<a href="https://colab.research.google.com/github/ankitkumarsahu454554-svg/Rare_Disease_Ai/blob/main/Rare_Disease_Prediction_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

required_files = [
    DATA / "clean_hpo_annotations.csv",
    DATA / "disease_level_split.csv",
    DATA / "three_model_comparison_summary.csv",
    DATA / "rare_disease_training_graph.graphml",
    DATA / "rare_disease_gnn_training_only.pt",
    PROJECT / "hp.obo",
]

print("\nPROJECT FILE CHECK")
missing = []

for path in required_files:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status}: {path}")
    if not path.exists():
        missing.append(path.name)

if missing:
    print("\nMissing files:", missing)
else:
    print("\nAll required files are available.")


Mounted at /content/drive

PROJECT FILE CHECK
OK: /content/drive/MyDrive/Rare Disease ai/processed/clean_hpo_annotations.csv
OK: /content/drive/MyDrive/Rare Disease ai/processed/disease_level_split.csv
OK: /content/drive/MyDrive/Rare Disease ai/processed/three_model_comparison_summary.csv
OK: /content/drive/MyDrive/Rare Disease ai/processed/rare_disease_training_graph.graphml
OK: /content/drive/MyDrive/Rare Disease ai/processed/rare_disease_gnn_training_only.pt
OK: /content/drive/MyDrive/Rare Disease ai/hp.obo

All required files are available.


In [ ]:

import pandas as pd
import numpy as np

annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)

comparison = pd.read_csv(
    DATA / "three_model_comparison_summary.csv"
)

split["split"] = split["split"].str.lower().str.strip()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

print("ANNOTATION ROWS:", len(annotations))
print("DISEASES:", annotations["database_id"].nunique())
print("PHENOTYPES:", annotations["hpo_id"].nunique())
print("TRAINING DISEASES:", len(train_ids))
print("HELD-OUT DISEASES:", len(test_ids))
print("TRAIN/TEST OVERLAP:", len(train_ids & test_ids))

print("\nEXISTING BENCHMARK")
print(comparison.to_string(index=False))

assert not (train_ids & test_ids), "Train/test leakage detected!"
print("\nVALIDATION CHECK PASSED")


ANNOTATION ROWS: 286047
DISEASES: 12879
PHENOTYPES: 11658
TRAINING DISEASES: 7340
HELD-OUT DISEASES: 1836
TRAIN/TEST OVERLAP: 0

EXISTING BENCHMARK
                   Model  Profiles  Mean Recall@10  Mean MRR
          Phenotype-only      1827        0.213532  0.520367
           Gene-informed      1827        0.133764  0.379349
GNN phenotype embeddings      1827        0.086528  0.278151

VALIDATION CHECK PASSED


In [ ]:

from collections import defaultdict
import numpy as np
import pandas as pd

# Build disease phenotype profiles from the curated knowledge base.
# This is for knowledge-base disease ranking, not unseen-disease generalization.
profiles = (
    annotations.groupby("database_id")["hpo_id"]
    .apply(lambda x: set(x.dropna().astype(str)))
    .to_dict()
)

# Information content: rarity of each phenotype across disease profiles.
term_count = defaultdict(int)
for terms in profiles.values():
    for term in terms:
        term_count[term] += 1

n_diseases = len(profiles)
ic = {
    term: -np.log(count / n_diseases)
    for term, count in term_count.items()
    if count > 0
}

def rank_diseases(input_phenotypes, top_k=10):
    """Rank known diseases by information-weighted phenotype overlap."""
    input_terms = set(input_phenotypes) & set(ic)
    if not input_terms:
        return pd.DataFrame(columns=["disease_id", "score"])

    denominator = sum(ic[t] for t in input_terms)
    if denominator <= 0:
        denominator = len(input_terms)

    results = []
    for disease_id, disease_terms in profiles.items():
        overlap = input_terms & disease_terms
        score = sum(ic[t] for t in overlap) / denominator
        results.append((disease_id, score))

    results.sort(key=lambda x: x[1], reverse=True)
    return pd.DataFrame(
        results[:top_k], columns=["disease_id", "score"]
    )

# Simulate patient profiles by hiding one phenotype from each eligible
# held-out disease. The full curated profiles remain the candidate database.
rng = np.random.default_rng(42)
test_disease_ids = sorted(test_ids)
evaluation = []

for disease_id in test_disease_ids:
    full_terms = profiles.get(disease_id, set())
    known_terms = sorted(full_terms & set(ic))

    if len(known_terms) < 2:
        continue

    # Hold out one phenotype as the simulated unknown finding.
    hidden_term = known_terms[int(rng.integers(0, len(known_terms)))]
    input_terms = set(known_terms) - {hidden_term}

    ranked = rank_diseases(input_terms, top_k=10)
    ranked_ids = ranked["disease_id"].tolist()

    # The correct disease is the target; report disease-level ranking metrics.
    full_ranking = rank_diseases(input_terms, top_k=len(profiles))
    all_ranked_ids = full_ranking["disease_id"].tolist()

    rank = (
        all_ranked_ids.index(disease_id) + 1
        if disease_id in all_ranked_ids else None
    )

    evaluation.append({
        "disease_id": disease_id,
        "input_phenotype_count": len(input_terms),
        "hidden_phenotype": hidden_term,
        "correct_disease_rank": rank,
        "recall_at_1": int(rank is not None and rank <= 1),
        "recall_at_5": int(rank is not None and rank <= 5),
        "recall_at_10": int(rank is not None and rank <= 10),
        "reciprocal_rank": 1 / rank if rank else 0.0,
    })

eval_df = pd.DataFrame(evaluation)
results_path = DATA / "phenotype_disease_ranking_evaluation.csv"
eval_df.to_csv(results_path, index=False)

print("Evaluation profiles:", len(eval_df))
print("Results saved to:", results_path)

if len(eval_df):
    print("\nDisease-ranking metrics")
    print("Recall@1: ", round(eval_df["recall_at_1"].mean(), 4))
    print("Recall@5: ", round(eval_df["recall_at_5"].mean(), 4))
    print("Recall@10:", round(eval_df["recall_at_10"].mean(), 4))
    print("MRR:      ", round(eval_df["reciprocal_rank"].mean(), 4))
else:
    print("No eligible profiles; check the disease and phenotype data.")


Evaluation profiles: 1832
Results saved to: /content/drive/MyDrive/Rare Disease ai/processed/phenotype_disease_ranking_evaluation.csv

Disease-ranking metrics
Recall@1:  0.9301
Recall@5:  0.9733
Recall@10: 0.9776
MRR:       0.9487


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "disease_level_split.csv")
results = pd.read_csv(
    DATA / "phenotype_disease_ranking_evaluation.csv"
)

print("Split columns:", split.columns.tolist())
print("Split counts:")
print(split["split"].value_counts().to_string())

print("\nEvaluation columns:", results.columns.tolist())
print("Evaluation profiles:", len(results))
print("Unique evaluated diseases:",
      results["disease_id"].nunique())

print("\nSaved results are preliminary; do not interpret them as diagnostic accuracy.")


Split columns: ['disease_id', 'split']
Split counts:
split
train    7340
test     1836

Evaluation columns: ['disease_id', 'input_phenotype_count', 'hidden_phenotype', 'correct_disease_rank', 'recall_at_1', 'recall_at_5', 'recall_at_10', 'reciprocal_rank']
Evaluation profiles: 1832
Unique evaluated diseases: 1832

Saved results are preliminary; do not interpret them as diagnostic accuracy.


In [ ]:

import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
split["split"] = split["split"].str.lower().str.strip()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
train_profiles = {d: profiles[d] for d in train_ids if d in profiles}

# Weight phenotypes by rarity across training disease profiles only.
counts = defaultdict(int)
for terms in train_profiles.values():
    for term in terms:
        counts[term] += 1

n = max(len(train_profiles), 1)
ic = {t: -np.log(c / n) for t, c in counts.items() if c > 0}

# Test only diseases with enough phenotypes to form input and target.
rng = np.random.default_rng(42)
rows = []

for disease in sorted(test_ids):
    terms = sorted(profiles.get(disease, set()) & set(ic))
    if len(terms) < 2:
        continue

    # Randomly hide 20% of terms, leaving at least one input term.
    n_hide = max(1, int(round(0.2 * len(terms))))
    n_hide = min(n_hide, len(terms) - 1)
    hidden_idx = set(rng.choice(len(terms), size=n_hide, replace=False))
    hidden = {terms[i] for i in hidden_idx}
    observed = set(terms) - hidden

    # Candidate list contains training diseases only.
    scored = []
    denom = sum(ic[t] for t in observed) or 1.0

    for candidate, candidate_terms in train_profiles.items():
        overlap = observed & candidate_terms
        score = sum(ic[t] for t in overlap) / denom
        scored.append((candidate, score))

    scored.sort(key=lambda x: x[1], reverse=True)

    # Because test disease IDs are excluded from candidates, direct
    # correct-disease rank is undefined for this split.
    # Record similarity to the nearest training disease for diagnostics.
    best_candidate, best_score = scored[0] if scored else (None, 0.0)

    rows.append({
        "test_disease_id": disease,
        "input_phenotypes": len(observed),
        "hidden_phenotypes": len(hidden),
        "nearest_training_disease": best_candidate,
        "nearest_training_score": best_score,
    })

out = pd.DataFrame(rows)
path = DATA / "heldout_profile_nearest_training_disease.csv"
out.to_csv(path, index=False)

print("Held-out profiles evaluated:", len(out))
print("Saved:", path)
print("\nThis is a nearest-training-disease diagnostic only.")
print("It does not provide Recall@K or MRR for the correct held-out disease,")
print("because the held-out disease is intentionally absent from candidates.")


Held-out profiles evaluated: 1829
Saved: /content/drive/MyDrive/Rare Disease ai/processed/heldout_profile_nearest_training_disease.csv

This is a nearest-training-disease diagnostic only.
It does not provide Recall@K or MRR for the correct held-out disease,
because the held-out disease is intentionally absent from candidates.


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
split["split"] = split["split"].str.lower().str.strip()

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
train_profiles = {d: profiles[d] for d in train_ids if d in profiles}
test_profiles = {d: profiles[d] for d in test_ids if d in profiles}

# Estimate phenotype information content using training diseases only.
freq = {}
for terms in train_profiles.values():
    for term in terms:
        freq[term] = freq.get(term, 0) + 1

n_train = max(len(train_profiles), 1)
ic = {t: -np.log(c / n_train) for t, c in freq.items()}

# Score all known disease profiles using the same phenotype-based rule.
# Candidate annotations are used for retrieval; no test disease identity
# embeddings are trained.
candidate_profiles = {
    **train_profiles,
    **{d: test_profiles[d] for d in test_profiles}
}

rng = np.random.default_rng(42)
rows = []

for disease_id, full_terms in sorted(test_profiles.items()):
    terms = sorted(set(full_terms) & set(ic))
    if len(terms) < 2:
        continue

    # Hide 20% of known phenotypes to simulate an incomplete profile.
    n_hide = min(max(1, round(0.2 * len(terms))), len(terms) - 1)
    hidden_idx = set(rng.choice(len(terms), size=n_hide, replace=False))
    observed = {t for i, t in enumerate(terms) if i not in hidden_idx}

    weights = {t: ic[t] for t in observed}
    denom = sum(weights.values()) or 1.0

    scores = []
    for candidate_id, candidate_terms in candidate_profiles.items():
        overlap = observed & candidate_terms
        score = sum(weights[t] for t in overlap) / denom
        scores.append((candidate_id, score))

    # Stable deterministic tie handling.
    scores.sort(key=lambda x: (-x[1], x[0]))
    ranking = [candidate for candidate, _ in scores]
    rank = ranking.index(disease_id) + 1

    rows.append({
        "disease_id": disease_id,
        "rank": rank,
        "recall_at_1": int(rank <= 1),
        "recall_at_5": int(rank <= 5),
        "recall_at_10": int(rank <= 10),
        "reciprocal_rank": 1 / rank
    })

metrics = pd.DataFrame(rows)
out_file = DATA / "disease_ranking_retrieval_metrics.csv"
metrics.to_csv(out_file, index=False)

print("Evaluated profiles:", len(metrics))
print("Recall@1:", round(metrics["recall_at_1"].mean(), 4))
print("Recall@5:", round(metrics["recall_at_5"].mean(), 4))
print("Recall@10:", round(metrics["recall_at_10"].mean(), 4))
print("MRR:", round(metrics["reciprocal_rank"].mean(), 4))
print("Saved:", out_file)
print("\nInterpretation: known-profile retrieval only; not clinical diagnostic accuracy.")


Evaluated profiles: 1829
Recall@1: 0.9333
Recall@5: 0.9727
Recall@10: 0.9787
MRR: 0.9503
Saved: /content/drive/MyDrive/Rare Disease ai/processed/disease_ranking_retrieval_metrics.csv

Interpretation: known-profile retrieval only; not clinical diagnostic accuracy.


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

results = pd.read_csv(DATA / "disease_ranking_retrieval_metrics.csv")

print("Profiles:", len(results))
print("Duplicate disease IDs:", results["disease_id"].duplicated().sum())
print("\nMetric summary:")
print(results[[
    "recall_at_1",
    "recall_at_5",
    "recall_at_10",
    "reciprocal_rank"
]].agg(["mean", "median", "std"]).round(4))

print("\nSaved baseline:", DATA / "disease_ranking_retrieval_metrics.csv")


Profiles: 1829
Duplicate disease IDs: 0

Metric summary:
        recall_at_1  recall_at_5  recall_at_10  reciprocal_rank
mean         0.9333       0.9727        0.9787           0.9503
median       1.0000       1.0000        1.0000           1.0000
std          0.2496       0.1631        0.1445           0.1926

Saved baseline: /content/drive/MyDrive/Rare Disease ai/processed/disease_ranking_retrieval_metrics.csv


In [ ]:

from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
ROOT = Path("/content/drive/MyDrive/Rare Disease ai")

ANN = DATA / "clean_hpo_annotations.csv"
SPLIT = DATA / "disease_level_split.csv"
OBO = ROOT / "hp.obo"

ann = pd.read_csv(
    ANN, low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(SPLIT, dtype={"disease_id": "string"})
split["split"] = split["split"].str.lower().str.strip()

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
train_profiles = {d: profiles[d] for d in train_ids if d in profiles}
test_profiles = {d: profiles[d] for d in test_ids if d in profiles}

# Parse HPO parent relationships.
parents = defaultdict(set)
current_id = None
in_term = False

with open(OBO, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line == "[Term]":
            current_id, in_term = None, True
        elif line.startswith("[") and line.endswith("]"):
            in_term = False
        elif in_term:
            if line.startswith("id: "):
                current_id = line[4:]
            elif line.startswith("is_a: ") and current_id:
                parents[current_id].add(line.split()[1])

ancestor_cache = {}

def ancestors(term):
    if term not in ancestor_cache:
        seen, stack = set(), list(parents.get(term, []))
        while stack:
            node = stack.pop()
            if node not in seen:
                seen.add(node)
                stack.extend(parents.get(node, []))
        seen.add(term)
        ancestor_cache[term] = seen
    return ancestor_cache[term]

# Train-only information content over propagated phenotype profiles.
propagated_train = {}
for disease, terms in train_profiles.items():
    expanded = set()
    for term in terms:
        expanded.update(ancestors(term))
    propagated_train[disease] = expanded

counts = defaultdict(int)
for terms in propagated_train.values():
    for term in terms:
        counts[term] += 1

n_train = max(len(propagated_train), 1)
ic = {t: -np.log(c / n_train) for t, c in counts.items() if c > 0}

# Candidate profiles use ontology propagation. Test disease annotations
# are used only as the target profile in this retrieval benchmark.
candidate_profiles = {}
for disease, terms in profiles.items():
    expanded = set()
    for term in terms:
        expanded.update(ancestors(term))
    candidate_profiles[disease] = expanded

rng = np.random.default_rng(42)
rows = []

for disease, raw_terms in sorted(test_profiles.items()):
    known = sorted(set(raw_terms) & set(ic))
    if len(known) < 2:
        continue

    hide_n = min(max(1, round(0.2 * len(known))), len(known) - 1)
    hidden_idx = set(rng.choice(len(known), size=hide_n, replace=False))
    observed = {t for i, t in enumerate(known) if i not in hidden_idx}

    observed_expanded = set()
    for term in observed:
        observed_expanded.update(ancestors(term))

    weights = {t: ic.get(t, 0.0) for t in observed}
    denom = sum(weights.values()) or 1.0

    scores = []
    for candidate, candidate_terms in candidate_profiles.items():
        # Direct and ancestor overlap, weighted by the specificity
        # of each observed term. Each observed term contributes at most once.
        score = sum(
            weight for term, weight in weights.items()
            if ancestors(term) & candidate_terms
        ) / denom
        scores.append((candidate, score))

    scores.sort(key=lambda x: (-x[1], x[0]))
    ranking = [candidate for candidate, _ in scores]
    rank = ranking.index(disease) + 1

    rows.append({
        "disease_id": disease,
        "rank": rank,
        "recall_at_1": int(rank <= 1),
        "recall_at_5": int(rank <= 5),
        "recall_at_10": int(rank <= 10),
        "reciprocal_rank": 1 / rank
    })

result = pd.DataFrame(rows)
output = DATA / "hierarchy_aware_retrieval_metrics.csv"
result.to_csv(output, index=False)

print("Profiles evaluated:", len(result))
print("Recall@1:", round(result["recall_at_1"].mean(), 4))
print("Recall@5:", round(result["recall_at_5"].mean(), 4))
print("Recall@10:", round(result["recall_at_10"].mean(), 4))
print("MRR:", round(result["reciprocal_rank"].mean(), 4))
print("Saved:", output)
print("Interpret as knowledge-base retrieval, not diagnostic accuracy.")


Profiles evaluated: 1829
Recall@1: 0.0
Recall@5: 0.0
Recall@10: 0.0
MRR: 0.0004
Saved: /content/drive/MyDrive/Rare Disease ai/processed/hierarchy_aware_retrieval_metrics.csv
Interpret as knowledge-base retrieval, not diagnostic accuracy.


In [ ]:

from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
ROOT = Path("/content/drive/MyDrive/Rare Disease ai")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
split["split"] = split["split"].str.lower().str.strip()

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

test_ids = set(split.loc[split["split"] == "test", "disease_id"])
profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

# Estimate information content from training diseases only.
train_ids = set(split.loc[split["split"] == "train", "disease_id"])
train_profiles = {d: profiles[d] for d in train_ids if d in profiles}

freq = defaultdict(int)
for terms in train_profiles.values():
    for term in terms:
        freq[term] += 1

n_train = max(len(train_profiles), 1)
ic = {t: -np.log(c / n_train) for t, c in freq.items() if c > 0}

# Parse HPO parent links.
obo = ROOT / "hp.obo"
parents = defaultdict(set)
current_id = None
in_term = False

with open(obo, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line == "[Term]":
            current_id, in_term = None, True
        elif line.startswith("[") and line.endswith("]"):
            in_term = False
        elif in_term:
            if line.startswith("id: "):
                current_id = line[4:]
            elif line.startswith("is_a: ") and current_id:
                parents[current_id].add(line.split()[1])

cache = {}

def ancestors(term):
    if term not in cache:
        seen = set()
        stack = list(parents.get(term, []))
        while stack:
            node = stack.pop()
            if node not in seen:
                seen.add(node)
                stack.extend(parents.get(node, []))
        seen.add(term)
        cache[term] = seen
    return cache[term]

# Candidate disease profiles are curated knowledge-base profiles.
candidates = {
    d: set(terms) for d, terms in profiles.items()
    if terms
}

rng = np.random.default_rng(42)
records = []

for disease in sorted(test_ids):
    full = sorted(set(profiles.get(disease, set())) & set(ic))
    if len(full) < 2:
        continue

    hide_n = min(max(1, round(0.2 * len(full))), len(full) - 1)
    hidden_idx = set(rng.choice(len(full), size=hide_n, replace=False))
    observed = {t for i, t in enumerate(full) if i not in hidden_idx}
    weights = {t: ic[t] for t in observed}
    denom = sum(weights.values()) or 1.0

    direct_scores = []
    hierarchy_scores = []

    for candidate, candidate_terms in candidates.items():
        # Method A: direct phenotype overlap.
        direct = sum(
            w for term, w in weights.items()
            if term in candidate_terms
        ) / denom

        # Method B: each input term is matched to a candidate term
        # only if the terms are identical or one is an ancestor of the other.
        # Similarity is discounted for broader, less specific matches.
        candidate_ancestors = set()
        for term in candidate_terms:
            candidate_ancestors.update(ancestors(term))

        hscore = 0.0
        for term, w in weights.items():
            if term in candidate_terms:
                hscore += w
            elif ancestors(term) & candidate_terms:
                # Match to a broader candidate phenotype: partial credit.
                hscore += 0.25 * w
            elif term in candidate_ancestors:
                # Candidate phenotype is more specific than the input term.
                hscore += 0.5 * w

        hierarchy = hscore / denom
        direct_scores.append((candidate, direct))
        hierarchy_scores.append((candidate, hierarchy))

    def get_rank(scored):
        # Deterministic tie handling; disease IDs break score ties.
        scored.sort(key=lambda x: (-x[1], x[0]))
        ranked_ids = [x[0] for x in scored]
        return ranked_ids.index(disease) + 1

    rd = get_rank(direct_scores)
    rh = get_rank(hierarchy_scores)

    records.append({
        "disease_id": disease,
        "direct_rank": rd,
        "direct_reciprocal_rank": 1 / rd,
        "direct_recall_at_1": int(rd <= 1),
        "direct_recall_at_5": int(rd <= 5),
        "direct_recall_at_10": int(rd <= 10),
        "hierarchy_rank": rh,
        "hierarchy_reciprocal_rank": 1 / rh,
        "hierarchy_recall_at_1": int(rh <= 1),
        "hierarchy_recall_at_5": int(rh <= 5),
        "hierarchy_recall_at_10": int(rh <= 10),
    })

result = pd.DataFrame(records)
out_path = DATA / "direct_vs_hierarchy_scoring_comparison.csv"
result.to_csv(out_path, index=False)

summary = pd.DataFrame([
    {
        "method": "Direct overlap",
        "profiles": len(result),
        "Recall@1": result["direct_recall_at_1"].mean(),
        "Recall@5": result["direct_recall_at_5"].mean(),
        "Recall@10": result["direct_recall_at_10"].mean(),
        "MRR": result["direct_reciprocal_rank"].mean(),
    },
    {
        "method": "HPO-aware",
        "profiles": len(result),
        "Recall@1": result["hierarchy_recall_at_1"].mean(),
        "Recall@5": result["hierarchy_recall_at_5"].mean(),
        "Recall@10": result["hierarchy_recall_at_10"].mean(),
        "MRR": result["hierarchy_reciprocal_rank"].mean(),
    },
])

print(summary.round(4).to_string(index=False))
print("\nSaved detailed comparison:", out_path)
print("Note: curated-profile retrieval only, not clinical diagnostic accuracy.")


        method  profiles  Recall@1  Recall@5  Recall@10    MRR
Direct overlap      1829    0.9306    0.9716     0.9787 0.9483
     HPO-aware      1829    0.9306    0.9716     0.9787 0.9483

Saved detailed comparison: /content/drive/MyDrive/Rare Disease ai/processed/direct_vs_hierarchy_scoring_comparison.csv
Note: curated-profile retrieval only, not clinical diagnostic accuracy.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

candidates = [
    PROJECT / "genes_to_disease.txt",
    DATA / "genes_to_disease.txt",
    PROJECT / "genes_to_phenotype.txt",
]

for path in candidates:
    print(("FOUND " if path.exists() else "MISSING "), path)


FOUND  /content/drive/MyDrive/Rare Disease ai/genes_to_disease.txt
MISSING  /content/drive/MyDrive/Rare Disease ai/processed/genes_to_disease.txt
FOUND  /content/drive/MyDrive/Rare Disease ai/genes_to_phenotype.txt


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
gene_file = PROJECT / "genes_to_disease.txt"

with open(gene_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 7:
            break


ncbi_gene_id	gene_symbol	association_type	disease_id	source
NCBIGene:64170	CARD9	MENDELIAN	OMIM:212050	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:51256	TBC1D7	MENDELIAN	OMIM:248000	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:28981	IFT81	MENDELIAN	OMIM:617895	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:8216	LZTR1	MENDELIAN	OMIM:616564	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:6505	SLC1A1	POLYGENIC	OMIM:615232	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:5949	RBP3	MENDELIAN	OMIM:615233	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:583	BBS2	MENDELIAN	OMIM:616562	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
GENE_FILE = PROJECT / "genes_to_disease.txt"

# Read the tab-separated gene-disease file.
gene_df = pd.read_csv(
    GENE_FILE,
    sep="\t",
    comment="#",
    dtype=str,
    low_memory=False
)

gene_df.columns = gene_df.columns.str.strip()
gene_df["disease_id"] = gene_df["disease_id"].str.strip()
gene_df["gene_symbol"] = gene_df["gene_symbol"].str.strip()

gene_df = gene_df.dropna(subset=["disease_id", "gene_symbol"])
gene_df = gene_df[
    (gene_df["disease_id"] != "") &
    (gene_df["gene_symbol"] != "")
].drop_duplicates(["disease_id", "gene_symbol"])

# Save a normalized mapping for later inference.
gene_map = (
    gene_df.groupby("disease_id")["gene_symbol"]
    .apply(lambda x: sorted(set(x)))
    .to_dict()
)

mapping = pd.DataFrame([
    {"disease_id": disease, "gene_symbols": ";".join(genes)}
    for disease, genes in gene_map.items()
])

mapping_path = DATA / "normalized_disease_gene_mapping.csv"
mapping.to_csv(mapping_path, index=False)

print("Gene-disease associations:", len(gene_df))
print("Diseases with gene associations:", len(gene_map))
print("Unique gene symbols:", gene_df["gene_symbol"].nunique())
print("Association types:")
print(gene_df["association_type"].value_counts(dropna=False).to_string())
print("Saved:", mapping_path)

print("\nImportant: these are disease-associated genes, not patient-specific genetic findings.")


Gene-disease associations: 16097
Diseases with gene associations: 11168
Unique gene symbols: 5550
Association types:
association_type
UNKNOWN      8423
MENDELIAN    7093
POLYGENIC     581
Saved: /content/drive/MyDrive/Rare Disease ai/processed/normalized_disease_gene_mapping.csv

Important: these are disease-associated genes, not patient-specific genetic findings.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

# Load disease phenotype profiles.
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

phenotype_map = (
    ann.groupby("database_id")["hpo_id"]
    .apply(set)
    .to_dict()
)

# Load normalized disease-gene mapping.
gene_df = pd.read_csv(
    DATA / "normalized_disease_gene_mapping.csv",
    dtype=str
).fillna("")

gene_map = {
    row["disease_id"]: set(
        g.strip().upper()
        for g in row["gene_symbols"].split(";")
        if g.strip()
    )
    for _, row in gene_df.iterrows()
}

# Estimate phenotype information content from training diseases only.
split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"]
)

term_frequency = {}
for disease in train_ids:
    for term in phenotype_map.get(disease, set()):
        term_frequency[term] = term_frequency.get(term, 0) + 1

n_train = max(len(train_ids), 1)
ic = {
    term: -np.log(count / n_train)
    for term, count in term_frequency.items()
    if count > 0
}

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    """
    Rank diseases from HPO phenotype IDs and optional patient candidate genes.

    Scores are ranking scores, NOT calibrated probabilities.
    Patient genes should be supported by actual genetic analysis.
    """
    observed = {
        str(term).strip()
        for term in patient_hpo_terms
        if str(term).strip() in ic
    }

    if not observed:
        raise ValueError(
            "No recognized HPO terms. Supply valid IDs such as HP:0001250."
        )

    weights = {term: ic[term] for term in observed}
    denom = sum(weights.values()) or 1.0

    genes = {
        str(gene).strip().upper()
        for gene in (patient_genes or [])
        if str(gene).strip()
    }

    scored = []
    for disease, disease_terms in phenotype_map.items():
        overlap = observed & disease_terms
        phenotype_score = sum(weights[t] for t in overlap) / denom

        disease_genes = gene_map.get(disease, set())
        gene_score = (
            len(genes & disease_genes) / len(genes)
            if genes else 0.0
        )

        # Use phenotype-only scoring when patient genes are unavailable.
        final_score = (
            (1 - gene_weight) * phenotype_score + gene_weight * gene_score
            if genes else phenotype_score
        )

        scored.append({
            "disease_id": disease,
            "phenotype_score": phenotype_score,
            "gene_score": gene_score if genes else np.nan,
            "combined_score": final_score,
            "matching_phenotypes": len(overlap),
            "matching_genes": (
                ";".join(sorted(genes & disease_genes)) if genes else ""
            )
        })

    result = pd.DataFrame(scored).sort_values(
        ["combined_score", "phenotype_score", "disease_id"],
        ascending=[False, False, True]
    ).head(top_k).reset_index(drop=True)

    result.insert(0, "rank", range(1, len(result) + 1))
    return result

print("Prediction function ready.")
print("Phenotype profiles:", len(phenotype_map))
print("Diseases with gene mappings:", len(gene_map))
print("\nExample call:")
print("predict_diseases(['HP:0001250', 'HP:0001263'], top_k=10)")


Prediction function ready.
Phenotype profiles: 12879
Diseases with gene mappings: 11168

Example call:
predict_diseases(['HP:0001250', 'HP:0001263'], top_k=10)


In [ ]:
predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)

,rank,disease_id,phenotype_score,gene_score,combined_score,matching_phenotypes,matching_genes
0,1,DECIPHER:45,1.0,NaN,1.0,2,
1,2,OMIM:100300,1.0,NaN,1.0,2,
2,3,OMIM:103050,1.0,NaN,1.0,2,
3,4,OMIM:105830,1.0,NaN,1.0,2,
4,5,OMIM:114290,1.0,NaN,1.0,2,
5,6,OMIM:117360,1.0,NaN,1.0,2,
6,7,OMIM:117550,1.0,NaN,1.0,2,
7,8,OMIM:122470,1.0,NaN,1.0,2,
8,9,OMIM:124000,1.0,NaN,1.0,2,
9,10,OMIM:135900,1.0,NaN,1.0,2,


In [ ]:

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    observed = {
        str(t).strip()
        for t in patient_hpo_terms
        if str(t).strip() in ic
    }

    if not observed:
        raise ValueError("No recognized HPO terms. Use valid HPO IDs.")

    weights = {t: ic[t] for t in observed}
    input_weight = sum(weights.values()) or 1.0

    genes = {
        str(g).strip().upper()
        for g in (patient_genes or [])
        if str(g).strip()
    }

    results = []

    for disease, disease_terms in phenotype_map.items():
        overlap = observed & disease_terms

        # How much of the patient's weighted phenotype input is explained?
        precision = sum(weights[t] for t in overlap) / input_weight

        # How much of the candidate disease profile is explained?
        recall = len(overlap) / max(len(disease_terms), 1)

        # Penalize candidates that match only a tiny part of their profile.
        phenotype_score = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        disease_genes = gene_map.get(disease, set())
        matching_genes = genes & disease_genes

        gene_score = (
            len(matching_genes) / len(genes) if genes else np.nan
        )

        combined_score = (
            (1 - gene_weight) * phenotype_score + gene_weight * gene_score
            if genes else phenotype_score
        )

        results.append({
            "disease_id": disease,
            "phenotype_score": phenotype_score,
            "gene_score": gene_score,
            "combined_score": combined_score,
            "matching_phenotypes": len(overlap),
            "matching_genes": ";".join(sorted(matching_genes))
        })

    result = pd.DataFrame(results).sort_values(
        ["combined_score", "matching_phenotypes", "disease_id"],
        ascending=[False, False, True]
    ).head(top_k).reset_index(drop=True)

    result.insert(0, "rank", range(1, len(result) + 1))
    return result


In [ ]:
predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)

,rank,disease_id,phenotype_score,gene_score,combined_score,matching_phenotypes,matching_genes
0,1,ORPHA:2798,0.666667,NaN,0.666667,2,
1,2,OMIM:300852,0.571429,NaN,0.571429,2,
2,3,OMIM:611718,0.571429,NaN,0.571429,2,
3,4,OMIM:605375,0.505061,NaN,0.505061,1,
4,5,ORPHA:46348,0.505061,NaN,0.505061,1,
5,6,OMIM:616887,0.500000,NaN,0.500000,2,
6,7,ORPHA:85336,0.500000,NaN,0.500000,2,
7,8,OMIM:613657,0.444444,NaN,0.444444,2,
8,9,OMIM:619615,0.444444,NaN,0.444444,2,
9,10,DECIPHER:74,0.403232,NaN,0.403232,1,


In [ ]:

test_genes = ["SCN1A", "KCNQ2", "MECP2"]

gene_test = predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    patient_genes=test_genes,
    top_k=10
)

print(gene_test.to_string(index=False))


 rank  disease_id  phenotype_score  gene_score  combined_score  matching_phenotypes matching_genes
    1  ORPHA:2798         0.666667         0.0        0.500000                    2               
    2 OMIM:300852         0.571429         0.0        0.428571                    2               
    3 OMIM:611718         0.571429         0.0        0.428571                    2               
    4 OMIM:605375         0.505061         0.0        0.378795                    1               
    5 ORPHA:46348         0.505061         0.0        0.378795                    1               
    6 OMIM:616887         0.500000         0.0        0.375000                    2               
    7 ORPHA:85336         0.500000         0.0        0.375000                    2               
    8 OMIM:613657         0.444444         0.0        0.333333                    2               
    9 OMIM:619615         0.444444         0.0        0.333333                    2               
   10 DECI

In [ ]:

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    observed = {
        str(t).strip()
        for t in patient_hpo_terms
        if str(t).strip() in ic
    }

    if not observed:
        raise ValueError("No recognized HPO terms. Use valid HPO IDs.")

    weights = {t: ic[t] for t in observed}
    denominator = sum(weights.values()) or 1.0

    genes = {
        str(g).strip().upper()
        for g in (patient_genes or [])
        if str(g).strip()
    }

    results = []

    for disease, disease_terms in phenotype_map.items():
        overlap = observed & disease_terms

        precision = sum(weights[t] for t in overlap) / denominator
        recall = len(overlap) / max(len(disease_terms), 1)

        phenotype_score = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        disease_genes = gene_map.get(disease, set())
        matching_genes = genes & disease_genes
        gene_score = len(matching_genes) / len(genes) if genes else np.nan

        if genes and matching_genes:
            combined_score = (
                (1 - gene_weight) * phenotype_score
                + gene_weight * gene_score
            )
        else:
            combined_score = phenotype_score

        results.append({
            "disease_id": disease,
            "phenotype_score": phenotype_score,
            "gene_score": gene_score,
            "combined_score": combined_score,
            "matching_phenotypes": len(overlap),
            "matching_genes": ";".join(sorted(matching_genes))
        })

    result = pd.DataFrame(results).sort_values(
        ["combined_score", "matching_phenotypes", "disease_id"],
        ascending=[False, False, True]
    ).head(top_k).reset_index(drop=True)

    result.insert(0, "rank", range(1, len(result) + 1))
    return result


print("Updated prediction function successfully.")


Updated prediction function successfully.


In [ ]:

predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    patient_genes=["SCN1A", "KCNQ2", "MECP2"],
    top_k=10
)


,rank,disease_id,phenotype_score,gene_score,combined_score,matching_phenotypes,matching_genes
0,1,ORPHA:2798,0.666667,0.0,0.666667,2,
1,2,OMIM:300852,0.571429,0.0,0.571429,2,
2,3,OMIM:611718,0.571429,0.0,0.571429,2,
3,4,OMIM:605375,0.505061,0.0,0.505061,1,
4,5,ORPHA:46348,0.505061,0.0,0.505061,1,
5,6,OMIM:616887,0.500000,0.0,0.500000,2,
6,7,ORPHA:85336,0.500000,0.0,0.500000,2,
7,8,OMIM:613657,0.444444,0.0,0.444444,2,
8,9,OMIM:619615,0.444444,0.0,0.444444,2,
9,10,DECIPHER:74,0.403232,0.0,0.403232,1,


In [ ]:

# Build a disease-ID-to-name mapping from the existing annotations.
disease_names = (
    ann.dropna(subset=["database_id", "disease_name"])
    .assign(
        database_id=lambda df: df["database_id"].str.strip(),
        disease_name=lambda df: df["disease_name"].astype(str).str.strip()
    )
    .groupby("database_id")["disease_name"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

def explain_predictions(patient_hpo_terms, patient_genes=None, top_k=10):
    results = predict_diseases(
        patient_hpo_terms=patient_hpo_terms,
        patient_genes=patient_genes,
        top_k=top_k
    ).copy()

    results["disease_name"] = results["disease_id"].map(disease_names)
    results["disease_name"] = results["disease_name"].fillna(
        "Disease name unavailable in annotations"
    )

    # Display matching phenotype IDs for transparency.
    observed = set(patient_hpo_terms)
    results["matching_phenotype_ids"] = results["disease_id"].apply(
        lambda d: ";".join(sorted(observed & phenotype_map.get(d, set())))
    )

    columns = [
        "rank", "disease_id", "disease_name", "combined_score",
        "matching_phenotypes", "matching_phenotype_ids",
        "matching_genes"
    ]
    return results[columns]

print("Explanation function ready.")


Explanation function ready.


In [ ]:

explain_predictions(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)



,rank,disease_id,disease_name,combined_score,matching_phenotypes,matching_phenotype_ids,matching_genes
0,1,ORPHA:2798,Pachygyria-intellectual disability-epilepsy sy...,0.666667,2,HP:0001250;HP:0001263,
1,2,OMIM:300852,"Intellectual developmental disorder, X-linked 88",0.571429,2,HP:0001250;HP:0001263,
2,3,OMIM:611718,"Hypomagnesemia 4, renal",0.571429,2,HP:0001250;HP:0001263,
3,4,OMIM:605375,"Epilepsy, nocturnal frontal lobe, 3",0.505061,1,HP:0001250,
4,5,ORPHA:46348,Paroxysmal extreme pain disorder,0.505061,1,HP:0001250,
5,6,OMIM:616887,"Intellectual developmental disorder, autosomal...",0.500000,2,HP:0001250;HP:0001263,
6,7,ORPHA:85336,"X-linked neurodegenerative syndrome, Hamel type",0.500000,2,HP:0001250;HP:0001263,
7,8,OMIM:613657,D-2-hydroxyglutaric aciduria 2,0.444444,2,HP:0001250;HP:0001263,
8,9,OMIM:619615,"Deafness, autosomal recessive 119",0.444444,2,HP:0001250;HP:0001263,
9,10,DECIPHER:74,15q13.3 microdeletion syndrome,0.403232,1,HP:0001250,


In [ ]:

from pathlib import Path
import json

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
MODEL_DIR = Path("/content/drive/MyDrive/Rare Disease ai/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the model configuration and artifact locations.
config = {
    "model_name": "Rare Disease Phenotype Ranking Prototype",
    "version": "1.0",
    "phenotype_scoring": "information_content_weighted_f1",
    "gene_weight": 0.25,
    "phenotype_annotations": str(DATA / "clean_hpo_annotations.csv"),
    "gene_mapping": str(DATA / "normalized_disease_gene_mapping.csv"),
    "disease_split": str(DATA / "disease_level_split.csv"),
    "clinical_validation": "not_yet_performed",
    "scores_are_probabilities": False
}

config_path = MODEL_DIR / "prediction_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Configuration saved:", config_path)
print("Prediction function is available in the current notebook session.")
print("Clinical validation is still required before real-world use.")


Configuration saved: /content/drive/MyDrive/Rare Disease ai/model/prediction_config.json
Prediction function is available in the current notebook session.
Clinical validation is still required before real-world use.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Current functions available:")
print("predict_diseases:", callable(globals().get("predict_diseases")))
print("explain_predictions:", callable(globals().get("explain_predictions")))

if not callable(globals().get("predict_diseases")):
    raise RuntimeError("Run the cell defining predict_diseases first.")

if not callable(globals().get("explain_predictions")):
    raise RuntimeError("Run the cell defining explain_predictions first.")

print("\nBoth functions are available in this session.")
print("Next we will export them into a reusable Python module.")


Current functions available:
predict_diseases: True
explain_predictions: True

Both functions are available in this session.
Next we will export them into a reusable Python module.


In [ ]:

import inspect
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

predictor = globals().get("predict_diseases")
explainer = globals().get("explain_predictions")

if not callable(predictor) or not callable(explainer):
    raise RuntimeError(
        "Run the cells defining both prediction functions first."
    )

# Export the function source code.
predictor_source = inspect.getsource(predictor)
explainer_source = inspect.getsource(explainer)

module_source = '''"""Rare Disease Prediction AI — research prototype.

This module contains ranking functions exported from the Colab notebook.
It is not a clinically validated diagnostic system.
"""

import pandas as pd
import numpy as np

''' + predictor_source + "\n\n" + explainer_source + "\n"

module_path = MODEL_DIR / "rare_disease_predictor.py"
module_path.write_text(module_source, encoding="utf-8")

print("Exported:", module_path)
print("Bytes written:", module_path.stat().st_size)
print("Note: the module also requires its data mappings and IC values to be loaded.")


Exported: /content/drive/MyDrive/Rare Disease ai/model/rare_disease_predictor.py
Bytes written: 2993
Note: the module also requires its data mappings and IC values to be loaded.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

loader_code = r'''
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

def load_prediction_data():
    ann = pd.read_csv(
        DATA / "clean_hpo_annotations.csv",
        low_memory=False,
        dtype={"database_id": "string", "hpo_id": "string"}
    ).dropna(subset=["database_id", "hpo_id"])

    ann["database_id"] = ann["database_id"].str.strip()
    ann["hpo_id"] = ann["hpo_id"].str.strip()

    phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

    gene_df = pd.read_csv(
        DATA / "normalized_disease_gene_mapping.csv",
        dtype=str
    ).fillna("")

    gene_map = {
        row["disease_id"]: {
            g.strip().upper()
            for g in row["gene_symbols"].split(";")
            if g.strip()
        }
        for _, row in gene_df.iterrows()
    }

    split = pd.read_csv(
        DATA / "disease_level_split.csv",
        dtype={"disease_id": "string"}
    )
    train_ids = set(
        split.loc[split["split"].str.lower() == "train", "disease_id"]
    )

    counts = defaultdict(int)
    for disease in train_ids:
        for term in phenotype_map.get(disease, set()):
            counts[term] += 1

    n_train = max(len(train_ids), 1)
    ic = {
        term: -np.log(count / n_train)
        for term, count in counts.items() if count > 0
    }

    disease_names = (
        ann.dropna(subset=["disease_name"])
        .groupby("database_id")["disease_name"]
        .agg(lambda x: x.value_counts().index[0])
        .to_dict()
    )

    return {
        "annotations": ann,
        "phenotype_map": phenotype_map,
        "gene_map": gene_map,
        "ic": ic,
        "disease_names": disease_names
    }
'''

loader_path = MODEL_DIR / "rare_disease_data_loader.py"
loader_path.write_text(loader_code, encoding="utf-8")

print("Saved reusable data loader:", loader_path)
print("It loads annotations, gene mappings, training-only IC weights, and disease names.")


Saved reusable data loader: /content/drive/MyDrive/Rare Disease ai/model/rare_disease_data_loader.py
It loads annotations, gene mappings, training-only IC weights, and disease names.


In [ ]:

import sys
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from rare_disease_data_loader import load_prediction_data

model_data = load_prediction_data()

ann = model_data["annotations"]
phenotype_map = model_data["phenotype_map"]
gene_map = model_data["gene_map"]
ic = model_data["ic"]
disease_names = model_data["disease_names"]

print("DATA LOADER TEST PASSED")
print("Disease phenotype profiles:", len(phenotype_map))
print("Diseases with gene mappings:", len(gene_map))
print("Phenotype information weights:", len(ic))
print("Disease names:", len(disease_names))


DATA LOADER TEST PASSED
Disease phenotype profiles: 12879
Diseases with gene mappings: 11168
Phenotype information weights: 9982
Disease names: 12879


In [ ]:

import sys
from pathlib import Path
import importlib

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from rare_disease_data_loader import load_prediction_data

# Load all required data into a single dictionary.
model_data = load_prediction_data()

# Check that every dependency required by the predictor is present.
required = [
    "annotations",
    "phenotype_map",
    "gene_map",
    "ic",
    "disease_names",
]

missing = [key for key in required if key not in model_data]
if missing:
    raise RuntimeError(f"Missing predictor dependencies: {missing}")

print("Prediction engine dependencies verified.")
print("Ready to build the standalone predictor module.")


Prediction engine dependencies verified.
Ready to build the standalone predictor module.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

module_code = r'''
from rare_disease_data_loader import load_prediction_data

_DATA = load_prediction_data()
PHENOTYPE_MAP = _DATA["phenotype_map"]
GENE_MAP = _DATA["gene_map"]
IC = _DATA["ic"]
DISEASE_NAMES = _DATA["disease_names"]

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    observed = {
        str(t).strip() for t in patient_hpo_terms
        if str(t).strip() in IC
    }
    if not observed:
        raise ValueError("No recognized HPO terms were supplied.")

    weights = {t: IC[t] for t in observed}
    denominator = sum(weights.values()) or 1.0
    genes = {
        str(g).strip().upper() for g in (patient_genes or [])
        if str(g).strip()
    }

    results = []
    for disease, disease_terms in PHENOTYPE_MAP.items():
        overlap = observed & disease_terms
        precision = sum(weights[t] for t in overlap) / denominator
        recall = len(overlap) / max(len(disease_terms), 1)
        phenotype_score = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        matching_genes = genes & GENE_MAP.get(disease, set())
        gene_score = len(matching_genes) / len(genes) if genes else None

        # Do not penalize candidates when no patient gene matches.
        if genes and matching_genes:
            combined_score = (
                (1 - gene_weight) * phenotype_score
                + gene_weight * gene_score
            )
        else:
            combined_score = phenotype_score

        results.append({
            "disease_id": disease,
            "disease_name": DISEASE_NAMES.get(
                disease, "Disease name unavailable"
            ),
            "phenotype_score": phenotype_score,
            "gene_score": gene_score,
            "combined_score": combined_score,
            "matching_phenotypes": len(overlap),
            "matching_phenotype_ids": ";".join(sorted(overlap)),
            "matching_genes": ";".join(sorted(matching_genes))
        })

    import pandas as pd
    result = pd.DataFrame(results).sort_values(
        ["combined_score", "matching_phenotypes", "disease_id"],
        ascending=[False, False, True]
    ).head(top_k).reset_index(drop=True)

    result.insert(0, "rank", range(1, len(result) + 1))
    return result
'''

module_path = MODEL_DIR / "rare_disease_ai.py"
module_path.write_text(module_code, encoding="utf-8")

print("Standalone predictor saved:", module_path)
print("Next: import it and test it.")


Standalone predictor saved: /content/drive/MyDrive/Rare Disease ai/model/rare_disease_ai.py
Next: import it and test it.


In [ ]:

import sys
import importlib

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import rare_disease_ai
importlib.reload(rare_disease_ai)

test_result = rare_disease_ai.predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)

print(test_result.to_string(index=False))


 rank  disease_id                                                disease_name  phenotype_score gene_score  combined_score  matching_phenotypes matching_phenotype_ids matching_genes
    1  ORPHA:2798        Pachygyria-intellectual disability-epilepsy syndrome         0.666667       None        0.666667                    2  HP:0001250;HP:0001263               
    2 OMIM:300852            Intellectual developmental disorder, X-linked 88         0.571429       None        0.571429                    2  HP:0001250;HP:0001263               
    3 OMIM:611718                                     Hypomagnesemia 4, renal         0.571429       None        0.571429                    2  HP:0001250;HP:0001263               
    4 OMIM:605375                         Epilepsy, nocturnal frontal lobe, 3         0.505061       None        0.505061                    1             HP:0001250               
    5 ORPHA:46348                            Paroxysmal extreme pain disorder         0.505061 

In [ ]:

from pathlib import Path
import sys

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import rare_disease_ai

def run_rare_disease_ai(hpo_terms, candidate_genes=None, top_k=10):
    """Return ranked disease candidates for supplied HPO terms."""
    results = rare_disease_ai.predict_diseases(
        patient_hpo_terms=hpo_terms,
        patient_genes=candidate_genes,
        top_k=top_k
    )

    display_columns = [
        "rank",
        "disease_id",
        "disease_name",
        "combined_score",
        "matching_phenotypes",
        "matching_phenotype_ids",
        "matching_genes"
    ]

    return results[display_columns]

print("Prediction interface ready.")
print("Call run_rare_disease_ai([...]) with valid HPO identifiers.")


Prediction interface ready.
Call run_rare_disease_ai([...]) with valid HPO identifiers.


In [ ]:

run_rare_disease_ai(
    hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)


,rank,disease_id,disease_name,combined_score,matching_phenotypes,matching_phenotype_ids,matching_genes
0,1,ORPHA:2798,Pachygyria-intellectual disability-epilepsy sy...,0.666667,2,HP:0001250;HP:0001263,
1,2,OMIM:300852,"Intellectual developmental disorder, X-linked 88",0.571429,2,HP:0001250;HP:0001263,
2,3,OMIM:611718,"Hypomagnesemia 4, renal",0.571429,2,HP:0001250;HP:0001263,
3,4,OMIM:605375,"Epilepsy, nocturnal frontal lobe, 3",0.505061,1,HP:0001250,
4,5,ORPHA:46348,Paroxysmal extreme pain disorder,0.505061,1,HP:0001250,
5,6,OMIM:616887,"Intellectual developmental disorder, autosomal...",0.500000,2,HP:0001250;HP:0001263,
6,7,ORPHA:85336,"X-linked neurodegenerative syndrome, Hamel type",0.500000,2,HP:0001250;HP:0001263,
7,8,OMIM:613657,D-2-hydroxyglutaric aciduria 2,0.444444,2,HP:0001250;HP:0001263,
8,9,OMIM:619615,"Deafness, autosomal recessive 119",0.444444,2,HP:0001250;HP:0001263,
9,10,DECIPHER:74,15q13.3 microdeletion syndrome,0.403232,1,HP:0001250,


In [ ]:

import pandas as pd

# Build a searchable phenotype-name lookup from the existing annotations.
phenotype_lookup = (
    ann.dropna(subset=["hpo_id", "disease_name"])
    [["hpo_id", "disease_name"]]
    .drop_duplicates()
    .rename(columns={"disease_name": "associated_disease_name"})
)

def search_phenotypes(query, limit=20):
    """Search annotation text for a phenotype-related disease description."""
    query = str(query).strip().lower()
    if not query:
        raise ValueError("Enter a search term.")

    matches = phenotype_lookup[
        phenotype_lookup["associated_disease_name"]
        .str.lower()
        .str.contains(query, regex=False, na=False)
    ].head(limit)

    return matches.reset_index(drop=True)

print("Phenotype search helper ready.")


Phenotype search helper ready.


In [ ]:

from pathlib import Path
import pandas as pd

OBO = Path("/content/drive/MyDrive/Rare Disease ai/hp.obo")

def load_hpo_terms(obo_path):
    terms = []
    current = None

    def save_term(term):
        if (
            term
            and term.get("id", "").startswith("HP:")
            and term.get("name")
            and not term.get("obsolete", False)
        ):
            terms.append({
                "id": term["id"],
                "name": term["name"]
            })

    with open(obo_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line == "[Term]":
                save_term(current)
                current = {}
            elif line.startswith("[") and line.endswith("]"):
                save_term(current)
                current = None
            elif current is not None:
                if line.startswith("id: "):
                    current["id"] = line[4:].strip()
                elif line.startswith("name: "):
                    current["name"] = line[6:].strip()
                elif line == "is_obsolete: true":
                    current["obsolete"] = True

    save_term(current)

    return pd.DataFrame(terms).drop_duplicates("id").reset_index(drop=True)

hpo_terms = load_hpo_terms(OBO)

def search_hpo(query, limit=20):
    query = str(query).strip().lower()
    if not query:
        raise ValueError("Enter a phenotype name.")

    return (
        hpo_terms[
            hpo_terms["name"].str.lower().str.contains(query, regex=False)
        ]
        .head(limit)
        .reset_index(drop=True)
    )

print("Official HPO terms loaded:", len(hpo_terms))
print("\nSeizure search test:")
print(search_hpo("seizure").to_string(index=False))


Official HPO terms loaded: 19894

Seizure search test:
        id                                                          name
HP:0001250                                                       Seizure
HP:0001327                              Photosensitive myoclonic seizure
HP:0002069                                Bilateral tonic-clonic seizure
HP:0002121                       Generalized non-motor (absence) seizure
HP:0002123                                 Generalized myoclonic seizure
HP:0002173                                         Hypoglycemic seizures
HP:0002197                                     Generalized-onset seizure
HP:0002199                                         Hypocalcemic seizures
HP:0002266                                          Focal clonic seizure
HP:0002349                                           Focal aware seizure
HP:0002373 Febrile seizure (within the age range of 3 months to 6 years)
HP:0002384                              Focal impaired awareness seiz

In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output

# Search and selection controls
search_box = widgets.Text(
    description="Find:",
    placeholder="e.g. seizure, hypotonia",
    layout=widgets.Layout(width="500px")
)

search_button = widgets.Button(
    description="Search HPO",
    button_style="primary"
)

results_dropdown = widgets.SelectMultiple(
    options=[],
    description="Matches:",
    rows=8,
    layout=widgets.Layout(width="700px")
)

selected_terms = widgets.SelectMultiple(
    options=[],
    description="Selected:",
    rows=6,
    layout=widgets.Layout(width="700px")
)

add_button = widgets.Button(description="Add selected")
remove_button = widgets.Button(description="Remove selected")
predict_button = widgets.Button(
    description="Rank diseases",
    button_style="success"
)

output = widgets.Output()
selected_map = {}

def do_search(_):
    matches = search_hpo(search_box.value, limit=100)
    results_dropdown.options = [
        (f"{row['name']} ({row['id']})", row["id"])
        for _, row in matches.iterrows()
    ]

def add_terms(_):
    for term in results_dropdown.value:
        selected_map[term] = hpo_terms.loc[
            hpo_terms["id"] == term, "name"
        ].iloc[0]
    selected_terms.options = [
        (f"{name} ({term})", term)
        for term, name in selected_map.items()
    ]

def remove_terms(_):
    for term in selected_terms.value:
        selected_map.pop(term, None)
    selected_terms.options = [
        (f"{name} ({term})", term)
        for term, name in selected_map.items()
    ]

def run_prediction(_):
    with output:
        clear_output()
        if not selected_map:
            print("Search and select at least one phenotype first.")
            return

        display(
            run_rare_disease_ai(
                hpo_terms=list(selected_map.keys()),
                top_k=10
            )
        )

search_button.on_click(do_search)
add_button.on_click(add_terms)
remove_button.on_click(remove_terms)
predict_button.on_click(run_prediction)

display(
    widgets.VBox([
        widgets.HTML("<h3>Rare Disease Prediction AI — Research Prototype</h3>"),
        search_box,
        search_button,
        results_dropdown,
        add_button,
        selected_terms,
        remove_button,
        predict_button,
        output
    ])
)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
split["split"] = split["split"].str.lower().str.strip()

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

# Information content is estimated from training diseases only.
freq = {}
for disease in train_ids:
    for term in profiles.get(disease, set()):
        freq[term] = freq.get(term, 0) + 1

n_train = max(len(train_ids), 1)
ic = {term: -np.log(count / n_train) for term, count in freq.items()}

rng = np.random.default_rng(42)
rows = []

for disease in sorted(test_ids):
    full_profile = set(profiles.get(disease, set()))
    known = sorted(full_profile & set(ic))

    if len(known) < 2:
        continue

    # Hold out 20% of the disease's known phenotypes.
    n_hide = min(max(1, round(0.2 * len(known))), len(known) - 1)
    hidden_idx = set(rng.choice(len(known), size=n_hide, replace=False))

    hidden = {term for i, term in enumerate(known) if i in hidden_idx}
    observed = {term for i, term in enumerate(known) if i not in hidden_idx}

    # Remove hidden terms from the target candidate profile.
    # Other diseases retain their known profiles.
    candidate_profiles = {
        candidate: set(terms)
        for candidate, terms in profiles.items()
        if terms
    }
    candidate_profiles[disease] = full_profile - hidden

    weights = {term: ic[term] for term in observed}
    denominator = sum(weights.values()) or 1.0

    scores = []
    for candidate, candidate_terms in candidate_profiles.items():
        overlap = observed & candidate_terms
        score = sum(weights[t] for t in overlap) / denominator
        scores.append((candidate, score))

    scores.sort(key=lambda x: (-x[1], x[0]))
    ranking = [candidate for candidate, _ in scores]

    # Rank the correct held-out disease against all candidates.
    rank = ranking.index(disease) + 1

    rows.append({
        "disease_id": disease,
        "observed_phenotypes": len(observed),
        "hidden_phenotypes": len(hidden),
        "correct_disease_rank": rank,
        "recall_at_1": int(rank <= 1),
        "recall_at_5": int(rank <= 5),
        "recall_at_10": int(rank <= 10),
        "reciprocal_rank": 1.0 / rank
    })

result = pd.DataFrame(rows)
output = DATA / "leakage_reduced_retrieval_evaluation.csv"
result.to_csv(output, index=False)

print("Profiles evaluated:", len(result))
if len(result):
    print("Recall@1:", round(result["recall_at_1"].mean(), 4))
    print("Recall@5:", round(result["recall_at_5"].mean(), 4))
    print("Recall@10:", round(result["recall_at_10"].mean(), 4))
    print("MRR:", round(result["reciprocal_rank"].mean(), 4))
else:
    print("No eligible profiles.")
print("Saved:", output)
print("Research retrieval evaluation only; not clinical diagnostic accuracy.")


Profiles evaluated: 1829
Recall@1: 0.9295
Recall@5: 0.9716
Recall@10: 0.9787
MRR: 0.9475
Saved: /content/drive/MyDrive/Rare Disease ai/processed/leakage_reduced_retrieval_evaluation.csv
Research retrieval evaluation only; not clinical diagnostic accuracy.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

print("Project folder exists:", PROJECT.exists())

if PROJECT.exists():
    print("\nProject folder contents:")
    for p in sorted(PROJECT.iterdir()):
        print(("FOLDER " if p.is_dir() else "FILE   ") + p.name)

    processed = PROJECT / "processed"
    if processed.exists():
        print("\nProcessed folder contents:")
        for p in sorted(processed.iterdir()):
            print(("FOLDER " if p.is_dir() else "FILE   ") + p.name)
else:
    print("Project folder not found. Check the folder name in Google Drive.")

Mounted at /content/drive
Project folder exists: True

Project folder contents:
FOLDER all_phenopackets
FILE   genes_to_disease.txt
FILE   genes_to_phenotype.txt
FILE   hp.obo
FOLDER model
FILE   phenotype.hpoa
FOLDER processed

Processed folder contents:
FILE   baseline_coverage_comparison_audit.csv
FILE   baseline_model_comparison.csv
FILE   candidate_coverage_audit.csv
FILE   clean_hpo_annotations.csv
FILE   direct_vs_hierarchy_scoring_comparison.csv
FILE   disease_level_split.csv
FILE   disease_ranking_retrieval_metrics.csv
FILE   evaluation_eligibility_audit.csv
FILE   evaluation_protocol_checks.csv
FILE   final_artifact_inventory.csv
FILE   final_consolidated_results.csv
FILE   final_project_handover_inventory.csv
FILE   gene_informed_phenotype_recovery_metrics.csv
FILE   gene_informed_phenotype_recovery_top10.csv
FILE   gnn_checkpoint_compatibility.csv
FILE   gnn_evaluation_results.csv
FILE   gnn_phenotype_recovery_metrics.csv
FILE   gnn_training_annotations.csv
FILE   gnn_zero_

In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

# Verify saved artifacts
required = {
    "annotations": DATA / "clean_hpo_annotations.csv",
    "training_graph": DATA / "rare_disease_training_graph.graphml",
    "training_checkpoint": DATA / "rare_disease_gnn_training_only.pt",
    "disease_split": DATA / "disease_level_split.csv",
    "results": DATA / "final_consolidated_results.csv",
}

print("Artifact verification:")
for name, path in required.items():
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

# Reload lightweight evaluation data
annotations = pd.read_csv(required["annotations"])
disease_split = pd.read_csv(required["disease_split"])
previous_results = pd.read_csv(required["results"])

print("\nAnnotations:", annotations.shape)
print("Disease split:", disease_split.shape)
print("Previous results:", previous_results.shape)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Artifact verification:
annotations: OK
training_graph: OK
training_checkpoint: OK
disease_split: OK
results: OK


/tmp/ipykernel_302/827847694.py:22: DtypeWarning: Columns (2,6,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations = pd.read_csv(required["annotations"])



Annotations: (286047, 12)
Disease split: (9176, 2)
Previous results: (11, 6)
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from pathlib import Path
import torch

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
checkpoint_path = DATA / "rare_disease_gnn_training_only.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} entries")
            print("Sample keys:", list(value.keys())[:8])
        elif isinstance(value, torch.Tensor):
            print(f"{key}: tensor shape {tuple(value.shape)}")
        elif isinstance(value, (str, int, float, bool, list, tuple)):
            print(f"{key}: {str(value)[:200]}")
        else:
            print(f"{key}: {type(value).__name__}")
else:
    print("Checkpoint contents require architecture inspection.")



Checkpoint type: dict
Checkpoint keys: ['model_state_dict', 'num_nodes', 'hidden_dim', 'disease_ids', 'phenotype_ids', 'disease_to_idx', 'phenotype_to_idx', 'training_disease_count', 'training_association_count', 'validation_loss', 'seed']

model_state_dict: dictionary with 5 entries
Sample keys: ['embedding.weight', 'conv1.bias', 'conv1.lin.weight', 'conv2.bias', 'conv2.lin.weight']
num_nodes: 17322
hidden_dim: 64
disease_ids: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400', 'OMIM:101600', 'OMIM:101800', 'OMIM:102200', 'OMIM:102370', 'OMIM:102500', 'OMIM:102700', 'OMIM:103050', 'OMIM:103285', 'OMI
phenotype_ids: ['HP:0000002', 'HP:0000003', 'HP:0000006', 'HP:0000007', 'HP:0000008', 'HP:0000009', 'HP:0000010', 'HP:0000011', 'HP:0000012', 'HP:0000013', 'HP:0000014', 'HP:0000015', 'HP:0000016', 'HP:0000017', 'HP

disease_to_idx: dictionary with 7340 entries
Sample keys: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400', 'OMIM:101600', '

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)

%pip install torch-geometric

import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
checkpoint_path = DATA / "rare_disease_gnn_training_only.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

class RareDiseaseGNN(nn.Module):
    def __init__(self, num_nodes, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, edge_index):
        x = self.embedding.weight
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = RareDiseaseGNN(
    num_nodes=checkpoint["num_nodes"],
    hidden_dim=checkpoint["hidden_dim"]
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("GNN restored successfully.")
print("Device:", device)
print("Training diseases:", checkpoint["training_disease_count"])
print("Training associations:", checkpoint["training_association_count"])
print("Validation loss:", checkpoint["validation_loss"])
print("Disease index entries:", len(checkpoint["disease_to_idx"]))
print("Phenotype index entries:", len(checkpoint["phenotype_to_idx"]))

PyTorch version: 2.11.0+cu128
CUDA version: 12.8
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.4 MB/s eta 0:00:00
GNN restored successfully.
Device: cuda
Training diseases: 7340
Training associations: 179653
Validation loss: 0.57393878698349
Disease index entries: 7340
Phenotype index entries: 9982


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "three_model_comparison_summary.csv",
    "baseline_model_comparison.csv",
    "hierarchy_aware_retrieval_metrics.csv",
    "ic_weighted_heldout_ranking_results.csv",
    "leakage_reduced_retrieval_evaluation.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 65}\n{name}")

    if path.exists():
        df = pd.read_csv(path, low_memory=False)
        print("Shape:", df.shape)
        print(df.head(8).to_string(index=False))
    else:
        print("File missing")



three_model_comparison_summary.csv
Shape: (3, 4)
                   Model  Profiles  Mean Recall@10  Mean MRR
          Phenotype-only      1827        0.213532  0.520367
           Gene-informed      1827        0.133764  0.379349
GNN phenotype embeddings      1827        0.086528  0.278151

baseline_model_comparison.csv
Shape: (2, 3)
                  Model  Mean Recall@10  Mean Reciprocal Rank
Phenotype co-occurrence        0.212949              0.518946
  Gene-informed scoring        0.198043              0.474340

hierarchy_aware_retrieval_metrics.csv
Shape: (1829, 6)
 disease_id  rank  recall_at_1  recall_at_5  recall_at_10  reciprocal_rank
OMIM:100100    49            0            0             0         0.020408
OMIM:101900    65            0            0             0         0.015385
OMIM:102530    75            0            0             0         0.013333
OMIM:102900    79            0            0             0         0.012658
OMIM:103500    86            0            0 

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

for name in [
    "evaluation_protocol_checks.csv",
    "self_match_sanity_check.csv",
    "phenotype_disease_ranking_evaluation.csv",
    "disease_ranking_retrieval_metrics.csv",
]:
    path = DATA / name
    print(f"\n--- {name} ---")

    if not path.exists():
        print("Not found")
        continue

    df = pd.read_csv(path, low_memory=False)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(5).to_string(index=False))

print("\n--- Disease split counts ---")
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
print("Columns:", split.columns.tolist())
print(split.nunique().to_string())
print(split.head(5).to_string(index=False))


--- evaluation_protocol_checks.csv ---
Shape: (6, 2)
Columns: ['Check', 'Value']
                              Check  Value
               Input phenotype rows  21647
       Hidden target phenotype rows  22562
          Input/target pair overlap      0
             Training diseases used   7340
Held-out diseases in training index      0

--- self_match_sanity_check.csv ---
Shape: (5, 3)
Columns: ['Metric', 'Value', 'Evaluation']
                    Metric       Value              Evaluation
         Queries evaluated 1832.000000 Self-match sanity check
  Mean rank of own profile    1.111354 Self-match sanity check
Median rank of own profile    1.000000 Self-match sanity check
     Top-1 self-match rate    0.979258 Self-match sanity check
 Recall@10 self-match rate    0.996179 Self-match sanity check

--- phenotype_disease_ranking_evaluation.csv ---
Shape: (1832, 8)
Columns: ['disease_id', 'input_phenotype_count', 'hidden_phenotype', 'correct_disease_rank', 'recall_at_1', 'recall_at_5'

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Check split integrity
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
split["split"] = split["split"].str.lower().str.strip()

print("DISEASE SPLIT")
print(split["split"].value_counts(dropna=False))
print("Duplicate disease IDs:", split["disease_id"].duplicated().sum())

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])
print("Train/test ID overlap:", len(train_ids & test_ids))

# Compare held-out IDs with the candidate coverage audit
coverage_path = DATA / "candidate_coverage_audit.csv"
if coverage_path.exists():
    coverage = pd.read_csv(coverage_path, low_memory=False)
    print("\nCANDIDATE COVERAGE AUDIT")
    print("Columns:", coverage.columns.tolist())
    print(coverage.head(8).to_string(index=False))

# Check saved ranking metrics
ranking_path = DATA / "disease_ranking_retrieval_metrics.csv"
ranking = pd.read_csv(ranking_path, low_memory=False)

print("\nRANKING METRICS")
print("Rows:", len(ranking))
print("Rank summary:")
print(ranking["rank"].describe())
print("Ranks <= 1:", int((ranking["rank"] <= 1).sum()))
print("Ranks <= 10:", int((ranking["rank"] <= 10).sum()))

if "disease_id" in ranking.columns:
    ranked_ids = set(ranking["disease_id"].dropna().astype(str))
    print("Ranking IDs overlapping test split:", len(ranked_ids & test_ids))
    print("Ranking IDs overlapping train split:", len(ranked_ids & train_ids))

DISEASE SPLIT
split
train    7340
test     1836
Name: count, dtype: int64
Duplicate disease IDs: 0
Train/test ID overlap: 0

CANDIDATE COVERAGE AUDIT
Columns: ['disease_id', 'input_count', 'covered_input_count', 'candidate_count']
 disease_id  input_count  covered_input_count  candidate_count
OMIM:100100            8                    8             7375
OMIM:101900            4                    4             4511
OMIM:102530            2                    2             6539
OMIM:102900            1                    0                0
OMIM:103500            4                    4             1021
OMIM:103780            1                    1              157
OMIM:104290            5                    5             6335
OMIM:105150            3                    3             5797

RANKING METRICS
Rows: 1829
Rank summary:
count    1829.000000
mean       11.486058
std       122.665070
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max      2534

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "gnn_training_annotations.csv",
    "heldout_input_phenotypes.csv",
    "heldout_target_phenotypes.csv",
    "phenotype_baseline_per_profile.csv",
    "evaluation_eligibility_audit.csv",
    "evaluation_protocol_checks.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 60}\n{name}")

    if not path.exists():
        print("MISSING")
        continue

    df = pd.read_csv(path, low_memory=False)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))


gnn_training_annotations.csv
Shape: (179653, 12)
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']
database_id                                  disease_name qualifier     hpo_id     reference evidence onset frequency sex modifier aspect               biocuration
OMIM:619340 Developmental and epileptic encephalopathy 96       NaN HP:0011097 PMID:31675180      PCS   NaN       1/2 NaN      NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96       NaN HP:0002187 PMID:31675180      PCS   NaN       1/1 NaN      NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96       NaN HP:0001518 PMID:31675180      PCS   NaN       1/2 NaN      NaN      P HPO:probinson[2021-06-21]

heldout_input_phenotypes.csv
Shape: (21647, 2)
Columns: ['disease_id', 'hpo_id']
 disease_id     hpo_id
OMIM:100100 HP:0030211
OMIM:100100

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

for folder in [PROJECT, PROJECT / "model"]:
    print(f"\n--- Python files in {folder} ---")
    if folder.exists():
        for path in sorted(folder.glob("*.py")):
            print(path.name)


--- Python files in /content/drive/MyDrive/Rare Disease ai ---

--- Python files in /content/drive/MyDrive/Rare Disease ai/model ---
rare_disease_ai.py
rare_disease_data_loader.py
rare_disease_predictor.py


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

keywords = (
    "predict", "rank", "score", "similarity",
    "candidate", "phenotype", "recall", "reciprocal"
)

for filename in [
    "rare_disease_ai.py",
    "rare_disease_predictor.py",
    "rare_disease_data_loader.py",
]:
    path = MODEL_DIR / filename
    print(f"\n{'=' * 70}\nFILE: {filename}")

    if not path.exists():
        print("File not found")
        continue

    lines = path.read_text(errors="replace").splitlines()
    matches = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    shown = set()
    for i in matches:
        start, end = max(0, i - 3), min(len(lines), i + 5)
        if any(j in shown for j in range(start, end)):
            continue
        print(f"\n--- lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")
            shown.add(j)


FILE: rare_disease_ai.py

--- lines 1-6 ---
1: 
2: from rare_disease_data_loader import load_prediction_data
3: 
4: _DATA = load_prediction_data()
5: PHENOTYPE_MAP = _DATA["phenotype_map"]
6: GENE_MAP = _DATA["gene_map"]

--- lines 7-14 ---
7: IC = _DATA["ic"]
8: DISEASE_NAMES = _DATA["disease_names"]
9: 
10: def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
11:                      gene_weight=0.25):
12:     observed = {
13:         str(t).strip() for t in patient_hpo_terms
14:         if str(t).strip() in IC

--- lines 24-31 ---
24:     }
25: 
26:     results = []
27:     for disease, disease_terms in PHENOTYPE_MAP.items():
28:         overlap = observed & disease_terms
29:         precision = sum(weights[t] for t in overlap) / denominator
30:         recall = len(overlap) / max(len(disease_terms), 1)
31:         phenotype_score = (

--- lines 34-41 ---
34:         )
35: 
36:         matching_genes = genes & GENE_MAP.get(disease, set())
37:         gene_score = l

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
)
split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype=str
)

train_ids = set(split.loc[split["split"] == "train", "disease_id"].str.strip())
test_ids = set(split.loc[split["split"] == "test", "disease_id"].str.strip())
all_candidate_ids = set(annotations["database_id"].dropna().str.strip())

print("Training diseases:", len(train_ids))
print("Held-out diseases:", len(test_ids))
print("All diseases in phenotype map:", len(all_candidate_ids))
print("Held-out diseases present in all-disease map:",
      len(test_ids & all_candidate_ids))
print("Training diseases present in all-disease map:",
      len(train_ids & all_candidate_ids))

train_annotations = annotations[
    annotations["database_id"].str.strip().isin(train_ids)
]
train_phenotype_map = (
    train_annotations.groupby("database_id")["hpo_id"]
    .apply(lambda s: set(s.dropna().str.strip()))
    .to_dict()
)

print("Training-only phenotype profiles:", len(train_phenotype_map))
print("Held-out IDs in training-only map:",
      len(set(train_phenotype_map) & test_ids))

Training diseases: 7340
Held-out diseases: 1836
All diseases in phenotype map: 12879
Held-out diseases present in all-disease map: 1836
Training diseases present in all-disease map: 7340
Training-only phenotype profiles: 7340
Held-out IDs in training-only map: 0


In [ ]:
  from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
)
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
inputs = pd.read_csv(DATA / "heldout_input_phenotypes.csv", dtype=str)
targets = pd.read_csv(DATA / "heldout_target_phenotypes.csv", dtype=str)

train_ids = set(split.loc[split["split"] == "train", "disease_id"].str.strip())
test_ids = set(split.loc[split["split"] == "test", "disease_id"].str.strip())

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
ann = ann[ann["database_id"].isin(train_ids)].dropna(subset=["hpo_id"])

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
input_map = inputs.groupby("disease_id")["hpo_id"].apply(set).to_dict()
target_map = targets.groupby("disease_id")["hpo_id"].apply(set).to_dict()

# Training-only phenotype information content
term_counts = defaultdict(int)
for terms in profiles.values():
    for term in terms:
        term_counts[term] += 1

n_train = max(len(profiles), 1)
ic = {term: -np.log(count / n_train) for term, count in term_counts.items()}

rows = []
for disease_id in sorted(test_ids & set(input_map) & set(target_map)):
    observed = input_map[disease_id]
    hidden = target_map[disease_id]
    observed_known = observed & set(ic)

    if not hidden or not observed_known:
        continue

    weights_total = sum(ic[t] for t in observed_known)
    scored = []

    for candidate, terms in profiles.items():
        overlap = observed_known & terms
        weighted_overlap = sum(ic[t] for t in overlap)
        precision = weighted_overlap / weights_total if weights_total else 0.0
        recall = len(overlap) / max(len(terms), 1)
        score = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        scored.append((candidate, score))

    scored.sort(key=lambda x: (-x[1], x[0]))
    # Score how well the top training-disease candidate predicts hidden phenotypes.
    top_candidate, _ = scored[0]
    recovered = len(hidden & profiles[top_candidate])
    rows.append({
        "heldout_disease_id": disease_id,
        "top_training_candidate": top_candidate,
        "hidden_phenotype_count": len(hidden),
        "recovered_hidden_phenotypes": recovered,
        "hidden_phenotype_recall": recovered / len(hidden),
        "candidate_count": len(scored),
    })

results = pd.DataFrame(rows)
out_path = DATA / "training_only_phenotype_recovery_audit.csv"
results.to_csv(out_path, index=False)

print("Evaluated profiles:", len(results))
if len(results):
    print("Mean hidden-phenotype recall:",
          round(results["hidden_phenotype_recall"].mean(), 4))
    print("Median hidden-phenotype recall:",
          round(results["hidden_phenotype_recall"].median(), 4))
    print("Mean candidate count:", round(results["candidate_count"].mean(), 1))
print("Saved:", out_path)

Evaluated profiles: 1828
Mean hidden-phenotype recall: 0.2055
Median hidden-phenotype recall: 0.1429
Mean candidate count: 7340.0
Saved: /content/drive/MyDrive/Rare Disease ai/processed/training_only_phenotype_recovery_audit.csv


In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
)
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
inputs = pd.read_csv(DATA / "heldout_input_phenotypes.csv", dtype=str)
targets = pd.read_csv(DATA / "heldout_target_phenotypes.csv", dtype=str)

train_ids = set(split.loc[split["split"] == "train", "disease_id"].str.strip())
test_ids = set(split.loc[split["split"] == "test", "disease_id"].str.strip())

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
ann = ann[ann["database_id"].isin(train_ids)].dropna(subset=["hpo_id"])

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
input_map = inputs.groupby("disease_id")["hpo_id"].apply(set).to_dict()
target_map = targets.groupby("disease_id")["hpo_id"].apply(set).to_dict()

term_counts = defaultdict(int)
for terms in profiles.values():
    for term in terms:
        term_counts[term] += 1

n_train = max(len(profiles), 1)
ic = {t: -np.log(c / n_train) for t, c in term_counts.items()}

methods = ["weighted_f1", "jaccard", "ic_overlap"]
rows = []

for disease_id in sorted(test_ids & set(input_map) & set(target_map)):
    observed = input_map[disease_id] & set(ic)
    hidden = target_map[disease_id]

    if not observed or not hidden:
        continue

    observed_ic = sum(ic[t] for t in observed)
    scored = {method: [] for method in methods}

    for candidate, terms in profiles.items():
        overlap = observed & terms
        union = observed | terms

        # Weighted F1 using training-only information content
        overlap_ic = sum(ic[t] for t in overlap)
        precision = overlap_ic / observed_ic if observed_ic else 0.0
        recall = overlap_ic / sum(ic[t] for t in terms if t in ic) \
            if any(t in ic for t in terms) else 0.0
        weighted_f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        # Standard Jaccard similarity
        jaccard = len(overlap) / len(union) if union else 0.0

        # Fraction of observed IC explained by candidate
        ic_overlap = overlap_ic / observed_ic if observed_ic else 0.0

        scored["weighted_f1"].append((candidate, weighted_f1))
        scored["jaccard"].append((candidate, jaccard))
        scored["ic_overlap"].append((candidate, ic_overlap))

    for method, candidates in scored.items():
        candidates.sort(key=lambda x: (-x[1], x[0]))
        top_candidate = candidates[0][0]
        candidate_terms = profiles[top_candidate]
        recovered = len(hidden & candidate_terms)

        rows.append({
            "disease_id": disease_id,
            "method": method,
            "top_training_candidate": top_candidate,
            "hidden_phenotype_count": len(hidden),
            "recovered_hidden_phenotypes": recovered,
            "hidden_phenotype_recall": recovered / len(hidden),
        })

results = pd.DataFrame(rows)
results.to_csv(DATA / "ranking_method_comparison.csv", index=False)

summary = (
    results.groupby("method")
    .agg(
        profiles=("disease_id", "nunique"),
        mean_hidden_recall=("hidden_phenotype_recall", "mean"),
        median_hidden_recall=("hidden_phenotype_recall", "median"),
    )
    .reset_index()
)

print(summary.to_string(index=False))
print("\nSaved:", DATA / "ranking_method_comparison.csv")

     method  profiles  mean_hidden_recall  median_hidden_recall
 ic_overlap      1828            0.341602              0.333333
    jaccard      1828            0.173206              0.100000
weighted_f1      1828            0.209105              0.142857

Saved: /content/drive/MyDrive/Rare Disease ai/processed/ranking_method_comparison.csv


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

results = pd.read_csv(DATA / "ranking_method_comparison.csv")
annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
)
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)

train_ids = set(
    split.loc[split["split"] == "train", "disease_id"].str.strip()
)

annotations["database_id"] = annotations["database_id"].str.strip()
annotations["hpo_id"] = annotations["hpo_id"].str.strip()

train = annotations[
    annotations["database_id"].isin(train_ids)
].dropna(subset=["hpo_id"])

profiles = train.groupby("database_id")["hpo_id"].apply(set).to_dict()

term_counts = {}
for terms in profiles.values():
    for term in terms:
        term_counts[term] = term_counts.get(term, 0) + 1

n_train = len(profiles)
ic = {
    term: -np.log(count / n_train)
    for term, count in term_counts.items()
}

inputs = pd.read_csv(DATA / "heldout_input_phenotypes.csv", dtype=str)
inputs["disease_id"] = inputs["disease_id"].str.strip()
inputs["hpo_id"] = inputs["hpo_id"].str.strip()
input_map = inputs.groupby("disease_id")["hpo_id"].apply(set).to_dict()

tie_count = 0
zero_score_count = 0
top_scores = []

for disease_id, observed in input_map.items():
    observed = observed & set(ic)
    if not observed:
        continue

    denominator = sum(ic[t] for t in observed)
    scores = []
    for disease, terms in profiles.items():
        score = (
            sum(ic[t] for t in observed & terms) / denominator
            if denominator else 0.0
        )
        scores.append(score)

    scores.sort(reverse=True)
    top_scores.append(scores[0])

    if scores[0] == 0:
        zero_score_count += 1
    if sum(np.isclose(s, scores[0]) for s in scores) > 1:
        tie_count += 1

print("Training candidate profiles:", len(profiles))
print("Queries with usable phenotypes:", len(top_scores))
print("Queries whose best IC-overlap score is zero:", zero_score_count)
print("Queries with tied top scores:", tie_count)
print("Mean best IC-overlap score:",
      round(float(np.mean(top_scores)), 4) if top_scores else "N/A")
print("\nSaved method comparison:")
print(
    results.groupby("method")["hidden_phenotype_recall"]
    .agg(["mean", "median", "count"])
    .round(4)
)

Training candidate profiles: 7340
Queries with usable phenotypes: 1828
Queries whose best IC-overlap score is zero: 0
Queries with tied top scores: 410
Mean best IC-overlap score: 0.6277

Saved method comparison:
               mean  median  count
method                            
ic_overlap   0.3416  0.3333   1828
jaccard      0.1732  0.1000   1828
weighted_f1  0.2091  0.1429   1828


In [ ]:
from pathlib import Path
import importlib.util

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("HPO ontology:", (PROJECT / "hp.obo").exists())
print("Existing hierarchy results:",
      (DATA / "hierarchy_aware_retrieval_metrics.csv").exists())

for package in ["obonet", "networkx"]:
    print(f"{package} installed:",
          importlib.util.find_spec(package) is not None)

HPO ontology: True
Existing hierarchy results: True
obonet installed: False
networkx installed: True


In [ ]:
%pip install obonet

from pathlib import Path
import pandas as pd
import obonet
import networkx as nx

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

# Load HPO ontology
with open(PROJECT / "hp.obo", "r", encoding="utf-8") as f:
    ontology = obonet.read_obo(f)

print("HPO ontology terms:", len(ontology))
print("Ontology edges:", ontology.number_of_edges())

# Inspect existing hierarchy-aware results
path = DATA / "hierarchy_aware_retrieval_metrics.csv"
df = pd.read_csv(path)

print("\nExisting hierarchy evaluation:", df.shape)
print(df.head().to_string(index=False))

metric_cols = [
    c for c in ["recall_at_1", "recall_at_5", "recall_at_10",
                "reciprocal_rank"] if c in df.columns
]
print("\nExisting metric averages:")
print(df[metric_cols].mean().round(4))

HPO ontology terms: 19894
Ontology edges: 24436

Existing hierarchy evaluation: (1829, 6)
 disease_id  rank  recall_at_1  recall_at_5  recall_at_10  reciprocal_rank
OMIM:100100    49            0            0             0         0.020408
OMIM:101900    65            0            0             0         0.015385
OMIM:102530    75            0            0             0         0.013333
OMIM:102900    79            0            0             0         0.012658
OMIM:103500    86            0            0             0         0.011628

Existing metric averages:
recall_at_1        0.0000
recall_at_5        0.0000
recall_at_10       0.0000
reciprocal_rank    0.0004
dtype: float64


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

hier = pd.read_csv(DATA / "hierarchy_aware_retrieval_metrics.csv")
coverage = pd.read_csv(DATA / "candidate_coverage_audit.csv")

print("HIERARCHY RANK DISTRIBUTION")
print(hier["rank"].describe(percentiles=[.25, .5, .75, .9, .99]))
print("Rank <= 10:", int((hier["rank"] <= 10).sum()))
print("Rank <= 100:", int((hier["rank"] <= 100).sum()))
print("Missing ranks:", int(hier["rank"].isna().sum()))

print("\nCANDIDATE COVERAGE")
print("Profiles:", len(coverage))
print("Zero candidates:", int((coverage["candidate_count"] == 0).sum()))
print("Candidate count summary:")
print(coverage["candidate_count"].describe(
    percentiles=[.25, .5, .75, .9]
))

print("\nSAMPLE LOW-COVERAGE PROFILES")
print(
    coverage.sort_values("candidate_count")
    .head(10).to_string(index=False)
)

HIERARCHY RANK DISTRIBUTION
count     1829.000000
mean      6405.753417
std       3472.250644
min         49.000000
25%       3658.000000
50%       6388.000000
75%       8936.000000
90%      11495.600000
99%      12741.600000
max      12876.000000
Name: rank, dtype: float64
Rank <= 10: 0
Rank <= 100: 7
Missing ranks: 0

CANDIDATE COVERAGE
Profiles: 1832
Zero candidates: 4
Candidate count summary:
count    1832.000000
mean     5906.044760
std      2218.459707
min         0.000000
25%      4692.750000
50%      6528.500000
75%      7556.500000
90%      8258.400000
max      9305.000000
Name: candidate_count, dtype: float64

SAMPLE LOW-COVERAGE PROFILES
 disease_id  input_count  covered_input_count  candidate_count
OMIM:619319            1                    0                0
OMIM:303900            1                    0                0
OMIM:102900            1                    0                0
OMIM:300813            1                    0                0
OMIM:136800            3    

In [ ]:
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

for p in sorted(DATA.glob("*")):
    if any(word in p.name.lower() for word in [
        "hierarchy", "ontology", "ranking", "baseline", "evaluation"
    ]):
        print(p.name)

baseline_coverage_comparison_audit.csv
baseline_model_comparison.csv
direct_vs_hierarchy_scoring_comparison.csv
disease_ranking_retrieval_metrics.csv
evaluation_eligibility_audit.csv
evaluation_protocol_checks.csv
gnn_evaluation_results.csv
hierarchy_aware_retrieval_metrics.csv
ic_weighted_heldout_ranking_results.csv
jaccard_baseline_top10.csv
leakage_reduced_retrieval_evaluation.csv
missing_evaluation_profiles.csv
ontology_fallback_candidates.csv
ontology_fallback_coverage_summary.csv
ontology_fallback_evaluation.csv
ontology_fallback_ranked.csv
ontology_fallback_target_comparison.csv
phenotype_baseline_per_profile.csv
phenotype_disease_ranking_evaluation.csv
phenotype_overlap_baseline_top10.csv
phenotype_recovery_baseline_metrics.csv
ranking_method_comparison.csv
zero_candidate_ontology_ancestors.csv
zero_candidate_ontology_check.csv
zero_candidate_ontology_parents.csv


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "direct_vs_hierarchy_scoring_comparison.csv",
    "ontology_fallback_coverage_summary.csv",
    "ontology_fallback_evaluation.csv",
    "phenotype_recovery_baseline_metrics.csv",
    "ranking_method_comparison.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 65}\n{name}")

    if not path.exists():
        print("Missing")
        continue

    df = pd.read_csv(path, low_memory=False)
    print("Shape:", df.shape)
    print(df.to_string(index=False, max_rows=12))


direct_vs_hierarchy_scoring_comparison.csv
Shape: (1829, 11)
 disease_id  direct_rank  direct_reciprocal_rank  direct_recall_at_1  direct_recall_at_5  direct_recall_at_10  hierarchy_rank  hierarchy_reciprocal_rank  hierarchy_recall_at_1  hierarchy_recall_at_5  hierarchy_recall_at_10
OMIM:100100            1                     1.0                   1                   1                    1               1                        1.0                      1                      1                       1
OMIM:101900            1                     1.0                   1                   1                    1               1                        1.0                      1                      1                       1
OMIM:102530            1                     1.0                   1                   1                    1               1                        1.0                      1                      1                       1
OMIM:102900            1                     1

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

direct = pd.read_csv(DATA / "direct_vs_hierarchy_scoring_comparison.csv")
methods = pd.read_csv(DATA / "ranking_method_comparison.csv")

summary = []

for label, df, prefix in [
    ("Direct scoring", direct, "direct"),
    ("Hierarchy scoring", direct, "hierarchy"),
]:
    summary.append({
        "Method": label,
        "Profiles": len(df),
        "Recall@1": df[f"{prefix}_recall_at_1"].mean(),
        "Recall@5": df[f"{prefix}_recall_at_5"].mean(),
        "Recall@10": df[f"{prefix}_recall_at_10"].mean(),
        "MRR": df[f"{prefix}_reciprocal_rank"].mean(),
    })

for method, group in methods.groupby("method"):
    summary.append({
        "Method": f"{method} (phenotype recovery)",
        "Profiles": group["disease_id"].nunique(),
        "Mean hidden-phenotype recall":
            group["hidden_phenotype_recall"].mean(),
        "Median hidden-phenotype recall":
            group["hidden_phenotype_recall"].median(),
    })

summary_df = pd.DataFrame(summary)
out = DATA / "ranking_method_consolidated_summary.csv"
summary_df.to_csv(out, index=False)

print(summary_df.round(4).to_string(index=False))
print("\nSaved:", out)

                          Method  Profiles  Recall@1  Recall@5  Recall@10    MRR  Mean hidden-phenotype recall  Median hidden-phenotype recall
                  Direct scoring      1829    0.9306    0.9716     0.9787 0.9483                           NaN                             NaN
               Hierarchy scoring      1829    0.9306    0.9716     0.9787 0.9483                           NaN                             NaN
 ic_overlap (phenotype recovery)      1828       NaN       NaN        NaN    NaN                        0.3416                          0.3333
    jaccard (phenotype recovery)      1828       NaN       NaN        NaN    NaN                        0.1732                          0.1000
weighted_f1 (phenotype recovery)      1828       NaN       NaN        NaN    NaN                        0.2091                          0.1429

Saved: /content/drive/MyDrive/Rare Disease ai/processed/ranking_method_consolidated_summary.csv


In [ ]:
from pathlib import Path
from collections import Counter

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
PHENO = PROJECT / "all_phenopackets"

print("Folder exists:", PHENO.exists())

if PHENO.exists():
    files = [p for p in PHENO.rglob("*") if p.is_file()]
    print("Total files:", len(files))

    extensions = Counter(p.suffix.lower() or "[no extension]" for p in files)
    print("\nFile types:")
    for ext, count in extensions.most_common(15):
        print(f"{ext}: {count}")

    print("\nExample paths:")
    for p in files[:25]:
        print(p.relative_to(PHENO))
else:
    print("Check whether the extracted folder is nested elsewhere in the project.")

Folder exists: True
Total files: 10428

File types:
.json: 10427
.tsv: 1

Example paths:
0.1.27/phenopacket_store.summary.tsv
0.1.27/ADAMTS10/PMID_19836009_Family_E_II_6.json
0.1.27/ADAMTS10/PMID_19836009_Family_E_II_3.json
0.1.27/ADAMTS10/PMID_19836009_Family_B_II_4.json
0.1.27/ADAMTS10/PMID_19836009_Family_B_II_2.json
0.1.27/ADAMTS10/PMID_25469541_patient.json
0.1.27/AAGAB/PMID_24573067_CASE_REPORT.json
0.1.27/AAGAB/PMID_28239884_Family_1_proband.json
0.1.27/AAGAB/PMID_28239884_Family_2_proband.json
0.1.27/AAGAB/PMID_28239884_Family_3_proband.json
0.1.27/ADA/PMID_28823388_3_month_old_girl.json
0.1.27/ADA/PMID_1680289_patient.json
0.1.27/ACTB/PMID_30315159_Patient_1.json
0.1.27/ACTB/PMID_30733661_4_year_old_girl.json
0.1.27/ACTB/PMID_30315159_Patient_4_Mother_of_Patient_3.json
0.1.27/ACTB/PMID_30315159_Patient_2_Father_of_Patient_1.json
0.1.27/ACTB/PMID_30315159_Patient_6.json
0.1.27/ACTB/PMID_30315159_Patient_N.json
0.1.27/ACTB/PMID_30315159_Patient_5.json
0.1.27/ACTB/PMID_30315159_P

In [ ]:
from pathlib import Path
import json
from collections import Counter

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
PHENO = PROJECT / "all_phenopackets"

json_files = list(PHENO.rglob("*.json"))
print("JSON files found:", len(json_files))

# Inspect up to 3 cases without printing potentially identifying case details
for path in json_files[:3]:
    print("\nFile:", path.name)

    with open(path, "r", encoding="utf-8") as f:
        case = json.load(f)

    print("Top-level keys:", list(case.keys()))

    subject = case.get("subject", {})
    print("Subject keys:", list(subject.keys()))

    phenotypes = case.get("phenotypicFeatures", [])
    print("Phenotypic features:", len(phenotypes))
    if phenotypes:
        print("First phenotype keys:", list(phenotypes[0].keys()))
        print("First phenotype type:",
              phenotypes[0].get("type", {}).get("id", "not found"))

    diseases = case.get("diseases", [])
    print("Recorded diagnoses:", len(diseases))
    if diseases:
        print("First diagnosis keys:", list(diseases[0].keys()))
        print("First diagnosis term:",
              diseases[0].get("term", {}).get("id", "not found"))

JSON files found: 10427

File: PMID_19836009_Family_E_II_6.json
Top-level keys: ['id', 'subject', 'phenotypicFeatures', 'interpretations', 'diseases', 'metaData']
Subject keys: ['id', 'timeAtLastEncounter', 'sex']
Phenotypic features: 9
First phenotype keys: ['type', 'excluded']
First phenotype type: HP:0007833
Recorded diagnoses: 1
First diagnosis keys: ['term']
First diagnosis term: OMIM:277600

File: PMID_19836009_Family_E_II_3.json
Top-level keys: ['id', 'subject', 'phenotypicFeatures', 'interpretations', 'diseases', 'metaData']
Subject keys: ['id', 'timeAtLastEncounter', 'sex']
Phenotypic features: 9
First phenotype keys: ['type', 'excluded']
First phenotype type: HP:0007833
Recorded diagnoses: 1
First diagnosis keys: ['term']
First diagnosis term: OMIM:277600

File: PMID_19836009_Family_B_II_4.json
Top-level keys: ['id', 'subject', 'phenotypicFeatures', 'interpretations', 'diseases', 'metaData']
Subject keys: ['id', 'timeAtLastEncounter', 'sex']
Phenotypic features: 9
First pheno

In [ ]:
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
out = DATA / "phenopacket_patient_cases.csv"

print("Output file exists:", out.exists())
if out.exists():
    print("Output size (MB):", round(out.stat().st_size / (1024 * 1024), 2))

Output file exists: False


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
PHENO = PROJECT / "all_phenopackets"
DATA = PROJECT / "processed"
DATA.mkdir(parents=True, exist_ok=True)

checkpoint = DATA / "phenopacket_parse_checkpoint.csv"
output = DATA / "phenopacket_patient_cases.csv"

# Resume from previously saved progress, if available
rows = []
done = set()

if checkpoint.exists():
    old = pd.read_csv(checkpoint, dtype=str).fillna("")
    rows = old.to_dict("records")
    done = set(old["source_file"])
    print("Resuming:", len(done), "cases already processed")

files = [p for p in PHENO.rglob("*.json")
         if str(p.relative_to(PHENO)) not in done]
print("Cases remaining:", len(files))

errors = 0
for i, path in enumerate(files, 1):
    try:
        with open(path, "r", encoding="utf-8") as f:
            case = json.load(f)

        observed, excluded = set(), set()
        for feature in case.get("phenotypicFeatures", []):
            term = feature.get("type", {}).get("id")
            if not term or not term.startswith("HP:"):
                continue
            (excluded if feature.get("excluded") is True else observed).add(term)

        observed -= excluded
        diagnoses = {
            d.get("term", {}).get("id")
            for d in case.get("diseases", [])
            if d.get("term", {}).get("id")
        }

        rows.append({
            "case_id": case.get("id") or path.stem,
            "source_file": str(path.relative_to(PHENO)),
            "hpo_terms": ";".join(sorted(observed)),
            "phenotype_count": len(observed),
            "diagnosis_ids": ";".join(sorted(diagnoses)),
            "diagnosis_count": len(diagnoses),
        })
    except Exception:
        errors += 1

    if i % 500 == 0:
        pd.DataFrame(rows).to_csv(checkpoint, index=False)
        print(f"Processed {i}/{len(files)} remaining files")

pd.DataFrame(rows).to_csv(output, index=False)
if checkpoint.exists():
    checkpoint.unlink()

print("\nCompleted.")
print("Cases parsed:", len(rows))
print("Errors in this run:", errors)
print("Saved:", output)

Resuming: 7000 cases already processed
Cases remaining: 3427
Processed 500/3427 remaining files
Processed 1000/3427 remaining files
Processed 1500/3427 remaining files
Processed 2000/3427 remaining files
Processed 2500/3427 remaining files
Processed 3000/3427 remaining files

Completed.
Cases parsed: 10427
Errors in this run: 0
Saved: /content/drive/MyDrive/Rare Disease ai/processed/phenopacket_patient_cases.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

print("Project exists:", PROJECT.exists())
print("Processed folder exists:", DATA.exists())
print("Model folder exists:", MODEL.exists())

print("\nProcessed files:")
for f in sorted(DATA.iterdir()):
    print(f.name)

Project exists: True
Processed folder exists: True
Model folder exists: True

Processed files:
baseline_coverage_comparison_audit.csv
baseline_model_comparison.csv
candidate_coverage_audit.csv
clean_hpo_annotations.csv
direct_vs_hierarchy_scoring_comparison.csv
disease_level_split.csv
disease_ranking_retrieval_metrics.csv
evaluation_eligibility_audit.csv
evaluation_protocol_checks.csv
final_artifact_inventory.csv
final_consolidated_results.csv
final_project_handover_inventory.csv
gene_informed_phenotype_recovery_metrics.csv
gene_informed_phenotype_recovery_top10.csv
gnn_checkpoint_compatibility.csv
gnn_evaluation_results.csv
gnn_phenotype_recovery_metrics.csv
gnn_training_annotations.csv
gnn_zero_shot_hidden_targets.csv
gnn_zero_shot_input_profiles.csv
heldout_gene_coverage.csv
heldout_input_phenotypes.csv
heldout_profile_nearest_training_disease.csv
heldout_target_phenotypes.csv
hierarchy_aware_retrieval_metrics.csv
ic_weighted_heldout_ranking_results.csv
jaccard_baseline_top10.csv
le

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

required_files = [
    DATA / "clean_hpo_annotations.csv",
    DATA / "phenopacket_patient_cases.csv",
    DATA / "disease_level_split.csv",
    DATA / "gnn_training_annotations.csv",
    DATA / "rare_disease_gnn_training_only.pt",
    DATA / "rare_disease_final_research_report.md",
]

print("PREPROCESSING AND ASSET CHECK")
print("-" * 45)

for file in required_files:
    status = "OK" if file.exists() else "MISSING"
    size = f"{file.stat().st_size / (1024 * 1024):.2f} MB" if file.exists() else "-"
    print(f"{status:8} | {size:>10} | {file.name}")

if all(file.exists() for file in required_files):
    print("\nAll required files are present.")
else:
    print("\nSome files are missing; we will investigate before proceeding.")

PREPROCESSING AND ASSET CHECK
---------------------------------------------
OK       |   34.24 MB | clean_hpo_annotations.csv
OK       |    1.87 MB | phenopacket_patient_cases.csv
OK       |    0.16 MB | disease_level_split.csv
OK       |   21.95 MB | gnn_training_annotations.csv
OK       |    4.73 MB | rare_disease_gnn_training_only.pt
OK       |    0.01 MB | rare_disease_final_research_report.md

All required files are present.


In [ ]:
import pandas as pd

cases_file = DATA / "phenopacket_patient_cases.csv"
annotations_file = DATA / "clean_hpo_annotations.csv"
split_file = DATA / "disease_level_split.csv"
training_file = DATA / "gnn_training_annotations.csv"

cases = pd.read_csv(cases_file, low_memory=False)
annotations = pd.read_csv(annotations_file, low_memory=False)
split = pd.read_csv(split_file, low_memory=False)
training = pd.read_csv(training_file, low_memory=False)

print("DATASET READINESS REPORT")
print("=" * 45)

print("\n1. PHENOPACKET PATIENT CASES")
print("Total cases:", len(cases))
print("Cases with phenotypes:",
      int((cases["phenotype_count"] > 0).sum()))
print("Cases with diagnoses:",
      int((cases["diagnosis_count"] > 0).sum()))
print("Cases with both:",
      int(((cases["phenotype_count"] > 0) &
           (cases["diagnosis_count"] > 0)).sum()))

print("\n2. DISEASE–PHENOTYPE KNOWLEDGE BASE")
print("Total association rows:", len(annotations))
print("Unique disease IDs:", annotations["database_id"].nunique())

print("\n3. DISEASE-LEVEL SPLIT")
print("Split columns:", split.columns.tolist())
print("Split rows:", len(split))

print("\n4. TRAINING ANNOTATIONS")
print("Training association rows:", len(training))
print("Training columns:", training.columns.tolist())

DATASET READINESS REPORT

1. PHENOPACKET PATIENT CASES
Total cases: 10427
Cases with phenotypes: 10424
Cases with diagnoses: 10427
Cases with both: 10424

2. DISEASE–PHENOTYPE KNOWLEDGE BASE
Total association rows: 286047
Unique disease IDs: 12879

3. DISEASE-LEVEL SPLIT
Split columns: ['disease_id', 'split']
Split rows: 9176

4. TRAINING ANNOTATIONS
Training association rows: 179653
Training columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']


In [ ]:
def split_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [
        x.strip()
        for x in str(value).split(";")
        if x.strip()
    ]

# Disease IDs present in the knowledge base
known_ids = set(
    annotations["database_id"].dropna().astype(str).str.strip()
)

# Extract the diagnosis IDs associated with each patient case
cases["diagnosis_list"] = cases["diagnosis_ids"].apply(split_ids)

# Check for exact diagnosis-ID matches
cases["has_matching_diagnosis"] = cases["diagnosis_list"].apply(
    lambda ids: any(d in known_ids for d in ids)
)

# Initial eligible cases: phenotype data, diagnosis, and a matching ID
eligible = cases[
    (cases["phenotype_count"] > 0)
    & (cases["diagnosis_count"] > 0)
    & cases["has_matching_diagnosis"]
].copy()

print("PATIENT DIAGNOSIS COVERAGE")
print("=" * 40)
print("Total patient cases:", len(cases))
print("Cases with a matching diagnosis ID:",
      int(cases["has_matching_diagnosis"].sum()))
print("Cases eligible for initial evaluation:", len(eligible))

matched_ids = {
    disease_id
    for ids in eligible["diagnosis_list"]
    for disease_id in ids
    if disease_id in known_ids
}

print("Unique matched diagnosis IDs:", len(matched_ids))

print("\nExamples of diagnosis IDs:")
print(cases.loc[cases["has_matching_diagnosis"],
                ["case_id", "diagnosis_ids"]].head(5).to_string(index=False))

PATIENT DIAGNOSIS COVERAGE
Total patient cases: 10427
Cases with a matching diagnosis ID: 10398
Cases eligible for initial evaluation: 10395
Unique matched diagnosis IDs: 778

Examples of diagnosis IDs:
                    case_id diagnosis_ids
PMID_19836009_Family_E_II_6   OMIM:277600
PMID_19836009_Family_E_II_3   OMIM:277600
PMID_19836009_Family_B_II_4   OMIM:277600
PMID_19836009_Family_B_II_2   OMIM:277600
      PMID_25469541_patient   OMIM:277600


In [ ]:
print("CASE STRUCTURE AUDIT")
print("=" * 45)

print("\nAvailable columns:")
print(cases.columns.tolist())

print("\nExample case IDs:")
print(cases["case_id"].head(10).to_list())

print("\nRepeated case IDs:")
print("Duplicate case IDs:", int(cases["case_id"].duplicated().sum()))

print("\nDiagnosis frequency:")
print(
    cases.loc[cases["has_matching_diagnosis"], "diagnosis_list"]
    .explode()
    .value_counts()
    .head(10)
    .to_string()
)

print("\nSource-file examples:")
print(cases["source_file"].head(10).to_string(index=False))

CASE STRUCTURE AUDIT

Available columns:
['case_id', 'source_file', 'hpo_terms', 'phenotype_count', 'diagnosis_ids', 'diagnosis_count', 'diagnosis_list', 'has_matching_diagnosis']

Example case IDs:
['PMID_19836009_Family_E_II_6', 'PMID_19836009_Family_E_II_3', 'PMID_19836009_Family_B_II_4', 'PMID_19836009_Family_B_II_2', 'PMID_25469541_patient', 'PMID_24573067_CASE_REPORT', 'PMID_28239884_Family_1_proband', 'PMID_28239884_Family_2_proband', 'PMID_28239884_Family_3_proband', 'PMID_28823388_3_month_old_girl']

Repeated case IDs:
Duplicate case IDs: 50

Diagnosis frequency:
diagnosis_list
OMIM:612164    477
OMIM:162200    421
OMIM:148050    334
OMIM:162000    199
OMIM:612313    157
OMIM:142900    150
OMIM:151660    127
OMIM:610253    125
OMIM:616364    117
OMIM:117360    104

Source-file examples:
0.1.27/ADAMTS10/PMID_19836009_Family_E_II_6.json
0.1.27/ADAMTS10/PMID_19836009_Family_E_II_3.json
0.1.27/ADAMTS10/PMID_19836009_Family_B_II_4.json
0.1.27/ADAMTS10/PMID_19836009_Family_B_II_2.js

In [ ]:
# Identify duplicate records
duplicate_cases = cases[cases["case_id"].duplicated(keep=False)].copy()

# Extract publication ID from case ID, e.g. PMID_19836009_...
cases["publication_id"] = cases["case_id"].str.extract(
    r"(PMID_\d+)", expand=False
)

print("DUPLICATE AND GROUPING AUDIT")
print("=" * 45)

print("Total records:", len(cases))
print("Duplicate case-ID rows:",
      int(cases["case_id"].duplicated().sum()))
print("Rows involved in duplicate IDs:",
      len(duplicate_cases))
print("Unique publication IDs:",
      cases["publication_id"].nunique())
print("Rows without a publication ID:",
      int(cases["publication_id"].isna().sum()))

print("\nPublications contributing the most cases:")
print(
    cases["publication_id"]
    .value_counts()
    .head(10)
    .to_string()
)

print("\nDuplicate case-ID examples:")
print(
    duplicate_cases[
        ["case_id", "source_file", "diagnosis_ids"]
    ].head(10).to_string(index=False)
)

DUPLICATE AND GROUPING AUDIT
Total records: 10427
Duplicate case-ID rows: 50
Rows involved in duplicate IDs: 100
Unique publication IDs: 1733
Rows without a publication ID: 0

Publications contributing the most cases:
publication_id
PMID_35190816    477
PMID_36446582    334
PMID_29290338    157
PMID_31021519    157
PMID_26178382    136
PMID_39013458    125
PMID_37843397    115
PMID_25802881    103
PMID_33674768     80
PMID_37761890     76

Duplicate case-ID examples:
                                    case_id                                                         source_file diagnosis_ids
PMID_36446582_Kutkowska_Kazmierczak2021_P23 0.1.27/ANKRD11/PMID_36446582_Kutkowska_Kazmierczak2021_P23 (1).json   OMIM:148050
PMID_36446582_Kutkowska_Kazmierczak2021_P23     0.1.27/ANKRD11/PMID_36446582_Kutkowska_Kazmierczak2021_P23.json   OMIM:148050
 PMID_20513137_individual_GUI_GSM_GSM492741      0.1.27/NF1/PMID_20513137_individual_GUI_GSM_GSM492741 (1).json   OMIM:162200
 PMID_20513137_individua

In [ ]:
from collections import Counter

def normalize_terms(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    return {
        term.strip()
        for term in str(value).split(";")
        if term.strip()
    }

duplicate_groups = cases[
    cases["case_id"].duplicated(keep=False)
].groupby("case_id", sort=False)

identical_duplicates = 0
different_duplicates = 0

for case_id, group in duplicate_groups:
    phenotype_sets = group["hpo_terms"].apply(normalize_terms)
    diagnosis_sets = group["diagnosis_list"].apply(
        lambda x: set(x) if isinstance(x, list) else set()
    )

    identical = (
        phenotype_sets.apply(lambda x: frozenset(x)).nunique() == 1
        and diagnosis_sets.apply(lambda x: frozenset(x)).nunique() == 1
    )

    if identical:
        identical_duplicates += 1
    else:
        different_duplicates += 1

print("DUPLICATE CONTENT CHECK")
print("=" * 40)
print("Duplicate case-ID groups:", len(duplicate_groups))
print("Groups with identical phenotype and diagnosis sets:",
      identical_duplicates)
print("Groups with differing contents:", different_duplicates)

DUPLICATE CONTENT CHECK
Duplicate case-ID groups: 50
Groups with identical phenotype and diagnosis sets: 50
Groups with differing contents: 0


In [ ]:
# Preserve the original `cases` DataFrame.
cases_eval = cases.drop_duplicates(
    subset="case_id",
    keep="first"
).copy()

print("CLEAN EVALUATION DATASET")
print("=" * 40)
print("Original records:", len(cases))
print("Records after deduplication:", len(cases_eval))
print("Duplicate records removed:", len(cases) - len(cases_eval))
print("Remaining duplicate case IDs:",
      int(cases_eval["case_id"].duplicated().sum()))

print("\nEligible cases after deduplication:",
      int((
          (cases_eval["phenotype_count"] > 0)
          & (cases_eval["diagnosis_count"] > 0)
          & cases_eval["has_matching_diagnosis"]
      ).sum()))

CLEAN EVALUATION DATASET
Original records: 10427
Records after deduplication: 10377
Duplicate records removed: 50
Remaining duplicate case IDs: 0

Eligible cases after deduplication: 10345


In [ ]:
import pandas as pd

files_to_check = {
    "protocol_checks": DATA / "evaluation_protocol_checks.csv",
    "eligibility_audit": DATA / "evaluation_eligibility_audit.csv",
    "final_results": DATA / "final_consolidated_results.csv",
    "training_annotations": DATA / "gnn_training_annotations.csv",
    "disease_split": DATA / "disease_level_split.csv",
}

for name, path in files_to_check.items():
    print(f"\n{'=' * 15} {name.upper()} {'=' * 15}")

    if path.exists():
        df = pd.read_csv(path, low_memory=False)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(5).to_string(index=False))
    else:
        print("FILE MISSING:", path.name)


=============== PROTOCOL_CHECKS ===============
Shape: (6, 2)
Columns: ['Check', 'Value']
                              Check  Value
               Input phenotype rows  21647
       Hidden target phenotype rows  22562
          Input/target pair overlap      0
             Training diseases used   7340
Held-out diseases in training index      0

=============== ELIGIBILITY_AUDIT ===============
Shape: (1836, 5)
Columns: ['disease_id', 'input_phenotype_count', 'target_phenotype_count', 'in_metrics', 'status']
 disease_id  input_phenotype_count  target_phenotype_count  in_metrics    status
OMIM:100100                      8                       8        True evaluated
OMIM:101900                      4                       5        True evaluated
OMIM:102530                      2                       2        True evaluated
OMIM:102900                      1                       2        True evaluated
OMIM:103500                      4                       5        True evaluate

In [ ]:
 # Build disease-to-phenotype sets from the saved knowledge base.
disease_col = "database_id"
phenotype_col = "hpo_id"

kb = annotations[[disease_col, phenotype_col]].dropna().copy()

kb[disease_col] = kb[disease_col].astype(str).str.strip()
kb[phenotype_col] = kb[phenotype_col].astype(str).str.strip()

disease_phenotypes = (
    kb.groupby(disease_col)[phenotype_col]
    .apply(set)
    .to_dict()
)

# Convert patient phenotype strings to sets.
cases_eval["phenotype_set"] = cases_eval["hpo_terms"].apply(
    normalize_terms
)

# Count exact phenotype-set matches against each recorded diagnosis.
def patient_diagnosis_overlap(row):
    patient_terms = row["phenotype_set"]
    matched_diagnoses = [
        d for d in row["diagnosis_list"]
        if d in disease_phenotypes
    ]

    if not patient_terms or not matched_diagnoses:
        return None

    return {
        d: len(patient_terms & disease_phenotypes[d])
        / len(patient_terms)
        for d in matched_diagnoses
    }

overlaps = cases_eval.apply(patient_diagnosis_overlap, axis=1)

print("PATIENT–KNOWLEDGE-BASE OVERLAP CHECK")
print("=" * 45)
print("Unique disease profiles in knowledge base:",
      len(disease_phenotypes))
print("Unique patient cases:", len(cases_eval))
print("Cases with phenotype sets:",
      int(cases_eval["phenotype_set"].map(bool).sum()))
print("Cases with diagnosis/knowledge-base comparison:",
      int(overlaps.notna().sum()))

PATIENT–KNOWLEDGE-BASE OVERLAP CHECK
Unique disease profiles in knowledge base: 12879
Unique patient cases: 10377
Cases with phenotype sets: 10374
Cases with diagnosis/knowledge-base comparison: 10345


In [ ]:
import re

def extract_pmid(value):
    if pd.isna(value):
        return None
    match = re.search(r"PMID[_:]?(\d+)", str(value), re.IGNORECASE)
    return match.group(1) if match else None

# Publication IDs from patient source files
cases_eval["pmid"] = cases_eval["source_file"].apply(extract_pmid)

# Publication IDs from disease–phenotype annotation references
annotations["pmid"] = annotations["reference"].apply(extract_pmid)

patient_pmids = set(cases_eval["pmid"].dropna())
annotation_pmids = set(annotations["pmid"].dropna())

shared_pmids = patient_pmids & annotation_pmids

cases_eval["publication_in_knowledge_base"] = (
    cases_eval["pmid"].isin(annotation_pmids)
)

print("PUBLICATION OVERLAP AUDIT")
print("=" * 45)
print("Unique patient publications:", len(patient_pmids))
print("Unique annotation publications:", len(annotation_pmids))
print("Publications appearing in both:", len(shared_pmids))
print("Patient cases from shared publications:",
      int(cases_eval["publication_in_knowledge_base"].sum()))
print("Patient cases without a matched PMID:",
      int(cases_eval["pmid"].isna().sum()))

print("\nExamples of shared publications:")
print(sorted(shared_pmids)[:15])

PUBLICATION OVERLAP AUDIT
Unique patient publications: 1733
Unique annotation publications: 9314
Publications appearing in both: 1240
Patient cases from shared publications: 7454
Patient cases without a matched PMID: 0

Examples of shared publications:
['10077612', '10198255', '10487826', '10498624', '10534505', '10560675', '10580070', '10805329', '10842287', '10851256', '10939567', '11050011', '11093277', '11118249', '11179005']


In [ ]:
audit_path = DATA / "patient_publication_overlap_audit.csv"

publication_audit = pd.DataFrame([{
    "unique_patient_publications": len(patient_pmids),
    "unique_annotation_publications": len(annotation_pmids),
    "shared_publications": len(shared_pmids),
    "patient_cases": len(cases_eval),
    "cases_from_shared_publications": int(
        cases_eval["publication_in_knowledge_base"].sum()
    ),
    "cases_without_matched_pmid": int(
        cases_eval["pmid"].isna().sum()
    ),
    "interpretation": (
        "Potential information overlap; shared PMIDs alone "
        "do not establish leakage."
    )
}])

publication_audit.to_csv(audit_path, index=False)

print(publication_audit.to_string(index=False))
print("\nSaved to:", audit_path)

 unique_patient_publications  unique_annotation_publications  shared_publications  patient_cases  cases_from_shared_publications  cases_without_matched_pmid                                                              interpretation
                        1733                            9314                 1240          10377                            7454                           0 Potential information overlap; shared PMIDs alone do not establish leakage.

Saved to: /content/drive/MyDrive/Rare Disease ai/processed/patient_publication_overlap_audit.csv


In [ ]:
import numpy as np
import pandas as pd

# Use only unique cases with phenotypes and a matching diagnosis.
eligible_cases = cases_eval[
    (cases_eval["phenotype_count"] > 0)
    & (cases_eval["diagnosis_count"] > 0)
    & cases_eval["has_matching_diagnosis"]
].copy()

# Every eligible case must have a publication ID.
assert eligible_cases["pmid"].notna().all(), (
    "Some eligible cases have no publication ID."
)

# Split publications, not individual patients.
unique_pmids = np.array(sorted(eligible_cases["pmid"].unique()))
rng = np.random.default_rng(42)
rng.shuffle(unique_pmids)

n_test = max(1, round(0.20 * len(unique_pmids)))
test_pmids = set(unique_pmids[:n_test])

eligible_cases["split"] = np.where(
    eligible_cases["pmid"].isin(test_pmids),
    "test",
    "development"
)

# Save the split manifest; preserve all original files.
split_path = DATA / "patient_publication_split.csv"

eligible_cases[
    ["case_id", "pmid", "split", "diagnosis_ids"]
].to_csv(split_path, index=False)

# Verify that no publication appears in both groups.
dev_pmids = set(
    eligible_cases.loc[
        eligible_cases["split"] == "development", "pmid"
    ]
)
test_pmids_actual = set(
    eligible_cases.loc[
        eligible_cases["split"] == "test", "pmid"
    ]
)

print("PUBLICATION-GROUPED SPLIT")
print("=" * 42)
print("Eligible cases:", len(eligible_cases))
print("Development cases:",
      int((eligible_cases["split"] == "development").sum()))
print("Test cases:",
      int((eligible_cases["split"] == "test").sum()))
print("Development publications:", len(dev_pmids))
print("Test publications:", len(test_pmids_actual))
print("Publication overlap:", len(dev_pmids & test_pmids_actual))
print("Saved to:", split_path)

PUBLICATION-GROUPED SPLIT
Eligible cases: 10345
Development cases: 8386
Test cases: 1959
Development publications: 1383
Test publications: 346
Publication overlap: 0
Saved to: /content/drive/MyDrive/Rare Disease ai/processed/patient_publication_split.csv


In [ ]:
# Use the publication-grouped split saved in Step 13.
patient_split = pd.read_csv(
    DATA / "patient_publication_split.csv",
    dtype=str
)

test_pmids = set(
    patient_split.loc[
        patient_split["split"] == "test", "pmid"
    ].dropna()
)

# Extract publication IDs from annotation references.
annotation_pmids = annotations["reference"].apply(extract_pmid)

annotations_audit = annotations.copy()
annotations_audit["pmid"] = annotation_pmids

test_reference_mask = annotations_audit["pmid"].isin(test_pmids)

print("TEST-PUBLICATION ANNOTATION AUDIT")
print("=" * 45)
print("Held-out test publications:", len(test_pmids))
print("Total annotation rows:", len(annotations_audit))
print("Annotation rows citing test publications:",
      int(test_reference_mask.sum()))
print("Unique test PMIDs found in annotations:",
      annotations_audit.loc[test_reference_mask, "pmid"].nunique())
print("Annotation rows without matched PMID:",
      int(annotations_audit["pmid"].isna().sum()))

print("\nExample overlapping annotation rows:")
print(
    annotations_audit.loc[
        test_reference_mask,
        ["database_id", "hpo_id", "reference"]
    ].head(10).to_string(index=False)
)

TEST-PUBLICATION ANNOTATION AUDIT
Held-out test publications: 346
Total annotation rows: 286047
Annotation rows citing test publications: 5314
Unique test PMIDs found in annotations: 248
Annotation rows without matched PMID: 167539

Example overlapping annotation rows:
database_id     hpo_id     reference
OMIM:620502 HP:0000750 PMID:36103875
OMIM:620502 HP:0002650 PMID:36103875
OMIM:620502 HP:0000343 PMID:36103875
OMIM:620502 HP:0000431 PMID:36103875
OMIM:620502 HP:0001250 PMID:36103875
OMIM:620502 HP:0001252 PMID:36103875
OMIM:620502 HP:0002360 PMID:36103875
OMIM:620502 HP:0003593 PMID:36103875
OMIM:620502 HP:0040183 PMID:36103875
OMIM:620502 HP:0012427 PMID:36103875


In [ ]:
# Exclude annotations citing held-out test publications.
safe_annotations = annotations_audit.loc[
    ~test_reference_mask
].copy()

# Save a separate, auditable dataset.
safe_path = DATA / "patient_test_leakage_reduced_annotations.csv"
safe_annotations.drop(columns=["pmid"]).to_csv(
    safe_path,
    index=False
)

# Verify the result.
remaining_test_pmids = (
    set(safe_annotations["pmid"].dropna()) & test_pmids
)

print("LEAKAGE-REDUCED ANNOTATIONS")
print("=" * 45)
print("Original annotation rows:", len(annotations_audit))
print("Excluded rows:", int(test_reference_mask.sum()))
print("Remaining annotation rows:", len(safe_annotations))
print("Remaining rows citing test publications:",
      len(remaining_test_pmids))
print("Saved to:", safe_path)

LEAKAGE-REDUCED ANNOTATIONS
Original annotation rows: 286047
Excluded rows: 5314
Remaining annotation rows: 280733
Remaining rows citing test publications: 0
Saved to: /content/drive/MyDrive/Rare Disease ai/processed/patient_test_leakage_reduced_annotations.csv


In [ ]:
# Load the leakage-reduced annotation file.
safe_annotations = pd.read_csv(
    safe_path,
    dtype=str,
    low_memory=False
)

# Build disease-to-phenotype profiles from the reduced annotations.
safe_profiles = (
    safe_annotations.dropna(subset=["database_id", "hpo_id"])
    .assign(
        database_id=lambda df: df["database_id"].str.strip(),
        hpo_id=lambda df: df["hpo_id"].str.strip()
    )
    .groupby("database_id")["hpo_id"]
    .apply(set)
    .to_dict()
)

# Identify eligible test cases.
test_cases = eligible_cases[
    eligible_cases["split"] == "test"
].copy()

test_cases["diagnosis_list"] = test_cases["diagnosis_ids"].apply(
    split_ids
)

test_cases["has_profile_after_filter"] = test_cases[
    "diagnosis_list"
].apply(
    lambda ids: any(d in safe_profiles for d in ids)
)

print("POST-FILTER DISEASE COVERAGE")
print("=" * 45)
print("Held-out patient cases:", len(test_cases))
print("Cases with at least one diagnosis profile:",
      int(test_cases["has_profile_after_filter"].sum()))
print("Cases without a diagnosis profile:",
      int((~test_cases["has_profile_after_filter"]).sum()))
print("Diseases with profiles after filtering:", len(safe_profiles))

# Save the coverage summary.
coverage_path = DATA / "patient_test_disease_coverage.csv"

pd.DataFrame([{
    "test_cases": len(test_cases),
    "cases_with_diagnosis_profile": int(
        test_cases["has_profile_after_filter"].sum()
    ),
    "cases_without_diagnosis_profile": int(
        (~test_cases["has_profile_after_filter"]).sum()
    ),
    "diseases_with_profiles": len(safe_profiles),
    "annotation_rows_after_filter": len(safe_annotations)
}]).to_csv(coverage_path, index=False)

print("\nCoverage report saved to:", coverage_path)

POST-FILTER DISEASE COVERAGE
Held-out patient cases: 1959
Cases with at least one diagnosis profile: 1551
Cases without a diagnosis profile: 408
Diseases with profiles after filtering: 12827

Coverage report saved to: /content/drive/MyDrive/Rare Disease ai/processed/patient_test_disease_coverage.csv


In [ ]:
print("PHENOTYPE BASELINE PREPARATION")
print("=" * 45)

print("Annotation columns:", safe_annotations.columns.tolist())
print("Patient phenotype examples:")

print(
    cases_eval.loc[
        cases_eval["phenotype_count"] > 0,
        ["case_id", "hpo_terms", "diagnosis_ids"]
    ].head(3).to_string(index=False)
)

# Check whether the HPO ontology file is available.
hpo_file = PROJECT / "hp.obo"

print("\nHPO ontology file exists:", hpo_file.exists())

# Count unique phenotype terms in the reduced annotations.
print(
    "Unique annotated HPO terms:",
    safe_annotations["hpo_id"].dropna().nunique()
)

PHENOTYPE BASELINE PREPARATION
Annotation columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']
Patient phenotype examples:
                    case_id                                                         hpo_terms diagnosis_ids
PMID_19836009_Family_E_II_6 HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003   OMIM:277600
PMID_19836009_Family_E_II_3 HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003   OMIM:277600
PMID_19836009_Family_B_II_4            HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0011003   OMIM:277600

HPO ontology file exists: True
Unique annotated HPO terms: 11611


In [ ]:
import math
import pandas as pd

# Build disease phenotype profiles from the filtered annotations.
baseline_annotations = safe_annotations.dropna(
    subset=["database_id", "hpo_id"]
).copy()

baseline_annotations["database_id"] = (
    baseline_annotations["database_id"].astype(str).str.strip()
)
baseline_annotations["hpo_id"] = (
    baseline_annotations["hpo_id"].astype(str).str.strip()
)

disease_profiles = (
    baseline_annotations.groupby("database_id")["hpo_id"]
    .apply(set)
    .to_dict()
)

# Information content: rarer disease-associated terms receive greater weight.
term_disease_counts = {}

for terms in disease_profiles.values():
    for term in terms:
        term_disease_counts[term] = term_disease_counts.get(term, 0) + 1

n_diseases = len(disease_profiles)

term_ic = {
    term: -math.log(count / n_diseases)
    for term, count in term_disease_counts.items()
}

print("BASELINE PROFILE SUMMARY")
print("=" * 40)
print("Disease profiles:", len(disease_profiles))
print("Unique weighted HPO terms:", len(term_ic))
print("Total disease–phenotype pairs:",
      sum(len(terms) for terms in disease_profiles.values()))
print("Example information-content weights:")
print(sorted(term_ic.items(), key=lambda x: x[1], reverse=True)[:10])

BASELINE PROFILE SUMMARY
Disease profiles: 12827
Unique weighted HPO terms: 11611
Total disease–phenotype pairs: 280733
Example information-content weights:
[('HP:0007590', 9.45930760329985), ('HP:6000946', 9.45930760329985), ('HP:6000920', 9.45930760329985), ('HP:0005733', 9.45930760329985), ('HP:0025096', 9.45930760329985), ('HP:0009590', 9.45930760329985), ('HP:0500089', 9.45930760329985), ('HP:0007935', 9.45930760329985), ('HP:6000934', 9.45930760329985), ('HP:0007343', 9.45930760329985)]


In [ ]:
import numpy as np
import pandas as pd

# Use the existing held-out patient cases.
test_cases = eligible_cases[
    eligible_cases["split"] == "test"
].copy()

# Keep only cases with at least one matching diagnosis profile.
test_cases["diagnosis_list"] = test_cases["diagnosis_ids"].apply(split_ids)

def weighted_overlap(query_terms, disease_terms):
    shared = query_terms & disease_terms
    denominator = sum(term_ic.get(t, 0.0) for t in query_terms)
    if denominator <= 0:
        return 0.0
    return sum(term_ic.get(t, 0.0) for t in shared) / denominator

results = []

for _, row in test_cases.iterrows():
    query_terms = normalize_terms(row["hpo_terms"])
    if not query_terms:
        continue

    # Rank all diseases using patient phenotypes only.
    ranked = sorted(
        (
            (disease_id, weighted_overlap(query_terms, terms))
            for disease_id, terms in disease_profiles.items()
        ),
        key=lambda x: x[1],
        reverse=True
    )

    ranked_ids = [d for d, _ in ranked]
    true_diagnoses = set(row["diagnosis_list"])
    ranks = [
        ranked_ids.index(d) + 1
        for d in true_diagnoses
        if d in ranked_ids
    ]

    results.append({
        "case_id": row["case_id"],
        "pmid": row["pmid"],
        "diagnosis_ids": row["diagnosis_ids"],
        "rank": min(ranks) if ranks else np.nan,
        "top1": int(bool(ranks) and min(ranks) == 1),
        "top5": int(bool(ranks) and min(ranks) <= 5),
        "top10": int(bool(ranks) and min(ranks) <= 10),
        "candidate_covered": int(bool(ranks))
    })

baseline_results = pd.DataFrame(results)

print("PHENOTYPE-ONLY BASELINE")
print("=" * 40)
print("Evaluated cases:", len(baseline_results))
print("Candidate coverage:",
      round(baseline_results["candidate_covered"].mean(), 4))
print("Top-1 accuracy:",
      round(baseline_results["top1"].mean(), 4))
print("Top-5 accuracy:",
      round(baseline_results["top5"].mean(), 4))
print("Top-10 accuracy:",
      round(baseline_results["top10"].mean(), 4))
print("Mean reciprocal rank:",
      round((1 / baseline_results["rank"].dropna()).mean(), 4))

out = DATA / "patient_phenotype_only_baseline.csv"
baseline_results.to_csv(out, index=False)
print("\nSaved results to:", out)

PHENOTYPE-ONLY BASELINE
Evaluated cases: 1959
Candidate coverage: 0.7917
Top-1 accuracy: 0.1138
Top-5 accuracy: 0.196
Top-10 accuracy: 0.221
Mean reciprocal rank: 0.1928

Saved results to: /content/drive/MyDrive/Rare Disease ai/processed/patient_phenotype_only_baseline.csv


In [ ]:
import pandas as pd
import numpy as np

results_file = DATA / "patient_phenotype_only_baseline.csv"
baseline_results = pd.read_csv(results_file)

covered = baseline_results[
    baseline_results["candidate_covered"] == 1
].copy()

print("BASELINE METRICS AUDIT")
print("=" * 45)

print("All test cases:", len(baseline_results))
print("Cases with diagnosis in candidate set:", len(covered))
print("Cases without diagnosis in candidate set:",
      len(baseline_results) - len(covered))

print("\nCOVERAGE-ADJUSTED METRICS (all cases)")
print("Top-1:", round(baseline_results["top1"].mean(), 4))
print("Top-5:", round(baseline_results["top5"].mean(), 4))
print("Top-10:", round(baseline_results["top10"].mean(), 4))

print("\nRANKING METRICS (covered cases only)")
if len(covered):
    print("Top-1:", round(covered["top1"].mean(), 4))
    print("Top-5:", round(covered["top5"].mean(), 4))
    print("Top-10:", round(covered["top10"].mean(), 4))
    print(
        "MRR:",
        round((1 / covered["rank"].dropna()).mean(), 4)
    )

print("\nRank summary:")
print(covered["rank"].describe().to_string())

audit_path = DATA / "patient_phenotype_baseline_metrics_audit.csv"

pd.DataFrame([
    {
        "metric_group": "all_cases",
        "cases": len(baseline_results),
        "coverage": baseline_results["candidate_covered"].mean(),
        "top1": baseline_results["top1"].mean(),
        "top5": baseline_results["top5"].mean(),
        "top10": baseline_results["top10"].mean(),
        "mrr": (1 / baseline_results["rank"].dropna()).sum()
              / len(baseline_results)
    },
    {
        "metric_group": "covered_cases",
        "cases": len(covered),
        "coverage": 1.0 if len(covered) else 0.0,
        "top1": covered["top1"].mean() if len(covered) else np.nan,
        "top5": covered["top5"].mean() if len(covered) else np.nan,
        "top10": covered["top10"].mean() if len(covered) else np.nan,
        "mrr": (1 / covered["rank"].dropna()).mean()
              if len(covered) else np.nan
    }
]).to_csv(audit_path, index=False)

print("\nSaved metrics audit to:", audit_path)

BASELINE METRICS AUDIT
All test cases: 1959
Cases with diagnosis in candidate set: 1551
Cases without diagnosis in candidate set: 408

COVERAGE-ADJUSTED METRICS (all cases)
Top-1: 0.1138
Top-5: 0.196
Top-10: 0.221

RANKING METRICS (covered cases only)
Top-1: 0.1438
Top-5: 0.2476
Top-10: 0.2792
MRR: 0.1928

Rank summary:
count     1551.000000
mean      2230.521599
std       3368.590345
min          1.000000
25%          6.000000
50%        258.000000
75%       3766.000000
max      10724.000000

Saved metrics audit to: /content/drive/MyDrive/Rare Disease ai/processed/patient_phenotype_baseline_metrics_audit.csv


In [ ]:
from pathlib import Path

hpo_file = PROJECT / "hp.obo"

print("ONTOLOGY CHECK")
print("=" * 40)
print("HPO file exists:", hpo_file.exists())
if hpo_file.exists():
    print("HPO file size (MB):",
          round(hpo_file.stat().st_size / (1024 * 1024), 2))

print("\nDISEASE PROFILE EXAMPLE")
example_disease = next(iter(disease_profiles))
print("Disease ID:", example_disease)
print("Number of HPO terms:",
      len(disease_profiles[example_disease]))
print("Example terms:",
      sorted(disease_profiles[example_disease])[:10])

print("\nPATIENT PROFILE EXAMPLE")
example_case = test_cases.iloc[0]
print("Case ID:", example_case["case_id"])
print("Phenotype count:", example_case["phenotype_count"])
print("Diagnosis IDs:", example_case["diagnosis_ids"])

ONTOLOGY CHECK
HPO file exists: True
HPO file size (MB): 10.36

DISEASE PROFILE EXAMPLE
Disease ID: DECIPHER:1
Number of HPO terms: 5
Example terms: ['HP:0000252', 'HP:0001249', 'HP:0001250', 'HP:0001252', 'HP:0001518']

PATIENT PROFILE EXAMPLE
Case ID: PMID_28239884_Family_1_proband
Phenotype count: 1
Diagnosis IDs: OMIM:148600


In [ ]:
try:
    import obonet
except ImportError:
    %pip -q install obonet
    import obonet

import networkx as nx

hpo_graph = obonet.read_obo(str(PROJECT / "hp.obo"))

print("HPO ONTOLOGY SUMMARY")
print("=" * 40)
print("Ontology terms:", len(hpo_graph.nodes))
print("Ontology edges:", len(hpo_graph.edges))

# HPO terms should normally be represented as HP: identifiers.
hpo_nodes = [n for n in hpo_graph.nodes if str(n).startswith("HP:")]
print("HP-prefixed nodes:", len(hpo_nodes))

# Inspect parent relationships for an example term.
example_term = "HP:0001250"
print("\nExample term:", example_term)
print("Term exists:", example_term in hpo_graph)

if example_term in hpo_graph:
    print("Direct parents:", list(hpo_graph.successors(example_term))[:10])

HPO ONTOLOGY SUMMARY
Ontology terms: 19894
Ontology edges: 24436
HP-prefixed nodes: 19894

Example term: HP:0001250
Term exists: True
Direct parents: ['HP:0012638']


In [ ]:
# Build a lookup of each HPO term and its ancestors.
# In this graph, successors are parents and ancestors are more general terms.

hpo_term_set = set(hpo_graph.nodes)

def get_hpo_ancestors(term):
    if term not in hpo_graph:
        return set()
    return nx.descendants(hpo_graph, term) | set(
        hpo_graph.successors(term)
    )

# Check term coverage before computing similarities.
patient_terms_all = {
    term
    for terms in test_cases["hpo_terms"].dropna()
    for term in normalize_terms(terms)
}

disease_terms_all = {
    term
    for terms in disease_profiles.values()
    for term in terms
}

print("ONTOLOGY TERM COVERAGE")
print("=" * 40)
print("Unique patient HPO terms:", len(patient_terms_all))
print("Patient terms in ontology:",
      len(patient_terms_all & hpo_term_set))
print("Unique disease HPO terms:", len(disease_terms_all))
print("Disease terms in ontology:",
      len(disease_terms_all & hpo_term_set))
print("Patient terms missing from ontology:",
      len(patient_terms_all - hpo_term_set))
print("Disease terms missing from ontology:",
      len(disease_terms_all - hpo_term_set))

ONTOLOGY TERM COVERAGE
Unique patient HPO terms: 2004
Patient terms in ontology: 2003
Unique disease HPO terms: 11611
Disease terms in ontology: 11611
Patient terms missing from ontology: 1
Disease terms missing from ontology: 0


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def hpo_ancestors(term):
    if term not in hpo_graph:
        return frozenset()

    return frozenset(
        nx.descendants(hpo_graph, term)
        | set(hpo_graph.successors(term))
    )

# Cache ontology ancestors for terms used in the test cases
# and the disease profiles.
terms_to_cache = patient_terms_all | disease_terms_all

for term in terms_to_cache:
    hpo_ancestors(term)

print("ONTOLOGY CACHE SUMMARY")
print("=" * 40)
print("Terms prepared:", len(terms_to_cache))
print("Example term:", "HP:0001250")
print("Example ancestors:", sorted(hpo_ancestors("HP:0001250")))

ONTOLOGY CACHE SUMMARY
Terms prepared: 11650
Example term: HP:0001250
Example ancestors: ['HP:0000001', 'HP:0000118', 'HP:0000707', 'HP:0012638']


In [ ]:
print("Baseline saved:",
      (DATA / "patient_phenotype_only_baseline.csv").exists())
print("Filtered annotations saved:",
      (DATA / "patient_test_leakage_reduced_annotations.csv").exists())
print("Publication split saved:",
      (DATA / "patient_publication_split.csv").exists())

Baseline saved: True
Filtered annotations saved: True
Publication split saved: True


In [ ]:
import torch
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Training-only GNN checkpoint:",
      (DATA / "rare_disease_gnn_training_only.pt").exists())
print("Existing all-disease GNN checkpoint:",
      (DATA / "rare_disease_gnn.pt").exists())
print("Saved phenotype baseline:",
      (DATA / "patient_phenotype_only_baseline.csv").exists())

CUDA available: True
GPU: Tesla T4


NameError: name 'DATA' is not defined

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not active yet. Do not start GNN training.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Project exists:", PROJECT.exists())
print("Processed folder exists:", DATA.exists())
print("Training checkpoint exists:",
      (DATA / "rare_disease_gnn_training_only.pt").exists())
print("Publication split exists:",
      (DATA / "patient_publication_split.csv").exists())

Mounted at /content/drive
GPU available: True
GPU: Tesla T4
Project exists: True
Processed folder exists: True
Training checkpoint exists: True
Publication split exists: True


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

files_to_check = [
    DATA / "rare_disease_gnn_training_only.pt",
    DATA / "patient_publication_split.csv",
    DATA / "patient_test_leakage_reduced_annotations.csv",
    DATA / "patient_test_disease_coverage.csv",
    DATA / "patient_phenotype_only_baseline.csv",
    DATA / "patient_phenotype_baseline_metrics_audit.csv",
    DATA / "gnn_training_annotations.csv",
    DATA / "disease_level_split.csv",
    PROJECT / "hp.obo",
]

for path in files_to_check:
    if path.exists():
        print(f"FOUND | {path.name} | {path.stat().st_size / (1024**2):.2f} MB")
    else:
        print(f"MISSING | {path.name}")

print("\nModel-folder Python files:")
if MODEL.exists():
    for path in sorted(MODEL.glob("*.py")):
        print(f"FOUND | {path.name}")

FOUND | rare_disease_gnn_training_only.pt | 4.73 MB
FOUND | patient_publication_split.csv | 0.59 MB
FOUND | patient_test_leakage_reduced_annotations.csv | 33.52 MB
FOUND | patient_test_disease_coverage.csv | 0.00 MB
FOUND | patient_phenotype_only_baseline.csv | 0.12 MB
FOUND | patient_phenotype_baseline_metrics_audit.csv | 0.00 MB
FOUND | gnn_training_annotations.csv | 21.95 MB
FOUND | disease_level_split.csv | 0.16 MB
FOUND | hp.obo | 10.36 MB

Model-folder Python files:
FOUND | rare_disease_ai.py
FOUND | rare_disease_data_loader.py
FOUND | rare_disease_predictor.py


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "patient_publication_split.csv",
    "patient_test_disease_coverage.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_phenotype_baseline_metrics_audit.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 65}\n{name}")

    if not path.exists():
        print("FILE MISSING")
        continue

    try:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(5).to_string(index=False))
    except Exception as e:
        print("Read error:", str(e))

print("\n" + "=" * 65)
print("Checkpoint file size:",
      f"{(DATA / 'rare_disease_gnn_training_only.pt').stat().st_size / (1024**2):.2f} MB")|

SyntaxError: invalid syntax (2139282329.py, line 31)

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "patient_publication_split.csv",
    "patient_test_disease_coverage.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_phenotype_baseline_metrics_audit.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 60}")
    print(name)

    if not path.exists():
        print("FILE MISSING")
        continue

    try:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(5).to_string(index=False))
    except Exception as e:
        print("Read error:", str(e))

print("\n" + "=" * 60)

checkpoint = DATA / "rare_disease_gnn_training_only.pt"

if checkpoint.exists():
    size_mb = checkpoint.stat().st_size / (1024 ** 2)
    print(f"Checkpoint file size: {size_mb:.2f} MB")
else:
    print("Checkpoint file missing")


patient_publication_split.csv
Shape: (10345, 4)
Columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
                    case_id     pmid       split diagnosis_ids
PMID_19836009_Family_E_II_6 19836009 development   OMIM:277600
PMID_19836009_Family_E_II_3 19836009 development   OMIM:277600
PMID_19836009_Family_B_II_4 19836009 development   OMIM:277600
PMID_19836009_Family_B_II_2 19836009 development   OMIM:277600
      PMID_25469541_patient 25469541 development   OMIM:277600

patient_test_disease_coverage.csv
Shape: (1, 5)
Columns: ['test_cases', 'cases_with_diagnosis_profile', 'cases_without_diagnosis_profile', 'diseases_with_profiles', 'annotation_rows_after_filter']
 test_cases  cases_with_diagnosis_profile  cases_without_diagnosis_profile  diseases_with_profiles  annotation_rows_after_filter
       1959                          1551                              408                   12827                        280733

patient_phenotype_only_baseline.csv
Shape: (1959, 8)
Columns:

In [ ]:
from pathlib import Path
import torch

from google.colab import drive
drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

print("Project exists:", PROJECT.exists())
print("Processed folder exists:", DATA.exists())
print("Model folder exists:", MODEL.exists())

for name in [
    "patient_publication_split.csv",
    "patient_test_disease_coverage.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_phenotype_baseline_metrics_audit.csv",
    "patient_test_leakage_reduced_annotations.csv",
    "rare_disease_gnn_training_only.pt",
]:
    path = DATA / name
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project exists: True
Processed folder exists: True
Model folder exists: True
patient_publication_split.csv: FOUND
patient_test_disease_coverage.csv: FOUND
patient_phenotype_only_baseline.csv: FOUND
patient_phenotype_baseline_metrics_audit.csv: FOUND
patient_test_leakage_reduced_annotations.csv: FOUND
rare_disease_gnn_training_only.pt: FOUND

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

print("=== Saved model files ===")
for folder in [MODEL, DATA]:
    print(f"\n{folder}")
    for p in sorted(folder.glob("*")):
        if p.is_file() and (
            p.suffix in {".pt", ".pth", ".py", ".json", ".yaml", ".yml"}
        ):
            print(f"{p.name:45} {p.stat().st_size / (1024**2):8.2f} MB")

print("\n=== Training annotations ===")
for name in [
    "gnn_training_annotations.csv",
    "clean_hpo_annotations.csv",
    "disease_level_split.csv",
]:
    p = DATA / name
    if p.exists():
        df = pd.read_csv(p, nrows=5)
        print(f"\n{name}")
        print("Columns:", df.columns.tolist())
        print(df.head(2).to_string(index=False))

print("\n=== Runtime ===")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
))

=== Saved model files ===

/content/drive/MyDrive/Rare Disease ai/model
prediction_config.json                            0.00 MB
rare_disease_ai.py                                0.00 MB
rare_disease_data_loader.py                       0.00 MB
rare_disease_predictor.py                         0.00 MB

/content/drive/MyDrive/Rare Disease ai/processed
rare_disease_gnn.pt                               6.69 MB
rare_disease_gnn_training_only.pt                 4.73 MB

=== Training annotations ===

gnn_training_annotations.csv
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']
database_id                                  disease_name  qualifier     hpo_id     reference evidence  onset frequency  sex  modifier aspect               biocuration
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0011097 PMID:31675180      PCS    NaN       1/2  NaN       NaN      P HPO:p

In [ ]:
from pathlib import Path
import torch

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
checkpoint_path = DATA / "rare_disease_gnn_training_only.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} entries")
            for name, tensor in list(value.items())[:12]:
                if torch.is_tensor(tensor):
                    print(f"  {name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")
                else:
                    print(f"  {name}: {type(tensor).__name__}")
        elif torch.is_tensor(value):
            print(f"{key}: tensor shape={tuple(value.shape)}")
        else:
            print(f"{key}: {type(value).__name__} = {str(value)[:150]}")
else:
    print("Unexpected checkpoint format.")

Checkpoint type: dict
Top-level keys: ['model_state_dict', 'num_nodes', 'hidden_dim', 'disease_ids', 'phenotype_ids', 'disease_to_idx', 'phenotype_to_idx', 'training_disease_count', 'training_association_count', 'validation_loss', 'seed']

model_state_dict: dictionary with 5 entries
  embedding.weight: shape=(17322, 64), dtype=torch.float32
  conv1.bias: shape=(64,), dtype=torch.float32
  conv1.lin.weight: shape=(64, 64), dtype=torch.float32
  conv2.bias: shape=(64,), dtype=torch.float32
  conv2.lin.weight: shape=(64, 64), dtype=torch.float32
num_nodes: int = 17322
hidden_dim: int = 64
disease_ids: list = ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400', 'OMIM:101600', 'OMIM:101800', 'OMIM:102200', 'OMIM:102370', 'OMIM:102500',
phenotype_ids: list = ['HP:0000002', 'HP:0000003', 'HP:0000006', 'HP:0000007', 'HP:0000008', 'HP:0000009', 'HP:0000010', 'HP:0000011', 'HP:0000012', 'HP:0000013', 'HP:00000

disease_to_idx: dictionary with 7340 entries
  OMIM:100300: in

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

for folder in [PROJECT / "model", PROJECT]:
    print(f"\n{'=' * 65}")
    print("Folder:", folder)

    for path in sorted(folder.glob("*.py")):
        print(f"\n--- {path.relative_to(PROJECT)} ---")
        lines = path.read_text(encoding="utf-8").splitlines()

        # Show model architecture, training, graph construction,
        # and prediction code without dumping unrelated files.
        keywords = (
            "class ", "def ", "GCNConv", "GATConv", "forward(",
            "optimizer", "loss", "train", "edge_index",
            "model_state_dict", "predict", "validation"
        )
        selected = [
            (i, line) for i, line in enumerate(lines)
            if any(word.lower() in line.lower() for word in keywords)
        ]

        if not selected:
            print("(No matching model-related lines)")
        else:
            for i, line in selected[:100]:
                print(f"{i + 1:4}: {line[:180]}")


Folder: /content/drive/MyDrive/Rare Disease ai/model

--- model/rare_disease_ai.py ---
   2: from rare_disease_data_loader import load_prediction_data
   4: _DATA = load_prediction_data()
  10: def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,

--- model/rare_disease_data_loader.py ---
  10: def load_prediction_data():
  40:     train_ids = set(
  41:         split.loc[split["split"].str.lower() == "train", "disease_id"]
  45:     for disease in train_ids:
  49:     n_train = max(len(train_ids), 1)
  51:         term: -np.log(count / n_train)

--- model/rare_disease_predictor.py ---
   1: """Rare Disease Prediction AI — research prototype.
  10: def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
  73: def explain_predictions(patient_hpo_terms, patient_genes=None, top_k=10):
  74:     results = predict_diseases(

Folder: /content/drive/MyDrive/Rare Disease ai


In [ ]:
from pathlib import Path

MODEL = Path("/content/drive/MyDrive/Rare Disease ai/model")

for filename in [
    "rare_disease_data_loader.py",
    "rare_disease_predictor.py",
    "rare_disease_ai.py",
]:
    path = MODEL / filename

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("FILE NOT FOUND")


rare_disease_data_loader.py

from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

def load_prediction_data():
    ann = pd.read_csv(
        DATA / "clean_hpo_annotations.csv",
        low_memory=False,
        dtype={"database_id": "string", "hpo_id": "string"}
    ).dropna(subset=["database_id", "hpo_id"])

    ann["database_id"] = ann["database_id"].str.strip()
    ann["hpo_id"] = ann["hpo_id"].str.strip()

    phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

    gene_df = pd.read_csv(
        DATA / "normalized_disease_gene_mapping.csv",
        dtype=str
    ).fillna("")

    gene_map = {
        row["disease_id"]: {
            g.strip().upper()
            for g in row["gene_symbols"].split(";")
            if g.strip()
        }
        for _, row in gene_df.iterrows()
    }

    split = pd.read_csv(
        DATA / "di

In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

# Load saved checkpoint on CPU
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

# Compare checkpoint disease IDs with the saved split
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"].dropna()
)
checkpoint_ids = set(ckpt["disease_ids"])

print("=== Checkpoint split verification ===")
print("Diseases in saved training split:", len(train_ids))
print("Diseases in checkpoint index:", len(checkpoint_ids))
print("Exact match:", train_ids == checkpoint_ids)
print("Missing from checkpoint:", len(train_ids - checkpoint_ids))
print("Unexpected in checkpoint:", len(checkpoint_ids - train_ids))

# Compare annotation files
print("\n=== Annotation comparison ===")
for filename in ["clean_hpo_annotations.csv", "gnn_training_annotations.csv"]:
    path = DATA / filename
    df = pd.read_csv(path, usecols=["database_id", "hpo_id"], dtype=str)
    pairs = set(
        zip(
            df["database_id"].dropna().str.strip(),
            df["hpo_id"].dropna().str.strip()
        )
    )
    print(filename, "| rows:", len(df), "| unique disease-term pairs:", len(pairs))
    if filename == "clean_hpo_annotations.csv":
        clean_pairs = pairs
    else:
        training_pairs = pairs

print("Training pairs also in clean annotations:", len(training_pairs & clean_pairs))
print("Training pairs absent from clean annotations:", len(training_pairs - clean_pairs))

# Locate scripts that may contain the actual GNN training loop
print("\n=== Possible training scripts ===")
for path in PROJECT.rglob("*.py"):
    if any(word in path.name.lower() for word in ["train", "gnn", "graph", "model"]):
        print(path.relative_to(PROJECT))

=== Checkpoint split verification ===
Diseases in saved training split: 7340
Diseases in checkpoint index: 7340
Exact match: True
Missing from checkpoint: 0
Unexpected in checkpoint: 0

=== Annotation comparison ===
clean_hpo_annotations.csv | rows: 286047 | unique disease-term pairs: 286047
gnn_training_annotations.csv | rows: 179653 | unique disease-term pairs: 179653
Training pairs also in clean annotations: 179653
Training pairs absent from clean annotations: 0

=== Possible training scripts ===


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

graph_files = [
    "rare_disease_training_graph.graphml",
    "rare_disease_knowledge_graph.graphml",
    "rare_disease_enriched_graph.graphml",
]

for name in graph_files:
    path = DATA / name
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}",
          f"({path.stat().st_size / (1024**2):.2f} MB)" if path.exists() else "")

print("\n=== Training checkpoint metadata ===")
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

print("Nodes:", ckpt["num_nodes"])
print("Hidden dimension:", ckpt["hidden_dim"])
print("Training diseases:", ckpt["training_disease_count"])
print("Training associations:", ckpt["training_association_count"])
print("Validation loss:", ckpt["validation_loss"])
print("Seed:", ckpt["seed"])

print("\n=== Training annotation index coverage ===")
ann = pd.read_csv(
    DATA / "gnn_training_annotations.csv",
    usecols=["database_id", "hpo_id"],
    dtype=str
).dropna()

disease_ids = set(ann["database_id"].str.strip())
phenotype_ids = set(ann["hpo_id"].str.strip())

print("Annotation diseases:", len(disease_ids))
print("Annotation phenotypes:", len(phenotype_ids))
print("Diseases outside checkpoint index:",
      len(disease_ids - set(ckpt["disease_ids"])))
print("Phenotypes outside checkpoint index:",
      len(phenotype_ids - set(ckpt["phenotype_ids"])))

rare_disease_training_graph.graphml: FOUND (20.36 MB)
rare_disease_knowledge_graph.graphml: FOUND (29.03 MB)
rare_disease_enriched_graph.graphml: FOUND (31.06 MB)

=== Training checkpoint metadata ===
Nodes: 17322
Hidden dimension: 64
Training diseases: 7340
Training associations: 179653
Validation loss: 0.57393878698349
Seed: 42

=== Training annotation index coverage ===
Annotation diseases: 7340
Annotation phenotypes: 9982
Diseases outside checkpoint index: 0
Phenotypes outside checkpoint index: 0


In [ ]:
from pathlib import Path
import networkx as nx
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Load only the training graph
graph_path = DATA / "rare_disease_training_graph.graphml"
G = nx.read_graphml(graph_path)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

# Inspect node and edge attributes
print("\nSample nodes:")
for node, attrs in list(G.nodes(data=True))[:5]:
    print(node, attrs)

print("\nSample edges:")
for u, v, attrs in list(G.edges(data=True))[:5]:
    print(u, "->", v, attrs)

# Compare expected training associations with graph edges
ann = pd.read_csv(
    DATA / "gnn_training_annotations.csv",
    usecols=["database_id", "hpo_id"],
    dtype=str
).dropna()

expected = set(zip(
    ann["database_id"].str.strip(),
    ann["hpo_id"].str.strip()
))

actual = set()
for u, v in G.edges():
    u, v = str(u), str(v)
    if u.startswith(("OMIM:", "ORPHA:", "DECIPHER:")) and v.startswith("HP:"):
        actual.add((u, v))
    elif v.startswith(("OMIM:", "ORPHA:", "DECIPHER:")) and u.startswith("HP:"):
        actual.add((v, u))

print("\n=== Association audit ===")
print("Expected training pairs:", len(expected))
print("Pairs found in graph:", len(actual))
print("Expected pairs missing from graph:", len(expected - actual))
print("Graph pairs absent from training annotations:", len(actual - expected))

Graph nodes: 24279
Graph edges: 189775

Sample nodes:
OMIM:619340 {'node_type': 'disease', 'name': 'Developmental and epileptic encephalopathy 96'}
OMIM:614102 {'node_type': 'disease', 'name': 'Immunoglobulin kappa light chain deficiency'}
OMIM:619426 {'node_type': 'disease', 'name': 'White-Kernohan syndrome'}
OMIM:610370 {'node_type': 'disease', 'name': 'Diarrhea 4, malabsorptive, congenital'}
OMIM:609621 {'node_type': 'disease', 'name': 'Short QT syndrome 2'}

Sample edges:
OMIM:619340 -> HP:0011097 {'relation': 'associated_with'}
OMIM:619340 -> HP:0002187 {'relation': 'associated_with'}
OMIM:619340 -> HP:0001518 {'relation': 'associated_with'}
OMIM:619340 -> HP:0032792 {'relation': 'associated_with'}
OMIM:619340 -> HP:0011451 {'relation': 'associated_with'}

=== Association audit ===
Expected training pairs: 179653
Pairs found in graph: 179653
Expected pairs missing from graph: 0
Graph pairs absent from training annotations: 0


In [ ]:
from pathlib import Path
from collections import Counter
import networkx as nx
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

G = nx.read_graphml(DATA / "rare_disease_training_graph.graphml")

print("=== Node types ===")
node_types = Counter(
    attrs.get("node_type", "UNSPECIFIED")
    for _, attrs in G.nodes(data=True)
)
for node_type, count in node_types.most_common():
    print(f"{node_type}: {count}")

print("\n=== Edge relations ===")
relations = Counter(
    attrs.get("relation", "UNSPECIFIED")
    for _, _, attrs in G.edges(data=True)
)
for relation, count in relations.most_common():
    print(f"{relation}: {count}")

print("\n=== Edge-type audit ===")
edge_types = Counter()
for u, v, attrs in G.edges(data=True):
    type_u = G.nodes[u].get("node_type", "UNSPECIFIED")
    type_v = G.nodes[v].get("node_type", "UNSPECIFIED")
    relation = attrs.get("relation", "UNSPECIFIED")
    edge_types[(type_u, relation, type_v)] += 1

for (type_u, relation, type_v), count in edge_types.most_common(25):
    print(f"{type_u} --{relation}--> {type_v}: {count}")

print("\n=== Example extra nodes ===")
for node, attrs in G.nodes(data=True):
    if attrs.get("node_type", "UNSPECIFIED") not in {"disease", "phenotype"}:
        print(node, attrs)
        if sum(
            1 for _, a in G.nodes(data=True)
            if a.get("node_type", "UNSPECIFIED") not in {"disease", "phenotype"}
        ) >= 10:
            break

=== Node types ===
phenotype: 11658
disease: 7340
gene: 5281

=== Edge relations ===
associated_with: 179653
disease_gene: 10122

=== Edge-type audit ===
disease --associated_with--> phenotype: 179653
disease --disease_gene--> gene: 10122

=== Example extra nodes ===
NCBIGene:64170 {'node_type': 'gene', 'gene_symbol': 'CARD9'}


In [ ]:
from pathlib import Path
import pandas as pd
import networkx as nx

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

G = nx.read_graphml(DATA / "rare_disease_training_graph.graphml")

split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"].dropna()
)
test_ids = set(
    split.loc[split["split"].str.lower() == "test", "disease_id"].dropna()
)

gene_edges = []
for u, v, attrs in G.edges(data=True):
    if attrs.get("relation") == "disease_gene":
        disease = u if G.nodes[u].get("node_type") == "disease" else v
        gene = v if disease == u else u
        gene_edges.append((disease, gene))

edge_diseases = {d for d, _ in gene_edges}

print("Disease-gene edges:", len(gene_edges))
print("Unique diseases with gene edges:", len(edge_diseases))
print("Edges involving training diseases:",
      sum(d in train_ids for d, _ in gene_edges))
print("Edges involving held-out diseases:",
      sum(d in test_ids for d, _ in gene_edges))
print("Diseases outside saved split:", len(edge_diseases - train_ids - test_ids))

print("\n=== Candidate gene mapping files ===")
for p in sorted(DATA.glob("*gene*mapping*.csv")):
    print(p.name, f"({p.stat().st_size / (1024**2):.2f} MB)")
    df = pd.read_csv(p, nrows=3, dtype=str)
    print("Columns:", df.columns.tolist())
    print(df.to_string(index=False))

Disease-gene edges: 10122
Unique diseases with gene edges: 7340
Edges involving training diseases: 10122
Edges involving held-out diseases: 0
Diseases outside saved split: 0

=== Candidate gene mapping files ===
normalized_disease_gene_mapping.csv (0.22 MB)
Columns: ['disease_id', 'gene_symbols']
 disease_id gene_symbols
OMIM:100100        CHRM3
OMIM:100300     ARHGAP31
OMIM:100800        FGFR3


In [ ]:
import importlib.util
import json
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL = PROJECT / "model"

print("=== Graph libraries ===")
for package in ["torch", "torch_geometric", "networkx", "sklearn"]:
    spec = importlib.util.find_spec(package)
    print(f"{package}: {'installed' if spec else 'not installed'}")

print("\n=== Prediction configuration ===")
config_path = MODEL / "prediction_config.json"

if config_path.exists():
    print(config_path.read_text(encoding="utf-8"))
else:
    print("Configuration file not found")

print("\n=== Project Python files ===")
for path in sorted(PROJECT.rglob("*.py")):
    print(path.relative_to(PROJECT))

=== Graph libraries ===
torch: installed
torch_geometric: not installed
networkx: installed
sklearn: installed

=== Prediction configuration ===
{
  "model_name": "Rare Disease Phenotype Ranking Prototype",
  "version": "1.0",
  "phenotype_scoring": "information_content_weighted_f1",
  "gene_weight": 0.25,
  "phenotype_annotations": "/content/drive/MyDrive/Rare Disease ai/processed/clean_hpo_annotations.csv",
  "gene_mapping": "/content/drive/MyDrive/Rare Disease ai/processed/normalized_disease_gene_mapping.csv",
  "disease_split": "/content/drive/MyDrive/Rare Disease ai/processed/disease_level_split.csv",
  "clinical_validation": "not_yet_performed",
  "scores_are_probabilities": false
}

=== Project Python files ===
model/rare_disease_ai.py
model/rare_disease_data_loader.py
model/rare_disease_predictor.py


In [ ]:
%pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch_geometric
from torch_geometric.nn import GCNConv

print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

layer = GCNConv(64, 64)
print("GCNConv import and initialization: OK")

PyTorch: 2.11.0+cu128
PyTorch Geometric: 2.8.1
CUDA available: True
GPU: Tesla T4
GCNConv import and initialization: OK


In [ ]:
from pathlib import Path
import torch
from torch_geometric.nn import GCNConv

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

class RareDiseaseGCN(torch.nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.embedding = torch.nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, edge_index):
        x = self.embedding.weight
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = RareDiseaseGCN(
    num_nodes=ckpt["num_nodes"],
    hidden_dim=ckpt["hidden_dim"]
)

model.load_state_dict(ckpt["model_state_dict"], strict=True)
model = model.to(device).eval()

print("Checkpoint loaded successfully.")
print("Parameter count:", sum(p.numel() for p in model.parameters()))
print("Device:", device)
print("Disease index:", len(ckpt["disease_to_idx"]))
print("Phenotype index:", len(ckpt["phenotype_to_idx"]))
print("Architecture:", model)

Checkpoint loaded successfully.
Parameter count: 1116928
Device: cuda
Disease index: 7340
Phenotype index: 9982
Architecture: RareDiseaseGCN(
  (embedding): Embedding(17322, 64)
  (conv1): GCNConv(64, 64)
  (conv2): GCNConv(64, 64)
)


In [ ]:
from pathlib import Path
import pandas as pd
import torch

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

# Verify the two index spaces are contiguous and non-overlapping.
disease_indices = sorted(disease_to_idx.values())
phenotype_indices = sorted(phenotype_to_idx.values())

print("Disease index range:", min(disease_indices), max(disease_indices))
print("Phenotype index range:", min(phenotype_indices), max(phenotype_indices))
print("Disease indexes contiguous:",
      disease_indices == list(range(len(disease_to_idx))))
print("Phenotype indexes contiguous:",
      phenotype_indices == list(range(len(disease_to_idx),
                                      len(disease_to_idx) + len(phenotype_to_idx))))

# Load only the training associations.
ann = pd.read_csv(
    DATA / "gnn_training_annotations.csv",
    usecols=["database_id", "hpo_id"],
    dtype=str
).dropna()

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
ann = ann.drop_duplicates(["database_id", "hpo_id"])

# Map associations using the saved checkpoint's exact indexes.
pairs = [
    (disease_to_idx[d], phenotype_to_idx[h])
    for d, h in zip(ann["database_id"], ann["hpo_id"])
    if d in disease_to_idx and h in phenotype_to_idx
]

edge_pairs = torch.tensor(pairs, dtype=torch.long).T.contiguous()

# GCN message passing over both directions of each association.
edge_index = torch.cat([edge_pairs, edge_pairs.flip(0)], dim=1).to(device)

print("\n=== Reconstructed graph ===")
print("Unique training associations mapped:", len(pairs))
print("Expected checkpoint associations:", ckpt["training_association_count"])
print("Edge index shape:", tuple(edge_index.shape))
print("Bidirectional edges:", edge_index.size(1))
print("All indexes in range:",
      int(edge_index.min()) >= 0 and
      int(edge_index.max()) < ckpt["num_nodes"])

Disease index range: 0 7339
Phenotype index range: 7340 17321
Disease indexes contiguous: True
Phenotype indexes contiguous: True

=== Reconstructed graph ===
Unique training associations mapped: 179653
Expected checkpoint associations: 179653
Edge index shape: (2, 359306)
Bidirectional edges: 359306
All indexes in range: True


In [ ]:
import torch

model.eval()

with torch.no_grad():
    embeddings = model(edge_index)

print("Embedding shape:", tuple(embeddings.shape))
print("Expected shape:", (ckpt["num_nodes"], ckpt["hidden_dim"]))
print("All embeddings finite:", bool(torch.isfinite(embeddings).all()))
print("Embedding device:", embeddings.device)

disease_embeddings = embeddings[:len(disease_to_idx)]
phenotype_embeddings = embeddings[
    len(disease_to_idx):
    len(disease_to_idx) + len(phenotype_to_idx)
]

print("Disease embeddings:", tuple(disease_embeddings.shape))
print("Phenotype embeddings:", tuple(phenotype_embeddings.shape))

Embedding shape: (17322, 64)
Expected shape: (17322, 64)
All embeddings finite: True
Embedding device: cuda:0
Disease embeddings: (7340, 64)
Phenotype embeddings: (9982, 64)


In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

cases_path = PROJECT / "all_phenopackets"
split_path = DATA / "patient_publication_split.csv"

print("Test split exists:", split_path.exists())
print("Phenopacket directory exists:", cases_path.exists())

split = pd.read_csv(split_path, dtype=str)
test_cases = split.loc[split["split"] == "test"].copy()

print("Publication-held-out test cases:", len(test_cases))
print("Unique test publications:", test_cases["pmid"].nunique())
print("Checkpoint phenotype vocabulary:", len(phenotype_to_idx))
print("\nTest split columns:", test_cases.columns.tolist())
print(test_cases.head(3).to_string(index=False))

Test split exists: True
Phenopacket directory exists: True
Publication-held-out test cases: 1959
Unique test publications: 346
Checkpoint phenotype vocabulary: 9982

Test split columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
                       case_id     pmid split diagnosis_ids
PMID_28239884_Family_1_proband 28239884  test   OMIM:148600
PMID_28239884_Family_2_proband 28239884  test   OMIM:148600
PMID_28239884_Family_3_proband 28239884  test   OMIM:148600


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_cases = split.loc[split["split"] == "test"].copy()

# Index JSON files by filename for direct lookup.
json_files = {p.name: p for p in PACKETS.glob("*.json")}
print("JSON files indexed:", len(json_files))

def extract_hpo_terms(obj):
    terms = set()

    def walk(x):
        if isinstance(x, dict):
            term = x.get("id")
            if isinstance(term, str) and term.startswith("HP:"):
                terms.add(term)
            for value in x.values():
                walk(value)
        elif isinstance(x, list):
            for value in x:
                walk(value)

    walk(obj)
    return terms

case_terms = {}
missing_files = []

for case_id in test_cases["case_id"]:
    path = json_files.get(f"{case_id}.json")
    if path is None:
        missing_files.append(case_id)
        continue

    with path.open(encoding="utf-8") as f:
        case_terms[case_id] = extract_hpo_terms(json.load(f))

known_terms = set(phenotype_to_idx)

coverage_rows = []
for row in test_cases.itertuples(index=False):
    terms = case_terms.get(row.case_id, set())
    recognized = terms & known_terms
    coverage_rows.append({
        "case_id": row.case_id,
        "pmid": row.pmid,
        "diagnosis_ids": row.diagnosis_ids,
        "n_patient_terms": len(terms),
        "n_recognized_terms": len(recognized),
        "term_coverage": len(recognized) / len(terms) if terms else 0.0,
    })

coverage = pd.DataFrame(coverage_rows)

print("\nTest cases:", len(test_cases))
print("Cases with missing JSON files:", len(missing_files))
print("Cases with at least one phenotype:", int((coverage["n_patient_terms"] > 0).sum()))
print("Cases with at least one recognized term:", int((coverage["n_recognized_terms"] > 0).sum()))
print("Mean per-case HPO vocabulary coverage:",
      round(coverage.loc[coverage["n_patient_terms"] > 0, "term_coverage"].mean(), 4))
print("Median per-case HPO vocabulary coverage:",
      round(coverage.loc[coverage["n_patient_terms"] > 0, "term_coverage"].median(), 4))

print("\nCoverage summary:")
print(coverage[["n_patient_terms", "n_recognized_terms", "term_coverage"]].describe().round(4))

JSON files indexed: 0

Test cases: 1959
Cases with missing JSON files: 1959
Cases with at least one phenotype: 0
Cases with at least one recognized term: 0
Mean per-case HPO vocabulary coverage: nan
Median per-case HPO vocabulary coverage: nan

Coverage summary:
       n_patient_terms  n_recognized_terms  term_coverage
count           1959.0              1959.0         1959.0
mean               0.0                 0.0            0.0
std                0.0                 0.0            0.0
min                0.0                 0.0            0.0
25%                0.0                 0.0            0.0
50%                0.0                 0.0            0.0
75%                0.0                 0.0            0.0
max                0.0                 0.0            0.0


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
PACKETS = PROJECT / "all_phenopackets"

print("Phenopacket directory exists:", PACKETS.exists())
print("Directory path:", PACKETS)

if PACKETS.exists():
    print("\nImmediate contents:")
    for p in list(PACKETS.iterdir())[:20]:
        print("DIR " if p.is_dir() else "FILE", p.name)

    print("\nFile counts:")
    print("JSON files directly inside:", sum(1 for _ in PACKETS.glob("*.json")))
    print("JSON files recursively:", sum(1 for _ in PACKETS.rglob("*.json")))
    print("TSV files recursively:", sum(1 for _ in PACKETS.rglob("*.tsv")))

Phenopacket directory exists: True
Directory path: /content/drive/MyDrive/Rare Disease ai/all_phenopackets

Immediate contents:
DIR  0.1.27

File counts:
JSON files directly inside: 0
JSON files recursively: 10427
TSV files recursively: 1


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_cases = split.loc[split["split"] == "test"].copy()

# Search recursively inside version subfolders.
json_files = {p.stem: p for p in PACKETS.rglob("*.json")}
print("JSON files indexed:", len(json_files))

def extract_hpo_terms(obj):
    terms = set()

    def walk(x):
        if isinstance(x, dict):
            term = x.get("id")
            if isinstance(term, str) and term.startswith("HP:"):
                terms.add(term)
            for value in x.values():
                walk(value)
        elif isinstance(x, list):
            for value in x:
                walk(value)

    walk(obj)
    return terms

known_terms = set(phenotype_to_idx)
coverage_rows = []
missing_files = []

for row in test_cases.itertuples(index=False):
    path = json_files.get(row.case_id)

    if path is None:
        missing_files.append(row.case_id)
        terms = set()
    else:
        with path.open(encoding="utf-8") as f:
            terms = extract_hpo_terms(json.load(f))

    recognized = terms & known_terms
    coverage_rows.append({
        "case_id": row.case_id,
        "pmid": row.pmid,
        "diagnosis_ids": row.diagnosis_ids,
        "n_patient_terms": len(terms),
        "n_recognized_terms": len(recognized),
        "term_coverage": len(recognized) / len(terms) if terms else 0.0,
    })

coverage = pd.DataFrame(coverage_rows)

print("\nTest cases:", len(test_cases))
print("Missing JSON files:", len(missing_files))
print("Cases with phenotype terms:", int((coverage["n_patient_terms"] > 0).sum()))
print("Cases with recognized terms:", int((coverage["n_recognized_terms"] > 0).sum()))
valid = coverage.loc[coverage["n_patient_terms"] > 0, "term_coverage"]
print("Mean HPO vocabulary coverage:", round(valid.mean(), 4) if len(valid) else "N/A")
print("Median HPO vocabulary coverage:", round(valid.median(), 4) if len(valid) else "N/A")

print("\nCoverage summary:")
print(coverage[["n_patient_terms", "n_recognized_terms", "term_coverage"]].describe().round(4))

JSON files indexed: 10427

Test cases: 1959
Missing JSON files: 0
Cases with phenotype terms: 1959
Cases with recognized terms: 1949
Mean HPO vocabulary coverage: 0.9904
Median HPO vocabulary coverage: 1.0

Coverage summary:
       n_patient_terms  n_recognized_terms  term_coverage
count        1959.0000           1959.0000      1959.0000
mean           23.7989             23.7147         0.9904
std            19.6747             19.6504         0.0777
min             1.0000              0.0000         0.0000
25%            10.0000             10.0000         1.0000
50%            18.0000             18.0000         1.0000
75%            31.0000             31.0000         1.0000
max           105.0000            105.0000         1.0000


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_case = split.loc[split["split"] == "test"].iloc[0]

json_path = next(
    p for p in PACKETS.rglob("*.json")
    if p.stem == test_case["case_id"]
)

with json_path.open(encoding="utf-8") as f:
    packet = json.load(f)

print("Case ID:", test_case["case_id"])
print("Expected diagnosis:", test_case["diagnosis_ids"])
print("JSON file:", json_path)
print("\nTop-level keys:", list(packet.keys()))

print("\nPhenotype-related structure:")
for key in ["phenotypicFeatures", "subject", "diseases", "interpretations"]:
    if key in packet:
        value = packet[key]
        print(f"\n{key}:")
        print(json.dumps(value[:3] if isinstance(value, list) else value,
                         indent=2)[:4000])

Case ID: PMID_28239884_Family_1_proband
Expected diagnosis: OMIM:148600
JSON file: /content/drive/MyDrive/Rare Disease ai/all_phenopackets/0.1.27/AAGAB/PMID_28239884_Family_1_proband.json

Top-level keys: ['id', 'subject', 'phenotypicFeatures', 'interpretations', 'diseases', 'metaData']

Phenotype-related structure:

phenotypicFeatures:
[
  {
    "type": {
      "id": "HP:0006482",
      "label": "Abnormal dental morphology"
    },
    "excluded": true
  },
  {
    "type": {
      "id": "HP:0007530",
      "label": "Punctate palmoplantar hyperkeratosis"
    },
    "onset": {
      "age": {
        "iso8601duration": "P18Y"
      }
    }
  },
  {
    "type": {
      "id": "HP:0001595",
      "label": "Abnormal hair morphology"
    },
    "excluded": true
  }
]

subject:
{
  "id": "Family 1 proband",
  "timeAtLastEncounter": {
    "age": {
      "iso8601duration": "P43Y"
    }
  },
  "sex": "FEMALE"
}

diseases:
[
  {
    "term": {
      "id": "OMIM:148600",
      "label": "Keratoderma, 

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_cases = split.loc[split["split"] == "test"].copy()

json_files = {p.stem: p for p in PACKETS.rglob("*.json")}

rows = []
missing = []

for row in test_cases.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        missing.append(row.case_id)
        continue

    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()

    for feature in packet.get("phenotypicFeatures", []):
        term = feature.get("type", {}).get("id")
        if not isinstance(term, str) or not term.startswith("HP:"):
            continue

        if feature.get("excluded") is True:
            excluded.add(term)
        else:
            positive.add(term)

    # Excluded and positive cannot both represent present findings.
    positive -= excluded

    rows.append({
        "case_id": row.case_id,
        "pmid": row.pmid,
        "diagnosis_ids": row.diagnosis_ids,
        "positive_hpo_terms": sorted(positive),
        "excluded_hpo_terms": sorted(excluded),
        "n_positive": len(positive),
        "n_excluded": len(excluded),
    })

patient_features = pd.DataFrame(rows)

print("Test cases in split:", len(test_cases))
print("Cases extracted:", len(patient_features))
print("Missing files:", len(missing))
print("Cases with positive phenotypes:",
      int((patient_features["n_positive"] > 0).sum()))
print("Total positive observations:",
      int(patient_features["n_positive"].sum()))
print("Total excluded observations:",
      int(patient_features["n_excluded"].sum()))

print("\nExample:")
print(patient_features[
    ["case_id", "diagnosis_ids", "n_positive", "n_excluded"]
].head(5).to_string(index=False))

Test cases in split: 1959
Cases extracted: 1959
Missing files: 0
Cases with positive phenotypes: 1959
Total positive observations: 18874
Total excluded observations: 26931

Example:
                       case_id diagnosis_ids  n_positive  n_excluded
PMID_28239884_Family_1_proband   OMIM:148600           1           3
PMID_28239884_Family_2_proband   OMIM:148600           1           3
PMID_28239884_Family_3_proband   OMIM:148600           1           3
          PMID_1680289_patient   OMIM:102700           6           0
 PMID_30733661_4_year_old_girl   OMIM:620475          11           0


In [ ]:
import torch
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

ckpt_path = DATA / "rare_disease_gnn_training_only.pt"
ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=True
)

train_split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype=str
)

print("Checkpoint disease IDs:", len(ckpt["disease_ids"]))
print("Checkpoint phenotype IDs:", len(ckpt["phenotype_ids"]))
print("Checkpoint node count:", ckpt["num_nodes"])
print("Checkpoint hidden dimension:", ckpt["hidden_dim"])

print("\nCheckpoint mapping examples:")
print("Disease IDs:", ckpt["disease_ids"][:5])
print("Phenotype IDs:", ckpt["phenotype_ids"][:5])

print("\nDisease split columns:", train_split.columns.tolist())
print(train_split.head(3).to_string(index=False))

print("\nCheckpoint disease index sample:")
for disease_id in ckpt["disease_ids"][:5]:
    print(disease_id, "->", ckpt["disease_to_idx"].get(disease_id))

print("\nCheckpoint phenotype index sample:")
for hpo_id in ckpt["phenotype_ids"][:5]:
    print(hpo_id, "->", ckpt["phenotype_to_idx"].get(hpo_id))

Checkpoint disease IDs: 7340
Checkpoint phenotype IDs: 9982
Checkpoint node count: 17322
Checkpoint hidden dimension: 64

Checkpoint mapping examples:
Disease IDs: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400']
Phenotype IDs: ['HP:0000002', 'HP:0000003', 'HP:0000006', 'HP:0000007', 'HP:0000008']

Disease split columns: ['disease_id', 'split']
 disease_id split
OMIM:100100  test
OMIM:100300 train
OMIM:100800 train

Checkpoint disease index sample:
OMIM:100300 -> 0
OMIM:100800 -> 1
OMIM:101000 -> 2
OMIM:101200 -> 3
OMIM:101400 -> 4

Checkpoint phenotype index sample:
HP:0000002 -> 7340
HP:0000003 -> 7341
HP:0000006 -> 7342
HP:0000007 -> 7343
HP:0000008 -> 7344


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch_geometric.nn import GCNConv

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load checkpoint and test split
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)
test_split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_split = test_split[test_split["split"] == "test"].copy()

# Reconstruct the checkpoint's GCN architecture
class RareDiseaseGCN(torch.nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.embedding = torch.nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, edge_index):
        x = self.embedding.weight
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

model = RareDiseaseGCN(ckpt["num_nodes"], ckpt["hidden_dim"])
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model = model.to(DEVICE).eval()

# Rebuild the disease-phenotype training graph from verified mappings
ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)
ann = ann.drop_duplicates()

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

pairs = []
for r in ann.itertuples(index=False):
    d = disease_to_idx.get(r.disease_id)
    h = phenotype_to_idx.get(r.hpo_id)
    if d is not None and h is not None:
        pairs.append((d, h))
        pairs.append((h, d))

edge_index = torch.tensor(pairs, dtype=torch.long).T.contiguous().to(DEVICE)

# Compute embeddings
with torch.no_grad():
    embeddings = model(edge_index).cpu().numpy()

disease_ids = ckpt["disease_ids"]
disease_vecs = embeddings[
    [disease_to_idx[d] for d in disease_ids]
]
phenotype_ids = ckpt["phenotype_ids"]
phenotype_vecs = embeddings[
    [phenotype_to_idx[h] for h in phenotype_ids]
]
hpo_to_row = {h: i for i, h in enumerate(phenotype_ids)}

# Index patient JSONs
json_files = {p.stem: p for p in PACKETS.rglob("*.json")}

results = []
for row in test_split.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        continue

    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    present, excluded = set(), set()
    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        if feature.get("excluded") is True:
            excluded.add(hpo)
        else:
            present.add(hpo)
    present -= excluded

    rows = [hpo_to_row[h] for h in present if h in hpo_to_row]
    if not rows:
        continue

    # Mean cosine similarity between positive phenotype embeddings
    # and each candidate disease embedding.
    pvecs = phenotype_vecs[rows]
    pvecs = pvecs / np.maximum(
        np.linalg.norm(pvecs, axis=1, keepdims=True), 1e-12
    )
    dvecs = disease_vecs / np.maximum(
        np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
    )
    scores = (pvecs @ dvecs.T).mean(axis=0)
    order = np.argsort(-scores)

    candidates = [disease_ids[i] for i in order]
    target_ids = set(str(row.diagnosis_ids).split(";"))

    ranks = [
        i + 1 for i, d in enumerate(candidates)
        if d in target_ids
    ]
    results.append({
        "case_id": row.case_id,
        "diagnosis_ids": row.diagnosis_ids,
        "n_positive_hpo": len(present),
        "n_recognized_hpo": len(rows),
        "target_in_candidate_set": bool(ranks),
        "rank": min(ranks) if ranks else np.nan,
        "top1": bool(ranks and min(ranks) <= 1),
        "top5": bool(ranks and min(ranks) <= 5),
        "top10": bool(ranks and min(ranks) <= 10),
        "mrr": 1 / min(ranks) if ranks else 0.0,
        "top_prediction": candidates[0],
    })

gnn_results = pd.DataFrame(results)
out_path = DATA / "patient_gnn_training_disease_rankings.csv"
gnn_results.to_csv(out_path, index=False)

print("Device:", DEVICE)
print("Cases ranked:", len(gnn_results))
print("Candidate diseases:", len(disease_ids))
print("Target coverage:", round(gnn_results["target_in_candidate_set"].mean(), 4))
print("Top-1:", round(gnn_results["top1"].mean(), 4))
print("Top-5:", round(gnn_results["top5"].mean(), 4))
print("Top-10:", round(gnn_results["top10"].mean(), 4))
print("MRR:", round(gnn_results["mrr"].mean(), 4))
print("Saved:", out_path)

AttributeError: 'Pandas' object has no attribute 'disease_id'

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(DATA / "gnn_training_annotations.csv", nrows=5)

print("Columns:", ann.columns.tolist())
print("\nSample rows:")
print(ann.to_string(index=False))

Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']

Sample rows:
database_id                                  disease_name  qualifier     hpo_id     reference evidence  onset frequency  sex  modifier aspect               biocuration
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0011097 PMID:31675180      PCS    NaN       1/2  NaN       NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0002187 PMID:31675180      PCS    NaN       1/1  NaN       NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0001518 PMID:31675180      PCS    NaN       1/2  NaN       NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0032792 PMID:31675180      PCS    NaN       1/2  NaN       NaN      P HPO:probinson[202

In [ ]:
import pandas as pd
import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)

# Keep explicitly negated annotations out of positive graph edges.
if "qualifier" in ann.columns:
    ann = ann[
        ann["qualifier"].isna()
        | ~ann["qualifier"].str.contains(
            "NOT", case=False, na=False
        )
    ]

ann = ann.drop_duplicates(subset=["database_id", "hpo_id"])

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

pairs = []
unmapped_diseases = set()
unmapped_phenotypes = set()

for r in ann.itertuples(index=False):
    d = disease_to_idx.get(r.database_id)
    h = phenotype_to_idx.get(r.hpo_id)

    if d is None:
        unmapped_diseases.add(r.database_id)
    if h is None:
        unmapped_phenotypes.add(r.hpo_id)

    if d is not None and h is not None:
        pairs.append((d, h))
        pairs.append((h, d))

edge_index = torch.tensor(
    pairs, dtype=torch.long
).T.contiguous().to(DEVICE)

print("Annotation pairs after filtering:", len(ann))
print("Mapped bidirectional edges:", edge_index.shape[1])
print("Unmapped disease IDs:", len(unmapped_diseases))
print("Unmapped HPO IDs:", len(unmapped_phenotypes))
print("Edge index shape:", tuple(edge_index.shape))
print("Device:", DEVICE)

Annotation pairs after filtering: 179287
Mapped bidirectional edges: 358574
Unmapped disease IDs: 0
Unmapped HPO IDs: 0
Edge index shape: (2, 358574)
Device: cuda


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

original = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)
original_pairs = original[["database_id", "hpo_id"]].drop_duplicates()

print("Original unique disease-HPO pairs:", len(original_pairs))
print("Rows with qualifier values:")
print(original["qualifier"].value_counts(dropna=False).to_string())

print("\nPairs with a non-empty qualifier:")
print(
    original.loc[
        original["qualifier"].notna(),
        ["database_id", "hpo_id", "qualifier"]
    ].drop_duplicates().head(20).to_string(index=False)
)

print("\nCheckpoint association count:", 179653)

Original unique disease-HPO pairs: 179653
Rows with qualifier values:
qualifier
NaN    179287
NOT       366

Pairs with a non-empty qualifier:
 database_id     hpo_id qualifier
  ORPHA:1401 HP:0001270       NOT
  ORPHA:1401 HP:0200041       NOT
ORPHA:240094 HP:0002063       NOT
ORPHA:240094 HP:0002067       NOT
 ORPHA:79408 HP:0000982       NOT
 ORPHA:79408 HP:0002671       NOT
 ORPHA:79408 HP:0006297       NOT
 ORPHA:79408 HP:0012252       NOT
 ORPHA:79409 HP:0000478       NOT
 ORPHA:79409 HP:0000982       NOT
 ORPHA:79409 HP:0001965       NOT
 ORPHA:79409 HP:0002671       NOT
 ORPHA:79409 HP:0002860       NOT
 ORPHA:79409 HP:0006297       NOT
 ORPHA:79409 HP:0012056       NOT
 ORPHA:79409 HP:0012252       NOT
 ORPHA:79411 HP:0000079       NOT
 ORPHA:79411 HP:0000478       NOT
 ORPHA:79411 HP:0000982       NOT
 ORPHA:79411 HP:0001510       NOT

Checkpoint association count: 179653


In [ ]:
import pandas as pd
import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)
ann = ann.drop_duplicates(subset=["database_id", "hpo_id"])

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

pairs = []
unmapped = 0

for r in ann.itertuples(index=False):
    d = disease_to_idx.get(r.database_id)
    h = phenotype_to_idx.get(r.hpo_id)

    if d is None or h is None:
        unmapped += 1
        continue

    pairs.append((d, h))
    pairs.append((h, d))

edge_index = torch.tensor(
    pairs, dtype=torch.long
).T.contiguous().to(DEVICE)

print("Unique annotation pairs:", len(ann))
print("Expected checkpoint associations:", ckpt["training_association_count"])
print("Reconstructed associations:", edge_index.shape[1] // 2)
print("Directed edges:", edge_index.shape[1])
print("Unmapped pairs:", unmapped)
print("Edge index shape:", tuple(edge_index.shape))

Unique annotation pairs: 179653
Expected checkpoint associations: 179653
Reconstructed associations: 179653
Directed edges: 359306
Unmapped pairs: 0
Edge index shape: (2, 359306)


In [ ]:
# Use the verified graph reconstructed in Step 23D.
# Do not filter out NOT-qualified associations for this checkpoint reproduction.
assert edge_index.shape == (2, 359306)

model = RareDiseaseGCN(
    ckpt["num_nodes"],
    ckpt["hidden_dim"]
)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model = model.to(DEVICE).eval()

with torch.no_grad():
    embeddings = model(edge_index).cpu().numpy()

print("Embedding shape:", embeddings.shape)
print("All embeddings finite:", bool(__import__("numpy").isfinite(embeddings).all()))

Embedding shape: (17322, 64)
All embeddings finite: True


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

# Verify the expected in-memory variables
assert embeddings.shape == (ckpt["num_nodes"], ckpt["hidden_dim"])

disease_ids = ckpt["disease_ids"]
phenotype_ids = ckpt["phenotype_ids"]
disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

disease_vecs = embeddings[[disease_to_idx[d] for d in disease_ids]]
phenotype_vecs = embeddings[[phenotype_to_idx[h] for h in phenotype_ids]]

def normalize_rows(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

disease_vecs = normalize_rows(disease_vecs)
hpo_to_row = {h: i for i, h in enumerate(phenotype_ids)}

test_split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_split = test_split[test_split["split"] == "test"].copy()
json_files = {p.stem: p for p in PACKETS.rglob("*.json")}

results = []

for row in test_split.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        continue

    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()

    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        if feature.get("excluded") is True:
            excluded.add(hpo)
        else:
            positive.add(hpo)

    positive -= excluded
    recognized = [hpo_to_row[h] for h in positive if h in hpo_to_row]

    target_ids = set(str(row.diagnosis_ids).split(";"))
    candidate_indices = [
        i for i, disease_id in enumerate(disease_ids)
        if disease_id in target_ids
    ]

    if not recognized:
        rank = np.nan
        top_prediction = None
    else:
        patient_vecs = normalize_rows(phenotype_vecs[recognized])
        scores = (patient_vecs @ disease_vecs.T).mean(axis=0)
        order = np.argsort(-scores)

        rank_positions = [
            int(np.where(order == i)[0][0]) + 1
            for i in candidate_indices
        ]
        rank = min(rank_positions) if rank_positions else np.nan
        top_prediction = disease_ids[int(order[0])]

    results.append({
        "case_id": row.case_id,
        "diagnosis_ids": row.diagnosis_ids,
        "n_positive_hpo": len(positive),
        "n_recognized_hpo": len(recognized),
        "target_in_candidate_set": bool(candidate_indices),
        "rank": rank,
        "top1": bool(pd.notna(rank) and rank <= 1),
        "top5": bool(pd.notna(rank) and rank <= 5),
        "top10": bool(pd.notna(rank) and rank <= 10),
        "mrr": 1.0 / rank if pd.notna(rank) else 0.0,
        "top_prediction": top_prediction,
    })

gnn_results = pd.DataFrame(results)
out_path = DATA / "patient_gnn_training_disease_rankings.csv"
gnn_results.to_csv(out_path, index=False)

print("Cases evaluated:", len(gnn_results))
print("Candidate diseases:", len(disease_ids))
print("Target candidate coverage:",
      round(gnn_results["target_in_candidate_set"].mean(), 4))
print("Top-1:", round(gnn_results["top1"].mean(), 4))
print("Top-5:", round(gnn_results["top5"].mean(), 4))
print("Top-10:", round(gnn_results["top10"].mean(), 4))
print("MRR:", round(gnn_results["mrr"].mean(), 4))
print("Saved:", out_path)

Cases evaluated: 1959
Candidate diseases: 7340
Target candidate coverage: 0.7616
Top-1: 0.0316
Top-5: 0.0572
Top-10: 0.0781
MRR: 0.0476
Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_gnn_training_disease_rankings.csv


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)
ann = ann.drop_duplicates(["database_id", "hpo_id"])

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

# Reuse the embeddings already computed.
E = embeddings
rng = np.random.default_rng(42)

positive_scores = []
negative_scores = []

# Sample positive pairs and randomly corrupted pairs.
sample = ann.sample(n=min(20000, len(ann)), random_state=42)

for r in sample.itertuples(index=False):
    d = disease_to_idx.get(r.database_id)
    h = phenotype_to_idx.get(r.hpo_id)
    if d is None or h is None:
        continue

    dv = E[d]
    hv = E[h]
    positive_scores.append(
        np.dot(dv, hv) /
        max(np.linalg.norm(dv) * np.linalg.norm(hv), 1e-12)
    )

    # Random disease–phenotype pair; may occasionally be a true pair.
    random_d = int(rng.integers(0, len(ckpt["disease_ids"])))
    random_d_id = ckpt["disease_ids"][random_d]

    if random_d_id == r.database_id:
        random_d = (random_d + 1) % len(ckpt["disease_ids"])

    ndv = E[random_d]
    negative_scores.append(
        np.dot(ndv, hv) /
        max(np.linalg.norm(ndv) * np.linalg.norm(hv), 1e-12)
    )

print("Positive pairs:", len(positive_scores))
print("Random comparison pairs:", len(negative_scores))
print("Mean positive cosine:", round(float(np.mean(positive_scores)), 4))
print("Mean random cosine:", round(float(np.mean(negative_scores)), 4))
print("Median positive cosine:", round(float(np.median(positive_scores)), 4))
print("Median random cosine:", round(float(np.median(negative_scores)), 4))
print("Positive score > random score:",
      round(float(np.mean(np.array(positive_scores) >
                          np.array(negative_scores))), 4))

Positive pairs: 20000
Random comparison pairs: 20000
Mean positive cosine: 0.7935
Mean random cosine: 0.5904
Median positive cosine: 0.9026
Median random cosine: 0.753
Positive score > random score: 0.7345


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

baseline_path = DATA / "patient_phenotype_only_baseline.csv"
gnn_path = DATA / "patient_gnn_training_disease_rankings.csv"

baseline = pd.read_csv(baseline_path)
gnn = pd.read_csv(gnn_path)

print("Baseline columns:", baseline.columns.tolist())
print("GNN columns:", gnn.columns.tolist())

print("\nBaseline sample:")
print(baseline.head(3).to_string(index=False))

print("\nGNN sample:")
print(gnn.head(3).to_string(index=False))

if "case_id" in baseline.columns and "case_id" in gnn.columns:
    print("\nUnique baseline case IDs:", baseline["case_id"].nunique())
    print("Unique GNN case IDs:", gnn["case_id"].nunique())
    print("Overlapping case IDs:",
          baseline["case_id"].isin(gnn["case_id"]).sum())

Baseline columns: ['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']
GNN columns: ['case_id', 'diagnosis_ids', 'n_positive_hpo', 'n_recognized_hpo', 'target_in_candidate_set', 'rank', 'top1', 'top5', 'top10', 'mrr', 'top_prediction']

Baseline sample:
                       case_id     pmid diagnosis_ids  rank  top1  top5  top10  candidate_covered
PMID_28239884_Family_1_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_2_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_3_proband 28239884   OMIM:148600   3.0     0     1      1                  1

GNN sample:
                       case_id diagnosis_ids  n_positive_hpo  n_recognized_hpo  target_in_candidate_set  rank  top1  top5  top10  mrr top_prediction
PMID_28239884_Family_1_proband   OMIM:148600               1                 1                     True   1.0  True  True   True  1.0    OMIM:148600
PMID_

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")
gnn = pd.read_csv(DATA / "patient_gnn_training_disease_rankings.csv")

# Join by case ID and verify the recorded diagnosis agrees.
paired = baseline.merge(
    gnn,
    on="case_id",
    suffixes=("_baseline", "_gnn"),
    validate="one_to_one"
)

assert len(paired) == 1959
assert (
    paired["diagnosis_ids_baseline"].fillna("")
    == paired["diagnosis_ids_gnn"].fillna("")
).all()

def summarize(df, prefix):
    rank = pd.to_numeric(df[f"rank_{prefix}"], errors="coerce")
    return {
        "cases": len(df),
        "target_coverage": df[
            "candidate_covered" if prefix == "baseline"
            else "target_in_candidate_set"
        ].astype(bool).mean(),
        "Top-1": (rank <= 1).mean(),
        "Top-5": (rank <= 5).mean(),
        "Top-10": (rank <= 10).mean(),
        "MRR": (1 / rank).where(rank.notna(), 0).mean(),
    }

overall = pd.DataFrame({
    "Phenotype baseline": summarize(paired, "baseline"),
    "GNN embedding score": summarize(paired, "gnn"),
}).T

print("OVERALL RESULTS")
print(overall.round(4).to_string())

# Compare only cases where both methods contain the target diagnosis.
both_covered = paired[
    paired["candidate_covered"].astype(bool)
    & paired["target_in_candidate_set"].astype(bool)
].copy()

print("\nCASES COVERED BY BOTH METHODS:", len(both_covered))
conditional = pd.DataFrame({
    "Phenotype baseline": summarize(both_covered, "baseline"),
    "GNN embedding score": summarize(both_covered, "gnn"),
}).T
print(conditional.round(4).to_string())

# Paired win/tie/loss using reciprocal rank.
rb = pd.to_numeric(both_covered["rank_baseline"], errors="coerce")
rg = pd.to_numeric(both_covered["rank_gnn"], errors="coerce")
rr_baseline = (1 / rb).fillna(0)
rr_gnn = (1 / rg).fillna(0)

print("\nPAIRED RANK COMPARISON (both-covered cases)")
print("GNN better:", int((rr_gnn > rr_baseline).sum()))
print("Tie:", int((rr_gnn == rr_baseline).sum()))
print("Baseline better:", int((rr_gnn < rr_baseline).sum()))

out = DATA / "patient_baseline_vs_gnn_paired.csv"
paired.to_csv(out, index=False)
print("\nSaved paired results:", out)

OVERALL RESULTS
                      cases  target_coverage   Top-1   Top-5  Top-10     MRR
Phenotype baseline   1959.0           0.7917  0.1138  0.1960  0.2210  0.1527
GNN embedding score  1959.0           0.7616  0.0316  0.0572  0.0781  0.0476

CASES COVERED BY BOTH METHODS: 1181
                      cases  target_coverage   Top-1   Top-5  Top-10     MRR
Phenotype baseline   1181.0              1.0  0.1160  0.2235  0.2566  0.1659
GNN embedding score  1181.0              1.0  0.0212  0.0457  0.0754  0.0381

PAIRED RANK COMPARISON (both-covered cases)
GNN better: 528
Tie: 5
Baseline better: 648

Saved paired results: /content/drive/MyDrive/Rare Disease ai/processed/patient_baseline_vs_gnn_paired.csv


In [ ]:
import numpy as np
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# These variables should still exist from the embedding evaluation.
assert "disease_vecs" in globals()
assert "phenotype_vecs" in globals()

# Normalize disease vectors.
D = disease_vecs / np.maximum(
    np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
)

# Sample random pairs of distinct disease embeddings.
rng = np.random.default_rng(42)
n = min(50000, len(D) * 5)

i = rng.integers(0, len(D), size=n)
j = rng.integers(0, len(D), size=n)
keep = i != j

cosines = np.sum(D[i[keep]] * D[j[keep]], axis=1)

print("Disease embeddings:", D.shape)
print("Random distinct disease pairs:", len(cosines))
print("Mean cosine similarity:", round(float(cosines.mean()), 4))
print("Median cosine similarity:", round(float(np.median(cosines)), 4))
print("10th percentile:", round(float(np.percentile(cosines, 10)), 4))
print("90th percentile:", round(float(np.percentile(cosines, 90)), 4))

Disease embeddings: (7340, 64)
Random distinct disease pairs: 36695
Mean cosine similarity: 0.4671
Median cosine similarity: 0.5857
10th percentile: -0.1783
90th percentile: 0.9064


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Reuse the GNN ranking output and embeddings already in memory.
gnn = pd.read_csv(DATA / "patient_gnn_training_disease_rankings.csv")

D = disease_vecs / np.maximum(
    np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
)
P = phenotype_vecs / np.maximum(
    np.linalg.norm(phenotype_vecs, axis=1, keepdims=True), 1e-12
)

disease_to_col = {d: i for i, d in enumerate(ckpt["disease_ids"])}
hpo_to_row = {h: i for i, h in enumerate(ckpt["phenotype_ids"])}

json_files = {p.stem: p for p in PACKETS.rglob("*.json")}
diagnostic = []

for row in gnn.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        continue

    import json
    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()
    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        (excluded if feature.get("excluded") is True else positive).add(hpo)

    positive -= excluded
    ids = [hpo_to_row[h] for h in positive if h in hpo_to_row]
    if not ids:
        continue

    target_ids = set(str(row.diagnosis_ids).split(";"))
    target_cols = [disease_to_col[d] for d in target_ids if d in disease_to_col]
    if not target_cols:
        continue

    scores = (P[ids] @ D.T).mean(axis=0)
    target_score = max(scores[j] for j in target_cols)

    # Average score of all non-target candidates for this patient.
    mask = np.ones(len(scores), dtype=bool)
    mask[target_cols] = False
    other_scores = scores[mask]

    diagnostic.append({
        "case_id": row.case_id,
        "target_score": float(target_score),
        "mean_other_score": float(other_scores.mean()),
        "target_above_other_mean": bool(target_score > other_scores.mean()),
        "target_percentile": float((other_scores < target_score).mean()),
    })

diag = pd.DataFrame(diagnostic)

print("Cases with target in GNN candidate set:", len(diag))
print("Mean target score:", round(diag["target_score"].mean(), 4))
print("Mean non-target score:", round(diag["mean_other_score"].mean(), 4))
print("Target above non-target mean:",
      round(diag["target_above_other_mean"].mean(), 4))
print("Mean target percentile:",
      round(diag["target_percentile"].mean(), 4)).


SyntaxError: invalid syntax (597861943.py, line 73)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Reuse the GNN ranking output and embeddings already in memory.
gnn = pd.read_csv(DATA / "patient_gnn_training_disease_rankings.csv")

D = disease_vecs / np.maximum(
    np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
)
P = phenotype_vecs / np.maximum(
    np.linalg.norm(phenotype_vecs, axis=1, keepdims=True), 1e-12
)

disease_to_col = {d: i for i, d in enumerate(ckpt["disease_ids"])}
hpo_to_row = {h: i for i, h in enumerate(ckpt["phenotype_ids"])}

json_files = {p.stem: p for p in PACKETS.rglob("*.json")}
diagnostic = []

for row in gnn.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        continue

    import json
    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()
    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        (excluded if feature.get("excluded") is True else positive).add(hpo)

    positive -= excluded
    ids = [hpo_to_row[h] for h in positive if h in hpo_to_row]
    if not ids:
        continue

    target_ids = set(str(row.diagnosis_ids).split(";"))
    target_cols = [disease_to_col[d] for d in target_ids if d in disease_to_col]
    if not target_cols:
        continue

    scores = (P[ids] @ D.T).mean(axis=0)
    target_score = max(scores[j] for j in target_cols)

    # Average score of all non-target candidates for this patient.
    mask = np.ones(len(scores), dtype=bool)
    mask[target_cols] = False
    other_scores = scores[mask]

    diagnostic.append({
        "case_id": row.case_id,
        "target_score": float(target_score),
        "mean_other_score": float(other_scores.mean()),
        "target_above_other_mean": bool(target_score > other_scores.mean()),
        "target_percentile": float((other_scores < target_score).mean()),
    })

diag = pd.DataFrame(diagnostic)

print("Cases with target in GNN candidate set:", len(diag))
print("Mean target score:", round(diag["target_score"].mean(), 4))
print("Mean non-target score:", round(diag["mean_other_score"].mean(), 4))
print("Target above non-target mean:",
      round(diag["target_above_other_mean"].mean(), 4))
print("Mean target percentile:",
      round(diag["target_percentile"].mean(), 4))

Cases with target in GNN candidate set: 1492
Mean target score: 0.8527
Mean non-target score: 0.6097
Target above non-target mean: 0.9672
Mean target percentile: 0.8216


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
PACKETS = Path("/content/drive/MyDrive/Rare Disease ai/all_phenopackets")

gnn = pd.read_csv(DATA / "patient_gnn_training_disease_rankings.csv")
json_files = {p.stem: p for p in PACKETS.rglob("*.json")}

D = disease_vecs / np.maximum(
    np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
)
P = phenotype_vecs / np.maximum(
    np.linalg.norm(phenotype_vecs, axis=1, keepdims=True), 1e-12
)
disease_to_col = {d: i for i, d in enumerate(ckpt["disease_ids"])}
hpo_to_row = {h: i for i, h in enumerate(ckpt["phenotype_ids"])}

hard_cases = []

for row in gnn.itertuples(index=False):
    if pd.isna(row.rank) or row.rank <= 10:
        continue

    path = json_files.get(row.case_id)
    if path is None:
        continue

    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()
    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        (excluded if feature.get("excluded") is True else positive).add(hpo)

    positive -= excluded
    ids = [hpo_to_row[h] for h in positive if h in hpo_to_row]
    targets = [
        disease_to_col[d]
        for d in set(str(row.diagnosis_ids).split(";"))
        if d in disease_to_col
    ]

    if not ids or not targets:
        continue

    scores = (P[ids] @ D.T).mean(axis=0)
    order = np.argsort(-scores)

    target_score = max(scores[t] for t in targets)
    top_alternatives = [
        (ckpt["disease_ids"][i], round(float(scores[i]), 4))
        for i in order if i not in targets
    ][:5]

    hard_cases.append({
        "case_id": row.case_id,
        "diagnosis": row.diagnosis_ids,
        "target_rank": int(row.rank),
        "target_score": round(float(target_score), 4),
        "top_alternatives": top_alternatives,
        "positive_hpo_count": len(positive),
    })

hard_df = pd.DataFrame(hard_cases)
print("Cases with target rank > 10 and target represented:", len(hard_df))
print(hard_df.head(10).to_string(index=False))

Cases with target rank > 10 and target represented: 1339
                          case_id   diagnosis  target_rank  target_score                                                                                                      top_alternatives  positive_hpo_count
             PMID_1680289_patient OMIM:102700         3278        0.7151      [(ORPHA:394, 0.8285), (OMIM:618885, 0.8275), (OMIM:122470, 0.8274), (ORPHA:79282, 0.8273), (ORPHA:3253, 0.8245)]                   6
          PMID_21989597_Proband_2 OMIM:621196         1105        0.6168     [(OMIM:621080, 0.7609), (OMIM:619221, 0.7403), (ORPHA:99845, 0.7387), (OMIM:306955, 0.7316), (ORPHA:2248, 0.725)]                   2
          PMID_21989597_Proband_1 OMIM:621196         1105        0.6168     [(OMIM:621080, 0.7609), (OMIM:619221, 0.7403), (ORPHA:99845, 0.7387), (OMIM:306955, 0.7316), (ORPHA:2248, 0.725)]                   2
PMID_37951597_Family_22_Subject_1 OMIM:620785           23        0.9908  [(ORPHA:352665, 0.9938), 

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

for filename in [
    "patient_phenotype_only_baseline.csv",
    "patient_publication_split.csv",
    "patient_test_disease_coverage.csv",
]:
    path = DATA / filename
    print(f"\n--- {filename} ---")
    if path.exists():
        df = pd.read_csv(path, nrows=5)
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))
    else:
        print("File not found")

print("\nGNN result rows:", len(pd.read_csv(
    DATA / "patient_gnn_training_disease_rankings.csv"
)))



--- patient_phenotype_only_baseline.csv ---
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']
                       case_id     pmid diagnosis_ids  rank  top1  top5  top10  candidate_covered
PMID_28239884_Family_1_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_2_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_3_proband 28239884   OMIM:148600   3.0     0     1      1                  1

--- patient_publication_split.csv ---
Columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
                    case_id     pmid       split diagnosis_ids
PMID_19836009_Family_E_II_6 19836009 development   OMIM:277600
PMID_19836009_Family_E_II_3 19836009 development   OMIM:277600
PMID_19836009_Family_B_II_4 19836009 development   OMIM:277600

--- patient_test_disease_coverage.csv ---
Columns: ['test_cases', 'cases_with_diagnosis_profile', 'cases_without_dia

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
ann = pd.read_csv(DATA / "patient_test_leakage_reduced_annotations.csv", dtype=str)
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

development = split[split["split"] == "development"].copy()
gnn_diseases = set(ckpt["disease_ids"])
profile_diseases = set(ann["database_id"].dropna().unique())

development["target_ids"] = development["diagnosis_ids"].fillna("").str.split(";")
development["gnn_covered"] = development["target_ids"].apply(
    lambda ids: any(d in gnn_diseases for d in ids)
)
development["profile_covered"] = development["target_ids"].apply(
    lambda ids: any(d in profile_diseases for d in ids)
)

print("Development cases:", len(development))
print("Unique development publications:", development["pmid"].nunique())
print("GNN candidate coverage:", round(development["gnn_covered"].mean(), 4))
print("Phenotype-profile coverage:", round(development["profile_covered"].mean(), 4))
print("Cases covered by both:", int(
    (development["gnn_covered"] & development["profile_covered"]).sum()
))
print("Cases covered by neither:", int(
    (~development["gnn_covered"] & ~development["profile_covered"]).sum()
))


Development cases: 8386
Unique development publications: 1383
GNN candidate coverage: 0.7913
Phenotype-profile coverage: 0.9982
Cases covered by both: 6621
Cases covered by neither: 0


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)

dev_pmids = set(
    split.loc[split["split"] == "development", "pmid"]
    .dropna().astype(str)
)
ann_pmids = set(
    ann["reference"].dropna().astype(str)
    .str.extract(r"PMID:(\d+)")[0].dropna()
)

overlap = dev_pmids & ann_pmids

print("Development publications:", len(dev_pmids))
print("Distinct annotation publication references:", len(ann_pmids))
print("Overlapping publication IDs:", len(overlap))
print("Development publications overlapping annotations:",
      round(len(overlap) / max(len(dev_pmids), 1), 4))
print("Example overlapping IDs:", sorted(overlap)[:15])

Development publications: 1383
Distinct annotation publication references: 7149
Overlapping publication IDs: 806
Development publications overlapping annotations: 0.5828
Example overlapping IDs: ['10077612', '10487826', '10498624', '10560675', '10842287', '10939567', '11050011', '11093277', '11179005', '11181649', '11344241', '11410597', '11479728', '11506412', '11555793']


In [ ]:

from pathlib import Path
import torch
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=" * 55)
print("RARE DISEASE PREDICTION AI — RUNTIME CHECK")
print("=" * 55)

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch:", torch.__version__)
    print("CUDA version:", torch.version.cuda)
    print(
        "GPU memory (allocated/reserved):",
        round(torch.cuda.memory_allocated() / 1024**3, 2), "GB /",
        round(torch.cuda.memory_reserved() / 1024**3, 2), "GB"
    )
else:
    print("No GPU detected. Do not start GPU-intensive work yet.")
    print("In Colab, select Runtime > Change runtime type > T4 GPU if available.")

print("\nProject folder exists:", PROJECT.exists())

important_files = [
    "gnn_training_annotations.csv",
    "patient_publication_split.csv",
    "patient_test_leakage_reduced_annotations.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_gnn_training_disease_rankings.csv",
    "patient_baseline_vs_gnn_paired.csv",
    "rare_disease_gnn_training_only.pt",
    "rare_disease_training_graph.graphml",
]

print("\nSaved project files:")
for filename in important_files:
    path = DATA / filename
    if path.exists():
        print(f"[OK] {filename} ({path.stat().st_size / 1024**2:.2f} MB)")
    else:
        print(f"[MISSING] {filename}")

print("\nGPU runtime check complete.")


Mounted at /content/drive
RARE DISEASE PREDICTION AI — RUNTIME CHECK

GPU available: True
GPU: Tesla T4
PyTorch: 2.11.0+cu130
CUDA version: 13.0
GPU memory (allocated/reserved): 0.0 GB / 0.0 GB

Project folder exists: True

Saved project files:
[OK] gnn_training_annotations.csv (21.95 MB)
[OK] patient_publication_split.csv (0.59 MB)
[OK] patient_test_leakage_reduced_annotations.csv (33.52 MB)
[OK] patient_phenotype_only_baseline.csv (0.12 MB)
[OK] patient_gnn_training_disease_rankings.csv (0.19 MB)
[OK] patient_baseline_vs_gnn_paired.csv (0.25 MB)
[OK] rare_disease_gnn_training_only.pt (4.73 MB)
[OK] rare_disease_training_graph.graphml (20.36 MB)

GPU runtime check complete.


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)

dev_pmids = set(
    split.loc[split["split"] == "development", "pmid"].dropna()
)
test_pmids = set(
    split.loc[split["split"] == "test", "pmid"].dropna()
)

# Extract PMID references where available.
ref_pmid = ann["reference"].fillna("").str.extract(
    r"PMID:(\d+)"
)[0]

has_pmid = ref_pmid.notna()
dev_overlap = has_pmid & ref_pmid.isin(dev_pmids)
test_overlap = has_pmid & ref_pmid.isin(test_pmids)
either_overlap = dev_overlap | test_overlap

print("Original annotation rows:", len(ann))
print("Unique disease-HPO pairs:",
      ann[["database_id", "hpo_id"]].drop_duplicates().shape[0])
print("Rows with identifiable PMID:", int(has_pmid.sum()))
print("Rows with missing/unmatched PMID:", int((~has_pmid).sum()))
print("Rows citing development publications:", int(dev_overlap.sum()))
print("Rows citing test publications:", int(test_overlap.sum()))
print("Rows citing either split:", int(either_overlap.sum()))
print("Rows remaining after identifiable-reference filter:",
      int((~either_overlap).sum()))
print("Remaining unique disease-HPO pairs:",
      ann.loc[~either_overlap, ["database_id", "hpo_id"]]
      .drop_duplicates().shape[0])
print("Negated NOT rows:", int(
    ann["qualifier"].fillna("").str.upper().eq("NOT").sum()
))


Original annotation rows: 179653
Unique disease-HPO pairs: 179653
Rows with identifiable PMID: 90758
Rows with missing/unmatched PMID: 88895
Rows citing development publications: 16448
Rows citing test publications: 4180
Rows citing either split: 20628
Rows remaining after identifiable-reference filter: 159025
Remaining unique disease-HPO pairs: 159025
Negated NOT rows: 366


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)

dev_pmids = set(
    split.loc[split["split"] == "development", "pmid"].dropna()
)
test_pmids = set(
    split.loc[split["split"] == "test", "pmid"].dropna()
)

ann["reference_pmid"] = (
    ann["reference"].fillna("").str.extract(r"PMID:(\d+)")[0]
)

identified = ann["reference_pmid"].notna()
overlap = identified & (
    ann["reference_pmid"].isin(dev_pmids)
    | ann["reference_pmid"].isin(test_pmids)
)

filtered = ann.loc[~overlap].copy()

out = DATA / "gnn_annotations_excluding_dev_test_pmids.csv"
filtered.to_csv(out, index=False)

print("Saved:", out)
print("Rows saved:", len(filtered))
print("Unique disease-HPO pairs:",
      filtered[["database_id", "hpo_id"]].drop_duplicates().shape[0])
print("Missing/unmatched PMID rows retained:",
      int(filtered["reference_pmid"].isna().sum()))
print("NOT annotations retained:",
      int(filtered["qualifier"].fillna("").str.upper().eq("NOT").sum()))


Saved: /content/drive/MyDrive/Rare Disease ai/processed/gnn_annotations_excluding_dev_test_pmids.csv
Rows saved: 159025
Unique disease-HPO pairs: 159025
Missing/unmatched PMID rows retained: 88895
NOT annotations retained: 366


In [ ]:

import pandas as pd
import torch
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

filtered = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    dtype=str
)

ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

original_diseases = set(ckpt["disease_ids"])
original_hpos = set(ckpt["phenotype_ids"])

filtered_diseases = set(filtered["database_id"].dropna())
filtered_hpos = set(filtered["hpo_id"].dropna())

print("Original checkpoint diseases:", len(original_diseases))
print("Filtered diseases:", len(filtered_diseases))
print("Checkpoint diseases still represented:",
      len(original_diseases & filtered_diseases))
print("Checkpoint disease coverage:",
      round(len(original_diseases & filtered_diseases)
            / len(original_diseases), 4))

print("\nOriginal checkpoint HPO terms:", len(original_hpos))
print("Filtered HPO terms:", len(filtered_hpos))
print("Checkpoint HPO terms still represented:",
      len(original_hpos & filtered_hpos))
print("Checkpoint HPO coverage:",
      round(len(original_hpos & filtered_hpos)
            / len(original_hpos), 4))

print("\nFiltered unique pairs:",
      filtered[["database_id", "hpo_id"]].drop_duplicates().shape[0])
print("Negated pairs:",
      int(filtered["qualifier"].fillna("").str.upper().eq("NOT").sum()))
print("GPU available:", torch.cuda.is_available())


Original checkpoint diseases: 7340
Filtered diseases: 6983
Checkpoint diseases still represented: 6983
Checkpoint disease coverage: 0.9514

Original checkpoint HPO terms: 9982
Filtered HPO terms: 9617
Checkpoint HPO terms still represented: 9617
Checkpoint HPO coverage: 0.9634

Filtered unique pairs: 159025
Negated pairs: 366
GPU available: True


In [ ]:

import pandas as pd
import torch
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

filtered = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    dtype=str
)
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

train_ids = set(ckpt["disease_ids"])
all_train_split = set(
    split.loc[split["split"] == "train", "disease_id"].dropna()
)
test_ids = set(
    split.loc[split["split"] == "test", "disease_id"].dropna()
)

filtered_diseases = set(filtered["database_id"].dropna())
positive = filtered[
    filtered["qualifier"].fillna("").str.upper() != "NOT"
]
positive_diseases = set(positive["database_id"].dropna())

print("Checkpoint training diseases:", len(train_ids))
print("Training diseases with any filtered association:",
      len(train_ids & filtered_diseases))
print("Training diseases with positive associations:",
      len(train_ids & positive_diseases))
print("Training diseases with no positive associations:",
      len(train_ids - positive_diseases))
print("Held-out test diseases appearing in filtered annotations:",
      len(test_ids & filtered_diseases))
print("Filtered associations involving held-out test diseases:",
      int(filtered["database_id"].isin(test_ids).sum()))
print("Filtered NOT rows:",
      int(filtered["qualifier"].fillna("").str.upper().eq("NOT").sum()))
print("GPU available:", torch.cuda.is_available())


Checkpoint training diseases: 7340
Training diseases with any filtered association: 6983
Training diseases with positive associations: 6983
Training diseases with no positive associations: 357
Held-out test diseases appearing in filtered annotations: 0
Filtered associations involving held-out test diseases: 0
Filtered NOT rows: 366
GPU available: True


In [ ]:

import pandas as pd
import torch
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    dtype=str
)
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

# Keep positive disease-phenotype associations only.
positive = ann[
    ann["qualifier"].fillna("").str.upper() != "NOT"
].drop_duplicates(["database_id", "hpo_id"])

train_diseases = list(ckpt["disease_ids"])
train_phenotypes = list(ckpt["phenotype_ids"])

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

# Retain only associations between the original training diseases
# and the checkpoint's phenotype vocabulary.
positive = positive[
    positive["database_id"].isin(disease_to_idx)
    & positive["hpo_id"].isin(phenotype_to_idx)
]

edges = []
for row in positive.itertuples(index=False):
    d = disease_to_idx[row.database_id]
    h = phenotype_to_idx[row.hpo_id]
    edges.extend([(d, h), (h, d)])

edge_index_filtered = torch.tensor(
    edges, dtype=torch.long
).T.contiguous()

out = DATA / "rare_disease_gnn_filtered_edge_index.pt"
torch.save({
    "edge_index": edge_index_filtered,
    "disease_ids": train_diseases,
    "phenotype_ids": train_phenotypes,
    "num_nodes": ckpt["num_nodes"],
    "hidden_dim": ckpt["hidden_dim"],
    "positive_association_count": len(positive),
    "excluded_development_and_test_publications": True,
    "note": (
        "Identifiable development/test PMID references were filtered; "
        "annotations with missing or unmatched PMIDs may remain."
    ),
}, out)

print("Positive associations:", len(positive))
print("Directed edges:", edge_index_filtered.shape[1])
print("Disease nodes preserved:", len(train_diseases))
print("Phenotype nodes preserved:", len(train_phenotypes))
print("Saved filtered graph:", out)


Positive associations: 158659
Directed edges: 317318
Disease nodes preserved: 7340
Phenotype nodes preserved: 9982
Saved filtered graph: /content/drive/MyDrive/Rare Disease ai/processed/rare_disease_gnn_filtered_edge_index.pt


In [ ]:

from pathlib import Path
import json
import torch
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

graph_path = DATA / "rare_disease_gnn_filtered_edge_index.pt"
annotation_path = DATA / "gnn_annotations_excluding_dev_test_pmids.csv"

graph = torch.load(graph_path, map_location="cpu", weights_only=True)
ann = pd.read_csv(annotation_path, dtype=str)
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

edge_index = graph["edge_index"]
disease_ids = graph["disease_ids"]
phenotype_ids = graph["phenotype_ids"]
num_nodes = graph["num_nodes"]

# 1. Validate graph structure and mappings.
assert edge_index.ndim == 2 and edge_index.shape[0] == 2
assert edge_index.shape[1] == 2 * graph["positive_association_count"]
assert int(edge_index.min()) >= 0
assert int(edge_index.max()) < num_nodes
assert len(disease_ids) + len(phenotype_ids) == num_nodes
assert len(set(disease_ids)) == len(disease_ids)
assert len(set(phenotype_ids)) == len(phenotype_ids)

print("GRAPH VALIDATION")
print("Positive associations:", graph["positive_association_count"])
print("Directed edges:", edge_index.shape[1])
print("Node count:", num_nodes)
print("Edge indices valid: YES")

# 2. Check disease support after filtering.
positive = ann[
    ann["qualifier"].fillna("").str.upper() != "NOT"
].drop_duplicates(["database_id", "hpo_id"])

positive_diseases = set(positive["database_id"].dropna())
unsupported = sorted(set(disease_ids) - positive_diseases)

print("\nDISEASE SUPPORT")
print("Diseases in graph:", len(disease_ids))
print("Diseases with positive associations:",
      len(set(disease_ids) & positive_diseases))
print("Diseases without positive associations:", len(unsupported))

# 3. Save a separate reproducible training configuration.
config = {
    "model": "RareDiseaseGCN",
    "num_nodes": num_nodes,
    "hidden_dim": int(graph["hidden_dim"]),
    "positive_associations": int(graph["positive_association_count"]),
    "directed_edges": int(edge_index.shape[1]),
    "seed": 42,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "max_epochs": 100,
    "early_stopping_patience": 10,
    "graph_file": graph_path.name,
    "annotation_file": annotation_path.name,
    "publication_filter": (
        "Exclude identifiable development/test PMID references; "
        "missing or unmatched references remain."
    ),
    "negated_associations": "Excluded from positive graph edges",
    "original_checkpoint_preserved": True,
    "evaluation_note": (
        "Do not claim fully leakage-free evaluation because some "
        "annotation references are missing or unmatched."
    )
}

config_path = MODEL_DIR / "filtered_gnn_training_config.json"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

with config_path.open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("\nTRAINING CONFIG")
print("Saved:", config_path)
print("Maximum epochs:", config["max_epochs"])
print("Early stopping patience:", config["early_stopping_patience"])
print("Original checkpoint unchanged: YES")


GRAPH VALIDATION
Positive associations: 158659
Directed edges: 317318
Node count: 17322
Edge indices valid: YES

DISEASE SUPPORT
Diseases in graph: 7340
Diseases with positive associations: 6983
Diseases without positive associations: 357

TRAINING CONFIG
Saved: /content/drive/MyDrive/Rare Disease ai/model/filtered_gnn_training_config.json
Maximum epochs: 100
Early stopping patience: 10
Original checkpoint unchanged: YES


In [ ]:

import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

graph = torch.load(
    DATA / "rare_disease_gnn_filtered_edge_index.pt",
    map_location="cpu",
    weights_only=True
)

# Inspect the stored structure before converting anything.
print("Graph keys:", list(graph.keys()))
for key in ["edge_index", "disease_ids", "phenotype_ids"]:
    value = graph[key]
    print(
        key,
        "| type:", type(value).__name__,
        "| sample:", value[:3] if isinstance(value, (list, tuple)) else str(value)[:100]
    )


Graph keys: ['edge_index', 'disease_ids', 'phenotype_ids', 'num_nodes', 'hidden_dim', 'positive_association_count', 'excluded_development_and_test_publications', 'note']
edge_index | type: Tensor | sample: tensor([[ 4612, 12925,  4612,  ..., 13634,  5830, 16148],
        [12925,  4612,  8879,  ...,  5830,
disease_ids | type: list | sample: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000']
phenotype_ids | type: list | sample: ['HP:0000002', 'HP:0000003', 'HP:0000006']


In [ ]:

import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

graph = torch.load(
    DATA / "rare_disease_gnn_filtered_edge_index.pt",
    map_location="cpu",
    weights_only=True
)

edge_index = graph["edge_index"].long()
disease_ids = graph["disease_ids"]       # Keep as strings
phenotype_ids = graph["phenotype_ids"]   # Keep as strings

# Map disease and phenotype identifiers to numeric graph-node indices.
disease_nodes = set(range(len(disease_ids)))
phenotype_offset = len(disease_ids)
phenotype_nodes = set(
    range(phenotype_offset, phenotype_offset + len(phenotype_ids))
)

# Recover unique disease-phenotype pairs from the bidirectional graph.
pairs = set()
for a, b in edge_index.T.tolist():
    if a in disease_nodes and b in phenotype_nodes:
        pairs.add((a, b))
    elif b in disease_nodes and a in phenotype_nodes:
        pairs.add((b, a))

pairs = sorted(pairs)
pairs_tensor = torch.tensor(pairs, dtype=torch.long).T.contiguous()

assert pairs_tensor.shape[1] == graph["positive_association_count"], (
    f"Expected {graph['positive_association_count']} unique pairs, "
    f"recovered {pairs_tensor.shape[1]}"
)

# Reproducible 80/20 split.
generator = torch.Generator().manual_seed(42)
perm = torch.randperm(pairs_tensor.shape[1], generator=generator)
n_val = max(1, int(0.20 * len(perm)))

val_pairs = pairs_tensor[:, :0]  # Initialize with correct shape
val_pairs = pairs_tensor[:, perm[:n_val]]
train_pairs = pairs_tensor[:, perm[n_val:]]

# Only training associations are visible during message passing.
train_edge_index = torch.cat(
    [train_pairs, train_pairs.flip(0)], dim=1
)

torch.save({
    "train_pairs": train_pairs,
    "val_pairs": val_pairs,
    "train_edge_index": train_edge_index,
    "disease_ids": disease_ids,
    "phenotype_ids": phenotype_ids,
    "num_nodes": int(graph["num_nodes"]),
    "hidden_dim": int(graph["hidden_dim"]),
    "seed": 42,
    "split": "80/20 positive association edge split",
    "note": "Validation pairs excluded from message-passing edges."
}, DATA / "rare_disease_gnn_edge_split.pt")

print("EDGE SPLIT VALIDATED")
print("Unique positive pairs:", pairs_tensor.shape[1])
print("Training pairs:", train_pairs.shape[1])
print("Validation pairs:", val_pairs.shape[1])
print("Training directed edges:", train_edge_index.shape[1])
print("Validation edges excluded from training graph: YES")
print("Saved:", DATA / "rare_disease_gnn_edge_split.pt")


EDGE SPLIT VALIDATED
Unique positive pairs: 158659
Training pairs: 126928
Validation pairs: 31731
Training directed edges: 253856
Validation edges excluded from training graph: YES
Saved: /content/drive/MyDrive/Rare Disease ai/processed/rare_disease_gnn_edge_split.pt


In [ ]:

%pip install torch-geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.0 MB/s eta 0:00:00


In [ ]:

import torch
from torch_geometric.nn import GCNConv

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GCNConv import: SUCCESS")



PyTorch: 2.11.0+cu130
CUDA available: True
GCNConv import: SUCCESS


In [ ]:

from pathlib import Path
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

split = torch.load(
    DATA / "rare_disease_gnn_edge_split.pt",
    map_location="cpu",
    weights_only=True
)

class RareDiseaseGCN(nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, edge_index):
        x = F.relu(self.conv1(self.embedding.weight, edge_index))
        return self.conv2(x, edge_index)

class LinkPredictor(nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.gnn = RareDiseaseGCN(num_nodes, hidden_dim)
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, edge_index, pairs):
        z = self.gnn(edge_index)
        return self.scorer(
            torch.cat([z[pairs[0]], z[pairs[1]]], dim=1)
        ).squeeze(-1)

# Reproducible initialization; do not load the old checkpoint.
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

device = torch.device("cuda")
model = LinkPredictor(
    num_nodes=int(split["num_nodes"]),
    hidden_dim=int(split["hidden_dim"])
).to(device)

train_edge_index = split["train_edge_index"].to(device)
train_pairs = split["train_pairs"].to(device)
val_pairs = split["val_pairs"].to(device)

# Build the complete positive-pair set to prevent false negatives.
graph = torch.load(
    DATA / "rare_disease_gnn_filtered_edge_index.pt",
    map_location="cpu",
    weights_only=True
)
disease_ids = graph["disease_ids"]
phenotype_ids = graph["phenotype_ids"]

print("MODEL SETUP READY")
print("Device:", device)
print("Nodes:", int(split["num_nodes"]))
print("Training positive pairs:", train_pairs.shape[1])
print("Validation positive pairs:", val_pairs.shape[1])
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Original checkpoint loaded: NO")


MODEL SETUP READY
Device: cuda
Nodes: 17322
Training positive pairs: 126928
Validation positive pairs: 31731
Model parameters: 1125249
Original checkpoint loaded: NO


In [ ]:

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import copy
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Encode every known positive pair so negatives cannot be known positives.
n_diseases = len(disease_ids)
n_phenotypes = len(phenotype_ids)
n_nodes = int(split["num_nodes"])

all_pairs = torch.as_tensor(
    graph["edge_index"], dtype=torch.long
).cpu()

disease_node_set = set(range(n_diseases))
phenotype_node_set = set(range(n_diseases, n_diseases + n_phenotypes))

positive_pairs = set()
for a, b in all_pairs.T.tolist():
    if a in disease_node_set and b in phenotype_node_set:
        positive_pairs.add((a, b))
    elif b in disease_node_set and a in phenotype_node_set:
        positive_pairs.add((b, a))

positive_keys = {d * n_nodes + p for d, p in positive_pairs}
disease_nodes_cpu = torch.arange(n_diseases)
phenotype_nodes_cpu = torch.arange(n_diseases, n_diseases + n_phenotypes)

def sample_negative_pairs(count):
    """Sample uniformly from disease-phenotype pairs absent from known positives."""
    chosen = set()
    max_attempts = max(10000, count * 20)
    attempts = 0

    while len(chosen) < count and attempts < max_attempts:
        batch_n = min(max(2048, 2 * (count - len(chosen))), 100000)
        ds = torch.randint(0, n_diseases, (batch_n,)).tolist()
        ps = torch.randint(
            n_diseases, n_diseases + n_phenotypes, (batch_n,)
        ).tolist()

        for d, p in zip(ds, ps):
            key = d * n_nodes + p
            if key not in positive_keys:
                chosen.add((d, p))
                if len(chosen) >= count:
                    break
        attempts += batch_n

    if len(chosen) < count:
        raise RuntimeError("Could not sample enough negative pairs.")

    return torch.tensor(list(chosen), dtype=torch.long).T.contiguous().to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001, weight_decay=0.0001
)

best_auc = -1.0
best_state = None
patience = 10
bad_epochs = 0
max_epochs = 100

train_pos_cpu = split["train_pairs"].cpu()
val_pos_cpu = split["val_pairs"].cpu()

for epoch in range(1, max_epochs + 1):
    model.train()
    train_neg = sample_negative_pairs(train_pos_cpu.shape[1])

    train_pos = train_pos_cpu.to(device)
    neg_train = train_neg

    optimizer.zero_grad(set_to_none=True)
    pos_logits = model(train_edge_index, train_pos)
    neg_logits = model(train_edge_index, neg_train)

    logits = torch.cat([pos_logits, neg_logits])
    labels = torch.cat([
        torch.ones_like(pos_logits),
        torch.zeros_like(neg_logits)
    ])

    loss = F.binary_cross_entropy_with_logits(logits, labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    optimizer.step()

    # Validation uses held-out positives and fresh known-negative pairs.
    model.eval()
    with torch.no_grad():
        val_neg = sample_negative_pairs(val_pos_cpu.shape[1])
        val_pos = val_pos_cpu.to(device)
        val_logits_pos = model(train_edge_index, val_pos)
        val_logits_neg = model(train_edge_index, val_neg)

        y_true = np.concatenate([
            np.ones(val_logits_pos.numel()),
            np.zeros(val_logits_neg.numel())
        ])
        y_score = torch.cat([
            val_logits_pos, val_logits_neg
        ]).sigmoid().cpu().numpy()

        val_auc = roc_auc_score(y_true, y_score)
        val_ap = average_precision_score(y_true, y_score)

    print(
        f"Epoch {epoch:03d} | loss={loss.item():.4f} "
        f"| val_AUROC={val_auc:.4f} | val_AP={val_ap:.4f}"
    )

    if val_auc > best_auc:
        best_auc = val_auc
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print(f"Early stopping at epoch {epoch}.")
        break

if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")

model.load_state_dict(best_state)

checkpoint_path = MODEL_DIR / "rare_disease_gnn_filtered_best.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "num_nodes": n_nodes,
    "hidden_dim": int(split["hidden_dim"]),
    "disease_ids": disease_ids,
    "phenotype_ids": phenotype_ids,
    "best_validation_auc": best_auc,
    "seed": SEED,
    "training_pairs": int(train_pos_cpu.shape[1]),
    "validation_pairs": int(val_pos_cpu.shape[1]),
    "note": (
        "Fresh model trained on filtered graph. Edge-level validation; "
        "unmatched/missing publication references may permit residual leakage."
    )
}, checkpoint_path)

print("\nTRAINING COMPLETE")
print("Best validation AUROC:", round(best_auc, 4))
print("Best model saved:", checkpoint_path)
print("Original checkpoint unchanged: YES")


Epoch 001 | loss=0.6878 | val_AUROC=0.6899 | val_AP=0.6679
Epoch 002 | loss=0.6816 | val_AUROC=0.7035 | val_AP=0.6924
Epoch 003 | loss=0.6765 | val_AUROC=0.7102 | val_AP=0.7047
Epoch 004 | loss=0.6720 | val_AUROC=0.7151 | val_AP=0.7138
Epoch 005 | loss=0.6685 | val_AUROC=0.7178 | val_AP=0.7189
Epoch 006 | loss=0.6653 | val_AUROC=0.7164 | val_AP=0.7200
Epoch 007 | loss=0.6626 | val_AUROC=0.7166 | val_AP=0.7204
Epoch 008 | loss=0.6602 | val_AUROC=0.7107 | val_AP=0.7171
Epoch 009 | loss=0.6580 | val_AUROC=0.7118 | val_AP=0.7210
Epoch 010 | loss=0.6561 | val_AUROC=0.7026 | val_AP=0.7132
Epoch 011 | loss=0.6540 | val_AUROC=0.7020 | val_AP=0.7138
Epoch 012 | loss=0.6518 | val_AUROC=0.7001 | val_AP=0.7155
Epoch 013 | loss=0.6498 | val_AUROC=0.6977 | val_AP=0.7158
Epoch 014 | loss=0.6483 | val_AUROC=0.6952 | val_AP=0.7130
Epoch 015 | loss=0.6465 | val_AUROC=0.6921 | val_AP=0.7120
Early stopping at epoch 15.

TRAINING COMPLETE
Best validation AUROC: 0.7178
Best model saved: /content/drive/MyDri

In [ ]:

from pathlib import Path
import torch
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
DATA = PROJECT / "processed"

checkpoint_path = MODEL_DIR / "rare_disease_gnn_filtered_best.pt"
original_path = DATA / "rare_disease_gnn_training_only.pt"

assert checkpoint_path.exists(), "Filtered checkpoint not found."
assert original_path.exists(), "Original checkpoint not found."

safe_types = [
    np._core.multiarray.scalar,
    np.dtype,
]

with torch.serialization.safe_globals(safe_types):
    filtered = torch.load(
        checkpoint_path, map_location="cpu", weights_only=True
    )
    original = torch.load(
        original_path, map_location="cpu", weights_only=True
    )

print("CHECKPOINT VERIFICATION")
print("Filtered checkpoint exists: YES")
print("Original checkpoint exists: YES")
print("Filtered model nodes:", filtered["num_nodes"])
print("Hidden dimension:", filtered["hidden_dim"])
print("Best validation AUROC:", round(float(filtered["best_validation_auc"]), 4))
print("Filtered model parameters:", sum(
    tensor.numel()
    for tensor in filtered["model_state_dict"].values()
))
print("Original checkpoint still readable: YES")
print("Original checkpoint unchanged: YES")


UnpicklingError: Weights only load failed. In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: 

Can only build Tensor, Parameter, OrderedDict or types allowlisted via `add_safe_globals`, but got <class 'numpy.dtypes.Float64DType'>

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

In [ ]:

from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
DATA = PROJECT / "processed"

checkpoint_path = MODEL_DIR / "rare_disease_gnn_filtered_best.pt"
original_path = DATA / "rare_disease_gnn_training_only.pt"

assert checkpoint_path.exists(), f"Missing checkpoint: {checkpoint_path}"
assert original_path.exists(), f"Missing original checkpoint: {original_path}"

# Only use weights_only=False for these trusted, locally created files.
filtered = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("FILTERED CHECKPOINT VERIFIED")
print("Model nodes:", filtered["num_nodes"])
print("Hidden dimension:", filtered["hidden_dim"])
print("Best validation AUROC:", round(float(filtered["best_validation_auc"]), 4))
print("State-dict parameter count:", sum(
    value.numel()
    for value in filtered["model_state_dict"].values()
    if isinstance(value, torch.Tensor)
))
print("Original checkpoint file exists:", original_path.exists())
print("Filtered checkpoint loaded successfully: YES")


FILTERED CHECKPOINT VERIFIED
Model nodes: 17322
Hidden dimension: 64
Best validation AUROC: 0.7178
State-dict parameter count: 1125249
Original checkpoint file exists: True
Filtered checkpoint loaded successfully: YES


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = [
    "patient_publication_split.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_baseline_vs_gnn_paired.csv",
    "patient_test_disease_coverage.csv",
]

for filename in files:
    path = DATA / filename
    print(f"\n--- {filename} ---")
    if not path.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path, nrows=5)
    print("Columns:", df.columns.tolist())
    print(df.head(2).to_string(index=False))

# Check the current held-out test cohort size.
split_path = DATA / "patient_publication_split.csv"
if split_path.exists():
    splits = pd.read_csv(split_path)
    print("\nSPLIT COUNTS")
    print(splits["split"].value_counts().to_string())



--- patient_publication_split.csv ---
Columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
                    case_id     pmid       split diagnosis_ids
PMID_19836009_Family_E_II_6 19836009 development   OMIM:277600
PMID_19836009_Family_E_II_3 19836009 development   OMIM:277600

--- patient_phenotype_only_baseline.csv ---
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']
                       case_id     pmid diagnosis_ids  rank  top1  top5  top10  candidate_covered
PMID_28239884_Family_1_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_2_proband 28239884   OMIM:148600   3.0     0     1      1                  1

--- patient_baseline_vs_gnn_paired.csv ---
Columns: ['case_id', 'pmid', 'diagnosis_ids_baseline', 'rank_baseline', 'top1_baseline', 'top5_baseline', 'top10_baseline', 'candidate_covered', 'diagnosis_ids_gnn', 'n_positive_hpo', 'n_recognized_hpo', 'target_in_candidate_set', 'ra

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

# Load the filtered checkpoint and graph split.
ckpt = torch.load(
    MODEL_DIR / "rare_disease_gnn_filtered_best.pt",
    map_location="cpu",
    weights_only=False
)
split = torch.load(
    DATA / "rare_disease_gnn_edge_split.pt",
    map_location="cpu",
    weights_only=True
)

# Rebuild the same GNN architecture used in training.
class RareDiseaseGCN(nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, edge_index):
        x = F.relu(self.conv1(self.embedding.weight, edge_index))
        return self.conv2(x, edge_index)

class LinkPredictor(nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.gnn = RareDiseaseGCN(num_nodes, hidden_dim)
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, edge_index, pairs):
        z = self.gnn(edge_index)
        return self.scorer(
            torch.cat([z[pairs[0]], z[pairs[1]]], dim=1)
        ).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LinkPredictor(ckpt["num_nodes"], ckpt["hidden_dim"]).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

edge_index = split["train_edge_index"].to(device)
disease_ids = ckpt["disease_ids"]
phenotype_ids = ckpt["phenotype_ids"]
n_diseases = len(disease_ids)
n_phenotypes = len(phenotype_ids)

# Map HPO IDs to model node indices.
hpo_to_node = {
    hpo: n_diseases + i for i, hpo in enumerate(phenotype_ids)
}
disease_node_indices = torch.arange(n_diseases, device=device)

with torch.no_grad():
    embeddings = model.gnn(edge_index)

# Read held-out test cases and extract their known positive HPO terms.
split_df = pd.read_csv(DATA / "patient_publication_split.csv")
test_df = split_df[split_df["split"] == "test"].copy()

phenopacket_root = PROJECT / "all_phenopackets"
packet_by_id = {}
for path in phenopacket_root.rglob("*.json"):
    packet_by_id[path.stem] = path

def get_case_hpos(case_id):
    path = packet_by_id.get(case_id)
    if path is None:
        return []
    try:
        packet = json.loads(path.read_text())
        features = packet.get("phenotypicFeatures", [])
        positive = {
            f.get("type", {}).get("id")
            for f in features
            if f.get("type", {}).get("id")
            and f.get("excluded") is not True
        }
        excluded = {
            f.get("type", {}).get("id")
            for f in features
            if f.get("type", {}).get("id")
            and f.get("excluded") is True
        }
        return sorted((positive - excluded) & set(hpo_to_node))
    except Exception:
        return []

rows = []
for row in test_df.itertuples(index=False):
    hpos = get_case_hpos(row.case_id)
    target_ids = [x.strip() for x in str(row.diagnosis_ids).split(";") if x.strip()]
    recognized = [hpo_to_node[h] for h in hpos]

    result = {
        "case_id": row.case_id,
        "pmid": row.pmid,
        "diagnosis_ids": row.diagnosis_ids,
        "n_positive_hpo": len(hpos),
        "n_recognized_hpo": len(recognized),
        "target_in_candidate_set": any(d in set(disease_ids) for d in target_ids),
        "rank": np.nan,
        "top1": 0,
        "top5": 0,
        "top10": 0,
        "mrr": 0.0,
        "top_prediction": None,
    }

    if recognized:
        patient_vec = embeddings[recognized].mean(dim=0, keepdim=True)
        scores = F.cosine_similarity(
            patient_vec, embeddings[:n_diseases], dim=1
        ).cpu().numpy()

        order = np.argsort(-scores)
        ranked_ids = [disease_ids[i] for i in order]
        target_ranks = [
            ranked_ids.index(t) + 1 for t in target_ids
            if t in ranked_ids
        ]

        if target_ranks:
            rank = min(target_ranks)
            result.update({
                "rank": rank,
                "top1": int(rank <= 1),
                "top5": int(rank <= 5),
                "top10": int(rank <= 10),
                "mrr": 1.0 / rank,
            })
        result["top_prediction"] = ranked_ids[0] if ranked_ids else None

    rows.append(result)

results = pd.DataFrame(rows)
out_path = DATA / "patient_filtered_gnn_test_rankings.csv"
results.to_csv(out_path, index=False)

covered = results["rank"].notna()
print("PATIENT-LEVEL EVALUATION")
print("Held-out test cases:", len(results))
print("Cases with recognized HPO terms:", int((results["n_recognized_hpo"] > 0).sum()))
print("Cases with target in candidate vocabulary:", int(results["target_in_candidate_set"].sum()))
print("Cases with a calculable target rank:", int(covered.sum()))
print("Top-1 among ranked cases:", round(results.loc[covered, "top1"].mean(), 4) if covered.any() else "N/A")
print("Top-5 among ranked cases:", round(results.loc[covered, "top5"].mean(), 4) if covered.any() else "N/A")
print("Top-10 among ranked cases:", round(results.loc[covered, "top10"].mean(), 4) if covered.any() else "N/A")
print("MRR among ranked cases:", round(results.loc[covered, "mrr"].mean(), 4) if covered.any() else "N/A")
print("Saved:", out_path)


PATIENT-LEVEL EVALUATION
Held-out test cases: 1959
Cases with recognized HPO terms: 1949
Cases with target in candidate vocabulary: 1492
Cases with a calculable target rank: 1492
Top-1 among ranked cases: 0.002
Top-5 among ranked cases: 0.002
Top-10 among ranked cases: 0.002
MRR among ranked cases: 0.0025
Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_filtered_gnn_test_rankings.csv


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

results = pd.read_csv(DATA / "patient_filtered_gnn_test_rankings.csv")
baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")

ranked = results[results["rank"].notna()].copy()

print("RANK DIAGNOSTICS")
print("Ranked cases:", len(ranked))
print("Rank value counts (first 15):")
print(ranked["rank"].value_counts().sort_index().head(15).to_string())
print("\nRank summary:")
print(ranked["rank"].describe().to_string())

print("\nTOP-K CONSISTENCY")
for k in [1, 5, 10, 50, 100]:
    print(f"Top-{k}: {(ranked['rank'] <= k).mean():.4f}")

print("\nDIAGNOSIS PARSING")
print("Example diagnosis IDs:")
print(ranked["diagnosis_ids"].head(10).to_list())
print("Example predicted IDs:")
print(ranked["top_prediction"].head(10).to_list())

print("\nBASELINE CONTEXT")
print("Baseline rows:", len(baseline))
print("Baseline top-1:", baseline["top1"].mean())
print("Baseline top-5:", baseline["top5"].mean())
print("Baseline top-10:", baseline["top10"].mean())
print("Baseline candidate coverage:", baseline["candidate_covered"].mean())

print("\nSaved ranking file:", DATA / "patient_filtered_gnn_test_rankings.csv")


RANK DIAGNOSTICS
Ranked cases: 1492
Rank value counts (first 15):
rank
1.0      3
127.0    1
131.0    1
142.0    1
147.0    1
236.0    1
242.0    1
244.0    1
268.0    1
297.0    1
319.0    1
401.0    1
405.0    1
410.0    1
411.0    2

Rank summary:
count    1492.000000
mean     4839.246649
std      2386.307142
min         1.000000
25%      3241.750000
50%      5928.500000
75%      6884.750000
max      7339.000000

TOP-K CONSISTENCY
Top-1: 0.0020
Top-5: 0.0020
Top-10: 0.0020
Top-50: 0.0020
Top-100: 0.0020

DIAGNOSIS PARSING
Example diagnosis IDs:
['OMIM:148600', 'OMIM:148600', 'OMIM:148600', 'OMIM:102700', 'OMIM:621196', 'OMIM:621196', 'OMIM:620785', 'OMIM:620785', 'OMIM:620785', 'OMIM:620785']
Example predicted IDs:
['OMIM:148600', 'OMIM:148600', 'OMIM:148600', 'ORPHA:254892', 'OMIM:618131', 'OMIM:618131', 'ORPHA:394', 'OMIM:606232', 'OMIM:606232', 'OMIM:606232']

BASELINE CONTEXT
Baseline rows: 1959
Baseline top-1: 0.1138335885655947
Baseline top-5: 0.19601837672281777
Baseline top-

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

ckpt = torch.load(
    MODEL_DIR / "rare_disease_gnn_filtered_best.pt",
    map_location="cpu",
    weights_only=False
)
split = torch.load(
    DATA / "rare_disease_gnn_edge_split.pt",
    map_location="cpu",
    weights_only=True
)

class RareDiseaseGCN(nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, edge_index):
        x = F.relu(self.conv1(self.embedding.weight, edge_index))
        return self.conv2(x, edge_index)

class LinkPredictor(nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.gnn = RareDiseaseGCN(num_nodes, hidden_dim)
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, edge_index, pairs):
        z = self.gnn(edge_index)
        return self.scorer(
            torch.cat([z[pairs[0]], z[pairs[1]]], dim=1)
        ).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LinkPredictor(ckpt["num_nodes"], ckpt["hidden_dim"]).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

edge_index = split["train_edge_index"].to(device)
disease_ids = ckpt["disease_ids"]
phenotype_ids = ckpt["phenotype_ids"]
n_diseases = len(disease_ids)
hpo_to_node = {
    hpo: n_diseases + i for i, hpo in enumerate(phenotype_ids)
}
disease_id_set = set(disease_ids)

# Reuse the previously generated patient rankings' case list,
# then extract phenotypes from the original phenopackets.
old_results = pd.read_csv(DATA / "patient_filtered_gnn_test_rankings.csv")
test_split = pd.read_csv(DATA / "patient_publication_split.csv")
test_split = test_split[test_split["split"] == "test"]
test_split = test_split.drop_duplicates("case_id").set_index("case_id")

phenopacket_root = PROJECT / "all_phenopackets"
packet_by_id = {p.stem: p for p in phenopacket_root.rglob("*.json")}

with torch.no_grad():
    z = model.gnn(edge_index)
    disease_z = z[:n_diseases]

rows = []
for old in old_results.itertuples(index=False):
    if old.case_id not in test_split.index:
        continue

    packet_path = packet_by_id.get(old.case_id)
    if packet_path is None:
        continue

    try:
        packet = json.loads(packet_path.read_text())
        features = packet.get("phenotypicFeatures", [])
        positive = {
            f.get("type", {}).get("id")
            for f in features
            if f.get("type", {}).get("id") and f.get("excluded") is not True
        }
        excluded = {
            f.get("type", {}).get("id")
            for f in features
            if f.get("type", {}).get("id") and f.get("excluded") is True
        }
        hpos = sorted((positive - excluded) & set(hpo_to_node))
    except Exception:
        continue

    target_ids = [
        x.strip() for x in str(test_split.loc[old.case_id, "diagnosis_ids"]).split(";")
        if x.strip()
    ]
    recognized_nodes = [hpo_to_node[h] for h in hpos]

    result = {
        "case_id": old.case_id,
        "diagnosis_ids": ";".join(target_ids),
        "n_recognized_hpo": len(recognized_nodes),
        "target_in_vocabulary": any(t in disease_id_set for t in target_ids),
        "rank": np.nan,
        "top1": 0,
        "top5": 0,
        "top10": 0,
        "mrr": 0.0,
        "top_prediction": None,
    }

    if recognized_nodes:
        patient_z = z[recognized_nodes].mean(dim=0)
        patient_batch = patient_z.unsqueeze(0).expand(n_diseases, -1)
        pairs = torch.stack([
            torch.arange(n_diseases, device=device),
            torch.full((n_diseases,), recognized_nodes[0], device=device)
        ])  # placeholder replaced below

        # Use the trained MLP scorer directly on disease and patient vectors.
        logits = model.scorer(
            torch.cat([disease_z, patient_batch], dim=1)
        ).squeeze(-1)
        order = torch.argsort(logits, descending=True).cpu().tolist()
        ranked_ids = [disease_ids[i] for i in order]

        target_ranks = [
            ranked_ids.index(t) + 1 for t in target_ids if t in ranked_ids
        ]
        if target_ranks:
            rank = min(target_ranks)
            result.update({
                "rank": rank,
                "top1": int(rank <= 1),
                "top5": int(rank <= 5),
                "top10": int(rank <= 10),
                "mrr": 1.0 / rank,
            })
        result["top_prediction"] = ranked_ids[0]

    rows.append(result)

out = pd.DataFrame(rows)
out_path = DATA / "patient_filtered_gnn_linkhead_rankings.csv"
out.to_csv(out_path, index=False)

covered = out["rank"].notna()
print("TRAINED-HEAD RANKING")
print("Cases evaluated:", len(out))
print("Cases with recognized HPO terms:", int((out["n_recognized_hpo"] > 0).sum()))
print("Targets in disease vocabulary:", int(out["target_in_vocabulary"].sum()))
print("Cases with a target rank:", int(covered.sum()))
if covered.any():
    print("Top-1:", round(out.loc[covered, "top1"].mean(), 4))
    print("Top-5:", round(out.loc[covered, "top5"].mean(), 4))
    print("Top-10:", round(out.loc[covered, "top10"].mean(), 4))
    print("MRR:", round(out.loc[covered, "mrr"].mean(), 4))
print("Saved:", out_path)


TRAINED-HEAD RANKING
Cases evaluated: 1959
Cases with recognized HPO terms: 1949
Targets in disease vocabulary: 1492
Cases with a target rank: 1492
Top-1: 0.0007
Top-5: 0.0007
Top-10: 0.0027
MRR: 0.0022
Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_filtered_gnn_linkhead_rankings.csv


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

baseline_path = DATA / "patient_phenotype_only_baseline.csv"
cosine_path = DATA / "patient_filtered_gnn_test_rankings.csv"
head_path = DATA / "patient_filtered_gnn_linkhead_rankings.csv"
split_path = DATA / "patient_publication_split.csv"

baseline = pd.read_csv(baseline_path)
cosine = pd.read_csv(cosine_path)
head = pd.read_csv(head_path)
splits = pd.read_csv(split_path)

# Restrict the baseline to held-out test cases.
test_ids = set(splits.loc[splits["split"] == "test", "case_id"])
baseline = baseline[baseline["case_id"].isin(test_ids)].copy()
cosine = cosine[cosine["case_id"].isin(test_ids)].copy()
head = head[head["case_id"].isin(test_ids)].copy()

# Ensure one row per case before joining.
baseline = baseline.drop_duplicates("case_id")
cosine = cosine.drop_duplicates("case_id")
head = head.drop_duplicates("case_id")

# Rename metrics to avoid ambiguous columns.
baseline = baseline[[
    "case_id", "rank", "top1", "top5", "top10", "candidate_covered"
]].rename(columns={
    "rank": "rank_baseline",
    "top1": "top1_baseline",
    "top5": "top5_baseline",
    "top10": "top10_baseline",
    "candidate_covered": "baseline_covered"
})

cosine = cosine[["case_id", "rank", "top1", "top5", "top10", "mrr"]].rename(
    columns={
        "rank": "rank_cosine",
        "top1": "top1_cosine",
        "top5": "top5_cosine",
        "top10": "top10_cosine",
        "mrr": "mrr_cosine"
    }
)

head = head[["case_id", "rank", "top1", "top5", "top10", "mrr"]].rename(
    columns={
        "rank": "rank_head",
        "top1": "top1_head",
        "top5": "top5_head",
        "top10": "top10_head",
        "mrr": "mrr_head"
    }
)

joined = baseline.merge(cosine, on="case_id", how="outer")
joined = joined.merge(head, on="case_id", how="outer")

# A comparable cohort requires a target rank from all three methods.
common = joined.dropna(subset=[
    "rank_baseline", "rank_cosine", "rank_head"
]).copy()

def metrics(df, prefix):
    rank = df[f"rank_{prefix}"]
    return {
        "Method": prefix,
        "Cases with rank": int(rank.notna().sum()),
        "Top-1": float((rank <= 1).mean()),
        "Top-5": float((rank <= 5).mean()),
        "Top-10": float((rank <= 10).mean()),
        "MRR": float((1 / rank).mean())
    }

summary = pd.DataFrame([
    metrics(common, "baseline"),
    metrics(common, "cosine"),
    metrics(common, "head")
])

print("COHORT COUNTS")
print("Held-out test cases:", len(test_ids))
print("Baseline test cases with ranks:", int(joined["rank_baseline"].notna().sum()))
print("Cosine GNN cases with ranks:", int(joined["rank_cosine"].notna().sum()))
print("Link-head cases with ranks:", int(joined["rank_head"].notna().sum()))
print("Cases ranked by all three methods:", len(common))

print("\nMETRICS ON THE SAME COMMON COHORT")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

out_path = DATA / "patient_three_method_common_cohort_comparison.csv"
joined.to_csv(out_path, index=False)
print("\nSaved joined case-level results:", out_path)



COHORT COUNTS
Held-out test cases: 1959
Baseline test cases with ranks: 1551
Cosine GNN cases with ranks: 1492
Link-head cases with ranks: 1492
Cases ranked by all three methods: 1181

METRICS ON THE SAME COMMON COHORT
  Method  Cases with rank  Top-1  Top-5  Top-10    MRR
baseline             1181 0.1160 0.2235  0.2566 0.1659
  cosine             1181 0.0025 0.0025  0.0025 0.0031
    head             1181 0.0008 0.0008  0.0008 0.0022

Saved joined case-level results: /content/drive/MyDrive/Rare Disease ai/processed/patient_three_method_common_cohort_comparison.csv


In [ ]:

from pathlib import Path
import json
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

split_df = pd.read_csv(DATA / "patient_publication_split.csv")
graph = torch.load(
    DATA / "rare_disease_gnn_filtered_edge_index.pt",
    map_location="cpu",
    weights_only=True
)

disease_vocab = set(graph["disease_ids"])
phenotype_vocab = set(graph["phenotype_ids"])

packet_root = PROJECT / "all_phenopackets"
packet_by_id = {p.stem: p for p in packet_root.rglob("*.json")}

development = split_df[split_df["split"] == "development"].copy()
test_ids = set(split_df.loc[split_df["split"] == "test", "case_id"])

rows = []
missing_packets = []
missing_diagnoses = []
unmatched_pmids = []

for row in development.itertuples(index=False):
    path = packet_by_id.get(row.case_id)
    if path is None:
        missing_packets.append(row.case_id)
        continue

    try:
        packet = json.loads(path.read_text())
    except Exception:
        missing_packets.append(row.case_id)
        continue

    features = packet.get("phenotypicFeatures", [])
    positive = {
        f.get("type", {}).get("id")
        for f in features
        if f.get("type", {}).get("id") and f.get("excluded") is not True
    }
    excluded = {
        f.get("type", {}).get("id")
        for f in features
        if f.get("type", {}).get("id") and f.get("excluded") is True
    }
    hpos = sorted((positive - excluded) & phenotype_vocab)

    diagnoses = [
        d.strip() for d in str(row.diagnosis_ids).split(";")
        if d.strip()
    ]
    known_diagnoses = [d for d in diagnoses if d in disease_vocab]

    if not known_diagnoses:
        missing_diagnoses.append(row.case_id)

    if not hpos or not known_diagnoses:
        continue

    rows.append({
        "case_id": row.case_id,
        "pmid": str(row.pmid),
        "diagnosis_ids": ";".join(known_diagnoses),
        "hpo_ids": ";".join(hpos),
        "n_hpo": len(hpos),
        "n_diagnoses": len(known_diagnoses),
    })

dev_dataset = pd.DataFrame(rows)
out_path = DATA / "patient_level_development_dataset.csv"
dev_dataset.to_csv(out_path, index=False)

# Basic split-integrity checks.
assert not set(dev_dataset["case_id"]) & test_ids
assert dev_dataset["case_id"].is_unique

print("DEVELOPMENT DATASET AUDIT")
print("Development cases in split:", len(development))
print("Cases saved with recognized HPOs and disease targets:", len(dev_dataset))
print("Missing/unreadable phenopackets:", len(missing_packets))
print("Cases without a diagnosis in graph vocabulary:", len(missing_diagnoses))
print("Unique development cases saved:", dev_dataset["case_id"].nunique())
print("Development/test case overlap:", len(set(dev_dataset["case_id"]) & test_ids))
print("Saved:", out_path)
print("\nLEAKAGE LIMITATION")
print("This cell checks case separation, not publication-level overlap in the graph.")
print("Some annotation references are missing or unmatched, so graph leakage remains possible.")


DEVELOPMENT DATASET AUDIT
Development cases in split: 8386
Cases saved with recognized HPOs and disease targets: 6636
Missing/unreadable phenopackets: 0
Cases without a diagnosis in graph vocabulary: 1750
Unique development cases saved: 6636
Development/test case overlap: 0
Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_level_development_dataset.csv

LEAKAGE LIMITATION
This cell checks case separation, not publication-level overlap in the graph.
Some annotation references are missing or unmatched, so graph leakage remains possible.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

required_files = [
    DATA / "patient_level_development_dataset.csv",
    DATA / "patient_publication_split.csv",
    DATA / "rare_disease_gnn_filtered_edge_index.pt",
    DATA / "rare_disease_gnn_edge_split.pt",
    MODEL_DIR / "rare_disease_gnn_filtered_best.pt",
]

print("PROJECT:", PROJECT)
print("Project exists:", PROJECT.exists())

for path in required_files:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

if (DATA / "patient_level_development_dataset.csv").exists():
    df = pd.read_csv(DATA / "patient_level_development_dataset.csv")
    print("\nDEVELOPMENT DATASET")
    print("Cases:", len(df))
    print("Unique cases:", df["case_id"].nunique())
    print("Columns:", df.columns.tolist())

print("\nResume point: Step 13 — patient-level dataset and leakage audit")


PROJECT: /content/drive/MyDrive/Rare Disease ai
Project exists: True
patient_level_development_dataset.csv: FOUND
patient_publication_split.csv: FOUND
rare_disease_gnn_filtered_edge_index.pt: FOUND
rare_disease_gnn_edge_split.pt: FOUND
rare_disease_gnn_filtered_best.pt: FOUND

DEVELOPMENT DATASET
Cases: 6636
Unique cases: 6636
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']

Resume point: Step 13 — patient-level dataset and leakage audit


In [ ]:

from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

dev = pd.read_csv(DATA / "patient_level_development_dataset.csv")
splits = pd.read_csv(DATA / "patient_publication_split.csv")
annotations_path = DATA / "gnn_annotations_excluding_dev_test_pmids.csv"

assert annotations_path.exists(), f"Missing file: {annotations_path}"

annotations = pd.read_csv(annotations_path, low_memory=False)

# Normalize publication identifiers to comparable strings.
def normalize_pmid(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    if value.endswith(".0"):
        value = value[:-2]
    return value if value and value.lower() not in {"nan", "none"} else None

dev_pmids = {
    p for p in dev["pmid"].map(normalize_pmid).dropna()
}
test_pmids = {
    p for p in splits.loc[splits["split"] == "test", "pmid"]
         .map(normalize_pmid).dropna()
}

# Find the reference column in the annotation file.
reference_col = "reference" if "reference" in annotations.columns else None
if reference_col is None:
    raise ValueError(
        f"No 'reference' column found. Columns: {annotations.columns.tolist()}"
    )

annotation_pmids = annotations[reference_col].map(normalize_pmid)
known_pmids = set(annotation_pmids.dropna())

summary = {
    "development_cases": int(len(dev)),
    "development_unique_pmids": int(len(dev_pmids)),
    "test_unique_pmids": int(len(test_pmids)),
    "annotation_rows": int(len(annotations)),
    "annotation_rows_with_reference": int(annotation_pmids.notna().sum()),
    "annotation_rows_without_reference": int(annotation_pmids.isna().sum()),
    "development_pmids_found_in_filtered_annotations": int(len(dev_pmids & known_pmids)),
    "test_pmids_found_in_filtered_annotations": int(len(test_pmids & known_pmids)),
    "development_pmids_overlap_count": int(len(dev_pmids & known_pmids)),
    "test_pmids_overlap_count": int(len(test_pmids & known_pmids)),
    "note": (
        "Reference matching is exact after basic string normalization. "
        "Missing or differently formatted references can conceal overlap."
    )
}

out_path = DATA / "patient_level_publication_leakage_audit.json"
out_path.write_text(json.dumps(summary, indent=2))

print("PUBLICATION OVERLAP AUDIT")
for key, value in summary.items():
    print(f"{key}: {value}")
print("\nSaved:", out_path)


PUBLICATION OVERLAP AUDIT
development_cases: 6636
development_unique_pmids: 1110
test_unique_pmids: 346
annotation_rows: 159025
annotation_rows_with_reference: 159025
annotation_rows_without_reference: 0
development_pmids_found_in_filtered_annotations: 0
test_pmids_found_in_filtered_annotations: 0
development_pmids_overlap_count: 0
test_pmids_overlap_count: 0
note: Reference matching is exact after basic string normalization. Missing or differently formatted references can conceal overlap.

Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_level_publication_leakage_audit.json


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

dev = pd.read_csv(DATA / "patient_level_development_dataset.csv")
test_split = pd.read_csv(DATA / "patient_publication_split.csv")
test_pmids = set(
    test_split.loc[test_split["split"] == "test", "pmid"].astype(str)
)

# Normalize PMID strings.
dev["pmid"] = dev["pmid"].astype(str).str.replace(r"\.0$", "", regex=True)
test_pmids = {p.replace(".0", "") for p in test_pmids}

# Verify no development rows use a held-out test publication.
overlap = set(dev["pmid"]) & test_pmids
assert not overlap, f"Development/test PMID overlap found: {len(overlap)}"

# Deterministic publication-level split.
unique_pmids = np.array(sorted(dev["pmid"].unique()))
rng = np.random.default_rng(42)
rng.shuffle(unique_pmids)

n_val_pmids = max(1, round(0.15 * len(unique_pmids)))
val_pmids = set(unique_pmids[:n_val_pmids])
train_pmids = set(unique_pmids[n_val_pmids:])

dev["subset"] = np.where(dev["pmid"].isin(val_pmids), "validation", "train")

train_df = dev[dev["subset"] == "train"].copy()
val_df = dev[dev["subset"] == "validation"].copy()

# Publication-level isolation checks.
assert set(train_df["pmid"]).isdisjoint(set(val_df["pmid"]))
assert set(train_df["case_id"]).isdisjoint(set(val_df["case_id"]))
assert set(dev["case_id"]).isdisjoint(
    set(test_split.loc[test_split["split"] == "test", "case_id"])
)

train_path = DATA / "patient_level_train.csv"
val_path = DATA / "patient_level_validation.csv"
split_summary_path = DATA / "patient_level_split_summary.json"

train_df.drop(columns=["subset"]).to_csv(train_path, index=False)
val_df.drop(columns=["subset"]).to_csv(val_path, index=False)

summary = {
    "seed": 42,
    "split_unit": "PMID/publication",
    "train_cases": int(len(train_df)),
    "validation_cases": int(len(val_df)),
    "train_pmids": int(train_df["pmid"].nunique()),
    "validation_pmids": int(val_df["pmid"].nunique()),
    "train_validation_pmid_overlap": int(
        len(set(train_df["pmid"]) & set(val_df["pmid"]))
    ),
    "development_test_pmid_overlap": int(len(overlap)),
    "test_set_used_for_training_or_validation": False,
}

split_summary_path.write_text(json.dumps(summary, indent=2))

print("PATIENT-LEVEL SPLIT READY")
for key, value in summary.items():
    print(f"{key}: {value}")
print("\nSaved:", train_path)
print("Saved:", val_path)
print("Saved:", split_summary_path)


PATIENT-LEVEL SPLIT READY
seed: 42
split_unit: PMID/publication
train_cases: 5730
validation_cases: 906
train_pmids: 944
validation_pmids: 166
train_validation_pmid_overlap: 0
development_test_pmid_overlap: 0
test_set_used_for_training_or_validation: False

Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_level_train.csv
Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_level_validation.csv
Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_level_split_summary.json


In [ ]:

from pathlib import Path
import pandas as pd
import torch
import json
from collections import Counter

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

train = pd.read_csv(DATA / "patient_level_train.csv")
val = pd.read_csv(DATA / "patient_level_validation.csv")

graph = torch.load(
    DATA / "rare_disease_gnn_filtered_edge_index.pt",
    map_location="cpu",
    weights_only=True
)

disease_vocab = set(graph["disease_ids"])
phenotype_vocab = set(graph["phenotype_ids"])

def parse_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split(";") if x.strip()]

summary = {}
for name, df in [("train", train), ("validation", val)]:
    diagnosis_lists = df["diagnosis_ids"].map(parse_ids)
    hpo_lists = df["hpo_ids"].map(parse_ids)

    labels_total = sum(len(ids) for ids in diagnosis_lists)
    labels_in_vocab = sum(
        sum(d in disease_vocab for d in ids) for ids in diagnosis_lists
    )
    hpos_total = sum(len(ids) for ids in hpo_lists)
    hpos_in_vocab = sum(
        sum(h in phenotype_vocab for h in ids) for ids in hpo_lists
    )

    summary[name] = {
        "cases": int(len(df)),
        "unique_pmids": int(df["pmid"].nunique()),
        "cases_with_multiple_diagnoses": int(
            diagnosis_lists.map(len).gt(1).sum()
        ),
        "diagnosis_labels": int(labels_total),
        "diagnosis_labels_in_vocabulary": int(labels_in_vocab),
        "cases_with_no_valid_target": int(
            diagnosis_lists.map(
                lambda ids: not any(d in disease_vocab for d in ids)
            ).sum()
        ),
        "total_hpo_terms": int(hpos_total),
        "hpo_terms_in_vocabulary": int(hpos_in_vocab),
        "cases_with_no_hpo_terms": int(hpo_lists.map(len).eq(0).sum()),
    }

# Check train/validation publication separation again.
summary["train_validation_pmid_overlap"] = len(
    set(train["pmid"].astype(str)) & set(val["pmid"].astype(str))
)
summary["disease_vocabulary_size"] = len(disease_vocab)
summary["phenotype_vocabulary_size"] = len(phenotype_vocab)

out_path = DATA / "patient_level_label_audit.json"
out_path.write_text(json.dumps(summary, indent=2))

print("PATIENT LABEL AUDIT")
print(json.dumps(summary, indent=2))
print("\nSaved:", out_path)


PATIENT LABEL AUDIT
{
  "train": {
    "cases": 5730,
    "unique_pmids": 944,
    "cases_with_multiple_diagnoses": 0,
    "diagnosis_labels": 5730,
    "diagnosis_labels_in_vocabulary": 5730,
    "cases_with_no_valid_target": 0,
    "total_hpo_terms": 49192,
    "hpo_terms_in_vocabulary": 49192,
    "cases_with_no_hpo_terms": 0
  },
  "validation": {
    "cases": 906,
    "unique_pmids": 166,
    "cases_with_multiple_diagnoses": 0,
    "diagnosis_labels": 906,
    "diagnosis_labels_in_vocabulary": 906,
    "cases_with_no_valid_target": 0,
    "total_hpo_terms": 7851,
    "hpo_terms_in_vocabulary": 7851,
    "cases_with_no_hpo_terms": 0
  },
  "train_validation_pmid_overlap": 0,
  "disease_vocabulary_size": 7340,
  "phenotype_vocabulary_size": 9982
}

Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_level_label_audit.json


In [ ]:

from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv(DATA / "patient_level_train.csv")
val_df = pd.read_csv(DATA / "patient_level_validation.csv")

graph = torch.load(
    DATA / "rare_disease_gnn_filtered_edge_index.pt",
    map_location="cpu",
    weights_only=True
)

disease_ids = graph["disease_ids"]
phenotype_ids = graph["phenotype_ids"]
disease_to_idx = {d: i for i, d in enumerate(disease_ids)}
phenotype_to_idx = {h: i for i, h in enumerate(phenotype_ids)}

def parse_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split(";") if x.strip()]

def encode_cases(df):
    cases = []
    for row in df.itertuples(index=False):
        hpos = [
            phenotype_to_idx[h]
            for h in parse_ids(row.hpo_ids)
            if h in phenotype_to_idx
        ]
        targets = [
            disease_to_idx[d]
            for d in parse_ids(row.diagnosis_ids)
            if d in disease_to_idx
        ]
        if hpos and targets:
            cases.append({
                "case_id": row.case_id,
                "pmid": str(row.pmid),
                "hpos": sorted(set(hpos)),
                "target": targets[0]
            })
    return cases

train_cases = encode_cases(train_df)
val_cases = encode_cases(val_df)

class PatientDiseaseRanker(nn.Module):
    def __init__(self, n_hpo, n_disease, dim=128):
        super().__init__()
        self.hpo_embedding = nn.Embedding(n_hpo, dim)
        self.disease_embedding = nn.Embedding(n_disease, dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(dim, dim)
        )
        self.disease_bias = nn.Embedding(n_disease, 1)

    def encode_patient(self, hpo_indices):
        h = self.hpo_embedding(hpo_indices)
        patient = h.mean(dim=0)
        return self.patient_projection(patient)

    def score_all_diseases(self, hpo_indices):
        patient = self.encode_patient(hpo_indices)
        disease = self.disease_embedding.weight
        scores = disease @ patient / (patient.norm() * disease.norm(dim=1).clamp_min(1e-8) + 1e-8)
        return scores + self.disease_bias.weight.squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ranker = PatientDiseaseRanker(
    n_hpo=len(phenotype_ids),
    n_disease=len(disease_ids),
    dim=128
).to(device)

config = {
    "model": "PatientDiseaseRanker",
    "dimension": 128,
    "dropout": 0.2,
    "seed": SEED,
    "training_cases": len(train_cases),
    "validation_cases": len(val_cases),
    "training_objective": "patient-to-disease ranking with sampled competing diseases",
    "device": str(device),
    "test_set_used_for_training_or_validation": False
}
config_path = MODEL_DIR / "patient_level_ranker_config.json"
config_path.write_text(json.dumps(config, indent=2))

print("PATIENT RANKER READY")
print("Device:", device)
print("Training cases:", len(train_cases))
print("Validation cases:", len(val_cases))
print("Disease candidates:", len(disease_ids))
print("HPO vocabulary:", len(phenotype_ids))
print("Parameters:", sum(p.numel() for p in ranker.parameters()))
print("Config saved:", config_path)


PATIENT RANKER READY
Device: cuda
Training cases: 5730
Validation cases: 906
Disease candidates: 7340
HPO vocabulary: 9982
Parameters: 2257580
Config saved: /content/drive/MyDrive/Rare Disease ai/model/patient_level_ranker_config.json


In [ ]:

import copy
import random
import numpy as np
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MAX_EPOCHS = 50
PATIENCE = 7
NEGATIVES_PER_CASE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(
    ranker.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Sample competing diseases; known diagnosis is excluded.
n_diseases = len(disease_ids)
def sample_negatives(target, count=NEGATIVES_PER_CASE):
    candidates = torch.randint(0, n_diseases, (count,))
    candidates = candidates[candidates != target]
    while len(candidates) < count:
        extra = torch.randint(0, n_diseases, (count - len(candidates),))
        extra = extra[extra != target]
        candidates = torch.cat([candidates, extra])
    return candidates[:count].to(device)

def evaluate(cases):
    ranker.eval()
    reciprocal_ranks, top1, top5, top10 = [], [], [], []
    with torch.no_grad():
        for case in cases:
            hpos = torch.tensor(case["hpos"], dtype=torch.long, device=device)
            scores = ranker.score_all_diseases(hpos)
            target = case["target"]
            target_score = scores[target]
            rank = int((scores > target_score).sum().item()) + 1
            reciprocal_ranks.append(1.0 / rank)
            top1.append(rank <= 1)
            top5.append(rank <= 5)
            top10.append(rank <= 10)
    return {
        "MRR": float(np.mean(reciprocal_ranks)),
        "Top1": float(np.mean(top1)),
        "Top5": float(np.mean(top5)),
        "Top10": float(np.mean(top10)),
    }

best_mrr = -1.0
best_state = None
bad_epochs = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    ranker.train()
    random.shuffle(train_cases)
    losses = []

    for case in train_cases:
        hpos = torch.tensor(case["hpos"], dtype=torch.long, device=device)
        target = case["target"]
        negatives = sample_negatives(target)

        optimizer.zero_grad(set_to_none=True)
        patient_vec = ranker.encode_patient(hpos)

        target_vec = ranker.disease_embedding.weight[target]
        neg_vecs = ranker.disease_embedding(negatives)

        target_score = (
            F.cosine_similarity(patient_vec.unsqueeze(0), target_vec.unsqueeze(0))
            + ranker.disease_bias.weight[target].squeeze()
        )
        neg_scores = (
            F.cosine_similarity(neg_vecs, patient_vec.unsqueeze(0).expand_as(neg_vecs))
            + ranker.disease_bias(negatives).squeeze(-1)
        )

        # Pairwise logistic ranking loss.
        loss = F.softplus(neg_scores - target_score).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ranker.parameters(), max_norm=2.0)
        optimizer.step()
        losses.append(float(loss.detach().item()))

    val_metrics = evaluate(val_cases)
    epoch_loss = float(np.mean(losses))
    history.append({"epoch": epoch, "loss": epoch_loss, **val_metrics})

    print(
        f"Epoch {epoch:02d} | loss={epoch_loss:.4f} "
        f"| val_MRR={val_metrics['MRR']:.4f} "
        f"| Top1={val_metrics['Top1']:.4f} "
        f"| Top5={val_metrics['Top5']:.4f} "
        f"| Top10={val_metrics['Top10']:.4f}"
    )

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        best_state = copy.deepcopy(ranker.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

if best_state is None:
    raise RuntimeError("No valid model checkpoint was produced.")

ranker.load_state_dict(best_state)

checkpoint_path = MODEL_DIR / "patient_level_ranker_best.pt"
torch.save({
    "model_state_dict": ranker.state_dict(),
    "n_diseases": len(disease_ids),
    "n_phenotypes": len(phenotype_ids),
    "hidden_dim": 128,
    "disease_ids": disease_ids,
    "phenotype_ids": phenotype_ids,
    "best_validation_mrr": best_mrr,
    "seed": SEED,
    "training_cases": len(train_cases),
    "validation_cases": len(val_cases),
    "note": (
        "Patient-level ranking model. Validation split by PMID. "
        "Unrecorded diseases used as sampled competitors are not guaranteed negatives."
    )
}, checkpoint_path)

history_path = DATA / "patient_level_training_history.csv"
import pandas as pd
pd.DataFrame(history).to_csv(history_path, index=False)

print("\nPATIENT-LEVEL TRAINING COMPLETE")
print("Best validation MRR:", round(best_mrr, 4))
print("Checkpoint saved:", checkpoint_path)
print("History saved:", history_path)


Epoch 01 | loss=0.5828 | val_MRR=0.0026 | Top1=0.0000 | Top5=0.0077 | Top10=0.0088
Epoch 02 | loss=0.3231 | val_MRR=0.0056 | Top1=0.0000 | Top5=0.0088 | Top10=0.0099
Epoch 03 | loss=0.1967 | val_MRR=0.0110 | Top1=0.0000 | Top5=0.0099 | Top10=0.0298
Epoch 04 | loss=0.1295 | val_MRR=0.0139 | Top1=0.0000 | Top5=0.0298 | Top10=0.0397
Epoch 05 | loss=0.0917 | val_MRR=0.0181 | Top1=0.0000 | Top5=0.0199 | Top10=0.0629
Epoch 06 | loss=0.0688 | val_MRR=0.0192 | Top1=0.0000 | Top5=0.0309 | Top10=0.0728
Epoch 07 | loss=0.0549 | val_MRR=0.0212 | Top1=0.0000 | Top5=0.0408 | Top10=0.0596
Epoch 08 | loss=0.0460 | val_MRR=0.0214 | Top1=0.0000 | Top5=0.0397 | Top10=0.0762
Epoch 09 | loss=0.0388 | val_MRR=0.0231 | Top1=0.0000 | Top5=0.0453 | Top10=0.0773
Epoch 10 | loss=0.0339 | val_MRR=0.0240 | Top1=0.0000 | Top5=0.0453 | Top10=0.0839
Epoch 11 | loss=0.0313 | val_MRR=0.0255 | Top1=0.0000 | Top5=0.0442 | Top10=0.0982
Epoch 12 | loss=0.0277 | val_MRR=0.0270 | Top1=0.0000 | Top5=0.0486 | Top10=0.0905
Epoc

In [ ]:

from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

history_path = DATA / "patient_level_training_history.csv"
checkpoint_path = MODEL_DIR / "patient_level_ranker_best.pt"

assert history_path.exists(), f"Missing: {history_path}"
assert checkpoint_path.exists(), f"Missing: {checkpoint_path}"

history = pd.read_csv(history_path)

print("History columns:", history.columns.tolist())
print("\nLast five history rows:")
print(history.tail(5).to_string(index=False))

# Identify the MRR column without assuming capitalization.
mrr_col = next(
    (c for c in history.columns if "mrr" in c.lower()),
    None
)
epoch_col = next(
    (c for c in history.columns if "epoch" in c.lower()),
    None
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nCHECKPOINT AUDIT")
print("Checkpoint MRR:", checkpoint.get("best_validation_mrr"))
print("Disease count:", len(checkpoint["disease_ids"]))
print("Phenotype count:", len(checkpoint["phenotype_ids"]))
print("Model parameters:", sum(
    x.numel()
    for x in checkpoint["model_state_dict"].values()
    if isinstance(x, torch.Tensor)
))

if mrr_col is not None:
    best_idx = history[mrr_col].astype(float).idxmax()
    best = history.loc[best_idx]
    print("\nBest history MRR:", float(best[mrr_col]))
    if epoch_col is not None:
        print("Best epoch:", int(best[epoch_col]))
    print(
        "Checkpoint matches history:",
        abs(float(best[mrr_col]) - float(checkpoint["best_validation_mrr"])) < 1e-6
    )
else:
    print("\nNo MRR column found. Review the displayed history columns.")


History columns: ['epoch', 'loss', 'MRR', 'Top1', 'Top5', 'Top10']

Last five history rows:
 epoch     loss      MRR     Top1     Top5    Top10
    28 0.014818 0.047371 0.011038 0.082781 0.123620
    29 0.014932 0.046306 0.011038 0.087196 0.122517
    30 0.015045 0.049286 0.011038 0.090508 0.123620
    31 0.014535 0.051332 0.011038 0.091611 0.119205
    32 0.014244 0.051033 0.011038 0.092715 0.112583

CHECKPOINT AUDIT
Checkpoint MRR: 0.052704318912176605
Disease count: 7340
Phenotype count: 9982
Model parameters: 2257580

Best history MRR: 0.0527043189121766
Best epoch: 25
Checkpoint matches history: True


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

val = pd.read_csv(DATA / "patient_level_validation.csv")
baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")

val_ids = set(val["case_id"])
baseline_val = baseline[baseline["case_id"].isin(val_ids)].copy()

print("VALIDATION BASELINE AUDIT")
print("Validation cases:", len(val))
print("Validation cases with baseline results:", baseline_val["case_id"].nunique())

if len(baseline_val):
    print("Baseline Top-1:", round(baseline_val["top1"].mean(), 4))
    print("Baseline Top-5:", round(baseline_val["top5"].mean(), 4))
    print("Baseline Top-10:", round(baseline_val["top10"].mean(), 4))
    print("Baseline coverage:", round(baseline_val["candidate_covered"].mean(), 4))
else:
    print("No validation cases found in the existing baseline file.")
    print("A baseline must be generated for validation cases before comparing methods.")

print("\nRanker validation cases:", len(val))
print("Saved ranker checkpoint:", (PROJECT / "model" / "patient_level_ranker_best.pt").exists())


VALIDATION BASELINE AUDIT
Validation cases: 906
Validation cases with baseline results: 0
No validation cases found in the existing baseline file.
A baseline must be generated for validation cases before comparing methods.

Ranker validation cases: 906
Saved ranker checkpoint: True


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
DATA = PROJECT / "processed"

keywords = [
    "phenotype_only_baseline",
    "candidate_covered",
    "hpo_id",
    "diagnosis_ids",
    "patient_phenotype"
]

print("SEARCHING PROJECT SCRIPTS")
for path in PROJECT.rglob("*.py"):
    if any(part.startswith(".") for part in path.parts):
        continue
    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue

    matches = [word for word in keywords if word.lower() in text.lower()]
    if matches:
        print(f"\nFILE: {path}")
        print("Matching terms:", ", ".join(matches))
        lines = text.splitlines()
        for i, line in enumerate(lines):
            if any(word.lower() in line.lower() for word in matches):
                start, end = max(0, i - 2), min(len(lines), i + 4)
                print(f"--- lines {start + 1}-{end} ---")
                print("\n".join(
                    f"{j + 1}: {lines[j]}" for j in range(start, end)
                ))
                print()

print("\nANNOTATION FILES")
for filename in [
    "gnn_training_annotations.csv",
    "gnn_annotations_excluding_dev_test_pmids.csv",
    "clean_hpo_annotations.csv",
]:
    path = DATA / filename
    print(filename, "FOUND" if path.exists() else "MISSING")


SEARCHING PROJECT SCRIPTS

FILE: /content/drive/MyDrive/Rare Disease ai/model/rare_disease_data_loader.py
Matching terms: hpo_id
--- lines 12-17 ---
12:         DATA / "clean_hpo_annotations.csv",
13:         low_memory=False,
14:         dtype={"database_id": "string", "hpo_id": "string"}
15:     ).dropna(subset=["database_id", "hpo_id"])
16: 
17:     ann["database_id"] = ann["database_id"].str.strip()

--- lines 13-18 ---
13:         low_memory=False,
14:         dtype={"database_id": "string", "hpo_id": "string"}
15:     ).dropna(subset=["database_id", "hpo_id"])
16: 
17:     ann["database_id"] = ann["database_id"].str.strip()
18:     ann["hpo_id"] = ann["hpo_id"].str.strip()

--- lines 16-21 ---
16: 
17:     ann["database_id"] = ann["database_id"].str.strip()
18:     ann["hpo_id"] = ann["hpo_id"].str.strip()
19: 
20:     phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
21: 

--- lines 18-23 ---
18:     ann["hpo_id"] = ann["hpo_id"].str.strip()
19: 
20:     

In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

for filename in [
    "rare_disease_data_loader.py",
    "rare_disease_predictor.py",
    "rare_disease_ai.py",
    "prediction_config.json",
]:
    path = MODEL_DIR / filename
    print(f"\n{'=' * 18} {filename} {'=' * 18}")

    if not path.exists():
        print("File not found")
        continue

    content = path.read_text(errors="ignore")
    print(content[:18000])



================== rare_disease_data_loader.py ==================

from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

def load_prediction_data():
    ann = pd.read_csv(
        DATA / "clean_hpo_annotations.csv",
        low_memory=False,
        dtype={"database_id": "string", "hpo_id": "string"}
    ).dropna(subset=["database_id", "hpo_id"])

    ann["database_id"] = ann["database_id"].str.strip()
    ann["hpo_id"] = ann["hpo_id"].str.strip()

    phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

    gene_df = pd.read_csv(
        DATA / "normalized_disease_gene_mapping.csv",
        dtype=str
    ).fillna("")

    gene_map = {
        row["disease_id"]: {
            g.strip().upper()
            for g in row["gene_symbols"].split(";")
            if g.strip()
        }
        for _, row in gene_df.iterrows()
    }

    s

In [ ]:

from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

# Load the same phenotype annotation source used by the original baseline.
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

# Reproduce the original information-content calculation.
split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"]
)
counts = defaultdict(int)
for disease in train_ids:
    for term in phenotype_map.get(disease, set()):
        counts[term] += 1

n_train = max(len(train_ids), 1)
ic = {
    term: -np.log(count / n_train)
    for term, count in counts.items() if count > 0
}

# Load validation cases only.
val = pd.read_csv(DATA / "patient_level_validation.csv")

def parse_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split(";") if x.strip()]

rows = []
disease_candidates = sorted(phenotype_map)

for row in val.itertuples(index=False):
    observed = {
        h for h in parse_ids(row.hpo_ids)
        if h in ic
    }
    targets = parse_ids(row.diagnosis_ids)

    result = {
        "case_id": row.case_id,
        "pmid": str(row.pmid),
        "diagnosis_ids": ";".join(targets),
        "n_recognized_hpo": len(observed),
        "target_in_candidate_set": any(t in phenotype_map for t in targets),
        "rank": np.nan,
        "top1": 0,
        "top5": 0,
        "top10": 0,
        "mrr": 0.0,
    }

    if observed:
        weights = {h: ic[h] for h in observed}
        denominator = sum(weights.values()) or 1.0
        scored = []

        for disease in disease_candidates:
            disease_terms = phenotype_map[disease]
            overlap = observed & disease_terms

            precision = sum(weights[h] for h in overlap) / denominator
            recall = len(overlap) / max(len(disease_terms), 1)
            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall > 0 else 0.0
            )
            scored.append((disease, f1, len(overlap)))

        scored.sort(key=lambda x: (-x[1], -x[2], x[0]))
        ranked_ids = [x[0] for x in scored]
        target_ranks = [
            ranked_ids.index(t) + 1 for t in targets if t in ranked_ids
        ]

        if target_ranks:
            rank = min(target_ranks)
            result.update({
                "rank": rank,
                "top1": int(rank <= 1),
                "top5": int(rank <= 5),
                "top10": int(rank <= 10),
                "mrr": 1.0 / rank,
            })

    rows.append(result)

out = pd.DataFrame(rows)
out_path = DATA / "patient_phenotype_only_validation.csv"
out.to_csv(out_path, index=False)

covered = out["rank"].notna()
print("VALIDATION BASELINE COMPLETE")
print("Validation cases:", len(out))
print("Cases with recognized HPO terms:", int((out["n_recognized_hpo"] > 0).sum()))
print("Targets in baseline candidate set:", int(out["target_in_candidate_set"].sum()))
print("Cases with target rank:", int(covered.sum()))
if covered.any():
    print("Top-1:", round(out.loc[covered, "top1"].mean(), 4))
    print("Top-5:", round(out.loc[covered, "top5"].mean(), 4))
    print("Top-10:", round(out.loc[covered, "top10"].mean(), 4))
    print("MRR:", round(out.loc[covered, "mrr"].mean(), 4))
print("Saved:", out_path)


VALIDATION BASELINE COMPLETE
Validation cases: 906
Cases with recognized HPO terms: 906
Targets in baseline candidate set: 906
Cases with target rank: 906
Top-1: 0.2638
Top-5: 0.4073
Top-10: 0.489
MRR: 0.336
Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_phenotype_only_validation.csv


In [ ]:

from pathlib import Path
import json
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

# 1. Verify project directories and key artifacts
print("=== 1. PROJECT / ARTIFACT CHECK ===")
for p in [
    DATA / "patient_level_validation.csv",
    DATA / "patient_phenotype_only_validation.csv",
    DATA / "patient_level_training_history.csv",
    MODEL / "patient_level_ranker_best.pt",
    MODEL / "patient_level_ranker_config.json",
]:
    print(f"{'OK' if p.exists() else 'MISSING'}  {p.name}")

# 2. Read model config
print("\n=== 2. RANKER CONFIG ===")
config_path = MODEL / "patient_level_ranker_config.json"
if config_path.exists():
    print(json.dumps(json.loads(config_path.read_text()), indent=2))
else:
    print("Config not found.")

# 3. Inspect checkpoint without training
print("\n=== 3. CHECKPOINT INSPECTION ===")
ckpt_path = MODEL / "patient_level_ranker_best.pt"
if ckpt_path.exists():
    try:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location="cpu")

    if isinstance(ckpt, dict):
        print("Keys:", list(ckpt.keys()))
        for key, value in ckpt.items():
            if torch.is_tensor(value):
                print(f"{key}: tensor {tuple(value.shape)}")
            elif isinstance(value, dict):
                print(f"{key}: dict ({len(value)} entries)")
            elif isinstance(value, (str, int, float, list, tuple)):
                print(f"{key}: {str(value)[:300]}")
    else:
        print("Checkpoint object type:", type(ckpt).__name__)
else:
    print("Checkpoint not found.")

# 4. Summarize validation baseline and training history
print("\n=== 4. VALIDATION METRICS / HISTORY ===")
baseline_path = DATA / "patient_phenotype_only_validation.csv"
if baseline_path.exists():
    baseline = pd.read_csv(baseline_path)
    print("Baseline cases:", len(baseline))
    print("Cases with target rank:", int(baseline["rank"].notna().sum()))
    for col in ["top1", "top5", "top10", "mrr"]:
        if col in baseline.columns:
            print(f"{col}: {baseline[col].mean():.4f}")

history_path = DATA / "patient_level_training_history.csv"
if history_path.exists():
    history = pd.read_csv(history_path)
    print("\nHistory shape:", history.shape)
    print("Columns:", history.columns.tolist())
    if "MRR" in history.columns:
        best_idx = history["MRR"].idxmax()
        print("Best history row:")
        print(history.loc[best_idx].to_dict())
else:
    print("Training history not found.")

print("\nGPU available now:", torch.cuda.is_available())
print("Inspection complete — no training performed.")


=== 1. PROJECT / ARTIFACT CHECK ===
OK  patient_level_validation.csv
OK  patient_phenotype_only_validation.csv
OK  patient_level_training_history.csv
OK  patient_level_ranker_best.pt
OK  patient_level_ranker_config.json

=== 2. RANKER CONFIG ===
{
  "model": "PatientDiseaseRanker",
  "dimension": 128,
  "dropout": 0.2,
  "seed": 42,
  "training_cases": 5730,
  "validation_cases": 906,
  "training_objective": "patient-to-disease ranking with sampled competing diseases",
  "device": "cuda",
  "test_set_used_for_training_or_validation": false
}

=== 3. CHECKPOINT INSPECTION ===
Keys: ['model_state_dict', 'n_diseases', 'n_phenotypes', 'hidden_dim', 'disease_ids', 'phenotype_ids', 'best_validation_mrr', 'seed', 'training_cases', 'validation_cases', 'note']
model_state_dict: dict (7 entries)
n_diseases: 7340
n_phenotypes: 9982
hidden_dim: 128
disease_ids: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400', 'OMIM:101600', 'OMIM:101800', 'OMIM:102200', 'OMIM:102370', 'O

In [ ]:

from pathlib import Path
from collections import defaultdict
import json
import numpy as np
import pandas as pd
import torch
from torch import nn

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA, MODEL_DIR = PROJECT / "processed", PROJECT / "model"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load checkpoint, validation cases, and baseline annotations
ckpt = torch.load(
    MODEL_DIR / "patient_level_ranker_best.pt",
    map_location="cpu",
    weights_only=False
)
val = pd.read_csv(DATA / "patient_level_validation.csv")
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": "string", "hpo_id": "string"},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

split = pd.read_csv(DATA / "disease_level_split.csv", dtype={"disease_id": "string"})
train_ids = set(split.loc[split["split"].str.lower() == "train", "disease_id"])
counts = defaultdict(int)
for d in train_ids:
    for h in phenotype_map.get(d, set()):
        counts[h] += 1
n_train = max(len(train_ids), 1)
ic = {h: -np.log(c / n_train) for h, c in counts.items() if c > 0}

def ids(value):
    if pd.isna(value):
        return []
    return [x.strip() for x in str(value).replace(",", ";").split(";") if x.strip()]

# 2. Rebuild the saved patient-level ranker architecture and load weights
class PatientDiseaseRanker(nn.Module):
    def __init__(self, n_hpo, n_disease, dim=128):
        super().__init__()
        self.hpo_embedding = nn.Embedding(n_hpo, dim)
        self.disease_embedding = nn.Embedding(n_disease, dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(dim, dim)
        )
        self.disease_bias = nn.Embedding(n_disease, 1)

    def score_all_diseases(self, hpo_indices):
        patient = self.hpo_embedding(hpo_indices).mean(dim=0)
        patient = self.patient_projection(patient)
        disease = self.disease_embedding.weight
        scores = disease @ patient / (
            patient.norm() * disease.norm(dim=1).clamp_min(1e-8) + 1e-8
        )
        return scores + self.disease_bias.weight.squeeze(-1)

disease_ids = list(ckpt["disease_ids"])
hpo_ids = list(ckpt["phenotype_ids"])
disease_index = {d: i for i, d in enumerate(disease_ids)}
hpo_index = {h: i for i, h in enumerate(hpo_ids)}

ranker = PatientDiseaseRanker(
    ckpt["n_phenotypes"], ckpt["n_diseases"], ckpt["hidden_dim"]
)
ranker.load_state_dict(ckpt["model_state_dict"])
ranker.to(DEVICE).eval()

baseline_candidates = set(phenotype_map)
ranker_candidates = set(disease_ids)
common_candidates = baseline_candidates & ranker_candidates
print("=== CANDIDATE SET AUDIT ===")
print("Baseline candidates:", len(baseline_candidates))
print("Ranker candidates:", len(ranker_candidates))
print("Shared candidates:", len(common_candidates))
print("Baseline-only:", len(baseline_candidates - ranker_candidates))
print("Ranker-only:", len(ranker_candidates - baseline_candidates))

# 3. Score each validation case using both models and the shared candidate set
records = []
with torch.no_grad():
    for row in val.itertuples(index=False):
        observed = set(ids(row.hpo_ids))
        targets = set(ids(row.diagnosis_ids))
        baseline_hpos = observed & set(ic)
        ranker_hpos = [hpo_index[h] for h in observed if h in hpo_index]

        base_scores = {}
        if baseline_hpos:
            weights = {h: ic[h] for h in baseline_hpos}
            denom = sum(weights.values()) or 1.0
            for d in baseline_candidates:
                terms = phenotype_map[d]
                overlap = baseline_hpos & terms
                precision = sum(weights[h] for h in overlap) / denom
                recall = len(overlap) / max(len(terms), 1)
                f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
                base_scores[d] = (f1, len(overlap))

        rank_scores = {}
        if ranker_hpos:
            tensor = torch.tensor(ranker_hpos, dtype=torch.long, device=DEVICE)
            scores = ranker.score_all_diseases(tensor).detach().cpu().numpy()
            rank_scores = {d: float(scores[i]) for i, d in enumerate(disease_ids)}

        # Match original tie-breaking rules for baseline; ranker ties use disease ID.
        def rank_targets(score_map, tie_map=None):
            ordered = sorted(
                score_map,
                key=lambda d: (
                    -score_map[d][0] if tie_map is not None else -score_map[d],
                    -(score_map[d][1]) if tie_map is not None else 0,
                    d
                )
            )
            positions = {d: i + 1 for i, d in enumerate(ordered)}
            found = [positions[t] for t in targets if t in positions]
            return min(found) if found else np.nan

        base_native = rank_targets(base_scores, tie_map=True) if base_scores else np.nan
        rank_native = rank_targets(rank_scores) if rank_scores else np.nan

        base_common = {d: base_scores[d] for d in common_candidates if d in base_scores}
        rank_common = {d: rank_scores[d] for d in common_candidates if d in rank_scores}
        base_common_rank = rank_targets(base_common, tie_map=True) if base_common else np.nan
        rank_common_rank = rank_targets(rank_common) if rank_common else np.nan

        records.append({
            "case_id": row.case_id,
            "pmid": str(row.pmid),
            "diagnosis_ids": ";".join(sorted(targets)),
            "n_observed_hpo": len(observed),
            "baseline_native_rank": base_native,
            "ranker_native_rank": rank_native,
            "baseline_common_rank": base_common_rank,
            "ranker_common_rank": rank_common_rank,
            "target_in_baseline_vocab": int(any(t in baseline_candidates for t in targets)),
            "target_in_ranker_vocab": int(any(t in ranker_candidates for t in targets)),
            "target_in_common_vocab": int(any(t in common_candidates for t in targets)),
        })

results = pd.DataFrame(records)
out_path = DATA / "patient_validation_matched_candidate_comparison.csv"
results.to_csv(out_path, index=False)

# 4. Summarize coverage and ranking metrics; report only cases with a rank
def metrics(series):
    ranks = pd.to_numeric(series, errors="coerce").dropna()
    if len(ranks) == 0:
        return {"n_ranked": 0, "Top1": np.nan, "Top5": np.nan, "Top10": np.nan, "MRR": np.nan}
    return {
        "n_ranked": int(len(ranks)),
        "Top1": round(float((ranks <= 1).mean()), 4),
        "Top5": round(float((ranks <= 5).mean()), 4),
        "Top10": round(float((ranks <= 10).mean()), 4),
        "MRR": round(float((1 / ranks).mean()), 4),
    }

summary = {
    "device": str(DEVICE),
    "validation_cases": int(len(results)),
    "target_coverage": {
        "baseline_vocab": int(results["target_in_baseline_vocab"].sum()),
        "ranker_vocab": int(results["target_in_ranker_vocab"].sum()),
        "shared_vocab": int(results["target_in_common_vocab"].sum()),
    },
    "baseline_native": metrics(results["baseline_native_rank"]),
    "ranker_native": metrics(results["ranker_native_rank"]),
    "baseline_shared_candidates": metrics(results["baseline_common_rank"]),
    "ranker_shared_candidates": metrics(results["ranker_common_rank"]),
    "saved_predictions": str(out_path),
}
summary_path = DATA / "patient_validation_matched_candidate_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("\n=== VALIDATION COMPARISON ===")
print(json.dumps(summary, indent=2))
print("\nSaved predictions:", out_path)
print("Saved summary:", summary_path)


=== CANDIDATE SET AUDIT ===
Baseline candidates: 12879
Ranker candidates: 7340
Shared candidates: 7340
Baseline-only: 5539
Ranker-only: 0

=== VALIDATION COMPARISON ===
{
  "device": "cuda",
  "validation_cases": 906,
  "target_coverage": {
    "baseline_vocab": 906,
    "ranker_vocab": 906,
    "shared_vocab": 906
  },
  "baseline_native": {
    "n_ranked": 906,
    "Top1": 0.2638,
    "Top5": 0.4073,
    "Top10": 0.489,
    "MRR": 0.336
  },
  "ranker_native": {
    "n_ranked": 906,
    "Top1": 0.021,
    "Top5": 0.0872,
    "Top10": 0.1104,
    "MRR": 0.0527
  },
  "baseline_shared_candidates": {
    "n_ranked": 906,
    "Top1": 0.2947,
    "Top5": 0.468,
    "Top10": 0.5419,
    "MRR": 0.3773
  },
  "ranker_shared_candidates": {
    "n_ranked": 906,
    "Top1": 0.021,
    "Top5": 0.0872,
    "Top10": 0.1104,
    "MRR": 0.0527
  },
  "saved_predictions": "/content/drive/MyDrive/Rare Disease ai/processed/patient_validation_matched_candidate_comparison.csv"
}

Saved predictions: /cont

In [ ]:

from pathlib import Path
from collections import Counter
import json
import numpy as np
import pandas as pd
import torch
from torch import nn

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA, MODEL_DIR = PROJECT / "processed", PROJECT / "model"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Training history and saved validation ranks
history = pd.read_csv(DATA / "patient_level_training_history.csv")
comparison = pd.read_csv(DATA / "patient_validation_matched_candidate_comparison.csv")
ckpt = torch.load(
    MODEL_DIR / "patient_level_ranker_best.pt",
    map_location="cpu",
    weights_only=False
)

print("=== 1. TRAINING HISTORY ===")
print(history.iloc[[0, history["MRR"].idxmax(), len(history)-1]].to_string(index=False))
print("\nBest epoch:", int(history.loc[history["MRR"].idxmax(), "epoch"]))
print("Last epoch:", int(history.iloc[-1]["epoch"]))
print("Loss: first =", round(float(history.iloc[0]["loss"]), 5),
      "| best-MRR epoch =", round(float(history.loc[history["MRR"].idxmax(), "loss"]), 5))

print("\n=== 2. RANK DISTRIBUTION ===")
ranks = pd.to_numeric(comparison["ranker_common_rank"], errors="coerce").dropna()
print("Ranked cases:", len(ranks))
print("Rank 1:", int((ranks == 1).sum()))
print("Ranks 2–5:", int(ranks.between(2, 5).sum()))
print("Ranks 6–10:", int(ranks.between(6, 10).sum()))
print("Ranks 11–100:", int(ranks.between(11, 100).sum()))
print("Ranks >100:", int((ranks > 100).sum()))
print("Median target rank:", float(ranks.median()))

# Rebuild model exactly as saved
class PatientDiseaseRanker(nn.Module):
    def __init__(self, n_hpo, n_disease, dim=128):
        super().__init__()
        self.hpo_embedding = nn.Embedding(n_hpo, dim)
        self.disease_embedding = nn.Embedding(n_disease, dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(dim, dim)
        )
        self.disease_bias = nn.Embedding(n_disease, 1)

    def score_all_diseases(self, hpo_indices):
        patient = self.hpo_embedding(hpo_indices).mean(dim=0)
        patient = self.patient_projection(patient)
        disease = self.disease_embedding.weight
        scores = disease @ patient / (
            patient.norm() * disease.norm(dim=1).clamp_min(1e-8) + 1e-8
        )
        return scores + self.disease_bias.weight.squeeze(-1)

disease_ids = list(ckpt["disease_ids"])
hpo_ids = list(ckpt["phenotype_ids"])
disease_idx = {d: i for i, d in enumerate(disease_ids)}
hpo_idx = {h: i for i, h in enumerate(hpo_ids)}

model = PatientDiseaseRanker(
    ckpt["n_phenotypes"], ckpt["n_diseases"], ckpt["hidden_dim"]
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()

# 3. Top-prediction frequency and score/bias diagnostics
val = pd.read_csv(DATA / "patient_level_validation.csv")
top_counts, all_top_scores, target_scores, target_biases = Counter(), [], [], []
bias = model.disease_bias.weight.detach().cpu().numpy().ravel()
emb = model.disease_embedding.weight.detach()
bias_order = np.argsort(-bias)

with torch.no_grad():
    for row in val.itertuples(index=False):
        hs = [hpo_idx[h.strip()] for h in str(row.hpo_ids).replace(",", ";").split(";")
              if h.strip() in hpo_idx]
        if not hs:
            continue
        scores = model.score_all_diseases(
            torch.tensor(hs, dtype=torch.long, device=device)
        ).detach().cpu().numpy()
        top_i = np.argsort(-scores)[:10]
        top_counts.update(disease_ids[i] for i in top_i[:1])
        all_top_scores.append(float(scores[top_i[0]]))

        targets = [x.strip() for x in str(row.diagnosis_ids).replace(",", ";").split(";")
                   if x.strip()]
        valid = [disease_idx[t] for t in targets if t in disease_idx]
        if valid:
            ti = valid[0]
            target_scores.append(float(scores[ti]))
            target_biases.append(float(bias[ti]))

print("\n=== 3. MOST FREQUENT TOP-1 PREDICTIONS ===")
print(pd.DataFrame(top_counts.most_common(15), columns=["disease_id", "top1_count"]).to_string(index=False))
print("Unique diseases ever predicted Top-1:", len(top_counts))
print("Validation cases scored:", len(all_top_scores))

print("\n=== 4. DISEASE BIAS / SCORE CHECK ===")
print("Bias min / median / max:",
      round(float(np.min(bias)), 4),
      round(float(np.median(bias)), 4),
      round(float(np.max(bias)), 4))
print("Top 10 diseases by learned bias:")
print(pd.DataFrame({
    "disease_id": [disease_ids[i] for i in bias_order[:10]],
    "bias": bias[bias_order[:10]]
}).to_string(index=False))
if target_scores:
    print("Correct-target raw score mean / median:",
          round(float(np.mean(target_scores)), 4),
          round(float(np.median(target_scores)), 4))
    print("Correct-target bias mean / median:",
          round(float(np.mean(target_biases)), 4),
          round(float(np.median(target_biases)), 4))
print("Top prediction score mean:", round(float(np.mean(all_top_scores)), 4))

report = {
    "best_epoch": int(history.loc[history["MRR"].idxmax(), "epoch"]),
    "best_validation_mrr": float(history["MRR"].max()),
    "median_target_rank": float(ranks.median()) if len(ranks) else None,
    "top1_unique_diseases": len(top_counts),
    "top1_frequency": dict(top_counts.most_common(30)),
    "bias_min_median_max": [float(np.min(bias)), float(np.median(bias)), float(np.max(bias))],
    "target_score_mean": float(np.mean(target_scores)) if target_scores else None,
    "target_bias_mean": float(np.mean(target_biases)) if target_biases else None,
}
report_path = DATA / "patient_ranker_diagnostics.json"
report_path.write_text(json.dumps(report, indent=2))
print("\nSaved diagnostic report:", report_path)


=== 1. TRAINING HISTORY ===
 epoch     loss      MRR     Top1     Top5    Top10
     1 0.582818 0.002557 0.000000 0.007726 0.008830
    25 0.015694 0.052704 0.020971 0.087196 0.110375
    32 0.014244 0.051033 0.011038 0.092715 0.112583

Best epoch: 25
Last epoch: 32
Loss: first = 0.58282 | best-MRR epoch = 0.01569

=== 2. RANK DISTRIBUTION ===
Ranked cases: 906
Rank 1: 19
Ranks 2–5: 60
Ranks 6–10: 21
Ranks 11–100: 230
Ranks >100: 576
Median target rank: 450.0

=== 3. MOST FREQUENT TOP-1 PREDICTIONS ===
 disease_id  top1_count
OMIM:612164         637
OMIM:148050         249
OMIM:162000          11
OMIM:142900           9
Unique diseases ever predicted Top-1: 4
Validation cases scored: 906

=== 4. DISEASE BIAS / SCORE CHECK ===
Bias min / median / max: -8.3389 -4.6719 4.4087
Top 10 diseases by learned bias:
 disease_id     bias
OMIM:612164 4.408734
OMIM:148050 4.129399
OMIM:162000 3.457870
OMIM:610253 3.321647
OMIM:616364 3.303920
OMIM:142900 3.267206
OMIM:615471 3.169854
OMIM:117360 3.0

In [ ]:

from pathlib import Path
from collections import Counter
import json
import numpy as np
import pandas as pd
import torch
from torch import nn

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA, MODEL_DIR = PROJECT / "processed", PROJECT / "model"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    MODEL_DIR / "patient_level_ranker_best.pt",
    map_location="cpu",
    weights_only=False
)
val = pd.read_csv(DATA / "patient_level_validation.csv")
disease_ids = list(ckpt["disease_ids"])
hpo_ids = list(ckpt["phenotype_ids"])
d_idx = {d: i for i, d in enumerate(disease_ids)}
h_idx = {h: i for i, h in enumerate(hpo_ids)}

class PatientDiseaseRanker(nn.Module):
    def __init__(self, n_hpo, n_disease, dim=128):
        super().__init__()
        self.hpo_embedding = nn.Embedding(n_hpo, dim)
        self.disease_embedding = nn.Embedding(n_disease, dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(dim, dim)
        )
        self.disease_bias = nn.Embedding(n_disease, 1)

    def components(self, hpo_indices):
        patient = self.hpo_embedding(hpo_indices).mean(dim=0)
        patient = self.patient_projection(patient)
        disease = self.disease_embedding.weight
        similarity = disease @ patient / (
            patient.norm() * disease.norm(dim=1).clamp_min(1e-8) + 1e-8
        )
        bias = self.disease_bias.weight.squeeze(-1)
        return similarity, bias

model = PatientDiseaseRanker(
    ckpt["n_phenotypes"], ckpt["n_diseases"], ckpt["hidden_dim"]
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()

def parse_ids(value):
    if pd.isna(value):
        return []
    return [x.strip() for x in str(value).replace(",", ";").split(";") if x.strip()]

rank_records = []
top_counts = {name: Counter() for name in ["original", "no_bias", "bias_only"]}

with torch.no_grad():
    for row in val.itertuples(index=False):
        observed = [h_idx[h] for h in parse_ids(row.hpo_ids) if h in h_idx]
        targets = set(parse_ids(row.diagnosis_ids))
        if not observed:
            continue

        similarity, bias = model.components(
            torch.tensor(observed, dtype=torch.long, device=device)
        )
        similarity = similarity.detach().cpu().numpy()
        bias = bias.detach().cpu().numpy()

        scores_by_method = {
            "original": similarity + bias,
            "no_bias": similarity,
            "bias_only": bias,
        }

        record = {"case_id": row.case_id, "diagnosis_ids": ";".join(sorted(targets))}
        for method, scores in scores_by_method.items():
            order = np.argsort(-scores, kind="stable")
            ranked_ids = [disease_ids[i] for i in order]
            top_counts[method][ranked_ids[0]] += 1
            positions = {d: i + 1 for i, d in enumerate(ranked_ids)}
            target_ranks = [positions[t] for t in targets if t in positions]
            rank = min(target_ranks) if target_ranks else np.nan
            record[f"{method}_rank"] = rank
            record[f"{method}_top1_disease"] = ranked_ids[0]
        rank_records.append(record)

out = pd.DataFrame(rank_records)
out_path = DATA / "patient_ranker_bias_ablation.csv"
out.to_csv(out_path, index=False)

def summarize(column):
    ranks = pd.to_numeric(out[column], errors="coerce").dropna()
    return {
        "n_ranked": int(len(ranks)),
        "Top1": round(float((ranks <= 1).mean()), 4) if len(ranks) else None,
        "Top5": round(float((ranks <= 5).mean()), 4) if len(ranks) else None,
        "Top10": round(float((ranks <= 10).mean()), 4) if len(ranks) else None,
        "MRR": round(float((1 / ranks).mean()), 4) if len(ranks) else None,
        "median_rank": float(ranks.median()) if len(ranks) else None,
    }

summary = {
    "device": str(device),
    "validation_cases_scored": len(out),
    "metrics": {
        "original": summarize("original_rank"),
        "no_bias": summarize("no_bias_rank"),
        "bias_only": summarize("bias_only_rank"),
    },
    "unique_top1_diseases": {
        method: len(counter) for method, counter in top_counts.items()
    },
    "most_common_top1": {
        method: counter.most_common(10) for method, counter in top_counts.items()
    },
    "saved_predictions": str(out_path),
}
summary_path = DATA / "patient_ranker_bias_ablation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print("\nSaved:", out_path)
print("Saved:", summary_path)


{
  "device": "cuda",
  "validation_cases_scored": 906,
  "metrics": {
    "original": {
      "n_ranked": 906,
      "Top1": 0.021,
      "Top5": 0.0872,
      "Top10": 0.1104,
      "MRR": 0.0527,
      "median_rank": 450.0
    },
    "no_bias": {
      "n_ranked": 906,
      "Top1": 0.2671,
      "Top5": 0.3444,
      "Top10": 0.3653,
      "MRR": 0.3043,
      "median_rank": 232.5
    },
    "bias_only": {
      "n_ranked": 906,
      "Top1": 0.0,
      "Top5": 0.0166,
      "Top10": 0.0442,
      "MRR": 0.0154,
      "median_rank": 477.0
    }
  },
  "unique_top1_diseases": {
    "original": 4,
    "no_bias": 196,
    "bias_only": 1
  },
  "most_common_top1": {
    "original": [
      [
        "OMIM:612164",
        637
      ],
      [
        "OMIM:148050",
        249
      ],
      [
        "OMIM:162000",
        11
      ],
      [
        "OMIM:142900",
        9
      ]
    ],
    "no_bias": [
      [
        "OMIM:151660",
        75
      ],
      [
        "OMIM:162000

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

base = pd.read_csv(DATA / "patient_validation_matched_candidate_comparison.csv")
ablation = pd.read_csv(DATA / "patient_ranker_bias_ablation.csv")

df = base[["case_id", "baseline_common_rank"]].merge(
    ablation[["case_id", "no_bias_rank"]],
    on="case_id",
    how="inner",
    validate="one_to_one"
)

df["baseline_common_rank"] = pd.to_numeric(df["baseline_common_rank"], errors="coerce")
df["no_bias_rank"] = pd.to_numeric(df["no_bias_rank"], errors="coerce")
df = df.dropna(subset=["baseline_common_rank", "no_bias_rank"]).reset_index(drop=True)

br = df["baseline_common_rank"].to_numpy(dtype=float)
nr = df["no_bias_rank"].to_numpy(dtype=float)
n = len(df)

if n < 100:
    raise ValueError(f"Only {n} matched cases found; inspect saved validation files.")

def metric_vector(r):
    return np.array([
        np.mean(r <= 1),
        np.mean(r <= 5),
        np.mean(r <= 10),
        np.mean(1.0 / r)
    ])

labels = ["Top1", "Top5", "Top10", "MRR"]
baseline_metrics = metric_vector(br)
ranker_metrics = metric_vector(nr)
observed_diff = ranker_metrics - baseline_metrics

# Case-level paired bootstrap; no model fitting or GPU required.
rng = np.random.default_rng(42)
B = 3000
boot_diffs = np.empty((B, 4))
for b in range(B):
    ix = rng.integers(0, n, size=n)
    boot_diffs[b] = metric_vector(nr[ix]) - metric_vector(br[ix])

ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5], axis=0)

print("=== PAIRED VALIDATION COMPARISON ===")
print("Matched cases:", n)
print("Positive difference means bias-free ranker is better.")
report_metrics = {}
for i, name in enumerate(labels):
    report_metrics[name] = {
        "baseline": round(float(baseline_metrics[i]), 4),
        "ranker_no_bias": round(float(ranker_metrics[i]), 4),
        "difference_ranker_minus_baseline": round(float(observed_diff[i]), 4),
        "bootstrap_95pct_CI_difference": [
            round(float(ci_low[i]), 4),
            round(float(ci_high[i]), 4)
        ]
    }
    print(
        f"{name}: baseline={baseline_metrics[i]:.4f}, "
        f"no_bias={ranker_metrics[i]:.4f}, "
        f"delta={observed_diff[i]:+.4f}, "
        f"95% CI [{ci_low[i]:+.4f}, {ci_high[i]:+.4f}]"
    )

# Paired patient-level wins by reciprocal rank.
base_rr, ranker_rr = 1 / br, 1 / nr
wins = int((ranker_rr > base_rr).sum())
ties = int((ranker_rr == base_rr).sum())
losses = int((ranker_rr < base_rr).sum())
print("\nReciprocal-rank patient comparison:")
print("Ranker wins:", wins, f"({wins/n:.1%})")
print("Ties:", ties, f"({ties/n:.1%})")
print("Baseline wins:", losses, f"({losses/n:.1%})")

report = {
    "matched_validation_cases": n,
    "bootstrap_replicates": B,
    "seed": 42,
    "metrics": report_metrics,
    "per_case_reciprocal_rank": {
        "ranker_wins": wins,
        "ties": ties,
        "baseline_wins": losses
    },
    "interpretation_note": (
        "Validation-only paired bootstrap; confidence intervals quantify "
        "case-sampling uncertainty, not external clinical generalizability."
    )
}
path = DATA / "patient_ranker_vs_baseline_paired_bootstrap.json"
path.write_text(json.dumps(report, indent=2))
print("\nSaved report:", path)


=== PAIRED VALIDATION COMPARISON ===
Matched cases: 906
Positive difference means bias-free ranker is better.
Top1: baseline=0.2947, no_bias=0.2671, delta=-0.0276, 95% CI [-0.0717, +0.0188]
Top5: baseline=0.4680, no_bias=0.3444, delta=-0.1236, 95% CI [-0.1744, -0.0717]
Top10: baseline=0.5419, no_bias=0.3653, delta=-0.1766, 95% CI [-0.2285, -0.1247]
MRR: baseline=0.3773, no_bias=0.3043, delta=-0.0730, 95% CI [-0.1169, -0.0279]

Reciprocal-rank patient comparison:
Ranker wins: 316 (34.9%)
Ties: 35 (3.9%)
Baseline wins: 555 (61.3%)

Saved report: /content/drive/MyDrive/Rare Disease ai/processed/patient_ranker_vs_baseline_paired_bootstrap.json


In [ ]:

from pathlib import Path
from collections import defaultdict
import json
import numpy as np
import pandas as pd
import torch
from torch import nn

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA, MODEL_DIR = PROJECT / "processed", PROJECT / "model"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load validation data and baseline annotations
val = pd.read_csv(DATA / "patient_level_validation.csv")
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": "string", "hpo_id": "string"},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

# Reproduce baseline information content
split = pd.read_csv(DATA / "disease_level_split.csv", dtype={"disease_id": "string"})
train_ids = set(split.loc[split["split"].str.lower() == "train", "disease_id"])
counts = defaultdict(int)
for d in train_ids:
    for h in phenotype_map.get(d, set()):
        counts[h] += 1
n_train = max(len(train_ids), 1)
ic = {h: -np.log(c / n_train) for h, c in counts.items() if c > 0}

# Load saved ranker; do not train
ckpt = torch.load(
    MODEL_DIR / "patient_level_ranker_best.pt",
    map_location="cpu",
    weights_only=False
)
disease_ids = list(ckpt["disease_ids"])
hpo_ids = list(ckpt["phenotype_ids"])
d_idx = {d: i for i, d in enumerate(disease_ids)}
h_idx = {h: i for i, h in enumerate(hpo_ids)}
candidates = sorted(set(disease_ids) & set(phenotype_map))
candidate_idx = np.array([d_idx[d] for d in candidates], dtype=int)

class PatientDiseaseRanker(nn.Module):
    def __init__(self, n_hpo, n_disease, dim=128):
        super().__init__()
        self.hpo_embedding = nn.Embedding(n_hpo, dim)
        self.disease_embedding = nn.Embedding(n_disease, dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(dim, dim)
        )
        self.disease_bias = nn.Embedding(n_disease, 1)

    def score_without_bias(self, hpo_indices):
        patient = self.hpo_embedding(hpo_indices).mean(dim=0)
        patient = self.patient_projection(patient)
        disease = self.disease_embedding.weight
        return disease @ patient / (
            patient.norm() * disease.norm(dim=1).clamp_min(1e-8) + 1e-8
        )

model = PatientDiseaseRanker(
    ckpt["n_phenotypes"], ckpt["n_diseases"], ckpt["hidden_dim"]
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()

def parse_ids(x):
    if pd.isna(x):
        return []
    return [z.strip() for z in str(x).replace(",", ";").split(";") if z.strip()]

def percentile_utility(scores):
    # 1 = best; convert scores to [0, 1], higher is better.
    order = np.argsort(np.argsort(-scores, kind="stable"), kind="stable")
    return 1.0 - order / max(len(scores) - 1, 1)

all_records = []
weights_to_test = np.round(np.arange(0.0, 1.01, 0.1), 2)

with torch.no_grad():
    for row in val.itertuples(index=False):
        observed = set(parse_ids(row.hpo_ids))
        targets = set(parse_ids(row.diagnosis_ids))
        recognized = observed & set(ic)

        if not recognized:
            continue

        weights = {h: ic[h] for h in recognized}
        denom = sum(weights.values()) or 1.0
        baseline_scores = []

        for d in candidates:
            terms = phenotype_map[d]
            overlap = recognized & terms
            precision = sum(weights[h] for h in overlap) / denom
            recall = len(overlap) / max(len(terms), 1)
            f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
            baseline_scores.append(f1)

        baseline_scores = np.asarray(baseline_scores, dtype=float)

        hs = [h_idx[h] for h in observed if h in h_idx]
        if not hs:
            continue
        ranker_scores = model.score_without_bias(
            torch.tensor(hs, dtype=torch.long, device=DEVICE)
        ).detach().cpu().numpy()[candidate_idx]

        b_utility = percentile_utility(baseline_scores)
        r_utility = percentile_utility(ranker_scores)

        # Candidate targets are guaranteed to be present in both vocabularies
        target_positions = [i for i, d in enumerate(candidates) if d in targets]
        if not target_positions:
            continue

        for alpha in weights_to_test:
            combined = (1.0 - alpha) * b_utility + alpha * r_utility
            order = np.argsort(-combined, kind="stable")
            inverse_rank = np.empty(len(order), dtype=int)
            inverse_rank[order] = np.arange(1, len(order) + 1)
            target_rank = int(min(inverse_rank[i] for i in target_positions))
            all_records.append({
                "case_id": row.case_id,
                "alpha_ranker": float(alpha),
                "target_rank": target_rank,
                "top1": int(target_rank <= 1),
                "top5": int(target_rank <= 5),
                "top10": int(target_rank <= 10),
                "mrr": 1.0 / target_rank
            })

pred = pd.DataFrame(all_records)
pred_path = DATA / "patient_validation_score_combination.csv"
pred.to_csv(pred_path, index=False)

summary = pred.groupby("alpha_ranker").agg(
    cases=("case_id", "nunique"),
    Top1=("top1", "mean"),
    Top5=("top5", "mean"),
    Top10=("top10", "mean"),
    MRR=("mrr", "mean")
).reset_index()

for col in ["Top1", "Top5", "Top10", "MRR"]:
    summary[col] = summary[col].round(4)

summary_path = DATA / "patient_validation_score_combination_summary.csv"
summary.to_csv(summary_path, index=False)

print("Shared candidates:", len(candidates))
print("Validation cases evaluated:", pred["case_id"].nunique())
print("\n=== SCORE COMBINATION RESULTS ===")
print(summary.to_string(index=False))
print("\nBest MRR weight:")
print(summary.loc[summary["MRR"].idxmax()].to_dict())
print("\nSaved predictions:", pred_path)
print("Saved summary:", summary_path)
print("Device:", DEVICE)


Shared candidates: 7340
Validation cases evaluated: 906

=== SCORE COMBINATION RESULTS ===
 alpha_ranker  cases   Top1   Top5  Top10    MRR
          0.0    906 0.2947 0.4680 0.5419 0.3773
          0.1    906 0.2031 0.4084 0.4514 0.3082
          0.2    906 0.2031 0.4161 0.4503 0.3092
          0.3    906 0.1987 0.4183 0.4525 0.3064
          0.4    906 0.1932 0.4227 0.4503 0.3018
          0.5    906 0.1921 0.4238 0.4581 0.2985
          0.6    906 0.1998 0.4139 0.4603 0.3002
          0.7    906 0.2053 0.4029 0.4514 0.3007
          0.8    906 0.2119 0.3929 0.4338 0.3019
          0.9    906 0.2285 0.3918 0.4260 0.3075
          1.0    906 0.2671 0.3444 0.3653 0.3043

Best MRR weight:
{'alpha_ranker': 0.0, 'cases': 906.0, 'Top1': 0.2947, 'Top5': 0.468, 'Top10': 0.5419, 'MRR': 0.3773}

Saved predictions: /content/drive/MyDrive/Rare Disease ai/processed/patient_validation_score_combination.csv
Saved summary: /content/drive/MyDrive/Rare Disease ai/processed/patient_validation_score_com

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

summary_path = DATA / "patient_validation_score_combination_summary.csv"
assert summary_path.exists(), f"Missing: {summary_path}"

summary = pd.read_csv(summary_path)
baseline_row = summary.loc[summary["alpha_ranker"] == 0.0].iloc[0]
best_row = summary.loc[summary["MRR"].idxmax()]

decision = {
    "decision": "retain_phenotype_only_baseline",
    "status": "research_prototype_not_clinically_validated",
    "selected_method": "information_content_weighted_phenotype_F1",
    "selected_validation_metrics": {
        "cases": int(baseline_row["cases"]),
        "top1": float(baseline_row["Top1"]),
        "top5": float(baseline_row["Top5"]),
        "top10": float(baseline_row["Top10"]),
        "mrr": float(baseline_row["MRR"]),
    },
    "score_combination_experiment": {
        "best_ranker_weight": float(best_row["alpha_ranker"]),
        "best_validation_mrr": float(best_row["MRR"]),
        "interpretation": (
            "No tested combination improved over the phenotype-only baseline."
        ),
    },
    "experimental_ranker": {
        "checkpoint": str(MODEL / "patient_level_ranker_best.pt"),
        "preserved": (MODEL / "patient_level_ranker_best.pt").exists(),
        "use_as_primary_model": False,
        "reason": "Bias term caused prediction collapse; bias-free ranker still underperformed baseline.",
    },
    "held_out_test_set_used_for_model_selection": False,
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
}

decision_path = DATA / "model_selection_decision.json"
decision_path.write_text(json.dumps(decision, indent=2))

print("MODEL SELECTION RECORD SAVED")
print(json.dumps(decision, indent=2))
print("\nPath:", decision_path)
print("\nNo model was retrained, overwritten, or deleted.")


MODEL SELECTION RECORD SAVED
{
  "decision": "retain_phenotype_only_baseline",
  "status": "research_prototype_not_clinically_validated",
  "selected_method": "information_content_weighted_phenotype_F1",
  "selected_validation_metrics": {
    "cases": 906,
    "top1": 0.2947,
    "top5": 0.468,
    "top10": 0.5419,
    "mrr": 0.3773
  },
  "score_combination_experiment": {
    "best_ranker_weight": 0.0,
    "best_validation_mrr": 0.3773,
    "interpretation": "No tested combination improved over the phenotype-only baseline."
  },
  "experimental_ranker": {
    "checkpoint": "/content/drive/MyDrive/Rare Disease ai/model/patient_level_ranker_best.pt",
    "preserved": true,
    "use_as_primary_model": false,
    "reason": "Bias term caused prediction collapse; bias-free ranker still underperformed baseline."
  },
  "held_out_test_set_used_for_model_selection": false,
  "recorded_at_utc": "2026-10-14T18:29:24.257713+00:00"
}

Path: /content/drive/MyDrive/Rare Disease ai/processed/model_se

In [ ]:

from pathlib import Path
import json
import re
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

def read_pmids(path):
    df = pd.read_csv(path, low_memory=False)
    if "pmid" not in df.columns:
        return set(), df
    values = df["pmid"].dropna().astype(str).str.strip()
    pmids = set()
    for value in values:
        # Extract numeric PMID tokens conservatively.
        for token in re.split(r"[;,|\s]+", value):
            token = token.strip()
            if token.isdigit():
                pmids.add(token)
    return pmids, df

# 1. Load publication split and identify development/test PMID sets.
split_path = DATA / "patient_publication_split.csv"
assert split_path.exists(), f"Missing split file: {split_path}"
split_df = pd.read_csv(split_path, low_memory=False)

pmid_col = "pmid"
split_col = "split"
if pmid_col not in split_df.columns or split_col not in split_df.columns:
    raise ValueError(
        f"Expected columns 'pmid' and 'split'. Found: {split_df.columns.tolist()}"
    )

dev_pmids = set(
    split_df.loc[
        split_df[split_col].astype(str).str.lower().isin(["development", "dev", "train"]),
        pmid_col
    ].dropna().astype(str).str.strip()
)
test_pmids = set(
    split_df.loc[
        split_df[split_col].astype(str).str.lower().isin(["test", "heldout", "held-out"]),
        pmid_col
    ].dropna().astype(str).str.strip()
)

# 2. Inspect reference coverage in the source annotation file.
annotation_path = DATA / "gnn_training_annotations.csv"
if not annotation_path.exists():
    annotation_path = DATA / "clean_hpo_annotations.csv"

ann = pd.read_csv(annotation_path, low_memory=False)
reference_col = next(
    (c for c in ["reference", "references", "pmid", "publication"] if c in ann.columns),
    None
)

missing_reference_rows = None
parseable_reference_rows = None
dev_overlap_rows = None
test_overlap_rows = None
unique_annotation_pmids = set()

if reference_col is not None:
    refs = ann[reference_col]
    missing_reference_rows = int(refs.isna().sum())
    parseable_reference_rows = 0
    dev_overlap_rows = 0
    test_overlap_rows = 0

    for ref in refs.dropna().astype(str):
        found = set(re.findall(r"(?<!\d)\d{5,9}(?!\d)", ref))
        if found:
            parseable_reference_rows += 1
            unique_annotation_pmids.update(found)
            if found & dev_pmids:
                dev_overlap_rows += 1
            if found & test_pmids:
                test_overlap_rows += 1

# 3. Check evaluation artifacts without opening or scoring the held-out test.
required_paths = [
    DATA / "patient_level_validation.csv",
    DATA / "patient_level_test.csv",
    DATA / "patient_phenotype_only_baseline.csv",
    DATA / "patient_phenotype_only_validation.csv",
    DATA / "model_selection_decision.json",
    MODEL / "patient_level_ranker_best.pt",
]
artifact_status = {
    str(p.relative_to(PROJECT)): {
        "exists": p.exists(),
        "bytes": p.stat().st_size if p.exists() and p.is_file() else None,
    }
    for p in required_paths
}

# 4. Record a conservative audit summary.
report = {
    "annotation_source_audited": str(annotation_path),
    "annotation_rows": int(len(ann)),
    "reference_column": reference_col,
    "missing_reference_rows": missing_reference_rows,
    "rows_with_parseable_numeric_references": parseable_reference_rows,
    "development_pmids_in_split": int(len(dev_pmids)),
    "test_pmids_in_split": int(len(test_pmids)),
    "unique_numeric_pmids_in_annotations": int(len(unique_annotation_pmids)),
    "annotation_rows_with_detected_development_pmid_overlap": dev_overlap_rows,
    "annotation_rows_with_detected_test_pmid_overlap": test_overlap_rows,
    "artifacts": artifact_status,
    "limitations": [
        "Numeric-reference parsing is an audit heuristic, not proof of provenance.",
        "Rows with missing or unmatched references cannot be confirmed publication-independent.",
        "No held-out test predictions or metrics were calculated in this step."
    ]
}

report_path = DATA / "annotation_leakage_readiness_audit.json"
report_path.write_text(json.dumps(report, indent=2))

print("=== ANNOTATION LEAKAGE / READINESS AUDIT ===")
print(json.dumps(report, indent=2))
print("\nSaved audit report:", report_path)


=== ANNOTATION LEAKAGE / READINESS AUDIT ===
{
  "annotation_source_audited": "/content/drive/MyDrive/Rare Disease ai/processed/gnn_training_annotations.csv",
  "annotation_rows": 179653,
  "reference_column": "reference",
  "missing_reference_rows": 0,
  "rows_with_parseable_numeric_references": 155411,
  "development_pmids_in_split": 1383,
  "test_pmids_in_split": 346,
  "unique_numeric_pmids_in_annotations": 11362,
  "annotation_rows_with_detected_development_pmid_overlap": 17288,
  "annotation_rows_with_detected_test_pmid_overlap": 5166,
  "artifacts": {
    "processed/patient_level_validation.csv": {
      "exists": true,
      "bytes": 134375
    },
    "processed/patient_level_test.csv": {
      "exists": false,
      "bytes": null
    },
    "processed/patient_phenotype_only_baseline.csv": {
      "exists": true,
      "bytes": 122758
    },
    "processed/patient_phenotype_only_validation.csv": {
      "exists": true,
      "bytes": 70673
    },
    "processed/model_selection_

In [ ]:

from pathlib import Path
import json
import re
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

split = pd.read_csv(DATA / "patient_publication_split.csv", low_memory=False)
split["pmid"] = split["pmid"].astype("string").str.strip()
split["split"] = split["split"].astype("string").str.lower().str.strip()

dev_pmids = set(split.loc[
    split["split"].isin(["development", "dev", "train"]), "pmid"
].dropna())
test_pmids = set(split.loc[
    split["split"].isin(["test", "heldout", "held-out"]), "pmid"
].dropna())

def inspect_annotations(path):
    if not path.exists():
        return {"exists": False, "path": str(path)}

    df = pd.read_csv(path, low_memory=False)
    ref_col = next(
        (c for c in ["reference", "references", "pmid", "publication"]
         if c in df.columns),
        None
    )
    if ref_col is None:
        return {
            "exists": True, "path": str(path), "rows": len(df),
            "reference_column": None
        }

    dev_rows, test_rows, parseable_rows = 0, 0, 0
    unique_pmids = set()
    for value in df[ref_col].dropna().astype(str):
        refs = set(re.findall(r"(?<!\d)\d{5,9}(?!\d)", value))
        if not refs:
            continue
        parseable_rows += 1
        unique_pmids.update(refs)
        dev_rows += int(bool(refs & dev_pmids))
        test_rows += int(bool(refs & test_pmids))

    return {
        "exists": True,
        "path": str(path),
        "rows": int(len(df)),
        "reference_column": ref_col,
        "null_reference_rows": int(df[ref_col].isna().sum()),
        "rows_with_numeric_reference": parseable_rows,
        "unique_numeric_pmids": len(unique_pmids),
        "rows_overlapping_development_pmids": dev_rows,
        "rows_overlapping_test_pmids": test_rows,
    }

# 1. Audit both original and filtered annotation sources.
annotation_results = {}
for filename in [
    "gnn_training_annotations.csv",
    "gnn_annotations_excluding_dev_test_pmids.csv",
    "patient_test_leakage_reduced_annotations.csv",
    "clean_hpo_annotations.csv",
]:
    annotation_results[filename] = inspect_annotations(DATA / filename)

# 2. Inspect the saved test baseline without using its scores for model selection.
baseline_path = DATA / "patient_phenotype_only_baseline.csv"
baseline_summary = {"exists": baseline_path.exists()}
if baseline_path.exists():
    baseline = pd.read_csv(baseline_path, low_memory=False)
    baseline_summary["rows"] = len(baseline)
    baseline_summary["columns"] = baseline.columns.tolist()
    baseline_summary["unique_case_ids"] = (
        int(baseline["case_id"].nunique()) if "case_id" in baseline else None
    )
    baseline_summary["unique_pmids"] = (
        int(baseline["pmid"].nunique()) if "pmid" in baseline else None
    )
    baseline_summary["duplicate_case_ids"] = (
        int(baseline["case_id"].duplicated().sum())
        if "case_id" in baseline else None
    )

# 3. Search for plausible existing test-case artifacts; do not fabricate one.
candidate_files = []
for folder in [DATA, PROJECT]:
    for p in folder.glob("**/*"):
        if p.is_file() and any(
            word in p.name.lower()
            for word in ["patient", "phenopacket", "test", "publication_split"]
        ):
            if p.suffix.lower() in [".csv", ".json", ".jsonl", ".parquet", ".pt"]:
                candidate_files.append(str(p))

report = {
    "development_pmid_count": len(dev_pmids),
    "test_pmid_count": len(test_pmids),
    "annotation_audits": annotation_results,
    "saved_test_baseline": baseline_summary,
    "candidate_patient_test_files": sorted(set(candidate_files)),
    "warning": (
        "PMID parsing is heuristic. Zero detected overlap alone does not prove "
        "complete provenance or absence of leakage."
    ),
}

out = DATA / "annotation_provenance_followup_audit.json"
out.write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print("\nSaved:", out)



{
  "development_pmid_count": 1383,
  "test_pmid_count": 346,
  "annotation_audits": {
    "gnn_training_annotations.csv": {
      "exists": true,
      "path": "/content/drive/MyDrive/Rare Disease ai/processed/gnn_training_annotations.csv",
      "rows": 179653,
      "reference_column": "reference",
      "null_reference_rows": 0,
      "rows_with_numeric_reference": 155411,
      "unique_numeric_pmids": 11362,
      "rows_overlapping_development_pmids": 17288,
      "rows_overlapping_test_pmids": 5166
    },
    "gnn_annotations_excluding_dev_test_pmids.csv": {
      "exists": true,
      "path": "/content/drive/MyDrive/Rare Disease ai/processed/gnn_annotations_excluding_dev_test_pmids.csv",
      "rows": 159025,
      "reference_column": "reference",
      "null_reference_rows": 0,
      "rows_with_numeric_reference": 134783,
      "unique_numeric_pmids": 10386,
      "rows_overlapping_development_pmids": 272,
      "rows_overlapping_test_pmids": 76
    },
    "patient_test_leakage

In [ ]:

from pathlib import Path
import pandas as pd
import re

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
dev_pmids = set(
    split.loc[split["split"].str.lower().eq("development"), "pmid"]
    .dropna().str.extract(r"(\d+)", expand=False).dropna()
)
test_pmids = set(
    split.loc[split["split"].str.lower().eq("test"), "pmid"]
    .dropna().str.extract(r"(\d+)", expand=False).dropna()
)

files = [
    "gnn_training_annotations.csv",
    "gnn_annotations_excluding_dev_test_pmids.csv",
    "patient_test_leakage_reduced_annotations.csv",
    "clean_hpo_annotations.csv",
]

print("ANNOTATION REFERENCE CHECK")
for name in files:
    path = DATA / name
    if not path.exists():
        print(f"{name}: MISSING")
        continue

    df = pd.read_csv(path, dtype=str, usecols=["reference"])
    refs = df["reference"].fillna("").str.strip()

    # Find PMID numbers anywhere in a reference, not just numeric-only values.
    extracted = refs.str.findall(r"(?i)PMID\s*:\s*(\d+)")
    pmid_sets = extracted.apply(set)

    dev_overlap = sum(bool(s & dev_pmids) for s in pmid_sets)
    test_overlap = sum(bool(s & test_pmids) for s in pmid_sets)
    no_pmid = sum(len(s) == 0 for s in pmid_sets)

    print(
        f"{name}: rows={len(df):,}, "
        f"rows_without_detected_PMID={no_pmid:,}, "
        f"dev_overlap_rows={dev_overlap:,}, "
        f"test_overlap_rows={test_overlap:,}"
    )


ANNOTATION REFERENCE CHECK
gnn_training_annotations.csv: rows=179,653, rows_without_detected_PMID=88,895, dev_overlap_rows=17,288, test_overlap_rows=5,166
gnn_annotations_excluding_dev_test_pmids.csv: rows=159,025, rows_without_detected_PMID=88,895, dev_overlap_rows=272, test_overlap_rows=76
patient_test_leakage_reduced_annotations.csv: rows=280,733, rows_without_detected_PMID=167,539, dev_overlap_rows=20,862, test_overlap_rows=1,259
clean_hpo_annotations.csv: rows=286,047, rows_without_detected_PMID=167,539, dev_overlap_rows=21,593, test_overlap_rows=6,573


In [ ]:

import pandas as pd
import re
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = [
    "gnn_training_annotations.csv",
    "gnn_annotations_excluding_dev_test_pmids.csv",
    "patient_test_leakage_reduced_annotations.csv",
    "clean_hpo_annotations.csv",
]

pmid_pattern = re.compile(r"(?i)PMID\s*:\s*(\d+)")

for filename in files:
    path = DATA / filename
    if not path.exists():
        print(f"\n{filename}: FILE NOT FOUND")
        continue

    df = pd.read_csv(path, dtype=str, low_memory=False)
    print(f"\n--- {filename} ---")
    print("Columns:", df.columns.tolist())

    if "reference" not in df.columns:
        print("No 'reference' column; inspect the column names above.")
        continue

    refs = df["reference"].fillna("").str.strip()
    undetected = refs[~refs.str.contains(pmid_pattern, na=False)]

    print("Rows without detected PMID:", len(undetected))
    print("Most common undetected reference values:")
    print(undetected.value_counts().head(12).to_string())

    samples = undetected[undetected.ne("")].drop_duplicates().head(8)
    if len(samples):
        print("Distinct examples:")
        for value in samples:
            print(" ", repr(value[:250]))




--- gnn_training_annotations.csv ---
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']


/tmp/ipykernel_2609/3474662809.py:32: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  undetected = refs[~refs.str.contains(pmid_pattern, na=False)]


Rows without detected PMID: 88895
Most common undetected reference values:
reference
ORPHA:904       186
ORPHA:261552    147
ORPHA:567       131
ORPHA:261537    127
ORPHA:3455      127
ORPHA:353284    123
ORPHA:96334     120
ORPHA:800       112
ORPHA:2092      107
ORPHA:84        106
ORPHA:818       106
ORPHA:500150    105
Distinct examples:
  'OMIM:614102'
  'OMIM:609621'
  'OMIM:613679'
  'OMIM:614116'
  'OMIM:137580'
  'OMIM:117650'
  'OMIM:616067'
  'OMIM:108770'

--- gnn_annotations_excluding_dev_test_pmids.csv ---
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration', 'reference_pmid']


/tmp/ipykernel_2609/3474662809.py:32: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  undetected = refs[~refs.str.contains(pmid_pattern, na=False)]


Rows without detected PMID: 88895
Most common undetected reference values:
reference
ORPHA:904       186
ORPHA:261552    147
ORPHA:567       131
ORPHA:261537    127
ORPHA:3455      127
ORPHA:353284    123
ORPHA:96334     120
ORPHA:800       112
ORPHA:2092      107
ORPHA:84        106
ORPHA:818       106
ORPHA:500150    105
Distinct examples:
  'OMIM:614102'
  'OMIM:609621'
  'OMIM:613679'
  'OMIM:614116'
  'OMIM:137580'
  'OMIM:117650'
  'OMIM:616067'
  'OMIM:108770'

--- patient_test_leakage_reduced_annotations.csv ---
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']


/tmp/ipykernel_2609/3474662809.py:32: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  undetected = refs[~refs.str.contains(pmid_pattern, na=False)]


Rows without detected PMID: 167539
Most common undetected reference values:
reference
ISBN-13:978-0721606156    230
ORPHA:904                 186
ORPHA:261552              147
ORPHA:2152                140
ORPHA:567                 131
ORPHA:3455                127
ORPHA:261537              127
ORPHA:353284              123
ORPHA:353277              122
ORPHA:534                 120
ORPHA:96334               120
ORPHA:191                 118
Distinct examples:
  'OMIM:609153'
  'OMIM:614102'
  'OMIM:609621'
  'OMIM:212790'
  'OMIM:613679'
  'OMIM:614116'
  'OMIM:137580'
  'OMIM:117650'

--- clean_hpo_annotations.csv ---
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']


/tmp/ipykernel_2609/3474662809.py:32: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  undetected = refs[~refs.str.contains(pmid_pattern, na=False)]


Rows without detected PMID: 167539
Most common undetected reference values:
reference
ISBN-13:978-0721606156    230
ORPHA:904                 186
ORPHA:261552              147
ORPHA:2152                140
ORPHA:567                 131
ORPHA:3455                127
ORPHA:261537              127
ORPHA:353284              123
ORPHA:353277              122
ORPHA:534                 120
ORPHA:96334               120
ORPHA:191                 118
Distinct examples:
  'OMIM:609153'
  'OMIM:614102'
  'OMIM:609621'
  'OMIM:212790'
  'OMIM:613679'
  'OMIM:614116'
  'OMIM:137580'
  'OMIM:117650'


In [ ]:

import pandas as pd
import re
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
annotations = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    dtype=str,
    low_memory=False
)

# Normalize publication IDs
split["pmid"] = split["pmid"].str.extract(r"(\d+)", expand=False)
dev_pmids = set(split.loc[split["split"].str.lower().eq("development"), "pmid"].dropna())
test_pmids = set(split.loc[split["split"].str.lower().eq("test"), "pmid"].dropna())

print("Split columns:", split.columns.tolist())
print("Development PMIDs:", len(dev_pmids))
print("Test PMIDs:", len(test_pmids))

# Prefer the explicit PMID column if present; otherwise extract from reference
if "reference_pmid" in annotations.columns:
    annotations["audit_pmid"] = annotations["reference_pmid"].str.extract(
        r"(\d+)", expand=False
    )
else:
    annotations["audit_pmid"] = annotations["reference"].str.extract(
        r"(?i)PMID\s*:\s*(\d+)", expand=False
    )

dev_overlap = annotations[annotations["audit_pmid"].isin(dev_pmids)]
test_overlap = annotations[annotations["audit_pmid"].isin(test_pmids)]

print("\nFiltered annotation audit:")
print("Rows:", len(annotations))
print("Rows with recognized PMID:", annotations["audit_pmid"].notna().sum())
print("Development-overlap rows:", len(dev_overlap))
print("Test-overlap rows:", len(test_overlap))

cols = [
    c for c in [
        "database_id", "disease_name", "hpo_id",
        "reference", "reference_pmid", "evidence", "biocuration"
    ] if c in annotations.columns
]

print("\nDevelopment-overlap examples:")
print(dev_overlap[cols].head(8).to_string(index=False))

print("\nTest-overlap examples:")
print(test_overlap[cols].head(8).to_string(index=False))


Split columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
Development PMIDs: 1383
Test PMIDs: 346

Filtered annotation audit:
Rows: 159025
Rows with recognized PMID: 70130
Development-overlap rows: 0
Test-overlap rows: 0

Development-overlap examples:
Empty DataFrame
Columns: [database_id, disease_name, hpo_id, reference, reference_pmid, evidence, biocuration]
Index: []

Test-overlap examples:
Empty DataFrame
Columns: [database_id, disease_name, hpo_id, reference, reference_pmid, evidence, biocuration]
Index: []


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
df = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    dtype=str,
    low_memory=False
)

ref = df["reference"].fillna("").str.strip()
pmid = df["reference_pmid"].fillna("").str.strip()

print("Total rows:", len(df))
print("Blank reference_pmid:", pmid.eq("").sum())
print("Nonblank reference_pmid:", pmid.ne("").sum())
print("Reference values beginning with PMID:",
      ref.str.contains(r"(?i)^\s*PMID\s*:", regex=True).sum())
print("PMID column values that are numeric:",
      pmid.str.fullmatch(r"\d+").sum())
print("Examples where reference contains PMID but reference_pmid is blank:")
mask = ref.str.contains(r"(?i)PMID\s*:\s*\d+", regex=True) & pmid.eq("")
print(df.loc[mask, ["reference", "reference_pmid"]].head(10).to_string(index=False))



Total rows: 159025
Blank reference_pmid: 88895
Nonblank reference_pmid: 70130
Reference values beginning with PMID: 70012
PMID column values that are numeric: 70130
Examples where reference contains PMID but reference_pmid is blank:
Empty DataFrame
Columns: [reference, reference_pmid]
Index: []


In [ ]:

import pandas as pd
from pathlib import Path

path = Path("/content/drive/MyDrive/Rare Disease ai/processed/gnn_annotations_excluding_dev_test_pmids.csv")
df = pd.read_csv(path, dtype=str, low_memory=False)

ref = df["reference"].fillna("").str.strip()
pmid = df["reference_pmid"].fillna("").str.strip()

mask = pmid.ne("") & ~ref.str.contains(r"(?i)^\s*PMID\s*:", regex=True)

print("Rows to explain:", int(mask.sum()))
print("\nReference values for those rows:")
print(df.loc[mask, ["reference", "reference_pmid"]]
        .value_counts()
        .head(20)
        .to_string())


Rows to explain: 118

Reference values for those rows:
reference                               reference_pmid
OMIM:186579;PMID:11545688               11545688          7
OMIM:610832;PMID:17200672               17200672          5
OMIM:312170;PMID:17043409               17043409          3
OMIM:235730;PMID:29300384               29300384          3
OMIM:256800;PMID:8696348                8696348           2
OMIM:182250;PMID:23322711               23322711          2
OMIM:300908;PMID:29298156               29298156          2
OMIM:613559;PMID:20598281               20598281          2
OMIM:614437;PMID:20389311               20389311          2
OMIM:614866;PMID:14630978               14630978          2
OMIM:164400;PMID:14967775               14967775          1
OMIM:152700;PMID:9796581                9796581           1
OMIM:101600;PMID:15996217               15996217          1
ISBN-13:978-0721606156;PMID:30674241    30674241          1
OMIM:147250;PMID:11471164               11471164  

In [ ]:

import pandas as pd
import re
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
annotations = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    dtype=str,
    low_memory=False
)

dev = set(
    split.loc[split["split"].str.lower().eq("development"), "pmid"]
    .str.extract(r"(\d+)", expand=False).dropna()
)
test = set(
    split.loc[split["split"].str.lower().eq("test"), "pmid"]
    .str.extract(r"(\d+)", expand=False).dropna()
)

# Extract every PMID from each complete reference string
all_pmids = annotations["reference"].fillna("").apply(
    lambda s: set(re.findall(r"(?i)PMID\s*:\s*(\d+)", s))
)

dev_mask = all_pmids.apply(lambda ids: bool(ids & dev))
test_mask = all_pmids.apply(lambda ids: bool(ids & test))

print("Rows audited:", len(annotations))
print("Rows containing at least one PMID:", int(all_pmids.apply(bool).sum()))
print("Rows overlapping development PMIDs:", int(dev_mask.sum()))
print("Rows overlapping test PMIDs:", int(test_mask.sum()))

print("\nDevelopment overlap examples:")
print(annotations.loc[dev_mask, ["database_id", "hpo_id", "reference"]]
      .head(10).to_string(index=False))

print("\nTest overlap examples:")
print(annotations.loc[test_mask, ["database_id", "hpo_id", "reference"]]
      .head(10).to_string(index=False))


Rows audited: 159025
Rows containing at least one PMID: 70130
Rows overlapping development PMIDs: 272
Rows overlapping test PMIDs: 76

Development overlap examples:
database_id     hpo_id                                 reference
OMIM:611091 HP:0000486               PMID:22541562;PMID:22541559
OMIM:611091 HP:0004322               PMID:22541562;PMID:22541559
OMIM:611091 HP:0001250 PMID:22541562;PMID:36420349;PMID:22541559
OMIM:611091 HP:0001276               PMID:22541562;PMID:22541559
OMIM:105650 HP:0004322 PMID:10590074;PMID:10753603;PMID:35923690
OMIM:105650 HP:0000316               PMID:10753603;PMID:35923690
OMIM:614923 HP:0002069               PMID:36729635;PMID:22956686
OMIM:614923 HP:0001250               PMID:36729635;PMID:22956686
OMIM:614923 HP:0011182               PMID:36729635;PMID:22956686
OMIM:614923 HP:0001249               PMID:36729635;PMID:22956686

Test overlap examples:
database_id     hpo_id                                 reference
OMIM:617225 HP:0001251         

In [ ]:

import pandas as pd
import re
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

source_path = DATA / "gnn_annotations_excluding_dev_test_pmids.csv"
output_path = DATA / "gnn_annotations_publication_disjoint_candidate.csv"
split_path = DATA / "patient_publication_split.csv"

split = pd.read_csv(split_path, dtype=str)
df = pd.read_csv(source_path, dtype=str, low_memory=False)

dev_pmids = set(
    split.loc[split["split"].str.lower().eq("development"), "pmid"]
    .str.extract(r"(\d+)", expand=False).dropna()
)
test_pmids = set(
    split.loc[split["split"].str.lower().eq("test"), "pmid"]
    .str.extract(r"(\d+)", expand=False).dropna()
)
excluded_pmids = dev_pmids | test_pmids

def referenced_pmids(value):
    return set(re.findall(r"(?i)PMID\s*:\s*(\d+)", value or ""))

row_pmids = df["reference"].fillna("").apply(referenced_pmids)
overlap_mask = row_pmids.apply(lambda ids: bool(ids & excluded_pmids))

candidate = df.loc[~overlap_mask].copy()
candidate.to_csv(output_path, index=False)

# Verify the saved candidate against every PMID in each reference
saved = pd.read_csv(output_path, dtype=str, low_memory=False)
saved_pmids = saved["reference"].fillna("").apply(referenced_pmids)
remaining_overlap = saved_pmids.apply(lambda ids: bool(ids & excluded_pmids)).sum()

print("Original rows:", len(df))
print("Rows excluded for any dev/test PMID:", int(overlap_mask.sum()))
print("Candidate rows saved:", len(saved))
print("Remaining overlapping rows:", int(remaining_overlap))
print("Rows without a PMID retained:", int(saved_pmids.apply(lambda ids: not ids).sum()))
print("Saved to:", output_path)


Original rows: 159025
Rows excluded for any dev/test PMID: 332
Candidate rows saved: 158693
Remaining overlapping rows: 0
Rows without a PMID retained: 88895
Saved to: /content/drive/MyDrive/Rare Disease ai/processed/gnn_annotations_publication_disjoint_candidate.csv


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = {
    "Previous filtered annotations":
        DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    "Publication-disjoint candidate":
        DATA / "gnn_annotations_publication_disjoint_candidate.csv",
}

for label, path in files.items():
    df = pd.read_csv(path, usecols=["database_id", "hpo_id"], dtype=str)
    print(f"\n{label}")
    print("Rows:", len(df))
    print("Unique diseases:", df["database_id"].nunique())
    print("Unique HPO terms:", df["hpo_id"].nunique())
    print("Unique disease-HPO pairs:",
          df.drop_duplicates(["database_id", "hpo_id"]).shape[0])



Previous filtered annotations
Rows: 159025
Unique diseases: 6983
Unique HPO terms: 9617
Unique disease-HPO pairs: 159025

Publication-disjoint candidate
Rows: 158693
Unique diseases: 6980
Unique HPO terms: 9613
Unique disease-HPO pairs: 158693


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

old = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    usecols=["database_id", "hpo_id"], dtype=str
)
new = pd.read_csv(
    DATA / "gnn_annotations_publication_disjoint_candidate.csv",
    usecols=["database_id", "hpo_id"], dtype=str
)

old_diseases = set(old["database_id"].dropna())
new_diseases = set(new["database_id"].dropna())
old_hpo = set(old["hpo_id"].dropna())
new_hpo = set(new["hpo_id"].dropna())

report = {
    "missing_disease_ids": sorted(old_diseases - new_diseases),
    "missing_hpo_ids": sorted(old_hpo - new_hpo),
}

print("Diseases absent from candidate:", report["missing_disease_ids"])
print("HPO terms absent from candidate:", report["missing_hpo_ids"])

# Save a reproducible coverage report
out = DATA / "publication_disjoint_coverage_report.txt"
with open(out, "w") as f:
    f.write("Diseases absent from candidate:\n")
    f.write("\n".join(report["missing_disease_ids"]) or "None")
    f.write("\n\nHPO terms absent from candidate:\n")
    f.write("\n".join(report["missing_hpo_ids"]) or "None")
    f.write("\n")

print("Report saved:", out)


Diseases absent from candidate: ['OMIM:129600', 'OMIM:273900', 'OMIM:610187']
HPO terms absent from candidate: ['HP:0006346', 'HP:0500132', 'HP:0500143', 'HP:0500144']
Report saved: /content/drive/MyDrive/Rare Disease ai/processed/publication_disjoint_coverage_report.txt


In [ ]:

import pandas as pd
import json
import re
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PHENOPACKETS = PROJECT / "all_phenopackets"

missing_diseases = {
    "OMIM:129600", "OMIM:273900", "OMIM:610187"
}
missing_hpo = {
    "HP:0006346", "HP:0500132", "HP:0500143", "HP:0500144"
}

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
split["pmid"] = split["pmid"].str.extract(r"(\d+)", expand=False)
split["split"] = split["split"].str.lower()

print("=== Missing disease IDs in diagnosis labels ===")
for disease in sorted(missing_diseases):
    matches = split[
        split["diagnosis_ids"].fillna("").str.contains(
            re.escape(disease), regex=True
        )
    ]
    print(
        disease,
        "| matching cases:", len(matches),
        "| development:", int((matches["split"] == "development").sum()),
        "| test:", int((matches["split"] == "test").sum())
    )

# Scan phenotype packet JSON files and associate them with PMIDs
# using the PMID in the filename.
pmid_to_splits = {}
for _, row in split.dropna(subset=["pmid"]).iterrows():
    pmid_to_splits.setdefault(row["pmid"], set()).add(row["split"])

found = {hpo: {"development": set(), "test": set()} for hpo in missing_hpo}
scanned = 0
unmatched_files = 0

for path in PHENOPACKETS.rglob("*.json"):
    match = re.search(r"PMID[_\s:-]*(\d+)", path.name, flags=re.I)
    if not match:
        unmatched_files += 1
        continue

    pmid = match.group(1)
    splits = pmid_to_splits.get(pmid, set())
    if not splits:
        continue

    try:
        content = path.read_text(encoding="utf-8")
        packet = json.loads(content)
        scanned += 1
    except Exception:
        continue

    # Search the packet's full JSON text for each exact HPO identifier.
    for hpo in missing_hpo:
        if re.search(r"(?<![A-Z0-9:])" + re.escape(hpo) + r"(?![A-Z0-9])",
                     content):
            for cohort in ("development", "test"):
                if cohort in splits:
                    found[hpo][cohort].add(pmid)

print("\n=== Missing HPO terms in phenotype packets ===")
print("Matching PMID-linked JSON files scanned:", scanned)
for hpo in sorted(missing_hpo):
    print(
        hpo,
        "| development publications:", len(found[hpo]["development"]),
        "| test publications:", len(found[hpo]["test"])
    )

print("\nNote: PMID-linked files only; filename matching may miss some packets.")


=== Missing disease IDs in diagnosis labels ===
OMIM:129600 | matching cases: 44 | development: 37 | test: 7
OMIM:273900 | matching cases: 8 | development: 8 | test: 0
OMIM:610187 | matching cases: 11 | development: 10 | test: 1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

files_to_check = [
    DATA / "patient_publication_split.csv",
    DATA / "patient_phenotype_only_baseline.csv",
    DATA / "gnn_training_annotations.csv",
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    DATA / "gnn_annotations_publication_disjoint_candidate.csv",
    DATA / "publication_disjoint_coverage_report.txt",
    DATA / "annotation_provenance_followup_audit.json",
    DATA / "rare_disease_gnn_training_only.pt",
    MODEL_DIR / "rare_disease_gnn_filtered_best.pt",
    MODEL_DIR / "patient_level_ranker_best.pt",
]

print("PROJECT EXISTS:", PROJECT.exists())
print("PROCESSED DIRECTORY EXISTS:", DATA.exists())
print("MODEL DIRECTORY EXISTS:", MODEL_DIR.exists())

print("\nFILE STATUS")
for path in files_to_check:
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"FOUND  | {path.name} | {size_mb:.2f} MB")
    else:
        print(f"MISSING | {path.name}")

if (DATA / "patient_publication_split.csv").exists():
    split = pd.read_csv(DATA / "patient_publication_split.csv")
    print("\nPUBLICATION SPLIT")
    print(split["split"].value_counts(dropna=False).to_string())

if (DATA / "gnn_annotations_publication_disjoint_candidate.csv").exists():
    annotations = pd.read_csv(
        DATA / "gnn_annotations_publication_disjoint_candidate.csv",
        usecols=["database_id", "hpo_id"]
    )
    print("\nPUBLICATION-DISJOINT CANDIDATE")
    print("Rows:", len(annotations))
    print("Unique diseases:", annotations["database_id"].nunique())
    print("Unique HPO terms:", annotations["hpo_id"].nunique())

PROJECT EXISTS: True
PROCESSED DIRECTORY EXISTS: True
MODEL DIRECTORY EXISTS: True

FILE STATUS
FOUND  | patient_publication_split.csv | 0.59 MB
FOUND  | patient_phenotype_only_baseline.csv | 0.12 MB
FOUND  | gnn_training_annotations.csv | 21.95 MB
FOUND  | gnn_annotations_excluding_dev_test_pmids.csv | 19.80 MB
FOUND  | gnn_annotations_publication_disjoint_candidate.csv | 19.74 MB
FOUND  | publication_disjoint_coverage_report.txt | 0.00 MB
FOUND  | annotation_provenance_followup_audit.json | 0.26 MB
FOUND  | rare_disease_gnn_training_only.pt | 4.73 MB
FOUND  | rare_disease_gnn_filtered_best.pt | 4.63 MB
FOUND  | patient_level_ranker_best.pt | 8.95 MB

PUBLICATION SPLIT
split
development    8386
test           1959

PUBLICATION-DISJOINT CANDIDATE
Rows: 158693
Unique diseases: 6980
Unique HPO terms: 9613


In [ ]:
import json
import re
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PHENOPACKETS = PROJECT / "all_phenopackets"

missing_hpo = {
    "HP:0006346",
    "HP:0500132",
    "HP:0500143",
    "HP:0500144",
}

# Load the existing publication split
split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
split["pmid"] = split["pmid"].str.extract(r"(\d+)", expand=False)
split["split"] = split["split"].str.lower()

pmid_to_splits = {}
for _, row in split.dropna(subset=["pmid"]).iterrows():
    pmid_to_splits.setdefault(row["pmid"], set()).add(row["split"])

# Track distinct PMIDs containing each missing HPO term
found = {
    hpo: {"development": set(), "test": set()}
    for hpo in missing_hpo
}

files_scanned = 0
json_errors = 0

for path in PHENOPACKETS.rglob("*.json"):
    match = re.search(r"PMID[_\s:-]*(\d+)", path.name, flags=re.I)
    if not match:
        continue

    pmid = match.group(1)
    cohorts = pmid_to_splits.get(pmid, set())

    if not cohorts:
        continue

    try:
        content = path.read_text(encoding="utf-8")
        json.loads(content)
        files_scanned += 1
    except Exception:
        json_errors += 1
        continue

    for hpo in missing_hpo:
        if re.search(
            r"(?<![A-Z0-9:])" + re.escape(hpo) + r"(?![A-Z0-9])",
            content
        ):
            for cohort in ("development", "test"):
                if cohort in cohorts:
                    found[hpo][cohort].add(pmid)

print("PMID-linked phenotype files scanned:", files_scanned)
print("JSON read/parse errors:", json_errors)

print("\nMISSING HPO TERM AUDIT")
for hpo in sorted(missing_hpo):
    print(
        f"{hpo}: "
        f"development PMIDs={len(found[hpo]['development'])}, "
        f"test PMIDs={len(found[hpo]['test'])}"
    )

KeyboardInterrupt: 

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("No GPU detected. Check Runtime > Change runtime type.")


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU count: 1
GPU memory (GB): 15.64


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
REPORT = DATA / "null_value_audit.csv"

results = []

for path in sorted(DATA.glob("*.csv")):
    try:
        # Process in chunks to reduce memory use
        for chunk_number, chunk in enumerate(
            pd.read_csv(path, chunksize=50000, low_memory=False)
        ):
            for col in chunk.columns:
                series = chunk[col]
                null_count = int(series.isna().sum())

                # Count empty/whitespace-only strings separately
                if pd.api.types.is_object_dtype(series):
                    blank_count = int(
                        (series.notna() & series.str.strip().eq("")).sum()
                    )
                else:
                    blank_count = 0

                if null_count or blank_count:
                    results.append({
                        "file": path.name,
                        "column": col,
                        "chunk": chunk_number,
                        "null_values": null_count,
                        "blank_strings": blank_count,
                    })

    except Exception as e:
        print(f"Could not audit {path.name}: {e}")

audit = pd.DataFrame(results)

if audit.empty:
    audit = pd.DataFrame(
        columns=["file", "column", "chunk", "null_values", "blank_strings"]
    )
else:
    audit = (
        audit.groupby(["file", "column"], as_index=False)
        [["null_values", "blank_strings"]].sum()
    )

audit.to_csv(REPORT, index=False)

print("CSV files scanned:", len(list(DATA.glob("*.csv"))))
print("Columns with missing/blank values:", len(audit))
print("Report saved to:", REPORT)

if not audit.empty:
    print("\nColumns with the most missing/blank values:")
    print(
        audit.assign(
            total_missing=audit["null_values"] + audit["blank_strings"]
        )
        .sort_values("total_missing", ascending=False)
        .head(30)
        .to_string(index=False)
    )
else:
    print("No null or blank values found in the scanned CSV files.")


CSV files scanned: 80
Columns with missing/blank values: 51
Report saved to: /content/drive/MyDrive/Rare Disease ai/processed/null_value_audit.csv

Columns with the most missing/blank values:
                                              file         column  null_values  blank_strings  total_missing
                         clean_hpo_annotations.csv            sex       285469              0         285469
                         clean_hpo_annotations.csv      qualifier       285314              0         285314
                         clean_hpo_annotations.csv       modifier       285023              0         285023
                         clean_hpo_annotations.csv          onset       283117              0         283117
      patient_test_leakage_reduced_annotations.csv            sex       280156              0         280156
      patient_test_leakage_reduced_annotations.csv      qualifier       280000              0         280000
      patient_test_leakage_reduced_annotation

In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
path = DATA / "gnn_annotations_publication_disjoint_candidate.csv"

required = ["database_id", "hpo_id"]
rows = 0
missing_disease = 0
missing_hpo = 0
missing_either = 0
pair_counts = {}

for chunk in pd.read_csv(
    path,
    usecols=required,
    dtype=str,
    chunksize=50000,
    low_memory=False
):
    rows += len(chunk)

    disease = chunk["database_id"].fillna("").str.strip()
    hpo = chunk["hpo_id"].fillna("").str.strip()

    bad_disease = disease.eq("")
    bad_hpo = hpo.eq("")

    missing_disease += int(bad_disease.sum())
    missing_hpo += int(bad_hpo.sum())
    missing_either += int((bad_disease | bad_hpo).sum())

    valid = chunk.loc[~(bad_disease | bad_hpo), required]
    counts = valid.value_counts()
    for pair, count in counts.items():
        pair_counts[pair] = pair_counts.get(pair, 0) + int(count)

duplicate_excess = sum(count - 1 for count in pair_counts.values() if count > 1)

print("Rows checked:", rows)
print("Missing/blank disease IDs:", missing_disease)
print("Missing/blank HPO IDs:", missing_hpo)
print("Rows missing either required ID:", missing_either)
print("Unique valid disease-HPO pairs:", len(pair_counts))
print("Duplicate pair rows beyond first occurrence:", duplicate_excess)


Rows checked: 158693
Missing/blank disease IDs: 0
Missing/blank HPO IDs: 0
Rows missing either required ID: 0
Unique valid disease-HPO pairs: 158693
Duplicate pair rows beyond first occurrence: 0


In [ ]:

import re
import json
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PHENOPACKETS = PROJECT / "all_phenopackets"

missing_hpo = [
    "HP:0006346",
    "HP:0500132",
    "HP:0500143",
    "HP:0500144",
]

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
split["pmid"] = split["pmid"].str.extract(r"(\d+)", expand=False)
split["split"] = split["split"].str.lower().str.strip()

pmid_to_splits = {}
for pmid, group in split.dropna(subset=["pmid"]).groupby("pmid"):
    pmid_to_splits[pmid] = set(group["split"])

found = {
    hpo: {"development": set(), "test": set()}
    for hpo in missing_hpo
}

scanned = 0
errors = 0
matched_files = 0

print("Indexing phenotype JSON filenames...", flush=True)

for path in PHENOPACKETS.rglob("*.json"):
    match = re.search(r"PMID[_\s:-]*(\d+)", path.name, flags=re.I)
    if not match:
        continue

    pmid = match.group(1)
    cohorts = pmid_to_splits.get(pmid, set())
    if not cohorts:
        continue

    matched_files += 1

    try:
        # JSON parsing is unnecessary for searching these exact HPO IDs.
        content = path.read_text(encoding="utf-8")
        scanned += 1
    except (OSError, UnicodeError) as e:
        errors += 1
        continue

    for hpo in missing_hpo:
        if hpo in content:
            for cohort in ("development", "test"):
                if cohort in cohorts:
                    found[hpo][cohort].add(pmid)

    if scanned % 200 == 0:
        print(f"Files scanned: {scanned}/{matched_files} matched so far",
              flush=True)

print("\n=== MISSING HPO TERM AUDIT ===")
print("Matching files found:", matched_files)
print("Files read successfully:", scanned)
print("File read errors:", errors)

for hpo in missing_hpo:
    print(
        f"{hpo}: development PMIDs="
        f"{len(found[hpo]['development'])}, test PMIDs="
        f"{len(found[hpo]['test'])}"
    )

report = {
    "files_matched": matched_files,
    "files_scanned": scanned,
    "file_read_errors": errors,
    "missing_hpo_results": {
        hpo: {
            "development_pmids": sorted(found[hpo]["development"]),
            "test_pmids": sorted(found[hpo]["test"]),
        }
        for hpo in missing_hpo
    },
    "note": "Only PMID-linked filenames were checked."
}

report_path = DATA / "missing_hpo_phenotype_audit.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Report saved:", report_path)
print("Audit complete.")


Indexing phenotype JSON filenames...
Files scanned: 200/200 matched so far
Files scanned: 400/400 matched so far
Files scanned: 600/600 matched so far
Files scanned: 800/800 matched so far
Files scanned: 1000/1000 matched so far
Files scanned: 1200/1200 matched so far
Files scanned: 1400/1400 matched so far
Files scanned: 1600/1600 matched so far
Files scanned: 1800/1800 matched so far
Files scanned: 2000/2000 matched so far
Files scanned: 2200/2200 matched so far
Files scanned: 2400/2400 matched so far
Files scanned: 2600/2600 matched so far
Files scanned: 2800/2800 matched so far
Files scanned: 3000/3000 matched so far
Files scanned: 3200/3200 matched so far
Files scanned: 3400/3400 matched so far
Files scanned: 3600/3600 matched so far
Files scanned: 3800/3800 matched so far
Files scanned: 4000/4000 matched so far
Files scanned: 4200/4200 matched so far
Files scanned: 4400/4400 matched so far
Files scanned: 4600/4600 matched so far
Files scanned: 4800/4800 matched so far
Files scann

In [ ]:

import pandas as pd
import re
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

candidate_path = DATA / "gnn_annotations_publication_disjoint_candidate.csv"
split_path = DATA / "patient_publication_split.csv"

annotations = pd.read_csv(
    candidate_path, usecols=["database_id"], dtype=str
)
split = pd.read_csv(split_path, dtype=str)

supported_diseases = set(annotations["database_id"].dropna().str.strip())
print("Diseases in candidate graph:", len(supported_diseases))
print("Publication split columns:", split.columns.tolist())

# Show diagnosis label format before attempting coverage calculations
print("\nExample diagnosis labels:")
print(split[["split", "diagnosis_ids"]].head(10).to_string(index=False))

def extract_ids(value):
    if not isinstance(value, str):
        return set()
    return set(re.findall(
        r"(?:OMIM|ORPHA|MONDO|DECIPHER|DOID):[A-Za-z0-9_.-]+",
        value,
        flags=re.I
    ))

split["diagnosis_id_set"] = split["diagnosis_ids"].apply(extract_ids)

for cohort in ["development", "test"]:
    cohort_df = split[split["split"].str.lower() == cohort]
    labeled_cases = 0
    cases_with_supported_diagnosis = 0
    unsupported_case_count = 0
    unsupported_diagnoses = set()

    for ids in cohort_df["diagnosis_id_set"]:
        if not ids:
            continue
        labeled_cases += 1
        supported = ids & supported_diseases

        if supported:
            cases_with_supported_diagnosis += 1
        else:
            unsupported_case_count += 1
            unsupported_diagnoses.update(ids)

    print(f"\n{cohort.upper()} coverage")
    print("Cases with parsed diagnosis IDs:", labeled_cases)
    print("Cases with at least one supported diagnosis:",
          cases_with_supported_diagnosis)
    print("Cases with no supported diagnosis:", unsupported_case_count)
    print("Distinct unsupported diagnosis IDs:", len(unsupported_diagnoses))
    print("Unsupported ID examples:", sorted(unsupported_diagnoses)[:15])


Diseases in candidate graph: 6980
Publication split columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']

Example diagnosis labels:
      split diagnosis_ids
development   OMIM:277600
development   OMIM:277600
development   OMIM:277600
development   OMIM:277600
development   OMIM:277600
development   OMIM:148600
       test   OMIM:148600
       test   OMIM:148600
       test   OMIM:148600
development   OMIM:102700

DEVELOPMENT coverage
Cases with parsed diagnosis IDs: 8386
Cases with at least one supported diagnosis: 3679
Cases with no supported diagnosis: 4707
Distinct unsupported diagnosis IDs: 459
Unsupported ID examples: ['OMIM:103500', 'OMIM:107970', 'OMIM:115200', 'OMIM:117360', 'OMIM:129600', 'OMIM:132800', 'OMIM:135100', 'OMIM:146000', 'OMIM:147791', 'OMIM:147920', 'OMIM:161200', 'OMIM:162200', 'OMIM:167320', 'OMIM:175050', 'OMIM:177170']

TEST coverage
Cases with parsed diagnosis IDs: 1959
Cases with at least one supported diagnosis: 833
Cases with no supported diagnosis: 11

In [ ]:

import pandas as pd
from pathlib import Path
import re

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)

def normalize_id(x):
    if pd.isna(x):
        return ""
    return str(x).strip().upper()

def extract_diagnoses(x):
    if pd.isna(x):
        return set()
    return {
        normalize_id(v)
        for v in re.findall(
            r"(?:OMIM|ORPHA|MONDO|DECIPHER|DOID):[A-Za-z0-9_.-]+",
            str(x),
            flags=re.I
        )
    }

split["split"] = split["split"].str.lower().str.strip()
split["diagnoses"] = split["diagnosis_ids"].apply(extract_diagnoses)

annotation_files = {
    "Previous filtered": DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    "Publication-disjoint candidate": DATA / "gnn_annotations_publication_disjoint_candidate.csv",
}

for label, path in annotation_files.items():
    ann = pd.read_csv(path, usecols=["database_id"], dtype=str)
    vocabulary = set(ann["database_id"].dropna().map(normalize_id))

    print(f"\n=== {label} ===")
    print("Vocabulary diseases:", len(vocabulary))

    for cohort in ["development", "test"]:
        cases = split[split["split"] == cohort]
        supported_cases = 0
        unsupported_cases = 0
        unsupported_ids = set()

        for diagnoses in cases["diagnoses"]:
            if not diagnoses:
                continue

            if diagnoses & vocabulary:
                supported_cases += 1
            else:
                unsupported_cases += 1
                unsupported_ids.update(diagnoses)

        total = supported_cases + unsupported_cases
        coverage = supported_cases / total if total else 0

        print(
            f"{cohort}: supported={supported_cases}, "
            f"unsupported={unsupported_cases}, "
            f"coverage={coverage:.1%}, "
            f"distinct unsupported IDs={len(unsupported_ids)}"
        )

        if cohort == "test":
            print("Unsupported test ID examples:",
                  sorted(unsupported_ids)[:20])



=== Previous filtered ===
Vocabulary diseases: 6983
development: supported=3734, unsupported=4652, coverage=44.5%, distinct unsupported IDs=456
test: supported=841, unsupported=1118, coverage=42.9%, distinct unsupported IDs=159
Unsupported test ID examples: ['OMIM:107480', 'OMIM:115200', 'OMIM:117360', 'OMIM:130050', 'OMIM:147920', 'OMIM:162200', 'OMIM:206700', 'OMIM:231095', 'OMIM:236200', 'OMIM:240300', 'OMIM:258480', 'OMIM:275400', 'OMIM:300867', 'OMIM:300868', 'OMIM:301109', 'OMIM:301111', 'OMIM:301142', 'OMIM:304110', 'OMIM:600373', 'OMIM:601414']

=== Publication-disjoint candidate ===
Vocabulary diseases: 6980
development: supported=3679, unsupported=4707, coverage=43.9%, distinct unsupported IDs=459
test: supported=833, unsupported=1126, coverage=42.5%, distinct unsupported IDs=161
Unsupported test ID examples: ['OMIM:107480', 'OMIM:115200', 'OMIM:117360', 'OMIM:129600', 'OMIM:130050', 'OMIM:147920', 'OMIM:162200', 'OMIM:206700', 'OMIM:231095', 'OMIM:236200', 'OMIM:240300', 'O

In [ ]:

import pandas as pd
import re
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)

def extract_ids(value):
    if pd.isna(value):
        return set()
    return {
        x.upper()
        for x in re.findall(
            r"(?:OMIM|ORPHA|MONDO|DECIPHER|DOID):[A-Za-z0-9_.-]+",
            str(value),
            flags=re.I
        )
    }

test = split[split["split"].str.lower().eq("test")].copy()
test["diagnoses"] = test["diagnosis_ids"].apply(extract_ids)

test_diagnoses = set().union(*test["diagnoses"]) if len(test) else set()

files = {
    "Original annotations": DATA / "gnn_training_annotations.csv",
    "Previous filtered": DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    "Publication-disjoint candidate": DATA / "gnn_annotations_publication_disjoint_candidate.csv",
}

for name, path in files.items():
    ann = pd.read_csv(path, usecols=["database_id"], dtype=str)
    vocab = set(ann["database_id"].dropna().str.strip().str.upper())

    overlap = test_diagnoses & vocab
    missing = test_diagnoses - vocab

    print(f"\n{name}")
    print("Unique test diagnosis IDs:", len(test_diagnoses))
    print("Represented in annotations:", len(overlap))
    print("Not represented:", len(missing))
    print("Missing ID examples:", sorted(missing)[:15])



Original annotations
Unique test diagnosis IDs: 258
Represented in annotations: 195
Not represented: 63
Missing ID examples: ['OMIM:107480', 'OMIM:115200', 'OMIM:130050', 'OMIM:147920', 'OMIM:162200', 'OMIM:206700', 'OMIM:231095', 'OMIM:236200', 'OMIM:240300', 'OMIM:258480', 'OMIM:275400', 'OMIM:300867', 'OMIM:300868', 'OMIM:304110', 'OMIM:600373']

Previous filtered
Unique test diagnosis IDs: 258
Represented in annotations: 99
Not represented: 159
Missing ID examples: ['OMIM:107480', 'OMIM:115200', 'OMIM:117360', 'OMIM:130050', 'OMIM:147920', 'OMIM:162200', 'OMIM:206700', 'OMIM:231095', 'OMIM:236200', 'OMIM:240300', 'OMIM:258480', 'OMIM:275400', 'OMIM:300867', 'OMIM:300868', 'OMIM:301109']

Publication-disjoint candidate
Unique test diagnosis IDs: 258
Represented in annotations: 97
Not represented: 161
Missing ID examples: ['OMIM:107480', 'OMIM:115200', 'OMIM:117360', 'OMIM:129600', 'OMIM:130050', 'OMIM:147920', 'OMIM:162200', 'OMIM:206700', 'OMIM:231095', 'OMIM:236200', 'OMIM:240300

In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test = split[split["split"].str.lower().eq("test")].copy()

def normalize_ids(series):
    return set(series.dropna().astype(str).str.strip().str.upper()) - {""}

test_ids = set()
for value in test["diagnosis_ids"].dropna():
    import re
    test_ids.update(
        x.upper() for x in re.findall(
            r"(?:OMIM|ORPHA|MONDO|DECIPHER|DOID):[A-Za-z0-9_.-]+",
            str(value), flags=re.I
        )
    )

paths = {
    "original": DATA / "gnn_training_annotations.csv",
    "previous": DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    "candidate": DATA / "gnn_annotations_publication_disjoint_candidate.csv",
}

vocabs = {}
for name, path in paths.items():
    ids = pd.read_csv(path, usecols=["database_id"], dtype=str)["database_id"]
    vocabs[name] = normalize_ids(ids)

original_only = sorted(test_ids & (vocabs["original"] - vocabs["previous"]))
previous_only = sorted(test_ids & (vocabs["previous"] - vocabs["candidate"]))
missing_original = sorted(test_ids - vocabs["original"])

print("Test diagnosis IDs:", len(test_ids))
print("Present originally, absent after previous filtering:", len(original_only))
print("Present in previous, absent in candidate:", len(previous_only))
print("Absent even from original annotations:", len(missing_original))

print("\nExamples lost after previous filtering:")
print(original_only[:30])

print("\nExamples lost in the candidate step:")
print(previous_only[:30])

print("\nExamples absent from original annotations:")
print(missing_original[:30])


Test diagnosis IDs: 258
Present originally, absent after previous filtering: 96
Present in previous, absent in candidate: 2
Absent even from original annotations: 63

Examples lost after previous filtering:
['OMIM:117360', 'OMIM:301109', 'OMIM:301111', 'OMIM:301142', 'OMIM:608908', 'OMIM:609296', 'OMIM:609622', 'OMIM:610687', 'OMIM:610842', 'OMIM:611182', 'OMIM:614199', 'OMIM:615193', 'OMIM:615277', 'OMIM:615829', 'OMIM:616457', 'OMIM:616873', 'OMIM:616975', 'OMIM:617433', 'OMIM:617516', 'OMIM:617560', 'OMIM:617622', 'OMIM:617865', 'OMIM:617988', 'OMIM:618977', 'OMIM:619598', 'OMIM:619656', 'OMIM:619657', 'OMIM:620344', 'OMIM:620376', 'OMIM:620398']

Examples lost in the candidate step:
['OMIM:129600', 'OMIM:610187']

Examples absent from original annotations:
['OMIM:107480', 'OMIM:115200', 'OMIM:130050', 'OMIM:147920', 'OMIM:162200', 'OMIM:206700', 'OMIM:231095', 'OMIM:236200', 'OMIM:240300', 'OMIM:258480', 'OMIM:275400', 'OMIM:300867', 'OMIM:300868', 'OMIM:304110', 'OMIM:600373', 'OM

In [ ]:

import pandas as pd
import re
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)

def extract_ids(value):
    if pd.isna(value):
        return set()
    return {
        x.upper() for x in re.findall(
            r"(?:OMIM|ORPHA|MONDO|DECIPHER|DOID):[A-Za-z0-9_.-]+",
            str(value), flags=re.I
        )
    }

def extract_pmids(value):
    if pd.isna(value):
        return set()
    return set(re.findall(r"(?<!\d)\d{7,9}(?!\d)", str(value)))

dev_pmids = set()
test_pmids = set()

for _, row in split.iterrows():
    pmids = extract_pmids(row.get("publication_ids", ""))
    if str(row["split"]).lower() == "development":
        dev_pmids.update(pmids)
    elif str(row["split"]).lower() == "test":
        test_pmids.update(pmids)

original = pd.read_csv(
    DATA / "gnn_training_annotations.csv",
    dtype=str,
    keep_default_na=False
)

# Reproduce the diagnosis IDs lost between original and previous filtered files.
previous = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    usecols=["database_id"],
    dtype=str
)

original_vocab = set(original["database_id"].str.strip().str.upper())
previous_vocab = set(previous["database_id"].dropna().str.strip().str.upper())

test_ids = set().union(
    *(extract_ids(x) for x in split.loc[
        split["split"].str.lower().eq("test"), "diagnosis_ids"
    ])
)

lost_ids = sorted(test_ids & original_vocab - previous_vocab)

print("Test diagnosis IDs lost after previous filtering:", len(lost_ids))
print("Scanning their original annotation rows...")

rows = original[
    original["database_id"].str.strip().str.upper().isin(lost_ids)
].copy()

def classify_reference(value):
    pmids = extract_pmids(value)
    has_dev = bool(pmids & dev_pmids)
    has_test = bool(pmids & test_pmids)

    if has_dev and has_test:
        return "References development AND test PMIDs"
    if has_dev:
        return "References development PMID"
    if has_test:
        return "References test PMID"
    if pmids:
        return "Has PMID, neither split"
    return "No recognizable PMID"

rows["reference_class"] = rows["reference"].apply(classify_reference)

print("\nRows for lost diagnoses:", len(rows))
print("\nReference classification:")
print(rows["reference_class"].value_counts(dropna=False).to_string())

print("\nExample rows:")
cols = [
    c for c in ["database_id", "hpo_id", "reference", "reference_class"]
    if c in rows.columns
]
print(rows[cols].head(15).to_string(index=False))


Test diagnosis IDs lost after previous filtering: 96
Scanning their original annotation rows...

Rows for lost diagnoses: 4985

Reference classification:
reference_class
Has PMID, neither split    4985

Example rows:
database_id     hpo_id                   reference         reference_class
OMIM:621344 HP:0004325               PMID:34509675 Has PMID, neither split
OMIM:621344 HP:0002421               PMID:34509675 Has PMID, neither split
OMIM:621344 HP:0000486 PMID:34509675;PMID:40576023 Has PMID, neither split
OMIM:621344 HP:0002540 PMID:34509675;PMID:40576023 Has PMID, neither split
OMIM:621344 HP:0000365               PMID:40576023 Has PMID, neither split
OMIM:621344 HP:0004322               PMID:34509675 Has PMID, neither split
OMIM:621344 HP:0001332               PMID:40576023 Has PMID, neither split
OMIM:621344 HP:0002141               PMID:40576023 Has PMID, neither split
OMIM:621344 HP:0002020               PMID:40576023 Has PMID, neither split
OMIM:621344 HP:0003593 PMID:34509

In [ ]:

import pandas as pd
import re
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

original = pd.read_csv(
    DATA / "gnn_training_annotations.csv", dtype=str, keep_default_na=False
)
previous = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    dtype=str, keep_default_na=False
)
candidate = pd.read_csv(
    DATA / "gnn_annotations_publication_disjoint_candidate.csv",
    dtype=str, keep_default_na=False
)

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)

def ids(series):
    return set(series.astype(str).str.strip().str.upper()) - {""}

test = split[split["split"].str.lower().eq("test")]
test_ids = set()
for value in test["diagnosis_ids"].dropna():
    test_ids.update(re.findall(
        r"(?:OMIM|ORPHA|MONDO|DECIPHER|DOID):[A-Za-z0-9_.-]+",
        value, flags=re.I
    ))

lost_ids = (
    test_ids
    & ids(original["database_id"])
    - ids(previous["database_id"])
)

def pair_set(df):
    return set(zip(
        df["database_id"].str.strip().str.upper(),
        df["hpo_id"].str.strip().str.upper()
    ))

orig_pairs = pair_set(original)
prev_pairs = pair_set(previous)
cand_pairs = pair_set(candidate)

lost_pairs = {
    pair for pair in orig_pairs if pair[0] in lost_ids
}

print("Lost test diagnosis IDs:", len(lost_ids))
print("Original disease-HPO pairs for these IDs:", len(lost_pairs))
print("Pairs retained in previous filtered file:", len(lost_pairs & prev_pairs))
print("Pairs retained in candidate:", len(lost_pairs & cand_pairs))
print("Pairs absent from both filtered files:", len(lost_pairs - prev_pairs))
print("Pairs present in previous but absent in candidate:",
      len((lost_pairs & prev_pairs) - cand_pairs))

print("\nExample lost pairs:")
for disease, hpo in sorted(lost_pairs - prev_pairs)[:20]:
    print(disease, hpo)


Lost test diagnosis IDs: 96
Original disease-HPO pairs for these IDs: 4985
Pairs retained in previous filtered file: 0
Pairs retained in candidate: 0
Pairs absent from both filtered files: 4985
Pairs present in previous but absent in candidate: 0

Example lost pairs:
OMIM:117360 HP:0000006
OMIM:117360 HP:0000028
OMIM:117360 HP:0000089
OMIM:117360 HP:0000218
OMIM:117360 HP:0000252
OMIM:117360 HP:0000448
OMIM:117360 HP:0000474
OMIM:117360 HP:0000486
OMIM:117360 HP:0000508
OMIM:117360 HP:0000514
OMIM:117360 HP:0000526
OMIM:117360 HP:0000540
OMIM:117360 HP:0000555
OMIM:117360 HP:0000563
OMIM:117360 HP:0000565
OMIM:117360 HP:0000570
OMIM:117360 HP:0000582
OMIM:117360 HP:0000616
OMIM:117360 HP:0000617
OMIM:117360 HP:0000639


In [ ]:

import pandas as pd
import re
import json
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

original = pd.read_csv(
    DATA / "gnn_training_annotations.csv",
    dtype=str, keep_default_na=False
)
split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)

def pmids(value):
    return set(re.findall(r"(?<!\d)\d{7,9}(?!\d)", str(value)))

dev_pmids, test_pmids = set(), set()

for _, row in split.iterrows():
    p = pmids(row.get("publication_ids", ""))
    if row["split"].lower() == "development":
        dev_pmids.update(p)
    elif row["split"].lower() == "test":
        test_pmids.update(p)

test = split[split["split"].str.lower().eq("test")]
test_diseases = set()
for value in test["diagnosis_ids"].dropna():
    test_diseases.update(re.findall(
        r"(?:OMIM|ORPHA|MONDO|DECIPHER|DOID):[A-Za-z0-9_.-]+",
        value, flags=re.I
    ))

previous = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    usecols=["database_id"], dtype=str
)
previous_vocab = set(previous["database_id"].dropna().str.upper().str.strip())
original_vocab = set(original["database_id"].str.upper().str.strip())

lost = (test_diseases & original_vocab) - previous_vocab
rows = original[
    original["database_id"].str.upper().str.strip().isin(lost)
].copy()

def classify(ref):
    p = pmids(ref)
    if p & dev_pmids and p & test_pmids:
        return "development_and_test"
    if p & dev_pmids:
        return "development"
    if p & test_pmids:
        return "test"
    if p:
        return "other_PMID"
    return "no_PMID"

rows["reference_class"] = rows["reference"].apply(classify)

report = {
    "lost_test_diagnosis_ids": len(lost),
    "original_rows_for_lost_ids": len(rows),
    "reference_class_counts": rows["reference_class"].value_counts().to_dict(),
    "example_rows": rows[
        ["database_id", "hpo_id", "reference", "reference_class"]
    ].head(25).to_dict("records"),
}

out = DATA / "lost_test_diagnosis_filter_audit.json"
out.write_text(json.dumps(report, indent=2, default=str))

print(json.dumps(report, indent=2, default=str))
print("\nSaved report:", out)


{
  "lost_test_diagnosis_ids": 96,
  "original_rows_for_lost_ids": 4985,
  "reference_class_counts": {
    "other_PMID": 4985
  },
  "example_rows": [
    {
      "database_id": "OMIM:621344",
      "hpo_id": "HP:0004325",
      "reference": "PMID:34509675",
      "reference_class": "other_PMID"
    },
    {
      "database_id": "OMIM:621344",
      "hpo_id": "HP:0002421",
      "reference": "PMID:34509675",
      "reference_class": "other_PMID"
    },
    {
      "database_id": "OMIM:621344",
      "hpo_id": "HP:0000486",
      "reference": "PMID:34509675;PMID:40576023",
      "reference_class": "other_PMID"
    },
    {
      "database_id": "OMIM:621344",
      "hpo_id": "HP:0002540",
      "reference": "PMID:34509675;PMID:40576023",
      "reference_class": "other_PMID"
    },
    {
      "database_id": "OMIM:621344",
      "hpo_id": "HP:0000365",
      "reference": "PMID:40576023",
      "reference_class": "other_PMID"
    },
    {
      "database_id": "OMIM:621344",
      "hpo_id"

In [ ]:

import pandas as pd
import re
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
original = pd.read_csv(
    DATA / "gnn_training_annotations.csv", dtype=str, keep_default_na=False
)
previous = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    dtype=str, keep_default_na=False
)

def get_pmids(x):
    return set(re.findall(r"(?<!\d)\d{7,9}(?!\d)", str(x)))

dev, test = set(), set()
for _, r in split.iterrows():
    p = get_pmids(r.get("publication_ids", ""))
    if r["split"].lower() == "development":
        dev.update(p)
    elif r["split"].lower() == "test":
        test.update(p)

print("Split PMID counts:", len(dev), "development;", len(test), "test")
print("PMID 34509675 in development:", "34509675" in dev)
print("PMID 34509675 in test:", "34509675" in test)
print("PMID 40576023 in development:", "40576023" in dev)
print("PMID 40576023 in test:", "40576023" in test)

for disease in ["OMIM:621344", "OMIM:117360"]:
    a = original[original["database_id"].str.upper().str.strip() == disease]
    b = previous[previous["database_id"].str.upper().str.strip() == disease]
    print(f"\n{disease}: original rows={len(a)}, previous rows={len(b)}")
    print("Original references:", a["reference"].head(5).tolist())


Split PMID counts: 0 development; 0 test
PMID 34509675 in development: False
PMID 34509675 in test: False
PMID 40576023 in development: False
PMID 40576023 in test: False

OMIM:621344: original rows=120, previous rows=0
Original references: ['PMID:34509675', 'PMID:34509675', 'PMID:34509675;PMID:40576023', 'PMID:34509675;PMID:40576023', 'PMID:40576023']

OMIM:117360: original rows=141, previous rows=0
Original references: ['PMID:37821226', 'PMID:25981959;PMID:37821226;PMID:27062503;PMID:25794864', 'PMID:22986007', 'PMID:37964426;PMID:35743164;PMID:22986007', 'PMID:22986007']


In [ ]:

import pandas as pd
from pathlib import Path

path = Path("/content/drive/MyDrive/Rare Disease ai/processed/patient_publication_split.csv")
split = pd.read_csv(path, dtype=str)

print("Columns:")
print(split.columns.tolist())

print("\nSample rows:")
print(split.head(3).to_string(index=False))

print("\nNon-empty counts:")
print(split.notna().sum().to_string())



Columns:
['case_id', 'pmid', 'split', 'diagnosis_ids']

Sample rows:
                    case_id     pmid       split diagnosis_ids
PMID_19836009_Family_E_II_6 19836009 development   OMIM:277600
PMID_19836009_Family_E_II_3 19836009 development   OMIM:277600
PMID_19836009_Family_B_II_4 19836009 development   OMIM:277600

Non-empty counts:
case_id          10345
pmid             10345
split            10345
diagnosis_ids    10345


In [ ]:

import pandas as pd
import re
import json
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
original = pd.read_csv(
    DATA / "gnn_training_annotations.csv",
    dtype=str, keep_default_na=False
)
previous = pd.read_csv(
    DATA / "gnn_annotations_excluding_dev_test_pmids.csv",
    usecols=["database_id"], dtype=str
)

def extract_pmids(value):
    return set(re.findall(r"(?<!\d)\d{7,9}(?!\d)", str(value)))

dev_pmids, test_pmids = set(), set()
for _, row in split.iterrows():
    p = extract_pmids(row["pmid"])
    if row["split"].lower() == "development":
        dev_pmids.update(p)
    elif row["split"].lower() == "test":
        test_pmids.update(p)

def normalize_ids(values):
    return set(values.astype(str).str.strip().str.upper()) - {""}

test_rows = split[split["split"].str.lower().eq("test")]
test_diseases = set()
for value in test_rows["diagnosis_ids"].dropna():
    test_diseases.update(re.findall(
        r"(?:OMIM|ORPHA|MONDO|DECIPHER|DOID):[A-Za-z0-9_.-]+",
        value, flags=re.I
    ))

lost_ids = (
    test_diseases
    & normalize_ids(original["database_id"])
    - normalize_ids(previous["database_id"])
)

rows = original[
    original["database_id"].str.strip().str.upper().isin(lost_ids)
].copy()

def classify(ref):
    p = extract_pmids(ref)
    d, t = bool(p & dev_pmids), bool(p & test_pmids)
    if d and t: return "development_and_test"
    if d: return "development"
    if t: return "test"
    if p: return "other_PMID"
    return "no_PMID"

rows["reference_class"] = rows["reference"].apply(classify)

print("Development PMIDs:", len(dev_pmids))
print("Test PMIDs:", len(test_pmids))
print("Lost test diagnosis IDs:", len(lost_ids))
print("Rows for lost diagnoses:", len(rows))
print("\nReference classification:")
print(rows["reference_class"].value_counts().to_string())

report_path = DATA / "lost_test_diagnosis_filter_audit_corrected.json"
report_path.write_text(json.dumps({
    "development_pmid_count": len(dev_pmids),
    "test_pmid_count": len(test_pmids),
    "lost_test_diagnosis_ids": sorted(lost_ids),
    "reference_class_counts": rows["reference_class"].value_counts().to_dict(),
    "example_rows": rows[
        ["database_id", "hpo_id", "reference", "reference_class"]
    ].head(30).to_dict("records")
}, indent=2, default=str))

print("\nSaved:", report_path)


Development PMIDs: 1383
Test PMIDs: 346
Lost test diagnosis IDs: 96
Rows for lost diagnoses: 4985

Reference classification:
reference_class
test                    2378
development             1750
development_and_test     857

Saved: /content/drive/MyDrive/Rare Disease ai/processed/lost_test_diagnosis_filter_audit_corrected.json


In [ ]:

from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
DATA = PROJECT / "processed"

checks = [
    DATA / "patient_publication_split.csv",
    DATA / "patient_phenotype_only_baseline.csv",
    DATA / "gnn_annotations_publication_disjoint_candidate.csv",
    MODEL_DIR / "rare_disease_gnn_filtered_best.pt",
    MODEL_DIR / "patient_level_ranker_best.pt",
]

for path in checks:
    print(f"{'FOUND' if path.exists() else 'MISSING'} | {path.name}"
          + (f" | {path.stat().st_size / (1024**2):.1f} MB"
             if path.exists() and path.is_file() else ""))

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


FOUND | patient_publication_split.csv | 0.6 MB
FOUND | patient_phenotype_only_baseline.csv | 0.1 MB
FOUND | gnn_annotations_publication_disjoint_candidate.csv | 19.7 MB
FOUND | rare_disease_gnn_filtered_best.pt | 4.6 MB
FOUND | patient_level_ranker_best.pt | 9.0 MB

CUDA available: True
GPU: Tesla T4


In [ ]:

import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
checkpoints = [
    PROJECT / "model" / "rare_disease_gnn_filtered_best.pt",
    PROJECT / "model" / "patient_level_ranker_best.pt",
]

for path in checkpoints:
    print(f"\n{'=' * 60}\n{path.name}")
    try:
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )
        if isinstance(checkpoint, dict):
            print("Top-level keys:", list(checkpoint.keys())[:40])
            for key, value in checkpoint.items():
                if isinstance(value, dict):
                    print(f"  {key}: dictionary with {len(value)} entries")
                elif isinstance(value, torch.Tensor):
                    print(f"  {key}: tensor shape {tuple(value.shape)}")
                elif isinstance(value, (str, int, float, bool, type(None))):
                    print(f"  {key}: {value}")
        else:
            print("Checkpoint type:", type(checkpoint).__name__)
    except Exception as e:
        print("Inspection error:", type(e).__name__, str(e)[:400])



rare_disease_gnn_filtered_best.pt
Inspection error: UnpicklingError Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code exe

patient_level_ranker_best.pt
Top-level keys: ['model_state_dict', 'n_diseases', 'n_phenotypes', 'hidden_dim', 'disease_ids', 'phenotype_ids', 'best_validation_mrr', 'seed', 'training_cases', 'validation_cases', 'note']
  model_state_dict: dictionary with 7 entries
  n_diseases: 7340
  n_phenotypes: 9982
  hidden_dim: 128
  best_validation_mrr: 0.052704318912176605
  seed: 42
  training_cases: 5730
  validation_cases: 906
  note: Patient-level ranking model. Validation split by PMID. Unrecorded diseases used as sampled competitors a

In [ ]:

import torch
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/rare_disease_gnn_filtered_best.pt"
)

try:
    checkpoint = torch.load(
        path, map_location="cpu", weights_only=True
    )
except Exception as e:
    print("Weights-only loading failed:", type(e).__name__)
    print("Only use the fallback if this is your own trusted checkpoint.")

    # Set this to True ONLY if you created this checkpoint yourself
    # and trust its contents.
    TRUST_MY_OWN_CHECKPOINT = False

    if not TRUST_MY_OWN_CHECKPOINT:
        print("Stopped safely; no unrestricted loading attempted.")
        checkpoint = None
    else:
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=False
        )

if checkpoint is not None:
    print("Checkpoint type:", type(checkpoint).__name__)
    if isinstance(checkpoint, dict):
        print("Top-level keys:", list(checkpoint.keys())[:50])
        for key, value in checkpoint.items():
            if isinstance(value, dict):
                print(f"{key}: dictionary, {len(value)} entries")
            elif isinstance(value, torch.Tensor):
                print(f"{key}: tensor, shape={tuple(value.shape)}")
            elif isinstance(value, (str, int, float, bool, type(None))):
                print(f"{key}: {value}")


Weights-only loading failed: UnpicklingError
Only use the fallback if this is your own trusted checkpoint.
Stopped safely; no unrestricted loading attempted.


In [ ]:

from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/rare_disease_gnn_filtered_best.pt"
)

with path.open("rb") as f:
    header = f.read(2_000_000)

print("File size (MB):", round(path.stat().st_size / (1024**2), 2))
print("PyTorch ZIP checkpoint:", header.startswith(b"PK"))

for term in [
    b"model_state_dict",
    b"disease_ids",
    b"phenotype_ids",
    b"n_diseases",
    b"n_phenotypes",
    b"best_validation",
]:
    print(term.decode(), "found in readable bytes:", term in header)


File size (MB): 4.63
PyTorch ZIP checkpoint: True
model_state_dict found in readable bytes: True
disease_ids found in readable bytes: True
phenotype_ids found in readable bytes: True
n_diseases found in readable bytes: False
n_phenotypes found in readable bytes: False
best_validation found in readable bytes: True


In [ ]:

import zipfile
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/rare_disease_gnn_filtered_best.pt"
)

with zipfile.ZipFile(path, "r") as z:
    names = z.namelist()
    print("Archive entries:")
    for name in names[:40]:
        info = z.getinfo(name)
        print(f"{name} | {info.file_size:,} bytes")

    print("\nSmall metadata-like entries:")
    for name in names:
        if any(k in name.lower() for k in ["data.pkl", "metadata", "version"]):
            print(name)


Archive entries:
rare_disease_gnn_filtered_best/data.pkl | 354,762 bytes
rare_disease_gnn_filtered_best/.format_version | 1 bytes
rare_disease_gnn_filtered_best/.storage_alignment | 2 bytes
rare_disease_gnn_filtered_best/byteorder | 6 bytes
rare_disease_gnn_filtered_best/data/0 | 4,434,432 bytes
rare_disease_gnn_filtered_best/data/1 | 256 bytes
rare_disease_gnn_filtered_best/data/2 | 16,384 bytes
rare_disease_gnn_filtered_best/data/3 | 256 bytes
rare_disease_gnn_filtered_best/data/4 | 16,384 bytes
rare_disease_gnn_filtered_best/data/5 | 32,768 bytes
rare_disease_gnn_filtered_best/data/6 | 256 bytes
rare_disease_gnn_filtered_best/data/7 | 256 bytes
rare_disease_gnn_filtered_best/data/8 | 4 bytes
rare_disease_gnn_filtered_best/version | 2 bytes
rare_disease_gnn_filtered_best/.data/serialization_id | 40 bytes

Small metadata-like entries:
rare_disease_gnn_filtered_best/data.pkl
rare_disease_gnn_filtered_best/.format_version
rare_disease_gnn_filtered_best/version


In [ ]:

import torch
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
ranker_path = PROJECT / "model" / "patient_level_ranker_best.pt"

ckpt = torch.load(ranker_path, map_location="cpu", weights_only=True)

ann = pd.read_csv(
    DATA / "gnn_annotations_publication_disjoint_candidate.csv",
    usecols=["database_id", "hpo_id"], dtype=str
)
split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)

ann_diseases = set(ann["database_id"].dropna().str.strip().str.upper())
ann_hpos = set(ann["hpo_id"].dropna().str.strip().str.upper())

model_diseases = {
    str(x).strip().upper() for x in ckpt["disease_ids"]
}
model_hpos = {
    str(x).strip().upper() for x in ckpt["phenotype_ids"]
}

print("Ranker disease vocabulary:", len(model_diseases))
print("Candidate disease vocabulary:", len(ann_diseases))
print("Shared disease IDs:", len(model_diseases & ann_diseases))
print("Ranker-only disease IDs:", len(model_diseases - ann_diseases))
print("Candidate-only disease IDs:", len(ann_diseases - model_diseases))

print("\nRanker phenotype vocabulary:", len(model_hpos))
print("Candidate phenotype vocabulary:", len(ann_hpos))
print("Shared phenotype IDs:", len(model_hpos & ann_hpos))
print("Ranker-only phenotype IDs:", len(model_hpos - ann_hpos))
print("Candidate-only phenotype IDs:", len(ann_hpos - model_hpos))

print("\nSplit rows:", len(split))
print("Test rows:", (split["split"].str.lower() == "test").sum())


Ranker disease vocabulary: 7340
Candidate disease vocabulary: 6980
Shared disease IDs: 6980
Ranker-only disease IDs: 360
Candidate-only disease IDs: 0

Ranker phenotype vocabulary: 9982
Candidate phenotype vocabulary: 9613
Shared phenotype IDs: 9613
Ranker-only phenotype IDs: 369
Candidate-only phenotype IDs: 0

Split rows: 10345
Test rows: 1959


In [ ]:

import torch
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/patient_level_ranker_best.pt"
)
ckpt = torch.load(path, map_location="cpu", weights_only=True)

print("Saved training note:", ckpt.get("note"))
print("Seed:", ckpt.get("seed"))
print("Training cases:", ckpt.get("training_cases"))
print("Validation cases:", ckpt.get("validation_cases"))
print("Best validation MRR:", ckpt.get("best_validation_mrr"))

print("\nFirst 15 disease IDs:")
print(list(ckpt["disease_ids"][:15]))

print("\nFirst 15 phenotype IDs:")
print(list(ckpt["phenotype_ids"][:15]))


Saved training note: Patient-level ranking model. Validation split by PMID. Unrecorded diseases used as sampled competitors are not guaranteed negatives.
Seed: 42
Training cases: 5730
Validation cases: 906
Best validation MRR: 0.052704318912176605

First 15 disease IDs:
['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400', 'OMIM:101600', 'OMIM:101800', 'OMIM:102200', 'OMIM:102370', 'OMIM:102500', 'OMIM:102700', 'OMIM:103050', 'OMIM:103285', 'OMIM:103580', 'OMIM:103900']

First 15 phenotype IDs:
['HP:0000002', 'HP:0000003', 'HP:0000006', 'HP:0000007', 'HP:0000008', 'HP:0000009', 'HP:0000010', 'HP:0000011', 'HP:0000012', 'HP:0000013', 'HP:0000014', 'HP:0000015', 'HP:0000016', 'HP:0000017', 'HP:0000019']


In [ ]:

import pandas as pd
import torch
import re
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

ckpt = torch.load(
    PROJECT / "model" / "patient_level_ranker_best.pt",
    map_location="cpu",
    weights_only=True
)

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test = split[split["split"].str.lower().eq("test")].copy()

disease_vocab = {
    str(x).strip().upper() for x in ckpt["disease_ids"]
}

def extract_diseases(value):
    return set(re.findall(
        r"(?:OMIM|ORPHA|MONDO|DECIPHER|DOID):[A-Za-z0-9_.-]+",
        str(value), flags=re.I
    ))

test["diagnoses"] = test["diagnosis_ids"].apply(extract_diseases)
test["supported"] = test["diagnoses"].apply(
    lambda ids: bool({x.upper() for x in ids} & disease_vocab)
)

print("Test patients:", len(test))
print("Patients with at least one diagnosis in ranker vocabulary:",
      int(test["supported"].sum()))
print("Patients with no diagnosis in ranker vocabulary:",
      int((~test["supported"]).sum()))
print("Patient-level vocabulary coverage:",
      round(test["supported"].mean(), 4))

print("\nUnique test diagnosis IDs:",
      len(set().union(*test["diagnoses"])))
print("Unique test diagnosis IDs in ranker vocabulary:",
      len(set().union(*test["diagnoses"]) & disease_vocab))


Test patients: 1959
Patients with at least one diagnosis in ranker vocabulary: 1492
Patients with no diagnosis in ranker vocabulary: 467
Patient-level vocabulary coverage: 0.7616

Unique test diagnosis IDs: 258
Unique test diagnosis IDs in ranker vocabulary: 195


In [ ]:

from pathlib import Path
import ast

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=" * 65)
print("1. PROJECT AND DATASET CHECK")
print("=" * 65)
print("Project exists:", PROJECT.exists())
print("Processed directory exists:", DATA.exists())
print("Model directory exists:", MODEL_DIR.exists())

required = [
    DATA / "patient_publication_split.csv",
    DATA / "gnn_annotations_publication_disjoint_candidate.csv",
    DATA / "patient_phenotype_only_baseline.csv",
    MODEL_DIR / "rare_disease_gnn_filtered_best.pt",
    MODEL_DIR / "patient_level_ranker_best.pt",
]

for p in required:
    print(f"{'FOUND' if p.exists() else 'MISSING'} | {p.relative_to(PROJECT)}")

print("\n" + "=" * 65)
print("2. TRAINING SCRIPT DISCOVERY")
print("=" * 65)

skip_dirs = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    "site-packages", "node_modules"
}
scripts = []
for p in PROJECT.rglob("*.py"):
    if not any(part in skip_dirs for part in p.parts):
        scripts.append(p)

keywords = (
    "train", "gnn", "ranker", "phenotype", "annotation",
    "publication_disjoint", "patient_publication_split"
)

matches = []
for p in scripts:
    try:
        text = p.read_text(errors="ignore")
    except Exception:
        continue
    relevant = [line.strip() for line in text.splitlines()
                if any(k in line.lower() for k in keywords)]
    if relevant:
        matches.append((p, relevant))

if not matches:
    print("No relevant Python training scripts found in the project.")
else:
    for p, lines in matches[:15]:
        print(f"\nSCRIPT: {p.relative_to(PROJECT)}")
        for line in lines[:12]:
            print("  ", line[:220])
        if len(lines) > 12:
            print(f"   ... {len(lines) - 12} more matching lines")

print("\n" + "=" * 65)
print("3. NOTEBOOK DISCOVERY")
print("=" * 65)

notebooks = []
for p in PROJECT.rglob("*.ipynb"):
    if not any(part in skip_dirs for part in p.parts):
        notebooks.append(p)

print("Notebook count:", len(notebooks))
for p in notebooks[:25]:
    print(p.relative_to(PROJECT))

print("\nAUDIT ONLY: no training started; no files modified.")


1. PROJECT AND DATASET CHECK
Project exists: True
Processed directory exists: True
Model directory exists: True
FOUND | processed/patient_publication_split.csv
FOUND | processed/gnn_annotations_publication_disjoint_candidate.csv
FOUND | processed/patient_phenotype_only_baseline.csv
FOUND | model/rare_disease_gnn_filtered_best.pt
FOUND | model/patient_level_ranker_best.pt

2. TRAINING SCRIPT DISCOVERY

SCRIPT: model/rare_disease_predictor.py
   for disease, disease_terms in phenotype_map.items():
   phenotype_score = (
   (1 - gene_weight) * phenotype_score
   combined_score = phenotype_score
   "phenotype_score": phenotype_score,
   "matching_phenotypes": len(overlap),
   ["combined_score", "matching_phenotypes", "disease_id"],
   "Disease name unavailable in annotations"
   # Display matching phenotype IDs for transparency.
   results["matching_phenotype_ids"] = results["disease_id"].apply(
   lambda d: ";".join(sorted(observed & phenotype_map.get(d, set())))
   "matching_phenotypes",

In [ ]:

from pathlib import Path
import re

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

patterns = [
    r"\.backward\s*\(",
    r"\.step\s*\(",
    r"optimizer",
    r"state_dict\s*\(",
    r"torch\.save",
    r"load_state_dict",
    r"DataLoader",
    r"def\s+train",
    r"epochs?",
    r"patient_level_ranker",
    r"rare_disease_gnn",
]

skip = {
    "__pycache__", ".git", ".ipynb_checkpoints",
    "node_modules", "site-packages"
}

files = [
    p for p in PROJECT.rglob("*.py")
    if not any(part in skip for part in p.parts)
]

print("Python files found:", len(files))

for path in files:
    try:
        lines = path.read_text(errors="ignore").splitlines()
    except Exception:
        continue

    hits = [
        (i + 1, line.strip())
        for i, line in enumerate(lines)
        if any(re.search(pattern, line, re.I) for pattern in patterns)
    ]

    if hits:
        print(f"\n--- {path.relative_to(PROJECT)} ---")
        for lineno, line in hits[:18]:
            print(f"{lineno}: {line[:200]}")
        if len(hits) > 18:
            print(f"... plus {len(hits) - 18} matching lines")

print("\nInspection only; no training started and no files modified.")


Python files found: 3

Inspection only; no training started and no files modified.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

for folder in [PROJECT / "model", PROJECT / "processed"]:
    print(f"\n=== {folder.relative_to(PROJECT)} ===")
    if not folder.exists():
        print("MISSING")
        continue

    for path in sorted(folder.iterdir()):
        if path.is_file():
            print(f"{path.name} ({path.stat().st_size / 1024:.1f} KB)")
        elif path.is_dir():
            print(f"{path.name}/")

print("\nInventory only — no files modified and no training started.")



=== model ===
__pycache__/
filtered_gnn_training_config.json (0.7 KB)
patient_level_ranker_best.pt (9167.6 KB)
patient_level_ranker_config.json (0.3 KB)
prediction_config.json (0.5 KB)
rare_disease_ai.py (2.3 KB)
rare_disease_data_loader.py (1.8 KB)
rare_disease_gnn_filtered_best.pt (4745.2 KB)
rare_disease_predictor.py (2.9 KB)

=== processed ===
annotation_leakage_readiness_audit.json (1.4 KB)
annotation_provenance_followup_audit.json (266.6 KB)
baseline_coverage_comparison_audit.csv (0.4 KB)
baseline_model_comparison.csv (0.2 KB)
candidate_coverage_audit.csv (39.2 KB)
clean_hpo_annotations.csv (35065.7 KB)
direct_vs_hierarchy_scoring_comparison.csv (66.8 KB)
disease_level_split.csv (159.0 KB)
disease_ranking_retrieval_metrics.csv (44.1 KB)
evaluation_eligibility_audit.csv (57.5 KB)
evaluation_protocol_checks.csv (0.2 KB)
final_artifact_inventory.csv (0.3 KB)
final_consolidated_results.csv (1.7 KB)
final_project_handover_inventory.csv (0.9 KB)
gene_informed_phenotype_recovery_metric

In [ ]:

import json
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
DATA = PROJECT / "processed"

files = [
    MODEL_DIR / "filtered_gnn_training_config.json",
    MODEL_DIR / "patient_level_ranker_config.json",
    DATA / "gnn_checkpoint_compatibility.csv",
    DATA / "patient_level_split_summary.json",
    DATA / "patient_level_training_history.csv",
    DATA / "model_selection_decision.json",
]

for path in files:
    print(f"\n{'=' * 65}")
    print(path.name)
    print("=" * 65)

    if not path.exists():
        print("MISSING")
        continue

    if path.suffix == ".json":
        try:
            print(json.dumps(json.loads(path.read_text()), indent=2)[:12000])
        except Exception as e:
            print("Could not parse JSON:", repr(e))
    elif path.suffix == ".csv":
        import pandas as pd
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print(df.head(8).to_string(index=False))
    else:
        print(path.read_text(errors="replace")[:12000])

print("\nRead-only inspection complete. No training started.")



filtered_gnn_training_config.json
{
  "model": "RareDiseaseGCN",
  "num_nodes": 17322,
  "hidden_dim": 64,
  "positive_associations": 158659,
  "directed_edges": 317318,
  "seed": 42,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "max_epochs": 100,
  "early_stopping_patience": 10,
  "graph_file": "rare_disease_gnn_filtered_edge_index.pt",
  "annotation_file": "gnn_annotations_excluding_dev_test_pmids.csv",
  "publication_filter": "Exclude identifiable development/test PMID references; missing or unmatched references remain.",
  "negated_associations": "Excluded from positive graph edges",
  "original_checkpoint_preserved": true,
  "evaluation_note": "Do not claim fully leakage-free evaluation because some annotation references are missing or unmatched."
}

patient_level_ranker_config.json
{
  "model": "PatientDiseaseRanker",
  "dimension": 128,
  "dropout": 0.2,
  "seed": 42,
  "training_cases": 5730,
  "validation_cases": 906,
  "training_objective": "patient-to-disease ranki

In [ ]:

import json
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

config = json.loads(
    (MODEL / "filtered_gnn_training_config.json").read_text()
)

print("Configured annotation file:", config["annotation_file"])
print("Configured graph file:", config["graph_file"])
print("Configured nodes:", config["num_nodes"])
print("Configured positive associations:", config["positive_associations"])

for name in [
    "gnn_annotations_excluding_dev_test_pmids.csv",
    "gnn_annotations_publication_disjoint_candidate.csv",
]:
    path = DATA / name
    df = pd.read_csv(path, usecols=["database_id", "hpo_id"])
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Unique diseases:", df["database_id"].nunique())
    print("Unique phenotypes:", df["hpo_id"].nunique())
    print("Unique disease-phenotype pairs:", len(df.drop_duplicates()))

graph_path = DATA / config["graph_file"]
print("\nConfigured graph exists:", graph_path.exists())
if graph_path.exists():
    print("Graph size (MB):", round(graph_path.stat().st_size / 1024**2, 2))

print("\nNo training started; no files modified.")


Configured annotation file: gnn_annotations_excluding_dev_test_pmids.csv
Configured graph file: rare_disease_gnn_filtered_edge_index.pt
Configured nodes: 17322
Configured positive associations: 158659

gnn_annotations_excluding_dev_test_pmids.csv
Rows: 159025
Unique diseases: 6983
Unique phenotypes: 9617
Unique disease-phenotype pairs: 159025

gnn_annotations_publication_disjoint_candidate.csv
Rows: 158693
Unique diseases: 6980
Unique phenotypes: 9613
Unique disease-phenotype pairs: 158693

Configured graph exists: True
Graph size (MB): 5.18

No training started; no files modified.


In [ ]:

import torch
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
path = DATA / "rare_disease_gnn_filtered_edge_index.pt"

obj = torch.load(path, map_location="cpu", weights_only=True)

print("Object type:", type(obj).__name__)

if isinstance(obj, dict):
    print("Top-level keys:", list(obj.keys()))
    for key, value in obj.items():
        if isinstance(value, torch.Tensor):
            print(f"{key}: tensor shape={tuple(value.shape)}, dtype={value.dtype}")
        elif isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
            print("  sample keys:", list(value.keys())[:5])
        elif isinstance(value, (list, tuple)):
            print(f"{key}: {type(value).__name__}, length={len(value)}")
            print("  sample:", value[:5])
        else:
            print(f"{key}: {type(value).__name__} = {str(value)[:200]}")
elif isinstance(obj, torch.Tensor):
    print("Tensor shape:", tuple(obj.shape), "dtype:", obj.dtype)
else:
    print("Value preview:", str(obj)[:1000])


Object type: dict
Top-level keys: ['edge_index', 'disease_ids', 'phenotype_ids', 'num_nodes', 'hidden_dim', 'positive_association_count', 'excluded_development_and_test_publications', 'note']
edge_index: tensor shape=(2, 317318), dtype=torch.int64
disease_ids: list, length=7340
  sample: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400']
phenotype_ids: list, length=9982
  sample: ['HP:0000002', 'HP:0000003', 'HP:0000006', 'HP:0000007', 'HP:0000008']
num_nodes: int = 17322
hidden_dim: int = 64
positive_association_count: int = 158659
excluded_development_and_test_publications: bool = True
note: str = Identifiable development/test PMID references were filtered; annotations with missing or unmatched PMIDs may remain.


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

path = DATA / "gnn_annotations_publication_disjoint_candidate.csv"
ann = pd.read_csv(path, usecols=["database_id", "hpo_id"], dtype=str)

ann["database_id"] = ann["database_id"].str.strip().str.upper()
ann["hpo_id"] = ann["hpo_id"].str.strip().str.upper()

print("Rows:", len(ann))
print("Missing disease IDs:", int(ann["database_id"].isna().sum()))
print("Missing phenotype IDs:", int(ann["hpo_id"].isna().sum()))
print("Duplicate disease-phenotype pairs:", int(ann.duplicated(
    ["database_id", "hpo_id"]
).sum()))

disease_ids = sorted(ann["database_id"].dropna().unique())
phenotype_ids = sorted(ann["hpo_id"].dropna().unique())

print("Unique diseases:", len(disease_ids))
print("Unique phenotypes:", len(phenotype_ids))
print("Total graph nodes:", len(disease_ids) + len(phenotype_ids))
print("Expected directed edges:", 2 * len(ann))

print("\nReady to construct candidate graph:",
      ann[["database_id", "hpo_id"]].notna().all().all()
      and not ann.duplicated(["database_id", "hpo_id"]).any())


Rows: 158693
Missing disease IDs: 0
Missing phenotype IDs: 0
Duplicate disease-phenotype pairs: 0
Unique diseases: 6980
Unique phenotypes: 9613
Total graph nodes: 16593
Expected directed edges: 317386

Ready to construct candidate graph: True


In [ ]:

import pandas as pd
import torch
import json
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "gnn_annotations_publication_disjoint_candidate.csv",
    usecols=["database_id", "hpo_id"],
    dtype=str
)

ann["database_id"] = ann["database_id"].str.strip().str.upper()
ann["hpo_id"] = ann["hpo_id"].str.strip().str.upper()
ann = ann.dropna(subset=["database_id", "hpo_id"]).drop_duplicates(
    ["database_id", "hpo_id"]
)

disease_ids = sorted(ann["database_id"].unique().tolist())
phenotype_ids = sorted(ann["hpo_id"].unique().tolist())

disease_to_idx = {d: i for i, d in enumerate(disease_ids)}
phenotype_offset = len(disease_ids)
phenotype_to_idx = {
    h: phenotype_offset + i for i, h in enumerate(phenotype_ids)
}

src = ann["database_id"].map(disease_to_idx).to_numpy()
dst = ann["hpo_id"].map(phenotype_to_idx).to_numpy()

forward = torch.tensor([src, dst], dtype=torch.long)
edge_index = torch.cat([forward, forward.flip(0)], dim=1)

artifact = {
    "edge_index": edge_index,
    "disease_ids": disease_ids,
    "phenotype_ids": phenotype_ids,
    "num_nodes": len(disease_ids) + len(phenotype_ids),
    "hidden_dim": 64,
    "positive_association_count": len(ann),
    "excluded_development_and_test_publications": True,
    "note": (
        "Built from gnn_annotations_publication_disjoint_candidate.csv. "
        "References without recognizable matching PMIDs may remain."
    ),
}

out = DATA / "rare_disease_gnn_publication_disjoint_edge_index.pt"
torch.save(artifact, out)

# Reload and verify the saved artifact.
check = torch.load(out, map_location="cpu", weights_only=True)

assert check["edge_index"].shape == (2, 2 * len(ann))
assert check["num_nodes"] == 16593
assert len(check["disease_ids"]) == 6980
assert len(check["phenotype_ids"]) == 9613
assert int(check["edge_index"].max()) < check["num_nodes"]

report = {
    "annotation_file": "gnn_annotations_publication_disjoint_candidate.csv",
    "graph_file": out.name,
    "annotation_pairs": len(ann),
    "directed_edges": check["edge_index"].shape[1],
    "diseases": len(check["disease_ids"]),
    "phenotypes": len(check["phenotype_ids"]),
    "nodes": check["num_nodes"],
}
(DATA / "publication_disjoint_graph_build_report.json").write_text(
    json.dumps(report, indent=2)
)

print("Graph built and reloaded successfully.")
print(json.dumps(report, indent=2))
print("Existing graph/checkpoints were not overwritten.")


/tmp/ipykernel_618/290294268.py:32: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  forward = torch.tensor([src, dst], dtype=torch.long)


Graph built and reloaded successfully.
{
  "annotation_file": "gnn_annotations_publication_disjoint_candidate.csv",
  "graph_file": "rare_disease_gnn_publication_disjoint_edge_index.pt",
  "annotation_pairs": 158693,
  "directed_edges": 317386,
  "diseases": 6980,
  "phenotypes": 9613,
  "nodes": 16593
}
Existing graph/checkpoints were not overwritten.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
search_files = [
    PROJECT / "model" / "rare_disease_ai.py",
    PROJECT / "model" / "rare_disease_predictor.py",
    PROJECT / "model" / "rare_disease_data_loader.py",
]

terms = [
    "class RareDiseaseGCN",
    "class ",
    "def forward",
    "def train",
    "optimizer",
    "torch.save",
    "load_state_dict",
    "edge_index",
    "negative",
    "contrastive",
    "CrossEntropyLoss",
    "BCEWithLogitsLoss",
]

for path in search_files:
    print(f"\n{'=' * 60}\n{path.name}\n{'=' * 60}")
    if not path.exists():
        print("Missing")
        continue

    lines = path.read_text(errors="ignore").splitlines()
    hits = [
        (i + 1, line.rstrip())
        for i, line in enumerate(lines)
        if any(term.lower() in line.lower() for term in terms)
    ]

    if not hits:
        print("No matching architecture/training terms found.")
    else:
        for line_no, line in hits[:35]:
            print(f"{line_no}: {line[:220]}")
        if len(hits) > 35:
            print(f"... {len(hits) - 35} additional matches")

print("\nInspection only. No training started.")



rare_disease_ai.py
No matching architecture/training terms found.

rare_disease_predictor.py
No matching architecture/training terms found.

rare_disease_data_loader.py
No matching architecture/training terms found.

Inspection only. No training started.


In [ ]:

import json
import zipfile
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL = PROJECT / "model"
DATA = PROJECT / "processed"

config_path = MODEL / "filtered_gnn_training_config.json"
graph_path = DATA / "rare_disease_gnn_publication_disjoint_edge_index.pt"
checkpoint_path = MODEL / "rare_disease_gnn_filtered_best.pt"

print("=== Training configuration ===")
print(json.dumps(json.loads(config_path.read_text()), indent=2))

print("\n=== Existing checkpoint archive ===")
with zipfile.ZipFile(checkpoint_path, "r") as z:
    for name in z.namelist():
        if name.endswith("data.pkl"):
            raw = z.read(name)
            print("Metadata bytes:", len(raw))
            print("Metadata preview:", repr(raw[:180]))

print("\n=== New graph artifact ===")
print("Exists:", graph_path.exists())
if graph_path.exists():
    print("Size (MB):", round(graph_path.stat().st_size / 1024**2, 2))

print("\nRead-only inspection complete; no training started.")


=== Training configuration ===
{
  "model": "RareDiseaseGCN",
  "num_nodes": 17322,
  "hidden_dim": 64,
  "positive_associations": 158659,
  "directed_edges": 317318,
  "seed": 42,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "max_epochs": 100,
  "early_stopping_patience": 10,
  "graph_file": "rare_disease_gnn_filtered_edge_index.pt",
  "annotation_file": "gnn_annotations_excluding_dev_test_pmids.csv",
  "publication_filter": "Exclude identifiable development/test PMID references; missing or unmatched references remain.",
  "negated_associations": "Excluded from positive graph edges",
  "original_checkpoint_preserved": true,
  "evaluation_note": "Do not claim fully leakage-free evaluation because some annotation references are missing or unmatched."
}

=== Existing checkpoint archive ===
Metadata bytes: 354762
Metadata preview: b'\x80\x02}q\x00(X\x10\x00\x00\x00model_state_dictq\x01ccollections\nOrderedDict\nq\x02)Rq\x03(X\x14\x00\x00\x00gnn.embedding.weightq\x04ctorch._utils\

In [ ]:

import zipfile
import pickletools
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
checkpoint_path = PROJECT / "model" / "rare_disease_gnn_filtered_best.pt"

with zipfile.ZipFile(checkpoint_path, "r") as z:
    data_name = next(n for n in z.namelist() if n.endswith("data.pkl"))
    raw = z.read(data_name)

print("Checkpoint state-dictionary key names:")
keys = []
for op, arg, pos in pickletools.genops(raw):
    if op.name == "BINUNICODE" and isinstance(arg, str):
        if arg.startswith("gnn.") or arg.startswith("model.") or arg.endswith(".weight") or arg.endswith(".bias"):
            if arg not in keys:
                keys.append(arg)

for key in keys[:100]:
    print(" ", key)

print("\nTotal candidate parameter names:", len(keys))
print("\nNo checkpoint loading, training, or file modification occurred.")


Checkpoint state-dictionary key names:
  gnn.embedding.weight
  gnn.conv1.bias
  gnn.conv1.lin.weight
  gnn.conv2.bias
  gnn.conv2.lin.weight
  scorer.0.weight
  scorer.0.bias
  scorer.2.weight
  scorer.2.bias
  gnn.embedding
  gnn.conv1
  gnn.conv1.aggr_module
  gnn.conv1.lin
  gnn.conv2
  gnn.conv2.aggr_module
  gnn.conv2.lin

Total candidate parameter names: 16

No checkpoint loading, training, or file modification occurred.


In [ ]:

from pathlib import Path
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

terms = [
    "RareDiseaseGCN",
    "conv1",
    "conv2",
    "scorer",
    "embedding",
    "negative sampling",
    "negative",
    "loss",
    "optimizer",
    "training objective",
]

files = [
    MODEL / "filtered_gnn_training_config.json",
    DATA / "gnn_evaluation_results.csv",
    DATA / "gnn_checkpoint_compatibility.csv",
    DATA / "final_project_handover_inventory.csv",
    DATA / "rare_disease_final_research_report.md",
]

for path in files:
    print(f"\n{'=' * 60}\n{path.name}\n{'=' * 60}")
    if not path.exists():
        print("MISSING")
        continue

    text = path.read_text(errors="ignore")
    lines = text.splitlines()
    found = [
        (i + 1, line.strip())
        for i, line in enumerate(lines)
        if any(term.lower() in line.lower() for term in terms)
    ]

    if found:
        for n, line in found[:15]:
            print(f"{n}: {line[:240]}")
    else:
        print("No architecture/objective details found.")

print("\nRead-only inspection complete. No training started.")



filtered_gnn_training_config.json
2: "model": "RareDiseaseGCN",

gnn_evaluation_results.csv
No architecture/objective details found.

gnn_checkpoint_compatibility.csv
No architecture/objective details found.

final_project_handover_inventory.csv
No architecture/objective details found.

rare_disease_final_research_report.md
59: pairs. Unrecorded associations are not necessarily true negatives.
100: - Missing knowledge-graph associations are not necessarily negative examples.
186: | GNN phenotype embeddings | 1827 | 0.0865 | 0.2782 |
188: In this experiment, the phenotype-only baseline achieved the highest Recall@10 and MRR. The gene-informed and GNN phenotype-embedding baselines did not outperform it.
200: | GNN phenotype embeddings | 1827 | 0.0865 | 0.2782 |
212: These link-prediction metrics compare recorded edges with sampled unrecorded pairs. Unrecorded pairs are not necessarily true negatives. They must not be interpreted as clinical diagnostic accuracy.
217: - The original GNN c

In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
extensions = {".py", ".ipynb", ".md", ".txt", ".json"}

needles = [
    "class RareDiseaseGCN",
    "class PatientDiseaseRanker",
    "scorer =",
    "conv1 =",
    "negative_sampling",
    "BCEWithLogitsLoss",
]

skip_dirs = {
    "__pycache__", ".git", ".ipynb_checkpoints",
    "node_modules"
}

matches = 0
for path in PROJECT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue
    if any(part in skip_dirs for part in path.parts):
        continue
    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue

    found = [s for s in needles if s.lower() in text.lower()]
    if found:
        matches += 1
        print(f"\n{path.relative_to(PROJECT)}")
        print("Matches:", ", ".join(found))
        lines = text.splitlines()
        for i, line in enumerate(lines):
            if any(s.lower() in line.lower() for s in found):
                print(f"  {i+1}: {line.strip()[:200]}")

print("\nFiles with matches:", matches)
print("Search only; no training or file changes.")



Files with matches: 0
Search only; no training or file changes.


In [ ]:

import torch
import importlib.util
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
graph_path = (
    PROJECT / "processed" /
    "rare_disease_gnn_publication_disjoint_edge_index.pt"
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

for package in ["torch_geometric", "torch_scatter", "sklearn", "pandas"]:
    print(f"{package} installed:", importlib.util.find_spec(package) is not None)

print("\nPublication-disjoint graph exists:", graph_path.exists())
if graph_path.exists():
    print("Graph size (MB):", round(graph_path.stat().st_size / 1024**2, 2))

print("\nEnvironment check complete; no training started.")



PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0
torch_geometric installed: False
torch_scatter installed: False
sklearn installed: True
pandas installed: True

Publication-disjoint graph exists: True
Graph size (MB): 5.17

Environment check complete; no training started.


In [ ]:

import torch
import json
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

graph_path = DATA / "rare_disease_gnn_publication_disjoint_edge_index.pt"
graph = torch.load(graph_path, map_location="cpu", weights_only=True)

disease_ids = graph["disease_ids"]
phenotype_ids = graph["phenotype_ids"]
edge_index = graph["edge_index"]

# Keep one copy of each disease -> phenotype association.
n_diseases = len(disease_ids)
forward_mask = (
    (edge_index[0] < n_diseases) &
    (edge_index[1] >= n_diseases)
)
positive_edges = edge_index[:, forward_mask].T.contiguous()

assert positive_edges.shape[0] == graph["positive_association_count"]
assert positive_edges.min().item() >= 0
assert positive_edges.max().item() < graph["num_nodes"]

# Reproducible 90/10 edge split.
generator = torch.Generator().manual_seed(42)
perm = torch.randperm(len(positive_edges), generator=generator)
n_val = max(1, int(0.10 * len(positive_edges)))

val_edges = positive_edges[perm[:n_val]]
train_edges = positive_edges[perm[n_val:]]

# Build training graph with both directions, using training positives only.
train_edge_index = torch.cat(
    [train_edges.T, train_edges[:, [1, 0]].T], dim=1
).contiguous()

out = DATA / "rare_disease_gnn_publication_disjoint_edge_split.pt"
torch.save({
    "train_edge_index": train_edge_index,
    "train_positive_edges": train_edges,
    "validation_positive_edges": val_edges,
    "disease_ids": disease_ids,
    "phenotype_ids": phenotype_ids,
    "num_nodes": graph["num_nodes"],
    "hidden_dim": graph["hidden_dim"],
    "seed": 42,
    "split_unit": "disease-phenotype edges",
    "validation_fraction": 0.10,
}, out)

print("Saved:", out.name)
print("Training positive associations:", len(train_edges))
print("Validation positive associations:", len(val_edges))
print("Training directed graph edges:", train_edge_index.shape[1])
print("Total nodes:", graph["num_nodes"])
print("Existing graph/checkpoints unchanged.")
print("NOTE: This is a random edge split, not a disease-level zero-shot split.")


Saved: rare_disease_gnn_publication_disjoint_edge_split.pt
Training positive associations: 142824
Validation positive associations: 15869
Training directed graph edges: 285648
Total nodes: 16593
Existing graph/checkpoints unchanged.
NOTE: This is a random edge split, not a disease-level zero-shot split.


In [ ]:

import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
split_path = PROJECT / "processed" / "rare_disease_gnn_publication_disjoint_edge_split.pt"

data = torch.load(split_path, map_location="cpu", weights_only=True)

train = data["train_positive_edges"].long()
val = data["validation_positive_edges"].long()
train_graph = data["train_edge_index"].long()
n_diseases = len(data["disease_ids"])
n_nodes = data["num_nodes"]

# Canonical disease-phenotype pairs: disease node first, phenotype node second.
train_keys = set((train[:, 0] * n_nodes + train[:, 1]).tolist())
val_keys = set((val[:, 0] * n_nodes + val[:, 1]).tolist())

assert len(train_keys) == len(train), "Duplicate training associations found."
assert len(val_keys) == len(val), "Duplicate validation associations found."
assert train_keys.isdisjoint(val_keys), "Train/validation association overlap!"
assert torch.all(train[:, 0] < n_diseases)
assert torch.all(train[:, 1] >= n_diseases)
assert torch.all(val[:, 0] < n_diseases)
assert torch.all(val[:, 1] >= n_diseases)

# Check no validation positive occurs in either direction in the message-passing graph.
graph_keys = set(
    (train_graph[0] * n_nodes + train_graph[1]).tolist()
)
val_reverse_keys = set((val[:, 1] * n_nodes + val[:, 0]).tolist())
assert val_keys.isdisjoint(graph_keys), "Validation edges leaked into training graph."
assert val_reverse_keys.isdisjoint(graph_keys), "Reverse validation edges leaked into training graph."

print("PASS: no train/validation positive overlap")
print("PASS: validation positives absent in both directions from training graph")
print("Training associations:", len(train))
print("Validation associations:", len(val))
print("Training graph directed edges:", train_graph.shape[1])
print("Disease nodes:", n_diseases)
print("Phenotype nodes:", len(data["phenotype_ids"]))
print("GPU available:", torch.cuda.is_available())
print("Ready to define the GNN training objective; no training started.")


PASS: no train/validation positive overlap
PASS: validation positives absent in both directions from training graph
Training associations: 142824
Validation associations: 15869
Training graph directed edges: 285648
Disease nodes: 6980
Phenotype nodes: 9613
GPU available: True
Ready to define the GNN training objective; no training started.


In [ ]:

import copy
import json
import random
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

split_path = DATA / "rare_disease_gnn_publication_disjoint_edge_split.pt"
d = torch.load(split_path, map_location="cpu", weights_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_diseases = len(d["disease_ids"])
n_nodes = d["num_nodes"]
hidden_dim = 64

train_pos = d["train_positive_edges"].long().to(device)
val_pos = d["validation_positive_edges"].long().to(device)
edge_index = d["train_edge_index"].long().to(device)

# Symmetric normalized adjacency with self-loops.
src, dst = edge_index
loops = torch.arange(n_nodes, device=device)
src = torch.cat([src, loops])
dst = torch.cat([dst, loops])
indices = torch.stack([src, dst])

adj = torch.sparse_coo_tensor(
    indices,
    torch.ones(indices.shape[1], device=device),
    (n_nodes, n_nodes)
).coalesce()

row, col = adj.indices()
deg = torch.bincount(row, minlength=n_nodes).float().clamp_min(1)
norm = deg[row].rsqrt() * deg[col].rsqrt()
adj = torch.sparse_coo_tensor(
    adj.indices(), norm, adj.shape, device=device
).coalesce()

class ExperimentalGNN(nn.Module):
    def __init__(self, n_nodes, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(n_nodes, hidden_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        self.conv1 = nn.Linear(hidden_dim, hidden_dim)
        self.conv2 = nn.Linear(hidden_dim, hidden_dim)
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def encode(self, adj):
        x = self.embedding.weight
        x = F.relu(self.conv1(torch.sparse.mm(adj, x)))
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(torch.sparse.mm(adj, x))
        return F.normalize(x, p=2, dim=1)

    def score(self, z, pairs):
        disease = z[pairs[:, 0]]
        phenotype = z[pairs[:, 1]]
        return self.scorer(torch.cat([disease, phenotype], dim=1)).squeeze(1)

model = ExperimentalGNN(n_nodes, hidden_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Disease-phenotype negative sampler.
# These are unobserved pairs, not verified biological negatives.
def sample_negatives(n, positives, n_diseases, n_nodes, device):
    phen_start = n_diseases
    phen_count = n_nodes - n_diseases
    positive_keys = set(
        (positives[:, 0] * n_nodes + positives[:, 1]).detach().cpu().tolist()
    )
    result = []
    attempts = 0
    while len(result) < n:
        batch_n = max(1024, 2 * (n - len(result)))
        diseases = torch.randint(n_diseases, (batch_n,))
        phenotypes = torch.randint(phen_count, (batch_n,)) + phen_start
        for a, b in zip(diseases.tolist(), phenotypes.tolist()):
            if a * n_nodes + b not in positive_keys:
                result.append((a, b))
                if len(result) == n:
                    break
        attempts += batch_n
        if attempts > n * 100:
            raise RuntimeError("Negative sampling stalled; inspect the graph.")
    return torch.tensor(result, dtype=torch.long, device=device)

@torch.no_grad()
def evaluate_mrr(model, adj, positives, n_diseases, n_nodes, n_neg=50):
    model.eval()
    z = model.encode(adj)
    # Sample candidate unobserved pairs for each validation positive.
    neg = sample_negatives(
        len(positives) * n_neg, positives, n_diseases, n_nodes, device
    )
    pos_scores = model.score(z, positives)
    neg_scores = model.score(z, neg).view(len(positives), n_neg)
    ranks = 1 + (neg_scores >= pos_scores[:, None]).sum(dim=1)
    return ranks.float().reciprocal().mean().item()

best_mrr = -1.0
best_state = None
patience = 8
stale = 0
max_epochs = 50
history = []
start = time.time()

for epoch in range(1, max_epochs + 1):
    model.train()
    optimizer.zero_grad()

    neg = sample_negatives(
        len(train_pos), train_pos, n_diseases, n_nodes, device
    )
    z = model.encode(adj)
    pos_scores = model.score(z, train_pos)
    neg_scores = model.score(z, neg)

    loss = (
        F.binary_cross_entropy_with_logits(
            pos_scores, torch.ones_like(pos_scores)
        )
        + F.binary_cross_entropy_with_logits(
            neg_scores, torch.zeros_like(neg_scores)
        )
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    val_mrr = evaluate_mrr(model, adj, val_pos, n_diseases, n_nodes)
    history.append({
        "epoch": epoch,
        "train_loss": float(loss.item()),
        "validation_sampled_mrr": val_mrr,
    })
    print(
        f"Epoch {epoch:02d}/{max_epochs} | "
        f"loss={loss.item():.4f} | validation sampled MRR={val_mrr:.4f}"
    )

    if val_mrr > best_mrr:
        best_mrr = val_mrr
        best_state = copy.deepcopy({
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        })
        stale = 0
    else:
        stale += 1

    if stale >= patience:
        print("Early stopping: validation MRR did not improve.")
        break

checkpoint_path = MODEL_DIR / "rare_disease_gnn_publication_disjoint_experiment.pt"
history_path = DATA / "rare_disease_gnn_publication_disjoint_experiment_history.csv"
config_path = MODEL_DIR / "rare_disease_gnn_publication_disjoint_experiment_config.json"

torch.save({
    "model_state_dict": best_state,
    "disease_ids": d["disease_ids"],
    "phenotype_ids": d["phenotype_ids"],
    "num_nodes": n_nodes,
    "n_diseases": n_diseases,
    "hidden_dim": hidden_dim,
    "best_validation_sampled_mrr": best_mrr,
    "seed": SEED,
    "architecture": "two-layer normalized sparse graph convolution with MLP edge scorer",
    "note": "Experimental edge-link prediction. Sampled negatives are unobserved, not confirmed false associations. Not clinical validation."
}, checkpoint_path)

pd.DataFrame(history).to_csv(history_path, index=False)
with open(config_path, "w") as f:
    json.dump({
        "seed": SEED,
        "epochs_run": len(history),
        "best_validation_sampled_mrr": best_mrr,
        "max_epochs": max_epochs,
        "early_stopping_patience": patience,
        "validation_negative_candidates_per_positive": 50,
        "training_graph": split_path.name,
        "checkpoint": checkpoint_path.name,
        "training_seconds": round(time.time() - start, 1),
        "device": str(device),
    }, f, indent=2)

print("\nGNN experiment finished.")
print("Best validation sampled MRR:", round(best_mrr, 5))
print("Checkpoint:", checkpoint_path)
print("History:", history_path)
print("Config:", config_path)
print("Original checkpoints were not modified.")


/tmp/ipykernel_618/1886816836.py:43: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj = torch.sparse_coo_tensor(


Epoch 01/50 | loss=1.3877 | validation sampled MRR=0.4850
Epoch 02/50 | loss=1.3799 | validation sampled MRR=0.4919
Epoch 03/50 | loss=1.3730 | validation sampled MRR=0.4917
Epoch 04/50 | loss=1.3659 | validation sampled MRR=0.4916
Epoch 05/50 | loss=1.3585 | validation sampled MRR=0.4912
Epoch 06/50 | loss=1.3513 | validation sampled MRR=0.4933
Epoch 07/50 | loss=1.3442 | validation sampled MRR=0.4912
Epoch 08/50 | loss=1.3368 | validation sampled MRR=0.4919
Epoch 09/50 | loss=1.3289 | validation sampled MRR=0.4960
Epoch 10/50 | loss=1.3207 | validation sampled MRR=0.4956
Epoch 11/50 | loss=1.3121 | validation sampled MRR=0.4926
Epoch 12/50 | loss=1.3027 | validation sampled MRR=0.4891
Epoch 13/50 | loss=1.2925 | validation sampled MRR=0.4877
Epoch 14/50 | loss=1.2816 | validation sampled MRR=0.4840
Epoch 15/50 | loss=1.2702 | validation sampled MRR=0.4852
Epoch 16/50 | loss=1.2579 | validation sampled MRR=0.4818
Epoch 17/50 | loss=1.2448 | validation sampled MRR=0.4788
Early stopping

In [ ]:

import json
import torch
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
DATA = PROJECT / "processed"

checkpoint_path = MODEL_DIR / "rare_disease_gnn_publication_disjoint_experiment.pt"
history_path = DATA / "rare_disease_gnn_publication_disjoint_experiment_history.csv"
config_path = MODEL_DIR / "rare_disease_gnn_publication_disjoint_experiment_config.json"

for p in [checkpoint_path, history_path, config_path]:
    print(f"{p.name}: exists={p.exists()}", end="")
    if p.exists():
        print(f", size={p.stat().st_size / 1024**2:.2f} MB")
    else:
        print()
        raise FileNotFoundError(p)

checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=True
)
history = pd.read_csv(history_path)
with open(config_path) as f:
    config = json.load(f)

print("\nCheckpoint metadata")
for key in [
    "n_diseases", "num_nodes", "hidden_dim",
    "best_validation_sampled_mrr", "seed", "architecture"
]:
    print(f"{key}: {checkpoint.get(key)}")

print("\nState dictionary tensors:", len(checkpoint["model_state_dict"]))
print("History rows:", len(history))
print("Best MRR in history:", round(history["validation_sampled_mrr"].max(), 5))
print("Config epochs run:", config["epochs_run"])
print("Config device:", config["device"])

assert len(history) == config["epochs_run"]
assert abs(
    history["validation_sampled_mrr"].max()
    - checkpoint["best_validation_sampled_mrr"]
) < 1e-6
assert len(checkpoint["disease_ids"]) == checkpoint["n_diseases"]

print("\nPASS: saved GNN artifacts are readable and internally consistent.")
print("No training started.")



rare_disease_gnn_publication_disjoint_experiment.pt: exists=True, size=4.44 MB
rare_disease_gnn_publication_disjoint_experiment_history.csv: exists=True, size=0.00 MB
rare_disease_gnn_publication_disjoint_experiment_config.json: exists=True, size=0.00 MB

Checkpoint metadata
n_diseases: 6980
num_nodes: 16593
hidden_dim: 64
best_validation_sampled_mrr: 0.49595126509666443
seed: 42
architecture: two-layer normalized sparse graph convolution with MLP edge scorer

State dictionary tensors: 9
History rows: 17
Best MRR in history: 0.49595
Config epochs run: 17
Config device: cuda

PASS: saved GNN artifacts are readable and internally consistent.
No training started.


In [ ]:

import json
import pandas as pd
import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=== Patient-level data files ===")
for p in sorted(DATA.glob("*patient*")):
    if p.is_file():
        print(f"{p.name} | {p.stat().st_size / 1024**2:.2f} MB")

print("\n=== Candidate CSV schemas ===")
for p in sorted(DATA.glob("*.csv")):
    if any(s in p.name.lower() for s in ["patient_level", "patient_publication_split"]):
        try:
            df = pd.read_csv(p, nrows=5)
            print(f"\n{p.name}")
            print("Columns:", list(df.columns))
            print("Sample rows:", len(df))
        except Exception as e:
            print(f"{p.name}: could not inspect ({e})")

checkpoint_path = MODEL_DIR / "patient_level_ranker_best.pt"
config_path = MODEL_DIR / "patient_level_ranker_config.json"
decision_path = DATA / "model_selection_decision.json"

print("\n=== Existing ranker checkpoint ===")
assert checkpoint_path.exists(), f"Missing checkpoint: {checkpoint_path}"
ranker = torch.load(
    checkpoint_path, map_location="cpu", weights_only=True
)
for key in [
    "n_diseases", "n_phenotypes", "hidden_dim",
    "best_validation_mrr", "seed", "training_cases",
    "validation_cases", "note"
]:
    print(f"{key}: {ranker.get(key)}")

if config_path.exists():
    print("\nRanker config:")
    print(json.dumps(json.load(open(config_path)), indent=2))

if decision_path.exists():
    print("\nSaved model-selection decision:")
    print(json.dumps(json.load(open(decision_path)), indent=2))

print("\nInspection complete. No training or file changes made.")


=== Patient-level data files ===
patient_baseline_vs_gnn_paired.csv | 0.25 MB
patient_filtered_gnn_linkhead_rankings.csv | 0.17 MB
patient_filtered_gnn_test_rankings.csv | 0.19 MB
patient_gnn_training_disease_rankings.csv | 0.19 MB
patient_level_development_dataset.csv | 0.93 MB
patient_level_label_audit.json | 0.00 MB
patient_level_publication_leakage_audit.json | 0.00 MB
patient_level_split_summary.json | 0.00 MB
patient_level_train.csv | 0.80 MB
patient_level_training_history.csv | 0.00 MB
patient_level_validation.csv | 0.13 MB
patient_phenotype_baseline_metrics_audit.csv | 0.00 MB
patient_phenotype_only_baseline.csv | 0.12 MB
patient_phenotype_only_validation.csv | 0.07 MB
patient_publication_overlap_audit.csv | 0.00 MB
patient_publication_split.csv | 0.59 MB
patient_ranker_bias_ablation.csv | 0.08 MB
patient_ranker_bias_ablation_summary.json | 0.00 MB
patient_ranker_diagnostics.json | 0.00 MB
patient_ranker_vs_baseline_paired_bootstrap.json | 0.00 MB
patient_test_disease_coverage.

In [ ]:

import ast
import json
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

train_path = DATA / "patient_level_train.csv"
val_path = DATA / "patient_level_validation.csv"
gnn_path = MODEL_DIR / "rare_disease_gnn_publication_disjoint_experiment.pt"

train = pd.read_csv(train_path)
val = pd.read_csv(val_path)

def parse_ids(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, SyntaxError):
        pass
    # Fallback for simple comma/semicolon-delimited strings.
    separator = ";" if ";" in text else ","
    return [x.strip().strip("'\"[] ") for x in text.split(separator)
            if x.strip().strip("'\"[] ")]

train_pmids = set(train["pmid"].dropna().astype(str))
val_pmids = set(val["pmid"].dropna().astype(str))

print("=== Split integrity ===")
print("Training cases:", len(train))
print("Validation cases:", len(val))
print("Training unique PMIDs:", len(train_pmids))
print("Validation unique PMIDs:", len(val_pmids))
print("Shared PMIDs:", len(train_pmids & val_pmids))
print("Duplicate training case IDs:", int(train["case_id"].duplicated().sum()))
print("Duplicate validation case IDs:", int(val["case_id"].duplicated().sum()))

assert len(train_pmids & val_pmids) == 0, "PMID leakage between train and validation!"

train_dx = set(x for v in train["diagnosis_ids"] for x in parse_ids(v))
val_dx = set(x for v in val["diagnosis_ids"] for x in parse_ids(v))
train_hpo = set(x for v in train["hpo_ids"] for x in parse_ids(v))
val_hpo = set(x for v in val["hpo_ids"] for x in parse_ids(v))

print("\n=== Parsed dataset coverage ===")
print("Training unique diagnosis IDs:", len(train_dx))
print("Validation unique diagnosis IDs:", len(val_dx))
print("Training unique HPO IDs:", len(train_hpo))
print("Validation unique HPO IDs:", len(val_hpo))
print("Validation cases with no diagnosis labels:",
      int(train.shape[0] * 0 + val["diagnosis_ids"].apply(
          lambda x: len(parse_ids(x)) == 0).sum()))
print("Validation cases with no HPO terms:",
      int(val["hpo_ids"].apply(lambda x: len(parse_ids(x)) == 0).sum()))

gnn = __import__("torch").load(
    gnn_path, map_location="cpu", weights_only=True
)
gnn_dx = set(map(str, gnn["disease_ids"]))
gnn_hpo = set(map(str, gnn["phenotype_ids"]))

print("\n=== New GNN vocabulary coverage ===")
print("Train diagnoses covered:", len(train_dx & gnn_dx), "/", len(train_dx))
print("Validation diagnoses covered:", len(val_dx & gnn_dx), "/", len(val_dx))
print("Train HPO terms covered:", len(train_hpo & gnn_hpo), "/", len(train_hpo))
print("Validation HPO terms covered:", len(val_hpo & gnn_hpo), "/", len(val_hpo))

print("\nAudit complete. No training or file changes made.")


=== Split integrity ===
Training cases: 5730
Validation cases: 906
Training unique PMIDs: 944
Validation unique PMIDs: 166
Shared PMIDs: 0
Duplicate training case IDs: 0
Duplicate validation case IDs: 0

=== Parsed dataset coverage ===
Training unique diagnosis IDs: 499
Validation unique diagnosis IDs: 125
Training unique HPO IDs: 3373
Validation unique HPO IDs: 1366
Validation cases with no diagnosis labels: 0
Validation cases with no HPO terms: 0

=== New GNN vocabulary coverage ===
Train diagnoses covered: 209 / 499
Validation diagnoses covered: 56 / 125
Train HPO terms covered: 3126 / 3373
Validation HPO terms covered: 1327 / 1366

Audit complete. No training or file changes made.


In [ ]:

import ast
import torch
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

def parse_ids(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, (list, tuple, set)):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, SyntaxError):
        pass
    text = str(value).strip().strip("[]")
    if not text:
        return []
    sep = ";" if ";" in text else ","
    return [x.strip().strip("'\" ") for x in text.split(sep) if x.strip()]

train = pd.read_csv(DATA / "patient_level_train.csv")
val = pd.read_csv(DATA / "patient_level_validation.csv")

ckpt_path = MODEL_DIR / "patient_level_ranker_best.pt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)

ranker_diseases = set(map(str, ckpt["disease_ids"]))
ranker_phenotypes = set(map(str, ckpt["phenotype_ids"]))

train_dx = [set(parse_ids(x)) for x in train["diagnosis_ids"]]
val_dx = [set(parse_ids(x)) for x in val["diagnosis_ids"]]
train_hpo = [set(parse_ids(x)) for x in train["hpo_ids"]]
val_hpo = [set(parse_ids(x)) for x in val["hpo_ids"]]

def coverage(rows, vocab):
    total = sum(len(r) for r in rows)
    mapped = sum(len(r & vocab) for r in rows)
    cases_with_mapped_label = sum(bool(r & vocab) for r in rows)
    return {
        "total_label_occurrences": total,
        "mapped_label_occurrences": mapped,
        "label_coverage": round(mapped / total, 4) if total else None,
        "cases_with_any_mapped_label": cases_with_mapped_label,
        "total_cases": len(rows),
    }

print("Existing ranker vocabulary sizes:")
print("Diseases:", len(ranker_diseases))
print("Phenotypes:", len(ranker_phenotypes))

print("\nTraining diagnosis coverage:")
print(coverage(train_dx, ranker_diseases))
print("\nValidation diagnosis coverage:")
print(coverage(val_dx, ranker_diseases))
print("\nTraining phenotype coverage:")
print(coverage(train_hpo, ranker_phenotypes))
print("\nValidation phenotype coverage:")
print(coverage(val_hpo, ranker_phenotypes))

print("\nSample raw ID representations:")
print("diagnosis_ids:", repr(train["diagnosis_ids"].iloc[0]))
print("hpo_ids:", repr(train["hpo_ids"].iloc[0]))

print("\nAudit only; no training or files changed.")


Existing ranker vocabulary sizes:
Diseases: 7340
Phenotypes: 9982

Training diagnosis coverage:
{'total_label_occurrences': 5730, 'mapped_label_occurrences': 5730, 'label_coverage': 1.0, 'cases_with_any_mapped_label': 5730, 'total_cases': 5730}

Validation diagnosis coverage:
{'total_label_occurrences': 906, 'mapped_label_occurrences': 906, 'label_coverage': 1.0, 'cases_with_any_mapped_label': 906, 'total_cases': 906}

Training phenotype coverage:
{'total_label_occurrences': 49192, 'mapped_label_occurrences': 49192, 'label_coverage': 1.0, 'cases_with_any_mapped_label': 5730, 'total_cases': 5730}

Validation phenotype coverage:
{'total_label_occurrences': 7851, 'mapped_label_occurrences': 7851, 'label_coverage': 1.0, 'cases_with_any_mapped_label': 906, 'total_cases': 906}

Sample raw ID representations:
diagnosis_ids: 'OMIM:277600'
hpo_ids: 'HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003'

Audit only; no training or files changed.


In [ ]:

import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
checkpoint_path = PROJECT / "model" / "patient_level_ranker_best.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint metadata:")
for key in [
    "n_diseases", "n_phenotypes", "hidden_dim",
    "best_validation_mrr", "seed",
    "training_cases", "validation_cases"
]:
    print(f"  {key}: {checkpoint.get(key)}")

state = checkpoint["model_state_dict"]
print("\nModel parameter tensors:")
for name, tensor in state.items():
    print(f"  {name:45s} {tuple(tensor.shape)}")

print("\nTotal parameters:",
      sum(t.numel() for t in state.values()))

print("\nOriginal checkpoint preserved:", checkpoint_path.exists())
print("Inspection only; no training or file changes made.")


Checkpoint metadata:
  n_diseases: 7340
  n_phenotypes: 9982
  hidden_dim: 128
  best_validation_mrr: 0.052704318912176605
  seed: 42
  training_cases: 5730
  validation_cases: 906

Model parameter tensors:
  hpo_embedding.weight                          (9982, 128)
  disease_embedding.weight                      (7340, 128)
  patient_projection.0.weight                   (128, 128)
  patient_projection.0.bias                     (128,)
  patient_projection.3.weight                   (128, 128)
  patient_projection.3.bias                     (128,)
  disease_bias.weight                           (7340, 1)

Total parameters: 2257580

Original checkpoint preserved: True
Inspection only; no training or file changes made.


In [ ]:

import json
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

baseline_path = DATA / "patient_phenotype_only_validation.csv"
decision_path = DATA / "model_selection_decision.json"
train_path = DATA / "patient_level_train.csv"
val_path = DATA / "patient_level_validation.csv"

for path in [baseline_path, decision_path, train_path, val_path]:
    assert path.exists(), f"Missing required file: {path}"

baseline = pd.read_csv(baseline_path)
train = pd.read_csv(train_path)
val = pd.read_csv(val_path)
decision = json.loads(decision_path.read_text())

print("=== Saved model-selection decision ===")
print("Selected method:", decision["selected_method"])
print("Decision:", decision["decision"])
print("Baseline validation metrics:", decision["selected_validation_metrics"])
print("Best ranker weight in combination experiment:",
      decision["score_combination_experiment"]["best_ranker_weight"])

print("\n=== Baseline validation file ===")
print("Rows:", len(baseline))
print("Columns:", list(baseline.columns))
print(baseline.head(3).to_string(index=False))

print("\n=== Ranker input datasets ===")
print("Training rows:", len(train))
print("Validation rows:", len(val))
print("Training columns:", list(train.columns))
print("Validation columns:", list(val.columns))

print("\nInspection complete; no files changed and no training started.")



=== Saved model-selection decision ===
Selected method: information_content_weighted_phenotype_F1
Decision: retain_phenotype_only_baseline
Baseline validation metrics: {'cases': 906, 'top1': 0.2947, 'top5': 0.468, 'top10': 0.5419, 'mrr': 0.3773}
Best ranker weight in combination experiment: 0.0

=== Baseline validation file ===
Rows: 906
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_recognized_hpo', 'target_in_candidate_set', 'rank', 'top1', 'top5', 'top10', 'mrr']
                                       case_id     pmid diagnosis_ids  n_recognized_hpo  target_in_candidate_set  rank  top1  top5  top10      mrr
                       PMID_28830446_Patient_2 28830446   OMIM:615688                14                     True    19     0     0      0 0.052632
                       PMID_28830446_Patient_1 28830446   OMIM:615688                17                     True    19     0     0      0 0.052632
PMID_32480058_Case_F5P1_original_pedigree_II_2 32480058   OMIM:618052                 

In [ ]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

val = pd.read_csv(DATA / "patient_level_validation.csv", dtype={"case_id": str})
baseline = pd.read_csv(
    DATA / "patient_phenotype_only_validation.csv",
    dtype={"case_id": str}
)

assert val["case_id"].is_unique, "Duplicate validation case IDs."
assert baseline["case_id"].is_unique, "Duplicate baseline case IDs."

val_ids = set(val["case_id"])
baseline_ids = set(baseline["case_id"])

print("Validation cases:", len(val_ids))
print("Baseline result cases:", len(baseline_ids))
print("Cases shared:", len(val_ids & baseline_ids))
print("Validation cases missing from baseline:", len(val_ids - baseline_ids))
print("Baseline cases missing from validation:", len(baseline_ids - val_ids))

if "diagnosis_ids" in baseline.columns:
    joined = val[["case_id", "diagnosis_ids"]].merge(
        baseline[["case_id", "diagnosis_ids"]],
        on="case_id",
        how="inner",
        suffixes=("_input", "_baseline")
    )
    mismatch = (
        joined["diagnosis_ids_input"].fillna("").astype(str)
        != joined["diagnosis_ids_baseline"].fillna("").astype(str)
    )
    print("Diagnosis-label mismatches:", int(mismatch.sum()))

assert val_ids == baseline_ids, "Baseline and validation case sets differ."
print("\nPASS: baseline results cover the same validation cases.")
print("No training started; no files changed.")


Validation cases: 906
Baseline result cases: 906
Cases shared: 906
Validation cases missing from baseline: 0
Baseline cases missing from validation: 0
Diagnosis-label mismatches: 0

PASS: baseline results cover the same validation cases.
No training started; no files changed.


In [ ]:

import ast
import copy
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA / "patient_level_train.csv")
val_df = pd.read_csv(DATA / "patient_level_validation.csv")
old_ckpt_path = MODEL_DIR / "patient_level_ranker_best.pt"
baseline_path = DATA / "patient_phenotype_only_validation.csv"

old_ckpt = torch.load(
    old_ckpt_path, map_location="cpu", weights_only=True
)
disease_ids = list(map(str, old_ckpt["disease_ids"]))
phenotype_ids = list(map(str, old_ckpt["phenotype_ids"]))
disease_to_idx = {x: i for i, x in enumerate(disease_ids)}
phenotype_to_idx = {x: i for i, x in enumerate(phenotype_ids)}

def parse_ids(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, SyntaxError):
        pass
    sep = ";" if ";" in text else ","
    return [
        x.strip().strip("'\"[] ")
        for x in text.split(sep)
        if x.strip().strip("'\"[] ")
    ]

def encode_cases(df):
    cases = []
    for _, row in df.iterrows():
        hpos = sorted({
            phenotype_to_idx[x]
            for x in parse_ids(row["hpo_ids"])
            if x in phenotype_to_idx
        })
        diagnoses = sorted({
            disease_to_idx[x]
            for x in parse_ids(row["diagnosis_ids"])
            if x in disease_to_idx
        })
        if hpos and diagnoses:
            cases.append({
                "case_id": str(row["case_id"]),
                "pmid": str(row["pmid"]),
                "hpos": hpos,
                "diagnoses": diagnoses,
            })
    return cases

train_cases = encode_cases(train_df)
val_cases = encode_cases(val_df)

assert len(train_cases) == len(train_df), "Some training cases failed encoding."
assert len(val_cases) == len(val_df), "Some validation cases failed encoding."
assert not (
    set(x["pmid"] for x in train_cases)
    & set(x["pmid"] for x in val_cases)
), "PMID overlap between train and validation!"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_diseases = len(disease_ids)
n_phenotypes = len(phenotype_ids)
dim = 128

class BiasFreePatientRanker(nn.Module):
    def __init__(self):
        super().__init__()
        self.hpo_embedding = nn.Embedding(n_phenotypes, dim)
        self.disease_embedding = nn.Embedding(n_diseases, dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(dim, dim),
        )
        # Deliberately omit the disease-bias term.
        nn.init.normal_(self.hpo_embedding.weight, std=0.05)
        nn.init.normal_(self.disease_embedding.weight, std=0.05)

    def patient_vector(self, hpos):
        x = self.hpo_embedding(torch.tensor(hpos, device=device))
        return self.patient_projection(x.mean(dim=0))

    def scores(self, hpos):
        patient = self.patient_vector(hpos)
        return self.disease_embedding.weight @ patient

model = BiasFreePatientRanker().to(device)
optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)

# Sampled ranking loss: recorded diagnoses are positives;
# sampled unrecorded diseases are competitors, not confirmed negatives.
def training_loss(case):
    patient = model.patient_vector(case["hpos"])
    pos_ids = torch.tensor(case["diagnoses"], device=device)
    pos_vec = model.disease_embedding(pos_ids)
    pos_scores = pos_vec @ patient

    positive_set = set(case["diagnoses"])
    candidates = []
    while len(candidates) < 64:
        batch = torch.randint(
            0, n_diseases, (128,), device=device
        ).tolist()
        for idx in batch:
            if idx not in positive_set:
                candidates.append(idx)
                if len(candidates) == 64:
                    break
    neg_ids = torch.tensor(candidates, device=device)
    neg_scores = model.disease_embedding(neg_ids) @ patient

    # Smooth pairwise logistic ranking objective.
    return F.softplus(neg_scores[None, :] - pos_scores[:, None]).mean()

@torch.no_grad()
def evaluate(cases, top_k=10):
    model.eval()
    reciprocal_ranks, top1, top5, top10 = [], [], [], []
    for case in cases:
        scores = model.scores(case["hpos"])
        target = torch.tensor(case["diagnoses"], device=device)
        target_score = scores[target].max()
        rank = 1 + int((scores > target_score).sum().item())
        reciprocal_ranks.append(1.0 / rank)
        top1.append(float(rank <= 1))
        top5.append(float(rank <= 5))
        top10.append(float(rank <= 10))
    return {
        "cases": len(cases),
        "mrr": float(np.mean(reciprocal_ranks)),
        "top1": float(np.mean(top1)),
        "top5": float(np.mean(top5)),
        "top10": float(np.mean(top10)),
    }

# Keep runtime reasonable on a Colab T4.
MAX_EPOCHS = 40
PATIENCE = 6
best_mrr = -1.0
best_state = None
stale = 0
history = []
start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    order = torch.randperm(len(train_cases)).tolist()
    losses = []

    for i in order:
        optimizer.zero_grad(set_to_none=True)
        loss = training_loss(train_cases[i])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(float(loss.item()))

    metrics = evaluate(val_cases)
    history.append({
        "epoch": epoch,
        "train_loss": float(np.mean(losses)),
        **{f"validation_{k}": v for k, v in metrics.items() if k != "cases"},
    })
    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"loss={np.mean(losses):.4f} | "
        f"MRR={metrics['mrr']:.4f} | "
        f"Top1={metrics['top1']:.4f} | "
        f"Top5={metrics['top5']:.4f} | "
        f"Top10={metrics['top10']:.4f}"
    )

    if metrics["mrr"] > best_mrr:
        best_mrr = metrics["mrr"]
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        stale = 0
    else:
        stale += 1

    if stale >= PATIENCE:
        print("Early stopping.")
        break

# Restore best validation checkpoint and calculate final metrics.
model.load_state_dict(best_state)
best_metrics = evaluate(val_cases)

checkpoint_out = MODEL_DIR / "patient_level_ranker_bias_free_experiment.pt"
history_out = DATA / "patient_level_ranker_bias_free_experiment_history.csv"
metrics_out = DATA / "patient_level_ranker_bias_free_experiment_metrics.json"
config_out = MODEL_DIR / "patient_level_ranker_bias_free_experiment_config.json"

torch.save({
    "model_state_dict": best_state,
    "n_diseases": n_diseases,
    "n_phenotypes": n_phenotypes,
    "hidden_dim": dim,
    "disease_ids": disease_ids,
    "phenotype_ids": phenotype_ids,
    "best_validation_mrr": best_metrics["mrr"],
    "validation_metrics": best_metrics,
    "seed": SEED,
    "training_cases": len(train_cases),
    "validation_cases": len(val_cases),
    "note": (
        "Experimental bias-free patient ranker. Validation split by PMID. "
        "Sampled unrecorded diseases are not guaranteed negatives. "
        "Not clinically validated."
    ),
}, checkpoint_out)

pd.DataFrame(history).to_csv(history_out, index=False)
metrics_out.write_text(json.dumps(best_metrics, indent=2))
config_out.write_text(json.dumps({
    "model": "BiasFreePatientRanker",
    "seed": SEED,
    "epochs_run": len(history),
    "best_validation_mrr": best_metrics["mrr"],
    "device": str(device),
    "training_seconds": round(time.time() - start, 1),
    "training_file": "patient_level_train.csv",
    "validation_file": "patient_level_validation.csv",
    "original_ranker_modified": False,
}, indent=2))

baseline = pd.read_csv(baseline_path)
baseline_metrics = {
    "mrr": float(baseline["mrr"].mean()),
    "top1": float(baseline["top1"].mean()),
    "top5": float(baseline["top5"].mean()),
    "top10": float(baseline["top10"].mean()),
}

print("\n=== Validation comparison ===")
print("Phenotype-only baseline:", {
    k: round(v, 4) for k, v in baseline_metrics.items()
})
print("New bias-free ranker:", {
    k: round(v, 4) for k, v in best_metrics.items() if k != "cases"
})
print("MRR difference:", round(best_metrics["mrr"] - baseline_metrics["mrr"], 4))
print("\nSaved new checkpoint:", checkpoint_out)
print("Saved history:", history_out)
print("Saved metrics:", metrics_out)
print("Original ranker checkpoint preserved.")
print("Held-out test set was not used.")



Epoch 01/40 | loss=0.1436 | MRR=0.0125 | Top1=0.0000 | Top5=0.0143 | Top10=0.0232
Epoch 02/40 | loss=0.0658 | MRR=0.0209 | Top1=0.0000 | Top5=0.0309 | Top10=0.0640
Epoch 03/40 | loss=0.0574 | MRR=0.0133 | Top1=0.0000 | Top5=0.0166 | Top10=0.0464
Epoch 04/40 | loss=0.0543 | MRR=0.0149 | Top1=0.0000 | Top5=0.0199 | Top10=0.0397
Epoch 05/40 | loss=0.0506 | MRR=0.0190 | Top1=0.0000 | Top5=0.0309 | Top10=0.0640
Epoch 06/40 | loss=0.0503 | MRR=0.0133 | Top1=0.0000 | Top5=0.0221 | Top10=0.0331
Epoch 07/40 | loss=0.0493 | MRR=0.0123 | Top1=0.0000 | Top5=0.0110 | Top10=0.0132
Epoch 08/40 | loss=0.0487 | MRR=0.0114 | Top1=0.0000 | Top5=0.0110 | Top10=0.0199
Early stopping.

=== Validation comparison ===
Phenotype-only baseline: {'mrr': 0.336, 'top1': 0.2638, 'top5': 0.4073, 'top10': 0.489}
New bias-free ranker: {'mrr': 0.0209, 'top1': 0.0, 'top5': 0.0309, 'top10': 0.064}
MRR difference: -0.3151

Saved new checkpoint: /content/drive/MyDrive/Rare Disease ai/model/patient_level_ranker_bias_free_exp

In [ ]:

import json
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

baseline = pd.read_csv(DATA / "patient_phenotype_only_validation.csv")
ranker = pd.read_csv(DATA / "patient_level_ranker_bias_free_experiment_history.csv")
decision = json.loads((DATA / "model_selection_decision.json").read_text())

print("=== Baseline CSV metrics ===")
for col in ["mrr", "top1", "top5", "top10"]:
    print(f"{col}: {baseline[col].mean():.6f}")

print("\n=== Saved model-selection metrics ===")
print(json.dumps(decision["selected_validation_metrics"], indent=2))

print("\n=== Ranker best epoch ===")
best_idx = ranker["validation_mrr"].idxmax()
print(ranker.loc[best_idx].to_string())

print("\n=== Ranker checkpoint ===")
checkpoint = torch.load(
    MODEL_DIR / "patient_level_ranker_bias_free_experiment.pt",
    map_location="cpu",
    weights_only=True
)
print("Saved validation metrics:")
print(json.dumps(checkpoint["validation_metrics"], indent=2))
print("Parameter tensors:", len(checkpoint["model_state_dict"]))

print("\nAudit complete. No training or files changed.")


=== Baseline CSV metrics ===
mrr: 0.336028
top1: 0.263797
top5: 0.407285
top10: 0.488962

=== Saved model-selection metrics ===
{
  "cases": 906,
  "top1": 0.2947,
  "top5": 0.468,
  "top10": 0.5419,
  "mrr": 0.3773
}

=== Ranker best epoch ===
epoch               2.000000
train_loss          0.065830
validation_mrr      0.020924
validation_top1     0.000000
validation_top5     0.030905
validation_top10    0.064018

=== Ranker checkpoint ===
Saved validation metrics:
{
  "cases": 906,
  "mrr": 0.020924354840425125,
  "top1": 0.0,
  "top5": 0.03090507726269316,
  "top10": 0.0640176600441501
}
Parameter tensors: 6

Audit complete. No training or files changed.


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = [
    "patient_phenotype_only_validation.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_baseline_vs_gnn_paired.csv",
    "patient_three_method_common_cohort_comparison.csv",
]

for name in files:
    path = DATA / name
    if not path.exists():
        print(f"\n{name}: NOT FOUND")
        continue

    df = pd.read_csv(path)
    print(f"\n=== {name} ===")
    print("Rows:", len(df))
    print("Columns:", list(df.columns))

    metric_cols = [
        c for c in df.columns
        if c.lower() in {"mrr", "top1", "top5", "top10", "recall@10"}
    ]
    if metric_cols:
        print("Metric means:")
        print(df[metric_cols].mean(numeric_only=True).round(6).to_dict())

    if "method" in df.columns:
        print("Methods:", df["method"].dropna().unique()[:10].tolist())
    if "model" in df.columns:
        print("Models:", df["model"].dropna().unique()[:10].tolist())

print("\nRead-only audit complete.")



=== patient_phenotype_only_validation.csv ===
Rows: 906
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_recognized_hpo', 'target_in_candidate_set', 'rank', 'top1', 'top5', 'top10', 'mrr']
Metric means:
{'top1': 0.263797, 'top5': 0.407285, 'top10': 0.488962, 'mrr': 0.336028}

=== patient_phenotype_only_baseline.csv ===
Rows: 1959
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']
Metric means:
{'top1': 0.113834, 'top5': 0.196018, 'top10': 0.221031}

=== patient_baseline_vs_gnn_paired.csv ===
Rows: 1959
Columns: ['case_id', 'pmid', 'diagnosis_ids_baseline', 'rank_baseline', 'top1_baseline', 'top5_baseline', 'top10_baseline', 'candidate_covered', 'diagnosis_ids_gnn', 'n_positive_hpo', 'n_recognized_hpo', 'target_in_candidate_set', 'rank_gnn', 'top1_gnn', 'top5_gnn', 'top10_gnn', 'mrr', 'top_prediction']
Metric means:
{'mrr': 0.047607}

=== patient_three_method_common_cohort_comparison.csv ===
Rows: 1959
Columns: ['case_id', 'rank_baseline

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
baseline_val = pd.read_csv(DATA / "patient_phenotype_only_validation.csv", dtype=str)
# Check that targets are covered and count HPO terms
covered = baseline_val["target_in_candidate_set"].astype(str).value_counts()
print("Target in candidate set:", covered.to_dict())
print("Mean #HPO:", baseline_val["n_recognized_hpo"].astype(int).mean())

# Inspect candidate coverage from baseline CSV
baseline_full = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv", dtype=str)
print("\nFull baseline (1959 cases) coverage:")
print("Coverage mean:", baseline_full["candidate_covered"].astype(float).mean())
print("Top-1 baseline mean:", baseline_full["top1"].astype(int).mean())

# Compare diagnosis label consistency
val = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)
merged = val.merge(baseline_val[["case_id","diagnosis_ids"]], on="case_id", how="inner", suffixes=("_orig","_baseline"))
mismatches = merged[merged["diagnosis_ids_orig"] != merged["diagnosis_ids_baseline"]]
print("\nDiagnosis label mismatches in overlap:", mismatches.shape[0])


Target in candidate set: {'True': 906}
Mean #HPO: 8.665562913907285

Full baseline (1959 cases) coverage:
Coverage mean: 0.7917304747320061
Top-1 baseline mean: 0.1138335885655947

Diagnosis label mismatches in overlap: 0


In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

# Compare the saved validation metrics with the metrics
# actually present in the validation prediction CSV.
baseline_path = DATA / "patient_phenotype_only_validation.csv"
decision_path = DATA / "model_selection_decision.json"

baseline = pd.read_csv(baseline_path)
print("Baseline validation file:", baseline_path.name)
print("Rows:", len(baseline))
print("Columns:", baseline.columns.tolist())

if "rank" in baseline.columns:
    ranks = pd.to_numeric(baseline["rank"], errors="coerce")
    valid = ranks.notna() & (ranks > 0)
    reciprocal_rank = 1.0 / ranks[valid]

    print("\nMetrics recomputed from saved ranks")
    print("Cases with valid ranks:", int(valid.sum()))
    print("MRR:", round(reciprocal_rank.mean(), 6))
    print("Top-1:", round((ranks[valid] == 1).mean(), 6))
    print("Top-5:", round((ranks[valid] <= 5).mean(), 6))
    print("Top-10:", round((ranks[valid] <= 10).mean(), 6))
else:
    print("\nNo 'rank' column found; cannot recompute rank metrics.")

if decision_path.exists():
    decision = json.loads(decision_path.read_text())
    print("\nSaved model-selection decision")
    print(json.dumps(decision, indent=2))
else:
    print("\nModel-selection decision JSON not found.")

print("\nAudit complete. No model was trained or checkpoint changed.")


Baseline validation file: patient_phenotype_only_validation.csv
Rows: 906
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_recognized_hpo', 'target_in_candidate_set', 'rank', 'top1', 'top5', 'top10', 'mrr']

Metrics recomputed from saved ranks
Cases with valid ranks: 906
MRR: 0.336028
Top-1: 0.263797
Top-5: 0.407285
Top-10: 0.488962

Saved model-selection decision
{
  "decision": "retain_phenotype_only_baseline",
  "status": "research_prototype_not_clinically_validated",
  "selected_method": "information_content_weighted_phenotype_F1",
  "selected_validation_metrics": {
    "cases": 906,
    "top1": 0.2947,
    "top5": 0.468,
    "top10": 0.5419,
    "mrr": 0.3773
  },
  "score_combination_experiment": {
    "best_ranker_weight": 0.0,
    "best_validation_mrr": 0.3773,
    "interpretation": "No tested combination improved over the phenotype-only baseline."
  },
  "experimental_ranker": {
    "checkpoint": "/content/drive/MyDrive/Rare Disease ai/model/patient_level_ranker_best.pt",
    

In [ ]:

from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

# Inspect relevant existing metric and experiment artifacts.
keywords = (
    "baseline_metrics_audit",
    "ranker_diagnostics",
    "score_combination",
    "matched_candidate",
    "training_history",
    "bias_free_experiment_metrics",
    "model_selection_decision",
)

files = sorted(
    p for folder in (DATA, MODEL_DIR)
    if folder.exists()
    for p in folder.iterdir()
    if p.is_file() and any(k in p.name.lower() for k in keywords)
)

print("Relevant existing artifacts found:", len(files))

for path in files:
    print(f"\n{'=' * 70}\n{path.name}")

    try:
        if path.suffix.lower() == ".json":
            obj = json.loads(path.read_text())
            print(json.dumps(obj, indent=2)[:7000])

        elif path.suffix.lower() == ".csv":
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())

            metric_cols = [
                c for c in df.columns
                if any(k in c.lower() for k in
                       ("mrr", "top1", "top5", "top10", "rank", "case_id"))
            ]

            if metric_cols:
                print(df[metric_cols].head(5).to_string(index=False))

                numeric = df.select_dtypes(include="number")
                metric_numeric = [
                    c for c in numeric.columns
                    if any(k in c.lower() for k in
                           ("mrr", "top1", "top5", "top10", "rank"))
                ]
                if metric_numeric:
                    print("\nNumeric summaries:")
                    print(numeric[metric_numeric].describe().round(5).to_string())
            else:
                print(df.head(3).to_string(index=False))

        else:
            print("File type skipped; no changes made.")

    except Exception as e:
        print("Could not inspect this file:", repr(e))

print("\nRead-only artifact inspection complete.")


Relevant existing artifacts found: 9

model_selection_decision.json
{
  "decision": "retain_phenotype_only_baseline",
  "status": "research_prototype_not_clinically_validated",
  "selected_method": "information_content_weighted_phenotype_F1",
  "selected_validation_metrics": {
    "cases": 906,
    "top1": 0.2947,
    "top5": 0.468,
    "top10": 0.5419,
    "mrr": 0.3773
  },
  "score_combination_experiment": {
    "best_ranker_weight": 0.0,
    "best_validation_mrr": 0.3773,
    "interpretation": "No tested combination improved over the phenotype-only baseline."
  },
  "experimental_ranker": {
    "checkpoint": "/content/drive/MyDrive/Rare Disease ai/model/patient_level_ranker_best.pt",
    "preserved": true,
    "use_as_primary_model": false,
    "reason": "Bias term caused prediction collapse; bias-free ranker still underperformed baseline."
  },
  "held_out_test_set_used_for_model_selection": false,
  "recorded_at_utc": "2026-10-14T18:29:24.257713+00:00"
}

patient_level_ranker_bia

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import ast

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

train = pd.read_csv(DATA / "patient_level_train.csv", dtype=str)
val = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)
baseline = pd.read_csv(
    DATA / "patient_phenotype_only_validation.csv", dtype=str
)
matched = pd.read_csv(
    DATA / "patient_validation_matched_candidate_comparison.csv"
)

def parse_ids(value):
    if pd.isna(value):
        return []
    s = str(value).strip()
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple, set)):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, SyntaxError):
        pass
    return [x.strip().strip("'\"") for x in s.split(",") if x.strip()]

# Diagnose label cardinality: one diagnosis vs multiple per case.
train_dx_counts = train["diagnosis_ids"].map(lambda x: len(parse_ids(x)))
val_dx_counts = val["diagnosis_ids"].map(lambda x: len(parse_ids(x)))

print("TRAIN")
print("Cases:", len(train))
print("Diagnosis labels per case:")
print(train_dx_counts.value_counts().sort_index().to_dict())

print("\nVALIDATION")
print("Cases:", len(val))
print("Diagnosis labels per case:")
print(val_dx_counts.value_counts().sort_index().to_dict())

# Confirm the same validation cases and targets are represented.
base_ids = set(baseline["case_id"].astype(str))
matched_ids = set(matched["case_id"].astype(str))
val_ids = set(val["case_id"].astype(str))

print("\nCASE-SET CHECK")
print("Validation cases:", len(val_ids))
print("Baseline cases:", len(base_ids))
print("Matched-comparison cases:", len(matched_ids))
print("Validation vs baseline identical:", val_ids == base_ids)
print("Validation vs matched identical:", val_ids == matched_ids)

print("\nCANDIDATE/RANK CHECK")
print("Baseline target coverage:")
print(baseline["target_in_candidate_set"].value_counts(dropna=False).to_dict())
print("Baseline native MRR:", pd.to_numeric(baseline["mrr"]).mean())
print("Matched shared-candidate MRR:",
      (1 / pd.to_numeric(matched["baseline_common_rank"])).mean())
print("Matched ranker shared-candidate MRR:",
      (1 / pd.to_numeric(matched["ranker_common_rank"])).mean())

print("\nRead-only check complete; no files or checkpoints changed.")


TRAIN
Cases: 5730
Diagnosis labels per case:
{1: 5730}

VALIDATION
Cases: 906
Diagnosis labels per case:
{1: 906}

CASE-SET CHECK
Validation cases: 906
Baseline cases: 906
Matched-comparison cases: 906
Validation vs baseline identical: True
Validation vs matched identical: True

CANDIDATE/RANK CHECK
Baseline target coverage:
{'True': 906}
Baseline native MRR: 0.3360284586400965
Matched shared-candidate MRR: 0.3772558856502456
Matched ranker shared-candidate MRR: 0.052704318912176605

Read-only check complete; no files or checkpoints changed.


In [ ]:

from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

train_df = pd.read_csv(DATA / "patient_level_train.csv", dtype=str)
val_df = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)
ckpt = torch.load(
    MODEL_DIR / "patient_level_ranker_best.pt",
    map_location="cpu",
    weights_only=True
)

print("TRAIN HPO EXAMPLES")
for x in train_df["hpo_ids"].head(5):
    print(repr(x))

print("\nVALIDATION HPO EXAMPLES")
for x in val_df["hpo_ids"].head(5):
    print(repr(x))

print("\nDIAGNOSIS EXAMPLES")
for x in train_df["diagnosis_ids"].head(3):
    print(repr(x))

print("\nCHECKPOINT KEYS")
print(list(ckpt.keys()))

for key in ("disease_ids", "phenotype_ids"):
    value = ckpt.get(key)
    print(f"\n{key}: type={type(value).__name__}")
    if isinstance(value, dict):
        print("First entries:", list(value.items())[:5])
    elif isinstance(value, (list, tuple)):
        print("First entries:", value[:5])
    else:
        print("Preview:", repr(value)[:500])

print("\nHPO column dtype:", train_df["hpo_ids"].dtype)
print("Non-empty train HPO values:",
      int(train_df["hpo_ids"].fillna("").str.strip().ne("").sum()))
print("Non-empty validation HPO values:",
      int(val_df["hpo_ids"].fillna("").str.strip().ne("").sum()))


TRAIN HPO EXAMPLES
'HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003'
'HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003'
'HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0011003'
'HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003'
'HP:0001156;HP:0001387;HP:0001642;HP:0002650;HP:0002938;HP:0004322;HP:0011003;HP:0011623;HP:0030961'

VALIDATION HPO EXAMPLES
'HP:0001954;HP:0002027;HP:0002140;HP:0002301;HP:0003493;HP:0005197;HP:0010702;HP:0011107;HP:0025528;HP:0030783;HP:0030880;HP:0033260;HP:0033833;HP:0034056'
'HP:0000623;HP:0001954;HP:0002027;HP:0002140;HP:0002301;HP:0002315;HP:0002321;HP:0002829;HP:0003493;HP:0005197;HP:0011107;HP:0011227;HP:0012246;HP:0025343;HP:0030783;HP:0033505;HP:0033833'
'HP:0000218;HP:0000369;HP:0001639;HP:0001712;HP:0200128'
'HP:0000465;HP:0001639;HP:0001644;HP:0001712;HP:0002751;HP:0004751;HP:0012385;HP:0034665'
'HP:0001653;HP:0001712;HP:0004322;HP:0030718;HP:0031295'

DIAGNOSIS EXAMPLES
'OMIM:277600'
'OMIM:27760

In [ ]:

from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

train_df = pd.read_csv(DATA / "patient_level_train.csv", dtype=str)
val_df = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)

ckpt = torch.load(
    MODEL_DIR / "patient_level_ranker_best.pt",
    map_location="cpu",
    weights_only=True
)

disease_ids = list(map(str, ckpt["disease_ids"]))
phenotype_ids = list(map(str, ckpt["phenotype_ids"]))
d2i = {x: i for i, x in enumerate(disease_ids)}
p2i = {x: i for i, x in enumerate(phenotype_ids)}

def parse_semicolon_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split(";") if x.strip()]

def audit_split(df, name):
    missing_diseases = []
    no_hpo = []
    hpos_with_no_vocab_match = []
    cases_with_zero_recognized_hpos = 0
    recognized_hpo_counts = []

    for _, row in df.iterrows():
        dx = parse_semicolon_ids(row["diagnosis_ids"])
        hpos = parse_semicolon_ids(row["hpo_ids"])

        if len(dx) != 1 or dx[0] not in d2i:
            missing_diseases.append(str(row["case_id"]))

        if not hpos:
            no_hpo.append(str(row["case_id"]))

        recognized = [h for h in hpos if h in p2i]
        recognized_hpo_counts.append(len(recognized))

        if hpos and not recognized:
            cases_with_zero_recognized_hpos += 1

        hpos_with_no_vocab_match.extend(h for h in hpos if h not in p2i)

    print(f"\n{name}")
    print("Cases:", len(df))
    print("Cases with missing/invalid disease target:", len(missing_diseases))
    print("Cases with no HPO IDs:", len(no_hpo))
    print("Cases with zero recognized HPOs:", cases_with_zero_recognized_hpos)
    print("Total HPO occurrences:", sum(len(parse_semicolon_ids(x)) for x in df["hpo_ids"]))
    print("Recognized HPO occurrences:", sum(recognized_hpo_counts))
    print("HPO occurrence coverage:",
          round(sum(recognized_hpo_counts) /
                max(1, sum(len(parse_semicolon_ids(x)) for x in df["hpo_ids"])), 6))
    print("Unique HPO IDs absent from vocabulary:",
          len(set(hpos_with_no_vocab_match)))
    print("Example absent IDs:", sorted(set(hpos_with_no_vocab_match))[:10])

    return {
        "cases": len(df),
        "invalid_disease_cases": len(missing_diseases),
        "no_hpo_cases": len(no_hpo),
        "zero_recognized_hpo_cases": cases_with_zero_recognized_hpos
    }

train_result = audit_split(train_df, "TRAIN")
val_result = audit_split(val_df, "VALIDATION")

assert train_result["invalid_disease_cases"] == 0
assert val_result["invalid_disease_cases"] == 0
assert train_result["no_hpo_cases"] == 0
assert val_result["no_hpo_cases"] == 0
assert train_result["zero_recognized_hpo_cases"] == 0
assert val_result["zero_recognized_hpo_cases"] == 0

print("\nPASS: all cases have one known disease target and at least one recognized HPO.")
print("No training performed and no checkpoints changed.")



TRAIN
Cases: 5730
Cases with missing/invalid disease target: 0
Cases with no HPO IDs: 0
Cases with zero recognized HPOs: 0
Total HPO occurrences: 49192
Recognized HPO occurrences: 49192
HPO occurrence coverage: 1.0
Unique HPO IDs absent from vocabulary: 0
Example absent IDs: []

VALIDATION
Cases: 906
Cases with missing/invalid disease target: 0
Cases with no HPO IDs: 0
Cases with zero recognized HPOs: 0
Total HPO occurrences: 7851
Recognized HPO occurrences: 7851
HPO occurrence coverage: 1.0
Unique HPO IDs absent from vocabulary: 0
Example absent IDs: []

PASS: all cases have one known disease target and at least one recognized HPO.
No training performed and no checkpoints changed.


In [ ]:

# ============================================================
# CELL 4C — Corrected Bias-Aware Softmax Ranker Experiment
# ============================================================

from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load data and original vocabulary (read-only)
train_df = pd.read_csv(DATA / "patient_level_train.csv", dtype=str)
val_df = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)

original_checkpoint_path = MODEL_DIR / "patient_level_ranker_best.pt"
old_ckpt = torch.load(
    original_checkpoint_path,
    map_location="cpu",
    weights_only=True
)

disease_ids = list(map(str, old_ckpt["disease_ids"]))
phenotype_ids = list(map(str, old_ckpt["phenotype_ids"]))

d2i = {x: i for i, x in enumerate(disease_ids)}
p2i = {x: i for i, x in enumerate(phenotype_ids)}
n_diseases = len(disease_ids)
n_phenotypes = len(phenotype_ids)

print("Device:", device)
print("Diseases:", n_diseases, "| Phenotypes:", n_phenotypes)

# 3. Parse semicolon-separated IDs correctly
def parse_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split(";") if x.strip()]

def encode_frame(df, split_name):
    cases = []

    for _, row in df.iterrows():
        dx_ids = parse_ids(row["diagnosis_ids"])
        hp_ids = parse_ids(row["hpo_ids"])

        if len(dx_ids) != 1:
            raise ValueError(
                f"{split_name}: expected one diagnosis for {row['case_id']}, "
                f"found {len(dx_ids)}"
            )

        if dx_ids[0] not in d2i:
            raise ValueError(
                f"{split_name}: disease missing from vocabulary: {dx_ids[0]}"
            )

        hp_indices = list(dict.fromkeys(
            p2i[h] for h in hp_ids if h in p2i
        ))

        if not hp_indices:
            raise ValueError(
                f"{split_name}: no recognized HPOs for {row['case_id']}"
            )

        cases.append({
            "case_id": str(row["case_id"]),
            "diagnosis_id": dx_ids[0],
            "dx": d2i[dx_ids[0]],
            "hpos": hp_indices
        })

    print(f"{split_name} encoded cases: {len(cases)}")
    return cases

train_cases = encode_frame(train_df, "TRAIN")
val_cases = encode_frame(val_df, "VALIDATION")

assert len(train_cases) == 5730
assert len(val_cases) == 906

# 4. Model
HIDDEN = 128

class BiasSoftmaxRanker(nn.Module):
    def __init__(self):
        super().__init__()

        self.hpo_embedding = nn.Embedding(n_phenotypes, HIDDEN)
        self.disease_embedding = nn.Embedding(n_diseases, HIDDEN)

        self.patient_projection = nn.Sequential(
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(HIDDEN, HIDDEN)
        )

        self.disease_bias = nn.Parameter(torch.zeros(n_diseases))

        nn.init.normal_(self.hpo_embedding.weight, mean=0.0, std=0.05)
        nn.init.normal_(self.disease_embedding.weight, mean=0.0, std=0.05)

    def score(self, hpos):
        h = torch.as_tensor(
            hpos, dtype=torch.long, device=device
        )

        patient = self.patient_projection(
            self.hpo_embedding(h).mean(dim=0)
        )

        # FIX: use the embedding weights, not the Embedding module.
        return F.linear(
            patient.unsqueeze(0),
            self.disease_embedding.weight,
            self.disease_bias
        ).squeeze(0)

model = BiasSoftmaxRanker().to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# 5. Validation metrics across the complete disease vocabulary
@torch.no_grad()
def evaluate(cases):
    model.eval()
    rows = []

    for case in cases:
        scores = model.score(case["hpos"])
        target_score = scores[case["dx"]]

        # Rank 1 means no disease has a strictly higher score.
        rank = int((scores > target_score).sum().item()) + 1

        rows.append({
            "case_id": case["case_id"],
            "diagnosis_id": case["diagnosis_id"],
            "rank": rank,
            "top1": int(rank <= 1),
            "top5": int(rank <= 5),
            "top10": int(rank <= 10),
            "mrr": 1.0 / rank
        })

    predictions = pd.DataFrame(rows)

    metrics = {
        "cases": int(len(predictions)),
        "mrr": float(predictions["mrr"].mean()),
        "top1": float(predictions["top1"].mean()),
        "top5": float(predictions["top5"].mean()),
        "top10": float(predictions["top10"].mean())
    }

    return metrics, predictions

# 6. Train with full-disease cross-entropy
EPOCHS = 20
PATIENCE = 4

best_mrr = -1.0
best_epoch = 0
best_state = None
stale_epochs = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    order = np.random.permutation(len(train_cases))
    losses = []

    for idx in order:
        case = train_cases[int(idx)]

        optimizer.zero_grad(set_to_none=True)

        logits = model.score(case["hpos"]).unsqueeze(0)
        target = torch.tensor(
            [case["dx"]], dtype=torch.long, device=device
        )

        loss = F.cross_entropy(logits, target)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        losses.append(float(loss.item()))

    val_metrics, _ = evaluate(val_cases)

    record = {
        "epoch": epoch,
        "train_loss": float(np.mean(losses)),
        **{f"val_{k}": v for k, v in val_metrics.items()}
    }
    history.append(record)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss={record['train_loss']:.4f} | "
        f"MRR={val_metrics['mrr']:.4f} | "
        f"Top1={val_metrics['top1']:.4f} | "
        f"Top5={val_metrics['top5']:.4f} | "
        f"Top10={val_metrics['top10']:.4f}"
    )

    if val_metrics["mrr"] > best_mrr:
        best_mrr = val_metrics["mrr"]
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        stale_epochs = 0
    else:
        stale_epochs += 1

    if stale_epochs >= PATIENCE:
        print("Early stopping.")
        break

assert best_state is not None

# Restore the best validation checkpoint.
model.load_state_dict(best_state)
final_metrics, predictions = evaluate(val_cases)

# 7. Save ONLY to new experiment-specific filenames
checkpoint_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
history_path = DATA / "patient_ranker_bias_softmax_experiment_history.csv"
pred_path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
metrics_path = DATA / "patient_ranker_bias_softmax_experiment_metrics.json"
config_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment_config.json"

torch.save({
    "model_state_dict": best_state,
    "disease_ids": disease_ids,
    "phenotype_ids": phenotype_ids,
    "hidden_dim": HIDDEN,
    "best_epoch": best_epoch,
    "best_validation_metrics": final_metrics,
    "seed": SEED,
    "training_cases": len(train_cases),
    "validation_cases": len(val_cases),
    "objective": "single-label full-disease cross-entropy",
    "disease_bias": True,
    "note": "Research experiment; not clinically validated."
}, checkpoint_path)

pd.DataFrame(history).to_csv(history_path, index=False)
predictions.to_csv(pred_path, index=False)
metrics_path.write_text(json.dumps(final_metrics, indent=2))

config_path.write_text(json.dumps({
    "seed": SEED,
    "hidden_dim": HIDDEN,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "epochs_run": len(history),
    "best_epoch": best_epoch,
    "patience": PATIENCE,
    "objective": "single-label full-disease cross-entropy",
    "disease_bias": True,
    "device": str(device)
}, indent=2))

# 8. Confirm original checkpoint still exists.
assert original_checkpoint_path.exists()

print("\nBEST EPOCH:", best_epoch)
print("FINAL VALIDATION METRICS:")
print(json.dumps(final_metrics, indent=2))
print("\nSaved new checkpoint:", checkpoint_path)
print("Saved validation predictions:", pred_path)
print("Original ranker checkpoint was not overwritten.")


Device: cuda
Diseases: 7340 | Phenotypes: 9982
TRAIN encoded cases: 5730
VALIDATION encoded cases: 906
Epoch 01 | Loss=5.2553 | MRR=0.0509 | Top1=0.0320 | Top5=0.0596 | Top10=0.0883
Epoch 02 | Loss=3.6179 | MRR=0.1355 | Top1=0.0861 | Top5=0.1843 | Top10=0.2163
Epoch 03 | Loss=2.7584 | MRR=0.2305 | Top1=0.1965 | Top5=0.2682 | Top10=0.2903
Epoch 04 | Loss=2.0961 | MRR=0.2340 | Top1=0.1876 | Top5=0.2925 | Top10=0.3245
Epoch 05 | Loss=1.6358 | MRR=0.2793 | Top1=0.2362 | Top5=0.3256 | Top10=0.3532
Epoch 06 | Loss=1.3340 | MRR=0.3045 | Top1=0.2759 | Top5=0.3311 | Top10=0.3587
Epoch 07 | Loss=1.1126 | MRR=0.3162 | Top1=0.2848 | Top5=0.3477 | Top10=0.3786
Epoch 08 | Loss=0.9410 | MRR=0.3110 | Top1=0.2704 | Top5=0.3521 | Top10=0.3797
Epoch 09 | Loss=0.7855 | MRR=0.3219 | Top1=0.2903 | Top5=0.3565 | Top10=0.3819
Epoch 10 | Loss=0.6863 | MRR=0.2932 | Top1=0.2439 | Top5=0.3499 | Top10=0.3863
Epoch 11 | Loss=0.6040 | MRR=0.3192 | Top1=0.2815 | Top5=0.3620 | Top10=0.3863
Epoch 12 | Loss=0.5421 | MRR

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import json
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("Project folder exists:", PROJECT.exists())
print("Processed data folder exists:", DATA.exists())
print("Model folder exists:", MODEL_DIR.exists())

files_to_check = [
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    MODEL_DIR / "patient_level_ranker_best.pt",
    MODEL_DIR / "rare_disease_gnn_publication_disjoint_experiment.pt",
    DATA / "patient_ranker_bias_softmax_experiment_metrics.json",
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv",
    DATA / "patient_level_validation.csv",
    DATA / "patient_phenotype_only_validation.csv",
]

print("\nSaved artifacts:")
for path in files_to_check:
    print(f"{'FOUND' if path.exists() else 'MISSING'}  {path.name}")

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project folder exists: True
Processed data folder exists: True
Model folder exists: True

Saved artifacts:
FOUND  patient_ranker_bias_softmax_experiment.pt
FOUND  patient_level_ranker_best.pt
FOUND  rare_disease_gnn_publication_disjoint_experiment.pt
FOUND  patient_ranker_bias_softmax_experiment_metrics.json
FOUND  patient_ranker_bias_softmax_experiment_validation.csv
FOUND  patient_level_validation.csv
FOUND  patient_phenotype_only_validation.csv

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

# 1. Read saved neural-ranker metrics
metrics_path = DATA / "patient_ranker_bias_softmax_experiment_metrics.json"

print("=" * 60)
print("BIAS-AWARE SOFTMAX RANKER — SAVED METRICS")
print("=" * 60)

with open(metrics_path, "r") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

# 2. Inspect validation prediction columns and sample rows
prediction_path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
predictions = pd.read_csv(prediction_path)

print("\n" + "=" * 60)
print("NEURAL-RANKER VALIDATION FILE")
print("=" * 60)
print("Shape:", predictions.shape)
print("Columns:", predictions.columns.tolist())
print(predictions.head(5).to_string(index=False))

# 3. Inspect phenotype baseline validation file
baseline_path = DATA / "patient_phenotype_only_validation.csv"
baseline = pd.read_csv(baseline_path)

print("\n" + "=" * 60)
print("PHENOTYPE BASELINE VALIDATION FILE")
print("=" * 60)
print("Shape:", baseline.shape)
print("Columns:", baseline.columns.tolist())
print(baseline.head(5).to_string(index=False))

BIAS-AWARE SOFTMAX RANKER — SAVED METRICS
{
  "cases": 906,
  "mrr": 0.3341009854885823,
  "top1": 0.30132450331125826,
  "top5": 0.3631346578366446,
  "top10": 0.39514348785871967
}

NEURAL-RANKER VALIDATION FILE
Shape: (906, 7)
Columns: ['case_id', 'diagnosis_id', 'rank', 'top1', 'top5', 'top10', 'mrr']
                                       case_id diagnosis_id  rank  top1  top5  top10      mrr
                       PMID_28830446_Patient_2  OMIM:615688   548     0     0      0 0.001825
                       PMID_28830446_Patient_1  OMIM:615688   545     0     0      0 0.001835
PMID_32480058_Case_F5P1_original_pedigree_II_2  OMIM:618052  6971     0     0      0 0.000143
PMID_32480058_Case_F4P2_original_pedigree_IV_5  OMIM:618052  6645     0     0      0 0.000150
                      PMID_32480058_Case_F10P2  OMIM:618052  4784     0     0      0 0.000209

PHENOTYPE BASELINE VALIDATION FILE
Shape: (906, 10)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_recognized_hpo', 'target_in

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

comparison_path = DATA / "patient_validation_matched_candidate_comparison.csv"
summary_path = DATA / "patient_validation_matched_candidate_summary.json"

print("Matched comparison file exists:", comparison_path.exists())
print("Matched summary file exists:", summary_path.exists())

if comparison_path.exists():
    df = pd.read_csv(comparison_path)

    print("\nShape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst 5 rows:")
    print(df.head().to_string(index=False))

if summary_path.exists():
    import json
    with open(summary_path, "r") as f:
        summary = json.load(f)

    print("\nSaved matched-candidate summary:")
    print(json.dumps(summary, indent=2))

Matched comparison file exists: True
Matched summary file exists: True

Shape: (906, 11)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_observed_hpo', 'baseline_native_rank', 'ranker_native_rank', 'baseline_common_rank', 'ranker_common_rank', 'target_in_baseline_vocab', 'target_in_ranker_vocab', 'target_in_common_vocab']

First 5 rows:
                                       case_id     pmid diagnosis_ids  n_observed_hpo  baseline_native_rank  ranker_native_rank  baseline_common_rank  ranker_common_rank  target_in_baseline_vocab  target_in_ranker_vocab  target_in_common_vocab
                       PMID_28830446_Patient_2 28830446   OMIM:615688              14                    19                2831                     7                2831                         1                       1                       1
                       PMID_28830446_Patient_1 28830446   OMIM:615688              17                    19                2711                    11                2711   

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=== PROCESSED FILES RELATED TO PREDICTION ===")
keywords = [
    "phenotype", "baseline", "disease", "hpo", "ontology",
    "annotation", "rank", "mapping", "label", "metric"
]

for path in sorted(DATA.iterdir()):
    if path.is_file() and any(k in path.name.lower() for k in keywords):
        print(f"{path.name}  ({path.stat().st_size:,} bytes)")

print("\n=== MODEL FILES ===")
for path in sorted(MODEL_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name}  ({path.stat().st_size:,} bytes)")

print("\n=== PROJECT ROOT FILES ===")
for path in sorted(PROJECT.iterdir()):
    if path.is_file():
        print(f"{path.name}  ({path.stat().st_size:,} bytes)")

=== PROCESSED FILES RELATED TO PREDICTION ===
annotation_leakage_readiness_audit.json  (1,396 bytes)
annotation_provenance_followup_audit.json  (272,991 bytes)
baseline_coverage_comparison_audit.csv  (429 bytes)
baseline_model_comparison.csv  (166 bytes)
clean_hpo_annotations.csv  (35,907,264 bytes)
disease_level_split.csv  (162,821 bytes)
disease_ranking_retrieval_metrics.csv  (45,159 bytes)
gene_informed_phenotype_recovery_metrics.csv  (90,137 bytes)
gene_informed_phenotype_recovery_top10.csv  (647,512 bytes)
gnn_annotations_excluding_dev_test_pmids.csv  (20,759,359 bytes)
gnn_annotations_publication_disjoint_candidate.csv  (20,694,622 bytes)
gnn_phenotype_recovery_metrics.csv  (74,139 bytes)
gnn_training_annotations.csv  (23,014,254 bytes)
heldout_input_phenotypes.csv  (496,634 bytes)
heldout_profile_nearest_training_disease.csv  (85,055 bytes)
heldout_target_phenotypes.csv  (517,620 bytes)
hierarchy_aware_retrieval_metrics.csv  (83,214 bytes)
ic_weighted_heldout_ranking_results.csv

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

files_to_inspect = [
    MODEL_DIR / "rare_disease_predictor.py",
    MODEL_DIR / "rare_disease_ai.py",
    MODEL_DIR / "rare_disease_data_loader.py",
]

for path in files_to_inspect:
    print("\n" + "=" * 80)
    print("FILE:", path.name)
    print("=" * 80)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    print(path.read_text(encoding="utf-8"))


FILE: rare_disease_predictor.py
"""Rare Disease Prediction AI — research prototype.

This module contains ranking functions exported from the Colab notebook.
It is not a clinically validated diagnostic system.
"""

import pandas as pd
import numpy as np

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    observed = {
        str(t).strip()
        for t in patient_hpo_terms
        if str(t).strip() in ic
    }

    if not observed:
        raise ValueError("No recognized HPO terms. Use valid HPO IDs.")

    weights = {t: ic[t] for t in observed}
    denominator = sum(weights.values()) or 1.0

    genes = {
        str(g).strip().upper()
        for g in (patient_genes or [])
        if str(g).strip()
    }

    results = []

    for disease, disease_terms in phenotype_map.items():
        overlap = observed & disease_terms

        precision = sum(weights[t] for t in overlap) / denominator
        recall = len(overlap) / 

In [ ]:
from pathlib import Path
import sys
import traceback

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

# Make the existing project modules importable
sys.path.insert(0, str(MODEL_DIR))

try:
    from rare_disease_data_loader import load_prediction_data
    from rare_disease_ai import predict_diseases

    print("Loading prediction data...")
    data = load_prediction_data()

    print("\nData loaded successfully")
    print("Diseases with phenotype annotations:", len(data["phenotype_map"]))
    print("Recognized HPO terms:", len(data["ic"]))
    print("Diseases with gene mappings:", len(data["gene_map"]))
    print("Disease names available:", len(data["disease_names"]))

    # Software smoke test only — not a clinical case
    example_terms = ["HP:0001250", "HP:0001251", "HP:0001263"]

    recognized = [term for term in example_terms if term in data["ic"]]
    print("\nExample HPO terms recognized:", recognized)

    results = predict_diseases(
        patient_hpo_terms=recognized,
        top_k=5
    )

    print("\nTop 5 test predictions:")
    print(results.to_string(index=False))

    assert not results.empty, "Prediction returned no results."
    assert "disease_id" in results.columns
    assert "combined_score" in results.columns

    print("\nPASS: prediction pipeline smoke test completed.")

except Exception:
    print("\nFAIL: prediction pipeline needs fixing.")
    traceback.print_exc()

Loading prediction data...

Data loaded successfully
Diseases with phenotype annotations: 12879
Recognized HPO terms: 9982
Diseases with gene mappings: 11168
Disease names available: 12879

Example HPO terms recognized: ['HP:0001250', 'HP:0001251', 'HP:0001263']

Top 5 test predictions:
 rank  disease_id                                             disease_name  phenotype_score gene_score  combined_score  matching_phenotypes matching_phenotype_ids matching_genes
    1  ORPHA:2798     Pachygyria-intellectual disability-epilepsy syndrome         0.524358       None        0.524358                    2  HP:0001250;HP:0001263               
    2  ORPHA:3233           Cochleosaccular degeneration-cataract syndrome         0.516822       None        0.516822                    2  HP:0001250;HP:0001251               
    3 ORPHA:85338 X-linked intellectual disability-ataxia-apraxia syndrome         0.516822       None        0.516822                    2  HP:0001250;HP:0001251               


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
APP_PATH = MODEL_DIR / "streamlit_app.py"

app_code = r'''
import sys
from pathlib import Path

import streamlit as st

MODEL_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(MODEL_DIR))

from rare_disease_ai import predict_diseases

st.set_page_config(
    page_title="Rare Disease Prediction AI",
    page_icon="🧬",
    layout="wide"
)

st.title("🧬 Rare Disease Prediction AI")
st.caption("Phenotype-based candidate ranking | Research prototype")

st.warning(
    "Research use only. This tool prioritizes candidate diseases from "
    "phenotype matches; it is not a clinical diagnostic system and "
    "must not replace assessment by a qualified healthcare professional."
)

st.markdown(
    """
    Enter Human Phenotype Ontology (HPO) identifiers to rank candidate
    diseases. You can optionally enter gene symbols when available.
    """
)

with st.form("prediction_form"):
    hpo_input = st.text_area(
        "Patient phenotype HPO IDs",
        placeholder="HP:0001250\nHP:0001251\nHP:0001263",
        help="Enter one HPO ID per line or separate IDs with commas or semicolons."
    )

    gene_input = st.text_input(
        "Optional gene symbols",
        placeholder="SCN1A, MECP2",
        help="Optional. Enter gene symbols separated by commas, semicolons, or new lines."
    )

    top_k = st.slider("Number of candidates to show", 5, 20, 10)

    submitted = st.form_submit_button(
        "Rank candidate diseases",
        type="primary"
    )

if submitted:
    hpo_terms = [
        item.strip()
        for item in hpo_input.replace(",", "\n").replace(";", "\n").splitlines()
        if item.strip()
    ]
    genes = [
        item.strip().upper()
        for item in gene_input.replace(",", "\n").replace(";", "\n").splitlines()
        if item.strip()
    ]

    if not hpo_terms:
        st.error("Enter at least one HPO ID.")
    else:
        try:
            with st.spinner("Ranking candidate diseases..."):
                results = predict_diseases(
                    patient_hpo_terms=hpo_terms,
                    patient_genes=genes or None,
                    top_k=top_k
                )

            st.subheader("Ranked candidate diseases")
            st.write(
                "Scores indicate relative matching under this prototype's "
                "ranking method; they are not probabilities of having a disease."
            )

            st.dataframe(
                results,
                use_container_width=True,
                hide_index=True,
                column_config={
                    "rank": st.column_config.NumberColumn("Rank"),
                    "disease_id": st.column_config.TextColumn("Disease ID"),
                    "disease_name": st.column_config.TextColumn("Disease"),
                    "phenotype_score": st.column_config.NumberColumn(
                        "Phenotype score", format="%.4f"
                    ),
                    "gene_score": st.column_config.NumberColumn(
                        "Gene score", format="%.4f"
                    ),
                    "combined_score": st.column_config.NumberColumn(
                        "Ranking score", format="%.4f"
                    ),
                    "matching_phenotypes": st.column_config.NumberColumn(
                        "Matching phenotypes"
                    ),
                    "matching_phenotype_ids": st.column_config.TextColumn(
                        "Matching HPO IDs"
                    ),
                    "matching_genes": st.column_config.TextColumn(
                        "Matching genes"
                    ),
                }
            )

            st.subheader("Candidate explanations")
            for _, row in results.iterrows():
                disease_name = row.get("disease_name", "Unknown disease")
                disease_id = row.get("disease_id", "")
                with st.expander(
                    f"#{int(row['rank'])} — {disease_name} ({disease_id})"
                ):
                    st.write(
                        f"**Ranking score:** {row['combined_score']:.4f}"
                    )
                    st.write(
                        f"**Matching phenotype count:** "
                        f"{int(row['matching_phenotypes'])}"
                    )
                    st.write(
                        "**Matching HPO IDs:** "
                        f"{row.get('matching_phenotype_ids', '') or 'None'}"
                    )
                    matching_genes = row.get("matching_genes", "")
                    st.write(
                        "**Matching genes:** "
                        f"{matching_genes if matching_genes else 'None'}"
                    )

            st.download_button(
                "Download results as CSV",
                data=results.to_csv(index=False).encode("utf-8"),
                file_name="rare_disease_ranked_candidates.csv",
                mime="text/csv"
            )

        except ValueError as exc:
            st.error(str(exc))
        except Exception as exc:
            st.error(
                "Prediction failed. Check the project files and error details."
            )
            st.exception(exc)

st.divider()
st.caption(
    "BTech project prototype | Phenotype similarity ranking | "
    "Outputs require expert interpretation."
)
'''

APP_PATH.write_text(app_code, encoding="utf-8")

print("App file created:", APP_PATH)
print("File size:", APP_PATH.stat().st_size, "bytes")
print("\nNext we will check the app dependencies and launch method.")

App file created: /content/drive/MyDrive/Rare Disease ai/model/streamlit_app.py
File size: 5313 bytes

Next we will check the app dependencies and launch method.


In [ ]:
import importlib.util
import sys
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

print("Python:", sys.version.split()[0])

packages = ["streamlit", "pandas", "numpy"]

print("\n=== Dependency check ===")
for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'INSTALLED' if installed else 'MISSING'}")

print("\n=== App files ===")
for filename in [
    "streamlit_app.py",
    "rare_disease_ai.py",
    "rare_disease_data_loader.py",
]:
    path = MODEL_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

print("\n=== App syntax check ===")
app_path = MODEL_DIR / "streamlit_app.py"

try:
    compile(app_path.read_text(encoding="utf-8"), str(app_path), "exec")
    print("PASS: Python syntax is valid.")
except SyntaxError as exc:
    print("FAIL:", exc)

Python: 3.13.15

=== Dependency check ===
streamlit: MISSING
pandas: INSTALLED
numpy: INSTALLED

=== App files ===
streamlit_app.py: FOUND
rare_disease_ai.py: FOUND
rare_disease_data_loader.py: FOUND

=== App syntax check ===
PASS: Python syntax is valid.


In [ ]:
%pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 74.6 MB/s eta 0:00:00


In [ ]:
import streamlit
import pandas
import numpy

print("Streamlit:", streamlit.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("PASS: required app packages are available.")

Streamlit: 1.66.0
Pandas: 2.2.3
NumPy: 2.1.3
PASS: required app packages are available.


In [ ]:
from pathlib import Path
import subprocess
import sys
import time

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
APP_PATH = MODEL_DIR / "streamlit_app.py"
LOG_PATH = PROJECT / "streamlit.log"
PID_PATH = PROJECT / "streamlit.pid"

assert APP_PATH.exists(), f"App not found: {APP_PATH}"

# Stop a previous app started by this notebook, if its PID is recorded.
if PID_PATH.exists():
    try:
        old_pid = int(PID_PATH.read_text().strip())
        subprocess.run(
            ["kill", str(old_pid)],
            check=False,
            capture_output=True
        )
    except (ValueError, OSError):
        pass

log_file = open(LOG_PATH, "w")
process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(APP_PATH),
        "--server.port", "8501",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false"
    ],
    cwd=str(MODEL_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

PID_PATH.write_text(str(process.pid))
time.sleep(5)

print("Streamlit process PID:", process.pid)
print("Log file:", LOG_PATH)
print("\n=== Recent startup logs ===")

if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("\n".join(lines[-25:]) if lines else "No logs yet.")

print("\nThe server process may be running, but Colab has not automatically exposed it as a public URL.")

Streamlit process PID: 5279
Log file: /content/drive/MyDrive/Rare Disease ai/streamlit.log

=== Recent startup logs ===
2026-10-18 05:17:31.059 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.185.15.22:8501


The server process may be running, but Colab has not automatically exposed it as a public URL.


In [ ]:
from google.colab import output

print("Opening the Streamlit app preview...")
output.serve_kernel_port_as_window(8501)

Opening the Streamlit app preview...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import subprocess

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
LOG_PATH = PROJECT / "streamlit.log"

print("=== Streamlit server process ===")
result = subprocess.run(
    ["ps", "-p", "5279", "-o", "pid,stat,cmd"],
    capture_output=True,
    text=True
)
print(result.stdout or "Process 5279 is not running.")

print("\n=== Recent Streamlit logs ===")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("\n".join(lines[-60:]) if lines else "Log file is empty.")
else:
    print("Log file not found.")

=== Streamlit server process ===
    PID STAT CMD
   5279 Ssl  /usr/bin/python3 -m streamlit run /content/drive/MyDrive/Rare Disease ai/model/streamlit_app.py --server.port 8501 --server.headless true --browser.gatherUsageStats false


=== Recent Streamlit logs ===
2026-10-18 05:17:31.059 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.185.15.22:8501

2026-10-18 05:18:46.658 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-s-kkb-use1c1-72o3qmkkaet4-c.us-east1-1.prod.colab.dev, host=gpu-t4-s-kkb-use1c1-72o3qmkkaet4.us-east1-c.c.codatalab-user-runtimes.internal:8007
2026-10-18 05:18:48.752 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-s-kkb-use1c1-72o3qmkkaet4-c.us-east1-1.prod.colab.dev, host=gpu-t4-s-kkb-use1c1-72o3qmkkaet4.us-east1-c.c.codatalab-u

In [ ]:
from pathlib import Path
import subprocess
import sys
import time

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
APP_PATH = MODEL_DIR / "streamlit_app.py"
LOG_PATH = PROJECT / "streamlit.log"
PID_PATH = PROJECT / "streamlit.pid"

# Stop the previous Streamlit process
if PID_PATH.exists():
    try:
        old_pid = int(PID_PATH.read_text().strip())
        subprocess.run(
            ["kill", str(old_pid)],
            check=False,
            capture_output=True
        )
        time.sleep(2)
    except (ValueError, OSError):
        pass

# Restart with settings for a proxied Colab preview
with open(LOG_PATH, "w") as log:
    process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", str(APP_PATH),
            "--server.port", "8501",
            "--server.address", "0.0.0.0",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
            "--server.enableWebsocketCompression", "false",
            "--browser.gatherUsageStats", "false",
        ],
        cwd=str(MODEL_DIR),
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

PID_PATH.write_text(str(process.pid))

time.sleep(5)

print("Streamlit PID:", process.pid)
print("\n=== Startup logs ===")
lines = LOG_PATH.read_text(errors="replace").splitlines()
print("\n".join(lines[-20:]) if lines else "No logs yet.")

Streamlit PID: 6233

=== Startup logs ===
2026-10-18 05:21:10.153 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.185.15.22:8501



In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

sys.path.insert(0, str(MODEL_DIR))

from rare_disease_data_loader import load_prediction_data
from rare_disease_ai import predict_diseases

print("Loading saved annotations and validation cases...")
data = load_prediction_data()

validation_path = DATA / "patient_level_validation.csv"
val = pd.read_csv(validation_path, dtype=str).fillna("")

print("Validation cases:", len(val))
print("Validation columns:", val.columns.tolist())

# Parse semicolon-separated HPO and diagnosis IDs.
def parse_ids(value):
    return {
        item.strip()
        for item in str(value).split(";")
        if item.strip()
    }

records = []
skipped = []

for _, row in val.iterrows():
    case_id = row["case_id"]
    hpo_terms = sorted(parse_ids(row["hpo_ids"]))
    targets = parse_ids(row["diagnosis_ids"])

    recognized = [h for h in hpo_terms if h in data["ic"]]

    if not recognized or not targets:
        skipped.append(case_id)
        continue

    try:
        # Rank the complete candidate set, not just the top 10.
        ranked = predict_diseases(
            patient_hpo_terms=recognized,
            top_k=len(data["phenotype_map"])
        )

        target_ranks = ranked.loc[
            ranked["disease_id"].isin(targets), "rank"
        ]

        if target_ranks.empty:
            target_rank = np.nan
        else:
            target_rank = int(target_ranks.min())

        records.append({
            "case_id": case_id,
            "target_diagnoses": ";".join(sorted(targets)),
            "n_input_hpo": len(hpo_terms),
            "n_recognized_hpo": len(recognized),
            "target_rank": target_rank,
            "target_retrieved": not target_ranks.empty,
        })

    except Exception as exc:
        skipped.append(case_id)
        print(f"Skipped {case_id}: {exc}")

report = pd.DataFrame(records)

if report.empty:
    print("No cases could be evaluated. We need to inspect the validation format.")
else:
    evaluated = report.dropna(subset=["target_rank"]).copy()
    ranks = evaluated["target_rank"].to_numpy()

    metrics = {
        "cases_in_validation": len(val),
        "evaluated_cases": len(evaluated),
        "skipped_cases": len(skipped),
        "target_retrieval_rate": float(report["target_retrieved"].mean()),
        "MRR": float(np.mean(1.0 / ranks)) if len(ranks) else 0.0,
        "Top1": float(np.mean(ranks <= 1)) if len(ranks) else 0.0,
        "Top5": float(np.mean(ranks <= 5)) if len(ranks) else 0.0,
        "Top10": float(np.mean(ranks <= 10)) if len(ranks) else 0.0,
    }

    print("\n=== Held-out validation results ===")
    for key, value in metrics.items():
        print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

    output_path = DATA / "app_predictor_validation_diagnostic.csv"
    report.to_csv(output_path, index=False)
    print("\nSaved case-level diagnostic:", output_path)

    print("\nExample cases:")
    print(report.head(10).to_string(index=False))

Loading saved annotations and validation cases...
Validation cases: 906
Validation columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']

=== Held-out validation results ===
cases_in_validation: 906
evaluated_cases: 906
skipped_cases: 0
target_retrieval_rate: 1.0000
MRR: 0.3360
Top1: 0.2638
Top5: 0.4073
Top10: 0.4890

Saved case-level diagnostic: /content/drive/MyDrive/Rare Disease ai/processed/app_predictor_validation_diagnostic.csv

Example cases:
                                       case_id target_diagnoses  n_input_hpo  n_recognized_hpo  target_rank  target_retrieved
                       PMID_28830446_Patient_2      OMIM:615688           14                14           19              True
                       PMID_28830446_Patient_1      OMIM:615688           17                17           19              True
PMID_32480058_Case_F5P1_original_pedigree_II_2      OMIM:618052            5                 5          133              True
PMID_32480058_C

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": "string", "hpo_id": "string"},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

phenotype_map = (
    ann.groupby("database_id")["hpo_id"]
    .apply(set)
    .to_dict()
)

# Compute IC using training diseases only.
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"]
)

counts = defaultdict(int)
for disease in train_ids:
    for term in phenotype_map.get(disease, set()):
        counts[term] += 1

n_train = max(len(train_ids), 1)
ic = {
    term: -np.log(count / n_train)
    for term, count in counts.items()
    if count > 0
}

val = pd.read_csv(
    DATA / "patient_level_validation.csv",
    dtype=str
).fillna("")

def parse_ids(value):
    return {
        item.strip()
        for item in str(value).split(";")
        if item.strip()
    }

records = []

for _, row in val.iterrows():
    observed = parse_ids(row["hpo_ids"])
    targets = parse_ids(row["diagnosis_ids"])
    observed = observed & set(ic)

    if not observed or not targets:
        continue

    ic_weights = {h: ic[h] for h in observed}
    denominator = sum(ic_weights.values()) or 1.0

    scores = []
    for disease, terms in phenotype_map.items():
        overlap = observed & terms

        # Method 1: IC-weighted precision/recall F1
        precision = sum(ic_weights[h] for h in overlap) / denominator
        recall = len(overlap) / max(len(terms), 1)
        ic_f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        # Method 2: Jaccard similarity
        union = observed | terms
        jaccard = len(overlap) / len(union) if union else 0.0

        scores.append((disease, ic_f1, jaccard))

    score_df = pd.DataFrame(
        scores, columns=["disease_id", "ic_f1", "jaccard"]
    )

    for method in ["ic_f1", "jaccard"]:
        ranked = score_df.sort_values(
            [method, "disease_id"],
            ascending=[False, True]
        ).reset_index(drop=True)

        target_rows = ranked.index[
            ranked["disease_id"].isin(targets)
        ]

        rank = int(target_rows[0] + 1) if len(target_rows) else np.nan

        records.append({
            "case_id": row["case_id"],
            "method": method,
            "target_rank": rank,
            "mrr": 1 / rank if pd.notna(rank) else 0.0,
            "top1": int(pd.notna(rank) and rank <= 1),
            "top5": int(pd.notna(rank) and rank <= 5),
            "top10": int(pd.notna(rank) and rank <= 10),
        })

results = pd.DataFrame(records)
out_path = DATA / "phenotype_scoring_method_comparison.csv"
results.to_csv(out_path, index=False)

summary = (
    results.groupby("method")
    .agg(
        cases=("case_id", "nunique"),
        MRR=("mrr", "mean"),
        Top1=("top1", "mean"),
        Top5=("top5", "mean"),
        Top10=("top10", "mean"),
    )
    .reset_index()
)

print("=== Scoring method comparison ===")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nSaved:", out_path)


=== Scoring method comparison ===
 method  cases    MRR   Top1   Top5  Top10
  ic_f1    906 0.3360 0.2638 0.4073 0.4890
jaccard    906 0.2950 0.2285 0.3389 0.4437

Saved: /content/drive/MyDrive/Rare Disease ai/processed/phenotype_scoring_method_comparison.csv


In [ ]:
from pathlib import Path
import re

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
OBO_PATH = PROJECT / "hp.obo"

assert OBO_PATH.exists(), f"HPO ontology file not found: {OBO_PATH}"

terms = []
current = None

with open(OBO_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.rstrip()

        if line == "[Term]":
            if current and current.get("id") and current.get("name"):
                terms.append(current)
            current = {}
        elif line.startswith("["):
            if current and current.get("id") and current.get("name"):
                terms.append(current)
            current = None
        elif current is not None:
            if line.startswith("id: "):
                current["id"] = line[4:].strip()
            elif line.startswith("name: "):
                current["name"] = line[6:].strip()
            elif line.startswith("is_obsolete: true"):
                current["obsolete"] = True

if current and current.get("id") and current.get("name"):
    terms.append(current)

# Remove obsolete terms and keep only standard HPO identifiers.
terms = [
    t for t in terms
    if not t.get("obsolete", False)
    and re.fullmatch(r"HP:\d{7}", t["id"])
]

print("Active HPO terms parsed:", len(terms))
print("\nExample symptom-name mappings:")
for term in terms[:15]:
    print(f'{term["id"]}  |  {term["name"]}')

search_terms = [
    t for t in terms
    if any(word in t["name"].lower() for word in [
        "seizure", "developmental delay", "microcephaly",
        "hearing impairment", "muscle weakness"
    ])
]

print("\nExample searchable terms:")
for term in search_terms[:25]:
    print(f'{term["id"]}  |  {term["name"]}')

Active HPO terms parsed: 19894

Example symptom-name mappings:
HP:0000001  |  All
HP:0000002  |  Abnormality of body height
HP:0000003  |  Multicystic kidney dysplasia
HP:0000005  |  Mode of inheritance
HP:0000006  |  Autosomal dominant inheritance
HP:0000007  |  Autosomal recessive inheritance
HP:0000008  |  Abnormal morphology of female internal genitalia
HP:0000009  |  Functional abnormality of the bladder
HP:0000010  |  Recurrent urinary tract infections
HP:0000011  |  Neurogenic bladder
HP:0000012  |  Urinary urgency
HP:0000013  |  Hypoplasia of the uterus
HP:0000014  |  Abnormality of the bladder
HP:0000015  |  Bladder diverticulum
HP:0000016  |  Urinary retention

Example searchable terms:
HP:0000183  |  Tongue muscle weakness
HP:0000252  |  Microcephaly
HP:0000253  |  Progressive microcephaly
HP:0000365  |  Hearing impairment
HP:0000399  |  Prelingual sensorineural hearing impairment
HP:0000405  |  Conductive hearing impairment
HP:0000407  |  Sensorineural hearing impairment
HP

In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
OBO_PATH = PROJECT / "hp.obo"

terms = []
current = None

with open(OBO_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.rstrip()

        if line == "[Term]":
            if current and current.get("id") and current.get("name"):
                terms.append(current)
            current = {}
        elif line.startswith("["):
            if current and current.get("id") and current.get("name"):
                terms.append(current)
            current = None
        elif current is not None:
            if line.startswith("id: "):
                current["hpo_id"] = line[4:].strip()
            elif line.startswith("name: "):
                current["name"] = line[6:].strip()
            elif line.startswith("is_obsolete: true"):
                current["obsolete"] = True

if current and current.get("hpo_id") and current.get("name"):
    terms.append(current)

catalogue = pd.DataFrame(terms)
catalogue = catalogue[
    catalogue["hpo_id"].astype(str).str.fullmatch(r"HP:\d{7}")
    & ~catalogue.get(
        "obsolete", pd.Series(False, index=catalogue.index)
    ).fillna(False)
].copy()

catalogue = catalogue[["hpo_id", "name"]].drop_duplicates("hpo_id")
catalogue["search_name"] = catalogue["name"].str.lower()
catalogue = catalogue.sort_values("name").reset_index(drop=True)

output_path = DATA / "hpo_symptom_catalogue.csv"
catalogue.to_csv(output_path, index=False)

print("Catalogue saved:", output_path)
print("Unique symptom terms:", len(catalogue))
print("\nSearch examples:")
for query in ["seizure", "microcephaly", "hearing impairment",
              "developmental delay"]:
    matches = catalogue[
        catalogue["search_name"].str.contains(
            query, regex=False, na=False
        )
    ].head(5)
    print(f"\n{query}:")
    print(matches[["hpo_id", "name"]].to_string(index=False))

KeyError: 'hpo_id'

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
OBO_PATH = PROJECT / "hp.obo"

terms = []
current_id = None
current_name = None
obsolete = False

def save_term():
    if (
        current_id
        and current_name
        and current_id.startswith("HP:")
        and len(current_id) == 10
        and current_id[3:].isdigit()
        and not obsolete
    ):
        terms.append({
            "hpo_id": current_id,
            "name": current_name
        })

with open(OBO_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip()

        if line == "[Term]":
            save_term()
            current_id = None
            current_name = None
            obsolete = False

        elif line.startswith("["):
            save_term()
            current_id = None
            current_name = None
            obsolete = False

        elif line.startswith("id: "):
            current_id = line[4:].strip()

        elif line.startswith("name: "):
            current_name = line[6:].strip()

        elif line == "is_obsolete: true":
            obsolete = True

save_term()

catalogue = pd.DataFrame(terms, columns=["hpo_id", "name"])
catalogue = catalogue.drop_duplicates(subset="hpo_id")
catalogue["search_name"] = catalogue["name"].str.lower()
catalogue = catalogue.sort_values("name").reset_index(drop=True)

output_path = DATA / "hpo_symptom_catalogue.csv"
catalogue.to_csv(output_path, index=False)

print("Catalogue saved:", output_path)
print("Number of active HPO terms:", len(catalogue))
print("\nSample symptom matches:")

for query in [
    "seizure",
    "microcephaly",
    "hearing impairment",
    "developmental delay"
]:
    matches = catalogue[
        catalogue["search_name"].str.contains(
            query, regex=False, na=False
        )
    ].head(5)

    print(f"\n{query}:")
    print(matches[["hpo_id", "name"]].to_string(index=False))

Catalogue saved: /content/drive/MyDrive/Rare Disease ai/processed/hpo_symptom_catalogue.csv
Number of active HPO terms: 19894

Sample symptom matches:

seizure:
    hpo_id                                             name
HP:0011149            Absence seizure with eyelid myoclonia
HP:0010819                                   Atonic seizure
HP:0007270                         Atypical absence seizure
HP:0032837 Bilateral asymmetric neonatal sequential seizure
HP:0032839  Bilateral symmetric neonatal sequential seizure

microcephaly:
    hpo_id                     name
HP:0000252             Microcephaly
HP:0040196        Mild microcephaly
HP:0011451     Primary microcephaly
HP:0000253 Progressive microcephaly
HP:0005484   Secondary microcephaly

hearing impairment:
    hpo_id                                             name
HP:0008615     Adult onset sensorineural hearing impairment
HP:0008513          Bilateral conductive hearing impairment
HP:0008619       Bilateral sensorineural hearin

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
APP_PATH = MODEL_DIR / "streamlit_app.py"

app_code = r'''
import sys
from pathlib import Path

import pandas as pd
import streamlit as st

MODEL_DIR = Path(__file__).resolve().parent
PROJECT = MODEL_DIR.parent
DATA_DIR = PROJECT / "processed"

sys.path.insert(0, str(MODEL_DIR))
from rare_disease_ai import predict_diseases

st.set_page_config(
    page_title="Rare Disease Prediction AI",
    page_icon="🧬",
    layout="wide"
)

st.title("🧬 Rare Disease Prediction AI")
st.caption("Phenotype-based rare-disease candidate ranking")

st.warning(
    "Research prototype only. Results are candidate rankings, not "
    "diagnoses or probabilities. Consult qualified medical professionals."
)

@st.cache_data
def load_catalogue():
    path = DATA_DIR / "hpo_symptom_catalogue.csv"
    df = pd.read_csv(path, dtype=str).fillna("")
    return df

catalogue = load_catalogue()
name_to_id = dict(zip(catalogue["name"], catalogue["hpo_id"]))

st.markdown("### 1. Select patient symptoms")
st.write("Search by symptom name and select all applicable terms.")

search_text = st.text_input(
    "Search symptom names",
    placeholder="Try seizure, hearing impairment, or microcephaly"
)

if search_text.strip():
    matches = catalogue[
        catalogue["name"].str.contains(
            search_text.strip(), case=False, regex=False, na=False
        )
    ].head(100)

    if matches.empty:
        st.info("No matching symptom names found. Try another search.")
        options = []
    else:
        options = [
            f'{row["name"]} ({row["hpo_id"]})'
            for _, row in matches.iterrows()
        ]
else:
    options = []

selected_display = st.multiselect(
    "Selected symptoms",
    options=options,
    help="Search for a symptom above, then select it here."
)

st.markdown("### 2. Or enter HPO IDs manually")
manual_input = st.text_area(
    "HPO IDs (optional)",
    placeholder="HP:0001250\nHP:0001263",
    help="Enter one ID per line, or separate IDs with commas or semicolons."
)

gene_input = st.text_input(
    "Optional gene symbols",
    placeholder="SCN1A, MECP2"
)

top_k = st.slider("Number of candidates", 5, 20, 10)

if st.button("Rank candidate diseases", type="primary"):
    selected_ids = [
        item.rsplit("(", 1)[-1].rstrip(")")
        for item in selected_display
    ]

    manual_ids = [
        item.strip()
        for item in manual_input.replace(",", "\n")
        .replace(";", "\n").splitlines()
        if item.strip()
    ]

    input_ids = list(dict.fromkeys(selected_ids + manual_ids))

    if not input_ids:
        st.error("Select at least one symptom or enter an HPO ID.")
    else:
        try:
            # The predictor ignores unrecognized IDs. Report them clearly.
            from rare_disease_data_loader import load_prediction_data
            data = load_prediction_data()
            recognized = [term for term in input_ids if term in data["ic"]]
            unknown = [term for term in input_ids if term not in data["ic"]]

            st.write(
                f"Recognized {len(recognized)} of {len(input_ids)} "
                "entered HPO IDs."
            )
            if unknown:
                st.warning("Unrecognized HPO IDs: " + ", ".join(unknown))
            if not recognized:
                st.error(
                    "None of the entered symptoms are recognized by the "
                    "current predictor's annotation vocabulary."
                )
                st.stop()

            genes = [
                item.strip().upper()
                for item in gene_input.replace(",", "\n")
                .replace(";", "\n").splitlines()
                if item.strip()
            ]

            with st.spinner("Ranking candidate diseases..."):
                results = predict_diseases(
                    patient_hpo_terms=recognized,
                    patient_genes=genes or None,
                    top_k=top_k
                )

            st.markdown("### 3. Ranked candidate diseases")
            st.caption(
                "Ranking scores are relative similarity scores, not "
                "probabilities of having a disease."
            )

            st.dataframe(results, use_container_width=True, hide_index=True)

            st.markdown("### 4. Why each candidate was ranked")
            for _, row in results.iterrows():
                with st.expander(
                    f'#{int(row["rank"])} — {row["disease_name"]} '
                    f'({row["disease_id"]})'
                ):
                    matched = [
                        x for x in str(
                            row.get("matching_phenotype_ids", "")
                        ).split(";") if x
                    ]
                    st.write(
                        f'**Phenotype score:** '
                        f'{row["phenotype_score"]:.4f}'
                    )
                    st.write(
                        f'**Matched input symptoms:** '
                        f'{len(matched)} of {len(recognized)}'
                    )
                    st.write(
                        "**Matching HPO IDs:** "
                        + (", ".join(matched) if matched else "None")
                    )
                    if row.get("matching_genes"):
                        st.write("**Matching genes:** " + row["matching_genes"])

            st.download_button(
                "Download results as CSV",
                results.to_csv(index=False).encode("utf-8"),
                file_name="rare_disease_ranked_candidates.csv",
                mime="text/csv"
            )

        except Exception as exc:
            st.error("Prediction failed.")
            st.exception(exc)

st.divider()
st.caption(
    "BTech project research prototype. This tool does not provide "
    "medical diagnoses and has not been clinically validated."
)
'''

APP_PATH.write_text(app_code, encoding="utf-8")
compile(app_code, str(APP_PATH), "exec")

print("Updated app:", APP_PATH)
print("Python syntax: PASS")
print("Symptom catalogue:", (PROJECT / "processed" / "hpo_symptom_catalogue.csv").exists())

Updated app: /content/drive/MyDrive/Rare Disease ai/model/streamlit_app.py
Python syntax: PASS
Symptom catalogue: True


In [ ]:
from pathlib import Path
import subprocess

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
LOG_PATH = PROJECT / "streamlit.log"

print("=== Recent Streamlit logs ===")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("\n".join(lines[-30:]))
else:
    print("Log file not found.")

=== Recent Streamlit logs ===
2026-10-18 05:21:10.153 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.185.15.22:8501

2026-10-18 05:23:01.180 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


In [ ]:
from pathlib import Path
import requests

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
APP_PATH = PROJECT / "model" / "streamlit_app.py"

print("Updated app contains symptom search:",
      "Search symptom names" in APP_PATH.read_text(encoding="utf-8"))

try:
    response = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=10)
    print("Streamlit health status:", response.status_code)
    print("Response:", response.text)
except Exception as exc:
    print("Health check failed:", repr(exc))

Updated app contains symptom search: True
Streamlit health status: 200
Response: ok


In [ ]:
import requests
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
APP_PATH = PROJECT / "model" / "streamlit_app.py"

print("Symptom search in app:", "Search symptom names" in APP_PATH.read_text())
print("Server health:", requests.get(
    "http://127.0.0.1:8501/_stcore/health", timeout=10
).status_code)
print("Preview helper available:", hasattr(
    __import__("google.colab").colab.output,
    "serve_kernel_port_as_iframe"
))

Symptom search in app: True
Server health: 200
Preview helper available: True


In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=850)

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import py_compile

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
APP_PATH = PROJECT / "model" / "streamlit_app.py"

source = APP_PATH.read_text(encoding="utf-8")

# Update deprecated Streamlit dataframe arguments
source = source.replace(
    "use_container_width=True",
    'width="stretch"'
)

APP_PATH.write_text(source, encoding="utf-8")
py_compile.compile(str(APP_PATH), doraise=True)

checks = {
    "Symptom-name search": "Search symptom names" in source,
    "Multiple symptom selection": "multiselect" in source,
    "Disease prediction": "predict_diseases" in source,
    "CSV export": "download_button" in source,
    "Explanations": "expander" in source,
}

print("APP CODE CHECK")
for feature, passed in checks.items():
    print(f"{'PASS' if passed else 'CHECK'} — {feature}")

print("\nPython syntax: PASS")
print("App file:", APP_PATH)
print("All required features present:", all(checks.values()))

APP CODE CHECK
PASS — Symptom-name search
PASS — Multiple symptom selection
PASS — Disease prediction
PASS — CSV export
PASS — Explanations

Python syntax: PASS
App file: /content/drive/MyDrive/Rare Disease ai/model/streamlit_app.py
All required features present: True


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

# Check key validation and evaluation files
files_to_check = [
    DATA / "patient_level_validation.csv",
    DATA / "app_predictor_validation_diagnostic.csv",
    DATA / "phenotype_scoring_method_comparison.csv",
    DATA / "hpo_symptom_catalogue.csv",
    MODEL_DIR / "rare_disease_ai.py",
    MODEL_DIR / "rare_disease_data_loader.py",
]

print("PROJECT AUDIT")
for path in files_to_check:
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path.name}")

validation_path = DATA / "patient_level_validation.csv"

if validation_path.exists():
    df = pd.read_csv(validation_path)
    print("\nValidation dataset shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Rows with missing case IDs:", df["case_id"].isna().sum())
    print("Unique cases:", df["case_id"].nunique())

    if "n_hpo" in df.columns:
        print("HPO terms per case:")
        print(df["n_hpo"].describe().round(2))

    if "n_diagnoses" in df.columns:
        print("\nDiagnoses per case:")
        print(df["n_diagnoses"].describe().round(2))

PROJECT AUDIT
FOUND: patient_level_validation.csv
FOUND: app_predictor_validation_diagnostic.csv
FOUND: phenotype_scoring_method_comparison.csv
FOUND: hpo_symptom_catalogue.csv
FOUND: rare_disease_ai.py
FOUND: rare_disease_data_loader.py

Validation dataset shape: (906, 6)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']
Rows with missing case IDs: 0
Unique cases: 906
HPO terms per case:
count    906.00
mean       8.67
std        6.31
min        1.00
25%        4.00
50%        7.00
75%       11.00
max       51.00
Name: n_hpo, dtype: float64

Diagnoses per case:
count    906.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: n_diagnoses, dtype: float64


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
APP_PATH = PROJECT / "model" / "streamlit_app.py"
PREDICTOR_PATH = PROJECT / "model" / "rare_disease_ai.py"

app_source = APP_PATH.read_text(encoding="utf-8")
predictor_source = PREDICTOR_PATH.read_text(encoding="utf-8")

def show_matches(label, source, terms, context=3):
    print(f"\n{'=' * 18} {label} {'=' * 18}")
    lines = source.splitlines()
    matched = set()

    for i, line in enumerate(lines):
        if any(term.lower() in line.lower() for term in terms):
            for j in range(max(0, i-context), min(len(lines), i+context+1)):
                matched.add(j)

    for j in sorted(matched):
        print(f"{j+1:4}: {lines[j]}")

show_matches(
    "APP RESULT / EXPLANATION LOGIC",
    app_source,
    ["predict_diseases", "explain", "expander", "score", "phenotype", "diagnosis"],
)

show_matches(
    "PREDICTOR OUTPUT / SCORING LOGIC",
    predictor_source,
    ["def predict_diseases", "return", "score", "phenotype", "gene_weight"],
)



================== APP RESULT / EXPLANATION LOGIC ==================
  10: DATA_DIR = PROJECT / "processed"
  11: 
  12: sys.path.insert(0, str(MODEL_DIR))
  13: from rare_disease_ai import predict_diseases
  14: 
  15: st.set_page_config(
  16:     page_title="Rare Disease Prediction AI",
  19: )
  20: 
  21: st.title("🧬 Rare Disease Prediction AI")
  22: st.caption("Phenotype-based rare-disease candidate ranking")
  23: 
  24: st.warning(
  25:     "Research prototype only. Results are candidate rankings, not "
 127:             ]
 128: 
 129:             with st.spinner("Ranking candidate diseases..."):
 130:                 results = predict_diseases(
 131:                     patient_hpo_terms=recognized,
 132:                     patient_genes=genes or None,
 133:                     top_k=top_k
 135: 
 136:             st.markdown("### 3. Ranked candidate diseases")
 137:             st.caption(
 138:                 "Ranking scores are relative similarity scores, not "
 139:  

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
APP_PATH = PROJECT / "model" / "streamlit_app.py"
CATALOGUE_PATH = PROJECT / "processed" / "hpo_symptom_catalogue.csv"

# Show the current explanation section
lines = APP_PATH.read_text(encoding="utf-8").splitlines()
print("=== CURRENT EXPLANATION SECTION ===")
for i, line in enumerate(lines[140:180], start=141):
    print(f"{i:3}: {line}")

# Inspect the symptom-name lookup table
catalogue = pd.read_csv(CATALOGUE_PATH)
print("\n=== HPO CATALOGUE ===")
print("Columns:", catalogue.columns.tolist())
print(catalogue[["hpo_id", "name"]].head(8).to_string(index=False))
print("Unique HPO IDs:", catalogue["hpo_id"].nunique())

=== CURRENT EXPLANATION SECTION ===
141: 
142:             st.dataframe(results, width="stretch", hide_index=True)
143: 
144:             st.markdown("### 4. Why each candidate was ranked")
145:             for _, row in results.iterrows():
146:                 with st.expander(
147:                     f'#{int(row["rank"])} — {row["disease_name"]} '
148:                     f'({row["disease_id"]})'
149:                 ):
150:                     matched = [
151:                         x for x in str(
152:                             row.get("matching_phenotype_ids", "")
153:                         ).split(";") if x
154:                     ]
155:                     st.write(
156:                         f'**Phenotype score:** '
157:                         f'{row["phenotype_score"]:.4f}'
158:                     )
159:                     st.write(
160:                         f'**Matched input symptoms:** '
161:                         f'{len(matched)} of {len(recognized)}'
162: 

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

print("Python runtime:", sys.version.split()[0])
print("GPU available:", torch.cuda.is_available())
print("Google Drive project exists:", PROJECT.exists())
print("App file exists:", (PROJECT / "model" / "streamlit_app.py").exists())
print("Symptom catalogue exists:",
      (PROJECT / "processed" / "hpo_symptom_catalogue.csv").exists())
print("Predictor checkpoint files:")
for p in sorted((PROJECT / "model").glob("*.pt")):
    print(" -", p.name)

Python runtime: 3.13.16
GPU available: False
Google Drive project exists: True
App file exists: True
Symptom catalogue exists: True
Predictor checkpoint files:
 - patient_level_ranker_best.pt
 - patient_level_ranker_bias_free_experiment.pt
 - patient_ranker_bias_softmax_experiment.pt
 - rare_disease_gnn_filtered_best.pt
 - rare_disease_gnn_publication_disjoint_experiment.pt


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import py_compile

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
APP_PATH = PROJECT / "model" / "streamlit_app.py"

source = APP_PATH.read_text(encoding="utf-8")

print("=== APP CHECK ===")
print("Symptom search:", "Search symptom names" in source)
print("Prediction function:", "predict_diseases" in source)
print("Explanation section:", "Matching symptoms:" in source)
print("Catalogue loader defined:", "def load_symptom_catalogue" in source)

try:
    py_compile.compile(str(APP_PATH), doraise=True)
    print("Python syntax: PASS")
except Exception as exc:
    print("Python syntax: FAIL")
    print(exc)

print("\n=== CHECKPOINTS ===")
for path in sorted((PROJECT / "model").glob("*.pt")):
    print(f"{path.name}: {path.stat().st_size / (1024**2):.1f} MB")

=== APP CHECK ===
Symptom search: True
Prediction function: True
Explanation section: False
Catalogue loader defined: False
Python syntax: PASS

=== CHECKPOINTS ===
patient_level_ranker_best.pt: 9.0 MB
patient_level_ranker_bias_free_experiment.pt: 8.9 MB
patient_ranker_bias_softmax_experiment.pt: 9.0 MB
rare_disease_gnn_filtered_best.pt: 4.6 MB
rare_disease_gnn_publication_disjoint_experiment.pt: 4.4 MB


In [ ]:
from pathlib import Path
import py_compile

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
APP_PATH = PROJECT / "model" / "streamlit_app.py"
source = APP_PATH.read_text(encoding="utf-8")

# 1. Add the catalogue loader if it is missing.
if "def load_symptom_catalogue" not in source:
    import_line = "import streamlit as st"
    loader = '''

@st.cache_data
def load_symptom_catalogue():
    """Load HPO identifiers and human-readable phenotype names."""
    import pandas as pd
    catalogue_path = DATA_DIR / "hpo_symptom_catalogue.csv"
    catalogue = pd.read_csv(catalogue_path)
    return catalogue.drop_duplicates("hpo_id")
'''
    if import_line not in source:
        raise RuntimeError("Could not locate the Streamlit import; no changes saved.")
    source = source.replace(import_line, import_line + loader, 1)

# 2. Replace the old HPO-ID-only explanation with readable names.
old = '''                    st.write(
                        "**Matching HPO IDs:** "
                        + (", ".join(matched) if matched else "None")
                    )
'''

new = '''                    catalogue = load_symptom_catalogue()
                    name_lookup = dict(
                        zip(catalogue["hpo_id"], catalogue["name"])
                    )
                    st.write("**Matching symptoms:**")
                    if matched:
                        for hpo_id in matched:
                            symptom_name = name_lookup.get(
                                hpo_id, "Name unavailable"
                            )
                            st.markdown(f"- {symptom_name} ({hpo_id})")
                    else:
                        st.write("No matching input phenotypes.")
'''

if old not in source:
    if "Matching symptoms:" not in source:
        raise RuntimeError(
            "The expected old explanation block was not found. "
            "No file changes were written; inspect the current explanation block."
        )

# Only replace when the old block is present.
if old in source:
    source = source.replace(old, new, 1)

# Save backup before writing.
backup_path = APP_PATH.with_suffix(".py.day3_backup")
backup_path.write_text(APP_PATH.read_text(encoding="utf-8"), encoding="utf-8")
APP_PATH.write_text(source, encoding="utf-8")
py_compile.compile(str(APP_PATH), doraise=True)

print("Catalogue loader:", "PASS" if "def load_symptom_catalogue" in source else "FAIL")
print("Readable symptom explanations:", "PASS" if "Matching symptoms:" in source else "FAIL")
print("Python syntax: PASS")
print("Backup saved:", backup_path.name)

Catalogue loader: PASS
Readable symptom explanations: PASS
Python syntax: PASS
Backup saved: streamlit_app.py.day3_backup


In [ ]:
from pathlib import Path
import pandas as pd
import sys

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

sys.path.insert(0, str(MODEL_DIR))

catalogue = pd.read_csv(DATA / "hpo_symptom_catalogue.csv")
name_lookup = dict(zip(catalogue["hpo_id"], catalogue["name"]))

test_ids = [
    "HP:0001250",  # Seizure
    "HP:0001263",  # Global developmental delay
    "HP:0000252",  # Microcephaly
]

print("=== HPO NAME LOOKUP ===")
for hpo_id in test_ids:
    print(f"{hpo_id}: {name_lookup.get(hpo_id, 'NOT FOUND')}")

print("\n=== PREDICTION SMOKE TEST ===")
from rare_disease_ai import predict_diseases

results = predict_diseases(
    patient_hpo_terms=test_ids,
    top_k=5
)

print("Result rows:", len(results))
print("Result columns:", results.columns.tolist())
print(results[[
    "rank", "disease_id", "disease_name",
    "phenotype_score", "matching_phenotypes",
    "matching_phenotype_ids"
]].to_string(index=False))

=== HPO NAME LOOKUP ===
HP:0001250: Seizure
HP:0001263: Global developmental delay
HP:0000252: Microcephaly

=== PREDICTION SMOKE TEST ===
Result rows: 5
Result columns: ['rank', 'disease_id', 'disease_name', 'phenotype_score', 'gene_score', 'combined_score', 'matching_phenotypes', 'matching_phenotype_ids', 'matching_genes']
 rank  disease_id                                         disease_name  phenotype_score  matching_phenotypes matching_phenotype_ids
    1   ORPHA:226                Dihydropteridine reductase deficiency         0.580827                    2  HP:0000252;HP:0001263
    2  ORPHA:2798 Pachygyria-intellectual disability-epilepsy syndrome         0.546296                    2  HP:0001250;HP:0001263
    3  DECIPHER:1                             Wolf-Hirschhorn syndrome         0.510443                    2  HP:0000252;HP:0001250
    4  DECIPHER:4                           Angelman syndrome (Type 1)         0.510443                    2  HP:0000252;HP:0001250
    5 DECIPHE

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
DATA_DIR = PROJECT / "processed"

print("=== ENVIRONMENT CHECK ===")
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

print("\n=== PROJECT CHECK ===")
print("Project exists:", PROJECT.exists())
print("App exists:", (MODEL_DIR / "streamlit_app.py").exists())
print("Predictor exists:", (MODEL_DIR / "rare_disease_ai.py").exists())
print("Validation data exists:",
      (DATA_DIR / "patient_level_validation.csv").exists())
print("HPO catalogue exists:",
      (DATA_DIR / "hpo_symptom_catalogue.csv").exists())

print("\n=== SAVED MODEL CHECKPOINTS ===")
if MODEL_DIR.exists():
    for file in sorted(MODEL_DIR.glob("*.pt")):
        print(f"{file.name} — {file.stat().st_size / (1024**2):.1f} MB")

=== ENVIRONMENT CHECK ===
GPU available: True
GPU: Tesla T4
CUDA version: 13.0

=== PROJECT CHECK ===
Project exists: True
App exists: True
Predictor exists: True
Validation data exists: True
HPO catalogue exists: True

=== SAVED MODEL CHECKPOINTS ===
patient_level_ranker_best.pt — 9.0 MB
patient_level_ranker_bias_free_experiment.pt — 8.9 MB
patient_ranker_bias_softmax_experiment.pt — 9.0 MB
rare_disease_gnn_filtered_best.pt — 4.6 MB
rare_disease_gnn_publication_disjoint_experiment.pt — 4.4 MB


In [ ]:
from pathlib import Path
import pandas as pd
import py_compile

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
DATA_DIR = PROJECT / "processed"
APP_PATH = MODEL_DIR / "streamlit_app.py"

# Check syntax and saved explanation code
py_compile.compile(str(APP_PATH), doraise=True)
source = APP_PATH.read_text(encoding="utf-8")

print("=== APP CHECK ===")
print("Python syntax: PASS")
print("Symptom search:", "Search symptom names" in source)
print("Catalogue loader:", "def load_symptom_catalogue" in source)
print("Readable explanations:", "Matching symptoms:" in source)

# Test HPO IDs against the saved catalogue
catalogue = pd.read_csv(DATA_DIR / "hpo_symptom_catalogue.csv")
lookup = dict(zip(catalogue["hpo_id"], catalogue["name"]))

print("\n=== HPO NAME LOOKUP ===")
for hpo_id in ["HP:0001250", "HP:0001263", "HP:0000252"]:
    print(f"{hpo_id}: {lookup.get(hpo_id, 'NOT FOUND')}")

print("\nCatalogue entries:", len(catalogue))
print("Unique HPO IDs:", catalogue["hpo_id"].nunique())

=== APP CHECK ===
Python syntax: PASS
Symptom search: True
Catalogue loader: True
Readable explanations: True

=== HPO NAME LOOKUP ===
HP:0001250: Seizure
HP:0001263: Global developmental delay
HP:0000252: Microcephaly

Catalogue entries: 19894
Unique HPO IDs: 19894


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

sys.path.insert(0, str(MODEL_DIR))

from rare_disease_ai import predict_diseases

test_hpo_ids = [
    "HP:0001250",  # Seizure
    "HP:0001263",  # Global developmental delay
    "HP:0000252",  # Microcephaly
]

results = predict_diseases(
    patient_hpo_terms=test_hpo_ids,
    top_k=5
)

print("=== PREDICTION PIPELINE TEST ===")
print("Input HPO terms:", test_hpo_ids)
print("Number of candidates:", len(results))
print("Columns:", results.columns.tolist())

display(results[[
    "rank",
    "disease_id",
    "disease_name",
    "phenotype_score",
    "matching_phenotypes",
    "matching_phenotype_ids"
]])

# Translate matches into readable names
catalogue = pd.read_csv(PROJECT / "processed" / "hpo_symptom_catalogue.csv")
name_lookup = dict(zip(catalogue["hpo_id"], catalogue["name"]))

print("\n=== TOP CANDIDATE EXPLANATION ===")
if not results.empty:
    top = results.iloc[0]
    print("Candidate:", top["disease_name"], f'({top["disease_id"]})')
    print("Phenotype similarity score:", round(float(top["phenotype_score"]), 4))

    matched = [
        hpo for hpo in str(top["matching_phenotype_ids"]).split(";") if hpo
    ]
    print("Matched phenotype names:")
    if matched:
        for hpo in matched:
            print(f"- {name_lookup.get(hpo, 'Name unavailable')} ({hpo})")
    else:
        print("- No overlapping input phenotypes")

print("\nNote: this is a software smoke test, not a clinical diagnosis.")

=== PREDICTION PIPELINE TEST ===
Input HPO terms: ['HP:0001250', 'HP:0001263', 'HP:0000252']
Number of candidates: 5
Columns: ['rank', 'disease_id', 'disease_name', 'phenotype_score', 'gene_score', 'combined_score', 'matching_phenotypes', 'matching_phenotype_ids', 'matching_genes']


,rank,disease_id,disease_name,phenotype_score,matching_phenotypes,matching_phenotype_ids
0,1,ORPHA:226,Dihydropteridine reductase deficiency,0.580827,2,HP:0000252;HP:0001263
1,2,ORPHA:2798,Pachygyria-intellectual disability-epilepsy sy...,0.546296,2,HP:0001250;HP:0001263
2,3,DECIPHER:1,Wolf-Hirschhorn syndrome,0.510443,2,HP:0000252;HP:0001250
3,4,DECIPHER:4,Angelman syndrome (Type 1),0.510443,2,HP:0000252;HP:0001250
4,5,DECIPHER:54,Angelman syndrome (Type 2),0.510443,2,HP:0000252;HP:0001250



=== TOP CANDIDATE EXPLANATION ===
Candidate: Dihydropteridine reductase deficiency (ORPHA:226)
Phenotype similarity score: 0.5808
Matched phenotype names:
- Microcephaly (HP:0000252)
- Global developmental delay (HP:0001263)

Note: this is a software smoke test, not a clinical diagnosis.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = {
    "App predictor validation": DATA / "app_predictor_validation_diagnostic.csv",
    "Scoring method comparison": DATA / "phenotype_scoring_method_comparison.csv",
}

for label, path in files.items():
    print(f"\n{'=' * 12} {label} {'=' * 12}")

    if not path.exists():
        print("File not found:", path.name)
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(10))

    # Show numeric summaries where available
    numeric = df.select_dtypes(include="number")
    if not numeric.empty:
        print("Numeric summary:")
        display(numeric.describe().round(4))


============ App predictor validation ============
Shape: (906, 6)
Columns: ['case_id', 'target_diagnoses', 'n_input_hpo', 'n_recognized_hpo', 'target_rank', 'target_retrieved']


,case_id,target_diagnoses,n_input_hpo,n_recognized_hpo,target_rank,target_retrieved
0,PMID_28830446_Patient_2,OMIM:615688,14,14,19,True
1,PMID_28830446_Patient_1,OMIM:615688,17,17,19,True
2,PMID_32480058_Case_F5P1_original_pedigree_II_2,OMIM:618052,5,5,133,True
3,PMID_32480058_Case_F4P2_original_pedigree_IV_5,OMIM:618052,8,8,13,True
4,PMID_32480058_Case_F10P2,OMIM:618052,5,5,108,True
5,PMID_32480058_Case_F6P1_original_pedigree_II_1,OMIM:618052,16,16,1,True
6,PMID_32480058_Case_F7P3,OMIM:618052,11,11,1,True
7,PMID_32480058_Case_F7P2,OMIM:618052,19,19,1,True
8,PMID_32480058_Case_F8P1,OMIM:618052,6,6,14,True
9,PMID_32480058_Case_F4P1_original_pedigree_IV_1,OMIM:618052,6,6,60,True


Numeric summary:


,n_input_hpo,n_recognized_hpo,target_rank
count,906.0000,906.0000,906.0000
mean,8.6656,8.6656,152.8962
std,6.3086,6.3086,659.7678
min,1.0000,1.0000,1.0000
25%,4.0000,4.0000,1.0000
50%,7.0000,7.0000,12.0000
75%,11.0000,11.0000,54.0000
max,51.0000,51.0000,6927.0000



============ Scoring method comparison ============
Shape: (1812, 7)
Columns: ['case_id', 'method', 'target_rank', 'mrr', 'top1', 'top5', 'top10']


,case_id,method,target_rank,mrr,top1,top5,top10
0,PMID_28830446_Patient_2,ic_f1,19,0.052632,0,0,0
1,PMID_28830446_Patient_2,jaccard,22,0.045455,0,0,0
2,PMID_28830446_Patient_1,ic_f1,19,0.052632,0,0,0
3,PMID_28830446_Patient_1,jaccard,26,0.038462,0,0,0
4,PMID_32480058_Case_F5P1_original_pedigree_II_2,ic_f1,133,0.007519,0,0,0
5,PMID_32480058_Case_F5P1_original_pedigree_II_2,jaccard,167,0.005988,0,0,0
6,PMID_32480058_Case_F4P2_original_pedigree_IV_5,ic_f1,13,0.076923,0,0,0
7,PMID_32480058_Case_F4P2_original_pedigree_IV_5,jaccard,20,0.050000,0,0,0
8,PMID_32480058_Case_F10P2,ic_f1,108,0.009259,0,0,0
9,PMID_32480058_Case_F10P2,jaccard,301,0.003322,0,0,0


Numeric summary:


,target_rank,mrr,top1,top5,top10
count,1812.0000,1812.0000,1812.0000,1812.0000,1812.0000
mean,169.9299,0.3155,0.2461,0.3731,0.4663
std,672.9860,0.4074,0.4309,0.4838,0.4990
min,1.0000,0.0001,0.0000,0.0000,0.0000
25%,2.0000,0.0175,0.0000,0.0000,0.0000
50%,14.5000,0.0690,0.0000,0.0000,0.0000
75%,57.2500,0.5000,0.0000,1.0000,1.0000
max,6927.0000,1.0000,1.0000,1.0000,1.0000


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

path = DATA / "phenotype_scoring_method_comparison.csv"
df = pd.read_csv(path)

print("Cases per scoring method:")
print(df.groupby("method")["case_id"].nunique())

print("\n=== METRICS BY METHOD ===")
summary = df.groupby("method").agg(
    cases=("case_id", "nunique"),
    MRR=("mrr", "mean"),
    Top1=("top1", "mean"),
    Top5=("top5", "mean"),
    Top10=("top10", "mean"),
    median_target_rank=("target_rank", "median"),
    mean_target_rank=("target_rank", "mean"),
).round(4)

display(summary)

print("\n=== APP PREDICTOR RANK DISTRIBUTION ===")
app_path = DATA / "app_predictor_validation_diagnostic.csv"
app = pd.read_csv(app_path)

print("Cases:", len(app))
print("Target rank percentiles:")
print(app["target_rank"].quantile([0, .25, .5, .75, .9, .95, 1]).to_string())

for k in [1, 5, 10, 20, 50]:
    print(f"Top-{k} target retrieval: {(app['target_rank'] <= k).mean():.4f}")

Cases per scoring method:
method
ic_f1      906
jaccard    906
Name: case_id, dtype: int64

=== METRICS BY METHOD ===


,cases,MRR,Top1,Top5,Top10,median_target_rank,mean_target_rank
method,,,,,,,
ic_f1,906,0.336,0.2638,0.4073,0.4890,12.0,152.8962
jaccard,906,0.295,0.2285,0.3389,0.4437,18.0,186.9636



=== APP PREDICTOR RANK DISTRIBUTION ===
Cases: 906
Target rank percentiles:
0.00       1.0
0.25       1.0
0.50      12.0
0.75      54.0
0.90     224.5
0.95     605.0
1.00    6927.0
Top-1 target retrieval: 0.2638
Top-5 target retrieval: 0.4073
Top-10 target retrieval: 0.4890
Top-20 target retrieval: 0.5850
Top-50 target retrieval: 0.7417


In [ ]:
from pathlib import Path
import pandas as pd
import sys

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

sys.path.insert(0, str(MODEL_DIR))

from rare_disease_data_loader import load_prediction_data

validation = pd.read_csv(DATA / "patient_level_validation.csv")
diagnostic = pd.read_csv(
    DATA / "app_predictor_validation_diagnostic.csv"
)

data = load_prediction_data()
phenotype_map = data["phenotype_map"]

# Join validation inputs to their recorded target ranks.
df = validation.merge(
    diagnostic[["case_id", "target_rank", "target_retrieved"]],
    on="case_id",
    how="inner"
)

rows = []
for _, case in df.nlargest(20, "target_rank").iterrows():
    target_ids = [
        x for x in str(case["diagnosis_ids"]).split(";") if x
    ]
    input_hpo = set(
        x for x in str(case["hpo_ids"]).split(";") if x
    )

    target_terms = set()
    for disease_id in target_ids:
        target_terms |= phenotype_map.get(disease_id, set())

    rows.append({
        "case_id": case["case_id"],
        "target_diagnosis": ";".join(target_ids),
        "target_rank": int(case["target_rank"]),
        "input_hpo_count": len(input_hpo),
        "target_phenotype_count": len(target_terms),
        "input_target_overlap": len(input_hpo & target_terms),
        "target_has_phenotype_map": any(
            disease_id in phenotype_map for disease_id in target_ids
        ),
    })

hard_cases = pd.DataFrame(rows)
display(hard_cases)

print("\n=== DIAGNOSTIC SUMMARY ===")
print("Hard cases with target phenotype data:",
      hard_cases["target_has_phenotype_map"].mean())
print("Mean input/target phenotype overlap:",
      round(hard_cases["input_target_overlap"].mean(), 2))

,case_id,target_diagnosis,target_rank,input_hpo_count,target_phenotype_count,input_target_overlap,target_has_phenotype_map
0,PMID_31057532_Family_G_individual_G_II_2,OMIM:616873,6927,6,16,0,True
1,PMID_31057532_Family_A_individual_A_II_1,OMIM:616873,6652,4,16,0,True
2,PMID_31057532_Family_C_individual_C_II_1,OMIM:616873,6652,5,16,0,True
3,PMID_31057532_Family_H_individual_H_II_1,OMIM:616873,6590,2,16,0,True
4,PMID_31057532_Family_I_individual_I_II_2,OMIM:616873,6419,1,16,0,True
5,PMID_31057532_Family_I_individual_I_II_1,OMIM:616873,6419,2,16,0,True
6,PMID_27939403_D_I_1,OMIM:616873,6415,3,16,0,True
7,PMID_27761245_BRC081,OMIM:604804,5624,3,17,0,True
8,PMID_30099644_IV_12,OMIM:600138,3220,1,13,0,True
9,PMID_23714749_Family_1_patient_1,OMIM:256810,2235,5,190,5,True



=== DIAGNOSTIC SUMMARY ===
Hard cases with target phenotype data: 1.0
Mean input/target phenotype overlap: 3.0


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

case_id = "PMID_31057532_Family_G_individual_G_II_2"
target_id = "OMIM:616873"

validation = pd.read_csv(DATA / "patient_level_validation.csv")
cases = validation[validation["case_id"] == case_id]

print("=== VALIDATION CASE ===")
if cases.empty:
    print("Case not found.")
else:
    case = cases.iloc[0]
    patient_hpo = set(
        x for x in str(case["hpo_ids"]).split(";") if x
    )
    print("Case:", case_id)
    print("Target:", case["diagnosis_ids"])
    print("Patient HPO IDs:", sorted(patient_hpo))

annotations_path = DATA / "clean_hpo_annotations.csv"
annotations = pd.read_csv(annotations_path)

print("\n=== ANNOTATION SCHEMA ===")
print("Columns:", annotations.columns.tolist())
print("Rows:", len(annotations))
display(annotations.head(5))

mapping_path = DATA / "normalized_disease_gene_mapping.csv"
mapping = pd.read_csv(mapping_path)

print("\n=== DISEASE MAPPING SCHEMA ===")
print("Columns:", mapping.columns.tolist())
print("Rows:", len(mapping))
display(mapping.head(5))

=== VALIDATION CASE ===
Case: PMID_31057532_Family_G_individual_G_II_2
Target: OMIM:616873
Patient HPO IDs: ['HP:0001744', 'HP:0002028', 'HP:0002205', 'HP:0002720', 'HP:0004315', 'HP:0004812']

=== ANNOTATION SCHEMA ===
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']
Rows: 286047


/tmp/ipykernel_1445/2840130701.py:26: DtypeWarning: Columns (2,6,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations = pd.read_csv(annotations_path)


,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
3,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0032792,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
4,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011451,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]



=== DISEASE MAPPING SCHEMA ===
Columns: ['disease_id', 'gene_symbols']
Rows: 11168


,disease_id,gene_symbols
0,OMIM:100100,CHRM3
1,OMIM:100300,ARHGAP31
2,OMIM:100800,FGFR3
3,OMIM:101000,NF2
4,OMIM:101200,FGFR2


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

target_id = "OMIM:616873"
case_id = "PMID_31057532_Family_G_individual_G_II_2"

validation = pd.read_csv(DATA / "patient_level_validation.csv")
case = validation.loc[validation["case_id"] == case_id].iloc[0]

patient_hpo = set(
    term for term in str(case["hpo_ids"]).split(";") if term
)

annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    usecols=["database_id", "disease_name", "hpo_id", "evidence", "reference"]
)

target_rows = annotations[
    annotations["database_id"].astype(str).str.strip() == target_id
].copy()

target_hpo = set(target_rows["hpo_id"].dropna().astype(str).str.strip())

print("=== TARGET ANNOTATION AUDIT ===")
print("Target disease:", target_id)
print("Target disease names:", target_rows["disease_name"].dropna().unique()[:5])
print("Annotation rows:", len(target_rows))
print("Unique target HPO terms:", len(target_hpo))
print("Patient HPO terms:", sorted(patient_hpo))
print("Target HPO terms:", sorted(target_hpo))
print("Exact overlap:", sorted(patient_hpo & target_hpo))
print("Patient terms absent from target annotations:",
      sorted(patient_hpo - target_hpo))
print("Target annotation evidence counts:")
print(target_rows["evidence"].value_counts(dropna=False).head(10))


=== TARGET ANNOTATION AUDIT ===
Target disease: OMIM:616873
Target disease names: ['Immunodeficiency, common variable, 13']
Annotation rows: 16
Unique target HPO terms: 16
Patient HPO terms: ['HP:0001744', 'HP:0002028', 'HP:0002205', 'HP:0002720', 'HP:0004315', 'HP:0004812']
Target HPO terms: ['HP:0000006', 'HP:0001561', 'HP:0001876', 'HP:0002718', 'HP:0002841', 'HP:0003596', 'HP:0003621', 'HP:0003829', 'HP:0004313', 'HP:0004429', 'HP:0005387', 'HP:0006721', 'HP:0010976', 'HP:0011462', 'HP:0011463', 'HP:0025116']
Exact overlap: []
Patient terms absent from target annotations: ['HP:0001744', 'HP:0002028', 'HP:0002205', 'HP:0002720', 'HP:0004315', 'HP:0004812']
Target annotation evidence counts:
evidence
PCS    16
Name: count, dtype: int64


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"
sys.path.insert(0, str(MODEL_DIR))

from rare_disease_data_loader import load_prediction_data
from rare_disease_ai import predict_diseases

target_id = "OMIM:616873"
case_id = "PMID_31057532_Family_G_individual_G_II_2"

validation = pd.read_csv(DATA / "patient_level_validation.csv")
case = validation.loc[validation["case_id"] == case_id].iloc[0]
patient_hpo = [x for x in str(case["hpo_ids"]).split(";") if x]

data = load_prediction_data()
phenotype_map = data["phenotype_map"]
loaded_terms = phenotype_map.get(target_id, set())

source = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    usecols=["database_id", "hpo_id"]
)
source_terms = set(
    source.loc[source["database_id"] == target_id, "hpo_id"]
    .dropna().astype(str).str.strip()
)

print("=== SOURCE VS LOADED PROFILE ===")
print("Source unique HPO terms:", len(source_terms))
print("Loaded predictor HPO terms:", len(loaded_terms))
print("Source-only terms:", sorted(source_terms - loaded_terms))
print("Loaded-only terms:", sorted(loaded_terms - source_terms))
print("Patient/source overlap:", sorted(set(patient_hpo) & source_terms))
print("Patient/loaded overlap:", sorted(set(patient_hpo) & loaded_terms))

print("\n=== REPRODUCE TARGET RANK ===")
results = predict_diseases(patient_hpo_terms=patient_hpo, top_k=10)
target = results[results["disease_id"] == target_id]
print("Target present in top 10:", not target.empty)
if not target.empty:
    print(target.to_string(index=False))
else:
    print("Target is outside top 10, as expected for this case.")

=== SOURCE VS LOADED PROFILE ===
Source unique HPO terms: 16
Loaded predictor HPO terms: 16
Source-only terms: []
Loaded-only terms: []
Patient/source overlap: []
Patient/loaded overlap: []

=== REPRODUCE TARGET RANK ===
Target present in top 10: False
Target is outside top 10, as expected for this case.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== ONTOLOGY FILES ===")
for path in sorted(DATA.glob("*")):
    if any(word in path.name.lower() for word in [
        "hpo", "ontology", "obo", "ic", "phenotype", "validation"
    ]):
        print(path.name, f"({path.stat().st_size / (1024**2):.2f} MB)")

=== ONTOLOGY FILES ===
app_predictor_validation_diagnostic.csv (0.05 MB)
clean_hpo_annotations.csv (34.24 MB)
disease_ranking_retrieval_metrics.csv (0.04 MB)
gene_informed_phenotype_recovery_metrics.csv (0.09 MB)
gene_informed_phenotype_recovery_top10.csv (0.62 MB)
gnn_annotations_publication_disjoint_candidate.csv (19.74 MB)
gnn_phenotype_recovery_metrics.csv (0.07 MB)
heldout_input_phenotypes.csv (0.47 MB)
heldout_target_phenotypes.csv (0.49 MB)
hierarchy_aware_retrieval_metrics.csv (0.08 MB)
hpo_symptom_catalogue.csv (1.40 MB)
ic_weighted_heldout_ranking_results.csv (0.07 MB)
matched_phenotype_gene_comparison.csv (0.14 MB)
missing_hpo_phenotype_audit.json (0.00 MB)
ontology_fallback_candidates.csv (0.00 MB)
ontology_fallback_coverage_summary.csv (0.00 MB)
ontology_fallback_evaluation.csv (0.00 MB)
ontology_fallback_ranked.csv (0.00 MB)
ontology_fallback_target_comparison.csv (0.00 MB)
patient_level_publication_leakage_audit.json (0.00 MB)
patient_level_ranker_bias_free_experiment_me

In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = [
    "hierarchy_aware_retrieval_metrics.csv",
    "ontology_fallback_coverage_summary.csv",
    "ontology_fallback_evaluation.csv",
    "ontology_fallback_target_comparison.csv",
    "gene_informed_phenotype_recovery_metrics.csv",
    "gnn_phenotype_recovery_metrics.csv",
    "phenotype_recovery_baseline_metrics.csv",
    "disease_ranking_retrieval_metrics.csv",
]

for filename in files:
    path = DATA / filename
    print(f"\n{'=' * 14} {filename} {'=' * 14}")

    if not path.exists() or path.stat().st_size == 0:
        print("Missing or empty")
        continue

    try:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(8))
    except Exception as exc:
        print("Could not read as CSV:", exc)

print("\n=== ONTOLOGY FALLBACK SUMMARY JSON ===")
for path in sorted(DATA.glob("ontology_fallback*.json")):
    print(path.name)
    try:
        print(json.dumps(json.loads(path.read_text()), indent=2)[:4000])
    except Exception as exc:
        print("Could not read JSON:", exc)


============== hierarchy_aware_retrieval_metrics.csv ==============
Shape: (1829, 6)
Columns: ['disease_id', 'rank', 'recall_at_1', 'recall_at_5', 'recall_at_10', 'reciprocal_rank']


,disease_id,rank,recall_at_1,recall_at_5,recall_at_10,reciprocal_rank
0,OMIM:100100,49,0,0,0,0.020408
1,OMIM:101900,65,0,0,0,0.015385
2,OMIM:102530,75,0,0,0,0.013333
3,OMIM:102900,79,0,0,0,0.012658
4,OMIM:103500,86,0,0,0,0.011628
5,OMIM:103780,88,0,0,0,0.011364
6,OMIM:104290,95,0,0,0,0.010526
7,OMIM:105150,104,0,0,0,0.009615



============== ontology_fallback_coverage_summary.csv ==============
Shape: (4, 4)
Columns: ['disease_id', 'fallback_candidate_count', 'best_specificity_score', 'closest_ontology_distance']


,disease_id,fallback_candidate_count,best_specificity_score,closest_ontology_distance
0,OMIM:102900,4,0.5,2
1,OMIM:300813,3,0.5,2
2,OMIM:303900,7,1.0,1
3,OMIM:619319,8,1.0,1



============== ontology_fallback_evaluation.csv ==============
Shape: (4, 6)
Columns: ['disease_id', 'candidate_count', 'target_count', 'matched_targets', 'target_recall', 'matched_hpo_ids']


...


============== phenotype_recovery_baseline_metrics.csv ==============
Shape: (3, 3)
Columns: ['Metric', 'Value', 'Evaluation']


,Metric,Value,Evaluation
0,Mean Recall@10,0.212949,Phenotype recovery baseline
1,Mean Reciprocal Rank,0.518946,Phenotype recovery baseline
2,Profiles evaluated,1832.000000,Phenotype recovery baseline



============== disease_ranking_retrieval_metrics.csv ==============
Shape: (1829, 6)
Columns: ['disease_id', 'rank', 'recall_at_1', 'recall_at_5', 'recall_at_10', 'reciprocal_rank']


,disease_id,rank,recall_at_1,recall_at_5,recall_at_10,reciprocal_rank
0,OMIM:100100,1,1,1,1,1.0
1,OMIM:101900,1,1,1,1,1.0
2,OMIM:102530,1,1,1,1,1.0
3,OMIM:102900,1,1,1,1,1.0
4,OMIM:103500,1,1,1,1,1.0
5,OMIM:103780,1,1,1,1,1.0
6,OMIM:104290,1,1,1,1,1.0
7,OMIM:105150,1,1,1,1,1.0



=== ONTOLOGY FALLBACK SUMMARY JSON ===


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

keywords = [
    "hierarchy", "ontology_fallback", "phenotype_recovery",
    "disease_ranking", "gnn", "validation", "evaluation"
]

print("=== POSSIBLE EXPERIMENT SCRIPTS ===")
matches = []

for path in PROJECT.rglob("*"):
    if not path.is_file():
        continue
    if path.suffix.lower() not in {".py", ".ipynb", ".md", ".txt"}:
        continue
    if any(word in path.name.lower() for word in keywords):
        matches.append(path)

if matches:
    for path in sorted(matches):
        print(path.relative_to(PROJECT))
else:
    print("No matching scripts found.")

print("\n=== CURRENT EVALUATION FILES ===")
for name in [
    "hierarchy_aware_retrieval_metrics.csv",
    "disease_ranking_retrieval_metrics.csv",
    "phenotype_recovery_baseline_metrics.csv",
    "app_predictor_validation_diagnostic.csv",
]:
    path = PROJECT / "processed" / name
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {name}")

=== POSSIBLE EXPERIMENT SCRIPTS ===
No matching scripts found.

=== CURRENT EVALUATION FILES ===
FOUND: hierarchy_aware_retrieval_metrics.csv
FOUND: disease_ranking_retrieval_metrics.csv
FOUND: phenotype_recovery_baseline_metrics.csv
FOUND: app_predictor_validation_diagnostic.csv


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

for folder in [PROJECT, PROJECT / "model", PROJECT / "processed"]:
    print(f"\n=== {folder.relative_to(PROJECT) or '.'} ===")

    if not folder.exists():
        print("Folder not found")
        continue

    files = sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in {".py", ".ipynb"}
    )

    if files:
        for path in files:
            print(f"{path.name} ({path.stat().st_size:,} bytes)")
    else:
        print("No Python scripts or notebooks directly in this folder")


=== . ===
No Python scripts or notebooks directly in this folder

=== model ===
rare_disease_ai.py (2,329 bytes)
rare_disease_data_loader.py (1,798 bytes)
rare_disease_predictor.py (2,993 bytes)
streamlit_app.py (6,556 bytes)

=== processed ===
No Python scripts or notebooks directly in this folder


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

for filename in [
    "model/rare_disease_ai.py",
    "model/rare_disease_data_loader.py",
    "model/rare_disease_predictor.py",
]:
    path = PROJECT / filename
    print(f"\n{'=' * 20} {filename} {'=' * 20}")

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    print(path.read_text(encoding="utf-8"))


==================== model/rare_disease_ai.py ====================

from rare_disease_data_loader import load_prediction_data

_DATA = load_prediction_data()
PHENOTYPE_MAP = _DATA["phenotype_map"]
GENE_MAP = _DATA["gene_map"]
IC = _DATA["ic"]
DISEASE_NAMES = _DATA["disease_names"]

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    observed = {
        str(t).strip() for t in patient_hpo_terms
        if str(t).strip() in IC
    }
    if not observed:
        raise ValueError("No recognized HPO terms were supplied.")

    weights = {t: IC[t] for t in observed}
    denominator = sum(weights.values()) or 1.0
    genes = {
        str(g).strip().upper() for g in (patient_genes or [])
        if str(g).strip()
    }

    results = []
    for disease, disease_terms in PHENOTYPE_MAP.items():
        overlap = observed & disease_terms
        precision = sum(weights[t] for t in overlap) / denominator
        recall = len(overlap)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

split_path = DATA / "disease_level_split.csv"
validation_path = DATA / "patient_level_validation.csv"

split = pd.read_csv(split_path, dtype={"disease_id": str})
validation = pd.read_csv(validation_path, dtype=str)

train_ids = set(
    split.loc[split["split"].str.lower().eq("train"), "disease_id"].dropna()
)
val_ids = set(
    split.loc[split["split"].str.lower().isin(["val", "valid", "validation"]), "disease_id"].dropna()
)
test_ids = set(
    split.loc[split["split"].str.lower().eq("test"), "disease_id"].dropna()
)

target_ids = set()
for value in validation["diagnosis_ids"].dropna():
    target_ids.update(
        x.strip() for x in str(value).split(";") if x.strip()
    )

print("=== SPLIT OVERVIEW ===")
print("Split labels:", split["split"].value_counts(dropna=False).to_dict())
print("Training diseases:", len(train_ids))
print("Validation diseases:", len(val_ids))
print("Test diseases:", len(test_ids))
print("Unique patient-validation target IDs:", len(target_ids))

print("\n=== TARGET OVERLAP ===")
print("Targets appearing in training split:", len(target_ids & train_ids))
print("Targets appearing in validation split:", len(target_ids & val_ids))
print("Targets appearing in test split:", len(target_ids & test_ids))
print("Targets absent from all split IDs:", len(target_ids - train_ids - val_ids - test_ids))

print("\nExamples of target IDs found in training:", sorted(target_ids & train_ids)[:10])
print("Examples of target IDs absent from split:", sorted(target_ids - train_ids - val_ids - test_ids)[:10])


=== SPLIT OVERVIEW ===
Split labels: {'train': 7340, 'test': 1836}
Training diseases: 7340
Validation diseases: 0
Test diseases: 1836
Unique patient-validation target IDs: 125

=== TARGET OVERLAP ===
Targets appearing in training split: 125
Targets appearing in validation split: 0
Targets appearing in test split: 0
Targets absent from all split IDs: 0

Examples of target IDs found in training: ['OMIM:102370', 'OMIM:115150', 'OMIM:117360', 'OMIM:120970', 'OMIM:121050', 'OMIM:122470', 'OMIM:123000', 'OMIM:129600', 'OMIM:130000', 'OMIM:130010']
Examples of target IDs absent from split: []


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

validation = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)

train_ids = set(
    split.loc[split["split"].str.lower().eq("train"), "disease_id"].dropna()
)
test_ids = set(
    split.loc[split["split"].str.lower().eq("test"), "disease_id"].dropna()
)
target_ids = {
    item.strip()
    for value in validation["diagnosis_ids"].dropna()
    for item in str(value).split(";")
    if item.strip()
}

profile_ids = set(ann["database_id"].str.strip().unique())
ann["database_id"] = ann["database_id"].str.strip()

print("=== ANNOTATION PROFILE COVERAGE ===")
print("Unique diseases in annotation source:", len(profile_ids))
print("Training diseases with profiles:", len(train_ids & profile_ids))
print("Test diseases with profiles:", len(test_ids & profile_ids))
print("Patient targets with profiles:", len(target_ids & profile_ids))
print("Patient targets without profiles:", len(target_ids - profile_ids))

print("\n=== ANNOTATION ROW DISTRIBUTION ===")
print("Rows belonging to training diseases:", int(ann["database_id"].isin(train_ids).sum()))
print("Rows belonging to test diseases:", int(ann["database_id"].isin(test_ids).sum()))
print("Rows belonging to patient target diseases:", int(ann["database_id"].isin(target_ids).sum()))

print("\n=== SAMPLE TARGETS ===")
print("Targets with profiles:", sorted(target_ids & profile_ids)[:10])
print("Targets without profiles:", sorted(target_ids - profile_ids)[:10])

=== ANNOTATION PROFILE COVERAGE ===
Unique diseases in annotation source: 12879
Training diseases with profiles: 7340
Test diseases with profiles: 1836
Patient targets with profiles: 125
Patient targets without profiles: 0

=== ANNOTATION ROW DISTRIBUTION ===
Rows belonging to training diseases: 179653
Rows belonging to test diseases: 44213
Rows belonging to patient target diseases: 6298

=== SAMPLE TARGETS ===
Targets with profiles: ['OMIM:102370', 'OMIM:115150', 'OMIM:117360', 'OMIM:120970', 'OMIM:121050', 'OMIM:122470', 'OMIM:123000', 'OMIM:129600', 'OMIM:130000', 'OMIM:130010']
Targets without profiles: []


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

patients = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

rows = []
for _, row in patients.head(10).iterrows():
    targets = {
        x.strip() for x in str(row["diagnosis_ids"]).split(";")
        if x.strip() and x.strip().lower() != "nan"
    }
    observed = {
        x.strip() for x in str(row["hpo_ids"]).split(";")
        if x.strip() and x.strip().lower() != "nan"
    }

    for target in sorted(targets):
        disease_terms = profiles.get(target, set())
        overlap = observed & disease_terms
        rows.append({
            "case_id": row["case_id"],
            "target_disease": target,
            "patient_hpo_count": len(observed),
            "disease_profile_hpo_count": len(disease_terms),
            "exact_overlap_count": len(overlap),
            "patient_hpos": ";".join(sorted(observed)),
            "overlapping_hpos": ";".join(sorted(overlap))
        })

display(pd.DataFrame(rows))

,case_id,target_disease,patient_hpo_count,disease_profile_hpo_count,exact_overlap_count,patient_hpos,overlapping_hpos
0,PMID_28830446_Patient_2,OMIM:615688,14,64,5,HP:0001954;HP:0002027;HP:0002140;HP:0002301;HP...,HP:0001954;HP:0002027;HP:0002301;HP:0030880;HP...
1,PMID_28830446_Patient_1,OMIM:615688,17,64,7,HP:0000623;HP:0001954;HP:0002027;HP:0002140;HP...,HP:0001954;HP:0002027;HP:0002301;HP:0002315;HP...
2,PMID_32480058_Case_F5P1_original_pedigree_II_2,OMIM:618052,5,88,5,HP:0000218;HP:0000369;HP:0001639;HP:0001712;HP...,HP:0000218;HP:0000369;HP:0001639;HP:0001712;HP...
3,PMID_32480058_Case_F4P2_original_pedigree_IV_5,OMIM:618052,8,88,8,HP:0000465;HP:0001639;HP:0001644;HP:0001712;HP...,HP:0000465;HP:0001639;HP:0001644;HP:0001712;HP...
4,PMID_32480058_Case_F10P2,OMIM:618052,5,88,5,HP:0001653;HP:0001712;HP:0004322;HP:0030718;HP...,HP:0001653;HP:0001712;HP:0004322;HP:0030718;HP...
5,PMID_32480058_Case_F6P1_original_pedigree_II_1,OMIM:618052,16,88,16,HP:0000293;HP:0000337;HP:0000343;HP:0000347;HP...,HP:0000293;HP:0000337;HP:0000343;HP:0000347;HP...
6,PMID_32480058_Case_F7P3,OMIM:618052,11,88,11,HP:0000316;HP:0000465;HP:0001639;HP:0001663;HP...,HP:0000316;HP:0000465;HP:0001639;HP:0001663;HP...
7,PMID_32480058_Case_F7P2,OMIM:618052,19,88,19,HP:0000316;HP:0000347;HP:0000369;HP:0000465;HP...,HP:0000316;HP:0000347;HP:0000369;HP:0000465;HP...
8,PMID_32480058_Case_F8P1,OMIM:618052,6,88,6,HP:0000316;HP:0001688;HP:0005133;HP:0005301;HP...,HP:0000316;HP:0001688;HP:0005133;HP:0005301;HP...
9,PMID_32480058_Case_F4P1_original_pedigree_IV_1,OMIM:618052,6,88,6,HP:0000465;HP:0001639;HP:0001644;HP:0006677;HP...,HP:0000465;HP:0001639;HP:0001644;HP:0006677;HP...


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

patients = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

rows = []
for _, row in patients.iterrows():
    observed = {
        x.strip() for x in str(row["hpo_ids"]).split(";")
        if x.strip() and x.strip().lower() != "nan"
    }
    targets = {
        x.strip() for x in str(row["diagnosis_ids"]).split(";")
        if x.strip() and x.strip().lower() != "nan"
    }

    for target in targets:
        disease_terms = profiles.get(target, set())
        overlap = observed & disease_terms
        rows.append({
            "case_id": row["case_id"],
            "target_disease": target,
            "patient_hpo_count": len(observed),
            "disease_profile_hpo_count": len(disease_terms),
            "overlap_count": len(overlap),
            "patient_coverage": len(overlap) / len(observed) if observed else np.nan,
            "disease_coverage": len(overlap) / len(disease_terms) if disease_terms else np.nan
        })

audit = pd.DataFrame(rows)

print("Cases evaluated:", len(audit))
print("Cases with no target profile:", int((audit["disease_profile_hpo_count"] == 0).sum()))
print("Cases with zero exact overlap:", int((audit["overlap_count"] == 0).sum()))
print("Cases with complete patient-term overlap:", int((audit["patient_coverage"] == 1).sum()))

print("\n=== PATIENT TERM COVERAGE DISTRIBUTION ===")
print(audit["patient_coverage"].describe(percentiles=[.1, .25, .5, .75, .9]).round(3))

print("\n=== OVERLAP GROUPS ===")
audit["overlap_group"] = pd.cut(
    audit["patient_coverage"],
    bins=[-0.001, 0, 0.5, 0.999999, 1.001],
    labels=["zero", "partial <=50%", "partial >50%", "complete"]
)
display(audit["overlap_group"].value_counts(dropna=False).rename("cases").to_frame())

print("\n=== LOWEST COVERAGE CASES ===")
display(
    audit.sort_values(["patient_coverage", "overlap_count"])
    [["case_id", "target_disease", "patient_hpo_count",
      "disease_profile_hpo_count", "overlap_count", "patient_coverage"]]
    .head(15)
)

Cases evaluated: 906
Cases with no target profile: 0
Cases with zero exact overlap: 9
Cases with complete patient-term overlap: 744

=== PATIENT TERM COVERAGE DISTRIBUTION ===
count    906.000
mean       0.920
std        0.217
min        0.000
10%        0.714
25%        1.000
50%        1.000
75%        1.000
90%        1.000
max        1.000
Name: patient_coverage, dtype: float64

=== OVERLAP GROUPS ===


,cases
overlap_group,
complete,744
partial >50%,92
partial <=50%,61
zero,9



=== LOWEST COVERAGE CASES ===


,case_id,target_disease,patient_hpo_count,disease_profile_hpo_count,overlap_count,patient_coverage
94,PMID_27761245_BRC081,OMIM:604804,3,17,0,0.000000
328,PMID_31057532_Family_I_individual_I_II_2,OMIM:616873,1,16,0,0.000000
334,PMID_31057532_Family_A_individual_A_II_1,OMIM:616873,4,16,0,0.000000
337,PMID_31057532_Family_C_individual_C_II_1,OMIM:616873,5,16,0,0.000000
340,PMID_27939403_D_I_1,OMIM:616873,3,16,0,0.000000
346,PMID_31057532_Family_H_individual_H_II_1,OMIM:616873,2,16,0,0.000000
348,PMID_31057532_Family_G_individual_G_II_2,OMIM:616873,6,16,0,0.000000
349,PMID_31057532_Family_I_individual_I_II_1,OMIM:616873,2,16,0,0.000000
647,PMID_30099644_IV_12,OMIM:600138,1,13,0,0.000000
339,PMID_28096536_patient,OMIM:616873,12,16,1,0.083333


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

patients = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)
diagnostic_path = DATA / "app_predictor_validation_diagnostic.csv"

if not diagnostic_path.exists():
    raise FileNotFoundError(diagnostic_path)

diagnostic = pd.read_csv(diagnostic_path, dtype={"case_id": str})
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

coverage_rows = []
for _, row in patients.iterrows():
    observed = {
        x.strip() for x in str(row["hpo_ids"]).split(";")
        if x.strip() and x.strip().lower() != "nan"
    }
    targets = {
        x.strip() for x in str(row["diagnosis_ids"]).split(";")
        if x.strip() and x.strip().lower() != "nan"
    }

    for target in targets:
        disease_terms = profiles.get(target, set())
        overlap = observed & disease_terms
        coverage_rows.append({
            "case_id": row["case_id"],
            "target_disease": target,
            "patient_coverage": len(overlap) / len(observed) if observed else np.nan,
            "overlap_count": len(overlap),
            "patient_hpo_count": len(observed),
            "disease_profile_hpo_count": len(disease_terms)
        })

coverage = pd.DataFrame(coverage_rows)

print("Diagnostic columns:", diagnostic.columns.tolist())
print("Coverage rows:", len(coverage))
print("Diagnostic rows:", len(diagnostic))

display(diagnostic.head(3))

# Join on case ID; retain the target disease where the diagnostic includes it.
merged = coverage.merge(diagnostic, on="case_id", how="left", suffixes=("", "_diagnostic"))

print("\nJoined rows:", len(merged))
print("Rows without a diagnostic match:", int(merged["target_rank"].isna().sum()) if "target_rank" in merged.columns else "target_rank column not found")

if "target_rank" in merged.columns:
    merged["rank_group"] = pd.cut(
        merged["target_rank"],
        bins=[0, 1, 5, 10, 50, float("inf")],
        labels=["Top 1", "2–5", "6–10", "11–50", "Below 50"]
    )
    display(
        merged.groupby("rank_group", observed=False)
        .agg(
            cases=("case_id", "count"),
            mean_patient_coverage=("patient_coverage", "mean"),
            median_patient_coverage=("patient_coverage", "median"),
            mean_overlap=("overlap_count", "mean")
        ).round(3)
    )

Diagnostic columns: ['case_id', 'target_diagnoses', 'n_input_hpo', 'n_recognized_hpo', 'target_rank', 'target_retrieved']
Coverage rows: 906
Diagnostic rows: 906


,case_id,target_diagnoses,n_input_hpo,n_recognized_hpo,target_rank,target_retrieved
0,PMID_28830446_Patient_2,OMIM:615688,14,14,19,True
1,PMID_28830446_Patient_1,OMIM:615688,17,17,19,True
2,PMID_32480058_Case_F5P1_original_pedigree_II_2,OMIM:618052,5,5,133,True



Joined rows: 906
Rows without a diagnostic match: 0


,cases,mean_patient_coverage,median_patient_coverage,mean_overlap
rank_group,,,,
Top 1,239,0.956,1.0,12.381
2–5,130,0.969,1.0,7.023
6–10,74,0.941,1.0,6.649
11–50,229,0.955,1.0,6.576
Below 50,234,0.813,1.0,5.457


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

patients = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)
diagnostic = pd.read_csv(DATA / "app_predictor_validation_diagnostic.csv", dtype=str)

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
profile_sizes = ann.groupby("database_id")["hpo_id"].nunique().rename("disease_profile_size")

patients["target_disease"] = patients["diagnosis_ids"].str.split(";").str[0].str.strip()

df = (
    patients[["case_id", "target_disease", "hpo_ids"]]
    .merge(diagnostic[["case_id", "target_rank"]], on="case_id", how="left")
    .join(profile_sizes, on="target_disease")
)

df["patient_hpo_count"] = df["hpo_ids"].apply(
    lambda x: len({
        t.strip() for t in str(x).split(";")
        if t.strip() and t.strip().lower() != "nan"
    })
)
df["target_rank"] = pd.to_numeric(df["target_rank"], errors="coerce")

df["rank_group"] = pd.cut(
    df["target_rank"],
    bins=[0, 1, 5, 10, 50, np.inf],
    labels=["Top 1", "2–5", "6–10", "11–50", "Below 50"]
)

print("=== DISEASE PROFILE SIZE BY TARGET RANK ===")
display(
    df.groupby("rank_group", observed=False)
    .agg(
        cases=("case_id", "count"),
        median_profile_size=("disease_profile_size", "median"),
        mean_profile_size=("disease_profile_size", "mean"),
        median_patient_hpos=("patient_hpo_count", "median"),
        median_target_rank=("target_rank", "median")
    ).round(2)
)

print("\n=== RANK CORRELATIONS ===")
print("Spearman correlation: target rank vs profile size:",
      round(df[["target_rank", "disease_profile_size"]].corr(method="spearman").iloc[0, 1], 3))
print("Spearman correlation: target rank vs patient HPO count:",
      round(df[["target_rank", "patient_hpo_count"]].corr(method="spearman").iloc[0, 1], 3))

print("\n=== CASES WITH COMPLETE COVERAGE BUT POOR RANK ===")
# Recalculate exact patient-term coverage for the target disease.
profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
coverage = []

for _, row in df.iterrows():
    observed = {
        t.strip() for t in str(row["hpo_ids"]).split(";")
        if t.strip() and t.strip().lower() != "nan"
    }
    target_terms = profiles.get(row["target_disease"], set())
    coverage.append(len(observed & target_terms) / len(observed) if observed else np.nan)

df["patient_coverage"] = coverage

display(
    df[(df["patient_coverage"] == 1) & (df["target_rank"] > 50)]
    .sort_values("target_rank", ascending=False)
    [["case_id", "target_disease", "target_rank",
      "patient_hpo_count", "disease_profile_size", "patient_coverage"]]
    .head(15)
)


=== DISEASE PROFILE SIZE BY TARGET RANK ===


,cases,median_profile_size,mean_profile_size,median_patient_hpos,median_target_rank
rank_group,,,,,
Top 1,239,30.0,47.59,11.0,1.0
2–5,130,19.0,42.88,4.0,4.0
6–10,74,33.5,47.49,7.0,8.0
11–50,229,81.0,65.44,7.0,24.0
Below 50,234,89.0,90.92,7.0,162.5



=== RANK CORRELATIONS ===
Spearman correlation: target rank vs profile size: 0.343
Spearman correlation: target rank vs patient HPO count: -0.262

=== CASES WITH COMPLETE COVERAGE BUT POOR RANK ===


,case_id,target_disease,target_rank,patient_hpo_count,disease_profile_size,patient_coverage
558,PMID_23714749_Family_1_patient_1,OMIM:256810,2235,5,190,1.0
560,PMID_23714749_Family_7_patient_10,OMIM:256810,2089,5,190,1.0
831,PMID_27040691_Individual_7_1,OMIM:616900,2038,5,118,1.0
821,PMID_27040691_Individual_4_1,OMIM:616900,1881,5,118,1.0
383,PMID_24091540_na,OMIM:117360,1822,9,141,1.0
829,PMID_27040691_Individual_2_1,OMIM:616900,1614,7,118,1.0
834,PMID_27040691_Individual_9_1,OMIM:616900,1601,5,118,1.0
590,PMID_20358602_Patient_C25,OMIM:122470,1596,7,129,1.0
376,PMID_39315527_Family_16_Najmabadi_et_al_II_3,OMIM:621354,1485,4,131,1.0
378,PMID_39315527_Family_16_Najmabadi_et_al_II_2,OMIM:621354,1485,4,131,1.0


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

sys.path.insert(0, str(PROJECT / "model"))

from rare_disease_ai import predict_diseases, PHENOTYPE_MAP

case_id = "PMID_23714749_Family_1_patient_1"
target_disease = "OMIM:256810"

patients = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str)
row = patients.loc[patients["case_id"] == case_id]

if row.empty:
    raise ValueError(f"Case not found: {case_id}")

patient_hpos = [
    term.strip()
    for term in row.iloc[0]["hpo_ids"].split(";")
    if term.strip()
]

target_terms = PHENOTYPE_MAP.get(target_disease, set())
overlap = set(patient_hpos) & target_terms

print("Case:", case_id)
print("Target disease:", target_disease)
print("Patient HPO terms:", patient_hpos)
print("Target profile size:", len(target_terms))
print("Exact overlap:", sorted(overlap))
print("Patient-term coverage:", len(overlap) / len(patient_hpos) if patient_hpos else 0)

# Score all candidates using the existing model.
ranked = predict_diseases(patient_hpos, top_k=len(PHENOTYPE_MAP))

target_row = ranked.loc[ranked["disease_id"] == target_disease]

print("\n=== TARGET SCORE ===")
if target_row.empty:
    print("Target disease was not returned by the predictor.")
else:
    display(target_row)

print("\n=== TOP 10 COMPETING CANDIDATES ===")
display(
    ranked.head(10)[
        ["rank", "disease_id", "disease_name", "phenotype_score",
         "matching_phenotypes", "matching_phenotype_ids"]
    ]
)

print("\n=== TARGET RANK FROM THIS RUN ===")
if target_row.empty:
    print("Not found")
else:
    print("Rank:", int(target_row.iloc[0]["rank"]))

Case: PMID_23714749_Family_1_patient_1
Target disease: OMIM:256810
Patient HPO terms: ['HP:0001252', 'HP:0001263', 'HP:0001943', 'HP:0002151', 'HP:0003256']
Target profile size: 190
Exact overlap: ['HP:0001252', 'HP:0001263', 'HP:0001943', 'HP:0002151', 'HP:0003256']
Patient-term coverage: 1.0

=== TARGET SCORE ===


,rank,disease_id,disease_name,phenotype_score,gene_score,combined_score,matching_phenotypes,matching_phenotype_ids,matching_genes
2234,2235,OMIM:256810,Mitochondrial DNA depletion syndrome 6 (hepato...,0.051282,None,0.051282,5,HP:0001252;HP:0001263;HP:0001943;HP:0002151;HP...,



=== TOP 10 COMPETING CANDIDATES ===


,rank,disease_id,disease_name,phenotype_score,matching_phenotypes,matching_phenotype_ids
0,1,OMIM:613933,Acetyl-CoA carboxylase deficiency,0.424835,3,HP:0001252;HP:0001263;HP:0002151
1,2,OMIM:614055,Acetyl-Coa acetyltransferase-2 deficiency,0.424835,3,HP:0001252;HP:0001263;HP:0002151
2,3,OMIM:616111,"Mitochondrial complex III deficiency, nuclear ...",0.346104,4,HP:0001252;HP:0001263;HP:0001943;HP:0002151
3,4,OMIM:137900,Globulin anomaly involving beta (2a)-globulin,0.339457,1,HP:0003256
4,5,OMIM:618245,"Mitochondrial complex I deficiency, nuclear ty...",0.333320,2,HP:0001252;HP:0002151
5,6,OMIM:500017,Mitochondrial Leigh syndrome,0.331069,3,HP:0001252;HP:0001263;HP:0002151
6,7,OMIM:266150,Pyruvate carboxylase deficiency,0.318542,4,HP:0001252;HP:0001263;HP:0001943;HP:0002151
7,8,OMIM:614741,Mitochondrial pyruvate carrier deficiency,0.295046,4,HP:0001252;HP:0001263;HP:0001943;HP:0002151
8,9,ORPHA:391457,"HSD10 disease, neonatal type",0.293862,2,HP:0001263;HP:0003256
9,10,OMIM:609016,Long-Chain 3-hydroxyacyl-coa dehydrogenase def...,0.289254,2,HP:0001252;HP:0001943



=== TARGET RANK FROM THIS RUN ===
Rank: 2235


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = [
    "patient_validation_matched_candidate_comparison.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_phenotype_only_validation.csv",
    "patient_validation_score_combination.csv",
    "phenotype_baseline_per_profile.csv",
    "phenotype_disease_ranking_evaluation.csv",
    "phenotype_overlap_baseline_top10.csv",
    "ic_weighted_heldout_ranking_results.csv",
]

for filename in files:
    path = DATA / filename
    print(f"\n{'=' * 18} {filename} {'=' * 18}")

    if not path.exists():
        print("Not found")
        continue

    try:
        df = pd.read_csv(path, low_memory=False)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(3))
    except Exception as exc:
        print("Read error:", repr(exc))


================== patient_validation_matched_candidate_comparison.csv ==================
Shape: (906, 11)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_observed_hpo', 'baseline_native_rank', 'ranker_native_rank', 'baseline_common_rank', 'ranker_common_rank', 'target_in_baseline_vocab', 'target_in_ranker_vocab', 'target_in_common_vocab']


,case_id,pmid,diagnosis_ids,n_observed_hpo,baseline_native_rank,ranker_native_rank,baseline_common_rank,ranker_common_rank,target_in_baseline_vocab,target_in_ranker_vocab,target_in_common_vocab
0,PMID_28830446_Patient_2,28830446,OMIM:615688,14,19,2831,7,2831,1,1,1
1,PMID_28830446_Patient_1,28830446,OMIM:615688,17,19,2711,11,2711,1,1,1
2,PMID_32480058_Case_F5P1_original_pedigree_II_2,32480058,OMIM:618052,5,133,2188,77,2188,1,1,1



================== patient_phenotype_only_baseline.csv ==================
Shape: (1959, 8)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']


,case_id,pmid,diagnosis_ids,rank,top1,top5,top10,candidate_covered
0,PMID_28239884_Family_1_proband,28239884,OMIM:148600,3.0,0,1,1,1
1,PMID_28239884_Family_2_proband,28239884,OMIM:148600,3.0,0,1,1,1
2,PMID_28239884_Family_3_proband,28239884,OMIM:148600,3.0,0,1,1,1



================== patient_phenotype_only_validation.csv ==================
Shape: (906, 10)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_recognized_hpo', 'target_in_candidate_set', 'rank', 'top1', 'top5', 'top10', 'mrr']


...

,disease_id,input_phenotype_count,hidden_phenotype,correct_disease_rank,recall_at_1,recall_at_5,recall_at_10,reciprocal_rank
0,OMIM:100100,15,HP:0000028,1,1,1,1,1.0
1,OMIM:101900,8,HP:0025092,1,1,1,1,1.0
2,OMIM:102530,3,HP:0012205,1,1,1,1,1.0



================== phenotype_overlap_baseline_top10.csv ==================
Shape: (18164, 4)
Columns: ['heldout_disease_id', 'candidate_disease_id', 'overlap_score', 'rank']


,heldout_disease_id,candidate_disease_id,overlap_score,rank
0,OMIM:100100,ORPHA:2970,4,1
1,OMIM:100100,OMIM:191800,3,2
2,OMIM:100100,OMIM:219100,3,3



================== ic_weighted_heldout_ranking_results.csv ==================
Shape: (1835, 6)
Columns: ['heldout_disease_id', 'observed_phenotypes', 'known_observed_phenotypes', 'top1_candidate_disease', 'top10_contains_phenotype_overlap', 'reciprocal_rank_first_overlap']


,heldout_disease_id,observed_phenotypes,known_observed_phenotypes,top1_candidate_disease,top10_contains_phenotype_overlap,reciprocal_rank_first_overlap
0,OMIM:100100,16,16,ORPHA:2970,True,1.0
1,OMIM:101900,9,9,OMIM:616295,True,1.0
2,OMIM:102530,4,3,OMIM:613958,True,1.0


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

path = DATA / "patient_validation_score_combination.csv"
df = pd.read_csv(path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nRows per alpha setting:")
print(df.groupby("alpha_ranker")["case_id"].nunique().to_string())

summary = (
    df.groupby("alpha_ranker")
    .agg(
        cases=("case_id", "nunique"),
        mean_rank=("target_rank", "mean"),
        median_rank=("target_rank", "median"),
        MRR=("mrr", "mean"),
        Top1=("top1", "mean"),
        Top5=("top5", "mean"),
        Top10=("top10", "mean"),
    )
    .round(4)
)

print("\n=== SCORE-COMBINATION SUMMARY ===")
display(summary)

print("\n=== BEST ALPHA BY MRR ===")
display(summary.sort_values("MRR", ascending=False).head(5))

print("\n=== BASELINE ALPHA ROWS ===")
display(df.loc[df["alpha_ranker"] == 0].head(5))

Shape: (9966, 7)
Columns: ['case_id', 'alpha_ranker', 'target_rank', 'top1', 'top5', 'top10', 'mrr']

Rows per alpha setting:
alpha_ranker
0.0    906
0.1    906
0.2    906
0.3    906
0.4    906
0.5    906
0.6    906
0.7    906
0.8    906
0.9    906
1.0    906

=== SCORE-COMBINATION SUMMARY ===


,cases,mean_rank,median_rank,MRR,Top1,Top5,Top10
alpha_ranker,,,,,,,
0.0,906,92.3168,8.0,0.3773,0.2947,0.4680,0.5419
0.1,906,151.1004,22.0,0.3082,0.2031,0.4084,0.4514
0.2,906,251.3002,29.0,0.3092,0.2031,0.4161,0.4503
0.3,906,386.7693,34.0,0.3064,0.1987,0.4183,0.4525
0.4,906,568.0916,42.0,0.3018,0.1932,0.4227,0.4503
0.5,906,828.5121,43.5,0.2985,0.1921,0.4238,0.4581
0.6,906,1156.9459,44.5,0.3002,0.1998,0.4139,0.4603
0.7,906,1471.6976,49.5,0.3007,0.2053,0.4029,0.4514
0.8,906,1743.9161,52.0,0.3019,0.2119,0.3929,0.4338



=== BEST ALPHA BY MRR ===


,cases,mean_rank,median_rank,MRR,Top1,Top5,Top10
alpha_ranker,,,,,,,
0.0,906,92.3168,8.0,0.3773,0.2947,0.4680,0.5419
0.2,906,251.3002,29.0,0.3092,0.2031,0.4161,0.4503
0.1,906,151.1004,22.0,0.3082,0.2031,0.4084,0.4514
0.9,906,1977.5166,89.5,0.3075,0.2285,0.3918,0.4260
0.3,906,386.7693,34.0,0.3064,0.1987,0.4183,0.4525



=== BASELINE ALPHA ROWS ===


,case_id,alpha_ranker,target_rank,top1,top5,top10,mrr
0,PMID_28830446_Patient_2,0.0,7,0,0,1,0.142857
11,PMID_28830446_Patient_1,0.0,11,0,0,0,0.090909
22,PMID_32480058_Case_F5P1_original_pedigree_II_2,0.0,77,0,0,0,0.012987
33,PMID_32480058_Case_F4P2_original_pedigree_IV_5,0.0,10,0,0,1,0.100000
44,PMID_32480058_Case_F10P2,0.0,64,0,0,0,0.015625


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

path = DATA / "patient_validation_score_combination.csv"
df = pd.read_csv(path)

print("=== DATA INTEGRITY ===")
print("Rows:", len(df))
print("Unique cases:", df["case_id"].nunique())
print("Alpha values:", sorted(df["alpha_ranker"].unique()))
print("Missing values:")
print(df[["case_id", "alpha_ranker", "target_rank", "mrr"]].isna().sum())

print("\n=== RANK DISTRIBUTION BY ALPHA ===")
print(
    df.groupby("alpha_ranker")["target_rank"]
      .agg(["min", "median", "mean", "max"])
      .round(2)
      .to_string()
)

print("\n=== ONE CASE ACROSS ALL ALPHAS ===")
example_case = df["case_id"].iloc[0]
print("Case:", example_case)
print(
    df[df["case_id"] == example_case]
      .sort_values("alpha_ranker")[
          ["alpha_ranker", "target_rank", "top1", "top5", "top10", "mrr"]
      ]
      .to_string(index=False)
)

=== DATA INTEGRITY ===
Rows: 9966
Unique cases: 906
Alpha values: [np.float64(0.0), np.float64(0.1), np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(0.7), np.float64(0.8), np.float64(0.9), np.float64(1.0)]
Missing values:
case_id         0
alpha_ranker    0
target_rank     0
mrr             0
dtype: int64

=== RANK DISTRIBUTION BY ALPHA ===
              min  median     mean   max
alpha_ranker                            
0.0             1     8.0    92.32  4014
0.1             1    22.0   151.10  4020
0.2             1    29.0   251.30  4462
0.3             1    34.0   386.77  4918
0.4             1    42.0   568.09  5493
0.5             1    43.5   828.51  5934
0.6             1    44.5  1156.95  6204
0.7             1    49.5  1471.70  6388
0.8             1    52.0  1743.92  6486
0.9             1    89.5  1977.52  6956
1.0             1   232.5  2183.85  7320

=== ONE CASE ACROSS ALL ALPHAS ===
Case: PMID_28830446_Patient_2
 alpha_ran

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

keywords = (
    "alpha_ranker",
    "ranker_score",
    "baseline_score",
    "combined_score",
    "score_combination",
)

print("=== SCORE-BLENDING CODE SEARCH ===")

found = False

for path in PROJECT.rglob("*.py"):
    if any(part.startswith(".") for part in path.parts):
        continue

    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except Exception:
        continue

    matches = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    if matches:
        found = True
        print(f"\n--- {path.relative_to(PROJECT)} ---")

        shown = set()
        for i in matches:
            start = max(0, i - 3)
            end = min(len(lines), i + 4)

            if any(j in shown for j in range(start, end)):
                continue

            print(f"\nLines {start + 1}-{end}:")
            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)

if not found:
    print("No matching blending code found in project Python files.")
    print("The experiment may have been run directly in a notebook cell.")

=== SCORE-BLENDING CODE SEARCH ===

--- model/rare_disease_predictor.py ---

Lines 45-51:
  45:         gene_score = len(matching_genes) / len(genes) if genes else np.nan
  46: 
  47:         if genes and matching_genes:
  48:             combined_score = (
  49:                 (1 - gene_weight) * phenotype_score
  50:                 + gene_weight * gene_score
  51:             )

Lines 56-62:
  56:             "disease_id": disease,
  57:             "phenotype_score": phenotype_score,
  58:             "gene_score": gene_score,
  59:             "combined_score": combined_score,
  60:             "matching_phenotypes": len(overlap),
  61:             "matching_genes": ";".join(sorted(matching_genes))
  62:         })

Lines 89-95:
  89:     )
  90: 
  91:     columns = [
  92:         "rank", "disease_id", "disease_name", "combined_score",
  93:         "matching_phenotypes", "matching_phenotype_ids",
  94:         "matching_genes"
  95:     ]

--- model/rare_disease_ai.py ---

Lin

In [ ]:
from pathlib import Path
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

print("=== SEARCHING NOTEBOOKS FOR RANKER BLENDING ===")

found = False
notebook_count = 0
keywords = (
    "alpha_ranker",
    "patient_validation_score_combination",
    "combined_score",
    "ranker_native_rank",
)

for path in PROJECT.rglob("*.ipynb"):
    if any(part.startswith(".") for part in path.parts):
        continue

    try:
        notebook = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue

    notebook_count += 1

    for cell_index, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))

        if any(word in source.lower() for word in keywords):
            found = True
            print(f"\n--- Notebook: {path.relative_to(PROJECT)} ---")
            print(f"Cell index: {cell_index}")
            print(source[:7000])

print(f"\nNotebooks inspected: {notebook_count}")

if not found:
    print("No matching experiment code found in saved notebooks.")

=== SEARCHING NOTEBOOKS FOR RANKER BLENDING ===

Notebooks inspected: 0
No matching experiment code found in saved notebooks.


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== MATCHED-CANDIDATE SUMMARY ===")

summary_path = DATA / "patient_validation_matched_candidate_summary.json"

if summary_path.exists():
    with open(summary_path, "r", encoding="utf-8") as f:
        summary_data = json.load(f)
    print(json.dumps(summary_data, indent=2))
else:
    print("Summary JSON not found.")

print("\n=== MATCHED-CANDIDATE TABLE ===")

table_path = DATA / "patient_validation_matched_candidate_comparison.csv"

if table_path.exists():
    df = pd.read_csv(table_path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    rank_cols = [
        "baseline_native_rank",
        "ranker_native_rank",
        "baseline_common_rank",
        "ranker_common_rank",
    ]

    print("\nRank summary:")
    display(df[rank_cols].describe().round(2))

    print("\nTarget vocabulary coverage:")
    for col in [
        "target_in_baseline_vocab",
        "target_in_ranker_vocab",
        "target_in_common_vocab",
    ]:
        if col in df.columns:
            print(f"{col}: {df[col].sum()} / {len(df)}")
else:
    print("Comparison CSV not found.")

=== MATCHED-CANDIDATE SUMMARY ===
{
  "device": "cuda",
  "validation_cases": 906,
  "target_coverage": {
    "baseline_vocab": 906,
    "ranker_vocab": 906,
    "shared_vocab": 906
  },
  "baseline_native": {
    "n_ranked": 906,
    "Top1": 0.2638,
    "Top5": 0.4073,
    "Top10": 0.489,
    "MRR": 0.336
  },
  "ranker_native": {
    "n_ranked": 906,
    "Top1": 0.021,
    "Top5": 0.0872,
    "Top10": 0.1104,
    "MRR": 0.0527
  },
  "baseline_shared_candidates": {
    "n_ranked": 906,
    "Top1": 0.2947,
    "Top5": 0.468,
    "Top10": 0.5419,
    "MRR": 0.3773
  },
  "ranker_shared_candidates": {
    "n_ranked": 906,
    "Top1": 0.021,
    "Top5": 0.0872,
    "Top10": 0.1104,
    "MRR": 0.0527
  },
  "saved_predictions": "/content/drive/MyDrive/Rare Disease ai/processed/patient_validation_matched_candidate_comparison.csv"
}

=== MATCHED-CANDIDATE TABLE ===
Shape: (906, 11)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_observed_hpo', 'baseline_native_rank', 'ranker_native_rank', 

,baseline_native_rank,ranker_native_rank,baseline_common_rank,ranker_common_rank
count,906.00,906.00,906.00,906.00
mean,152.90,2029.25,92.32,2029.25
std,659.77,2327.19,385.08,2327.19
min,1.00,1.00,1.00,1.00
25%,1.00,65.00,1.00,65.00
50%,12.00,450.00,8.00,450.00
75%,54.00,4346.00,30.00,4346.00
max,6927.00,6949.00,4014.00,6949.00



Target vocabulary coverage:
target_in_baseline_vocab: 906 / 906
target_in_ranker_vocab: 906 / 906
target_in_common_vocab: 906 / 906


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

keywords = (
    "patient_level_ranker",
    "class Patient",
    "class Ranker",
    "torch.load",
    "load_state_dict",
    "TripletMarginLoss",
    "MarginRankingLoss",
    "BCEWithLogitsLoss",
    "CrossEntropyLoss",
)

print("=== LEARNED RANKER IMPLEMENTATION SEARCH ===")

found = False

for path in PROJECT.rglob("*.py"):
    if any(part.startswith(".") for part in path.parts):
        continue

    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except Exception:
        continue

    matches = [
        i for i, line in enumerate(lines)
        if any(word.lower() in line.lower() for word in keywords)
    ]

    if matches:
        found = True
        print(f"\n--- {path.relative_to(PROJECT)} ---")

        shown = set()
        for i in matches:
            start = max(0, i - 5)
            end = min(len(lines), i + 9)

            if any(j in shown for j in range(start, end)):
                continue

            print(f"\nLines {start + 1}-{end}:")
            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)

if not found:
    print("No matching implementation found in Python files.")

print("\n=== CHECKPOINT FILES ===")
model_dir = PROJECT / "model"

for path in sorted(model_dir.glob("*.pt")):
    print(f"{path.name} — {path.stat().st_size / (1024**2):.2f} MB")


=== LEARNED RANKER IMPLEMENTATION SEARCH ===
No matching implementation found in Python files.

=== CHECKPOINT FILES ===
patient_level_ranker_best.pt — 8.95 MB
patient_level_ranker_bias_free_experiment.pt — 8.92 MB
patient_ranker_bias_softmax_experiment.pt — 8.95 MB
rare_disease_gnn_filtered_best.pt — 4.63 MB
rare_disease_gnn_publication_disjoint_experiment.pt — 4.44 MB


In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

checkpoint_names = [
    "patient_level_ranker_best.pt",
    "patient_level_ranker_bias_free_experiment.pt",
    "patient_ranker_bias_softmax_experiment.pt",
]

print("=== PATIENT RANKER CHECKPOINT AUDIT ===")

for name in checkpoint_names:
    path = MODEL_DIR / name
    print(f"\n--- {name} ---")

    if not path.exists():
        print("File not found.")
        continue

    try:
        # Checkpoint files are project-generated artifacts.
        # weights_only=True avoids unpickling arbitrary Python objects.
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        print("Top-level type:", type(checkpoint).__name__)

        if isinstance(checkpoint, dict):
            print("Top-level keys:", list(checkpoint.keys()))

            for key, value in checkpoint.items():
                if isinstance(value, dict):
                    print(
                        f"  {key}: dict with {len(value)} entries; "
                        f"sample keys: {list(value.keys())[:8]}"
                    )
                elif isinstance(value, torch.Tensor):
                    print(
                        f"  {key}: Tensor shape={tuple(value.shape)}, "
                        f"dtype={value.dtype}"
                    )
                elif isinstance(value, (str, int, float, bool, type(None))):
                    print(f"  {key}: {value}")
                else:
                    print(f"  {key}: {type(value).__name__}")

        else:
            print("Unexpected checkpoint structure.")

    except Exception as exc:
        print("Safe metadata inspection failed:", type(exc).__name__, str(exc)[:300])

print("\nAudit complete. No checkpoints were modified.")

=== PATIENT RANKER CHECKPOINT AUDIT ===

--- patient_level_ranker_best.pt ---
Top-level type: dict
Top-level keys: ['model_state_dict', 'n_diseases', 'n_phenotypes', 'hidden_dim', 'disease_ids', 'phenotype_ids', 'best_validation_mrr', 'seed', 'training_cases', 'validation_cases', 'note']
  model_state_dict: dict with 7 entries; sample keys: ['hpo_embedding.weight', 'disease_embedding.weight', 'patient_projection.0.weight', 'patient_projection.0.bias', 'patient_projection.3.weight', 'patient_projection.3.bias', 'disease_bias.weight']
  n_diseases: 7340
  n_phenotypes: 9982
  hidden_dim: 128
  disease_ids: list
  phenotype_ids: list
  best_validation_mrr: 0.052704318912176605
  seed: 42
  training_cases: 5730
  validation_cases: 906
  note: Patient-level ranking model. Validation split by PMID. Unrecorded diseases used as sampled competitors are not guaranteed negatives.

--- patient_level_ranker_bias_free_experiment.pt ---
Top-level type: dict
Top-level keys: ['model_state_dict', 'n_dis

In [ ]:
from pathlib import Path
import torch

MODEL_DIR = Path("/content/drive/MyDrive/Rare Disease ai/model")

names = [
    "patient_level_ranker_bias_free_experiment.pt",
    "patient_ranker_bias_softmax_experiment.pt",
]

for name in names:
    path = MODEL_DIR / name
    print(f"\n=== {name} ===")

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print("Keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if key == "model_state_dict":
            print("Model tensors:")
            for tensor_name, tensor in value.items():
                print(f"  {tensor_name}: {tuple(tensor.shape)}")
        elif isinstance(value, (str, int, float, bool, type(None))):
            print(f"{key}: {value}")
        elif isinstance(value, list):
            print(f"{key}: list, length={len(value)}")
        elif isinstance(value, dict):
            print(f"{key}: dict, length={len(value)}")
        else:
            print(f"{key}: {type(value).__name__}")

print("\nCheckpoint audit complete; no files modified.")



=== patient_level_ranker_bias_free_experiment.pt ===
Keys: ['model_state_dict', 'n_diseases', 'n_phenotypes', 'hidden_dim', 'disease_ids', 'phenotype_ids', 'best_validation_mrr', 'validation_metrics', 'seed', 'training_cases', 'validation_cases', 'note']
Model tensors:
  hpo_embedding.weight: (9982, 128)
  disease_embedding.weight: (7340, 128)
  patient_projection.0.weight: (128, 128)
  patient_projection.0.bias: (128,)
  patient_projection.3.weight: (128, 128)
  patient_projection.3.bias: (128,)
n_diseases: 7340
n_phenotypes: 9982
hidden_dim: 128
disease_ids: list, length=7340
phenotype_ids: list, length=9982
best_validation_mrr: 0.020924354840425125
validation_metrics: dict, length=5
seed: 42
training_cases: 5730
validation_cases: 906
note: Experimental bias-free patient ranker. Validation split by PMID. Sampled unrecorded diseases are not guaranteed negatives. Not clinically validated.

=== patient_ranker_bias_softmax_experiment.pt ===
Keys: ['model_state_dict', 'disease_ids', 'phe

In [ ]:
from pathlib import Path
import torch
import json

MODEL_DIR = Path("/content/drive/MyDrive/Rare Disease ai/model")

checkpoints = [
    "patient_level_ranker_best.pt",
    "patient_level_ranker_bias_free_experiment.pt",
    "patient_ranker_bias_softmax_experiment.pt",
]

for name in checkpoints:
    path = MODEL_DIR / name
    print(f"\n{'=' * 65}")
    print(name)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    for key in [
        "best_validation_mrr",
        "validation_metrics",
        "best_validation_metrics",
        "best_epoch",
        "objective",
        "note",
    ]:
        if key in checkpoint:
            value = checkpoint[key]

            if isinstance(value, torch.Tensor):
                value = value.tolist()

            print(f"\n{key}:")
            print(json.dumps(value, indent=2, default=str))

print("\nMetric inspection complete. No files modified.")


patient_level_ranker_best.pt

best_validation_mrr:
0.052704318912176605

note:
"Patient-level ranking model. Validation split by PMID. Unrecorded diseases used as sampled competitors are not guaranteed negatives."

patient_level_ranker_bias_free_experiment.pt

best_validation_mrr:
0.020924354840425125

validation_metrics:
{
  "cases": 906,
  "mrr": 0.020924354840425125,
  "top1": 0.0,
  "top5": 0.03090507726269316,
  "top10": 0.0640176600441501
}

note:
"Experimental bias-free patient ranker. Validation split by PMID. Sampled unrecorded diseases are not guaranteed negatives. Not clinically validated."

patient_ranker_bias_softmax_experiment.pt

best_validation_metrics:
{
  "cases": 906,
  "mrr": 0.3341009854885823,
  "top1": 0.30132450331125826,
  "top5": 0.3631346578366446,
  "top10": 0.39514348785871967
}

best_epoch:
15

objective:
"single-label full-disease cross-entropy"

note:
"Research experiment; not clinically validated."

Metric inspection complete. No files modified.


In [ ]:
from pathlib import Path
import torch

MODEL_DIR = Path("/content/drive/MyDrive/Rare Disease ai/model")

path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=True
)

print("=== SOFTMAX EXPERIMENT METRICS ===")

metrics = checkpoint.get("best_validation_metrics", {})
for key, value in metrics.items():
    print(f"{key}: {value}")

print("\n=== TRAINING CONFIGURATION ===")
for key in [
    "best_epoch",
    "objective",
    "training_cases",
    "validation_cases",
    "seed",
    "hidden_dim",
    "disease_bias",
    "note",
]:
    if key in checkpoint:
        print(f"{key}: {checkpoint[key]}")

print("\n=== CHECKPOINT MODEL TENSORS ===")
for key, value in checkpoint["model_state_dict"].items():
    if isinstance(value, torch.Tensor):
        print(f"{key}: {tuple(value.shape)}")

print("\nRead-only inspection complete.")


=== SOFTMAX EXPERIMENT METRICS ===
cases: 906
mrr: 0.3341009854885823
top1: 0.30132450331125826
top5: 0.3631346578366446
top10: 0.39514348785871967

=== TRAINING CONFIGURATION ===
best_epoch: 15
objective: single-label full-disease cross-entropy
training_cases: 5730
validation_cases: 906
seed: 42
hidden_dim: 128
disease_bias: True
note: Research experiment; not clinically validated.

=== CHECKPOINT MODEL TENSORS ===
disease_bias: (7340,)
hpo_embedding.weight: (9982, 128)
disease_embedding.weight: (7340, 128)
patient_projection.0.weight: (128, 128)
patient_projection.0.bias: (128,)
patient_projection.3.weight: (128, 128)
patient_projection.3.bias: (128,)

Read-only inspection complete.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "patient_validation_score_combination.csv",
    "patient_validation_matched_candidate_comparison.csv",
    "app_predictor_validation_diagnostic.csv",
    "patient_level_validation.csv",
]

for name in files:
    path = DATA / name
    print(f"\n=== {name} ===")

    if not path.exists():
        print("Not found")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    # Show whether the file explicitly identifies a model or experiment.
    model_columns = [
        col for col in df.columns
        if any(word in col.lower() for word in
               ["model", "method", "experiment", "alpha", "ranker", "softmax"])
    ]

    if model_columns:
        print("Model/experiment-related columns:")
        print(df[model_columns].head(3).to_string(index=False))

print("\nInspection complete. No files modified.")


=== patient_validation_score_combination.csv ===
Shape: (9966, 7)
Columns: ['case_id', 'alpha_ranker', 'target_rank', 'top1', 'top5', 'top10', 'mrr']
Model/experiment-related columns:
 alpha_ranker
          0.0
          0.1
          0.2

=== patient_validation_matched_candidate_comparison.csv ===
Shape: (906, 11)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_observed_hpo', 'baseline_native_rank', 'ranker_native_rank', 'baseline_common_rank', 'ranker_common_rank', 'target_in_baseline_vocab', 'target_in_ranker_vocab', 'target_in_common_vocab']
Model/experiment-related columns:
 ranker_native_rank  ranker_common_rank  target_in_ranker_vocab
               2831                2831                       1
               2711                2711                       1
               2188                2188                       1

=== app_predictor_validation_diagnostic.csv ===
Shape: (906, 6)
Columns: ['case_id', 'target_diagnoses', 'n_input_hpo', 'n_recognized_hpo', 'target_rank',

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

keywords = (
    "patient_ranker_bias_softmax_experiment",
    "single-label full-disease cross-entropy",
    "best_validation_metrics",
    "patient_projection",
    "disease_bias",
)

print("=== SEARCHING PROJECT FILES ===")
found = False

extensions = {".py", ".md", ".txt", ".json", ".yaml", ".yml", ".log"}

for path in PROJECT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue
    if any(part.startswith(".") for part in path.parts):
        continue

    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    matches = [word for word in keywords if word.lower() in text.lower()]

    if matches:
        found = True
        print(f"\nFile: {path.relative_to(PROJECT)}")
        print("Matched terms:", ", ".join(matches))

        lines = text.splitlines()
        shown = 0
        for i, line in enumerate(lines):
            if any(word.lower() in line.lower() for word in matches):
                start = max(0, i - 2)
                end = min(len(lines), i + 4)
                print(f"\nLines {start + 1}-{end}:")
                for j in range(start, end):
                    print(f"{j + 1:4}: {lines[j]}")
                shown += 1
                if shown >= 3:
                    print("Additional matches omitted for this file.")
                    break

if not found:
    print("No training or evaluation implementation found in the searched project files.")

print("\nSearch complete. No files modified.")

=== SEARCHING PROJECT FILES ===

File: model/patient_ranker_bias_softmax_experiment_config.json
Matched terms: single-label full-disease cross-entropy, disease_bias

Lines 7-12:
   7:   "best_epoch": 15,
   8:   "patience": 4,
   9:   "objective": "single-label full-disease cross-entropy",
  10:   "disease_bias": true,
  11:   "device": "cuda"
  12: }

Lines 8-12:
   8:   "patience": 4,
   9:   "objective": "single-label full-disease cross-entropy",
  10:   "disease_bias": true,
  11:   "device": "cuda"
  12: }


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
path = PROJECT / "model" / "patient_ranker_bias_softmax_experiment_config.json"

print("=== COMPLETE SOFTMAX EXPERIMENT CONFIG ===")

if path.exists():
    config = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(config, indent=2))
else:
    print("Configuration file not found.")

print("\n=== RELATED EXPERIMENT FILES ===")

for folder in [PROJECT / "model", PROJECT / "processed"]:
    print(f"\n{folder.name}/")
    for item in sorted(folder.iterdir()):
        if any(word in item.name.lower() for word in
               ["softmax", "ranker", "patient_validation", "score_combination"]):
            print(item.name)

=== COMPLETE SOFTMAX EXPERIMENT CONFIG ===
{
  "seed": 42,
  "hidden_dim": 128,
  "learning_rate": 0.0003,
  "weight_decay": 0.0001,
  "epochs_run": 19,
  "best_epoch": 15,
  "patience": 4,
  "objective": "single-label full-disease cross-entropy",
  "disease_bias": true,
  "device": "cuda"
}

=== RELATED EXPERIMENT FILES ===

model/
patient_level_ranker_best.pt
patient_level_ranker_bias_free_experiment.pt
patient_level_ranker_bias_free_experiment_config.json
patient_level_ranker_config.json
patient_ranker_bias_softmax_experiment.pt
patient_ranker_bias_softmax_experiment_config.json

processed/
patient_level_ranker_bias_free_experiment_history.csv
patient_level_ranker_bias_free_experiment_metrics.json
patient_ranker_bias_ablation.csv
patient_ranker_bias_ablation_summary.json
patient_ranker_bias_softmax_experiment_history.csv
patient_ranker_bias_softmax_experiment_metrics.json
patient_ranker_bias_softmax_experiment_validation.csv
patient_ranker_diagnostics.json
patient_ranker_vs_baseline

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = [
    "patient_ranker_bias_softmax_experiment_metrics.json",
    "patient_ranker_vs_baseline_paired_bootstrap.json",
    "patient_ranker_bias_ablation_summary.json",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 60}")
    print(name)

    if path.exists():
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps(data, indent=2))
        except Exception as e:
            print("Could not read JSON:", e)
    else:
        print("File not found.")

print("\n=== SOFTMAX VALIDATION PREDICTIONS ===")

path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"

if path.exists():
    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(5).to_string(index=False))
else:
    print("Validation predictions file not found.")

print("\nInspection complete. No files modified.")


patient_ranker_bias_softmax_experiment_metrics.json
{
  "cases": 906,
  "mrr": 0.3341009854885823,
  "top1": 0.30132450331125826,
  "top5": 0.3631346578366446,
  "top10": 0.39514348785871967
}

patient_ranker_vs_baseline_paired_bootstrap.json
{
  "matched_validation_cases": 906,
  "bootstrap_replicates": 3000,
  "seed": 42,
  "metrics": {
    "Top1": {
      "baseline": 0.2947,
      "ranker_no_bias": 0.2671,
      "difference_ranker_minus_baseline": -0.0276,
      "bootstrap_95pct_CI_difference": [
        -0.0717,
        0.0188
      ]
    },
    "Top5": {
      "baseline": 0.468,
      "ranker_no_bias": 0.3444,
      "difference_ranker_minus_baseline": -0.1236,
      "bootstrap_95pct_CI_difference": [
        -0.1744,
        -0.0717
      ]
    },
    "Top10": {
      "baseline": 0.5419,
      "ranker_no_bias": 0.3653,
      "difference_ranker_minus_baseline": -0.1766,
      "bootstrap_95pct_CI_difference": [
        -0.2285,
        -0.1247
      ]
    },
    "MRR": {
      "bas

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
df = pd.read_csv(path)

print("=== SOFTMAX METRIC VERIFICATION ===")
print("Rows:", len(df))
print("Unique cases:", df["case_id"].nunique())
print("Missing ranks:", df["rank"].isna().sum())
print("Minimum rank:", df["rank"].min())
print("Maximum rank:", df["rank"].max())

# Recompute metrics from target ranks.
ranks = pd.to_numeric(df["rank"], errors="coerce")
valid = ranks.notna() & (ranks >= 1)

print("\nValid ranked cases:", int(valid.sum()))

if valid.any():
    r = ranks[valid]
    print("Recomputed MRR:", round((1 / r).mean(), 6))
    print("Recomputed Top-1:", round((r <= 1).mean(), 6))
    print("Recomputed Top-5:", round((r <= 5).mean(), 6))
    print("Recomputed Top-10:", round((r <= 10).mean(), 6))

print("\n=== RANK DISTRIBUTION ===")
print(
    pd.cut(
        ranks,
        bins=[0, 1, 5, 10, 50, 100, 1000, float("inf")],
        labels=["1", "2–5", "6–10", "11–50", "51–100", "101–1000", ">1000"]
    ).value_counts(sort=False).to_string()
)

print("\nNo files modified.")

=== SOFTMAX METRIC VERIFICATION ===
Rows: 906
Unique cases: 906
Missing ranks: 0
Minimum rank: 1
Maximum rank: 7193

Valid ranked cases: 906
Recomputed MRR: 0.334101
Recomputed Top-1: 0.301325
Recomputed Top-5: 0.363135
Recomputed Top-10: 0.395143

=== RANK DISTRIBUTION ===
rank
1           273
2–5          56
6–10         29
11–50        41
51–100       32
101–1000     65
>1000       410

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

softmax = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)

baseline = pd.read_csv(
    DATA / "patient_validation_matched_candidate_comparison.csv"
)

print("=== CASE AND TARGET ALIGNMENT ===")
print("Softmax cases:", softmax["case_id"].nunique())
print("Baseline comparison cases:", baseline["case_id"].nunique())

softmax_cases = set(softmax["case_id"])
baseline_cases = set(baseline["case_id"])

print("Cases in both:", len(softmax_cases & baseline_cases))
print("Only in softmax:", len(softmax_cases - baseline_cases))
print("Only in baseline:", len(baseline_cases - softmax_cases))

print("\n=== TARGET LABEL ALIGNMENT ===")
if "diagnosis_id" in softmax.columns and "diagnosis_ids" in baseline.columns:
    merged = softmax.merge(
        baseline[["case_id", "diagnosis_ids"]],
        on="case_id",
        how="inner",
        validate="one_to_one"
    )

    matches = (
        merged["diagnosis_id"].astype(str)
        == merged["diagnosis_ids"].astype(str)
    )

    print("Target labels matching:", int(matches.sum()), "/", len(merged))
    print("Target labels mismatching:", int((~matches).sum()))
    if (~matches).any():
        print(merged.loc[~matches, ["case_id", "diagnosis_id", "diagnosis_ids"]].head(10))
else:
    print("Expected target columns not found; inspect column names.")

print("\nNo files modified.")

=== CASE AND TARGET ALIGNMENT ===
Softmax cases: 906
Baseline comparison cases: 906
Cases in both: 906
Only in softmax: 0
Only in baseline: 0

=== TARGET LABEL ALIGNMENT ===
Target labels matching: 906 / 906
Target labels mismatching: 0

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

softmax = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)

baseline = pd.read_csv(
    DATA / "patient_validation_matched_candidate_comparison.csv"
)

df = softmax.merge(
    baseline[[
        "case_id",
        "baseline_common_rank",
        "ranker_common_rank",
        "target_in_common_vocab"
    ]],
    on="case_id",
    how="inner",
    validate="one_to_one"
)

print("=== PAIRED RANK COMPARISON ===")
print("Matched cases:", len(df))

for col in [
    "rank",
    "baseline_common_rank",
    "ranker_common_rank"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

def metrics(ranks):
    ranks = pd.Series(ranks).dropna()
    return {
        "cases": len(ranks),
        "MRR": (1 / ranks).mean(),
        "Top1": (ranks <= 1).mean(),
        "Top5": (ranks <= 5).mean(),
        "Top10": (ranks <= 10).mean(),
        "Median rank": ranks.median()
    }

summary = pd.DataFrame({
    "Phenotype baseline": metrics(df["baseline_common_rank"]),
    "Original ranker": metrics(df["ranker_common_rank"]),
    "Softmax ranker": metrics(df["rank"])
}).T

display(summary.round(4))

print("\n=== PER-CASE MRR WIN / TIE COUNTS ===")
baseline_rr = 1 / df["baseline_common_rank"]
softmax_rr = 1 / df["rank"]

print("Softmax wins:", int((softmax_rr > baseline_rr).sum()))
print("Ties:", int((softmax_rr == baseline_rr).sum()))
print("Baseline wins:", int((baseline_rr > softmax_rr).sum()))

print("\nNo files modified.")

=== PAIRED RANK COMPARISON ===
Matched cases: 906


,cases,MRR,Top1,Top5,Top10,Median rank
Phenotype baseline,906.0,0.3773,0.2947,0.4680,0.5419,8.0
Original ranker,906.0,0.0527,0.0210,0.0872,0.1104,450.0
Softmax ranker,906.0,0.3341,0.3013,0.3631,0.3951,152.0



=== PER-CASE MRR WIN / TIE COUNTS ===
Softmax wins: 334
Ties: 32
Baseline wins: 540

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "patient_ranker_bias_softmax_experiment_history.csv",
    "patient_level_ranker_bias_free_experiment_history.csv",
]

for name in files:
    path = DATA / name
    print(f"\n=== {name} ===")

    if not path.exists():
        print("File not found.")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))
    display(df.tail(5))

print("\nRead-only inspection complete.")


=== patient_ranker_bias_softmax_experiment_history.csv ===
Shape: (19, 7)
Columns: ['epoch', 'train_loss', 'val_cases', 'val_mrr', 'val_top1', 'val_top5', 'val_top10']


,epoch,train_loss,val_cases,val_mrr,val_top1,val_top5,val_top10
0,1,5.255307,906,0.050934,0.032009,0.059603,0.088300
1,2,3.617910,906,0.135523,0.086093,0.184327,0.216336
2,3,2.758400,906,0.230470,0.196468,0.268212,0.290287


,epoch,train_loss,val_cases,val_mrr,val_top1,val_top5,val_top10
14,15,0.392794,906,0.334101,0.301325,0.363135,0.395143
15,16,0.356568,906,0.315388,0.272627,0.363135,0.391832
16,17,0.337070,906,0.324764,0.289183,0.357616,0.390728
17,18,0.311816,906,0.316394,0.277042,0.354305,0.388521
18,19,0.298530,906,0.334063,0.303532,0.371965,0.397351



=== patient_level_ranker_bias_free_experiment_history.csv ===
Shape: (8, 6)
Columns: ['epoch', 'train_loss', 'validation_mrr', 'validation_top1', 'validation_top5', 'validation_top10']


,epoch,train_loss,validation_mrr,validation_top1,validation_top5,validation_top10
0,1,0.143621,0.012548,0.0,0.014349,0.023179
1,2,0.065830,0.020924,0.0,0.030905,0.064018
2,3,0.057449,0.013261,0.0,0.016556,0.046358


,epoch,train_loss,validation_mrr,validation_top1,validation_top5,validation_top10
3,4,0.054256,0.014941,0.0,0.019868,0.039735
4,5,0.050561,0.019001,0.0,0.030905,0.064018
5,6,0.050292,0.013320,0.0,0.022075,0.033113
6,7,0.049275,0.012305,0.0,0.011038,0.013245
7,8,0.048671,0.011357,0.0,0.011038,0.019868



Read-only inspection complete.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
path = DATA / "patient_ranker_bias_softmax_experiment_history.csv"

df = pd.read_csv(path)

print("=== SOFTMAX TRAINING HISTORY ===")
print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== BEST EPOCH BY VALIDATION METRIC ===")
for metric in ["val_mrr", "val_top1", "val_top5", "val_top10"]:
    if metric in df.columns:
        idx = df[metric].idxmax()
        print(
            f"{metric}: epoch {int(df.loc[idx, 'epoch'])}, "
            f"value={df.loc[idx, metric]:.4f}"
        )

print("\n=== TRAIN LOSS TREND ===")
print(
    "First epoch:",
    round(float(df["train_loss"].iloc[0]), 4),
    "| Last epoch:",
    round(float(df["train_loss"].iloc[-1]), 4)
)

print("\nRead-only inspection complete.")

=== SOFTMAX TRAINING HISTORY ===
 epoch  train_loss  val_cases  val_mrr  val_top1  val_top5  val_top10
     1      5.2553        906   0.0509    0.0320    0.0596     0.0883
     2      3.6179        906   0.1355    0.0861    0.1843     0.2163
     3      2.7584        906   0.2305    0.1965    0.2682     0.2903
     4      2.0961        906   0.2340    0.1876    0.2925     0.3245
     5      1.6358        906   0.2793    0.2362    0.3256     0.3532
     6      1.3340        906   0.3045    0.2759    0.3311     0.3587
     7      1.1126        906   0.3162    0.2848    0.3477     0.3786
     8      0.9410        906   0.3110    0.2704    0.3521     0.3797
     9      0.7855        906   0.3219    0.2903    0.3565     0.3819
    10      0.6863        906   0.2932    0.2439    0.3499     0.3863
    11      0.6040        906   0.3192    0.2815    0.3620     0.3863
    12      0.5421        906   0.3230    0.2903    0.3598     0.3830
    13      0.4920        906   0.3041    0.2627    0.353

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "patient_ranker_bias_softmax_experiment_validation.csv",
    "patient_validation_matched_candidate_comparison.csv",
    "patient_validation_score_combination.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 65}\n{name}")

    if not path.exists():
        print("File not found")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

print("\nRead-only inspection complete.")


patient_ranker_bias_softmax_experiment_validation.csv
Shape: (906, 7)
Columns: ['case_id', 'diagnosis_id', 'rank', 'top1', 'top5', 'top10', 'mrr']
                                       case_id diagnosis_id  rank  top1  top5  top10      mrr
                       PMID_28830446_Patient_2  OMIM:615688   548     0     0      0 0.001825
                       PMID_28830446_Patient_1  OMIM:615688   545     0     0      0 0.001835
PMID_32480058_Case_F5P1_original_pedigree_II_2  OMIM:618052  6971     0     0      0 0.000143

patient_validation_matched_candidate_comparison.csv
Shape: (906, 11)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_observed_hpo', 'baseline_native_rank', 'ranker_native_rank', 'baseline_common_rank', 'ranker_common_rank', 'target_in_baseline_vocab', 'target_in_ranker_vocab', 'target_in_common_vocab']
                                       case_id     pmid diagnosis_ids  n_observed_hpo  baseline_native_rank  ranker_native_rank  baseline_common_rank  ranker_common_rank 

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

keywords = (
    "ranker",
    "candidate",
    "vocab",
    "mapping",
    "evaluation",
    "validation",
    "checkpoint",
)

print("=== RELEVANT PROJECT FILES ===")

for folder in [PROJECT / "model", PROJECT / "processed"]:
    if not folder.exists():
        continue

    print(f"\n[{folder.name}]")

    for path in sorted(folder.iterdir()):
        if path.is_file() and any(
            word in path.name.lower() for word in keywords
        ):
            print(f"{path.name} ({path.stat().st_size:,} bytes)")

print("\n=== MODEL DIRECTORY PYTHON FILES ===")
for path in sorted((PROJECT / "model").glob("*.py")):
    print(path.name)

print("\nRead-only file inventory complete.")

=== RELEVANT PROJECT FILES ===

[model]
patient_level_ranker_best.pt (9,387,579 bytes)
patient_level_ranker_bias_free_experiment.pt (9,357,805 bytes)
patient_level_ranker_bias_free_experiment_config.json (303 bytes)
patient_level_ranker_config.json (301 bytes)
patient_ranker_bias_softmax_experiment.pt (9,387,364 bytes)
patient_ranker_bias_softmax_experiment_config.json (249 bytes)

[processed]
app_predictor_validation_diagnostic.csv (47,216 bytes)
candidate_coverage_audit.csv (40,147 bytes)
evaluation_eligibility_audit.csv (58,872 bytes)
evaluation_protocol_checks.csv (220 bytes)
gnn_annotations_publication_disjoint_candidate.csv (20,694,622 bytes)
gnn_checkpoint_compatibility.csv (157 bytes)
gnn_evaluation_results.csv (295 bytes)
leakage_reduced_retrieval_evaluation.csv (53,952 bytes)
missing_evaluation_profiles.csv (157 bytes)
normalized_disease_gene_mapping.csv (230,797 bytes)
ontology_fallback_candidates.csv (1,408 bytes)
ontology_fallback_evaluation.csv (178 bytes)
patient_level_r

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "evaluation_protocol_checks.csv",
    "evaluation_eligibility_audit.csv",
    "patient_validation_matched_candidate_summary.json",
    "leakage_reduced_retrieval_evaluation.csv",
    "patient_ranker_bias_softmax_experiment_metrics.json",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 65}\n{name}")

    if not path.exists():
        print("File not found")
        continue

    if path.suffix == ".json":
        print(path.read_text())
    else:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(8).to_string(index=False))

print("\nRead-only audit complete.")


evaluation_protocol_checks.csv
Shape: (6, 2)
Columns: ['Check', 'Value']
                                         Check  Value
                          Input phenotype rows  21647
                  Hidden target phenotype rows  22562
                     Input/target pair overlap      0
                        Training diseases used   7340
           Held-out diseases in training index      0
Profiles with both input and target phenotypes   1832

evaluation_eligibility_audit.csv
Shape: (1836, 5)
Columns: ['disease_id', 'input_phenotype_count', 'target_phenotype_count', 'in_metrics', 'status']
 disease_id  input_phenotype_count  target_phenotype_count  in_metrics    status
OMIM:100100                      8                       8        True evaluated
OMIM:101900                      4                       5        True evaluated
OMIM:102530                      2                       2        True evaluated
OMIM:102900                      1                       2        True eva

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "candidate_coverage_audit.csv",
    "patient_ranker_diagnostics.json",
    "patient_ranker_bias_ablation_summary.json",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 60}\n{name}")

    if not path.exists():
        print("File not found")
        continue

    if path.suffix == ".json":
        print(path.read_text())
    else:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(10).to_string(index=False))
        print("\nData types:")
        print(df.dtypes.to_string())

print("\nRead-only inspection complete.")



candidate_coverage_audit.csv
Shape: (1832, 4)
Columns: ['disease_id', 'input_count', 'covered_input_count', 'candidate_count']
 disease_id  input_count  covered_input_count  candidate_count
OMIM:100100            8                    8             7375
OMIM:101900            4                    4             4511
OMIM:102530            2                    2             6539
OMIM:102900            1                    0                0
OMIM:103500            4                    4             1021
OMIM:103780            1                    1              157
OMIM:104290            5                    5             6335
OMIM:105150            3                    3             5797
OMIM:105200            7                    7             6792
OMIM:105250            3                    3             4729

Data types:
disease_id             object
input_count             int64
covered_input_count     int64
candidate_count         int64

patient_ranker_diagnostics.json
{
  "best_epo

In [ ]:
from pathlib import Path
import pandas as pd

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/processed/"
    "patient_ranker_bias_ablation.csv"
)

df = pd.read_csv(path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head().to_string(index=False))

print("\nData types:")
print(df.dtypes.to_string())

print("\nMissing values:")
print(df.isna().sum().to_string())

print("\nRead-only inspection complete.")

Shape: (906, 8)

Columns:
['case_id', 'diagnosis_ids', 'original_rank', 'original_top1_disease', 'no_bias_rank', 'no_bias_top1_disease', 'bias_only_rank', 'bias_only_top1_disease']

First 5 rows:
                                       case_id diagnosis_ids  original_rank original_top1_disease  no_bias_rank no_bias_top1_disease  bias_only_rank bias_only_top1_disease
                       PMID_28830446_Patient_2   OMIM:615688           2831           OMIM:148050          3557          OMIM:214800            2835            OMIM:612164
                       PMID_28830446_Patient_1   OMIM:615688           2711           OMIM:612164           979          OMIM:621500            2835            OMIM:612164
PMID_32480058_Case_F5P1_original_pedigree_II_2   OMIM:618052           2188           OMIM:148050          2212          OMIM:615278            2212            OMIM:612164
PMID_32480058_Case_F4P2_original_pedigree_IV_5   OMIM:618052           2170           OMIM:612164          1652     

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

softmax = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)
ablation = pd.read_csv(
    DATA / "patient_ranker_bias_ablation.csv"
)

print("=== CASE ALIGNMENT AUDIT ===")
print("Softmax rows:", len(softmax))
print("Ablation rows:", len(ablation))
print("Softmax unique cases:", softmax["case_id"].nunique())
print("Ablation unique cases:", ablation["case_id"].nunique())

merged = softmax.merge(
    ablation[["case_id", "diagnosis_ids"]],
    on="case_id",
    how="outer",
    indicator=True,
)

print("\nMerge status:")
print(merged["_merge"].value_counts().to_string())

matched = merged[merged["_merge"] == "both"]
print(
    "\nMatched target diagnoses:",
    (matched["diagnosis_id"] == matched["diagnosis_ids"]).sum(),
    "/",
    len(matched),
)

print("\nRead-only audit complete.")

=== CASE ALIGNMENT AUDIT ===
Softmax rows: 906
Ablation rows: 906
Softmax unique cases: 906
Ablation unique cases: 906

Merge status:
_merge
both          906
left_only       0
right_only      0

Matched target diagnoses: 906 / 906

Read-only audit complete.


In [ ]:
from pathlib import Path
import json

MODEL_DIR = Path("/content/drive/MyDrive/Rare Disease ai/model")

files = [
    "patient_ranker_bias_softmax_experiment_config.json",
    "patient_level_ranker_config.json",
    "patient_level_ranker_bias_free_experiment_config.json",
]

for name in files:
    path = MODEL_DIR / name
    print(f"\n=== {name} ===")

    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2))
    else:
        print("File not found")

print("\nRead-only inspection complete.")


=== patient_ranker_bias_softmax_experiment_config.json ===
{
  "seed": 42,
  "hidden_dim": 128,
  "learning_rate": 0.0003,
  "weight_decay": 0.0001,
  "epochs_run": 19,
  "best_epoch": 15,
  "patience": 4,
  "objective": "single-label full-disease cross-entropy",
  "disease_bias": true,
  "device": "cuda"
}

=== patient_level_ranker_config.json ===
{
  "model": "PatientDiseaseRanker",
  "dimension": 128,
  "dropout": 0.2,
  "seed": 42,
  "training_cases": 5730,
  "validation_cases": 906,
  "training_objective": "patient-to-disease ranking with sampled competing diseases",
  "device": "cuda",
  "test_set_used_for_training_or_validation": false
}

=== patient_level_ranker_bias_free_experiment_config.json ===
{
  "model": "BiasFreePatientRanker",
  "seed": 42,
  "epochs_run": 8,
  "best_validation_mrr": 0.020924354840425125,
  "device": "cuda",
  "training_seconds": 154.2,
  "training_file": "patient_level_train.csv",
  "validation_file": "patient_level_validation.csv",
  "original_ranke

In [ ]:
from pathlib import Path

MODEL_DIR = Path("/content/drive/MyDrive/Rare Disease ai/model")

for name in ["rare_disease_ai.py", "rare_disease_data_loader.py"]:
    path = MODEL_DIR / name
    print(f"\n{'=' * 65}\n{name}")

    if not path.exists():
        print("File not found")
        continue

    lines = path.read_text().splitlines()

    for i, line in enumerate(lines, start=1):
        if any(term in line.lower() for term in [
            "disease_map",
            "disease_to_idx",
            "disease2idx",
            "disease_names",
            "disease_ids",
            "candidate",
            "vocab",
            "state_dict",
            "checkpoint",
            "torch.load",
        ]):
            start = max(1, i - 2)
            end = min(len(lines), i + 3)
            print(f"\n--- lines {start}-{end} ---")
            for j in range(start, end + 1):
                print(f"{j:4}: {lines[j - 1]}")

print("\nRead-only source inspection complete.")


rare_disease_ai.py

--- lines 6-11 ---
   6: GENE_MAP = _DATA["gene_map"]
   7: IC = _DATA["ic"]
   8: DISEASE_NAMES = _DATA["disease_names"]
   9: 
  10: def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
  11:                      gene_weight=0.25):

--- lines 37-42 ---
  37:         gene_score = len(matching_genes) / len(genes) if genes else None
  38: 
  39:         # Do not penalize candidates when no patient gene matches.
  40:         if genes and matching_genes:
  41:             combined_score = (
  42:                 (1 - gene_weight) * phenotype_score

--- lines 48-53 ---
  48:         results.append({
  49:             "disease_id": disease,
  50:             "disease_name": DISEASE_NAMES.get(
  51:                 disease, "Disease name unavailable"
  52:             ),
  53:             "phenotype_score": phenotype_score,

rare_disease_data_loader.py

--- lines 53-58 ---
  53:     }
  54: 
  55:     disease_names = (
  56:         ann.dropna(subset=["

In [ ]:
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/"
    "rare_disease_data_loader.py"
)

lines = path.read_text().splitlines()

print("=== DATA LOADER SOURCE ===")
for i, line in enumerate(lines, start=1):
    print(f"{i:4}: {line}")

print("\nRead-only source inspection complete.")


=== DATA LOADER SOURCE ===
   1: 
   2: from pathlib import Path
   3: from collections import defaultdict
   4: import numpy as np
   5: import pandas as pd
   6: 
   7: PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
   8: DATA = PROJECT / "processed"
   9: 
  10: def load_prediction_data():
  11:     ann = pd.read_csv(
  12:         DATA / "clean_hpo_annotations.csv",
  13:         low_memory=False,
  14:         dtype={"database_id": "string", "hpo_id": "string"}
  15:     ).dropna(subset=["database_id", "hpo_id"])
  16: 
  17:     ann["database_id"] = ann["database_id"].str.strip()
  18:     ann["hpo_id"] = ann["hpo_id"].str.strip()
  19: 
  20:     phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
  21: 
  22:     gene_df = pd.read_csv(
  23:         DATA / "normalized_disease_gene_mapping.csv",
  24:         dtype=str
  25:     ).fillna("")
  26: 
  27:     gene_map = {
  28:         row["disease_id"]: {
  29:             g.strip().upper()
  30: 

In [ ]:
from pathlib import Path
import torch

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/"
    "patient_ranker_bias_softmax_experiment.pt"
)

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dict with {len(value)} entries")
        elif isinstance(value, (list, tuple)):
            print(f"{key}: {type(value).__name__} with {len(value)} items")
        elif hasattr(value, "shape"):
            print(f"{key}: tensor/array shape {tuple(value.shape)}")
        else:
            print(f"{key}: {str(value)[:200]}")

    state = checkpoint.get("model_state_dict")
    if state is None:
        state = checkpoint.get("state_dict")

    if isinstance(state, dict):
        print("\nState-dict parameter names and shapes:")
        for key, value in state.items():
            if hasattr(value, "shape"):
                print(f"{key}: {tuple(value.shape)}")
            else:
                print(f"{key}: {type(value).__name__}")
    elif all(hasattr(v, "shape") for v in checkpoint.values()):
        print("\nCheckpoint appears to be a raw state dictionary.")
        for key, value in checkpoint.items():
            print(f"{key}: {tuple(value.shape)}")
else:
    print("No dictionary metadata available.")

print("\nRead-only checkpoint inspection complete.")

Checkpoint type: dict

Top-level keys:
model_state_dict: dict with 7 entries
disease_ids: list with 7340 items
phenotype_ids: list with 9982 items
hidden_dim: 128
best_epoch: 15
best_validation_metrics: dict with 5 entries
seed: 42
training_cases: 5730
validation_cases: 906
objective: single-label full-disease cross-entropy
disease_bias: True
note: Research experiment; not clinically validated.

State-dict parameter names and shapes:
disease_bias: (7340,)
hpo_embedding.weight: (9982, 128)
disease_embedding.weight: (7340, 128)
patient_projection.0.weight: (128, 128)
patient_projection.0.bias: (128,)
patient_projection.3.weight: (128, 128)
patient_projection.3.bias: (128,)

Read-only checkpoint inspection complete.


In [ ]:
from pathlib import Path
import torch

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/"
    "patient_ranker_bias_softmax_experiment.pt"
)

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dict with {len(value)} entries")
        elif isinstance(value, (list, tuple)):
            print(f"{key}: {type(value).__name__} with {len(value)} items")
        elif hasattr(value, "shape"):
            print(f"{key}: tensor/array shape {tuple(value.shape)}")
        else:
            print(f"{key}: {str(value)[:200]}")

    state = checkpoint.get("model_state_dict")
    if state is None:
        state = checkpoint.get("state_dict")

    if isinstance(state, dict):
        print("\nState-dict parameter names and shapes:")
        for key, value in state.items():
            if hasattr(value, "shape"):
                print(f"{key}: {tuple(value.shape)}")
            else:
                print(f"{key}: {type(value).__name__}")
    elif all(hasattr(v, "shape") for v in checkpoint.values()):
        print("\nCheckpoint appears to be a raw state dictionary.")
        for key, value in checkpoint.items():
            print(f"{key}: {tuple(value.shape)}")
else:
    print("No dictionary metadata available.")

print("\nRead-only checkpoint inspection complete."22)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1988134980.py, line 47)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import torch

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/"
    "patient_ranker_bias_softmax_experiment.pt"
)

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dict with {len(value)} entries")
        elif isinstance(value, (list, tuple)):
            print(f"{key}: {type(value).__name__} with {len(value)} items")
        elif hasattr(value, "shape"):
            print(f"{key}: tensor/array shape {tuple(value.shape)}")
        else:
            print(f"{key}: {str(value)[:200]}")

    state = checkpoint.get("model_state_dict")
    if state is None:
        state = checkpoint.get("state_dict")

    if isinstance(state, dict):
        print("\nState-dict parameter names and shapes:")
        for key, value in state.items():
            if hasattr(value, "shape"):
                print(f"{key}: {tuple(value.shape)}")
            else:
                print(f"{key}: {type(value).__name__}")
    elif all(hasattr(v, "shape") for v in checkpoint.values()):
        print("\nCheckpoint appears to be a raw state dictionary.")
        for key, value in checkpoint.items():
            print(f"{key}: {tuple(value.shape)}")
else:
    print("No dictionary metadata available.")

print("\nRead-only checkpoint inspection complete.")

Checkpoint type: dict

Top-level keys:
model_state_dict: dict with 7 entries
disease_ids: list with 7340 items
phenotype_ids: list with 9982 items
hidden_dim: 128
best_epoch: 15
best_validation_metrics: dict with 5 entries
seed: 42
training_cases: 5730
validation_cases: 906
objective: single-label full-disease cross-entropy
disease_bias: True
note: Research experiment; not clinically validated.

State-dict parameter names and shapes:
disease_bias: (7340,)
hpo_embedding.weight: (9982, 128)
disease_embedding.weight: (7340, 128)
patient_projection.0.weight: (128, 128)
patient_projection.0.bias: (128,)
patient_projection.3.weight: (128, 128)
patient_projection.3.bias: (128,)

Read-only checkpoint inspection complete.


In [ ]:
from pathlib import Path
import torch

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/"
    "patient_ranker_bias_softmax_experiment.pt"
)

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

disease_ids = checkpoint["disease_ids"]
phenotype_ids = checkpoint["phenotype_ids"]
state = checkpoint["model_state_dict"]

print("=== CHECKPOINT MAPPING AUDIT ===")
print("Disease IDs:", len(disease_ids))
print("Disease IDs unique:", len(set(disease_ids)))
print("Disease embedding rows:", state["disease_embedding.weight"].shape[0])

print("\nPhenotype IDs:", len(phenotype_ids))
print("Phenotype IDs unique:", len(set(phenotype_ids)))
print("Phenotype embedding rows:", state["hpo_embedding.weight"].shape[0])

print("\nSample disease IDs:", disease_ids[:10])
print("Sample phenotype IDs:", phenotype_ids[:10])

assert len(disease_ids) == state["disease_embedding.weight"].shape[0]
assert len(set(disease_ids)) == len(disease_ids)
assert len(phenotype_ids) == state["hpo_embedding.weight"].shape[0]
assert len(set(phenotype_ids)) == len(phenotype_ids)

print("\nMapping audit passed.")

=== CHECKPOINT MAPPING AUDIT ===
Disease IDs: 7340
Disease IDs unique: 7340
Disease embedding rows: 7340

Phenotype IDs: 9982
Phenotype IDs unique: 9982
Phenotype embedding rows: 9982

Sample disease IDs: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400', 'OMIM:101600', 'OMIM:101800', 'OMIM:102200', 'OMIM:102370', 'OMIM:102500']
Sample phenotype IDs: ['HP:0000002', 'HP:0000003', 'HP:0000006', 'HP:0000007', 'HP:0000008', 'HP:0000009', 'HP:0000010', 'HP:0000011', 'HP:0000012', 'HP:0000013']

Mapping audit passed.


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

# Load the saved disease vocabulary from the softmax checkpoint
checkpoint = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)
ranker_diseases = set(checkpoint["disease_ids"])

# Load the phenotype profiles used by the baseline
annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    usecols=["database_id", "hpo_id"],
    dtype={"database_id": "string", "hpo_id": "string"},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

baseline_diseases = set(
    annotations["database_id"].str.strip().unique()
)

shared = ranker_diseases & baseline_diseases
ranker_only = ranker_diseases - baseline_diseases
baseline_only = baseline_diseases - ranker_diseases

print("=== DISEASE VOCABULARY OVERLAP ===")
print("Softmax ranker diseases:", len(ranker_diseases))
print("Baseline diseases with phenotype annotations:", len(baseline_diseases))
print("Shared disease IDs:", len(shared))
print("Ranker-only disease IDs:", len(ranker_only))
print("Baseline-only disease IDs:", len(baseline_only))

print("\nSample ranker-only IDs:", sorted(ranker_only)[:10])
print("Sample baseline-only IDs:", sorted(baseline_only)[:10])

print("\nRead-only audit complete.")

=== DISEASE VOCABULARY OVERLAP ===
Softmax ranker diseases: 7340
Baseline diseases with phenotype annotations: 12879
Shared disease IDs: 7340
Ranker-only disease IDs: 0
Baseline-only disease IDs: 5539

Sample ranker-only IDs: []
Sample baseline-only IDs: ['DECIPHER:1', 'DECIPHER:14', 'DECIPHER:15', 'DECIPHER:16', 'DECIPHER:17', 'DECIPHER:18', 'DECIPHER:19', 'DECIPHER:2', 'DECIPHER:20', 'DECIPHER:21']

Read-only audit complete.


In [ ]:

from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/"
    "rare_disease_ai.py"
)

print(path.read_text())


from rare_disease_data_loader import load_prediction_data

_DATA = load_prediction_data()
PHENOTYPE_MAP = _DATA["phenotype_map"]
GENE_MAP = _DATA["gene_map"]
IC = _DATA["ic"]
DISEASE_NAMES = _DATA["disease_names"]

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    observed = {
        str(t).strip() for t in patient_hpo_terms
        if str(t).strip() in IC
    }
    if not observed:
        raise ValueError("No recognized HPO terms were supplied.")

    weights = {t: IC[t] for t in observed}
    denominator = sum(weights.values()) or 1.0
    genes = {
        str(g).strip().upper() for g in (patient_genes or [])
        if str(g).strip()
    }

    results = []
    for disease, disease_terms in PHENOTYPE_MAP.items():
        overlap = observed & disease_terms
        precision = sum(weights[t] for t in overlap) / denominator
        recall = len(overlap) / max(len(disease_terms), 1)
        phenotype_score = (
          

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

path = DATA / "patient_level_validation.csv"
df = pd.read_csv(path, dtype=str)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3).to_string(index=False))

print("\nMissing values:")
print(df[["case_id", "diagnosis_ids", "hpo_ids"]].isna().sum().to_string())

print("\nHPO field examples:")
print(df["hpo_ids"].head(5).tolist())

print("\nRead-only inspection complete.")

Shape: (906, 6)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']

First 3 rows:
                                       case_id     pmid diagnosis_ids                                                                                                                                                                                    hpo_ids n_hpo n_diagnoses
                       PMID_28830446_Patient_2 28830446   OMIM:615688                                  HP:0001954;HP:0002027;HP:0002140;HP:0002301;HP:0003493;HP:0005197;HP:0010702;HP:0011107;HP:0025528;HP:0030783;HP:0030880;HP:0033260;HP:0033833;HP:0034056    14           1
                       PMID_28830446_Patient_1 28830446   OMIM:615688 HP:0000623;HP:0001954;HP:0002027;HP:0002140;HP:0002301;HP:0002315;HP:0002321;HP:0002829;HP:0003493;HP:0005197;HP:0011107;HP:0011227;HP:0012246;HP:0025343;HP:0030783;HP:0033505;HP:0033833    17           1
PMID_32480058_Case_F5P1_original_pedigree_II_2 32480058   OMIM:

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import numpy as np
from collections import defaultdict

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

# Load checkpoint vocabularies
checkpoint = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

ranker_diseases = set(checkpoint["disease_ids"])
ranker_phenotypes = set(checkpoint["phenotype_ids"])

# Load annotation profiles
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    usecols=["database_id", "hpo_id"],
    dtype={"database_id": "string", "hpo_id": "string"},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

# Recreate IC weights using the project's training disease split
split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"]
)

counts = defaultdict(int)
for disease in train_ids:
    for term in phenotype_map.get(disease, set()):
        counts[term] += 1

ic = {
    term: -np.log(count / max(len(train_ids), 1))
    for term, count in counts.items()
    if count > 0
}

validation = pd.read_csv(
    DATA / "patient_level_validation.csv",
    dtype=str
)

profiled_candidates = ranker_diseases & set(phenotype_map)
unprofiled_candidates = ranker_diseases - set(phenotype_map)

observed_terms = {
    term.strip()
    for value in validation["hpo_ids"]
    for term in value.split(";")
    if term.strip()
}

print("=== SAME-CANDIDATE EVALUATION PREFLIGHT ===")
print("Validation cases:", len(validation))
print("Checkpoint diseases:", len(ranker_diseases))
print("Candidates with phenotype profiles:", len(profiled_candidates))
print("Candidates without phenotype profiles:", len(unprofiled_candidates))
print("Checkpoint phenotype IDs:", len(ranker_phenotypes))
print("Validation unique HPO terms:", len(observed_terms))
print("Validation HPO terms with IC weights:", len(observed_terms & set(ic)))
print("Validation HPO terms without IC weights:", len(observed_terms - set(ic)))
print("Target diagnoses absent from checkpoint:",
      sum(d not in ranker_diseases
          for d in validation["diagnosis_ids"]))

print("\nPreflight complete. No files modified.")

=== SAME-CANDIDATE EVALUATION PREFLIGHT ===
Validation cases: 906
Checkpoint diseases: 7340
Candidates with phenotype profiles: 7340
Candidates without phenotype profiles: 0
Checkpoint phenotype IDs: 9982
Validation unique HPO terms: 1366
Validation HPO terms with IC weights: 1366
Validation HPO terms without IC weights: 0
Target diagnoses absent from checkpoint: 0

Preflight complete. No files modified.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
terms = [
    "PatientDiseaseRanker",
    "BiasFreePatientRanker",
    "patient_projection",
    "disease_bias",
]

extensions = {".py", ".ipynb", ".txt"}

print("=== SEARCHING PROJECT SOURCE FILES ===")

found = 0
for path in PROJECT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue
    if any(part.startswith(".") for part in path.parts):
        continue

    try:
        lines = path.read_text(errors="ignore").splitlines()
    except OSError:
        continue

    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, start=1)
        if any(term.lower() in line.lower() for term in terms)
    ]

    if matches:
        found += 1
        print(f"\n--- {path.relative_to(PROJECT)} ---")
        for line_no, line in matches[:20]:
            print(f"{line_no}: {line[:200]}")

print(f"\nFiles containing matching terms: {found}")
print("Read-only search complete.")

=== SEARCHING PROJECT SOURCE FILES ===

Files containing matching terms: 0
Read-only search complete.


In [ ]:
from pathlib import Path
import torch

path = Path(
    "/content/drive/MyDrive/Rare Disease ai/model/"
    "patient_ranker_bias_softmax_experiment.pt"
)

ckpt = torch.load(path, map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"]

print("=== SAVED MODEL STRUCTURE ===")
for name, tensor in state.items():
    print(f"{name}: shape={tuple(tensor.shape)}")

print("\n=== EMBEDDING AND BIAS CHECKS ===")
for name in [
    "hpo_embedding.weight",
    "disease_embedding.weight",
    "disease_bias",
]:
    tensor = state[name].float()
    print(
        f"{name}: "
        f"mean={tensor.mean().item():.6f}, "
        f"std={tensor.std().item():.6f}, "
        f"min={tensor.min().item():.6f}, "
        f"max={tensor.max().item():.6f}, "
        f"all_finite={torch.isfinite(tensor).all().item()}"
    )

print("\n=== CHECKPOINT METADATA ===")
for key in [
    "best_epoch",
    "best_validation_metrics",
    "objective",
    "disease_bias",
    "training_cases",
    "validation_cases",
    "note",
]:
    print(f"{key}: {ckpt.get(key)}")

print("\nRead-only inspection complete.")

=== SAVED MODEL STRUCTURE ===
disease_bias: shape=(7340,)
hpo_embedding.weight: shape=(9982, 128)
disease_embedding.weight: shape=(7340, 128)
patient_projection.0.weight: shape=(128, 128)
patient_projection.0.bias: shape=(128,)
patient_projection.3.weight: shape=(128, 128)
patient_projection.3.bias: shape=(128,)

=== EMBEDDING AND BIAS CHECKS ===
hpo_embedding.weight: mean=0.000044, std=0.071956, min=-0.833480, max=0.779088, all_finite=True
disease_embedding.weight: mean=-0.028740, std=0.410707, min=-1.607736, max=1.389406, all_finite=True
disease_bias: mean=-1.596143, std=0.344747, min=-1.765462, max=0.043093, all_finite=True

=== CHECKPOINT METADATA ===
best_epoch: 15
best_validation_metrics: {'cases': 906, 'mrr': 0.3341009854885823, 'top1': 0.30132450331125826, 'top5': 0.3631346578366446, 'top10': 0.39514348785871967}
objective: single-label full-disease cross-entropy
disease_bias: True
training_cases: 5730
validation_cases: 906
note: Research experiment; not clinically validated.



In [ ]:
import torch
import torch.nn as nn

print("=== LIVE MODEL DEFINITIONS ===")

for name, obj in list(globals().items()):
    if isinstance(obj, type) and issubclass(obj, nn.Module):
        if any(term in name.lower() for term in ["patient", "ranker", "disease"]):
            print(f"\nClass: {name}")
            try:
                import inspect
                print(inspect.getsource(obj))
            except (OSError, TypeError):
                print("Class exists, but source code is unavailable.")

print("\nInspection complete.")

=== LIVE MODEL DEFINITIONS ===

Inspection complete.


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
DATA = PROJECT / "processed"

files = [
    MODEL_DIR / "patient_ranker_bias_softmax_experiment_config.json",
    MODEL_DIR / "patient_level_ranker_config.json",
    DATA / "patient_ranker_bias_softmax_experiment_history.csv",
]

for path in files:
    print(f"\n=== {path.name} ===")

    if not path.exists():
        print("File not found")
        continue

    if path.suffix == ".json":
        print(json.dumps(json.loads(path.read_text()), indent=2))
    else:
        df = pd.read_csv(path)
        print("Columns:", df.columns.tolist())
        print("Last 5 epochs:")
        print(df.tail().to_string(index=False))

print("\nRead-only inspection complete.")



=== patient_ranker_bias_softmax_experiment_config.json ===
{
  "seed": 42,
  "hidden_dim": 128,
  "learning_rate": 0.0003,
  "weight_decay": 0.0001,
  "epochs_run": 19,
  "best_epoch": 15,
  "patience": 4,
  "objective": "single-label full-disease cross-entropy",
  "disease_bias": true,
  "device": "cuda"
}

=== patient_level_ranker_config.json ===
{
  "model": "PatientDiseaseRanker",
  "dimension": 128,
  "dropout": 0.2,
  "seed": 42,
  "training_cases": 5730,
  "validation_cases": 906,
  "training_objective": "patient-to-disease ranking with sampled competing diseases",
  "device": "cuda",
  "test_set_used_for_training_or_validation": false
}

=== patient_ranker_bias_softmax_experiment_history.csv ===
Columns: ['epoch', 'train_loss', 'val_cases', 'val_mrr', 'val_top1', 'val_top5', 'val_top10']
Last 5 epochs:
 epoch  train_loss  val_cases  val_mrr  val_top1  val_top5  val_top10
    15    0.392794        906 0.334101  0.301325  0.363135   0.395143
    16    0.356568        906 0.31538

In [ ]:
from pathlib import Path
from collections import defaultdict
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ---------------------------------------------------------
# 1. Paths and saved artifacts
# ---------------------------------------------------------
PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

checkpoint_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
validation_path = DATA / "patient_level_validation.csv"
annotation_path = DATA / "clean_hpo_annotations.csv"
split_path = DATA / "disease_level_split.csv"

checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)
state = checkpoint["model_state_dict"]

disease_ids = list(checkpoint["disease_ids"])
phenotype_ids = list(checkpoint["phenotype_ids"])
disease_to_idx = {d: i for i, d in enumerate(disease_ids)}
phenotype_to_idx = {h: i for i, h in enumerate(phenotype_ids)}

# ---------------------------------------------------------
# 2. Load validation cases and phenotype profiles
# ---------------------------------------------------------
validation = pd.read_csv(validation_path, dtype=str).fillna("")
annotations = pd.read_csv(
    annotation_path,
    usecols=["database_id", "hpo_id"],
    dtype={"database_id": "string", "hpo_id": "string"},
    low_memory=False,
).dropna(subset=["database_id", "hpo_id"])

annotations["database_id"] = annotations["database_id"].str.strip()
annotations["hpo_id"] = annotations["hpo_id"].str.strip()

phenotype_map = (
    annotations.groupby("database_id")["hpo_id"].apply(set).to_dict()
)

# Reconstruct IC exactly as in the existing data loader
split = pd.read_csv(split_path, dtype={"disease_id": "string"})
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"]
)
counts = defaultdict(int)

for disease in train_ids:
    for term in phenotype_map.get(disease, set()):
        counts[term] += 1

ic = {
    term: -np.log(count / max(len(train_ids), 1))
    for term, count in counts.items()
    if count > 0
}

# All 7,340 checkpoint diseases are candidates for BOTH methods
candidates = disease_ids
candidate_profiles = [phenotype_map.get(d, set()) for d in candidates]

# ---------------------------------------------------------
# 3. Reconstructed neural ranker (assumption-based)
# ---------------------------------------------------------
device = torch.device("cpu")
hpo_embedding = state["hpo_embedding.weight"].float()
disease_embedding = state["disease_embedding.weight"].float()
disease_bias = state["disease_bias"].float()

projection = nn.Sequential(
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 128),
)
projection[0].weight.data.copy_(state["patient_projection.0.weight"])
projection[0].bias.data.copy_(state["patient_projection.0.bias"])
projection[3].weight.data.copy_(state["patient_projection.3.weight"])
projection[3].bias.data.copy_(state["patient_projection.3.bias"])
projection.eval()

# ---------------------------------------------------------
# 4. Score every case with both methods
# ---------------------------------------------------------
results = []

for row in validation.itertuples(index=False):
    case_id = row.case_id
    target = row.diagnosis_ids.strip()
    observed_all = {
        term.strip() for term in row.hpo_ids.split(";") if term.strip()
    }

    # Match the baseline: only terms with IC weights contribute
    observed = observed_all & set(ic)
    if not observed:
        raise ValueError(f"No recognized phenotype terms for {case_id}")

    weights = {term: ic[term] for term in observed}
    denominator = sum(weights.values()) or 1.0

    phenotype_scores = np.zeros(len(candidates), dtype=np.float64)
    overlap_counts = np.zeros(len(candidates), dtype=np.int32)

    for j, profile in enumerate(candidate_profiles):
        overlap = observed & profile
        overlap_counts[j] = len(overlap)
        precision = sum(weights[t] for t in overlap) / denominator
        recall = len(overlap) / max(len(profile), 1)
        phenotype_scores[j] = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

    # Baseline tie-break: score, overlap count, disease ID
    baseline_order = sorted(
        range(len(candidates)),
        key=lambda j: (
            -phenotype_scores[j],
            -int(overlap_counts[j]),
            candidates[j],
        ),
    )

    # Reconstructed ranker: mean HPO embeddings -> projection
    hpo_indices = [
        phenotype_to_idx[t] for t in observed_all
        if t in phenotype_to_idx
    ]
    if not hpo_indices:
        raise ValueError(f"No checkpoint HPO embeddings for {case_id}")

    patient_vector = hpo_embedding[hpo_indices].mean(dim=0).unsqueeze(0)
    with torch.no_grad():
        patient_vector = projection(patient_vector).squeeze(0)
        ranker_scores = disease_embedding @ patient_vector + disease_bias

    ranker_scores = ranker_scores.numpy()
    ranker_order = sorted(
        range(len(candidates)),
        key=lambda j: (-float(ranker_scores[j]), candidates[j]),
    )

    if target not in disease_to_idx:
        raise ValueError(f"Target not in checkpoint vocabulary: {target}")

    baseline_rank = baseline_order.index(disease_to_idx[target]) + 1
    ranker_rank = ranker_order.index(disease_to_idx[target]) + 1

    results.append({
        "case_id": case_id,
        "diagnosis_id": target,
        "baseline_rank_7340": baseline_rank,
        "reconstructed_ranker_rank_7340": ranker_rank,
        "baseline_top1": int(baseline_rank == 1),
        "baseline_top5": int(baseline_rank <= 5),
        "baseline_top10": int(baseline_rank <= 10),
        "baseline_rr": 1.0 / baseline_rank,
        "reconstructed_ranker_top1": int(ranker_rank == 1),
        "reconstructed_ranker_top5": int(ranker_rank <= 5),
        "reconstructed_ranker_top10": int(ranker_rank <= 10),
        "reconstructed_ranker_rr": 1.0 / ranker_rank,
    })

result_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 5. Summarize metrics
# ---------------------------------------------------------
metric_columns = {
    "Top-1": ("baseline_top1", "reconstructed_ranker_top1"),
    "Top-5": ("baseline_top5", "reconstructed_ranker_top5"),
    "Top-10": ("baseline_top10", "reconstructed_ranker_top10"),
    "MRR": ("baseline_rr", "reconstructed_ranker_rr"),
}

summary = []
for metric, (base_col, ranker_col) in metric_columns.items():
    summary.append({
        "metric": metric,
        "phenotype_baseline": result_df[base_col].mean(),
        "reconstructed_ranker": result_df[ranker_col].mean(),
    })

summary_df = pd.DataFrame(summary)

print("=== RECONSTRUCTED SAME-CANDIDATE EVALUATION ===")
print("Cases:", len(result_df))
print("Candidates per case:", len(candidates))
print("Target diagnoses present:", result_df["diagnosis_id"].isin(candidates).sum())
print("\n", summary_df.to_string(
    index=False, float_format=lambda x: f"{x:.4f}"
))

print("\nMedian target ranks:")
print("Baseline:", result_df["baseline_rank_7340"].median())
print(
    "Reconstructed ranker:",
    result_df["reconstructed_ranker_rank_7340"].median()
)

print("\nIMPORTANT:")
print("- Ranker values are a reconstructed experiment, not verified original outputs.")
print("- No model training occurred.")
print("- No checkpoint or data files were modified.")

=== RECONSTRUCTED SAME-CANDIDATE EVALUATION ===
Cases: 906
Candidates per case: 7340
Target diagnoses present: 906

 metric  phenotype_baseline  reconstructed_ranker
 Top-1              0.2947                0.3013
 Top-5              0.4680                0.3631
Top-10              0.5419                0.3951
   MRR              0.3773                0.3341

Median target ranks:
Baseline: 8.0
Reconstructed ranker: 152.0

IMPORTANT:
- Ranker values are a reconstructed experiment, not verified original outputs.
- No model training occurred.
- No checkpoint or data files were modified.


In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)
state = ckpt["model_state_dict"]
disease_ids = list(ckpt["disease_ids"])
phenotype_ids = list(ckpt["phenotype_ids"])
disease_to_idx = {d: i for i, d in enumerate(disease_ids)}
phenotype_to_idx = {h: i for i, h in enumerate(phenotype_ids)}

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    usecols=["database_id", "hpo_id"],
    dtype={"database_id": "string", "hpo_id": "string"},
    low_memory=False
).dropna(subset=["database_id", "hpo_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

split = pd.read_csv(DATA / "disease_level_split.csv", dtype={"disease_id": "string"})
train_ids = set(split.loc[split["split"].str.lower() == "train", "disease_id"])
counts = defaultdict(int)
for disease in train_ids:
    for term in phenotype_map.get(disease, set()):
        counts[term] += 1
ic = {
    term: -np.log(count / max(len(train_ids), 1))
    for term, count in counts.items() if count > 0
}

validation = pd.read_csv(DATA / "patient_level_validation.csv", dtype=str).fillna("")
candidate_profiles = [phenotype_map.get(d, set()) for d in disease_ids]

hpo_embedding = state["hpo_embedding.weight"].float()
disease_embedding = state["disease_embedding.weight"].float()
disease_bias = state["disease_bias"].float()

projection = nn.Sequential(
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 128),
)
projection[0].weight.data.copy_(state["patient_projection.0.weight"])
projection[0].bias.data.copy_(state["patient_projection.0.bias"])
projection[3].weight.data.copy_(state["patient_projection.3.weight"])
projection[3].bias.data.copy_(state["patient_projection.3.bias"])
projection.eval()

baseline_scores_all = []
ranker_scores_all = []
targets = []
case_ids = []

for row in validation.itertuples(index=False):
    observed_all = {t.strip() for t in row.hpo_ids.split(";") if t.strip()}
    observed = observed_all & set(ic)
    if not observed:
        raise ValueError(f"No recognized terms for {row.case_id}")

    weights = {t: ic[t] for t in observed}
    denominator = sum(weights.values()) or 1.0
    base_scores = np.zeros(len(disease_ids), dtype=np.float64)

    for j, profile in enumerate(candidate_profiles):
        overlap = observed & profile
        precision = sum(weights[t] for t in overlap) / denominator
        recall = len(overlap) / max(len(profile), 1)
        base_scores[j] = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

    indices = [phenotype_to_idx[t] for t in observed_all if t in phenotype_to_idx]
    patient_vector = hpo_embedding[indices].mean(dim=0).unsqueeze(0)
    with torch.no_grad():
        patient_vector = projection(patient_vector).squeeze(0)
        neural_scores = (disease_embedding @ patient_vector + disease_bias).numpy()

    baseline_scores_all.append(base_scores)
    ranker_scores_all.append(neural_scores)
    targets.append(disease_to_idx[row.diagnosis_ids.strip()])
    case_ids.append(row.case_id)

baseline_scores_all = np.stack(baseline_scores_all)
ranker_scores_all = np.stack(ranker_scores_all)
targets = np.asarray(targets)

print("Score matrices:", baseline_scores_all.shape, ranker_scores_all.shape)
print("Patients:", len(case_ids))
print("Candidates per patient:", len(disease_ids))
print("Scores retained in memory only; no files modified.")

Score matrices: (906, 7340) (906, 7340)
Patients: 906
Candidates per patient: 7340
Scores retained in memory only; no files modified.


In [ ]:
import numpy as np
import pandas as pd

def minmax_rows(scores):
    row_min = scores.min(axis=1, keepdims=True)
    row_max = scores.max(axis=1, keepdims=True)
    return (scores - row_min) / np.maximum(row_max - row_min, 1e-12)

baseline_norm = minmax_rows(baseline_scores_all)
ranker_norm = minmax_rows(ranker_scores_all)

def evaluate_scores(scores, targets):
    n_cases = len(targets)
    reciprocal_ranks = []
    top1 = top5 = top10 = 0
    ranks = []

    for i in range(n_cases):
        target_score = scores[i, targets[i]]
        # Average-rank ties to avoid arbitrary disease-ID tie breaks
        rank = 1 + np.sum(scores[i] > target_score)
        rank += 0.5 * np.sum(scores[i] == target_score)
        ranks.append(rank)
        reciprocal_ranks.append(1.0 / rank)
        top1 += rank <= 1
        top5 += rank <= 5
        top10 += rank <= 10

    return {
        "Top1": top1 / n_cases,
        "Top5": top5 / n_cases,
        "Top10": top10 / n_cases,
        "MRR": np.mean(reciprocal_ranks),
        "Median_rank": float(np.median(ranks)),
    }

rows = []
for alpha in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    combined = (1 - alpha) * baseline_norm + alpha * ranker_norm
    metrics = evaluate_scores(combined, targets)
    rows.append({
        "ranker_weight": alpha,
        "baseline_weight": round(1 - alpha, 1),
        **metrics,
    })

blend_results = pd.DataFrame(rows)

print("=== EXPLORATORY SCORE-BLENDING RESULTS ===")
print(blend_results.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

print("\nBest blending weight by MRR:")
print(blend_results.loc[blend_results["MRR"].idxmax()].to_string())

print("\nBest blending weight by Top-5:")
print(blend_results.loc[blend_results["Top5"].idxmax()].to_string())

print("\nBest blending weight by Top-10:")
print(blend_results.loc[blend_results["Top10"].idxmax()].to_string())

print("\nNo training performed; no files saved.")

=== EXPLORATORY SCORE-BLENDING RESULTS ===
 ranker_weight  baseline_weight   Top1   Top5  Top10    MRR  Median_rank
        0.0000           1.0000 0.0000 0.4448 0.5276 0.2697       8.5000
        0.1000           0.9000 0.0000 0.4735 0.5806 0.2848       6.5000
        0.2000           0.8000 0.0000 0.4989 0.6600 0.3061       5.5000
        0.3000           0.7000 0.0000 0.5828 0.7263 0.3308       3.5000
        0.4000           0.6000 0.0000 0.6380 0.7428 0.3509       3.5000
        0.5000           0.5000 0.0000 0.6126 0.6965 0.3259       3.5000
        0.6000           0.4000 0.0000 0.5596 0.6137 0.2988       3.5000
        0.7000           0.3000 0.0000 0.5265 0.5541 0.2838       3.5000
        0.8000           0.2000 0.0000 0.4790 0.5210 0.2644       6.5000
        0.9000           0.1000 0.0000 0.4084 0.4349 0.2578      29.5000
        1.0000           0.0000 0.0000 0.3576 0.3907 0.2289     152.5000

Best blending weight by MRR:
ranker_weight      0.400000
baseline_weight    0.60

In [ ]:
import numpy as np
import pandas as pd

print("=== SCORE MATRIX CONSISTENCY AUDIT ===")
print("Baseline matrix:", baseline_scores_all.shape)
print("Ranker matrix:", ranker_scores_all.shape)
print("Number of targets:", len(targets))
print("Number of disease IDs:", len(disease_ids))

print("\nTarget index bounds:")
print("Minimum:", int(targets.min()))
print("Maximum:", int(targets.max()))
print("Valid indices:", bool(np.all((targets >= 0) & (targets < len(disease_ids)))))

print("\nFirst 10 target mappings:")
for i in range(10):
    print(
        case_ids[i],
        "| target index:", int(targets[i]),
        "| mapped disease:", disease_ids[targets[i]],
        "| expected diagnosis:", validation.iloc[i]["diagnosis_ids"],
        "| baseline score:", float(baseline_scores_all[i, targets[i]])
    )

print("\n=== BASELINE-ONLY METRICS USING THE CURRENT MATRICES ===")
baseline_metrics = evaluate_scores(baseline_scores_all, targets)
print(pd.Series(baseline_metrics).to_string())

print("\nNo files modified.")


=== SCORE MATRIX CONSISTENCY AUDIT ===
Baseline matrix: (906, 7340)
Ranker matrix: (906, 7340)
Number of targets: 906
Number of disease IDs: 7340

Target index bounds:
Minimum: 8
Maximum: 5349
Valid indices: True

First 10 target mappings:
PMID_28830446_Patient_2 | target index: 3274 | mapped disease: OMIM:615688 | expected diagnosis: OMIM:615688 | baseline score: 0.12457615350926553
PMID_28830446_Patient_1 | target index: 3274 | mapped disease: OMIM:615688 | expected diagnosis: OMIM:615688 | baseline score: 0.16629225610439574
PMID_32480058_Case_F5P1_original_pedigree_II_2 | target index: 4055 | mapped disease: OMIM:618052 | expected diagnosis: OMIM:618052 | baseline score: 0.1075268817204301
PMID_32480058_Case_F4P2_original_pedigree_IV_5 | target index: 4055 | mapped disease: OMIM:618052 | expected diagnosis: OMIM:618052 | baseline score: 0.16666666666666669
PMID_32480058_Case_F10P2 | target index: 4055 | mapped disease: OMIM:618052 | expected diagnosis: OMIM:618052 | baseline score:

In [ ]:
import numpy as np
import pandas as pd

print("=== DIRECT TARGET-RANK CHECK ===")

for i in range(10):
    row = baseline_scores_all[i]
    target_idx = int(targets[i])
    target_score = row[target_idx]
    higher = int(np.sum(row > target_score))
    equal = int(np.sum(row == target_score))
    top_score = float(np.max(row))
    target_is_max = bool(target_score == top_score)

    print({
        "case_id": case_ids[i],
        "target": disease_ids[target_idx],
        "target_score": round(float(target_score), 6),
        "top_score": round(top_score, 6),
        "diseases_scoring_higher": higher,
        "diseases_tied_with_target": equal,
        "target_is_max": target_is_max,
        "rank_with_ties": higher + (equal + 1) / 2
    })

print("\n=== SCORE SANITY CHECK ===")
print("Finite scores:", bool(np.isfinite(baseline_scores_all).all()))
print("Unique scores in first case:",
      len(np.unique(baseline_scores_all[0])))
print("Top 5 disease IDs in first case:")

top_indices = np.argsort(-baseline_scores_all[0], kind="stable")[:5]
for idx in top_indices:
    print(disease_ids[idx], float(baseline_scores_all[0, idx]))

print("\nNo files modified.")

=== DIRECT TARGET-RANK CHECK ===
{'case_id': 'PMID_28830446_Patient_2', 'target': 'OMIM:615688', 'target_score': 0.124576, 'top_score': 0.14719, 'diseases_scoring_higher': 6, 'diseases_tied_with_target': 1, 'target_is_max': False, 'rank_with_ties': 7.0}
{'case_id': 'PMID_28830446_Patient_1', 'target': 'OMIM:615688', 'target_score': 0.166292, 'top_score': 0.273659, 'diseases_scoring_higher': 10, 'diseases_tied_with_target': 1, 'target_is_max': False, 'rank_with_ties': 11.0}
{'case_id': 'PMID_32480058_Case_F5P1_original_pedigree_II_2', 'target': 'OMIM:618052', 'target_score': 0.107527, 'top_score': 0.411094, 'diseases_scoring_higher': 76, 'diseases_tied_with_target': 1, 'target_is_max': False, 'rank_with_ties': 77.0}
{'case_id': 'PMID_32480058_Case_F4P2_original_pedigree_IV_5', 'target': 'OMIM:618052', 'target_score': 0.166667, 'top_score': 0.280359, 'diseases_scoring_higher': 9, 'diseases_tied_with_target': 1, 'target_is_max': False, 'rank_with_ties': 10.0}
{'case_id': 'PMID_32480058_Ca

In [ ]:
import numpy as np

# Recalculate metrics independently, without relying on evaluate_scores()
n = len(targets)
ranks = np.empty(n, dtype=float)

for i in range(n):
    row = baseline_scores_all[i]
    target_score = row[int(targets[i])]
    higher = np.sum(row > target_score)
    equal = np.sum(row == target_score)
    ranks[i] = higher + (equal + 1) / 2

print("Cases:", n)
print("Top-1:", np.mean(ranks == 1))
print("Top-5:", np.mean(ranks <= 5))
print("Top-10:", np.mean(ranks <= 10))
print("MRR:", np.mean(1 / ranks))
print("Median rank:", np.median(ranks))

print("\nDirectly verified top-1 cases among first 10:",
      int(np.sum(ranks[:10] == 1)))
print("Ranks for first 10:", ranks[:10].tolist())
print("\nNo files modified.")

Cases: 906
Top-1: 0.2947019867549669
Top-5: 0.46799116997792495
Top-10: 0.5419426048565121
MRR: 0.37795605406346255
Median rank: 8.0

Directly verified top-1 cases among first 10: 3
Ranks for first 10: [7.0, 11.0, 77.0, 10.0, 64.0, 1.0, 1.0, 1.0, 8.0, 44.0]

No files modified.


In [ ]:
import numpy as np

required = [
    "baseline_scores_all",
    "ranker_scores_all",
    "targets",
    "disease_ids",
    "case_ids",
]

missing = [name for name in required if name not in globals()]

if missing:
    print("Missing variables:", missing)
    print("We need to restore the saved data before continuing.")
else:
    print("Baseline scores:", baseline_scores_all.shape)
    print("Ranker scores:", ranker_scores_all.shape)
    print("Targets:", len(targets))
    print("Disease candidates:", len(disease_ids))
    print("Patient cases:", len(case_ids))

    assert baseline_scores_all.shape == ranker_scores_all.shape
    assert baseline_scores_all.shape == (len(targets), len(disease_ids))
    assert len(targets) == len(case_ids) == 906
    assert np.isfinite(baseline_scores_all).all()
    assert np.isfinite(ranker_scores_all).all()

    print("\nPASS: score matrices are ready for the corrected comparison.")

Baseline scores: (906, 7340)
Ranker scores: (906, 7340)
Targets: 906
Disease candidates: 7340
Patient cases: 906

PASS: score matrices are ready for the corrected comparison.


In [ ]:
import numpy as np
import pandas as pd

def verified_metrics(scores, targets):
    ranks = np.empty(len(targets), dtype=float)

    for i, target_idx in enumerate(targets):
        row = scores[i]
        target_score = row[int(target_idx)]

        higher = np.sum(row > target_score)
        equal = np.sum(row == target_score)

        # Average rank for ties
        ranks[i] = higher + (equal + 1) / 2

    return {
        "Top-1": np.mean(ranks <= 1),
        "Top-5": np.mean(ranks <= 5),
        "Top-10": np.mean(ranks <= 10),
        "MRR": np.mean(1 / ranks),
        "Median rank": np.median(ranks),
    }

comparison = pd.DataFrame({
    "Baseline": verified_metrics(baseline_scores_all, targets),
    "Reconstructed ranker": verified_metrics(ranker_scores_all, targets),
}).T

print("=== VERIFIED COMPARISON ===")
display(comparison.round(4))

print("\n=== BASELINE CONSISTENCY CHECK ===")
expected = {
    "Top-1": 0.2947019867549669,
    "Top-5": 0.46799116997792495,
    "Top-10": 0.5419426048565121,
}

for metric, value in expected.items():
    actual = comparison.loc["Baseline", metric]
    print(f"{metric}: {actual:.4f} (expected {value:.4f})")

print("\nNo files modified.")

=== VERIFIED COMPARISON ===


,Top-1,Top-5,Top-10,MRR,Median rank
Baseline,0.2947,0.4680,0.5419,0.3780,8.0
Reconstructed ranker,0.3013,0.3631,0.3951,0.3341,152.0



=== BASELINE CONSISTENCY CHECK ===
Top-1: 0.2947 (expected 0.2947)
Top-5: 0.4680 (expected 0.4680)
Top-10: 0.5419 (expected 0.5419)

No files modified.


In [ ]:
import numpy as np
import pandas as pd

def minmax_rows(scores):
    row_min = scores.min(axis=1, keepdims=True)
    row_max = scores.max(axis=1, keepdims=True)
    span = row_max - row_min
    return (scores - row_min) / np.maximum(span, 1e-12)

baseline_norm = minmax_rows(baseline_scores_all)
ranker_norm = minmax_rows(ranker_scores_all)

weights = np.round(np.arange(0.0, 1.01, 0.1), 1)
results = []

for w in weights:
    combined = (1 - w) * baseline_norm + w * ranker_norm
    metrics = verified_metrics(combined, targets)

    results.append({
        "Ranker weight": w,
        "Baseline weight": round(1 - w, 1),
        **metrics
    })

blend_results = pd.DataFrame(results)

print("=== BLENDING RESULTS ===")
display(blend_results.round(4))

print("\nBaseline-only check:")
print(verified_metrics(baseline_norm, targets))

print("\nRanker-only check:")
print(verified_metrics(ranker_norm, targets))

print("\nNo files modified.")

=== BLENDING RESULTS ===


,Ranker weight,Baseline weight,Top-1,Top-5,Top-10,MRR,Median rank
0,0.0,1.0,0.2947,0.4680,0.5419,0.3780,8.0
1,0.1,0.9,0.3024,0.4912,0.5960,0.3975,6.0
2,0.2,0.8,0.3267,0.5442,0.6932,0.4273,5.0
3,0.3,0.7,0.3444,0.6325,0.7417,0.4607,3.0
4,0.4,0.6,0.3488,0.6678,0.7539,0.4883,3.0
5,0.5,0.5,0.2980,0.6291,0.7152,0.4498,3.0
6,0.6,0.4,0.2804,0.5695,0.6170,0.4151,3.0
7,0.7,0.3,0.2815,0.5353,0.5563,0.3978,3.0
8,0.8,0.2,0.2671,0.4934,0.5232,0.3713,6.0
9,0.9,0.1,0.3234,0.4194,0.4393,0.3734,29.0



Baseline-only check:
{'Top-1': np.float64(0.2947019867549669), 'Top-5': np.float64(0.46799116997792495), 'Top-10': np.float64(0.5419426048565121), 'MRR': np.float64(0.37795605406346255), 'Median rank': np.float64(8.0)}

Ranker-only check:
{'Top-1': np.float64(0.30132450331125826), 'Top-5': np.float64(0.3631346578366446), 'Top-10': np.float64(0.39514348785871967), 'MRR': np.float64(0.3341009854621938), 'Median rank': np.float64(152.0)}

No files modified.


In [ ]:
import numpy as np

# Use the best weight from the experiment
best_weight = 0.4

best_blend_scores = (
    (1 - best_weight) * baseline_norm
    + best_weight * ranker_norm
)

def reciprocal_ranks(scores, targets):
    rr = np.empty(len(targets), dtype=float)

    for i, target_idx in enumerate(targets):
        row = scores[i]
        target_score = row[int(target_idx)]

        higher = np.sum(row > target_score)
        equal = np.sum(row == target_score)
        rank = higher + (equal + 1) / 2

        rr[i] = 1.0 / rank

    return rr

baseline_rr = reciprocal_ranks(baseline_norm, targets)
blend_rr = reciprocal_ranks(best_blend_scores, targets)

difference = blend_rr - baseline_rr

# Paired bootstrap over patient cases
rng = np.random.default_rng(42)
n = len(difference)
bootstrap_deltas = np.empty(5000)

for b in range(len(bootstrap_deltas)):
    sample_idx = rng.integers(0, n, size=n)
    bootstrap_deltas[b] = difference[sample_idx].mean()

ci_low, ci_high = np.percentile(bootstrap_deltas, [2.5, 97.5])

print("=== BEST BLEND ROBUSTNESS CHECK ===")
print("Cases:", n)
print("Baseline MRR:", round(float(baseline_rr.mean()), 4))
print("Blend MRR:", round(float(blend_rr.mean()), 4))
print("MRR improvement:", round(float(difference.mean()), 4))
print("95% paired bootstrap CI:",
      (round(float(ci_low), 4), round(float(ci_high), 4)))

print("\nCases improved:", int(np.sum(difference > 0)))
print("Cases unchanged:", int(np.sum(difference == 0)))
print("Cases worsened:", int(np.sum(difference < 0)))

print("\nNo files modified.")

=== BEST BLEND ROBUSTNESS CHECK ===
Cases: 906
Baseline MRR: 0.378
Blend MRR: 0.4883
MRR improvement: 0.1103
95% paired bootstrap CI: (0.0896, 0.1317)

Cases improved: 389
Cases unchanged: 300
Cases worsened: 217

No files modified.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

terms = [
    "PatientDiseaseRanker",
    "patient_projection",
    "disease_bias",
    "hpo_embedding",
]

extensions = {".py", ".ipynb", ".txt"}

print("=== SEARCHING PROJECT SOURCE FILES ===")

matches = []

for path in PROJECT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue

    # Skip generated datasets and large result artifacts.
    if any(part.startswith(".") for part in path.parts):
        continue
    if path.stat().st_size > 10_000_000:
        continue

    try:
        content = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    found = [term for term in terms if term in content]
    if found:
        matches.append((str(path.relative_to(PROJECT)), found))

for filename, found in matches:
    print(f"{filename} | matches: {', '.join(found)}")

if not matches:
    print("No matching source found in project files.")

print("\nSearch complete. No files modified.")

=== SEARCHING PROJECT SOURCE FILES ===
No matching source found in project files.

Search complete. No files modified.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

print("=== PROJECT NOTEBOOKS ===")
notebooks = sorted(PROJECT.rglob("*.ipynb"))

if notebooks:
    for path in notebooks:
        print(path)
else:
    print("No notebooks found inside the project folder.")

print("\n=== RELEVANT LIVE RUNTIME OBJECTS ===")
for name, obj in list(globals().items()):
    if any(term in name.lower() for term in [
        "ranker", "patientdisease", "model_class", "forward"
    ]):
        try:
            print(name, "->", type(obj).__name__)
        except Exception:
            pass

print("\nInspection complete. No files modified.")

=== PROJECT NOTEBOOKS ===
No notebooks found inside the project folder.

=== RELEVANT LIVE RUNTIME OBJECTS ===
ranker_diseases -> set
ranker_only -> set
ranker_phenotypes -> set
ranker_scores -> ndarray
ranker_order -> list
ranker_rank -> int
ranker_col -> str
ranker_scores_all -> ndarray
ranker_norm -> ndarray

Inspection complete. No files modified.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from datetime import datetime

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
REPORT_DIR = PROJECT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Recalculate the final metrics from the existing score arrays.
baseline_metrics = verified_metrics(baseline_scores_all, targets)
ranker_metrics = verified_metrics(ranker_scores_all, targets)
blend_metrics = verified_metrics(best_blend_scores, targets)

report = {
    "experiment": "baseline_ranker_blend_preliminary",
    "created_at": datetime.now().astimezone().isoformat(),
    "validation_cases": int(len(targets)),
    "disease_candidates": int(len(disease_ids)),
    "baseline_metrics": {
        k: float(v) for k, v in baseline_metrics.items()
    },
    "reconstructed_ranker_metrics": {
        k: float(v) for k, v in ranker_metrics.items()
    },
    "blend": {
        "baseline_weight": 0.6,
        "ranker_weight": 0.4,
        "metrics": {k: float(v) for k, v in blend_metrics.items()},
        "mrr_difference_vs_baseline": float(
            blend_metrics["MRR"] - baseline_metrics["MRR"]
        ),
    },
    "paired_bootstrap": {
        "mrr_difference": float(difference.mean()),
        "ci_95_lower": float(ci_low),
        "ci_95_upper": float(ci_high),
        "improved_cases": int(np.sum(difference > 0)),
        "unchanged_cases": int(np.sum(difference == 0)),
        "worsened_cases": int(np.sum(difference < 0)),
        "bootstrap_replicates": 5000,
        "seed": 42,
    },
    "limitations": [
        "Neural ranker inference architecture is reconstructed, not verified.",
        "Blend weight was selected on the same 906 validation cases.",
        "Results are preliminary research metrics, not clinical validation.",
    ],
}

report_path = REPORT_DIR / "baseline_ranker_blend_preliminary.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Report saved:", report_path)
print("Baseline MRR:", round(baseline_metrics["MRR"], 4))
print("Ranker MRR:", round(ranker_metrics["MRR"], 4))
print("Blend MRR:", round(blend_metrics["MRR"], 4))
print("No model checkpoints modified.")

Report saved: /content/drive/MyDrive/Rare Disease ai/reports/baseline_ranker_blend_preliminary.json
Baseline MRR: 0.378
Ranker MRR: 0.3341
Blend MRR: 0.4883
No model checkpoints modified.


In [ ]:
from pathlib import Path
import json
import math

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
REPORT_PATH = PROJECT / "reports" / "baseline_ranker_blend_preliminary.json"

print("=== REPRODUCIBILITY AUDIT ===")

if not REPORT_PATH.exists():
    print("FAIL: Report file not found:", REPORT_PATH)
else:
    with REPORT_PATH.open("r", encoding="utf-8") as f:
        saved_report = json.load(f)

    print("PASS: Report exists and valid JSON was loaded.")
    print("Experiment:", saved_report.get("experiment"))
    print("Validation cases:", saved_report.get("validation_cases"))
    print("Disease candidates:", saved_report.get("disease_candidates"))

    baseline_mrr = saved_report["baseline_metrics"]["MRR"]
    ranker_mrr = saved_report["reconstructed_ranker_metrics"]["MRR"]
    blend_mrr = saved_report["blend"]["metrics"]["MRR"]

    print("\nSaved MRR values:")
    print("Baseline:", round(baseline_mrr, 4))
    print("Reconstructed ranker:", round(ranker_mrr, 4))
    print("Blend:", round(blend_mrr, 4))

    checks = {
        "Expected case count": saved_report["validation_cases"] == 906,
        "Expected candidate count": saved_report["disease_candidates"] == 7340,
        "Blend weight sums to 1": math.isclose(
            saved_report["blend"]["baseline_weight"]
            + saved_report["blend"]["ranker_weight"],
            1.0,
        ),
        "Blend MRR exceeds baseline MRR": blend_mrr > baseline_mrr,
        "Limitations documented": len(saved_report.get("limitations", [])) > 0,
    }

    print("\n=== CHECKS ===")
    for name, passed in checks.items():
        print(f"{'PASS' if passed else 'FAIL'}: {name}")

    print("\nNo files modified.")

=== REPRODUCIBILITY AUDIT ===
PASS: Report exists and valid JSON was loaded.
Experiment: baseline_ranker_blend_preliminary
Validation cases: 906
Disease candidates: 7340

Saved MRR values:
Baseline: 0.378
Reconstructed ranker: 0.3341
Blend: 0.4883

=== CHECKS ===
PASS: Expected case count
PASS: Expected candidate count
PASS: Blend weight sums to 1
PASS: Blend MRR exceeds baseline MRR
PASS: Limitations documented

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files_to_check = [
    PROJECT / "patient_ranker_bias_softmax_experiment_validation.csv",
    PROJECT / "patient_validation_matched_candidate_comparison.csv",
    DATA / "patient_level_validation.csv",
]

for path in files_to_check:
    print(f"\n=== {path.name} ===")

    if not path.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    for col in ["case_id", "diagnosis_id", "diagnosis_ids"]:
        if col in df.columns:
            print(f"Unique {col}:", df[col].nunique())

print("\nAudit complete. No files modified.")


=== patient_ranker_bias_softmax_experiment_validation.csv ===
NOT FOUND

=== patient_validation_matched_candidate_comparison.csv ===
NOT FOUND

=== patient_level_validation.csv ===
Rows: 906
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']
Unique case_id: 906
Unique diagnosis_ids: 125

Audit complete. No files modified.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

search_terms = [
    "patient_ranker_bias_softmax_experiment_validation.csv",
    "patient_validation_matched_candidate_comparison.csv",
    "patient_ranker_bias_softmax_experiment_metrics.json",
    "patient_ranker_bias_softmax_experiment_history.csv",
]

print("=== SEARCHING PROJECT FOR EXISTING ARTIFACTS ===")

for filename in search_terms:
    matches = list(PROJECT.rglob(filename))

    print(f"\n{filename}")
    if matches:
        for path in matches:
            print("FOUND:", path)
    else:
        print("NOT FOUND anywhere inside project folder")

print("\nSearch complete. No files modified.")

=== SEARCHING PROJECT FOR EXISTING ARTIFACTS ===

patient_ranker_bias_softmax_experiment_validation.csv
FOUND: /content/drive/MyDrive/Rare Disease ai/processed/patient_ranker_bias_softmax_experiment_validation.csv

patient_validation_matched_candidate_comparison.csv
FOUND: /content/drive/MyDrive/Rare Disease ai/processed/patient_validation_matched_candidate_comparison.csv

patient_ranker_bias_softmax_experiment_metrics.json
FOUND: /content/drive/MyDrive/Rare Disease ai/processed/patient_ranker_bias_softmax_experiment_metrics.json

patient_ranker_bias_softmax_experiment_history.csv
FOUND: /content/drive/MyDrive/Rare Disease ai/processed/patient_ranker_bias_softmax_experiment_history.csv

Search complete. No files modified.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

validation = pd.read_csv(DATA / "patient_level_validation.csv")

softmax_path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
baseline_path = DATA / "patient_validation_matched_candidate_comparison.csv"

softmax = pd.read_csv(softmax_path)
baseline = pd.read_csv(baseline_path)

print("=== SAVED PREDICTION AUDIT ===")
print("Reference validation cases:", len(validation))
print("Reference unique case IDs:", validation["case_id"].nunique())

for name, df in [
    ("Softmax ranker", softmax),
    ("Matched baseline comparison", baseline),
]:
    print(f"\n--- {name} ---")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if "case_id" in df.columns:
        print("Unique case IDs:", df["case_id"].nunique())
        print("Cases missing from reference:",
              len(set(validation["case_id"]) - set(df["case_id"])))
        print("Extra cases:",
              len(set(df["case_id"]) - set(validation["case_id"])))

    print(df.head(3).to_string(index=False))

print("\nNo files modified.")


=== SAVED PREDICTION AUDIT ===
Reference validation cases: 906
Reference unique case IDs: 906

--- Softmax ranker ---
Rows: 906
Columns: ['case_id', 'diagnosis_id', 'rank', 'top1', 'top5', 'top10', 'mrr']
Unique case IDs: 906
Cases missing from reference: 0
Extra cases: 0
                                       case_id diagnosis_id  rank  top1  top5  top10      mrr
                       PMID_28830446_Patient_2  OMIM:615688   548     0     0      0 0.001825
                       PMID_28830446_Patient_1  OMIM:615688   545     0     0      0 0.001835
PMID_32480058_Case_F5P1_original_pedigree_II_2  OMIM:618052  6971     0     0      0 0.000143

--- Matched baseline comparison ---
Rows: 906
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_observed_hpo', 'baseline_native_rank', 'ranker_native_rank', 'baseline_common_rank', 'ranker_common_rank', 'target_in_baseline_vocab', 'target_in_ranker_vocab', 'target_in_common_vocab']
Unique case IDs: 906
Cases missing from reference: 0
Extra cases: 0


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

softmax = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)
matched = pd.read_csv(
    DATA / "patient_validation_matched_candidate_comparison.csv"
)
metrics_path = DATA / "patient_ranker_bias_softmax_experiment_metrics.json"

def metrics_from_ranks(ranks):
    ranks = np.asarray(ranks, dtype=float)
    return {
        "Top-1": np.mean(ranks <= 1),
        "Top-5": np.mean(ranks <= 5),
        "Top-10": np.mean(ranks <= 10),
        "MRR": np.mean(1 / ranks),
        "Median rank": np.median(ranks),
    }

print("=== SAVED SOFTMAX RANKER METRICS ===")
print(pd.Series(metrics_from_ranks(softmax["rank"])).round(4).to_string())

print("\n=== SAVED MATCHED-COMPARISON METRICS ===")
for col in ["baseline_common_rank", "ranker_common_rank"]:
    print(f"\n{col}")
    print(pd.Series(metrics_from_ranks(matched[col])).round(4).to_string())

if metrics_path.exists():
    with metrics_path.open("r", encoding="utf-8") as f:
        saved_metrics = json.load(f)
    print("\n=== SAVED METRICS JSON ===")
    print(json.dumps(saved_metrics, indent=2)[:6000])

print("\nNo files modified.")

=== SAVED SOFTMAX RANKER METRICS ===
Top-1            0.3013
Top-5            0.3631
Top-10           0.3951
MRR              0.3341
Median rank    152.0000

=== SAVED MATCHED-COMPARISON METRICS ===

baseline_common_rank
Top-1          0.2947
Top-5          0.4680
Top-10         0.5419
MRR            0.3773
Median rank    8.0000

ranker_common_rank
Top-1            0.0210
Top-5            0.0872
Top-10           0.1104
MRR              0.0527
Median rank    450.0000

=== SAVED METRICS JSON ===
{
  "cases": 906,
  "mrr": 0.3341009854885823,
  "top1": 0.30132450331125826,
  "top5": 0.3631346578366446,
  "top10": 0.39514348785871967
}

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

saved = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)

print("=== SAVED VS RECONSTRUCTED RANK AUDIT ===")

case_to_row = {case_id: i for i, case_id in enumerate(case_ids)}

checked = 0
rank_matches = 0

for _, row in saved.head(20).iterrows():
    case_id = row["case_id"]

    if case_id not in case_to_row:
        print("Case missing:", case_id)
        continue

    i = case_to_row[case_id]
    target_idx = int(targets[i])
    scores = ranker_scores_all[i]
    target_score = scores[target_idx]

    reconstructed_rank = int(np.sum(scores > target_score) + 1)
    saved_rank = int(row["rank"])

    checked += 1
    rank_matches += reconstructed_rank == saved_rank

    print(
        f"{case_id} | saved={saved_rank} "
        f"| reconstructed={reconstructed_rank}"
    )

print(f"\nCases checked: {checked}")
print(f"Exact rank matches: {rank_matches}")
print("No files modified.")

=== SAVED VS RECONSTRUCTED RANK AUDIT ===
PMID_28830446_Patient_2 | saved=548 | reconstructed=548
PMID_28830446_Patient_1 | saved=545 | reconstructed=545
PMID_32480058_Case_F5P1_original_pedigree_II_2 | saved=6971 | reconstructed=6971
PMID_32480058_Case_F4P2_original_pedigree_IV_5 | saved=6645 | reconstructed=6645
PMID_32480058_Case_F10P2 | saved=4784 | reconstructed=4784
PMID_32480058_Case_F6P1_original_pedigree_II_1 | saved=5423 | reconstructed=5423
PMID_32480058_Case_F7P3 | saved=3790 | reconstructed=3790
PMID_32480058_Case_F7P2 | saved=4677 | reconstructed=4677
PMID_32480058_Case_F8P1 | saved=3416 | reconstructed=3416
PMID_32480058_Case_F4P1_original_pedigree_IV_1 | saved=6236 | reconstructed=6236
PMID_32480058_Case_F13P1 | saved=3844 | reconstructed=3844
PMID_32480058_Case_F11P1 | saved=3812 | reconstructed=3812
PMID_32480058_Case_F1P1_original_pedigree_A_IX_2 | saved=6197 | reconstructed=6197
PMID_32480058_Case_F2P3_original_pedigree_B_IV_3 | saved=5109 | reconstructed=5109
PMID_

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
saved_path = (
    PROJECT / "processed" /
    "patient_ranker_bias_softmax_experiment_validation.csv"
)

saved = pd.read_csv(saved_path)
case_to_row = {case_id: i for i, case_id in enumerate(case_ids)}

matched = 0
mismatches = []

for _, row in saved.iterrows():
    case_id = row["case_id"]

    if case_id not in case_to_row:
        mismatches.append((case_id, "missing case"))
        continue

    i = case_to_row[case_id]
    target_idx = int(targets[i])
    scores = ranker_scores_all[i]
    target_score = scores[target_idx]

    reconstructed_rank = int(np.sum(scores > target_score) + 1)
    saved_rank = int(row["rank"])

    if reconstructed_rank == saved_rank:
        matched += 1
    else:
        mismatches.append(
            (case_id, saved_rank, reconstructed_rank)
        )

print("=== FULL RANK CONSISTENCY AUDIT ===")
print("Saved cases:", len(saved))
print("Exact rank matches:", matched)
print("Mismatches:", len(mismatches))
print("Exact match rate:", round(matched / len(saved), 6))

if mismatches:
    print("\nFirst 10 mismatches:")
    for item in mismatches[:10]:
        print(item)

print("\nNo files modified.")

=== FULL RANK CONSISTENCY AUDIT ===
Saved cases: 906
Exact rank matches: 904
Mismatches: 2
Exact match rate: 0.997792

First 10 mismatches:
('PMID_33988247_Family_208_II_1', 3970, 3971)
('PMID_39315527_Family_3_IV_5', 2114, 2113)

No files modified.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
saved = pd.read_csv(
    PROJECT / "processed" /
    "patient_ranker_bias_softmax_experiment_validation.csv"
)

problem_cases = [
    "PMID_33988247_Family_208_II_1",
    "PMID_39315527_Family_3_IV_5",
]

case_to_row = {case_id: i for i, case_id in enumerate(case_ids)}

for case_id in problem_cases:
    saved_row = saved.loc[saved["case_id"] == case_id].iloc[0]
    i = case_to_row[case_id]
    target_idx = int(targets[i])
    scores = ranker_scores_all[i]
    target_score = scores[target_idx]

    higher = int(np.sum(scores > target_score))
    equal = int(np.sum(scores == target_score))
    reconstructed_rank = higher + 1

    print(f"\n=== {case_id} ===")
    print("Saved rank:", int(saved_row["rank"]))
    print("Reconstructed rank:", reconstructed_rank)
    print("Target disease:", disease_ids[target_idx])
    print("Target score:", repr(float(target_score)))
    print("Scores strictly higher:", higher)
    print("Candidates tied with target:", equal)

    # Inspect scores around the target's rank.
    order = np.argsort(-scores, kind="stable")
    start = max(0, reconstructed_rank - 3)
    end = min(len(order), reconstructed_rank + 2)

    print("Nearby candidates:")
    for pos in range(start, end):
        idx = order[pos]
        print(
            f"rank={pos + 1}, disease={disease_ids[idx]}, "
            f"score={scores[idx]:.12f}"
        )

print("\nNo files modified.")


=== PMID_33988247_Family_208_II_1 ===
Saved rank: 3970
Reconstructed rank: 3971
Target disease: OMIM:621269
Target score: -9.897055625915527
Scores strictly higher: 3970
Candidates tied with target: 1
Nearby candidates:
rank=3969, disease=OMIM:312000, score=-9.896781921387
rank=3970, disease=OMIM:618646, score=-9.897052764893
rank=3971, disease=OMIM:621269, score=-9.897055625916
rank=3972, disease=OMIM:619881, score=-9.897188186646
rank=3973, disease=OMIM:616067, score=-9.897385597229

=== PMID_39315527_Family_3_IV_5 ===
Saved rank: 2114
Reconstructed rank: 2113
Target disease: OMIM:621354
Target score: -17.619220733642578
Scores strictly higher: 2112
Candidates tied with target: 3
Nearby candidates:
rank=2111, disease=ORPHA:468726, score=-17.618654251099
rank=2112, disease=OMIM:182900, score=-17.619127273560
rank=2113, disease=OMIM:614940, score=-17.619220733643
rank=2114, disease=OMIM:618555, score=-17.619220733643
rank=2115, disease=OMIM:621354, score=-17.619220733643

No files mod

In [ ]:
from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

checkpoint_path = (
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
)

print("=== CHECKPOINT METADATA AUDIT ===")
print("Checkpoint exists:", checkpoint_path.exists())

if checkpoint_path.exists():
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print("\nCheckpoint keys:")
    if isinstance(checkpoint, dict):
        print(list(checkpoint.keys()))

        for key, value in checkpoint.items():
            if key == "model_state_dict":
                print("\nModel state dict:")
                for name, tensor in value.items():
                    print(name, tuple(tensor.shape), tensor.dtype)

            elif isinstance(value, (str, int, float, bool, type(None))):
                print(f"{key}: {value}")

            elif isinstance(value, list):
                print(f"{key}: list with {len(value)} entries")
    else:
        print("Checkpoint type:", type(checkpoint).__name__)

print("\nNo files modified.")

=== CHECKPOINT METADATA AUDIT ===
Checkpoint exists: True

Checkpoint keys:
['model_state_dict', 'disease_ids', 'phenotype_ids', 'hidden_dim', 'best_epoch', 'best_validation_metrics', 'seed', 'training_cases', 'validation_cases', 'objective', 'disease_bias', 'note']

Model state dict:
disease_bias (7340,) torch.float32
hpo_embedding.weight (9982, 128) torch.float32
disease_embedding.weight (7340, 128) torch.float32
patient_projection.0.weight (128, 128) torch.float32
patient_projection.0.bias (128,) torch.float32
patient_projection.3.weight (128, 128) torch.float32
patient_projection.3.bias (128,) torch.float32
disease_ids: list with 7340 entries
phenotype_ids: list with 9982 entries
hidden_dim: 128
best_epoch: 15
seed: 42
training_cases: 5730
validation_cases: 906
objective: single-label full-disease cross-entropy
disease_bias: True
note: Research experiment; not clinically validated.

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
saved_path = (
    PROJECT / "processed" /
    "patient_ranker_bias_softmax_experiment_validation.csv"
)

saved = pd.read_csv(saved_path)

problem_cases = [
    "PMID_33988247_Family_208_II_1",
    "PMID_39315527_Family_3_IV_5",
]

print("=== SAVED ROWS FOR THE TWO CASES ===")

for case_id in problem_cases:
    rows = saved.loc[saved["case_id"] == case_id]

    print(f"\nCase: {case_id}")
    if rows.empty:
        print("Case not found in saved CSV.")
    else:
        print(rows.to_string(index=False))

print("\nNo files modified.")

=== SAVED ROWS FOR THE TWO CASES ===

Case: PMID_33988247_Family_208_II_1
                      case_id diagnosis_id  rank  top1  top5  top10      mrr
PMID_33988247_Family_208_II_1  OMIM:621269  3970     0     0      0 0.000252

Case: PMID_39315527_Family_3_IV_5
                    case_id diagnosis_id  rank  top1  top5  top10      mrr
PMID_39315527_Family_3_IV_5  OMIM:621354  2114     0     0      0 0.000473

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

history_path = DATA / "patient_ranker_bias_softmax_experiment_history.csv"
metrics_path = DATA / "patient_ranker_bias_softmax_experiment_metrics.json"

print("=== SOFTMAX EXPERIMENT HISTORY ===")

if history_path.exists():
    history = pd.read_csv(history_path)
    print("Shape:", history.shape)
    print("Columns:", history.columns.tolist())
    print(history.tail(5).to_string(index=False))
else:
    print("History file not found.")

print("\n=== EXPERIMENT METADATA ===")

if metrics_path.exists():
    with metrics_path.open("r", encoding="utf-8") as f:
        metadata = json.load(f)
    print(json.dumps(metadata, indent=2))
else:
    print("Metrics JSON not found.")

print("\nNo files modified.")


=== SOFTMAX EXPERIMENT HISTORY ===
Shape: (19, 7)
Columns: ['epoch', 'train_loss', 'val_cases', 'val_mrr', 'val_top1', 'val_top5', 'val_top10']
 epoch  train_loss  val_cases  val_mrr  val_top1  val_top5  val_top10
    15    0.392794        906 0.334101  0.301325  0.363135   0.395143
    16    0.356568        906 0.315388  0.272627  0.363135   0.391832
    17    0.337070        906 0.324764  0.289183  0.357616   0.390728
    18    0.311816        906 0.316394  0.277042  0.354305   0.388521
    19    0.298530        906 0.334063  0.303532  0.371965   0.397351

=== EXPERIMENT METADATA ===
{
  "cases": 906,
  "mrr": 0.3341009854885823,
  "top1": 0.30132450331125826,
  "top5": 0.3631346578366446,
  "top10": 0.39514348785871967
}

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== EXISTING DATA SPLITS ===")

for path in sorted(DATA.glob("*split*.csv")):
    print(f"\nFile: {path.name}")

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

print("\n=== PATIENT VALIDATION DATASET ===")

validation_path = DATA / "patient_level_validation.csv"

if validation_path.exists():
    df = pd.read_csv(validation_path)
    print("Cases:", len(df))
    print("Unique cases:", df["case_id"].nunique())
    print("Unique diagnoses:", df["diagnosis_ids"].nunique())
    print("Diagnosis count distribution:")
    print(df["n_diagnoses"].value_counts().sort_index().to_string())

print("\nInspection complete. No files modified.")

=== EXISTING DATA SPLITS ===

File: disease_level_split.csv
Rows: 9176
Columns: ['disease_id', 'split']
 disease_id split
OMIM:100100  test
OMIM:100300 train
OMIM:100800 train

File: patient_publication_split.csv
Rows: 10345
Columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
                    case_id     pmid       split diagnosis_ids
PMID_19836009_Family_E_II_6 19836009 development   OMIM:277600
PMID_19836009_Family_E_II_3 19836009 development   OMIM:277600
PMID_19836009_Family_B_II_4 19836009 development   OMIM:277600

=== PATIENT VALIDATION DATASET ===
Cases: 906
Unique cases: 906
Unique diagnoses: 125
Diagnosis count distribution:
n_diagnoses
1    906

Inspection complete. No files modified.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

split_path = DATA / "patient_publication_split.csv"
validation_path = DATA / "patient_level_validation.csv"

split_df = pd.read_csv(split_path)
validation_df = pd.read_csv(validation_path)

print("=== PATIENT-PUBLICATION SPLIT ===")
print("Total rows:", len(split_df))
print("\nRows by split:")
print(split_df["split"].value_counts(dropna=False).to_string())

print("\nUnique cases by split:")
print(
    split_df.groupby("split")["case_id"]
    .nunique()
    .to_string()
)

print("\nUnique publications by split:")
print(
    split_df.groupby("split")["pmid"]
    .nunique()
    .to_string()
)

print("\n=== VALIDATION OVERLAP ===")
validation_cases = set(validation_df["case_id"].astype(str))
validation_pmids = set(validation_df["pmid"].astype(str))
split_cases = set(split_df["case_id"].astype(str))

print("Validation cases:", len(validation_cases))
print("Validation cases found in split:", len(validation_cases & split_cases))
print("Validation cases absent from split:", len(validation_cases - split_cases))

split_pmids = set(split_df["pmid"].astype(str))
print("Validation publications found in split:", len(validation_pmids & split_pmids))

print("\nValidation cases by split label:")
validation_with_split = validation_df.merge(
    split_df[["case_id", "split"]].drop_duplicates(),
    on="case_id",
    how="left"
)
print(
    validation_with_split["split"]
    .fillna("NOT FOUND")
    .value_counts()
    .to_string()
)

print("\nNo files modified.")

=== PATIENT-PUBLICATION SPLIT ===
Total rows: 10345

Rows by split:
split
development    8386
test           1959

Unique cases by split:
split
development    8386
test           1959

Unique publications by split:
split
development    1383
test            346

=== VALIDATION OVERLAP ===
Validation cases: 906
Validation cases found in split: 906
Validation cases absent from split: 0
Validation publications found in split: 166

Validation cases by split label:
split
development    906

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

split_df = pd.read_csv(DATA / "patient_publication_split.csv")
validation_df = pd.read_csv(DATA / "patient_level_validation.csv")

test_df = split_df[split_df["split"] == "test"].copy()
dev_df = split_df[split_df["split"] == "development"].copy()

print("=== HELD-OUT TEST SPLIT AUDIT ===")
print("Test rows:", len(test_df))
print("Unique test cases:", test_df["case_id"].nunique())
print("Unique test publications:", test_df["pmid"].nunique())
print("Unique test diagnoses:", test_df["diagnosis_ids"].nunique())

print("\n=== CASE AND PUBLICATION SEPARATION ===")
print(
    "Case overlap with development:",
    len(set(test_df["case_id"]) & set(dev_df["case_id"]))
)
print(
    "Publication overlap with development:",
    len(set(test_df["pmid"].astype(str)) &
        set(dev_df["pmid"].astype(str)))
)
print(
    "Case overlap with current validation:",
    len(set(test_df["case_id"]) &
        set(validation_df["case_id"]))
)
print(
    "Publication overlap with current validation:",
    len(set(test_df["pmid"].astype(str)) &
        set(validation_df["pmid"].astype(str)))
)

print("\n=== TEST DIAGNOSIS EXAMPLES ===")
print(
    test_df[["case_id", "pmid", "diagnosis_ids"]]
    .head(10)
    .to_string(index=False)
)

print("\nNo files modified.")

=== HELD-OUT TEST SPLIT AUDIT ===
Test rows: 1959
Unique test cases: 1959
Unique test publications: 346
Unique test diagnoses: 258

=== CASE AND PUBLICATION SEPARATION ===
Case overlap with development: 0
Publication overlap with development: 0
Case overlap with current validation: 0
Publication overlap with current validation: 0

=== TEST DIAGNOSIS EXAMPLES ===
                          case_id     pmid diagnosis_ids
   PMID_28239884_Family_1_proband 28239884   OMIM:148600
   PMID_28239884_Family_2_proband 28239884   OMIM:148600
   PMID_28239884_Family_3_proband 28239884   OMIM:148600
             PMID_1680289_patient  1680289   OMIM:102700
    PMID_30733661_4_year_old_girl 30733661   OMIM:620475
          PMID_21989597_Proband_2 21989597   OMIM:621196
          PMID_21989597_Proband_1 21989597   OMIM:621196
           PMID_28513613_family_1 28513613   OMIM:617297
           PMID_28513613_family_2 28513613   OMIM:617297
PMID_37951597_Family_22_Subject_1 37951597   OMIM:620785

No file

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

# Load held-out test cases and saved model vocabulary
split_df = pd.read_csv(DATA / "patient_publication_split.csv")
test_df = split_df[split_df["split"] == "test"].copy()

checkpoint_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

ranker_diseases = set(checkpoint["disease_ids"])

# Baseline disease profiles come from the existing HPO annotation file
annotations = pd.read_csv(DATA / "clean_hpo_annotations.csv")
baseline_diseases = set(
    annotations["database_id"].dropna().astype(str).unique()
)

# Compare target diagnosis coverage
test_df["diagnosis_ids"] = test_df["diagnosis_ids"].astype(str)

test_targets = set(test_df["diagnosis_ids"])
ranker_covered = test_df["diagnosis_ids"].isin(ranker_diseases)
baseline_covered = test_df["diagnosis_ids"].isin(baseline_diseases)

print("=== HELD-OUT TEST DISEASE COVERAGE ===")
print("Test cases:", len(test_df))
print("Unique target diagnoses:", test_df["diagnosis_ids"].nunique())
print("Ranker vocabulary size:", len(ranker_diseases))
print("Baseline disease profiles:", len(baseline_diseases))

print("\nCase-level target coverage:")
print(f"Ranker: {ranker_covered.sum()} / {len(test_df)} "
      f"({ranker_covered.mean():.2%})")
print(f"Baseline: {baseline_covered.sum()} / {len(test_df)} "
      f"({baseline_covered.mean():.2%})")

print("\nUnique-diagnosis coverage:")
print("Ranker:", len(test_targets & ranker_diseases),
      "/", len(test_targets))
print("Baseline:", len(test_targets & baseline_diseases),
      "/", len(test_targets))

print("\nExamples of targets missing from ranker vocabulary:")
print(sorted(test_targets - ranker_diseases)[:20])

print("\nNo files modified.")

=== HELD-OUT TEST DISEASE COVERAGE ===
Test cases: 1959
Unique target diagnoses: 258
Ranker vocabulary size: 7340
Baseline disease profiles: 12879

Case-level target coverage:
Ranker: 1492 / 1959 (76.16%)
Baseline: 1959 / 1959 (100.00%)

Unique-diagnosis coverage:
Ranker: 195 / 258
Baseline: 258 / 258

Examples of targets missing from ranker vocabulary:
['OMIM:107480', 'OMIM:115200', 'OMIM:130050', 'OMIM:147920', 'OMIM:162200', 'OMIM:206700', 'OMIM:231095', 'OMIM:236200', 'OMIM:240300', 'OMIM:258480', 'OMIM:275400', 'OMIM:300867', 'OMIM:300868', 'OMIM:304110', 'OMIM:600373', 'OMIM:601414', 'OMIM:602849', 'OMIM:604377', 'OMIM:605911', 'OMIM:606658']

No files modified.


/tmp/ipykernel_541/4120789089.py:23: DtypeWarning: Columns (2,6,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations = pd.read_csv(DATA / "clean_hpo_annotations.csv")


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

test_df = pd.read_csv(DATA / "patient_publication_split.csv")
test_df = test_df[test_df["split"] == "test"].copy()

validation_df = pd.read_csv(DATA / "patient_level_validation.csv")
annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False
)

# Check which test cases have phenotype records in the original patient dataset.
patient_hpo_path = DATA / "patient_level_validation.csv"

print("=== BASELINE EVALUATION PREFLIGHT ===")
print("Held-out test cases:", len(test_df))
print("Test cases with diagnosis IDs:",
      test_df["diagnosis_ids"].notna().sum())

# Confirm phenotype profiles exist for all test target diseases.
profile_diseases = set(
    annotations["database_id"].dropna().astype(str).unique()
)
target_diseases = set(test_df["diagnosis_ids"].dropna().astype(str))

print("Unique test target diseases:", len(target_diseases))
print("Targets with baseline profiles:",
      len(target_diseases & profile_diseases))
print("Targets without baseline profiles:",
      len(target_diseases - profile_diseases))

print("\nMissing-profile examples:")
print(sorted(target_diseases - profile_diseases)[:20])

print("\nNote: this checks disease-profile availability, not yet test-case HPO availability.")
print("No files modified.")

=== BASELINE EVALUATION PREFLIGHT ===
Held-out test cases: 1959
Test cases with diagnosis IDs: 1959
Unique test target diseases: 258
Targets with baseline profiles: 258
Targets without baseline profiles: 0

Missing-profile examples:
[]

Note: this checks disease-profile availability, not yet test-case HPO availability.
No files modified.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== RELEVANT DATA FILES ===")
for path in sorted(DATA.glob("*")):
    if path.is_file() and any(
        term in path.name.lower()
        for term in ["patient", "phenotype", "hpo", "case", "annotation"]
    ):
        print(f"{path.name} ({path.stat().st_size / 1024:.1f} KB)")

print("\n=== DATA LOADER SOURCE ===")
loader_path = PROJECT / "rare_disease_data_loader.py"

if loader_path.exists():
    source = loader_path.read_text(encoding="utf-8", errors="ignore")
    lines = source.splitlines()

    keywords = ["patient_level_validation", "phenotype_map",
                "hpo_ids", "patient_publication_split"]

    for i, line in enumerate(lines):
        if any(word in line.lower() for word in keywords):
            start = max(0, i - 2)
            end = min(len(lines), i + 4)
            print(f"\n--- Lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")
else:
    print("rare_disease_data_loader.py not found at project root.")

print("\nInspection complete. No files modified.")

=== RELEVANT DATA FILES ===
annotation_leakage_readiness_audit.json (1.4 KB)
annotation_provenance_followup_audit.json (266.6 KB)
clean_hpo_annotations.csv (35065.7 KB)
gene_informed_phenotype_recovery_metrics.csv (88.0 KB)
gene_informed_phenotype_recovery_top10.csv (632.3 KB)
gnn_annotations_excluding_dev_test_pmids.csv (20272.8 KB)
gnn_annotations_publication_disjoint_candidate.csv (20209.6 KB)
gnn_phenotype_recovery_metrics.csv (72.4 KB)
gnn_training_annotations.csv (22474.9 KB)
heldout_input_phenotypes.csv (485.0 KB)
heldout_target_phenotypes.csv (505.5 KB)
hpo_symptom_catalogue.csv (1438.4 KB)
matched_phenotype_gene_comparison.csv (138.4 KB)
missing_hpo_phenotype_audit.json (0.6 KB)
patient_baseline_vs_gnn_paired.csv (256.2 KB)
patient_filtered_gnn_linkhead_rankings.csv (171.9 KB)
patient_filtered_gnn_test_rankings.csv (193.4 KB)
patient_gnn_training_disease_rankings.csv (195.3 KB)
patient_level_development_dataset.csv (955.3 KB)
patient_level_label_audit.json (0.7 KB)
patient_lev

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = [
    "phenopacket_patient_cases.csv",
    "heldout_input_phenotypes.csv",
    "heldout_target_phenotypes.csv",
    "patient_level_development_dataset.csv",
    "patient_level_train.csv",
]

for filename in files:
    path = DATA / filename
    print(f"\n=== {filename} ===")

    if not path.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path, nrows=5, low_memory=False)
    print("Columns:", df.columns.tolist())
    print("Sample rows:")
    print(df.head(3).to_string(index=False))

print("\nInspection complete. No files modified.")


=== phenopacket_patient_cases.csv ===
Columns: ['case_id', 'source_file', 'hpo_terms', 'phenotype_count', 'diagnosis_ids', 'diagnosis_count']
Sample rows:
                    case_id                                      source_file                                                         hpo_terms  phenotype_count diagnosis_ids  diagnosis_count
PMID_19836009_Family_E_II_6 0.1.27/ADAMTS10/PMID_19836009_Family_E_II_6.json HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003                6   OMIM:277600                1
PMID_19836009_Family_E_II_3 0.1.27/ADAMTS10/PMID_19836009_Family_E_II_3.json HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003                6   OMIM:277600                1
PMID_19836009_Family_B_II_4 0.1.27/ADAMTS10/PMID_19836009_Family_B_II_4.json            HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0011003                5   OMIM:277600                1

=== heldout_input_phenotypes.csv ===
Columns: ['disease_id', 'hpo_id']
Sample r

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

split_df = pd.read_csv(DATA / "patient_publication_split.csv")
test_df = split_df[split_df["split"] == "test"].copy()

phenopackets = pd.read_csv(
    DATA / "phenopacket_patient_cases.csv",
    low_memory=False
)

# Compare test cases with the patient phenotype source
test_ids = set(test_df["case_id"].astype(str))
phenopacket_ids = set(phenopackets["case_id"].astype(str))

matched_ids = test_ids & phenopacket_ids
missing_ids = test_ids - phenopacket_ids

print("=== TEST CASE PHENOTYPE COVERAGE ===")
print("Held-out test cases:", len(test_ids))
print("Cases in phenotype source:", len(phenopacket_ids))
print("Matched test cases:", len(matched_ids))
print("Missing test cases:", len(missing_ids))

# Check that phenotype-source diagnosis labels agree with the split labels
check = test_df[["case_id", "diagnosis_ids"]].merge(
    phenopackets[["case_id", "diagnosis_ids", "hpo_terms", "phenotype_count"]],
    on="case_id",
    how="inner",
    suffixes=("_split", "_phenopacket")
)

check["diagnosis_match"] = (
    check["diagnosis_ids_split"].astype(str)
    == check["diagnosis_ids_phenopacket"].astype(str)
)

print("\nDiagnosis labels match:", int(check["diagnosis_match"].sum()))
print("Diagnosis label mismatches:", int((~check["diagnosis_match"]).sum()))
print("Matched cases with nonempty HPO terms:",
      int(check["hpo_terms"].fillna("").str.strip().ne("").sum()))
print("Median phenotype count:", check["phenotype_count"].median())

print("\nExamples of missing test cases:")
print(sorted(missing_ids)[:10])

print("\nNo files modified.")

=== TEST CASE PHENOTYPE COVERAGE ===
Held-out test cases: 1959
Cases in phenotype source: 10377
Matched test cases: 1959
Missing test cases: 0

Diagnosis labels match: 1959
Diagnosis label mismatches: 0
Matched cases with nonempty HPO terms: 1959
Median phenotype count: 8.0

Examples of missing test cases:
[]

No files modified.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
source_file = PROJECT / "rare_disease_ai.py"

print("Source exists:", source_file.exists())

if source_file.exists():
    source = source_file.read_text()
    lines = source.splitlines()

    keywords = [
        "def load_prediction_data",
        "phenotype_map",
        "information_content",
        "def predict",
        "precision",
        "recall",
        "harmonic",
        "def rank",
    ]

    shown = set()
    for keyword in keywords:
        matches = [
            i for i, line in enumerate(lines)
            if keyword.lower() in line.lower()
        ]
        for i in matches:
            start = max(0, i - 5)
            end = min(len(lines), i + 16)
            if any(j in shown for j in range(start, end)):
                continue
            print(f"\n--- Lines {start + 1}–{end} (match: {keyword}) ---")
            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)

print("\nInspection only — no files modified.")

Source exists: False

Inspection only — no files modified.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== PROJECT FILES ===")
for p in sorted(PROJECT.iterdir()):
    if p.is_file():
        print(f"{p.name} ({p.stat().st_size:,} bytes)")
    elif p.is_dir() and p.name != "processed":
        print(f"{p.name}/")

print("\n=== BASELINE / EVALUATION ARTIFACTS ===")
for p in sorted(DATA.glob("*")):
    if any(word in p.name.lower() for word in [
        "baseline", "annotation", "validation",
        "rank", "split", "phenopacket", "metric"
    ]):
        print(f"{p.name} ({p.stat().st_size:,} bytes)")

print("\nInspection only — no files modified.")

=== PROJECT FILES ===
all_phenopackets/
genes_to_disease.txt (1,492,039 bytes)
genes_to_phenotype.txt (20,821,704 bytes)
hp.obo (10,863,613 bytes)
model/
phenotype.hpoa (35,816,037 bytes)
reports/
streamlit.log (236 bytes)
streamlit.pid (5 bytes)

=== BASELINE / EVALUATION ARTIFACTS ===
annotation_leakage_readiness_audit.json (1,396 bytes)
annotation_provenance_followup_audit.json (272,991 bytes)
app_predictor_validation_diagnostic.csv (47,216 bytes)
baseline_coverage_comparison_audit.csv (429 bytes)
baseline_model_comparison.csv (166 bytes)
clean_hpo_annotations.csv (35,907,264 bytes)
disease_level_split.csv (162,821 bytes)
disease_ranking_retrieval_metrics.csv (45,159 bytes)
gene_informed_phenotype_recovery_metrics.csv (90,137 bytes)
gnn_annotations_excluding_dev_test_pmids.csv (20,759,359 bytes)
gnn_annotations_publication_disjoint_candidate.csv (20,694,622 bytes)
gnn_phenotype_recovery_metrics.csv (74,139 bytes)
gnn_training_annotations.csv (23,014,254 bytes)
hierarchy_aware_retrie

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "patient_phenotype_baseline_metrics_audit.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_phenotype_only_validation.csv",
    "baseline_model_comparison.csv",
    "baseline_coverage_comparison_audit.csv",
]

for filename in files:
    path = DATA / filename
    print(f"\n{'=' * 70}\n{filename}\n{'=' * 70}")

    if not path.exists():
        print("File not found.")
        continue

    df = pd.read_csv(path, low_memory=False)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

print("\nInspection only — no files modified.")


patient_phenotype_baseline_metrics_audit.csv
Shape: (2, 7)
Columns: ['metric_group', 'cases', 'coverage', 'top1', 'top5', 'top10', 'mrr']
 metric_group  cases  coverage     top1     top5    top10      mrr
    all_cases   1959   0.79173 0.113834 0.196018 0.221031 0.152666
covered_cases   1551   1.00000 0.143778 0.247582 0.279175 0.192826

patient_phenotype_only_baseline.csv
Shape: (1959, 8)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']
                       case_id     pmid diagnosis_ids  rank  top1  top5  top10  candidate_covered
PMID_28239884_Family_1_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_2_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_3_proband 28239884   OMIM:148600   3.0     0     1      1                  1

patient_phenotype_only_validation.csv
Shape: (906, 10)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_recognize

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

results = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")
audit = pd.read_csv(DATA / "patient_phenotype_baseline_metrics_audit.csv")

print("=== SAVED BASELINE TEST RESULT AUDIT ===")
print("Rows:", len(results))
print("Unique case IDs:", results["case_id"].nunique())
print("Duplicate case IDs:", results["case_id"].duplicated().sum())
print("Missing ranks:", results["rank"].isna().sum())
print("Candidate coverage rate:",
      results["candidate_covered"].mean())

for group_name, df in [
    ("all_cases", results),
    ("covered_cases", results[results["candidate_covered"] == 1]),
]:
    ranks = pd.to_numeric(df["rank"], errors="coerce")
    valid = ranks.notna() & (ranks > 0)
    r = ranks[valid]

    print(f"\n{group_name}:")
    print("Rows:", len(df))
    print("Valid positive ranks:", int(valid.sum()))
    print("Top-1 from ranks:", round((r <= 1).mean(), 6) if len(r) else None)
    print("Top-5 from ranks:", round((r <= 5).mean(), 6) if len(r) else None)
    print("Top-10 from ranks:", round((r <= 10).mean(), 6) if len(r) else None)
    print("MRR from ranks:", round((1 / r).mean(), 6) if len(r) else None)

print("\n=== RECORDED AUDIT ===")
print(audit.to_string(index=False))
print("\nRead-only verification — no files modified.")

=== SAVED BASELINE TEST RESULT AUDIT ===
Rows: 1959
Unique case IDs: 1959
Duplicate case IDs: 0
Missing ranks: 408
Candidate coverage rate: 0.7917304747320061

all_cases:
Rows: 1959
Valid positive ranks: 1551
Top-1 from ranks: 0.143778
Top-5 from ranks: 0.247582
Top-10 from ranks: 0.279175
MRR from ranks: 0.192826

covered_cases:
Rows: 1551
Valid positive ranks: 1551
Top-1 from ranks: 0.143778
Top-5 from ranks: 0.247582
Top-10 from ranks: 0.279175
MRR from ranks: 0.192826

=== RECORDED AUDIT ===
 metric_group  cases  coverage     top1     top5    top10      mrr
    all_cases   1959   0.79173 0.113834 0.196018 0.221031 0.152666
covered_cases   1551   1.00000 0.143778 0.247582 0.279175 0.192826

Read-only verification — no files modified.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
results = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")

uncovered = results[results["candidate_covered"] == 0].copy()

print("=== UNCOVERED TEST CASES ===")
print("Count:", len(uncovered))
print("\nColumn names:")
print(results.columns.tolist())

print("\nRank values for uncovered cases:")
print(uncovered["rank"].value_counts(dropna=False).to_string())

print("\nTop-1 / Top-5 / Top-10 flags:")
for col in ["top1", "top5", "top10"]:
    print(f"\n{col}:")
    print(uncovered[col].value_counts(dropna=False).to_string())

print("\nSample uncovered cases:")
print(uncovered.head(10).to_string(index=False))

print("\nRead-only inspection — no files modified.")

=== UNCOVERED TEST CASES ===
Count: 408

Column names:
['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']

Rank values for uncovered cases:
rank
NaN    408

Top-1 / Top-5 / Top-10 flags:

top1:
top1
0    408

top5:
top5
0    408

top10:
top10
0    408

Sample uncovered cases:
                          case_id     pmid diagnosis_ids  rank  top1  top5  top10  candidate_covered
PMID_37951597_Family_22_Subject_1 37951597   OMIM:620785   NaN     0     0      0                  0
 PMID_37951597_Family_5_Subject_1 37951597   OMIM:620785   NaN     0     0      0                  0
PMID_37951597_Family_25_Subject_1 37951597   OMIM:620785   NaN     0     0      0                  0
 PMID_37951597_Family_2_Subject_1 37951597   OMIM:620785   NaN     0     0      0                  0
PMID_37951597_Family_19_Subject_2 37951597   OMIM:620785   NaN     0     0      0                  0
 PMID_37951597_Family_3_Subject_1 37951597   OMIM:620785   NaN     0     0   

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv")
test = split.loc[split["split"] == "test", ["case_id", "diagnosis_ids"]].copy()

pred = pd.read_csv(
    DATA / "patient_phenotype_only_baseline.csv"
)[["case_id", "diagnosis_ids", "rank", "candidate_covered"]].copy()

test["case_id"] = test["case_id"].astype(str)
pred["case_id"] = pred["case_id"].astype(str)

check = test.merge(
    pred,
    on="case_id",
    how="outer",
    suffixes=("_test", "_prediction"),
    indicator=True
)

matched = check[check["_merge"] == "both"].copy()
label_match = (
    matched["diagnosis_ids_test"].fillna("").astype(str)
    == matched["diagnosis_ids_prediction"].fillna("").astype(str)
)

print("Held-out test cases:", len(test))
print("Saved baseline rows:", len(pred))
print("Cases matched:", int((check["_merge"] == "both").sum()))
print("Missing predictions:", int((check["_merge"] == "left_only").sum()))
print("Unexpected prediction cases:", int((check["_merge"] == "right_only").sum()))
print("Diagnosis labels match:", int(label_match.sum()))
print("Diagnosis label mismatches:", int((~label_match).sum()))
print("\nNo files modified.")

Held-out test cases: 1959
Saved baseline rows: 1959
Cases matched: 1959
Missing predictions: 0
Unexpected prediction cases: 0
Diagnosis labels match: 1959
Diagnosis label mismatches: 0

No files modified.


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

checkpoint_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
split_path = DATA / "patient_publication_split.csv"
phenotype_path = DATA / "phenopacket_patient_cases.csv"

print("=== REQUIRED FILES ===")
for p in [checkpoint_path, split_path, phenotype_path]:
    print(f"{p.name}: exists={p.exists()}", end="")
    if p.exists():
        print(f", size={p.stat().st_size:,} bytes")
    else:
        print()

if checkpoint_path.exists():
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    print("\n=== CHECKPOINT METADATA ===")
    print("Disease vocabulary:", len(ckpt["disease_ids"]))
    print("Phenotype vocabulary:", len(ckpt["phenotype_ids"]))
    print("Best epoch:", ckpt.get("best_epoch"))
    print("Training cases:", ckpt.get("training_cases"))
    print("Validation cases:", ckpt.get("validation_cases"))

if split_path.exists() and phenotype_path.exists():
    split = pd.read_csv(split_path)
    test_ids = set(split.loc[split["split"] == "test", "case_id"].astype(str))
    cases = pd.read_csv(phenotype_path, usecols=["case_id"])
    source_ids = set(cases["case_id"].astype(str))
    print("\nHeld-out test cases:", len(test_ids))
    print("Test cases in phenotype source:", len(test_ids & source_ids))
    print("Test cases missing from phenotype source:", len(test_ids - source_ids))

print("\nRead-only inspection — no files modified.")

=== REQUIRED FILES ===
patient_ranker_bias_softmax_experiment.pt: exists=True, size=9,387,364 bytes
patient_publication_split.csv: exists=True, size=617,350 bytes
phenopacket_patient_cases.csv: exists=True, size=1,960,299 bytes

=== CHECKPOINT METADATA ===
Disease vocabulary: 7340
Phenotype vocabulary: 9982
Best epoch: 15
Training cases: 5730
Validation cases: 906

Held-out test cases: 1959
Test cases in phenotype source: 1959
Test cases missing from phenotype source: 0

Read-only inspection — no files modified.


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

cases = pd.read_csv(
    DATA / "phenopacket_patient_cases.csv",
    low_memory=False
)
ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

print("=== PHENOTYPE SOURCE ===")
print("Columns:", cases.columns.tolist())
print(cases[["case_id", "hpo_terms", "phenotype_count", "diagnosis_ids"]].head(5).to_string(index=False))

print("\n=== CHECKPOINT VOCABULARY ===")
print("Example phenotype IDs:", ckpt["phenotype_ids"][:10])
print("Example disease IDs:", ckpt["disease_ids"][:10])

print("\n=== VOCABULARY MATCH CHECK ===")
test = pd.read_csv(DATA / "patient_publication_split.csv")
test_ids = set(test.loc[test["split"] == "test", "case_id"].astype(str))
test_cases = cases[cases["case_id"].astype(str).isin(test_ids)]

hpo_vocab = set(map(str, ckpt["phenotype_ids"]))
print("Test cases:", len(test_cases))
print("Unique HPO strings in source:", cases["hpo_terms"].nunique(dropna=True))
print("Checkpoint phenotype vocabulary size:", len(hpo_vocab))

print("\nInspection only — no files modified.")


=== PHENOTYPE SOURCE ===
Columns: ['case_id', 'source_file', 'hpo_terms', 'phenotype_count', 'diagnosis_ids', 'diagnosis_count']
                    case_id                                                                                          hpo_terms  phenotype_count diagnosis_ids
PMID_19836009_Family_E_II_6                                  HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003                6   OMIM:277600
PMID_19836009_Family_E_II_3                                  HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003                6   OMIM:277600
PMID_19836009_Family_B_II_4                                             HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0011003                5   OMIM:277600
PMID_19836009_Family_B_II_2                                  HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003                6   OMIM:277600
      PMID_25469541_patient HP:0001156;HP:0001387;HP:0001642;HP:0002650;HP:000293

In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

cases = pd.read_csv(
    DATA / "phenopacket_patient_cases.csv",
    low_memory=False
)
split = pd.read_csv(DATA / "patient_publication_split.csv")

ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

test_ids = set(
    split.loc[split["split"] == "test", "case_id"].astype(str)
)
test = cases[cases["case_id"].astype(str).isin(test_ids)].copy()

hpo_vocab = set(map(str, ckpt["phenotype_ids"]))

test["hpo_list"] = test["hpo_terms"].fillna("").apply(
    lambda s: [x.strip() for x in str(s).split(";") if x.strip()]
)
test["recognized_hpo"] = test["hpo_list"].apply(
    lambda terms: [h for h in terms if h in hpo_vocab]
)
test["input_hpo_count"] = test["hpo_list"].str.len()
test["recognized_hpo_count"] = test["recognized_hpo"].str.len()

total_terms = int(test["input_hpo_count"].sum())
recognized_terms = int(test["recognized_hpo_count"].sum())

print("=== HELD-OUT HPO COVERAGE ===")
print("Test cases:", len(test))
print("Total observed HPO terms:", total_terms)
print("Recognized HPO terms:", recognized_terms)
print("Term-level coverage:", round(recognized_terms / total_terms, 4) if total_terms else 0)
print("Cases with at least one recognized HPO:",
      int((test["recognized_hpo_count"] > 0).sum()))
print("Cases with zero recognized HPO:",
      int((test["recognized_hpo_count"] == 0).sum()))
print("Median observed HPO count:", test["input_hpo_count"].median())
print("Median recognized HPO count:", test["recognized_hpo_count"].median())

print("\nRead-only coverage check — no files modified.")

=== HELD-OUT HPO COVERAGE ===
Test cases: 1959
Total observed HPO terms: 18874
Recognized HPO terms: 18805
Term-level coverage: 0.9963
Cases with at least one recognized HPO: 1949
Cases with zero recognized HPO: 10
Median observed HPO count: 8.0
Median recognized HPO count: 8.0

Read-only coverage check — no files modified.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

checkpoint_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

# Load held-out cases and saved baseline results.
split = pd.read_csv(DATA / "patient_publication_split.csv")
cases = pd.read_csv(DATA / "phenopacket_patient_cases.csv", low_memory=False)
baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")

test_ids = set(split.loc[split["split"] == "test", "case_id"].astype(str))
test = cases[cases["case_id"].astype(str).isin(test_ids)].copy()
test = test.drop_duplicates("case_id").reset_index(drop=True)

# Reconstruct the saved ranker's inference architecture.
state = ckpt["model_state_dict"]
hidden_dim = int(ckpt.get("hidden_dim", 128))
n_hpo = len(ckpt["phenotype_ids"])
n_disease = len(ckpt["disease_ids"])

class SavedRanker(nn.Module):
    def __init__(self):
        super().__init__()
        self.hpo_embedding = nn.Embedding(n_hpo, hidden_dim)
        self.disease_embedding = nn.Embedding(n_disease, hidden_dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.disease_bias = nn.Parameter(torch.zeros(n_disease))

    def forward(self, hpo_indices):
        # Padding index 0 is excluded from the mean.
        mask = (hpo_indices != 0).float().unsqueeze(-1)
        embedded = self.hpo_embedding(hpo_indices)
        pooled = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        patient = self.patient_projection(pooled)
        return patient @ self.disease_embedding.weight.T + self.disease_bias

model = SavedRanker()
model.load_state_dict(state)
model.eval()

disease_ids = list(map(str, ckpt["disease_ids"]))
hpo_to_idx = {str(h): i for i, h in enumerate(ckpt["phenotype_ids"])}
disease_to_idx = {d: i for i, d in enumerate(disease_ids)}

# Encode the patient's observed HPO terms; no phenotype labels are added.
encoded = []
recognized_counts = []
for value in test["hpo_terms"].fillna(""):
    terms = [x.strip() for x in str(value).split(";") if x.strip()]
    indices = [hpo_to_idx[h] for h in terms if h in hpo_to_idx]
    encoded.append(indices)
    recognized_counts.append(len(indices))

max_len = max(1, max(map(len, encoded)))
x = torch.zeros((len(encoded), max_len), dtype=torch.long)
for i, indices in enumerate(encoded):
    if indices:
        x[i, :len(indices)] = torch.tensor(indices)

with torch.no_grad():
    scores = model(x).cpu().numpy()

# Evaluate each true diagnosis against the saved disease vocabulary.
targets = test["diagnosis_ids"].astype(str).tolist()
target_indices = np.array([disease_to_idx.get(d, -1) for d in targets])
target_covered = target_indices >= 0

# Rank is 1 + number of diseases with a strictly higher score.
ranks = np.full(len(test), np.nan)
for i in np.where(target_covered)[0]:
    target_score = scores[i, target_indices[i]]
    ranks[i] = 1 + np.sum(scores[i] > target_score)

def metrics_from_ranks(rank_values):
    r = np.asarray(rank_values, dtype=float)
    r = r[np.isfinite(r) & (r > 0)]
    if len(r) == 0:
        return {"cases": 0, "Top-1": np.nan, "Top-5": np.nan,
                "Top-10": np.nan, "MRR": np.nan, "Median rank": np.nan}
    return {
        "cases": len(r),
        "Top-1": np.mean(r <= 1),
        "Top-5": np.mean(r <= 5),
        "Top-10": np.mean(r <= 10),
        "MRR": np.mean(1 / r),
        "Median rank": np.median(r),
    }

# Baseline saved ranks are missing for uncovered baseline cases.
baseline["case_id"] = baseline["case_id"].astype(str)
test["case_id"] = test["case_id"].astype(str)
baseline_by_id = baseline.set_index("case_id")

baseline_ranks = test["case_id"].map(baseline_by_id["rank"]).to_numpy(dtype=float)
baseline_covered = test["case_id"].map(
    baseline_by_id["candidate_covered"]
).fillna(0).to_numpy().astype(bool)

common = target_covered & baseline_covered

# For full-test metrics, an uncovered target is a failed prediction.
ranker_full = np.where(target_covered, ranks, np.inf)
baseline_full = np.where(baseline_covered, baseline_ranks, np.inf)

def full_test_metrics(rank_values):
    r = np.asarray(rank_values, dtype=float)
    return {
        "cases": len(r),
        "coverage": np.mean(np.isfinite(r)),
        "Top-1": np.mean(r <= 1),
        "Top-5": np.mean(r <= 5),
        "Top-10": np.mean(r <= 10),
        "MRR": np.mean(np.where(np.isfinite(r), 1 / r, 0)),
    }

print("=== HELD-OUT TEST: SAVED CHECKPOINT INFERENCE ===")
print("Test cases:", len(test))
print("Recognized HPO terms:", sum(recognized_counts))
print("Cases with zero recognized HPO:", sum(n == 0 for n in recognized_counts))
print("Ranker disease-vocabulary coverage:", int(target_covered.sum()), "/", len(test))
print("Cases covered by both ranker and baseline:", int(common.sum()))

print("\nFull-test metrics (uncovered cases count as failures):")
print(pd.DataFrame([
    {"model": "Phenotype baseline", **full_test_metrics(baseline_full)},
    {"model": "Saved neural ranker", **full_test_metrics(ranker_full)},
]).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\nSame cases with both target diagnoses covered:")
print(pd.DataFrame([
    {"model": "Phenotype baseline", **metrics_from_ranks(baseline_ranks[common])},
    {"model": "Saved neural ranker", **metrics_from_ranks(ranks[common])},
]).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\nInference only — no training, checkpoints, or files modified.")

=== HELD-OUT TEST: SAVED CHECKPOINT INFERENCE ===
Test cases: 1959
Recognized HPO terms: 18805
Cases with zero recognized HPO: 10
Ranker disease-vocabulary coverage: 1492 / 1959
Cases covered by both ranker and baseline: 1181

Full-test metrics (uncovered cases count as failures):
              model  cases  coverage  Top-1  Top-5  Top-10    MRR
 Phenotype baseline   1959    0.7917 0.1138 0.1960  0.2210 0.1527
Saved neural ranker   1959    0.7616 0.1577 0.2266  0.2527 0.1911

Same cases with both target diagnoses covered:
              model  cases  Top-1  Top-5  Top-10    MRR  Median rank
 Phenotype baseline   1181 0.1160 0.2235  0.2566 0.1659     334.0000
Saved neural ranker   1181 0.2591 0.3709  0.4124 0.3128      31.0000

Inference only — no training, checkpoints, or files modified.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

saved_path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
saved = pd.read_csv(saved_path)

print("Saved validation rows:", len(saved))
print("Reconstructed validation scores available:",
      "ranker_scores_all" in globals())
print("Validation target indices available:",
      "targets" in globals())

if "ranker_scores_all" not in globals() or "targets" not in globals():
    print(
        "\nThe earlier validation score arrays are not available in this "
        "runtime. We should reconstruct validation scores from the checkpoint "
        "before comparing ranks."
    )
else:
    saved = saved.copy()
    saved["case_id"] = saved["case_id"].astype(str)

    # Recompute target ranks from the current reconstructed score matrix.
    reconstructed_ranks = []
    for i, target_idx in enumerate(targets):
        if int(target_idx) < 0:
            reconstructed_ranks.append(np.nan)
        else:
            row = ranker_scores_all[i]
            reconstructed_ranks.append(
                1 + int(np.sum(row > row[int(target_idx)]))
            )

    print("Reconstructed rank count:", len(reconstructed_ranks))
    print("\nSaved validation columns:", saved.columns.tolist())
    print(saved.head(3).to_string(index=False))

print("\nRead-only check — no files modified.")

Saved validation rows: 906
Reconstructed validation scores available: False
Validation target indices available: True

The earlier validation score arrays are not available in this runtime. We should reconstruct validation scores from the checkpoint before comparing ranks.

Read-only check — no files modified.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)
saved = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)
cases = pd.read_csv(DATA / "phenopacket_patient_cases.csv", low_memory=False)

hidden_dim = int(ckpt.get("hidden_dim", 128))
hpo_ids = list(map(str, ckpt["phenotype_ids"]))
disease_ids = list(map(str, ckpt["disease_ids"]))
hpo_to_idx = {h: i for i, h in enumerate(hpo_ids)}
disease_to_idx = {d: i for i, d in enumerate(disease_ids)}

class SavedRanker(nn.Module):
    def __init__(self):
        super().__init__()
        self.hpo_embedding = nn.Embedding(len(hpo_ids), hidden_dim)
        self.disease_embedding = nn.Embedding(len(disease_ids), hidden_dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.disease_bias = nn.Parameter(torch.zeros(len(disease_ids)))

    def forward(self, x):
        mask = (x != 0).float().unsqueeze(-1)
        embedded = self.hpo_embedding(x)
        pooled = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        patient = self.patient_projection(pooled)
        return patient @ self.disease_embedding.weight.T + self.disease_bias

model = SavedRanker()
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

validation = saved[["case_id", "diagnosis_id", "rank"]].copy()
validation["case_id"] = validation["case_id"].astype(str)
validation["diagnosis_id"] = validation["diagnosis_id"].astype(str)

case_lookup = cases.drop_duplicates("case_id").set_index("case_id")
encoded = []
target_indices = []
usable_rows = []

for row_i, row in validation.iterrows():
    case_id = row["case_id"]
    if case_id not in case_lookup.index:
        continue

    raw = str(case_lookup.loc[case_id, "hpo_terms"])
    terms = [t.strip() for t in raw.split(";") if t.strip()]
    indices = [hpo_to_idx[t] for t in terms if t in hpo_to_idx]
    target_idx = disease_to_idx.get(row["diagnosis_id"], -1)

    if target_idx < 0:
        continue

    encoded.append(indices)
    target_indices.append(target_idx)
    usable_rows.append(row_i)

max_len = max(1, max(map(len, encoded)))
x = torch.zeros((len(encoded), max_len), dtype=torch.long)
for i, ids in enumerate(encoded):
    if ids:
        x[i, :len(ids)] = torch.tensor(ids)

with torch.no_grad():
    scores = model(x).cpu().numpy()

calculated_ranks = []
for i, target_idx in enumerate(target_indices):
    target_score = scores[i, target_idx]
    calculated_ranks.append(1 + int(np.sum(scores[i] > target_score)))

comparison = validation.iloc[usable_rows].copy().reset_index(drop=True)
comparison["calculated_rank"] = calculated_ranks
comparison["rank_difference"] = (
    comparison["calculated_rank"] - comparison["rank"]
)

print("Validation rows in saved file:", len(saved))
print("Rows scored:", len(comparison))
print("Exact rank matches:",
      int((comparison["rank_difference"] == 0).sum()))
print("Rank mismatches:",
      int((comparison["rank_difference"] != 0).sum()))
print("Maximum absolute rank difference:",
      int(comparison["rank_difference"].abs().max()))

print("\nMismatch examples:")
print(
    comparison.loc[
        comparison["rank_difference"] != 0,
        ["case_id", "diagnosis_id", "rank",
         "calculated_rank", "rank_difference"]
    ].head(10).to_string(index=False)
)

print("\nInference audit only — no training or files modified.")

Validation rows in saved file: 906
Rows scored: 906
Exact rank matches: 900
Rank mismatches: 6
Maximum absolute rank difference: 1

Mismatch examples:
                      case_id diagnosis_id  rank  calculated_rank  rank_difference
PMID_33988247_Family_208_II_1  OMIM:621269  3970             3971                1
  PMID_39315527_Family_3_IV_5  OMIM:621354  2114             2113               -1
     PMID_18256394_Patient_13  OMIM:176670  3130             3129               -1
      PMID_18256394_Patient_9  OMIM:176670  3164             3165                1
          PMID_30099644_III_3  OMIM:600138  3557             3556               -1
  PMID_40965981_Individual_23  OMIM:621495  4730             4729               -1

Inference audit only — no training or files modified.


In [ ]:
import numpy as np
import pandas as pd

# Only compare cases where both models contain the true diagnosis.
common = np.asarray(target_covered, dtype=bool) & np.asarray(baseline_covered, dtype=bool)

ranker_r = np.asarray(ranks, dtype=float)[common]
baseline_r = np.asarray(baseline_ranks, dtype=float)[common]

if len(ranker_r) == 0:
    raise ValueError("No common-coverage cases available.")

def metric_vector(r):
    return np.array([
        np.mean(r <= 1),
        np.mean(r <= 5),
        np.mean(r <= 10),
        np.mean(1 / r),
    ])

names = ["Top-1", "Top-5", "Top-10", "MRR"]
observed_delta = metric_vector(ranker_r) - metric_vector(baseline_r)

rng = np.random.default_rng(42)
n = len(ranker_r)
n_boot = 5000
boot_deltas = np.empty((n_boot, 4), dtype=float)

for b in range(n_boot):
    idx = rng.integers(0, n, size=n)
    boot_deltas[b] = metric_vector(ranker_r[idx]) - metric_vector(baseline_r[idx])

ci_low, ci_high = np.percentile(boot_deltas, [2.5, 97.5], axis=0)

summary = pd.DataFrame({
    "metric": names,
    "baseline": metric_vector(baseline_r),
    "ranker": metric_vector(ranker_r),
    "ranker_minus_baseline": observed_delta,
    "95%_CI_lower": ci_low,
    "95%_CI_upper": ci_high,
})

print("Paired test cases:", n)
print("Bootstrap repetitions:", n_boot)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nBootstrap only — no training or files modified.")

Paired test cases: 1181
Bootstrap repetitions: 5000
metric  baseline  ranker  ranker_minus_baseline  95%_CI_lower  95%_CI_upper
 Top-1    0.1160  0.2591                 0.1431        0.1160        0.1710
 Top-5    0.2235  0.3709                 0.1473        0.1202        0.1753
Top-10    0.2566  0.4124                 0.1558        0.1279        0.1846
   MRR    0.1659  0.3128                 0.1469        0.1232        0.1718

Bootstrap only — no training or files modified.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

checkpoint_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

assert checkpoint_path.exists(), f"Checkpoint not found: {checkpoint_path}"

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded successfully.")
print("Disease vocabulary:", len(ckpt["disease_ids"]))
print("Phenotype vocabulary:", len(ckpt["phenotype_ids"]))
print("Best saved epoch:", ckpt.get("best_epoch"))
print("Training cases:", ckpt.get("training_cases"))
print("Validation cases:", ckpt.get("validation_cases"))

for filename in [
    "patient_publication_split.csv",
    "phenopacket_patient_cases.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_ranker_bias_softmax_experiment_validation.csv",
]:
    path = DATA / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

print("\nRead-only check. No files modified.")

Checkpoint loaded successfully.
Disease vocabulary: 7340
Phenotype vocabulary: 9982
Best saved epoch: 15
Training cases: 5730
Validation cases: 906
patient_publication_split.csv: FOUND
phenopacket_patient_cases.csv: FOUND
patient_phenotype_only_baseline.csv: FOUND
patient_ranker_bias_softmax_experiment_validation.csv: FOUND

Read-only check. No files modified.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Load test cases and saved baseline predictions.
split = pd.read_csv(DATA / "patient_publication_split.csv")
cases = pd.read_csv(DATA / "phenopacket_patient_cases.csv", low_memory=False)
baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")

test_ids = set(
    split.loc[split["split"] == "test", "case_id"].astype(str)
)

test = (
    cases[cases["case_id"].astype(str).isin(test_ids)]
    .drop_duplicates("case_id")
    .reset_index(drop=True)
)

# Reconstruct the saved checkpoint architecture.
hidden_dim = int(ckpt.get("hidden_dim", 128))
hpo_ids = list(map(str, ckpt["phenotype_ids"]))
disease_ids = list(map(str, ckpt["disease_ids"]))

hpo_to_idx = {h: i for i, h in enumerate(hpo_ids)}
disease_to_idx = {d: i for i, d in enumerate(disease_ids)}

class SavedRanker(nn.Module):
    def __init__(self):
        super().__init__()
        self.hpo_embedding = nn.Embedding(len(hpo_ids), hidden_dim)
        self.disease_embedding = nn.Embedding(len(disease_ids), hidden_dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.disease_bias = nn.Parameter(torch.zeros(len(disease_ids)))

    def forward(self, x):
        mask = (x != 0).float().unsqueeze(-1)
        embedded = self.hpo_embedding(x)
        pooled = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        patient = self.patient_projection(pooled)
        return patient @ self.disease_embedding.weight.T + self.disease_bias

model = SavedRanker()
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Encode observed patient HPO terms.
encoded = []
recognized_counts = []

for value in test["hpo_terms"].fillna(""):
    terms = [t.strip() for t in str(value).split(";") if t.strip()]
    indices = [hpo_to_idx[t] for t in terms if t in hpo_to_idx]
    encoded.append(indices)
    recognized_counts.append(len(indices))

max_len = max(1, max(map(len, encoded)))
x = torch.zeros((len(encoded), max_len), dtype=torch.long)

for i, ids in enumerate(encoded):
    if ids:
        x[i, :len(ids)] = torch.tensor(ids)

with torch.no_grad():
    scores = model(x).cpu().numpy()

# Rank the true diagnosis only where it exists in the disease vocabulary.
targets = test["diagnosis_ids"].astype(str).tolist()
target_indices = np.array(
    [disease_to_idx.get(d, -1) for d in targets]
)
target_covered = target_indices >= 0

ranks = np.full(len(test), np.nan)
for i in np.where(target_covered)[0]:
    target_score = scores[i, target_indices[i]]
    ranks[i] = 1 + np.sum(scores[i] > target_score)

# Align baseline results by case ID.
baseline["case_id"] = baseline["case_id"].astype(str)
test["case_id"] = test["case_id"].astype(str)
baseline_by_id = baseline.set_index("case_id")

baseline_ranks = test["case_id"].map(
    baseline_by_id["rank"]
).to_numpy(dtype=float)

baseline_covered = test["case_id"].map(
    baseline_by_id["candidate_covered"]
).fillna(0).to_numpy().astype(bool)

common = target_covered & baseline_covered

def metric_vector(r):
    return np.array([
        np.mean(r <= 1),
        np.mean(r <= 5),
        np.mean(r <= 10),
        np.mean(1.0 / r),
    ])

def full_metrics(r, covered):
    r = np.asarray(r, dtype=float)
    covered = np.asarray(covered, dtype=bool)
    return {
        "cases": len(r),
        "coverage": covered.mean(),
        "Top-1": np.mean(np.where(covered, r <= 1, False)),
        "Top-5": np.mean(np.where(covered, r <= 5, False)),
        "Top-10": np.mean(np.where(covered, r <= 10, False)),
        "MRR": np.mean(np.where(covered, 1 / np.where(covered, r, 1), 0)),
    }

print("=== FULL HELD-OUT TEST RESULTS ===")
print("Test cases:", len(test))
print("Ranker diagnosis coverage:", int(target_covered.sum()), "/", len(test))
print("Baseline diagnosis coverage:", int(baseline_covered.sum()), "/", len(test))
print("Common-coverage cases:", int(common.sum()))
print("Cases with zero recognized HPO terms:",
      int(np.sum(np.array(recognized_counts) == 0)))

print("\nAll test cases (uncovered diagnoses count as failures):")
print(pd.DataFrame([
    {"model": "Phenotype baseline",
     **full_metrics(baseline_ranks, baseline_covered)},
    {"model": "Saved neural ranker",
     **full_metrics(ranks, target_covered)},
]).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\nSame cases, both diagnoses covered:")
print(pd.DataFrame([
    {"model": "Phenotype baseline",
     **dict(zip(["Top-1", "Top-5", "Top-10", "MRR"],
                metric_vector(baseline_ranks[common])))},
    {"model": "Saved neural ranker",
     **dict(zip(["Top-1", "Top-5", "Top-10", "MRR"],
                metric_vector(ranks[common])))},
]).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\nInference only. No training or checkpoint changes.")

=== FULL HELD-OUT TEST RESULTS ===
Test cases: 1959
Ranker diagnosis coverage: 1492 / 1959
Baseline diagnosis coverage: 1551 / 1959
Common-coverage cases: 1181
Cases with zero recognized HPO terms: 10

All test cases (uncovered diagnoses count as failures):
              model  cases  coverage  Top-1  Top-5  Top-10    MRR
 Phenotype baseline   1959    0.7917 0.1138 0.1960  0.2210 0.1527
Saved neural ranker   1959    0.7616 0.1577 0.2266  0.2527 0.1911

Same cases, both diagnoses covered:
              model  Top-1  Top-5  Top-10    MRR
 Phenotype baseline 0.1160 0.2235  0.2566 0.1659
Saved neural ranker 0.2591 0.3709  0.4124 0.3128

Inference only. No training or checkpoint changes.


In [ ]:
import numpy as np
import pandas as pd

# Confirm Cell 2 variables are available.
required = [
    "test", "ranks", "baseline_ranks",
    "target_covered", "baseline_covered"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Run Cell 2 first. Missing variables: {missing}")

test_eval = test.reset_index(drop=True).copy()
ranker_ranks = np.asarray(ranks, dtype=float)
baseline_ranks_arr = np.asarray(baseline_ranks, dtype=float)
ranker_covered = np.asarray(target_covered, dtype=bool)
baseline_covered_arr = np.asarray(baseline_covered, dtype=bool)

if not all(
    len(a) == len(test_eval)
    for a in [
        ranker_ranks, baseline_ranks_arr,
        ranker_covered, baseline_covered_arr
    ]
):
    raise ValueError("Evaluation arrays do not align with test cases.")

# Compare only cases where both models contain the true diagnosis.
common = ranker_covered & baseline_covered_arr
paired = test_eval.loc[common].copy().reset_index(drop=True)
r_model = ranker_ranks[common]
r_base = baseline_ranks_arr[common]

# Extract publication ID from IDs such as PMID_28239884_Family_1_proband.
paired["pmid"] = (
    paired["case_id"].astype(str)
    .str.extract(r"^PMID_(\d+)", expand=False)
)

if paired["pmid"].isna().any():
    bad = paired.loc[paired["pmid"].isna(), "case_id"].head(10).tolist()
    raise ValueError(f"Could not extract publication IDs for cases: {bad}")

groups = [
    group.index.to_numpy()
    for _, group in paired.groupby("pmid", sort=False)
]

def metric_vector(r):
    return np.array([
        np.mean(r <= 1),
        np.mean(r <= 5),
        np.mean(r <= 10),
        np.mean(1.0 / r),
    ])

observed = metric_vector(r_model) - metric_vector(r_base)

rng = np.random.default_rng(42)
n_boot = 5000
boot = np.empty((n_boot, 4))
n_groups = len(groups)

for b in range(n_boot):
    sampled_groups = rng.integers(0, n_groups, size=n_groups)
    sampled_rows = np.concatenate([groups[j] for j in sampled_groups])

    boot[b] = (
        metric_vector(r_model[sampled_rows])
        - metric_vector(r_base[sampled_rows])
    )

ci_low, ci_high = np.percentile(boot, [2.5, 97.5], axis=0)

summary = pd.DataFrame({
    "metric": ["Top-1", "Top-5", "Top-10", "MRR"],
    "paired_improvement": observed,
    "95%_CI_lower": ci_low,
    "95%_CI_upper": ci_high,
})

print("Common-coverage test cases:", len(paired))
print("Distinct publications:", n_groups)
print("Bootstrap repetitions:", n_boot)
print(summary.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nStatistical evaluation only — no files modified.")

Common-coverage test cases: 1181
Distinct publications: 231
Bootstrap repetitions: 5000
metric  paired_improvement  95%_CI_lower  95%_CI_upper
 Top-1              0.1431        0.0674        0.2321
 Top-5              0.1473        0.0681        0.2381
Top-10              0.1558        0.0733        0.2503
   MRR              0.1469        0.0724        0.2343

Statistical evaluation only — no files modified.


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

split = pd.read_csv(DATA / "patient_publication_split.csv")
ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

disease_vocab = set(map(str, ckpt["disease_ids"]))

test = split[split["split"] == "test"].copy()
development = split[split["split"] == "development"].copy()

test["diagnosis_ids"] = test["diagnosis_ids"].astype(str)
development["diagnosis_ids"] = development["diagnosis_ids"].astype(str)

test_labels = set(test["diagnosis_ids"])
development_labels = set(development["diagnosis_ids"])

print("=== DISEASE VOCABULARY COVERAGE AUDIT ===")
print("Checkpoint disease vocabulary:", len(disease_vocab))
print("Unique test diagnoses:", len(test_labels))
print("Test diagnoses in vocabulary:",
      len(test_labels & disease_vocab))
print("Test diagnoses absent from vocabulary:",
      len(test_labels - disease_vocab))
print("Unique development diagnoses:", len(development_labels))
print("Development diagnoses in vocabulary:",
      len(development_labels & disease_vocab))
print("Development diagnoses absent from vocabulary:",
      len(development_labels - disease_vocab))

print("\nExamples of test diagnoses absent from vocabulary:")
print(sorted(test_labels - disease_vocab)[:25])

print("\nRead-only audit — no files modified.")

=== DISEASE VOCABULARY COVERAGE AUDIT ===
Checkpoint disease vocabulary: 7340
Unique test diagnoses: 258
Test diagnoses in vocabulary: 195
Test diagnoses absent from vocabulary: 63
Unique development diagnoses: 690
Development diagnoses in vocabulary: 554
Development diagnoses absent from vocabulary: 136

Examples of test diagnoses absent from vocabulary:
['OMIM:107480', 'OMIM:115200', 'OMIM:130050', 'OMIM:147920', 'OMIM:162200', 'OMIM:206700', 'OMIM:231095', 'OMIM:236200', 'OMIM:240300', 'OMIM:258480', 'OMIM:275400', 'OMIM:300867', 'OMIM:300868', 'OMIM:304110', 'OMIM:600373', 'OMIM:601414', 'OMIM:602849', 'OMIM:604377', 'OMIM:605911', 'OMIM:606658', 'OMIM:606693', 'OMIM:609192', 'OMIM:609322', 'OMIM:611291', 'OMIM:611431']

Read-only audit — no files modified.


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

split = pd.read_csv(DATA / "patient_publication_split.csv")
annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False
)

ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

disease_vocab = set(map(str, ckpt["disease_ids"]))

test = split[split["split"] == "test"].copy()
test["diagnosis_ids"] = test["diagnosis_ids"].astype(str)

missing_test_diseases = sorted(
    set(test["diagnosis_ids"]) - disease_vocab
)

annotations["database_id"] = annotations["database_id"].astype(str)
annotations["hpo_id"] = annotations["hpo_id"].astype(str)

profile_counts = (
    annotations.groupby("database_id")["hpo_id"]
    .nunique()
    .rename("hpo_profile_terms")
)

audit = pd.DataFrame({"disease_id": missing_test_diseases})
audit = audit.merge(
    profile_counts,
    left_on="disease_id",
    right_index=True,
    how="left"
)

audit["has_profile"] = audit["hpo_profile_terms"].fillna(0).gt(0)

print("=== MISSING TEST DISEASE PROFILE AUDIT ===")
print("Missing distinct test diseases:", len(audit))
print("Diseases with HPO profiles:", int(audit["has_profile"].sum()))
print("Diseases without HPO profiles:", int((~audit["has_profile"]).sum()))
print("Median HPO terms among profiled diseases:",
      audit.loc[audit["has_profile"], "hpo_profile_terms"].median())

print("\nSample missing diseases and profile sizes:")
print(audit.head(30).to_string(index=False))

print("\nRead-only audit — no files modified.")

=== MISSING TEST DISEASE PROFILE AUDIT ===
Missing distinct test diseases: 63
Diseases with HPO profiles: 63
Diseases without HPO profiles: 0
Median HPO terms among profiled diseases: 43.0

Sample missing diseases and profile sizes:
 disease_id  hpo_profile_terms  has_profile
OMIM:107480                 58         True
OMIM:115200                 14         True
OMIM:130050                 53         True
OMIM:147920                183         True
OMIM:162200                 59         True
OMIM:206700                 59         True
OMIM:231095                 32         True
OMIM:236200                 51         True
OMIM:240300                 63         True
OMIM:258480                 54         True
OMIM:275400                 41         True
OMIM:300867                 51         True
OMIM:300868                202         True
OMIM:304110                 52         True
OMIM:600373                 67         True
OMIM:601414                  6         True
OMIM:602849        

In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False
)

ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

print("=== ANNOTATION DATA ===")
print("Rows:", len(annotations))
print("Columns:", annotations.columns.tolist())
print("Unique disease IDs:", annotations["database_id"].nunique())
print("Unique HPO IDs:", annotations["hpo_id"].nunique())

print("\n=== RANKER VOCABULARY ===")
print("Disease IDs:", len(ckpt["disease_ids"]))

print("\n=== SAMPLE ANNOTATIONS ===")
print(annotations[["database_id", "hpo_id"]].head(10).to_string(index=False))

print("\nRead-only inspection — no files modified.")


=== ANNOTATION DATA ===
Rows: 286047
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']
Unique disease IDs: 12879
Unique HPO IDs: 11658

=== RANKER VOCABULARY ===
Disease IDs: 7340

=== SAMPLE ANNOTATIONS ===
database_id     hpo_id
OMIM:619340 HP:0011097
OMIM:619340 HP:0002187
OMIM:619340 HP:0001518
OMIM:619340 HP:0032792
OMIM:619340 HP:0011451
OMIM:619340 HP:0010851
OMIM:619340 HP:0001789
OMIM:619340 HP:0200134
OMIM:619340 HP:0001522
OMIM:619340 HP:0000006

Read-only inspection — no files modified.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

for filename in [
    "patient_level_validation.csv",
    "patient_phenotype_only_validation.csv",
    "patient_validation_matched_candidate_comparison.csv",
    "patient_level_split_summary.json",
]:
    path = DATA / filename
    print(f"\n=== {filename} ===")

    if not path.exists():
        print("Not found")
        continue

    if path.suffix == ".csv":
        df = pd.read_csv(path, low_memory=False)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))
    else:
        print(path.read_text())

print("\nRead-only inspection — no files modified.")


=== patient_level_validation.csv ===
Shape: (906, 6)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']
                                       case_id     pmid diagnosis_ids                                                                                                                                                                                    hpo_ids  n_hpo  n_diagnoses
                       PMID_28830446_Patient_2 28830446   OMIM:615688                                  HP:0001954;HP:0002027;HP:0002140;HP:0002301;HP:0003493;HP:0005197;HP:0010702;HP:0011107;HP:0025528;HP:0030783;HP:0030880;HP:0033260;HP:0033833;HP:0034056     14            1
                       PMID_28830446_Patient_1 28830446   OMIM:615688 HP:0000623;HP:0001954;HP:0002027;HP:0002140;HP:0002301;HP:0002315;HP:0002321;HP:0002829;HP:0003493;HP:0005197;HP:0011107;HP:0011227;HP:0012246;HP:0025343;HP:0030783;HP:0033505;HP:0033833     17            1
PMID_32480058_Case_F5P1_original_p

In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

validation = pd.read_csv(DATA / "patient_level_validation.csv")
annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    usecols=["database_id", "hpo_id"],
    low_memory=False
)

ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

ranker_diseases = set(map(str, ckpt["disease_ids"]))
profile_diseases = set(annotations["database_id"].astype(str))
targets = validation["diagnosis_ids"].astype(str)

target_in_ranker = targets.isin(ranker_diseases)
target_in_profiles = targets.isin(profile_diseases)

print("=== VALIDATION FALLBACK COVERAGE ===")
print("Validation cases:", len(validation))
print("Target in ranker vocabulary:", int(target_in_ranker.sum()))
print("Target missing from ranker vocabulary:", int((~target_in_ranker).sum()))
print("Target has annotation profile:", int(target_in_profiles.sum()))
print(
    "Cases potentially recoverable by profile fallback:",
    int(((~target_in_ranker) & target_in_profiles).sum())
)
print(
    "Cases missing from both sources:",
    int(((~target_in_ranker) & (~target_in_profiles)).sum())
)

print("\nRead-only audit — no files modified.")

=== VALIDATION FALLBACK COVERAGE ===
Validation cases: 906
Target in ranker vocabulary: 906
Target missing from ranker vocabulary: 0
Target has annotation profile: 906
Cases potentially recoverable by profile fallback: 0
Cases missing from both sources: 0

Read-only audit — no files modified.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

validation = pd.read_csv(DATA / "patient_level_validation.csv")
saved_ranker = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)
saved_baseline = pd.read_csv(
    DATA / "patient_phenotype_only_validation.csv"
)

validation["case_id"] = validation["case_id"].astype(str)
saved_ranker["case_id"] = saved_ranker["case_id"].astype(str)
saved_baseline["case_id"] = saved_baseline["case_id"].astype(str)

df = validation[["case_id", "pmid", "n_hpo"]].merge(
    saved_ranker[["case_id", "rank", "mrr", "top1", "top5", "top10"]],
    on="case_id",
    how="inner"
).rename(columns={
    "rank": "ranker_rank",
    "mrr": "ranker_mrr",
    "top1": "ranker_top1",
    "top5": "ranker_top5",
    "top10": "ranker_top10"
})

df = df.merge(
    saved_baseline[["case_id", "rank", "mrr", "top1", "top5", "top10"]],
    on="case_id",
    how="inner",
    suffixes=("", "_baseline")
)

df["hpo_group"] = pd.cut(
    df["n_hpo"],
    bins=[0, 5, 10, 20, np.inf],
    labels=["1–5 HPO", "6–10 HPO", "11–20 HPO", "21+ HPO"],
    include_lowest=True
)

summary = df.groupby("hpo_group", observed=False).agg(
    cases=("case_id", "count"),
    ranker_top1=("ranker_top1", "mean"),
    ranker_top5=("ranker_top5", "mean"),
    ranker_top10=("ranker_top10", "mean"),
    ranker_mrr=("ranker_mrr", "mean"),
    baseline_top1=("top1", "mean"),
    baseline_top5=("top5", "mean"),
    baseline_top10=("top10", "mean"),
    baseline_mrr=("mrr", "mean")
).reset_index()

print("=== VALIDATION PERFORMANCE BY PHENOTYPE COUNT ===")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nRead-only analysis — no files modified.")

=== VALIDATION PERFORMANCE BY PHENOTYPE COUNT ===
hpo_group  cases  ranker_top1  ranker_top5  ranker_top10  ranker_mrr  baseline_top1  baseline_top5  baseline_top10  baseline_mrr
  1–5 HPO    304       0.2993       0.3454        0.3618      0.3222         0.1118         0.3487          0.4441        0.2105
 6–10 HPO    349       0.3725       0.4585        0.4928      0.4168         0.2350         0.3066          0.3811        0.2879
11–20 HPO    211       0.2227       0.2701        0.3081      0.2499         0.4028         0.5403          0.6303        0.4775
  21+ HPO     42       0.1190       0.1667        0.2619      0.1563         0.9048         1.0000          1.0000        0.9333

Read-only analysis — no files modified.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = {
    "validation": "patient_level_validation.csv",
    "ranker": "patient_ranker_bias_softmax_experiment_validation.csv",
    "baseline": "patient_phenotype_only_validation.csv",
}

for name, filename in files.items():
    path = DATA / filename
    print(f"\n=== {name.upper()} ===")

    if not path.exists():
        print("FILE MISSING:", path)
        continue

    df = pd.read_csv(path, nrows=3)
    print("Columns:", df.columns.tolist())
    print(df.head(2).to_string(index=False))

print("\nRead-only inspection — no files modified.")


=== VALIDATION ===
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']
                case_id     pmid diagnosis_ids                                                                                                                                                                                    hpo_ids  n_hpo  n_diagnoses
PMID_28830446_Patient_2 28830446   OMIM:615688                                  HP:0001954;HP:0002027;HP:0002140;HP:0002301;HP:0003493;HP:0005197;HP:0010702;HP:0011107;HP:0025528;HP:0030783;HP:0030880;HP:0033260;HP:0033833;HP:0034056     14            1
PMID_28830446_Patient_1 28830446   OMIM:615688 HP:0000623;HP:0001954;HP:0002027;HP:0002140;HP:0002301;HP:0002315;HP:0002321;HP:0002829;HP:0003493;HP:0005197;HP:0011107;HP:0011227;HP:0012246;HP:0025343;HP:0030783;HP:0033505;HP:0033833     17            1

=== RANKER ===
Columns: ['case_id', 'diagnosis_id', 'rank', 'top1', 'top5', 'top10', 'mrr']
                case_id diagnosis_id  rank  

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

validation = pd.read_csv(DATA / "patient_level_validation.csv")
ranker = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)
baseline = pd.read_csv(
    DATA / "patient_phenotype_only_validation.csv"
)

for df in [validation, ranker, baseline]:
    df["case_id"] = df["case_id"].astype(str)

print("=== VALIDATION ALIGNMENT AUDIT ===")
print("Validation cases:", len(validation))
print("Ranker rows:", len(ranker))
print("Baseline rows:", len(baseline))

print("Unique validation IDs:", validation["case_id"].nunique())
print("Unique ranker IDs:", ranker["case_id"].nunique())
print("Unique baseline IDs:", baseline["case_id"].nunique())

validation_ids = set(validation["case_id"])
ranker_ids = set(ranker["case_id"])
baseline_ids = set(baseline["case_id"])

print("\nValidation/ranker case sets match:",
      validation_ids == ranker_ids)
print("Validation/baseline case sets match:",
      validation_ids == baseline_ids)

ranker_check = validation[["case_id", "diagnosis_ids"]].merge(
    ranker[["case_id", "diagnosis_id"]],
    on="case_id",
    how="inner"
)
ranker_labels_match = (
    ranker_check["diagnosis_ids"].astype(str)
    == ranker_check["diagnosis_id"].astype(str)
)
print("Ranker diagnosis labels match:",
      int(ranker_labels_match.sum()), "/", len(ranker_check))

baseline_check = validation[["case_id", "diagnosis_ids"]].merge(
    baseline[["case_id", "diagnosis_ids"]],
    on="case_id",
    how="inner",
    suffixes=("_expected", "_baseline")
)
baseline_labels_match = (
    baseline_check["diagnosis_ids_expected"].astype(str)
    == baseline_check["diagnosis_ids_baseline"].astype(str)
)
print("Baseline diagnosis labels match:",
      int(baseline_labels_match.sum()), "/", len(baseline_check))

print("\nRead-only audit — no files modified.")

=== VALIDATION ALIGNMENT AUDIT ===
Validation cases: 906
Ranker rows: 906
Baseline rows: 906
Unique validation IDs: 906
Unique ranker IDs: 906
Unique baseline IDs: 906

Validation/ranker case sets match: True
Validation/baseline case sets match: True
Ranker diagnosis labels match: 906 / 906
Baseline diagnosis labels match: 906 / 906

Read-only audit — no files modified.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

validation = pd.read_csv(DATA / "patient_level_validation.csv")
ranker = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)
baseline = pd.read_csv(
    DATA / "patient_phenotype_only_validation.csv"
)

for df in [validation, ranker, baseline]:
    df["case_id"] = df["case_id"].astype(str)

df = validation[["case_id", "pmid", "n_hpo"]].merge(
    ranker[["case_id", "rank", "mrr", "top1", "top5", "top10"]],
    on="case_id",
    how="inner"
).rename(columns={
    "rank": "ranker_rank",
    "mrr": "ranker_mrr",
    "top1": "ranker_top1",
    "top5": "ranker_top5",
    "top10": "ranker_top10"
})

df = df.merge(
    baseline[["case_id", "rank", "mrr", "top1", "top5", "top10"]],
    on="case_id",
    how="inner",
    suffixes=("", "_baseline")
)

def report(name, ranks):
    ranks = np.asarray(ranks, dtype=float)
    valid = np.isfinite(ranks) & (ranks > 0)
    r = ranks[valid]
    return {
        "strategy": name,
        "cases": len(r),
        "Top-1": np.mean(r <= 1),
        "Top-5": np.mean(r <= 5),
        "Top-10": np.mean(r <= 10),
        "MRR": np.mean(1 / r),
        "median_rank": np.median(r),
    }

rows = [
    report("Ranker", df["ranker_rank"]),
    report("Phenotype baseline", df["rank"]),
]

# A transparent routing experiment using HPO count thresholds.
for threshold in [5, 8, 10, 12, 15, 20]:
    chosen_ranks = np.where(
        df["n_hpo"].to_numpy() <= threshold,
        df["ranker_rank"].to_numpy(),
        df["rank"].to_numpy()
    )
    rows.append(report(f"Ranker if HPO <= {threshold}", chosen_ranks))

summary = pd.DataFrame(rows).sort_values("MRR", ascending=False)

print("=== VALIDATION ROUTING EXPERIMENT ===")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nValidation-only experiment. No training or files modified.")


=== VALIDATION ROUTING EXPERIMENT ===
           strategy  cases  Top-1  Top-5  Top-10    MRR  median_rank
Ranker if HPO <= 10    906 0.3797 0.4647  0.5044 0.4231      10.0000
Ranker if HPO <= 12    906 0.3687 0.4470  0.4857 0.4084      13.0000
 Ranker if HPO <= 8    906 0.3554 0.4437  0.4857 0.4022      12.0000
Ranker if HPO <= 15    906 0.3499 0.4238  0.4558 0.3871      37.5000
 Ranker if HPO <= 5    906 0.3267 0.4062  0.4614 0.3735      16.0000
Ranker if HPO <= 20    906 0.3377 0.4018  0.4294 0.3701      86.0000
 Phenotype baseline    906 0.2638 0.4073  0.4890 0.3360      12.0000
             Ranker    906 0.3013 0.3631  0.3951 0.3341     152.0000

Validation-only experiment. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv")
phenotypes = pd.read_csv(DATA / "phenopacket_patient_cases.csv")
baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")

# Find the saved reconstructed ranker test-rank file.
candidates = sorted(DATA.glob("*ranker*test*.csv"))
print("Ranker test files found:")
for p in candidates:
    print(" -", p.name)

print("\nBaseline rows:", len(baseline))
print("Split columns:", split.columns.tolist())
print("Phenotype columns:", phenotypes.columns.tolist())
print("Baseline columns:", baseline.columns.tolist())


Ranker test files found:

Baseline rows: 1959
Split columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
Phenotype columns: ['case_id', 'source_file', 'hpo_terms', 'phenotype_count', 'diagnosis_ids', 'diagnosis_count']
Baseline columns: ['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']


In [ ]:
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
MODEL_DIR = Path("/content/drive/MyDrive/Rare Disease ai/model")

print("=== PROCESSED FILES RELATED TO RANKING / TEST ===")
for p in sorted(DATA.iterdir()):
    if any(k in p.name.lower() for k in [
        "rank", "test", "baseline", "metric", "prediction"
    ]):
        print(p.name)

print("\n=== MODEL CHECKPOINTS ===")
for p in sorted(MODEL_DIR.glob("*.pt")):
    print(p.name)

=== PROCESSED FILES RELATED TO RANKING / TEST ===
baseline_coverage_comparison_audit.csv
baseline_model_comparison.csv
disease_ranking_retrieval_metrics.csv
gene_informed_phenotype_recovery_metrics.csv
gnn_annotations_excluding_dev_test_pmids.csv
gnn_phenotype_recovery_metrics.csv
hierarchy_aware_retrieval_metrics.csv
ic_weighted_heldout_ranking_results.csv
jaccard_baseline_top10.csv
lost_test_diagnosis_filter_audit.json
lost_test_diagnosis_filter_audit_corrected.json
ontology_fallback_ranked.csv
patient_baseline_vs_gnn_paired.csv
patient_filtered_gnn_linkhead_rankings.csv
patient_filtered_gnn_test_rankings.csv
patient_gnn_training_disease_rankings.csv
patient_level_ranker_bias_free_experiment_history.csv
patient_level_ranker_bias_free_experiment_metrics.json
patient_phenotype_baseline_metrics_audit.csv
patient_phenotype_only_baseline.csv
patient_ranker_bias_ablation.csv
patient_ranker_bias_ablation_summary.json
patient_ranker_bias_softmax_experiment_history.csv
patient_ranker_bias_sof

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "ranking_method_comparison.csv",
    "ranking_method_consolidated_summary.csv",
    "baseline_model_comparison.csv",
    "patient_baseline_vs_gnn_paired.csv",
    "patient_ranker_vs_baseline_paired_bootstrap.json",
    "patient_ranker_diagnostics.json",
]

for name in files:
    path = DATA / name
    print(f"\n{'='*25} {name} {'='*25}")

    if not path.exists():
        print("Not found")
        continue

    if path.suffix == ".csv":
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(8).to_string(index=False))
    else:
        print(path.read_text()[:7000])


========================= ranking_method_comparison.csv =========================
Shape: (5484, 6)
Columns: ['disease_id', 'method', 'top_training_candidate', 'hidden_phenotype_count', 'recovered_hidden_phenotypes', 'hidden_phenotype_recall']
 disease_id      method top_training_candidate  hidden_phenotype_count  recovered_hidden_phenotypes  hidden_phenotype_recall
OMIM:100100 weighted_f1            OMIM:191800                       8                            0                     0.00
OMIM:100100     jaccard            OMIM:191800                       8                            0                     0.00
OMIM:100100  ic_overlap             ORPHA:2970                       8                            6                     0.75
OMIM:101900 weighted_f1            OMIM:620415                       5                            0                     0.00
OMIM:101900     jaccard            OMIM:620415                       5                            0                     0.00
OMIM:1

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

phenotypes = pd.read_csv(DATA / "phenopacket_patient_cases.csv")
baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")
gnn = pd.read_csv(DATA / "patient_baseline_vs_gnn_paired.csv")

print("GNN test rows:", len(gnn))
print("Unique GNN cases:", gnn["case_id"].nunique())
print("Phenotype rows:", len(phenotypes))
print("Baseline rows:", len(baseline))

print("\nGNN columns:")
print(gnn.columns.tolist())

print("\nHPO-count fields available:")
print([
    c for c in phenotypes.columns
    if "hpo" in c.lower() or "phenotype" in c.lower()
])

print("\nGNN target coverage:")
print(gnn["target_in_candidate_set"].value_counts(dropna=False).to_string())

print("\nGNN ranking metrics (all rows):")
for col in ["top1_gnn", "top5_gnn", "top10_gnn", "mrr"]:
    if col in gnn.columns:
        print(f"{col}: {pd.to_numeric(gnn[col], errors='coerce').mean():.4f}")

GNN test rows: 1959
Unique GNN cases: 1959
Phenotype rows: 10427
Baseline rows: 1959

GNN columns:
['case_id', 'pmid', 'diagnosis_ids_baseline', 'rank_baseline', 'top1_baseline', 'top5_baseline', 'top10_baseline', 'candidate_covered', 'diagnosis_ids_gnn', 'n_positive_hpo', 'n_recognized_hpo', 'target_in_candidate_set', 'rank_gnn', 'top1_gnn', 'top5_gnn', 'top10_gnn', 'mrr', 'top_prediction']

HPO-count fields available:
['hpo_terms', 'phenotype_count']

GNN target coverage:
target_in_candidate_set
True     1492
False     467

GNN ranking metrics (all rows):
top1_gnn: 0.0316
top5_gnn: 0.0572
top10_gnn: 0.0781
mrr: 0.0476


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

phenotypes = pd.read_csv(DATA / "phenopacket_patient_cases.csv")
baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")
gnn = pd.read_csv(DATA / "patient_baseline_vs_gnn_paired.csv")

print("GNN test rows:", len(gnn))
print("Unique GNN cases:", gnn["case_id"].nunique())
print("Phenotype rows:", len(phenotypes))
print("Baseline rows:", len(baseline))

print("\nGNN columns:")
print(gnn.columns.tolist())

print("\nHPO-count fields available:")
print([
    c for c in phenotypes.columns
    if "hpo" in c.lower() or "phenotype" in c.lower()
])

print("\nGNN target coverage:")
print(gnn["target_in_candidate_set"].value_counts(dropna=False).to_string())

print("\nGNN ranking metrics (all rows):")
for col in ["top1_gnn", "top5_gnn", "top10_gnn", "mrr"]:
    if col in gnn.columns:
        print(f"{col}: {pd.to_numeric(gnn[col], errors='coerce').mean():.4f}")

GNN test rows: 1959
Unique GNN cases: 1959
Phenotype rows: 10427
Baseline rows: 1959

GNN columns:
['case_id', 'pmid', 'diagnosis_ids_baseline', 'rank_baseline', 'top1_baseline', 'top5_baseline', 'top10_baseline', 'candidate_covered', 'diagnosis_ids_gnn', 'n_positive_hpo', 'n_recognized_hpo', 'target_in_candidate_set', 'rank_gnn', 'top1_gnn', 'top5_gnn', 'top10_gnn', 'mrr', 'top_prediction']

HPO-count fields available:
['hpo_terms', 'phenotype_count']

GNN target coverage:
target_in_candidate_set
True     1492
False     467

GNN ranking metrics (all rows):
top1_gnn: 0.0316
top5_gnn: 0.0572
top10_gnn: 0.0781
mrr: 0.0476


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

phenotypes = pd.read_csv(DATA / "phenopacket_patient_cases.csv")
gnn = pd.read_csv(DATA / "patient_baseline_vs_gnn_paired.csv")

phenotypes["case_id"] = phenotypes["case_id"].astype(str)
gnn["case_id"] = gnn["case_id"].astype(str)

# Compute the number of observed HPO terms per row.
phenotypes["n_hpo_actual"] = phenotypes["hpo_terms"].fillna("").astype(str).apply(
    lambda x: len([t for t in x.split(";") if t.strip()])
)

# Collapse duplicate case IDs conservatively: use the maximum observed count.
hpo_counts = (
    phenotypes.groupby("case_id", as_index=False)["n_hpo_actual"]
    .max()
)

print("Unique phenotype case IDs:", hpo_counts["case_id"].nunique())
print("Duplicate phenotype rows:", phenotypes["case_id"].duplicated().sum())

df = gnn.merge(
    hpo_counts,
    on="case_id",
    how="left",
    validate="one_to_one"
)

print("Merged rows:", len(df))
print("Cases missing HPO counts:", df["n_hpo_actual"].isna().sum())

def metrics(name, ranks):
    ranks = pd.to_numeric(pd.Series(ranks), errors="coerce").to_numpy()
    ranks = np.where(np.isfinite(ranks) & (ranks > 0), ranks, np.inf)
    return {
        "strategy": name,
        "cases": len(ranks),
        "Top-1": np.mean(ranks <= 1),
        "Top-5": np.mean(ranks <= 5),
        "Top-10": np.mean(ranks <= 10),
        "MRR": np.mean(1 / ranks),
    }

rows = [
    metrics("Phenotype baseline", df["rank_baseline"]),
    metrics("GNN", df["rank_gnn"]),
]

for threshold in [5, 10, 15]:
    use_gnn = df["n_hpo_actual"].fillna(0).to_numpy() <= threshold
    routed = np.where(
        use_gnn,
        pd.to_numeric(df["rank_gnn"], errors="coerce").to_numpy(),
        pd.to_numeric(df["rank_baseline"], errors="coerce").to_numpy()
    )
    rows.append(metrics(f"GNN if HPO <= {threshold}", routed))

print("\n=== HELD-OUT TEST COMPARISON ===")
print(pd.DataFrame(rows).to_string(
    index=False, float_format=lambda x: f"{x:.4f}"
))
print("\nRead-only evaluation; no training or files modified.")

Unique phenotype case IDs: 10377
Duplicate phenotype rows: 50
Merged rows: 1959
Cases missing HPO counts: 0

=== HELD-OUT TEST COMPARISON ===
          strategy  cases  Top-1  Top-5  Top-10    MRR
Phenotype baseline   1959 0.1138 0.1960  0.2210 0.1527
               GNN   1959 0.0316 0.0572  0.0781 0.0476
   GNN if HPO <= 5   1959 0.0995 0.1582  0.1776 0.1284
  GNN if HPO <= 10   1959 0.0567 0.0949  0.1149 0.0773
  GNN if HPO <= 15   1959 0.0413 0.0730  0.0919 0.0591

Read-only evaluation; no training or files modified.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

phenotypes = pd.read_csv(DATA / "phenopacket_patient_cases.csv")
gnn = pd.read_csv(DATA / "patient_baseline_vs_gnn_paired.csv")

phenotypes["case_id"] = phenotypes["case_id"].astype(str)
gnn["case_id"] = gnn["case_id"].astype(str)

phenotypes["n_hpo_actual"] = phenotypes["hpo_terms"].fillna("").astype(str).apply(
    lambda x: len(set(t.strip() for t in x.split(";") if t.strip()))
)

counts = phenotypes.groupby("case_id")["n_hpo_actual"].max()
check = gnn[["case_id", "n_recognized_hpo"]].copy()
check["source_hpo_count"] = check["case_id"].map(counts)

check["difference"] = (
    check["source_hpo_count"] - check["n_recognized_hpo"]
)

print("Test cases:", len(check))
print("Missing source counts:", check["source_hpo_count"].isna().sum())
print("Exact count matches:", (check["difference"] == 0).sum())
print("Median source HPO count:", check["source_hpo_count"].median())
print("Median recognized HPO count:", check["n_recognized_hpo"].median())
print("Source-minus-recognized count distribution:")
print(check["difference"].value_counts().sort_index().head(15).to_string())

Test cases: 1959
Missing source counts: 0
Exact count matches: 1895
Median source HPO count: 8.0
Median recognized HPO count: 8.0
Source-minus-recognized count distribution:
difference
0    1895
1      59
2       5


In [ ]:
from pathlib import Path
import torch
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

checkpoint_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint exists:", checkpoint_path.exists())
print("Checkpoint keys:", list(checkpoint.keys()))

disease_ids = checkpoint.get("disease_ids", [])
phenotype_ids = checkpoint.get("phenotype_ids", [])

print("Checkpoint disease vocabulary:", len(disease_ids))
print("Checkpoint phenotype vocabulary:", len(phenotype_ids))

phenotypes = pd.read_csv(DATA / "phenopacket_patient_cases.csv")
test_ids = set(
    pd.read_csv(DATA / "patient_publication_split.csv")
    .query("split == 'test'")["case_id"].astype(str)
)
test_pheno = phenotypes[
    phenotypes["case_id"].astype(str).isin(test_ids)
]

print("Test cases:", len(test_ids))
print("Test cases with phenotype records:", test_pheno["case_id"].nunique())
print("Checkpoint best epoch:", checkpoint.get("best_epoch"))
print("Checkpoint training cases:", checkpoint.get("training_cases"))
print("Checkpoint validation cases:", checkpoint.get("validation_cases"))


Checkpoint exists: True
Checkpoint keys: ['model_state_dict', 'disease_ids', 'phenotype_ids', 'hidden_dim', 'best_epoch', 'best_validation_metrics', 'seed', 'training_cases', 'validation_cases', 'objective', 'disease_bias', 'note']
Checkpoint disease vocabulary: 7340
Checkpoint phenotype vocabulary: 9982
Test cases: 1959
Test cases with phenotype records: 1959
Checkpoint best epoch: 15
Checkpoint training cases: 5730
Checkpoint validation cases: 906


In [ ]:
from pathlib import Path
import torch
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

print("=== CHECKPOINT WEIGHT SHAPES ===")
for name, tensor in state.items():
    print(f"{name}: {tuple(tensor.shape)}")

print("\n=== VOCABULARY CHECK ===")
disease_ids = [str(x) for x in ckpt["disease_ids"]]
phenotype_ids = [str(x) for x in ckpt["phenotype_ids"]]

print("Unique disease IDs:", len(set(disease_ids)))
print("Unique phenotype IDs:", len(set(phenotype_ids)))
print("Hidden dimension:", ckpt["hidden_dim"])

split = pd.read_csv(DATA / "patient_publication_split.csv")
test_ids = set(
    split.loc[split["split"] == "test", "case_id"].astype(str)
)

phenotypes = pd.read_csv(DATA / "phenopacket_patient_cases.csv")
test_pheno = phenotypes[
    phenotypes["case_id"].astype(str).isin(test_ids)
].copy()

hpo_vocab = set(phenotype_ids)
observed_terms = set()

for value in test_pheno["hpo_terms"].dropna().astype(str):
    observed_terms.update(
        term.strip()
        for term in value.split(";")
        if term.strip()
    )

print("\n=== TEST INPUT COVERAGE ===")
print("Test phenotype rows:", len(test_pheno))
print("Unique test cases:", test_pheno["case_id"].nunique())
print("Distinct observed HPO terms:", len(observed_terms))
print("Observed HPO terms in checkpoint vocabulary:",
      len(observed_terms & hpo_vocab))
print("Observed HPO terms outside vocabulary:",
      len(observed_terms - hpo_vocab))

=== CHECKPOINT WEIGHT SHAPES ===
disease_bias: (7340,)
hpo_embedding.weight: (9982, 128)
disease_embedding.weight: (7340, 128)
patient_projection.0.weight: (128, 128)
patient_projection.0.bias: (128,)
patient_projection.3.weight: (128, 128)
patient_projection.3.bias: (128,)

=== VOCABULARY CHECK ===
Unique disease IDs: 7340
Unique phenotype IDs: 9982
Hidden dimension: 128

=== TEST INPUT COVERAGE ===
Test phenotype rows: 1959
Unique test cases: 1959
Distinct observed HPO terms: 2004
Observed HPO terms in checkpoint vocabulary: 1970
Observed HPO terms outside vocabulary: 34


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

disease_ids = [str(x) for x in ckpt["disease_ids"]]
phenotype_ids = [str(x) for x in ckpt["phenotype_ids"]]
disease_to_idx = {d: i for i, d in enumerate(disease_ids)}
hpo_to_idx = {h: i for i, h in enumerate(phenotype_ids)}

class SavedRanker(nn.Module):
    def __init__(self, n_hpo, n_disease, hidden_dim):
        super().__init__()
        self.hpo_embedding = nn.Embedding(n_hpo, hidden_dim)
        self.disease_embedding = nn.Embedding(n_disease, hidden_dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.disease_bias = nn.Parameter(torch.zeros(n_disease))

    def forward(self, x):
        mask = (x != 0).float().unsqueeze(-1)
        embedded = self.hpo_embedding(x)
        pooled = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        patient = self.patient_projection(pooled)
        return patient @ self.disease_embedding.weight.T + self.disease_bias

model = SavedRanker(
    len(phenotype_ids), len(disease_ids), int(ckpt["hidden_dim"])
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

split = pd.read_csv(DATA / "patient_publication_split.csv")
test_ids = set(
    split.loc[split["split"] == "test", "case_id"].astype(str)
)

phenotypes = pd.read_csv(DATA / "phenopacket_patient_cases.csv")
test_pheno = phenotypes[
    phenotypes["case_id"].astype(str).isin(test_ids)
].copy()

# Deduplicate cases by combining their observed HPO terms.
case_records = []
for case_id, group in test_pheno.groupby("case_id", sort=False):
    terms = set()
    for value in group["hpo_terms"].dropna().astype(str):
        terms.update(t.strip() for t in value.split(";") if t.strip())

    split_row = split.loc[split["case_id"].astype(str) == str(case_id)].iloc[0]
    targets = {
        t.strip() for t in str(split_row["diagnosis_ids"]).split(";")
        if t.strip() and t.strip() != "nan"
    }
    case_records.append({
        "case_id": str(case_id),
        "targets": targets,
        "hpos": sorted(terms),
    })

max_len = max(1, max(len(r["hpos"]) for r in case_records))
X = np.zeros((len(case_records), max_len), dtype=np.int64)

for i, record in enumerate(case_records):
    ids = [hpo_to_idx[h] for h in record["hpos"] if h in hpo_to_idx]
    if ids:
        X[i, :len(ids)] = ids

with torch.no_grad():
    scores = model(torch.tensor(X, dtype=torch.long)).numpy()

# Rank each known target; diagnoses outside the checkpoint vocabulary count as misses.
ranker_ranks = []
for i, record in enumerate(case_records):
    target_indices = [
        disease_to_idx[t] for t in record["targets"] if t in disease_to_idx
    ]
    if not target_indices:
        ranker_ranks.append(np.inf)
        continue

    row = scores[i]
    order = np.argsort(-row)
    positions = np.empty_like(order)
    positions[order] = np.arange(1, len(order) + 1)
    ranker_ranks.append(float(min(positions[j] for j in target_indices)))

ranker_ranks = np.asarray(ranker_ranks, dtype=float)

baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")
baseline["case_id"] = baseline["case_id"].astype(str)
baseline = baseline.set_index("case_id")

baseline_ranks = np.array([
    float(baseline.loc[r["case_id"], "rank"])
    if r["case_id"] in baseline.index
       and pd.notna(baseline.loc[r["case_id"], "rank"])
       and float(baseline.loc[r["case_id"], "rank"]) > 0
    else np.inf
    for r in case_records
])

def summarize(name, ranks):
    ranks = np.asarray(ranks, dtype=float)
    return {
        "method": name,
        "cases": len(ranks),
        "target_covered": int(np.isfinite(ranks).sum()),
        "Top-1": np.mean(ranks <= 1),
        "Top-5": np.mean(ranks <= 5),
        "Top-10": np.mean(ranks <= 10),
        "MRR": np.mean(np.where(np.isfinite(ranks), 1 / ranks, 0)),
        "median_rank_covered": (
            float(np.median(ranks[np.isfinite(ranks)]))
            if np.isfinite(ranks).any() else np.nan
        ),
    }

print("=== HELD-OUT TEST: RECONSTRUCTED RANKER VS SAVED BASELINE ===")
print(pd.DataFrame([
    summarize("Reconstructed bias-softmax ranker", ranker_ranks),
    summarize("Saved phenotype baseline", baseline_ranks),
]).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nTest cases evaluated:", len(case_records))
print("No training performed; no files modified.")
print("Caveat: ranker inference is reconstructed from checkpoint weights.")

=== HELD-OUT TEST: RECONSTRUCTED RANKER VS SAVED BASELINE ===
                           method  cases  target_covered  Top-1  Top-5  Top-10    MRR  median_rank_covered
Reconstructed bias-softmax ranker   1959            1492 0.1577 0.2266  0.2527 0.1911             150.0000
         Saved phenotype baseline   1959            1551 0.1138 0.1960  0.2210 0.1527             258.0000

Test cases evaluated: 1959
No training performed; no files modified.
Caveat: ranker inference is reconstructed from checkpoint weights.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Reuse the saved baseline and reconstruct the ranker ranks
# from the checkpoint, as in the previous cell.
# If ranker_ranks and baseline_ranks still exist in memory,
# this cell uses them directly.

if "ranker_ranks" not in globals() or "baseline_ranks" not in globals():
    raise RuntimeError(
        "Please rerun the previous ranker inference cell first; "
        "this comparison requires ranker_ranks and baseline_ranks."
    )

ranker_ranks = np.asarray(ranker_ranks, dtype=float)
baseline_ranks = np.asarray(baseline_ranks, dtype=float)

if len(ranker_ranks) != len(baseline_ranks):
    raise ValueError("Ranker and baseline arrays have different lengths.")

# Common coverage: both methods have a finite, positive target rank.
common = (
    np.isfinite(ranker_ranks) & (ranker_ranks > 0) &
    np.isfinite(baseline_ranks) & (baseline_ranks > 0)
)

def metrics(name, ranks):
    ranks = np.asarray(ranks, dtype=float)
    return {
        "method": name,
        "cases": len(ranks),
        "Top-1": np.mean(ranks <= 1),
        "Top-5": np.mean(ranks <= 5),
        "Top-10": np.mean(ranks <= 10),
        "MRR": np.mean(1.0 / ranks),
        "median_rank": np.median(ranks),
    }

print("Total test cases:", len(ranker_ranks))
print("Ranker target coverage:", np.isfinite(ranker_ranks).sum())
print("Baseline target coverage:", np.isfinite(baseline_ranks).sum())
print("Common-coverage cases:", int(common.sum()))

print("\n=== COMMON-COVERAGE COMPARISON ===")
result = pd.DataFrame([
    metrics("Reconstructed ranker", ranker_ranks[common]),
    metrics("Phenotype baseline", baseline_ranks[common]),
])
print(result.to_string(
    index=False, float_format=lambda x: f"{x:.4f}"
))

# Paired per-case differences in reciprocal rank.
rr_ranker = 1.0 / ranker_ranks[common]
rr_baseline = 1.0 / baseline_ranks[common]
delta = rr_ranker - rr_baseline

print("\nPaired MRR difference (ranker - baseline):", f"{delta.mean():.4f}")
print("Cases ranker wins on reciprocal rank:", int((delta > 0).sum()))
print("Cases tied:", int((delta == 0).sum()))
print("Cases baseline wins on reciprocal rank:", int((delta < 0).sum()))
print("\nRead-only comparison; no training or files modified.")

Total test cases: 1959
Ranker target coverage: 1492
Baseline target coverage: 1551
Common-coverage cases: 1181

=== COMMON-COVERAGE COMPARISON ===
              method  cases  Top-1  Top-5  Top-10    MRR  median_rank
Reconstructed ranker   1181 0.2591 0.3709  0.4124 0.3128      31.0000
  Phenotype baseline   1181 0.1160 0.2235  0.2566 0.1659     334.0000

Paired MRR difference (ranker - baseline): 0.1469
Cases ranker wins on reciprocal rank: 675
Cases tied: 77
Cases baseline wins on reciprocal rank: 429

Read-only comparison; no training or files modified.


In [ ]:
import numpy as np

# Uses ranker_ranks and baseline_ranks from the previous cells.
ranker_ranks = np.asarray(ranker_ranks, dtype=float)
baseline_ranks = np.asarray(baseline_ranks, dtype=float)

common = (
    np.isfinite(ranker_ranks) & (ranker_ranks > 0) &
    np.isfinite(baseline_ranks) & (baseline_ranks > 0)
)

r = ranker_ranks[common]
b = baseline_ranks[common]

def metric_values(ranks):
    return np.array([
        np.mean(ranks <= 1),
        np.mean(ranks <= 5),
        np.mean(ranks <= 10),
        np.mean(1.0 / ranks),
    ])

metric_names = ["Top-1", "Top-5", "Top-10", "MRR"]
observed_delta = metric_values(r) - metric_values(b)

rng = np.random.default_rng(42)
n_boot = 5000
boot_deltas = np.empty((n_boot, 4))
n = len(r)

for i in range(n_boot):
    idx = rng.integers(0, n, size=n)
    boot_deltas[i] = metric_values(r[idx]) - metric_values(b[idx])

print("Matched test cases:", n)
print("Paired bootstrap replicates:", n_boot)
print("\nRanker minus baseline, paired 95% bootstrap CIs:")

for j, name in enumerate(metric_names):
    lo, hi = np.quantile(boot_deltas[:, j], [0.025, 0.975])
    print(
        f"{name:7s}: delta={observed_delta[j]:.4f}, "
        f"95% CI [{lo:.4f}, {hi:.4f}]"
    )

print("\nInference caveat: ranker predictions were reconstructed from the checkpoint.")
print("No training or files modified.")

Matched test cases: 1181
Paired bootstrap replicates: 5000

Ranker minus baseline, paired 95% bootstrap CIs:
Top-1  : delta=0.1431, 95% CI [0.1160, 0.1710]
Top-5  : delta=0.1473, 95% CI [0.1202, 0.1753]
Top-10 : delta=0.1558, 95% CI [0.1279, 0.1846]
MRR    : delta=0.1469, 95% CI [0.1232, 0.1718]

Inference caveat: ranker predictions were reconstructed from the checkpoint.
No training or files modified.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

# Confirm the previous inference and baseline comparison are still in memory.
required = ["case_records", "ranker_ranks", "baseline_ranks"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Rerun the previous inference/comparison "
        "cells before running this export."
    )

ranker_ranks = np.asarray(ranker_ranks, dtype=float)
baseline_ranks = np.asarray(baseline_ranks, dtype=float)

if not (len(case_records) == len(ranker_ranks) == len(baseline_ranks)):
    raise ValueError("Case records and rank arrays have different lengths.")

def summarize(ranks):
    ranks = np.asarray(ranks, dtype=float)
    covered = np.isfinite(ranks) & (ranks > 0)
    return {
        "cases": int(len(ranks)),
        "target_covered": int(covered.sum()),
        "target_coverage": float(covered.mean()),
        "top1": float(np.mean(ranks <= 1)),
        "top5": float(np.mean(ranks <= 5)),
        "top10": float(np.mean(ranks <= 10)),
        "mrr": float(np.mean(np.where(covered, 1.0 / ranks, 0.0))),
        "median_rank_covered": (
            float(np.median(ranks[covered])) if covered.any() else None
        ),
    }

case_rows = []
for i, record in enumerate(case_records):
    rr = ranker_ranks[i]
    br = baseline_ranks[i]
    case_rows.append({
        "case_id": record["case_id"],
        "target_diagnoses": ";".join(sorted(record["targets"])),
        "n_observed_hpo": len(record["hpos"]),
        "ranker_target_covered": bool(np.isfinite(rr) and rr > 0),
        "ranker_rank": int(rr) if np.isfinite(rr) and rr > 0 else None,
        "ranker_top1": bool(rr <= 1),
        "ranker_top5": bool(rr <= 5),
        "ranker_top10": bool(rr <= 10),
        "ranker_reciprocal_rank": float(1.0 / rr) if np.isfinite(rr) and rr > 0 else 0.0,
        "baseline_target_covered": bool(np.isfinite(br) and br > 0),
        "baseline_rank": int(br) if np.isfinite(br) and br > 0 else None,
        "baseline_top1": bool(br <= 1),
        "baseline_top5": bool(br <= 5),
        "baseline_top10": bool(br <= 10),
        "baseline_reciprocal_rank": float(1.0 / br) if np.isfinite(br) and br > 0 else 0.0,
        "both_cover_target": bool(
            np.isfinite(rr) and rr > 0 and np.isfinite(br) and br > 0
        ),
    })

case_df = pd.DataFrame(case_rows)

csv_path = DATA / "patient_ranker_reconstructed_test_case_results.csv"
case_df.to_csv(csv_path, index=False)

common = (
    np.isfinite(ranker_ranks) & (ranker_ranks > 0) &
    np.isfinite(baseline_ranks) & (baseline_ranks > 0)
)

metrics = {
    "evaluation": "Held-out test ranking comparison",
    "test_cases": int(len(case_df)),
    "ranker": summarize(ranker_ranks),
    "phenotype_baseline": summarize(baseline_ranks),
    "common_coverage_cases": int(common.sum()),
    "common_coverage_ranker": summarize(ranker_ranks[common]),
    "common_coverage_baseline": summarize(baseline_ranks[common]),
    "paired_bootstrap_common_coverage": {
        "replicates": 5000,
        "seed": 42,
        "deltas_ranker_minus_baseline": {
            "top1": {
                "estimate": 0.1431,
                "ci95": [0.1160, 0.1710]
            },
            "top5": {
                "estimate": 0.1473,
                "ci95": [0.1202, 0.1753]
            },
            "top10": {
                "estimate": 0.1558,
                "ci95": [0.1279, 0.1846]
            },
            "mrr": {
                "estimate": 0.1469,
                "ci95": [0.1232, 0.1718]
            }
        },
        "note": "Previously calculated paired case bootstrap results."
    },
    "caveats": [
        "Ranker inference was reconstructed from the saved checkpoint.",
        "Original inference source code was not recovered or verified.",
        "467 test cases have target diagnoses outside the ranker vocabulary.",
        "Research prototype only; not clinically validated."
    ]
}

json_path = DATA / "patient_ranker_reconstructed_test_metrics.json"
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Export complete.")
print("Case results:", csv_path)
print("Metrics:", json_path)
print("Case rows:", len(case_df))
print("Unique case IDs:", case_df["case_id"].nunique())
print("Both methods cover target:", int(common.sum()))
print("Existing checkpoints modified: No")

Export complete.
Case results: /content/drive/MyDrive/Rare Disease ai/processed/patient_ranker_reconstructed_test_case_results.csv
Metrics: /content/drive/MyDrive/Rare Disease ai/processed/patient_ranker_reconstructed_test_metrics.json
Case rows: 1959
Unique case IDs: 1959
Both methods cover target: 1181
Existing checkpoints modified: No


In [ ]:
from pathlib import Path
import torch
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

files = {
    "Ranker checkpoint": MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    "Phenotype baseline": DATA / "patient_phenotype_only_baseline.csv",
    "Test case results": DATA / "patient_ranker_reconstructed_test_case_results.csv",
    "Test metrics": DATA / "patient_ranker_reconstructed_test_metrics.json",
    "Phenotype cases": DATA / "phenopacket_patient_cases.csv",
}

print("=== DEMO READINESS CHECK ===")
for label, path in files.items():
    print(f"{label}: {'FOUND' if path.exists() else 'MISSING'}")

if files["Ranker checkpoint"].exists():
    ckpt = torch.load(
        files["Ranker checkpoint"],
        map_location="cpu",
        weights_only=False
    )
    print("\nDisease candidates:", len(ckpt["disease_ids"]))
    print("Recognized HPO vocabulary:", len(ckpt["phenotype_ids"]))
    print("Hidden dimension:", ckpt["hidden_dim"])

print("\nExisting checkpoints will not be modified.")

=== DEMO READINESS CHECK ===
Ranker checkpoint: FOUND
Phenotype baseline: FOUND
Test case results: FOUND
Test metrics: FOUND
Phenotype cases: FOUND

Disease candidates: 7340
Recognized HPO vocabulary: 9982
Hidden dimension: 128

Existing checkpoints will not be modified.


In [ ]:

from pathlib import Path
import torch
import torch.nn as nn
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

CHECKPOINT_PATH = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
ANNOTATIONS_PATH = DATA / "clean_hpo_annotations.csv"

# Load saved checkpoint without retraining.
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

disease_ids = [str(x) for x in checkpoint["disease_ids"]]
phenotype_ids = [str(x) for x in checkpoint["phenotype_ids"]]

hpo_to_idx = {hpo: i for i, hpo in enumerate(phenotype_ids)}

class SavedRanker(nn.Module):
    def __init__(self, n_hpo, n_disease, hidden_dim):
        super().__init__()
        self.hpo_embedding = nn.Embedding(n_hpo, hidden_dim)
        self.disease_embedding = nn.Embedding(n_disease, hidden_dim)
        self.patient_projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.disease_bias = nn.Parameter(torch.zeros(n_disease))

    def forward(self, x):
        mask = (x != 0).float().unsqueeze(-1)
        embedded = self.hpo_embedding(x)
        pooled = (embedded * mask).sum(dim=1)
        pooled = pooled / mask.sum(dim=1).clamp(min=1)
        patient = self.patient_projection(pooled)
        return patient @ self.disease_embedding.weight.T + self.disease_bias

model = SavedRanker(
    len(phenotype_ids),
    len(disease_ids),
    int(checkpoint["hidden_dim"])
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Load disease names and HPO profiles for display.
annotations = pd.read_csv(ANNOTATIONS_PATH, low_memory=False)
annotations["database_id"] = annotations["database_id"].astype(str)
annotations["hpo_id"] = annotations["hpo_id"].astype(str)

disease_names = (
    annotations.dropna(subset=["disease_name"])
    .drop_duplicates("database_id")
    .set_index("database_id")["disease_name"]
    .to_dict()
)

def predict_diseases(hpo_terms, top_k=10):
    """
    hpo_terms: iterable of HPO IDs, e.g. ['HP:0001250', 'HP:0001263'].
    Returns ranked candidate diseases and input-coverage information.
    """
    if isinstance(hpo_terms, str):
        hpo_terms = hpo_terms.replace(",", ";").split(";")

    terms = sorted({
        str(term).strip()
        for term in hpo_terms
        if str(term).strip()
    })

    recognized = [term for term in terms if term in hpo_to_idx]
    unrecognized = [term for term in terms if term not in hpo_to_idx]

    if not recognized:
        return {
            "error": "No input HPO terms were recognized by the saved model.",
            "recognized_terms": [],
            "unrecognized_terms": unrecognized,
            "candidates": pd.DataFrame()
        }

    x = torch.tensor(
        [[hpo_to_idx[t] for t in recognized]],
        dtype=torch.long
    )

    with torch.no_grad():
        scores = model(x)[0]

    k = min(max(1, int(top_k)), len(disease_ids))
    top_indices = torch.topk(scores, k=k).indices.tolist()

    candidates = pd.DataFrame([
        {
            "rank": rank,
            "disease_id": disease_ids[idx],
            "disease_name": disease_names.get(
                disease_ids[idx], "Name unavailable"
            ),
            "model_score": float(scores[idx].item())
        }
        for rank, idx in enumerate(top_indices, start=1)
    ])

    return {
        "input_term_count": len(terms),
        "recognized_count": len(recognized),
        "unrecognized_count": len(unrecognized),
        "recognized_terms": recognized,
        "unrecognized_terms": unrecognized,
        "candidates": candidates
    }

print("Demo inference model loaded successfully.")
print("Disease candidates:", len(disease_ids))
print("HPO vocabulary:", len(phenotype_ids))
print("Ready to accept HPO IDs.")


Demo inference model loaded successfully.
Disease candidates: 7340
HPO vocabulary: 9982
Ready to accept HPO IDs.


In [ ]:
# Example phenotype input for a smoke test only — not a real patient.
example_hpo_terms = [
    "HP:0001250",
    "HP:0001263",
    "HP:0004322",
]

result = predict_diseases(example_hpo_terms, top_k=10)

if "error" in result:
    print(result["error"])
    print("Unrecognized terms:", result["unrecognized_terms"])
else:
    print("=== DEMO SMOKE TEST ===")
    print("Input HPO terms:", result["input_term_count"])
    print("Recognized terms:", result["recognized_count"])
    print("Unrecognized terms:", result["unrecognized_count"])

    print("\nRecognized:", result["recognized_terms"])
    print("Unrecognized:", result["unrecognized_terms"])

    print("\n=== TOP 10 CANDIDATE DISEASES ===")
    display(result["candidates"])

=== DEMO SMOKE TEST ===
Input HPO terms: 3
Recognized terms: 3
Unrecognized terms: 0

Recognized: ['HP:0001250', 'HP:0001263', 'HP:0004322']
Unrecognized: []

=== TOP 10 CANDIDATE DISEASES ===


,rank,disease_id,disease_name,model_score
0,1,OMIM:614800,"Short stature, optic nerve atrophy, and pelger...",17.763851
1,2,OMIM:272300,Sulfite oxidase deficiency,17.735012
2,3,OMIM:615829,Xia-Gibbs syndrome,16.731421
3,4,OMIM:610253,Kleefstra syndrome 1,15.159489
4,5,OMIM:620851,ReNU syndrome,14.705893
5,6,OMIM:148050,KBG syndrome,14.349082
6,7,OMIM:611091,"Intellectual developmental disorder, autosomal...",14.198501
7,8,OMIM:618089,Intellectual developmental disorder with dysmo...,14.113952
8,9,OMIM:612164,Developmental and epileptic encephalopathy 4,11.971632
9,10,OMIM:301310,"Anemia, sideroblastic, and spinocerebellar ataxia",11.740572


In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output

hpo_input = widgets.Textarea(
    value="HP:0001250; HP:0001263; HP:0004322",
    placeholder="Enter HPO IDs separated by semicolons or commas",
    description="HPO IDs:",
    layout=widgets.Layout(width="100%", height="100px"),
    style={"description_width": "initial"}
)

top_k_input = widgets.Dropdown(
    options=[5, 10, 15, 20],
    value=10,
    description="Candidates:",
    style={"description_width": "initial"}
)

predict_button = widgets.Button(
    description="Rank candidate diseases",
    button_style="primary",
    icon="search"
)

output = widgets.Output()

def on_predict_clicked(_):
    with output:
        clear_output(wait=True)

        terms = [
            term.strip()
            for term in hpo_input.value.replace(",", ";").split(";")
            if term.strip()
        ]

        if not terms:
            print("Please enter at least one HPO ID.")
            return

        result = predict_diseases(terms, top_k=top_k_input.value)

        if "error" in result:
            print(result["error"])
            print("Unrecognized terms:", result["unrecognized_terms"])
            return

        print(f"Input terms: {result['input_term_count']}")
        print(f"Recognized: {result['recognized_count']}")
        print(f"Unrecognized: {result['unrecognized_count']}")

        if result["unrecognized_terms"]:
            print("Unrecognized HPO IDs:", result["unrecognized_terms"])

        print("\nRanked candidate diseases:")
        display(result["candidates"].drop(columns=["model_score"]))

        print(
            "\nResearch prototype only. These rankings are candidate suggestions, "
            "not diagnoses. Consult a qualified medical genetics professional."
        )

predict_button.on_click(on_predict_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Rare Disease Prediction AI</h3>"),
    widgets.HTML(
        "<p>Enter Human Phenotype Ontology (HPO) IDs to rank candidate diseases.</p>"
    ),
    hpo_input,
    top_k_input,
    predict_button,
    output
]))


In [ ]:
# Final input-validation checks — no model training or file changes.

test_inputs = {
    "Recognized HPO terms": ["HP:0001250", "HP:0001263"],
    "Unknown HPO term": ["HP:9999999"],
    "Empty input": [],
}

for test_name, terms in test_inputs.items():
    result = predict_diseases(terms, top_k=5)

    print(f"\n=== {test_name} ===")

    if "error" in result:
        print("Handled safely:", result["error"])
    else:
        print("Recognized:", result["recognized_count"])
        print("Unrecognized:", result["unrecognized_count"])
        print("Candidate rows:", len(result["candidates"]))

print("\nInput validation checks completed.")



=== Recognized HPO terms ===
Recognized: 2
Unrecognized: 0
Candidate rows: 5

=== Unknown HPO term ===
Handled safely: No input HPO terms were recognized by the saved model.

=== Empty input ===
Handled safely: No input HPO terms were recognized by the saved model.

Input validation checks completed.


In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

hpo_input = widgets.Textarea(
    placeholder="Example: HP:0001250; HP:0001263; HP:0004322",
    description="HPO IDs:",
    layout=widgets.Layout(width="100%", height="110px"),
    style={"description_width": "initial"}
)

top_k_input = widgets.Dropdown(
    options=[5, 10, 15, 20],
    value=10,
    description="Show candidates:",
    style={"description_width": "initial"}
)

predict_button = widgets.Button(
    description="Rank Disease Candidates",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="250px", height="42px")
)

output = widgets.Output()

def run_prediction(_):
    with output:
        clear_output(wait=True)

        terms = [
            t.strip()
            for t in hpo_input.value.replace(",", ";").split(";")
            if t.strip()
        ]

        if not terms:
            display(HTML(
                "<p style='color:#b45309'><b>Please enter at least one HPO ID.</b></p>"
            ))
            return

        result = predict_diseases(terms, top_k=top_k_input.value)

        if "error" in result:
            display(HTML(
                "<p style='color:#b91c1c'><b>No usable phenotype terms found.</b>"
                " Check the HPO IDs and try again.</p>"
            ))
            if result.get("unrecognized_terms"):
                print("Unrecognized IDs:", ", ".join(result["unrecognized_terms"]))
            return

        display(HTML(
            "<h4>Input Summary</h4>"
            f"<p>Total terms: <b>{result['input_term_count']}</b> &nbsp; "
            f"Recognized: <b>{result['recognized_count']}</b> &nbsp; "
            f"Unrecognized: <b>{result['unrecognized_count']}</b></p>"
        ))

        if result["unrecognized_terms"]:
            print("Unrecognized HPO IDs:", ", ".join(result["unrecognized_terms"]))

        candidates = result["candidates"].copy()
        candidates["model_score"] = candidates["model_score"].map(
            lambda x: f"{x:.3f}"
        )
        candidates = candidates.rename(columns={
            "rank": "Rank",
            "disease_id": "Disease ID",
            "disease_name": "Candidate disease",
            "model_score": "Ranking score (not probability)"
        })

        display(HTML("<h4>Ranked Candidate Diseases</h4>"))
        display(candidates.reset_index(drop=True))

        display(HTML(
            "<div style='padding:12px;border:1px solid #d97706;border-radius:8px;"
            "margin-top:12px'>"
            "<b>Research prototype — not for clinical diagnosis.</b><br>"
            "Rankings are model-generated candidate suggestions. Scores are not "
            "probabilities, and a low rank does not rule out a disease. "
            "Clinical assessment and qualified medical genetics expertise are required."
            "</div>"
        ))

predict_button.on_click(run_prediction)

display(widgets.VBox([
    widgets.HTML(
        "<h2>Rare Disease Prediction AI</h2>"
        "<p>Phenotype-based ranking of candidate rare diseases using Human "
        "Phenotype Ontology (HPO) identifiers.</p>"
    ),
    hpo_input,
    top_k_input,
    predict_button,
    output
], layout=widgets.Layout(width="100%")))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:

from pathlib import Path
import json
import torch
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

checkpoint_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"

required_files = [
    DATA / "patient_level_validation.csv",
    DATA / "patient_publication_split.csv",
    DATA / "phenopacket_patient_cases.csv",
    DATA / "clean_hpo_annotations.csv",
    DATA / "patient_phenotype_only_baseline.csv",
    DATA / "patient_ranker_reconstructed_test_case_results.csv",
    DATA / "patient_ranker_reconstructed_test_metrics.json",
]

print("=== PROJECT AUDIT ===")
print("Project exists:", PROJECT.exists())

for path in required_files:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

print("\n=== CHECKPOINT ===")
print("Checkpoint exists:", checkpoint_path.exists())

if checkpoint_path.exists():
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )
    print("Disease vocabulary:", len(checkpoint["disease_ids"]))
    print("HPO vocabulary:", len(checkpoint["phenotype_ids"]))
    print("Hidden dimension:", checkpoint["hidden_dim"])
    print("Best epoch:", checkpoint.get("best_epoch"))

print("\n=== SPLIT SUMMARY ===")
split_path = DATA / "patient_publication_split.csv"

if split_path.exists():
    split = pd.read_csv(split_path)
    print("Rows:", len(split))
    print("Columns:", split.columns.tolist())
    if "split" in split.columns:
        print(split["split"].value_counts(dropna=False).to_string())

print("\nRead-only audit complete. No training or files modified.")


=== PROJECT AUDIT ===
Project exists: True
patient_level_validation.csv: FOUND
patient_publication_split.csv: FOUND
phenopacket_patient_cases.csv: FOUND
clean_hpo_annotations.csv: FOUND
patient_phenotype_only_baseline.csv: FOUND
patient_ranker_reconstructed_test_case_results.csv: FOUND
patient_ranker_reconstructed_test_metrics.json: FOUND

=== CHECKPOINT ===
Checkpoint exists: True
Disease vocabulary: 7340
HPO vocabulary: 9982
Hidden dimension: 128
Best epoch: 15

=== SPLIT SUMMARY ===
Rows: 10345
Columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
split
development    8386
test           1959

Read-only audit complete. No training or files modified.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Load development validation cases and disease-phenotype annotations.
val = pd.read_csv(DATA / "patient_level_validation.csv")
ann = pd.read_csv(DATA / "clean_hpo_annotations.csv", low_memory=False)

def split_ids(value):
    if pd.isna(value):
        return []
    return [
        x.strip() for x in str(value).replace(",", ";").split(";")
        if x.strip()
    ]

# Build disease-HPO profiles from the existing annotation resource.
ann = ann[["database_id", "hpo_id"]].dropna().copy()
ann["database_id"] = ann["database_id"].astype(str).str.strip()
ann["hpo_id"] = ann["hpo_id"].astype(str).str.strip()
ann = ann[
    ann["database_id"].ne("") &
    ann["hpo_id"].str.startswith("HP:")
].drop_duplicates()

diseases = sorted(ann["database_id"].unique())
hpos = sorted(ann["hpo_id"].unique())

disease_idx = {d: i for i, d in enumerate(diseases)}
hpo_idx = {h: i for i, h in enumerate(hpos)}

row = ann["database_id"].map(disease_idx).to_numpy()
col = ann["hpo_id"].map(hpo_idx).to_numpy()
profiles = csr_matrix(
    (np.ones(len(ann), dtype=np.float32), (row, col)),
    shape=(len(diseases), len(hpos))
)
profiles.data[:] = 1.0

# Information content: rarer annotation terms receive greater weight.
term_frequency = np.asarray((profiles > 0).sum(axis=0)).ravel()
n_diseases = len(diseases)
ic = np.log((n_diseases + 1) / (term_frequency + 1))

# Prepare validation cases.
val["case_id"] = val["case_id"].astype(str)
rank_rows = []

for _, case in val.iterrows():
    patient_hpos = [
        h for h in split_ids(case.get("hpo_ids", ""))
        if h in hpo_idx
    ]
    targets = set(split_ids(case.get("diagnosis_ids", "")))
    patient_hpos = sorted(set(patient_hpos))

    if not patient_hpos or not targets:
        continue

    pcols = np.array([hpo_idx[h] for h in patient_hpos], dtype=int)
    weights = ic[pcols]

    # Weighted overlap normalized by total information content in the patient.
    overlap = np.asarray(profiles[:, pcols] @ weights).ravel()
    denom = max(float(weights.sum()), 1e-12)
    scores = overlap / denom

    order = np.argsort(-scores, kind="stable")
    disease_ranks = np.empty(len(diseases), dtype=int)
    disease_ranks[order] = np.arange(1, len(diseases) + 1)

    available_targets = [t for t in targets if t in disease_idx]
    if not available_targets:
        continue

    # For multi-diagnosis cases, use the best-ranked covered target.
    target_rank = min(disease_ranks[disease_idx[t]] for t in available_targets)

    rank_rows.append({
        "case_id": case["case_id"],
        "rank": int(target_rank),
        "target_covered": True,
        "n_recognized_hpo": len(patient_hpos)
    })

results = pd.DataFrame(rank_rows)

if results.empty:
    raise RuntimeError(
        "No validation cases could be evaluated. "
        "Check the hpo_ids and diagnosis_ids formats."
    )

ranks = results["rank"].to_numpy(dtype=float)

print("=== VALIDATION IC-WEIGHTED OVERLAP BASELINE ===")
print("Evaluated cases:", len(results))
print("Validation cases available:", len(val))
print("Disease candidates:", len(diseases))
print(f"Hit@1:  {np.mean(ranks <= 1):.4f}")
print(f"Hit@5:  {np.mean(ranks <= 5):.4f}")
print(f"Hit@10: {np.mean(ranks <= 10):.4f}")
print(f"MRR:    {np.mean(1.0 / ranks):.4f}")
print(f"Median target rank: {np.median(ranks):.1f}")

print("\nNo model trained. No checkpoint or files modified.")
print("This is a validation-only exploratory baseline.")


=== VALIDATION IC-WEIGHTED OVERLAP BASELINE ===
Evaluated cases: 906
Validation cases available: 906
Disease candidates: 12879
Hit@1:  0.7638
Hit@5:  0.8587
Hit@10: 0.8874
MRR:    0.8070
Median target rank: 1.0

No model trained. No checkpoint or files modified.
This is a validation-only exploratory baseline.


In [ ]:

import numpy as np
import pandas as pd

DATA = "/content/drive/MyDrive/Rare Disease ai/processed"

val = pd.read_csv(f"{DATA}/patient_level_validation.csv")
ann = pd.read_csv(
    f"{DATA}/clean_hpo_annotations.csv",
    usecols=["database_id", "hpo_id"],
    low_memory=False
)

def ids(value):
    if pd.isna(value):
        return set()
    return {
        x.strip()
        for x in str(value).replace(",", ";").split(";")
        if x.strip()
    }

ann = ann.dropna().copy()
ann["database_id"] = ann["database_id"].astype(str).str.strip()
ann["hpo_id"] = ann["hpo_id"].astype(str).str.strip()

# Build profiles for the target diseases.
profiles = (
    ann[ann["hpo_id"].str.startswith("HP:")]
    .groupby("database_id")["hpo_id"]
    .agg(lambda s: set(s))
    .to_dict()
)

rows = []
for _, case in val.iterrows():
    patient_hpos = ids(case.get("hpo_ids", ""))
    targets = ids(case.get("diagnosis_ids", ""))

    covered_targets = [t for t in targets if t in profiles]
    if not patient_hpos or not covered_targets:
        continue

    best_overlap = max(
        len(patient_hpos & profiles[t]) / len(patient_hpos)
        for t in covered_targets
    )
    rows.append({
        "case_id": case["case_id"],
        "patient_hpo_count": len(patient_hpos),
        "best_target_overlap_fraction": best_overlap
    })

audit = pd.DataFrame(rows)

print("Validation cases audited:", len(audit))
print("Mean target-profile overlap:",
      round(audit["best_target_overlap_fraction"].mean(), 4))
print("Median target-profile overlap:",
      round(audit["best_target_overlap_fraction"].median(), 4))
print("Cases with >=50% patient HPO overlap:",
      int((audit["best_target_overlap_fraction"] >= 0.5).sum()))
print("Cases with >=80% patient HPO overlap:",
      int((audit["best_target_overlap_fraction"] >= 0.8).sum()))

print("\nInterpretation: overlap alone does not prove leakage.")
print("We must also establish whether target annotations were derived")
print("from the same patient reports or held-out publications.")


Validation cases audited: 906
Mean target-profile overlap: 0.9195
Median target-profile overlap: 1.0
Cases with >=50% patient HPO overlap: 846
Cases with >=80% patient HPO overlap: 797

Interpretation: overlap alone does not prove leakage.
We must also establish whether target annotations were derived
from the same patient reports or held-out publications.


In [ ]:

from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    usecols=[
        "database_id", "hpo_id", "reference",
        "evidence", "biocuration"
    ],
    low_memory=False
)

val = pd.read_csv(DATA / "patient_level_validation.csv")

val_pmids = set(
    val["pmid"].dropna().astype(str).str.extract(r"(\d+)")[0].dropna()
)

def extract_pmids(value):
    if pd.isna(value):
        return set()
    import re
    return set(re.findall(r"(?<!\d)\d{6,9}(?!\d)", str(value)))

ann["reference_pmids"] = ann["reference"].apply(extract_pmids)
ann["validation_pmid_overlap"] = ann["reference_pmids"].apply(
    lambda refs: bool(refs & val_pmids)
)

print("=== ANNOTATION PROVENANCE AUDIT ===")
print("Annotation rows:", len(ann))
print("Rows with a reference PMID:", int(ann["reference_pmids"].map(bool).sum()))
print("Rows referencing any validation PMID:", int(ann["validation_pmid_overlap"].sum()))
print("Distinct validation PMIDs:", len(val_pmids))

print("\nMost common evidence values:")
print(ann["evidence"].fillna("MISSING").value_counts().head(10).to_string())

print("\nMost common biocuration values:")
print(ann["biocuration"].fillna("MISSING").value_counts().head(10).to_string())

print("\nImportant: no overlap does not prove independence if the annotation")
print("resource was derived from the same reports indirectly.")
print("Read-only audit; no files modified.")


=== ANNOTATION PROVENANCE AUDIT ===
Annotation rows: 286047
Rows with a reference PMID: 206924
Rows referencing any validation PMID: 3249
Distinct validation PMIDs: 166

Most common evidence values:
evidence
TAS    136955
PCS    118797
IEA     30295

Most common biocuration values:
biocuration
ORPHA:orphadata[2026-09-02]              116612
HPO:iea[2009-02-17]                       21809
HPO:skoehler[2012-10-17]                   2999
HPO:skoehler[2017-07-13]                   2881
HPO:probinson[2009-02-17]                  2687
HPO:skoehler[2018-10-08]                   2260
ORCID:0000-0002-0736-9199[2024-04-01]      1823
HPO:skoehler[2010-06-20]                   1346
HPO:skoehler[2010-06-19]                   1072
HPO:skoehler[2019-04-18]                    955

Important: no overlap does not prove independence if the annotation
resource was derived from the same reports indirectly.
Read-only audit; no files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import re

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

val = pd.read_csv(DATA / "patient_level_validation.csv")
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    usecols=["database_id", "reference"],
    low_memory=False
)

def parse_ids(value):
    if pd.isna(value):
        return set()
    return {
        x.strip() for x in re.split(r"[;,|]", str(value))
        if x.strip()
    }

def pmids(value):
    if pd.isna(value):
        return set()
    return set(re.findall(r"(?<!\d)\d{6,9}(?!\d)", str(value)))

ann = ann.dropna(subset=["database_id"]).copy()
ann["database_id"] = ann["database_id"].astype(str).str.strip()
ann["ref_pmids"] = ann["reference"].apply(pmids)

# References associated with each disease profile
profile_refs = (
    ann.groupby("database_id")["ref_pmids"]
       .agg(lambda series: set().union(*series))
       .to_dict()
)

val["case_pmid"] = val["pmid"].apply(
    lambda x: next(iter(pmids(x)), None)
)

rows = []
for _, case in val.iterrows():
    targets = parse_ids(case["diagnosis_ids"])
    case_pmid = case["case_pmid"]

    target_refs = set().union(
        *(profile_refs.get(t, set()) for t in targets)
    ) if targets else set()

    rows.append({
        "case_id": case["case_id"],
        "has_target_profile": bool(targets & profile_refs.keys()),
        "own_pmid_in_target_refs": (
            case_pmid in target_refs if case_pmid else False
        ),
        "any_validation_pmid_in_target_refs": bool(
            target_refs & set(val["case_pmid"].dropna())
        ),
        "n_target_reference_pmids": len(target_refs)
    })

audit = pd.DataFrame(rows)

print("=== TARGET-SPECIFIC REFERENCE AUDIT ===")
print("Validation cases:", len(audit))
print("Cases with a target profile:", int(audit["has_target_profile"].sum()))
print("Cases whose own PMID appears in target references:",
      int(audit["own_pmid_in_target_refs"].sum()))
print("Cases whose target references include any validation PMID:",
      int(audit["any_validation_pmid_in_target_refs"].sum()))
print("Median target-profile reference PMIDs:",
      audit["n_target_reference_pmids"].median())
print("Cases with no target reference PMIDs:",
      int((audit["n_target_reference_pmids"] == 0).sum()))
print("\nRead-only audit; no files modified.")


=== TARGET-SPECIFIC REFERENCE AUDIT ===
Validation cases: 906
Cases with a target profile: 906
Cases whose own PMID appears in target references: 776
Cases whose target references include any validation PMID: 781
Median target-profile reference PMIDs: 2.0
Cases with no target reference PMIDs: 0

Read-only audit; no files modified.


In [ ]:

from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = [
    "patient_ranker_reconstructed_test_metrics.json",
    "patient_ranker_reconstructed_test_case_results.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_baseline_vs_gnn_paired.csv",
]

print("=== EXISTING TEST ARTIFACTS ===")
for name in files:
    path = DATA / name
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")

metrics_path = DATA / "patient_ranker_reconstructed_test_metrics.json"
if metrics_path.exists():
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
    print("\n=== RECONSTRUCTED RANKER METRICS ===")
    print(json.dumps(metrics, indent=2))

results_path = DATA / "patient_ranker_reconstructed_test_case_results.csv"
if results_path.exists():
    results = pd.read_csv(results_path)
    print("\n=== TEST RESULT TABLE ===")
    print("Rows:", len(results))
    print("Columns:", results.columns.tolist())
    print("Unique cases:", results["case_id"].nunique()
          if "case_id" in results.columns else "case_id column missing")

print("\nRead-only audit complete. No model trained or files modified.")


=== EXISTING TEST ARTIFACTS ===
patient_ranker_reconstructed_test_metrics.json: FOUND
patient_ranker_reconstructed_test_case_results.csv: FOUND
patient_phenotype_only_baseline.csv: FOUND
patient_baseline_vs_gnn_paired.csv: FOUND

=== RECONSTRUCTED RANKER METRICS ===
{
  "evaluation": "Held-out test ranking comparison",
  "test_cases": 1959,
  "ranker": {
    "cases": 1959,
    "target_covered": 1492,
    "target_coverage": 0.7616130678917815,
    "top1": 0.15773353751914243,
    "top5": 0.22664624808575803,
    "top10": 0.25267993874425726,
    "mrr": 0.19112964563013418,
    "median_rank_covered": 150.0
  },
  "phenotype_baseline": {
    "cases": 1959,
    "target_covered": 1551,
    "target_coverage": 0.7917304747320061,
    "top1": 0.1138335885655947,
    "top5": 0.19601837672281777,
    "top10": 0.22103113833588567,
    "mrr": 0.15266584058813337,
    "median_rank_covered": 258.0
  },
  "common_coverage_cases": 1181,
  "common_coverage_ranker": {
    "cases": 1181,
    "target_cove

In [ ]:

from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

files = [
    "patient_ranker_reconstructed_test_metrics.json",
    "patient_ranker_reconstructed_test_case_results.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_baseline_vs_gnn_paired.csv",
]

print("=== EXISTING TEST ARTIFACTS ===")
for name in files:
    path = DATA / name
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")

metrics_path = DATA / "patient_ranker_reconstructed_test_metrics.json"
if metrics_path.exists():
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
    print("\n=== RECONSTRUCTED RANKER METRICS ===")
    print(json.dumps(metrics, indent=2))

results_path = DATA / "patient_ranker_reconstructed_test_case_results.csv"
if results_path.exists():
    results = pd.read_csv(results_path)
    print("\n=== TEST RESULT TABLE ===")
    print("Rows:", len(results))
    print("Columns:", results.columns.tolist())
    print(
        "Unique cases:",
        results["case_id"].nunique()
        if "case_id" in results.columns
        else "case_id column missing"
    )

print("\nRead-only audit complete. No model trained or files modified.")


=== EXISTING TEST ARTIFACTS ===
patient_ranker_reconstructed_test_metrics.json: FOUND
patient_ranker_reconstructed_test_case_results.csv: FOUND
patient_phenotype_only_baseline.csv: FOUND
patient_baseline_vs_gnn_paired.csv: FOUND

=== RECONSTRUCTED RANKER METRICS ===
{
  "evaluation": "Held-out test ranking comparison",
  "test_cases": 1959,
  "ranker": {
    "cases": 1959,
    "target_covered": 1492,
    "target_coverage": 0.7616130678917815,
    "top1": 0.15773353751914243,
    "top5": 0.22664624808575803,
    "top10": 0.25267993874425726,
    "mrr": 0.19112964563013418,
    "median_rank_covered": 150.0
  },
  "phenotype_baseline": {
    "cases": 1959,
    "target_covered": 1551,
    "target_coverage": 0.7917304747320061,
    "top1": 0.1138335885655947,
    "top5": 0.19601837672281777,
    "top10": 0.22103113833588567,
    "mrr": 0.15266584058813337,
    "median_rank_covered": 258.0
  },
  "common_coverage_cases": 1181,
  "common_coverage_ranker": {
    "cases": 1181,
    "target_cove

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

path = DATA / "patient_ranker_reconstructed_test_case_results.csv"
df = pd.read_csv(path)

print("=== STEP 2: TEST RESULT QUALITY AUDIT ===")
print("Total rows:", len(df))
print("Unique cases:", df["case_id"].nunique())
print("Duplicate case IDs:", int(df["case_id"].duplicated().sum()))

for col in ["ranker_target_covered", "baseline_target_covered",
            "ranker_top1", "baseline_top1",
            "ranker_top5", "baseline_top5",
            "ranker_top10", "baseline_top10"]:
    if col in df.columns:
        print(f"Missing {col}: {df[col].isna().sum()}")

print("\n=== TARGET COVERAGE ===")
for col in ["ranker_target_covered", "baseline_target_covered"]:
    if col in df.columns:
        print(f"{col}: {df[col].sum()} / {len(df)} "
              f"({df[col].mean():.2%})")

print("\n=== TOP-K PERFORMANCE ===")
for model in ["ranker", "baseline"]:
    for k in [1, 5, 10]:
        col = f"{model}_top{k}"
        if col in df.columns:
            print(f"{model} Top-{k}: {df[col].mean():.4f}")

for col in ["ranker_reciprocal_rank", "baseline_reciprocal_rank"]:
    if col in df.columns:
        print(f"{col}: {df[col].mean():.4f}")

if "both_cover_target" in df.columns:
    common = df[df["both_cover_target"].astype(bool)].copy()
    print("\n=== MATCHED TARGET-COVERAGE CASES ===")
    print("Cases:", len(common))
    for model in ["ranker", "baseline"]:
        for k in [1, 5, 10]:
            col = f"{model}_top{k}"
            if col in common.columns:
                print(f"{model} Top-{k}: {common[col].mean():.4f}")
        rr = f"{model}_reciprocal_rank"
        if rr in common.columns:
            print(f"{model} MRR: {common[rr].mean():.4f}")

print("\nRead-only audit. No training or files modified.")


=== STEP 2: TEST RESULT QUALITY AUDIT ===
Total rows: 1959
Unique cases: 1959
Duplicate case IDs: 0
Missing ranker_target_covered: 0
Missing baseline_target_covered: 0
Missing ranker_top1: 0
Missing baseline_top1: 0
Missing ranker_top5: 0
Missing baseline_top5: 0
Missing ranker_top10: 0
Missing baseline_top10: 0

=== TARGET COVERAGE ===
ranker_target_covered: 1492 / 1959 (76.16%)
baseline_target_covered: 1551 / 1959 (79.17%)

=== TOP-K PERFORMANCE ===
ranker Top-1: 0.1577
ranker Top-5: 0.2266
ranker Top-10: 0.2527
baseline Top-1: 0.1138
baseline Top-5: 0.1960
baseline Top-10: 0.2210
ranker_reciprocal_rank: 0.1911
baseline_reciprocal_rank: 0.1527

=== MATCHED TARGET-COVERAGE CASES ===
Cases: 1181
ranker Top-1: 0.2591
ranker Top-5: 0.3709
ranker Top-10: 0.4124
ranker MRR: 0.3128
baseline Top-1: 0.1160
baseline Top-5: 0.2235
baseline Top-10: 0.2566
baseline MRR: 0.1659

Read-only audit. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

df = pd.read_csv(DATA / "patient_ranker_reconstructed_test_case_results.csv")

print("=== STEP 3: PERFORMANCE BY PHENOTYPE COUNT ===")

if "n_observed_hpo" not in df.columns:
    print("ERROR: n_observed_hpo column not found.")
else:
    bins = [0, 5, 10, 15, 20, float("inf")]
    labels = ["1-5", "6-10", "11-15", "16-20", "21+"]

    df["phenotype_group"] = pd.cut(
        df["n_observed_hpo"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    rows = []
    for group, part in df.groupby("phenotype_group", observed=False):
        if len(part) == 0:
            continue

        row = {
            "phenotype_count": str(group),
            "cases": len(part),
            "ranker_coverage": part["ranker_target_covered"].mean(),
            "baseline_coverage": part["baseline_target_covered"].mean(),
            "ranker_top1": part["ranker_top1"].mean(),
            "baseline_top1": part["baseline_top1"].mean(),
            "ranker_top5": part["ranker_top5"].mean(),
            "baseline_top5": part["baseline_top5"].mean(),
            "ranker_top10": part["ranker_top10"].mean(),
            "baseline_top10": part["baseline_top10"].mean(),
            "ranker_mrr": part["ranker_reciprocal_rank"].mean(),
            "baseline_mrr": part["baseline_reciprocal_rank"].mean(),
        }
        rows.append(row)

    summary = pd.DataFrame(rows)
    print(summary.round(4).to_string(index=False))

print("\nRead-only analysis. No training or files modified.")


=== STEP 3: PERFORMANCE BY PHENOTYPE COUNT ===
phenotype_count  cases  ranker_coverage  baseline_coverage  ranker_top1  baseline_top1  ranker_top5  baseline_top5  ranker_top10  baseline_top10  ranker_mrr  baseline_mrr
            1-5    564           0.7801             0.7855       0.1596         0.0674       0.2074         0.1578        0.2323          0.1862      0.1848        0.1101
           6-10    723           0.7497             0.8133       0.1438         0.1300       0.2185         0.2061        0.2337          0.2241      0.1786        0.1656
          11-15    359           0.6936             0.7883       0.1393         0.1337       0.2228         0.1950        0.2674          0.2201      0.1757        0.1657
          16-20    166           0.7831             0.7349       0.1386         0.0783       0.2048         0.1566        0.2349          0.2108      0.1734        0.1257
            21+    147           0.8912             0.7823       0.2857         0.2041       0.374

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

df = pd.read_csv(DATA / "patient_ranker_reconstructed_test_case_results.csv")

print("=== STEP 4: EVALUATION LOGIC AUDIT ===")
print("Rows:", len(df))

for model in ["ranker", "baseline"]:
    rank_col = f"{model}_rank"
    covered_col = f"{model}_target_covered"
    rr_col = f"{model}_reciprocal_rank"

    print(f"\n--- {model.upper()} ---")

    if not all(c in df.columns for c in [rank_col, covered_col, rr_col]):
        print("Required columns missing; skipping.")
        continue

    covered = df[covered_col].astype(bool)
    ranks = pd.to_numeric(df[rank_col], errors="coerce")
    rr = pd.to_numeric(df[rr_col], errors="coerce")

    print("Covered cases:", int(covered.sum()))
    print("Covered cases with missing rank:", int((covered & ranks.isna()).sum()))
    print("Uncovered cases with a rank:", int((~covered & ranks.notna()).sum()))
    print("Covered cases with rank < 1:",
          int((covered & ranks.notna() & (ranks < 1)).sum()))

    expected_rr = pd.Series(0.0, index=df.index)
    valid = covered & ranks.notna() & (ranks >= 1)
    expected_rr.loc[valid] = 1.0 / ranks.loc[valid]

    print("Reciprocal-rank mismatches:",
          int((~np.isclose(rr.fillna(-1), expected_rr, atol=1e-6)).sum()))

    for k in [1, 5, 10]:
        col = f"{model}_top{k}"
        if col in df.columns:
            expected_hit = (valid & (ranks <= k)).astype(int)
            actual_hit = pd.to_numeric(df[col], errors="coerce")
            mismatch = ~np.isclose(
                actual_hit.fillna(-1), expected_hit, atol=1e-6
            )
            print(f"Top-{k} mismatches:", int(mismatch.sum()))

print("\nAudit complete. Read-only; no files modified.")


=== STEP 4: EVALUATION LOGIC AUDIT ===
Rows: 1959

--- RANKER ---
Covered cases: 1492
Covered cases with missing rank: 0
Uncovered cases with a rank: 0
Covered cases with rank < 1: 0
Reciprocal-rank mismatches: 0
Top-1 mismatches: 0
Top-5 mismatches: 0
Top-10 mismatches: 0

--- BASELINE ---
Covered cases: 1551
Covered cases with missing rank: 0
Uncovered cases with a rank: 0
Covered cases with rank < 1: 0
Reciprocal-rank mismatches: 0
Top-1 mismatches: 0
Top-5 mismatches: 0
Top-10 mismatches: 0

Audit complete. Read-only; no files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

df = pd.read_csv(DATA / "patient_ranker_reconstructed_test_case_results.csv")

print("=== STEP 5: CHECKPOINT AND VOCABULARY AUDIT ===")

checkpoint_path = PROJECT / "model" / "patient_ranker_bias_softmax_experiment.pt"

print("Checkpoint exists:", checkpoint_path.exists())

if checkpoint_path.exists():
    import torch

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    disease_ids = checkpoint.get("disease_ids", [])
    phenotype_ids = checkpoint.get("phenotype_ids", [])

    print("Disease vocabulary size:", len(disease_ids))
    print("Phenotype vocabulary size:", len(phenotype_ids))
    print("Hidden dimension:", checkpoint.get("hidden_dim"))
    print("Best epoch:", checkpoint.get("best_epoch"))
    print("Saved validation metrics:", checkpoint.get("best_validation_metrics"))

    print("\nCheckpoint state keys:")
    state = checkpoint.get("model_state_dict", {})
    print(list(state.keys()))

print("\n=== TEST CASE COVERAGE COUNTS ===")
print("Total test cases:", len(df))
print("Ranker target covered:", int(df["ranker_target_covered"].sum()))
print("Ranker target not covered:", int((~df["ranker_target_covered"].astype(bool)).sum()))
print("Baseline target covered:", int(df["baseline_target_covered"].sum()))
print("Baseline target not covered:", int((~df["baseline_target_covered"].astype(bool)).sum()))

print("\n=== SAVED CHECKPOINT AUDIT COMPLETE ===")
print("Read-only. No training or files modified.")


=== STEP 5: CHECKPOINT AND VOCABULARY AUDIT ===
Checkpoint exists: True
Disease vocabulary size: 7340
Phenotype vocabulary size: 9982
Hidden dimension: 128
Best epoch: 15
Saved validation metrics: {'cases': 906, 'mrr': 0.3341009854885823, 'top1': 0.30132450331125826, 'top5': 0.3631346578366446, 'top10': 0.39514348785871967}

Checkpoint state keys:
['disease_bias', 'hpo_embedding.weight', 'disease_embedding.weight', 'patient_projection.0.weight', 'patient_projection.0.bias', 'patient_projection.3.weight', 'patient_projection.3.bias']

=== TEST CASE COVERAGE COUNTS ===
Total test cases: 1959
Ranker target covered: 1492
Ranker target not covered: 467
Baseline target covered: 1551
Baseline target not covered: 408

=== SAVED CHECKPOINT AUDIT COMPLETE ===
Read-only. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

df = pd.read_csv(DATA / "patient_ranker_reconstructed_test_case_results.csv")

print("=== STEP 6: CHECKPOINT VOCABULARY COVERAGE ===")

checkpoint_path = PROJECT / "model" / "patient_ranker_bias_softmax_experiment.pt"

import torch
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

disease_ids = set(map(str, checkpoint["disease_ids"]))

def parse_diagnoses(value):
    if pd.isna(value):
        return set()
    import re
    return {
        x.strip()
        for x in re.split(r"[;,|]", str(value))
        if x.strip()
    }

split_path = DATA / "patient_publication_split.csv"
split_df = pd.read_csv(split_path)

test_cases = split_df[split_df["split"].astype(str).str.lower() == "test"].copy()

print("Checkpoint disease vocabulary:", len(disease_ids))
print("Test split rows:", len(test_cases))
print("Result table rows:", len(df))

if "diagnosis_ids" in test_cases.columns:
    test_cases["target_ids"] = test_cases["diagnosis_ids"].apply(parse_diagnoses)

    target_sets = test_cases["target_ids"].tolist()
    all_targets = set().union(*target_sets) if target_sets else set()
    covered_targets = all_targets & disease_ids
    missing_targets = all_targets - disease_ids

    print("Unique test target diseases:", len(all_targets))
    print("Unique target diseases in vocabulary:", len(covered_targets))
    print("Unique target diseases missing:", len(missing_targets))
    print("Target disease vocabulary coverage:",
          round(len(covered_targets) / len(all_targets), 4)
          if all_targets else "N/A")
else:
    print("diagnosis_ids column not found in publication split.")

print("\nRead-only audit. No training or files modified.")


=== STEP 6: CHECKPOINT VOCABULARY COVERAGE ===
Checkpoint disease vocabulary: 7340
Test split rows: 1959
Result table rows: 1959
Unique test target diseases: 258
Unique target diseases in vocabulary: 195
Unique target diseases missing: 63
Target disease vocabulary coverage: 0.7558

Read-only audit. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

df = pd.read_csv(DATA / "patient_ranker_reconstructed_test_case_results.csv")

print("=== STEP 7: PAIRED PERFORMANCE AND COVERAGE AUDIT ===")

both = df["both_cover_target"].astype(bool)
ranker_rr = pd.to_numeric(df["ranker_reciprocal_rank"], errors="coerce").fillna(0)
baseline_rr = pd.to_numeric(df["baseline_reciprocal_rank"], errors="coerce").fillna(0)

print("Cases where both models cover the target:", int(both.sum()))
print("Cases where only ranker covers target:",
      int((df["ranker_target_covered"].astype(bool) &
           ~df["baseline_target_covered"].astype(bool)).sum()))
print("Cases where only baseline covers target:",
      int((df["baseline_target_covered"].astype(bool) &
           ~df["ranker_target_covered"].astype(bool)).sum()))
print("Cases where neither covers target:",
      int((~df["ranker_target_covered"].astype(bool) &
           ~df["baseline_target_covered"].astype(bool)).sum()))

paired = pd.DataFrame({
    "ranker_rr": ranker_rr[both],
    "baseline_rr": baseline_rr[both]
})

delta = paired["ranker_rr"] - paired["baseline_rr"]

print("\n=== COMMON-COVERAGE RECIPROCAL-RANK COMPARISON ===")
print("Cases:", len(paired))
print("Ranker wins:", int((delta > 1e-12).sum()))
print("Ties:", int((delta.abs() <= 1e-12).sum()))
print("Baseline wins:", int((delta < -1e-12).sum()))
print("Mean ranker MRR:", round(paired["ranker_rr"].mean(), 4))
print("Mean baseline MRR:", round(paired["baseline_rr"].mean(), 4))
print("Mean paired MRR difference:", round(delta.mean(), 4))

print("\nRead-only audit. No training or files modified.")


=== STEP 7: PAIRED PERFORMANCE AND COVERAGE AUDIT ===
Cases where both models cover the target: 1181
Cases where only ranker covers target: 311
Cases where only baseline covers target: 370
Cases where neither covers target: 97

=== COMMON-COVERAGE RECIPROCAL-RANK COMPARISON ===
Cases: 1181
Ranker wins: 675
Ties: 77
Baseline wins: 429
Mean ranker MRR: 0.3128
Mean baseline MRR: 0.1659
Mean paired MRR difference: 0.1469

Read-only audit. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

df = pd.read_csv(DATA / "patient_ranker_reconstructed_test_case_results.csv")

print("=== STEP 8: FINAL BASELINE AUDIT SUMMARY ===")

summary = {
    "test_cases": int(len(df)),
    "unique_cases": int(df["case_id"].nunique()),
    "ranker_target_coverage": round(float(df["ranker_target_covered"].mean()), 4),
    "baseline_target_coverage": round(float(df["baseline_target_covered"].mean()), 4),
    "ranker_top1": round(float(df["ranker_top1"].mean()), 4),
    "baseline_top1": round(float(df["baseline_top1"].mean()), 4),
    "ranker_top5": round(float(df["ranker_top5"].mean()), 4),
    "baseline_top5": round(float(df["baseline_top5"].mean()), 4),
    "ranker_top10": round(float(df["ranker_top10"].mean()), 4),
    "baseline_top10": round(float(df["baseline_top10"].mean()), 4),
    "ranker_mrr": round(float(df["ranker_reciprocal_rank"].mean()), 4),
    "baseline_mrr": round(float(df["baseline_reciprocal_rank"].mean()), 4),
    "both_cover_cases": int(df["both_cover_target"].astype(bool).sum()),
}

both = df["both_cover_target"].astype(bool)
ranker_rr = df.loc[both, "ranker_reciprocal_rank"].fillna(0)
baseline_rr = df.loc[both, "baseline_reciprocal_rank"].fillna(0)

summary["common_coverage_cases"] = int(both.sum())
summary["common_coverage_ranker_mrr"] = round(float(ranker_rr.mean()), 4)
summary["common_coverage_baseline_mrr"] = round(float(baseline_rr.mean()), 4)
summary["common_coverage_mrr_delta"] = round(
    float((ranker_rr - baseline_rr).mean()), 4
)

print(json.dumps(summary, indent=2))

print("\n=== AUDIT CONCLUSION ===")
print("1. Evaluation columns are internally consistent.")
print("2. Ranker has lower overall target coverage than baseline.")
print("3. Ranker has higher MRR on cases where both cover the target.")
print("4. Checkpoint inference remains reconstructed, not source-verified.")
print("5. IC baseline validation score is not trusted pending leakage audit.")
print("\nRead-only. No training or files modified.")


=== STEP 8: FINAL BASELINE AUDIT SUMMARY ===
{
  "test_cases": 1959,
  "unique_cases": 1959,
  "ranker_target_coverage": 0.7616,
  "baseline_target_coverage": 0.7917,
  "ranker_top1": 0.1577,
  "baseline_top1": 0.1138,
  "ranker_top5": 0.2266,
  "baseline_top5": 0.196,
  "ranker_top10": 0.2527,
  "baseline_top10": 0.221,
  "ranker_mrr": 0.1911,
  "baseline_mrr": 0.1527,
  "both_cover_cases": 1181,
  "common_coverage_cases": 1181,
  "common_coverage_ranker_mrr": 0.3128,
  "common_coverage_baseline_mrr": 0.1659,
  "common_coverage_mrr_delta": 0.1469
}

=== AUDIT CONCLUSION ===
1. Evaluation columns are internally consistent.
2. Ranker has lower overall target coverage than baseline.
3. Ranker has higher MRR on cases where both cover the target.
4. Checkpoint inference remains reconstructed, not source-verified.
5. IC baseline validation score is not trusted pending leakage audit.

Read-only. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import re

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 2: PUBLICATION OVERLAP AND SPLIT AUDIT ===")

split = pd.read_csv(DATA / "patient_publication_split.csv")
val = pd.read_csv(DATA / "patient_level_validation.csv")
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    usecols=["database_id", "reference"],
    low_memory=False
)

def get_pmids(value):
    if pd.isna(value):
        return set()
    return set(re.findall(r"(?<!\d)\d{6,9}(?!\d)", str(value)))

def get_ids(value):
    if pd.isna(value):
        return set()
    return {
        x.strip()
        for x in re.split(r"[;,|]", str(value))
        if x.strip()
    }

split["case_id"] = split["case_id"].astype(str)
val["case_id"] = val["case_id"].astype(str)

split["pmid_norm"] = split["pmid"].apply(
    lambda x: next(iter(get_pmids(x)), None)
)
val["pmid_norm"] = val["pmid"].apply(
    lambda x: next(iter(get_pmids(x)), None)
)

ann = ann.dropna(subset=["database_id"]).copy()
ann["database_id"] = ann["database_id"].astype(str).str.strip()
ann["ref_pmids"] = ann["reference"].apply(get_pmids)

profile_refs = (
    ann.groupby("database_id")["ref_pmids"]
    .agg(lambda series: set().union(*series))
    .to_dict()
)

all_validation_pmids = set(val["pmid_norm"].dropna())
test = split[split["split"].astype(str).str.lower() == "test"].copy()
development = split[
    split["split"].astype(str).str.lower() != "test"
].copy()

print("\n--- SPLIT INTEGRITY ---")
print("Split rows:", len(split))
print("Unique case IDs:", split["case_id"].nunique())
print("Duplicate case IDs:", int(split["case_id"].duplicated().sum()))
print("Development rows:", len(development))
print("Test rows:", len(test))
print("Validation rows:", len(val))
print("Validation cases absent from development:",
      int((~val["case_id"].isin(set(development["case_id"]))).sum()))

dev_pmids = set(development["pmid_norm"].dropna())
test_pmids = set(test["pmid_norm"].dropna())
print("Development/test shared PMIDs:", len(dev_pmids & test_pmids))

def audit_reference_overlap(frame, label):
    own_matches = 0
    any_validation_matches = 0
    target_profile_cases = 0
    cases_with_pmids = 0

    for _, row in frame.iterrows():
        targets = get_ids(row.get("diagnosis_ids", ""))
        own_pmid = row.get("pmid_norm")
        refs = set().union(
            *(profile_refs.get(target, set()) for target in targets)
        ) if targets else set()

        if targets and any(t in profile_refs for t in targets):
            target_profile_cases += 1
        if own_pmid:
            cases_with_pmids += 1
            if own_pmid in refs:
                own_matches += 1
        if refs & all_validation_pmids:
            any_validation_matches += 1

    print(f"\n--- {label} TARGET-REFERENCE OVERLAP ---")
    print("Cases with target profiles:", target_profile_cases, "/", len(frame))
    print("Cases with PMIDs:", cases_with_pmids)
    print("Own PMID in target disease references:", own_matches)
    print("Target references include any validation PMID:",
          any_validation_matches)

audit_reference_overlap(val, "VALIDATION")
audit_reference_overlap(test, "TEST")

print("\nAudit finished. Read-only; no files modified or models trained.")


=== STAGE 2: PUBLICATION OVERLAP AND SPLIT AUDIT ===

--- SPLIT INTEGRITY ---
Split rows: 10345
Unique case IDs: 10345
Duplicate case IDs: 0
Development rows: 8386
Test rows: 1959
Validation rows: 906
Validation cases absent from development: 0
Development/test shared PMIDs: 0

--- VALIDATION TARGET-REFERENCE OVERLAP ---
Cases with target profiles: 906 / 906
Cases with PMIDs: 906
Own PMID in target disease references: 776
Target references include any validation PMID: 781

--- TEST TARGET-REFERENCE OVERLAP ---
Cases with target profiles: 1959 / 1959
Cases with PMIDs: 1959
Own PMID in target disease references: 1509
Target references include any validation PMID: 220

Audit finished. Read-only; no files modified or models trained.


In [ ]:

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:

from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
checkpoint_path = PROJECT / "model" / "patient_ranker_bias_softmax_experiment.pt"

print("=== STAGE 3: ORIGINAL INFERENCE SOURCE AUDIT ===")
print("Checkpoint exists:", checkpoint_path.exists())

terms = [
    "patient_projection",
    "hpo_embedding",
    "disease_embedding",
    "disease_bias",
]

matches = []
for path in PROJECT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in {".py", ".ipynb", ".txt", ".md"}:
        continue
    if any(part.startswith(".") for part in path.parts):
        continue
    try:
        if path.stat().st_size > 5_000_000:
            continue
        content = path.read_text(encoding="utf-8", errors="ignore")
        found = [term for term in terms if term.lower() in content.lower()]
        if found:
            matches.append((path, found))
    except Exception:
        pass

print("\nProject files containing model terms:", len(matches))
for path, found in matches[:50]:
    print(f"- {path}: {', '.join(found)}")

if checkpoint_path.exists():
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )
    print("\n=== CHECKPOINT METADATA ===")
    for key in [
        "objective", "note", "seed", "best_epoch",
        "training_cases", "validation_cases"
    ]:
        print(f"{key}: {checkpoint.get(key, 'NOT SAVED')}")

print("\nRead-only audit complete. No files modified or models trained.")


=== STAGE 3: ORIGINAL INFERENCE SOURCE AUDIT ===
Checkpoint exists: True

Project files containing model terms: 0

=== CHECKPOINT METADATA ===
objective: single-label full-disease cross-entropy
note: Research experiment; not clinically validated.
seed: 42
best_epoch: 15
training_cases: 5730
validation_cases: 906

Read-only audit complete. No files modified or models trained.


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 4: TRAIN / VALIDATION / TEST CASE AUDIT ===")

split = pd.read_csv(DATA / "patient_publication_split.csv")
val = pd.read_csv(DATA / "patient_level_validation.csv")

split["case_id"] = split["case_id"].astype(str).str.strip()
val["case_id"] = val["case_id"].astype(str).str.strip()

train = split[split["split"].astype(str).str.lower() == "train"]
test = split[split["split"].astype(str).str.lower() == "test"]
development = split[split["split"].astype(str).str.lower() != "test"]

train_ids = set(train["case_id"])
test_ids = set(test["case_id"])
val_ids = set(val["case_id"])
development_ids = set(development["case_id"])

print("Training split cases:", len(train_ids))
print("Validation cases:", len(val_ids))
print("Test split cases:", len(test_ids))

print("\nOverlap checks:")
print("Train / validation shared cases:", len(train_ids & val_ids))
print("Train / test shared cases:", len(train_ids & test_ids))
print("Validation / test shared cases:", len(val_ids & test_ids))
print("Validation cases outside development:", len(val_ids - development_ids))

print("\nSaved checkpoint metadata reported 5,730 training cases.")
print("This audit checks split files, not the exact examples used during training.")
print("Read-only. No training or files modified.")


=== STAGE 4: TRAIN / VALIDATION / TEST CASE AUDIT ===
Training split cases: 0
Validation cases: 906
Test split cases: 1959

Overlap checks:
Train / validation shared cases: 0
Train / test shared cases: 0
Validation / test shared cases: 0
Validation cases outside development: 0

Saved checkpoint metadata reported 5,730 training cases.
This audit checks split files, not the exact examples used during training.
Read-only. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 4B: IDENTIFY ACTUAL SPLIT LABELS ===")

split = pd.read_csv(DATA / "patient_publication_split.csv")

print("\nColumns:")
print(split.columns.tolist())

print("\nSplit label counts:")
print(split["split"].astype(str).str.strip().value_counts(dropna=False).to_string())

print("\nSample rows:")
print(split.head(10).to_string(index=False))

print("\n=== CHECK VALIDATION CASES AGAINST SPLIT LABELS ===")
val = pd.read_csv(DATA / "patient_level_validation.csv")

split["case_id"] = split["case_id"].astype(str).str.strip()
val["case_id"] = val["case_id"].astype(str).str.strip()

labels_by_case = split.set_index("case_id")["split"].astype(str).str.strip()
print("Validation cases:", len(val))
print("Validation cases with split labels:")
print(val["case_id"].map(labels_by_case).value_counts(dropna=False).to_string())

print("\nRead-only audit. No training or files modified.")


=== STAGE 4B: IDENTIFY ACTUAL SPLIT LABELS ===

Columns:
['case_id', 'pmid', 'split', 'diagnosis_ids']

Split label counts:
split
development    8386
test           1959

Sample rows:
                       case_id     pmid       split diagnosis_ids
   PMID_19836009_Family_E_II_6 19836009 development   OMIM:277600
   PMID_19836009_Family_E_II_3 19836009 development   OMIM:277600
   PMID_19836009_Family_B_II_4 19836009 development   OMIM:277600
   PMID_19836009_Family_B_II_2 19836009 development   OMIM:277600
         PMID_25469541_patient 25469541 development   OMIM:277600
     PMID_24573067_CASE_REPORT 24573067 development   OMIM:148600
PMID_28239884_Family_1_proband 28239884        test   OMIM:148600
PMID_28239884_Family_2_proband 28239884        test   OMIM:148600
PMID_28239884_Family_3_proband 28239884        test   OMIM:148600
PMID_28823388_3_month_old_girl 28823388 development   OMIM:102700

=== CHECK VALIDATION CASES AGAINST SPLIT LABELS ===
Validation cases: 906
Validation case

In [ ]:

from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=== STAGE 5: TRAINING DATA PROVENANCE AUDIT ===")

checkpoint_path = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

print("Checkpoint training_cases:", ckpt.get("training_cases"))
print("Checkpoint validation_cases:", ckpt.get("validation_cases"))

split = pd.read_csv(DATA / "patient_publication_split.csv")
split["case_id"] = split["case_id"].astype(str).str.strip()

print("\nSplit labels:")
print(split["split"].value_counts().to_string())

print("\nCheckpoint says training cases:", ckpt.get("training_cases"))
print("Development split cases:", int((split["split"] == "development").sum()))
print("Test split cases:", int((split["split"] == "test").sum()))

# Search existing processed files for training manifests or case-level records.
print("\n=== POSSIBLE TRAINING MANIFESTS ===")
for path in sorted(DATA.glob("*")):
    if path.is_file() and any(
        word in path.name.lower()
        for word in ["train", "history", "manifest", "ranker"]
    ):
        print(path.name)

print("\n=== CHECKPOINT METADATA KEYS ===")
print(sorted(ckpt.keys()))

print("\nRead-only audit. No training or files modified.")


=== STAGE 5: TRAINING DATA PROVENANCE AUDIT ===
Checkpoint training_cases: 5730
Checkpoint validation_cases: 906

Split labels:
split
development    8386
test           1959

Checkpoint says training cases: 5730
Development split cases: 8386
Test split cases: 1959

=== POSSIBLE TRAINING MANIFESTS ===
gnn_training_annotations.csv
heldout_profile_nearest_training_disease.csv
patient_gnn_training_disease_rankings.csv
patient_level_ranker_bias_free_experiment_history.csv
patient_level_ranker_bias_free_experiment_metrics.json
patient_level_train.csv
patient_level_training_history.csv
patient_ranker_bias_ablation.csv
patient_ranker_bias_ablation_summary.json
patient_ranker_bias_softmax_experiment_history.csv
patient_ranker_bias_softmax_experiment_metrics.json
patient_ranker_bias_softmax_experiment_validation.csv
patient_ranker_diagnostics.json
patient_ranker_reconstructed_test_case_results.csv
patient_ranker_reconstructed_test_metrics.json
patient_ranker_vs_baseline_paired_bootstrap.json
rar

In [ ]:

from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 6: VERIFY TRAINING MANIFEST ===")

path = DATA / "patient_level_train.csv"

if not path.exists():
    print("Training manifest not found:", path)
else:
    train = pd.read_csv(path)
    print("File:", path.name)
    print("Rows:", len(train))
    print("Columns:", train.columns.tolist())

    if "case_id" in train.columns:
        ids = train["case_id"].astype(str).str.strip()
        print("Unique case IDs:", ids.nunique())
        print("Duplicate case IDs:", int(ids.duplicated().sum()))

        split_path = DATA / "patient_publication_split.csv"
        split = pd.read_csv(split_path)
        split["case_id"] = split["case_id"].astype(str).str.strip()

        development_ids = set(
            split.loc[split["split"] == "development", "case_id"]
        )
        test_ids = set(
            split.loc[split["split"] == "test", "case_id"]
        )

        train_ids = set(ids)
        print("Manifest IDs in development split:", len(train_ids & development_ids))
        print("Manifest IDs overlapping test:", len(train_ids & test_ids))
        print("Manifest IDs absent from split:", len(train_ids - development_ids - test_ids))

    print("\nSample rows:")
    print(train.head(5).to_string(index=False))

metrics_path = DATA / "patient_ranker_bias_softmax_experiment_metrics.json"
if metrics_path.exists():
    print("\n=== SAVED TRAINING METRICS ===")
    with open(metrics_path, "r") as f:
        print(json.dumps(json.load(f), indent=2)[:6000])

print("\nRead-only audit. No training or files modified.")


=== STAGE 6: VERIFY TRAINING MANIFEST ===
File: patient_level_train.csv
Rows: 5730
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']
Unique case IDs: 5730
Duplicate case IDs: 0
Manifest IDs in development split: 5730
Manifest IDs overlapping test: 0
Manifest IDs absent from split: 0

Sample rows:
                    case_id     pmid diagnosis_ids                                                                                            hpo_ids  n_hpo  n_diagnoses
PMID_19836009_Family_E_II_6 19836009   OMIM:277600                                  HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003      6            1
PMID_19836009_Family_E_II_3 19836009   OMIM:277600                                  HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003      6            1
PMID_19836009_Family_B_II_4 19836009   OMIM:277600                                             HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0011003      5     

In [ ]:

from pathlib import Path
import pandas as pd
import re

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 7: PUBLICATION-LEVEL TRAINING LEAKAGE CHECK ===")

train = pd.read_csv(DATA / "patient_level_train.csv")
val = pd.read_csv(DATA / "patient_level_validation.csv")
split = pd.read_csv(DATA / "patient_publication_split.csv")

def normalize_pmid(value):
    if pd.isna(value):
        return None
    match = re.search(r"\d{6,9}", str(value))
    return match.group(0) if match else None

for frame in [train, val, split]:
    frame["pmid_norm"] = frame["pmid"].apply(normalize_pmid)

train_pmids = set(train["pmid_norm"].dropna())
val_pmids = set(val["pmid_norm"].dropna())
test_pmids = set(
    split.loc[split["split"] == "test", "pmid_norm"].dropna()
)

train_cases = set(train["case_id"].astype(str))
val_cases = set(val["case_id"].astype(str))
test_cases = set(
    split.loc[split["split"] == "test", "case_id"].astype(str)
)

print("Training cases:", len(train_cases))
print("Validation cases:", len(val_cases))
print("Test cases:", len(test_cases))

print("\n--- CASE-LEVEL OVERLAP ---")
print("Train / validation:", len(train_cases & val_cases))
print("Train / test:", len(train_cases & test_cases))
print("Validation / test:", len(val_cases & test_cases))

print("\n--- PUBLICATION-LEVEL OVERLAP ---")
print("Unique training PMIDs:", len(train_pmids))
print("Unique validation PMIDs:", len(val_pmids))
print("Unique test PMIDs:", len(test_pmids))
print("Train / validation shared PMIDs:", len(train_pmids & val_pmids))
print("Train / test shared PMIDs:", len(train_pmids & test_pmids))
print("Validation / test shared PMIDs:", len(val_pmids & test_pmids))

print("\nRead-only audit. No training or files modified.")


=== STAGE 7: PUBLICATION-LEVEL TRAINING LEAKAGE CHECK ===
Training cases: 5730
Validation cases: 906
Test cases: 1959

--- CASE-LEVEL OVERLAP ---
Train / validation: 0
Train / test: 0
Validation / test: 0

--- PUBLICATION-LEVEL OVERLAP ---
Unique training PMIDs: 944
Unique validation PMIDs: 166
Unique test PMIDs: 346
Train / validation shared PMIDs: 0
Train / test shared PMIDs: 0
Validation / test shared PMIDs: 0

Read-only audit. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import json
import hashlib

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=== STAGE 8: REPRODUCIBILITY AND ARTIFACT INVENTORY ===")

files = [
    DATA / "patient_level_train.csv",
    DATA / "patient_level_validation.csv",
    DATA / "patient_publication_split.csv",
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv",
    DATA / "patient_ranker_bias_softmax_experiment_history.csv",
    DATA / "patient_ranker_bias_softmax_experiment_metrics.json",
    DATA / "patient_ranker_reconstructed_test_case_results.csv",
    DATA / "patient_ranker_reconstructed_test_metrics.json",
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
]

for path in files:
    if path.exists():
        digest = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
        print(f"FOUND | {path.name} | {path.stat().st_size:,} bytes | SHA256:{digest}")
    else:
        print(f"MISSING | {path.name}")

print("\n=== VALIDATION RESULT ROW AUDIT ===")
val_path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
if val_path.exists():
    val_results = pd.read_csv(val_path)
    print("Rows:", len(val_results))
    print("Columns:", val_results.columns.tolist())
    if "case_id" in val_results:
        print("Unique validation cases:", val_results["case_id"].nunique())
    print(val_results.head(5).to_string(index=False))

print("\n=== TRAINING HISTORY AUDIT ===")
history_path = DATA / "patient_ranker_bias_softmax_experiment_history.csv"
if history_path.exists():
    history = pd.read_csv(history_path)
    print("History rows:", len(history))
    print("History columns:", history.columns.tolist())
    print(history.tail(5).to_string(index=False))

print("\nRead-only inventory complete. No files modified or models trained.")


=== STAGE 8: REPRODUCIBILITY AND ARTIFACT INVENTORY ===
FOUND | patient_level_train.csv | 843,919 bytes | SHA256:c12c6baf1aa448ea
FOUND | patient_level_validation.csv | 134,375 bytes | SHA256:3aee5bff93fede2b
FOUND | patient_publication_split.csv | 617,350 bytes | SHA256:21fa4541fbb675e9
FOUND | patient_ranker_bias_softmax_experiment_validation.csv | 58,498 bytes | SHA256:70de076232a5444c
FOUND | patient_ranker_bias_softmax_experiment_history.csv | 2,026 bytes | SHA256:7d6fa3abce6dc887
FOUND | patient_ranker_bias_softmax_experiment_metrics.json | 140 bytes | SHA256:dc5f692a9bc593ce
FOUND | patient_ranker_reconstructed_test_case_results.csv | 257,098 bytes | SHA256:1be40cd9b8f9678d
FOUND | patient_ranker_reconstructed_test_metrics.json | 2,163 bytes | SHA256:1dff51e850921177
FOUND | patient_ranker_bias_softmax_experiment.pt | 9,387,364 bytes | SHA256:7671869b8e169bba

=== VALIDATION RESULT ROW AUDIT ===
Rows: 906
Columns: ['case_id', 'diagnosis_id', 'rank', 'top1', 'top5', 'top10', 'mrr

In [ ]:

from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 9: VALIDATION METRIC CONSISTENCY CHECK ===")

history_path = DATA / "patient_ranker_bias_softmax_experiment_history.csv"
results_path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
metrics_path = DATA / "patient_ranker_bias_softmax_experiment_metrics.json"

history = pd.read_csv(history_path)
results = pd.read_csv(results_path)

with open(metrics_path, "r") as f:
    saved_metrics = json.load(f)

print("\n--- SAVED METRICS JSON ---")
print(json.dumps(saved_metrics, indent=2))

print("\n--- BEST HISTORY EPOCH BY VALIDATION MRR ---")
best_idx = history["val_mrr"].idxmax()
print(history.loc[best_idx].to_string())

print("\n--- HISTORY FINAL EPOCH ---")
print(history.tail(1).to_string(index=False))

print("\n--- VALIDATION RESULT ROW CHECK ---")
print("Rows:", len(results))
print("Unique cases:", results["case_id"].nunique())
print("Duplicate cases:", int(results["case_id"].duplicated().sum()))
print("MRR from per-case reciprocal ranks:", round(results["mrr"].mean(), 6))
print("Top-1 from per-case flags:", round(results["top1"].mean(), 6))
print("Top-5 from per-case flags:", round(results["top5"].mean(), 6))
print("Top-10 from per-case flags:", round(results["top10"].mean(), 6))

print("\n--- SAVED BEST-EPOCH METRICS ---")
print("Best epoch in history:", int(history.loc[best_idx, "epoch"]))
print("Best history MRR:", round(float(history.loc[best_idx, "val_mrr"]), 6))
print("Best history Top-1:", round(float(history.loc[best_idx, "val_top1"]), 6))
print("Best history Top-5:", round(float(history.loc[best_idx, "val_top5"]), 6))
print("Best history Top-10:", round(float(history.loc[best_idx, "val_top10"]), 6))

print("\nRead-only audit. No training or files modified.")


=== STAGE 9: VALIDATION METRIC CONSISTENCY CHECK ===

--- SAVED METRICS JSON ---
{
  "cases": 906,
  "mrr": 0.3341009854885823,
  "top1": 0.30132450331125826,
  "top5": 0.3631346578366446,
  "top10": 0.39514348785871967
}

--- BEST HISTORY EPOCH BY VALIDATION MRR ---
epoch          15.000000
train_loss      0.392794
val_cases     906.000000
val_mrr         0.334101
val_top1        0.301325
val_top5        0.363135
val_top10       0.395143

--- HISTORY FINAL EPOCH ---
 epoch  train_loss  val_cases  val_mrr  val_top1  val_top5  val_top10
    19     0.29853        906 0.334063  0.303532  0.371965   0.397351

--- VALIDATION RESULT ROW CHECK ---
Rows: 906
Unique cases: 906
Duplicate cases: 0
MRR from per-case reciprocal ranks: 0.334101
Top-1 from per-case flags: 0.301325
Top-5 from per-case flags: 0.363135
Top-10 from per-case flags: 0.395143

--- SAVED BEST-EPOCH METRICS ---
Best epoch in history: 15
Best history MRR: 0.334101
Best history Top-1: 0.301325
Best history Top-5: 0.363135
Best 

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 10: TRAINING CONVERGENCE AUDIT ===")

history = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_history.csv"
).sort_values("epoch")

required = ["epoch", "train_loss", "val_mrr", "val_top1", "val_top5", "val_top10"]
missing = [c for c in required if c not in history.columns]

if missing:
    print("Missing columns:", missing)
else:
    print("Epochs recorded:", len(history))
    print("Epoch range:", int(history["epoch"].min()), "to", int(history["epoch"].max()))

    print("\n--- FIRST 5 EPOCHS ---")
    print(history[required].head().round(5).to_string(index=False))

    print("\n--- LAST 5 EPOCHS ---")
    print(history[required].tail().round(5).to_string(index=False))

    best_idx = history["val_mrr"].idxmax()
    best = history.loc[best_idx]
    final = history.iloc[-1]

    print("\n--- COMPARISON ---")
    print("Best MRR epoch:", int(best["epoch"]))
    print("Best validation MRR:", round(float(best["val_mrr"]), 5))
    print("Final epoch:", int(final["epoch"]))
    print("Final validation MRR:", round(float(final["val_mrr"]), 5))
    print("Final train loss:", round(float(final["train_loss"]), 5))
    print("Train loss decreased:",
          bool(history["train_loss"].iloc[-1] < history["train_loss"].iloc[0]))

    print("\n--- VALIDATION TREND ---")
    print("MRR increased from first to last epoch:",
          bool(history["val_mrr"].iloc[-1] > history["val_mrr"].iloc[0]))
    print("Best MRR minus final MRR:",
          round(float(best["val_mrr"] - final["val_mrr"]), 5))

print("\nRead-only audit. No training or files modified.")


=== STAGE 10: TRAINING CONVERGENCE AUDIT ===
Epochs recorded: 19
Epoch range: 1 to 19

--- FIRST 5 EPOCHS ---
 epoch  train_loss  val_mrr  val_top1  val_top5  val_top10
     1     5.25531  0.05093   0.03201   0.05960    0.08830
     2     3.61791  0.13552   0.08609   0.18433    0.21634
     3     2.75840  0.23047   0.19647   0.26821    0.29029
     4     2.09609  0.23400   0.18764   0.29249    0.32450
     5     1.63583  0.27929   0.23620   0.32561    0.35320

--- LAST 5 EPOCHS ---
 epoch  train_loss  val_mrr  val_top1  val_top5  val_top10
    15     0.39279  0.33410   0.30132   0.36313    0.39514
    16     0.35657  0.31539   0.27263   0.36313    0.39183
    17     0.33707  0.32476   0.28918   0.35762    0.39073
    18     0.31182  0.31639   0.27704   0.35430    0.38852
    19     0.29853  0.33406   0.30353   0.37196    0.39735

--- COMPARISON ---
Best MRR epoch: 15
Best validation MRR: 0.3341
Final epoch: 19
Final validation MRR: 0.33406
Final train loss: 0.29853
Train loss decreased

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 11: AUDIT EXISTING MODEL DIAGNOSTICS ===")

path = DATA / "patient_ranker_diagnostics.json"

if path.exists():
    with open(path, "r") as f:
        diagnostics = json.load(f)

    print("\n--- SAVED DIAGNOSTICS ---")
    print(json.dumps(diagnostics, indent=2)[:10000])
else:
    print("Diagnostics JSON not found:", path.name)

print("\n--- EXISTING ABLATION ARTIFACTS ---")
for name in [
    "patient_ranker_bias_ablation.csv",
    "patient_ranker_bias_ablation_summary.json",
    "patient_ranker_vs_baseline_paired_bootstrap.json",
    "training_only_phenotype_recovery_audit.csv",
]:
    p = DATA / name
    print(f"\n{name}: {'FOUND' if p.exists() else 'NOT FOUND'}")
    if p.exists() and p.suffix == ".csv":
        temp = pd.read_csv(p)
        print("Rows:", len(temp))
        print("Columns:", temp.columns.tolist())
        print(temp.head(5).to_string(index=False))
    elif p.exists() and p.suffix == ".json":
        try:
            print(json.dumps(json.loads(p.read_text()), indent=2)[:5000])
        except Exception as e:
            print("Could not parse JSON:", e)

print("\nRead-only audit. No training or files modified.")


=== STAGE 11: AUDIT EXISTING MODEL DIAGNOSTICS ===

--- SAVED DIAGNOSTICS ---
{
  "best_epoch": 25,
  "best_validation_mrr": 0.0527043189121766,
  "median_target_rank": 450.0,
  "top1_unique_diseases": 4,
  "top1_frequency": {
    "OMIM:612164": 637,
    "OMIM:148050": 249,
    "OMIM:162000": 11,
    "OMIM:142900": 9
  },
  "bias_min_median_max": [
    -8.338905334472656,
    -4.671901702880859,
    4.408734321594238
  ],
  "target_score_mean": -1.4318644955474678,
  "target_bias_mean": -1.678973670816563
}

--- EXISTING ABLATION ARTIFACTS ---

patient_ranker_bias_ablation.csv: FOUND
Rows: 906
Columns: ['case_id', 'diagnosis_ids', 'original_rank', 'original_top1_disease', 'no_bias_rank', 'no_bias_top1_disease', 'bias_only_rank', 'bias_only_top1_disease']
                                       case_id diagnosis_ids  original_rank original_top1_disease  no_bias_rank no_bias_top1_disease  bias_only_rank bias_only_top1_disease
                       PMID_28830446_Patient_2   OMIM:615688   

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 12: SELECT THE MOST RELIABLE EXISTING MODEL ===")

summary_path = DATA / "patient_ranker_bias_ablation_summary.json"
bootstrap_path = DATA / "patient_ranker_vs_baseline_paired_bootstrap.json"
metrics_path = DATA / "patient_ranker_bias_softmax_experiment_metrics.json"

with open(summary_path, "r") as f:
    ablation = json.load(f)

with open(bootstrap_path, "r") as f:
    bootstrap = json.load(f)

with open(metrics_path, "r") as f:
    saved_metrics = json.load(f)

print("\n--- EXISTING VALIDATION COMPARISON ---")
print("Ranker with bias:")
print(json.dumps(ablation["metrics"]["original"], indent=2))

print("\nRanker without bias:")
print(json.dumps(ablation["metrics"]["no_bias"], indent=2))

print("\nBias-only:")
print(json.dumps(ablation["metrics"]["bias_only"], indent=2))

print("\nPhenotype baseline vs ranker without bias:")
print(json.dumps(bootstrap["metrics"], indent=2))

print("\n--- CHECK FOR CONFLICTING SAVED RESULTS ---")
val_path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
val = pd.read_csv(val_path)

print("Saved validation rows:", len(val))
print("Unique validation cases:", val["case_id"].nunique())
print("Saved best-model metrics:", saved_metrics)

print("\n--- AUDIT DECISION ---")
print("Bias-heavy original configuration performs poorly in saved ablation.")
print("No-bias configuration is the more promising ranker variant.")
print("The phenotype baseline outperforms no-bias ranker on saved validation metrics.")
print("These validation comparisons do not establish clinical generalizability.")
print("\nRead-only audit. No training or files modified.")


=== STAGE 12: SELECT THE MOST RELIABLE EXISTING MODEL ===

--- EXISTING VALIDATION COMPARISON ---
Ranker with bias:
{
  "n_ranked": 906,
  "Top1": 0.021,
  "Top5": 0.0872,
  "Top10": 0.1104,
  "MRR": 0.0527,
  "median_rank": 450.0
}

Ranker without bias:
{
  "n_ranked": 906,
  "Top1": 0.2671,
  "Top5": 0.3444,
  "Top10": 0.3653,
  "MRR": 0.3043,
  "median_rank": 232.5
}

Bias-only:
{
  "n_ranked": 906,
  "Top1": 0.0,
  "Top5": 0.0166,
  "Top10": 0.0442,
  "MRR": 0.0154,
  "median_rank": 477.0
}

Phenotype baseline vs ranker without bias:
{
  "Top1": {
    "baseline": 0.2947,
    "ranker_no_bias": 0.2671,
    "difference_ranker_minus_baseline": -0.0276,
    "bootstrap_95pct_CI_difference": [
      -0.0717,
      0.0188
    ]
  },
  "Top5": {
    "baseline": 0.468,
    "ranker_no_bias": 0.3444,
    "difference_ranker_minus_baseline": -0.1236,
    "bootstrap_95pct_CI_difference": [
      -0.1744,
      -0.0717
    ]
  },
  "Top10": {
    "baseline": 0.5419,
    "ranker_no_bias": 0.3653,
 

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 13: CHECK VALIDATION ARTIFACT CONSISTENCY ===")

files = {
    "saved_ranker_validation": DATA / "patient_ranker_bias_softmax_experiment_validation.csv",
    "no_bias_ablation": DATA / "patient_ranker_bias_ablation.csv",
    "paired_bootstrap": DATA / "patient_ranker_vs_baseline_paired_bootstrap.json",
    "training_history": DATA / "patient_ranker_bias_softmax_experiment_history.csv",
}

for label, path in files.items():
    print(f"{label}: {'FOUND' if path.exists() else 'MISSING'}")

if all(p.exists() for p in files.values()):
    saved = pd.read_csv(files["saved_ranker_validation"])
    ablation = pd.read_csv(files["no_bias_ablation"])
    history = pd.read_csv(files["training_history"])

    print("\n--- SAVED RANKER VALIDATION ---")
    print("Rows:", len(saved))
    print("Unique cases:", saved["case_id"].nunique())
    print("MRR:", round(saved["mrr"].mean(), 6))
    print("Top1:", round(saved["top1"].mean(), 6))
    print("Top5:", round(saved["top5"].mean(), 6))
    print("Top10:", round(saved["top10"].mean(), 6))

    print("\n--- ABLATION CASE MATCH ---")
    print("Ablation rows:", len(ablation))
    print("Unique ablation cases:", ablation["case_id"].nunique())
    print("Shared case IDs:", len(set(saved["case_id"]) & set(ablation["case_id"])))

    if "original_rank" in ablation.columns and "no_bias_rank" in ablation.columns:
        print("Original rank median:", ablation["original_rank"].median())
        print("No-bias rank median:", ablation["no_bias_rank"].median())

    print("\n--- HISTORY CHECK ---")
    best = history.loc[history["val_mrr"].idxmax()]
    print("Best history epoch:", int(best["epoch"]))
    print("Best history MRR:", round(float(best["val_mrr"]), 6))
    print("Best history Top1:", round(float(best["val_top1"]), 6))
    print("Best history Top5:", round(float(best["val_top5"]), 6))
    print("Best history Top10:", round(float(best["val_top10"]), 6))

    print("\n--- DIAGNOSTIC ---")
    print("If the saved validation table and history agree, they represent the saved best-epoch metrics.")
    print("Ablation metrics are a separate experiment and should not be conflated with this checkpoint.")

print("\nRead-only audit. No training or files modified.")


=== STAGE 13: CHECK VALIDATION ARTIFACT CONSISTENCY ===
saved_ranker_validation: FOUND
no_bias_ablation: FOUND
paired_bootstrap: FOUND
training_history: FOUND

--- SAVED RANKER VALIDATION ---
Rows: 906
Unique cases: 906
MRR: 0.334101
Top1: 0.301325
Top5: 0.363135
Top10: 0.395143

--- ABLATION CASE MATCH ---
Ablation rows: 906
Unique ablation cases: 906
Shared case IDs: 906
Original rank median: 450.0
No-bias rank median: 232.5

--- HISTORY CHECK ---
Best history epoch: 15
Best history MRR: 0.334101
Best history Top1: 0.301325
Best history Top5: 0.363135
Best history Top10: 0.395143

--- DIAGNOSTIC ---
If the saved validation table and history agree, they represent the saved best-epoch metrics.
Ablation metrics are a separate experiment and should not be conflated with this checkpoint.

Read-only audit. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 14: CHECKPOINT METADATA VS VALIDATION HISTORY ===")

import torch

checkpoint_path = PROJECT / "model" / "patient_ranker_bias_softmax_experiment.pt"
ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

history = pd.read_csv(DATA / "patient_ranker_bias_softmax_experiment_history.csv")
saved_val = pd.read_csv(DATA / "patient_ranker_bias_softmax_experiment_validation.csv")

print("Checkpoint best epoch:", ckpt.get("best_epoch"))
print("History best epoch:", int(history.loc[history["val_mrr"].idxmax(), "epoch"]))
print("Checkpoint best validation metrics:")
print(ckpt.get("best_validation_metrics"))

print("\nSaved validation file metrics:")
for col in ["mrr", "top1", "top5", "top10"]:
    if col in saved_val.columns:
        print(f"{col}: {saved_val[col].mean():.6f}")

print("\n=== CHECKPOINT STRUCTURE ===")
state = ckpt.get("model_state_dict", {})
for key, value in state.items():
    if hasattr(value, "shape"):
        print(f"{key}: {tuple(value.shape)}")

print("\n=== PROVENANCE CONCLUSION ===")
print("Checkpoint and validation artifacts are present.")
print("This checks metadata consistency, not exact reproduction of original inference.")
print("Read-only. No training or files modified.")


=== STAGE 14: CHECKPOINT METADATA VS VALIDATION HISTORY ===
Checkpoint best epoch: 15
History best epoch: 15
Checkpoint best validation metrics:
{'cases': 906, 'mrr': 0.3341009854885823, 'top1': 0.30132450331125826, 'top5': 0.3631346578366446, 'top10': 0.39514348785871967}

Saved validation file metrics:
mrr: 0.334101
top1: 0.301325
top5: 0.363135
top10: 0.395143

=== CHECKPOINT STRUCTURE ===
disease_bias: (7340,)
hpo_embedding.weight: (9982, 128)
disease_embedding.weight: (7340, 128)
patient_projection.0.weight: (128, 128)
patient_projection.0.bias: (128,)
patient_projection.3.weight: (128, 128)
patient_projection.3.bias: (128,)

=== PROVENANCE CONCLUSION ===
Checkpoint and validation artifacts are present.
This checks metadata consistency, not exact reproduction of original inference.
Read-only. No training or files modified.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 15: FINAL DATA AND MODEL READINESS CHECK ===")

required = [
    DATA / "patient_level_train.csv",
    DATA / "patient_level_validation.csv",
    DATA / "patient_publication_split.csv",
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv",
    DATA / "patient_ranker_bias_softmax_experiment_history.csv",
    DATA / "patient_ranker_reconstructed_test_case_results.csv",
    DATA / "patient_ranker_reconstructed_test_metrics.json",
    PROJECT / "model" / "patient_ranker_bias_softmax_experiment.pt",
]

for path in required:
    print(f"{'OK' if path.exists() else 'MISSING'} | {path.name}")

train = pd.read_csv(DATA / "patient_level_train.csv")
val = pd.read_csv(DATA / "patient_level_validation.csv")
split = pd.read_csv(DATA / "patient_publication_split.csv")
test_results = pd.read_csv(DATA / "patient_ranker_reconstructed_test_case_results.csv")

print("\n--- DATA SIZES ---")
print("Training rows:", len(train))
print("Validation rows:", len(val))
print("Test result rows:", len(test_results))

print("\n--- BASIC INTEGRITY ---")
print("Training duplicate cases:", int(train["case_id"].duplicated().sum()))
print("Validation duplicate cases:", int(val["case_id"].duplicated().sum()))
print("Test result duplicate cases:", int(test_results["case_id"].duplicated().sum()))

train_ids = set(train["case_id"].astype(str))
val_ids = set(val["case_id"].astype(str))
test_ids = set(test_results["case_id"].astype(str))

print("Train/validation case overlap:", len(train_ids & val_ids))
print("Train/test case overlap:", len(train_ids & test_ids))
print("Validation/test case overlap:", len(val_ids & test_ids))

print("\n--- CURRENT LIMITATIONS ---")
print("1. Original inference implementation has not been recovered.")
print("2. Publication references overlap target diseases and patient reports.")
print("3. Ranker target coverage is lower than the phenotype baseline.")
print("4. Test results use reconstructed ranker inference.")

print("\nReadiness audit complete. Read-only; no files modified or models trained.")


=== STAGE 15: FINAL DATA AND MODEL READINESS CHECK ===
OK | patient_level_train.csv
OK | patient_level_validation.csv
OK | patient_publication_split.csv
OK | patient_ranker_bias_softmax_experiment_validation.csv
OK | patient_ranker_bias_softmax_experiment_history.csv
OK | patient_ranker_reconstructed_test_case_results.csv
OK | patient_ranker_reconstructed_test_metrics.json
OK | patient_ranker_bias_softmax_experiment.pt

--- DATA SIZES ---
Training rows: 5730
Validation rows: 906
Test result rows: 1959

--- BASIC INTEGRITY ---
Training duplicate cases: 0
Validation duplicate cases: 0
Test result duplicate cases: 0
Train/validation case overlap: 0
Train/test case overlap: 0
Validation/test case overlap: 0

--- CURRENT LIMITATIONS ---
1. Original inference implementation has not been recovered.
2. Publication references overlap target diseases and patient reports.
3. Ranker target coverage is lower than the phenotype baseline.
4. Test results use reconstructed ranker inference.

Readiness

In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=== STAGE 16: TRAINING STRATEGY INSPECTION ===")

files = {
    "train": DATA / "patient_level_train.csv",
    "validation": DATA / "patient_level_validation.csv",
    "test_results": DATA / "patient_ranker_reconstructed_test_case_results.csv",
    "phenotype_baseline": DATA / "patient_phenotype_only_baseline.csv",
    "hpo_annotations": DATA / "clean_hpo_annotations.csv",
}

frames = {}
for name, path in files.items():
    if path.exists():
        df = pd.read_csv(path, nrows=5 if name == "hpo_annotations" else None)
        frames[name] = df
        print(f"\n--- {name.upper()} ---")
        print("Shape inspected:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(2).to_string(index=False, max_cols=12))
    else:
        print(f"\nMISSING: {path.name}")

checkpoint = MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt"
print("\n--- CHECKPOINT ---")
print("Exists:", checkpoint.exists())
if checkpoint.exists():
    import torch
    ckpt = torch.load(checkpoint, map_location="cpu", weights_only=False)
    print("Best epoch:", ckpt.get("best_epoch"))
    print("Training cases:", ckpt.get("training_cases"))
    print("Validation cases:", ckpt.get("validation_cases"))
    print("Objective:", ckpt.get("objective"))
    print("Hidden dimension:", ckpt.get("hidden_dim"))
    print("Disease vocabulary:", len(ckpt.get("disease_ids", [])))
    print("Phenotype vocabulary:", len(ckpt.get("phenotype_ids", [])))
    print("Saved validation metrics:", ckpt.get("best_validation_metrics"))

print("\nInspection complete. No files modified; no model trained.")

=== STAGE 16: TRAINING STRATEGY INSPECTION ===

--- TRAIN ---
Shape inspected: (5730, 6)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']
                    case_id     pmid diagnosis_ids                                                           hpo_ids  n_hpo  n_diagnoses
PMID_19836009_Family_E_II_6 19836009   OMIM:277600 HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003      6            1
PMID_19836009_Family_E_II_3 19836009   OMIM:277600 HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003      6            1

--- VALIDATION ---
Shape inspected: (906, 6)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'hpo_ids', 'n_hpo', 'n_diagnoses']
                case_id     pmid diagnosis_ids                                                                                                                                                                                    hpo_ids  n_hpo  n_diagnoses
PMID_28830446_Patient_2 28830446   OMI

In [ ]:
# STAGE 17: AUDIT TRAINING LABELS AND VALIDATION QUALITY
# Read-only: no training, no checkpoint changes.

import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

train = pd.read_csv(DATA / "patient_level_train.csv")
val = pd.read_csv(DATA / "patient_level_validation.csv")
split = pd.read_csv(DATA / "patient_publication_split.csv")

def labels(series):
    return series.fillna("").astype(str).apply(
        lambda x: [v.strip() for v in x.replace(",", ";").split(";") if v.strip()]
    )

train_labels = labels(train["diagnosis_ids"])
val_labels = labels(val["diagnosis_ids"])

train_diseases = set(d for row in train_labels for d in row)
val_diseases = set(d for row in val_labels for d in row)

train_hpos = set(
    h.strip()
    for row in train["hpo_ids"].fillna("").astype(str)
    for h in row.split(";") if h.strip()
)
val_hpos = set(
    h.strip()
    for row in val["hpo_ids"].fillna("").astype(str)
    for h in row.split(";") if h.strip()
)

print("=== STAGE 17: LABEL AND VOCABULARY AUDIT ===")
print("Training cases:", len(train))
print("Validation cases:", len(val))
print("Training unique diseases:", len(train_diseases))
print("Validation unique diseases:", len(val_diseases))
print("Validation diseases seen in training:", len(val_diseases & train_diseases))
print("Validation diseases unseen in training:", len(val_diseases - train_diseases))
print("Training unique HPO terms:", len(train_hpos))
print("Validation unique HPO terms:", len(val_hpos))
print("Validation HPO terms seen in training:", len(val_hpos & train_hpos))
print("Validation HPO terms unseen in training:", len(val_hpos - train_hpos))

print("\n--- LABEL COUNTS ---")
print("Training cases with exactly 1 diagnosis:", sum(len(x) == 1 for x in train_labels))
print("Training cases with multiple diagnoses:", sum(len(x) > 1 for x in train_labels))
print("Validation cases with exactly 1 diagnosis:", sum(len(x) == 1 for x in val_labels))
print("Validation cases with multiple diagnoses:", sum(len(x) > 1 for x in val_labels))

print("\n--- PUBLICATION SPLIT LABELS ---")
print(split["split"].value_counts(dropna=False).to_string())

print("\nAudit complete. No files modified; no model trained.")

=== STAGE 17: LABEL AND VOCABULARY AUDIT ===
Training cases: 5730
Validation cases: 906
Training unique diseases: 499
Validation unique diseases: 125
Validation diseases seen in training: 70
Validation diseases unseen in training: 55
Training unique HPO terms: 3373
Validation unique HPO terms: 1366
Validation HPO terms seen in training: 1141
Validation HPO terms unseen in training: 225

--- LABEL COUNTS ---
Training cases with exactly 1 diagnosis: 5730
Training cases with multiple diagnoses: 0
Validation cases with exactly 1 diagnosis: 906
Validation cases with multiple diagnoses: 0

--- PUBLICATION SPLIT LABELS ---
split
development    8386
test           1959

Audit complete. No files modified; no model trained.


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

ckpt = torch.load(
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
    map_location="cpu",
    weights_only=False
)

val = pd.read_csv(DATA / "patient_level_validation.csv")
pred_path = DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
pred = pd.read_csv(pred_path)

disease_vocab = set(map(str, ckpt["disease_ids"]))
val["target"] = val["diagnosis_ids"].fillna("").astype(str).str.strip()
val["target_covered"] = val["target"].isin(disease_vocab)

print("=== STAGE 18: VALIDATION COVERAGE ===")
print("Checkpoint disease vocabulary:", len(disease_vocab))
print("Validation cases:", len(val))
print("Targets covered by checkpoint:", int(val["target_covered"].sum()))
print("Targets outside checkpoint:", int((~val["target_covered"]).sum()))
print("Coverage rate:", round(val["target_covered"].mean() * 100, 2), "%")

print("\n--- SAVED PREDICTION FILE ---")
print("Rows:", len(pred))
print("Columns:", pred.columns.tolist())

# Merge by case ID to avoid relying on row order.
if "case_id" in pred.columns:
    merged = val[["case_id", "target_covered"]].merge(
        pred, on="case_id", how="left", validate="one_to_one"
    )
    print("Cases matched to saved predictions:", int(merged[pred.columns[0] == merged[pred.columns[0]]].shape[0]) if False else int(merged["case_id"].notna().sum()))

    for col in ["top1", "top5", "top10", "reciprocal_rank", "ranker_top1",
                "ranker_top5", "ranker_top10", "ranker_reciprocal_rank"]:
        if col in merged.columns:
            print(f"\n{col}:")
            for label, subset in [
                ("All validation cases", merged),
                ("Target covered", merged[merged["target_covered"]]),
                ("Target unseen", merged[~merged["target_covered"]]),
            ]:
                values = pd.to_numeric(subset[col], errors="coerce").dropna()
                if len(values):
                    print(f"  {label}: n={len(values)}, mean={values.mean():.4f}")
else:
    print("No case_id column found; cannot safely match saved predictions.")

print("\nRead-only audit complete.")

=== STAGE 18: VALIDATION COVERAGE ===
Checkpoint disease vocabulary: 7340
Validation cases: 906
Targets covered by checkpoint: 906
Targets outside checkpoint: 0
Coverage rate: 100.0 %

--- SAVED PREDICTION FILE ---
Rows: 906
Columns: ['case_id', 'diagnosis_id', 'rank', 'top1', 'top5', 'top10', 'mrr']
Cases matched to saved predictions: 906

top1:
  All validation cases: n=906, mean=0.3013
  Target covered: n=906, mean=0.3013

top5:
  All validation cases: n=906, mean=0.3631
  Target covered: n=906, mean=0.3631

top10:
  All validation cases: n=906, mean=0.3951
  Target covered: n=906, mean=0.3951

Read-only audit complete.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

val = pd.read_csv(DATA / "patient_level_validation.csv")
pred = pd.read_csv(
    DATA / "patient_ranker_bias_softmax_experiment_validation.csv"
)

df = val.merge(
    pred,
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_case", "_pred")
)

df["n_hpo"] = pd.to_numeric(df["n_hpo"], errors="coerce")
df["rank"] = pd.to_numeric(df["rank"], errors="coerce")

# Count training examples for each disease.
train = pd.read_csv(DATA / "patient_level_train.csv")
train_counts = (
    train.assign(diagnosis_id=train["diagnosis_ids"].astype(str).str.strip())
         .groupby("diagnosis_id")["case_id"]
         .nunique()
)

df["target_disease"] = df["diagnosis_ids"].astype(str).str.strip()
df["training_examples_for_target"] = (
    df["target_disease"].map(train_counts).fillna(0).astype(int)
)

print("=== STAGE 19: VALIDATION ERROR ANALYSIS ===")
print("Matched validation cases:", len(df))
print("Missing saved ranks:", int(df["rank"].isna().sum()))
print("Overall Top-1:", round(df["top1"].mean(), 4))
print("Overall Top-5:", round(df["top5"].mean(), 4))
print("Overall Top-10:", round(df["top10"].mean(), 4))
print("Overall MRR:", round(df["mrr"].mean(), 4))

print("\n--- PERFORMANCE BY NUMBER OF PHENOTYPES ---")
df["phenotype_group"] = pd.cut(
    df["n_hpo"],
    bins=[-1, 3, 5, 10, 20, np.inf],
    labels=["0–3", "4–5", "6–10", "11–20", "21+"]
)
print(
    df.groupby("phenotype_group", observed=False)
      .agg(cases=("case_id", "count"),
           top1=("top1", "mean"),
           top5=("top5", "mean"),
           top10=("top10", "mean"),
           mrr=("mrr", "mean"))
      .round(4).to_string()
)

print("\n--- PERFORMANCE BY TARGET DISEASE TRAINING FREQUENCY ---")
df["frequency_group"] = pd.cut(
    df["training_examples_for_target"],
    bins=[-1, 0, 1, 5, 20, np.inf],
    labels=["0", "1", "2–5", "6–20", "21+"]
)
print(
    df.groupby("frequency_group", observed=False)
      .agg(cases=("case_id", "count"),
           top1=("top1", "mean"),
           top5=("top5", "mean"),
           top10=("top10", "mean"),
           mrr=("mrr", "mean"))
      .round(4).to_string()
)

print("\n--- WORST-RANKED CASES ---")
cols = ["case_id", "target_disease", "n_hpo",
        "training_examples_for_target", "rank", "top1", "top5", "top10"]
print(df.sort_values("rank", ascending=False)[cols].head(10).to_string(index=False))

print("\nRead-only analysis complete; no files changed or models trained.")

=== STAGE 19: VALIDATION ERROR ANALYSIS ===
Matched validation cases: 906
Missing saved ranks: 0
Overall Top-1: 0.3013
Overall Top-5: 0.3631
Overall Top-10: 0.3951
Overall MRR: 0.3341

--- PERFORMANCE BY NUMBER OF PHENOTYPES ---
                 cases    top1    top5   top10     mrr
phenotype_group                                       
0–3                179  0.2570  0.2849  0.2905  0.2700
4–5                125  0.3600  0.4320  0.4640  0.3969
6–10               349  0.3725  0.4585  0.4928  0.4168
11–20              211  0.2227  0.2701  0.3081  0.2499
21+                 42  0.1190  0.1667  0.2619  0.1563

--- PERFORMANCE BY TARGET DISEASE TRAINING FREQUENCY ---
                 cases    top1    top5   top10     mrr
frequency_group                                       
0                  423  0.0000  0.0000  0.0000  0.0004
1                   53  0.0755  0.1132  0.1321  0.1152
2–5                 65  0.2154  0.4154  0.5385  0.3197
6–20               232  0.6293  0.7543  0.8233  0.693

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

val = pd.read_csv(DATA / "patient_level_validation.csv")
ann = pd.read_csv(DATA / "clean_hpo_annotations.csv", low_memory=False)

required = {"case_id", "pmid", "diagnosis_ids"}
if not required.issubset(val.columns):
    raise ValueError(f"Validation file missing columns: {required - set(val.columns)}")

required_ann = {"database_id", "hpo_id", "reference"}
if not required_ann.issubset(ann.columns):
    raise ValueError(f"Annotation file missing columns: {required_ann - set(ann.columns)}")

def split_ids(value):
    if pd.isna(value):
        return []
    return [x.strip() for x in str(value).replace(",", ";").split(";") if x.strip()]

val = val.copy()
val["target_diseases"] = val["diagnosis_ids"].apply(split_ids)
ann = ann.copy()
ann["database_id"] = ann["database_id"].fillna("").astype(str).str.strip()
ann["hpo_id"] = ann["hpo_id"].fillna("").astype(str).str.strip()
ann["reference"] = ann["reference"].fillna("").astype(str).str.strip()

ann = ann[(ann["database_id"] != "") & (ann["hpo_id"] != "")]
ann["reference_pmids"] = ann["reference"].str.findall(r"(?i)PMID:\s*(\d+)")

# One row per validation case; report each target disease separately.
rows = []
for _, case in val.iterrows():
    case_pmid = str(case["pmid"]).strip()
    for disease in case["target_diseases"]:
        disease_ann = ann[ann["database_id"] == disease]
        all_hpos = set(disease_ann["hpo_id"])
        cited_same_pmid = disease_ann["reference_pmids"].apply(
            lambda refs: case_pmid in refs
        )
        same_pmid_hpos = set(disease_ann.loc[cited_same_pmid, "hpo_id"])
        remaining_hpos = set(disease_ann.loc[~cited_same_pmid, "hpo_id"])

        rows.append({
            "case_id": case["case_id"],
            "pmid": case_pmid,
            "disease": disease,
            "profile_hpo_count": len(all_hpos),
            "same_pmid_hpo_count": len(same_pmid_hpos),
            "remaining_hpo_count": len(remaining_hpos),
            "profile_has_other_reference": len(remaining_hpos) > 0,
        })

audit = pd.DataFrame(rows)

print("=== STAGE 20: PUBLICATION-AWARE PROFILE COVERAGE ===")
print("Validation cases:", val["case_id"].nunique())
print("Validation case-target pairs:", len(audit))
print("Pairs with any disease profile:", int((audit["profile_hpo_count"] > 0).sum()))
print("Pairs with profile after same-PMID exclusion:",
      int(audit["profile_has_other_reference"].sum()))
print("Pairs with no remaining HPO profile:",
      int((audit["remaining_hpo_count"] == 0).sum()))
print("Median original profile HPO count:", audit["profile_hpo_count"].median())
print("Median remaining profile HPO count:", audit["remaining_hpo_count"].median())

print("\n--- REMAINING PROFILE SIZE ---")
print(
    audit["remaining_hpo_count"]
    .value_counts()
    .sort_index()
    .head(15)
    .to_string()
)

print("\n--- EXAMPLES WITH NO REMAINING PROFILE ---")
print(
    audit.loc[audit["remaining_hpo_count"] == 0,
              ["case_id", "pmid", "disease", "profile_hpo_count", "same_pmid_hpo_count"]]
    .head(12)
    .to_string(index=False)
)

print("\nRead-only audit complete. No files modified; no model trained.")

=== STAGE 20: PUBLICATION-AWARE PROFILE COVERAGE ===
Validation cases: 906
Validation case-target pairs: 906
Pairs with any disease profile: 906
Pairs with profile after same-PMID exclusion: 675
Pairs with no remaining HPO profile: 231
Median original profile HPO count: 53.0
Median remaining profile HPO count: 15.0

--- REMAINING PROFILE SIZE ---
remaining_hpo_count
0     231
1      32
2       3
3       3
4       3
5       6
6       2
7      87
8      16
9      41
11      3
13     11
15     23
16     25
17      2

--- EXAMPLES WITH NO REMAINING PROFILE ---
                           case_id     pmid     disease  profile_hpo_count  same_pmid_hpo_count
   PMID_35534203_Family_R0036_II_1 35534203 OMIM:620744                  7                    7
PMID_35534203_Family_W0031_II_1_P3 35534203 OMIM:620744                  7                    7
       PMID_35534203_R0036_II_3_P2 35534203 OMIM:620744                  7                    7
 PMID_38753057_Family_D_individual 38753057 OMIM:6213

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import re

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

val = pd.read_csv(DATA / "patient_level_validation.csv")
ann = pd.read_csv(DATA / "clean_hpo_annotations.csv", low_memory=False)

def parse_ids(value):
    if pd.isna(value):
        return []
    return [x.strip() for x in str(value).replace(",", ";").split(";") if x.strip()]

ann = ann.copy()
for col in ["database_id", "hpo_id", "reference"]:
    ann[col] = ann[col].fillna("").astype(str).str.strip()

ann = ann[(ann["database_id"] != "") & (ann["hpo_id"] != "")]
ann["ref_pmids"] = ann["reference"].apply(
    lambda s: set(re.findall(r"(?i)PMID:\s*(\d+)", s))
)

# Keep annotation rows grouped by disease; for each case, remove rows
# whose reference list contains that case's own publication PMID.
disease_rows = {
    disease: group
    for disease, group in ann.groupby("database_id", sort=False)
}
all_diseases = sorted(disease_rows)

case_results = []

for i, case in val.iterrows():
    pmid = str(case["pmid"]).strip()
    target_ids = parse_ids(case["diagnosis_ids"])
    observed = set(parse_ids(case["hpo_ids"]))

    if not observed or not target_ids:
        continue

    scores = {}
    for disease, group in disease_rows.items():
        eligible = group[
            ~group["ref_pmids"].apply(lambda refs: pmid in refs)
        ]
        profile = set(eligible["hpo_id"])
        if not profile:
            continue

        overlap = len(observed & profile)
        # Normalized overlap reduces the advantage of very large profiles.
        score = overlap / np.sqrt(len(observed) * len(profile))
        scores[disease] = score

    for target in target_ids:
        if target not in scores:
            case_results.append({
                "case_id": case["case_id"],
                "target": target,
                "covered": False,
                "rank": np.nan,
                "top1": 0,
                "top5": 0,
                "top10": 0,
                "mrr": 0.0,
            })
            continue

        target_score = scores[target]
        # Competition ranking: number of diseases with a strictly higher score + 1.
        rank = 1 + sum(score > target_score for score in scores.values())

        case_results.append({
            "case_id": case["case_id"],
            "target": target,
            "covered": True,
            "rank": rank,
            "top1": int(rank <= 1),
            "top5": int(rank <= 5),
            "top10": int(rank <= 10),
            "mrr": 1.0 / rank,
        })

results = pd.DataFrame(case_results)

print("=== STAGE 21: PUBLICATION-AWARE BASELINE ===")
print("Validation cases:", val["case_id"].nunique())
print("Case-target evaluations:", len(results))
print("Targets with remaining profiles:", int(results["covered"].sum()))
print("Targets without remaining profiles:", int((~results["covered"]).sum()))
print("Target profile coverage:", round(results["covered"].mean() * 100, 2), "%")

covered = results[results["covered"]]
print("\n--- METRICS ON COVERED TARGETS ---")
if len(covered):
    print("Evaluated targets:", len(covered))
    print("Top-1:", round(covered["top1"].mean(), 4))
    print("Top-5:", round(covered["top5"].mean(), 4))
    print("Top-10:", round(covered["top10"].mean(), 4))
    print("MRR:", round(covered["mrr"].mean(), 4))
    print("Median target rank:", float(covered["rank"].median()))

print("\n--- METRICS INCLUDING UNCOVERED TARGETS AS MISSES ---")
print("Top-1:", round(results["top1"].mean(), 4))
print("Top-5:", round(results["top5"].mean(), 4))
print("Top-10:", round(results["top10"].mean(), 4))
print("MRR:", round(results["mrr"].mean(), 4))

print("\nRead-only experiment complete. No files modified; no model trained.")

=== STAGE 21: PUBLICATION-AWARE BASELINE ===
Validation cases: 906
Case-target evaluations: 906
Targets with remaining profiles: 675
Targets without remaining profiles: 231
Target profile coverage: 74.5 %

--- METRICS ON COVERED TARGETS ---
Evaluated targets: 675
Top-1: 0.0415
Top-5: 0.0652
Top-10: 0.0785
MRR: 0.0575
Median target rank: 474.0

--- METRICS INCLUDING UNCOVERED TARGETS AS MISSES ---
Top-1: 0.0309
Top-5: 0.0486
Top-10: 0.0585
MRR: 0.0429

Read-only experiment complete. No files modified; no model trained.


In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
MODEL_ID = "gemini-3.8-flash"

In [ ]:
from pathlib import Path
import time

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=== STAGE 21A: RUNTIME RECOVERY CHECK ===")
print("Python runtime:", "OK")
print("Check time:", time.strftime("%H:%M:%S"))

for p in [
    DATA / "patient_level_validation.csv",
    DATA / "clean_hpo_annotations.csv",
    MODEL_DIR / "patient_ranker_bias_softmax_experiment.pt",
]:
    print(f"{'OK' if p.exists() else 'MISSING'} | {p.name}")

print("Check complete. No files modified; no training performed.")

=== STAGE 21A: RUNTIME RECOVERY CHECK ===
Python runtime: OK
Check time: 02:52:17
OK | patient_level_validation.csv
OK | clean_hpo_annotations.csv
OK | patient_ranker_bias_softmax_experiment.pt
Check complete. No files modified; no training performed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

from pathlib import Path
import re, time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
t0 = time.time()

print("=== STAGE 21B: OPTIMIZED PUBLICATION-AWARE BASELINE ===")

val = pd.read_csv(DATA / "patient_level_validation.csv",
                  usecols=["case_id", "pmid", "diagnosis_ids", "hpo_ids"])
ann = pd.read_csv(DATA / "clean_hpo_annotations.csv",
                  usecols=["database_id", "hpo_id", "reference"])

def ids(x):
    if pd.isna(x): return []
    return [s.strip() for s in str(x).replace(",", ";").split(";") if s.strip()]

val["pmid"] = val["pmid"].astype(str).str.strip()
val["targets"] = val["diagnosis_ids"].apply(ids)
val["observed"] = val["hpo_ids"].apply(ids)

for c in ["database_id", "hpo_id", "reference"]:
    ann[c] = ann[c].fillna("").astype(str).str.strip()

ann = ann[(ann.database_id != "") & (ann.hpo_id != "")].drop_duplicates(
    ["database_id", "hpo_id", "reference"]
)
ann["ref_pmids"] = ann["reference"].str.findall(r"(?i)PMID:\s*(\d+)")
ann["ref_set"] = ann["ref_pmids"].apply(set)

# Vocabulary and patient matrix
hpos = sorted(set(h for row in val.observed for h in row) | set(ann.hpo_id))
h2i = {h:i for i,h in enumerate(hpos)}
diseases = sorted(ann.database_id.unique())
d2i = {d:i for i,d in enumerate(diseases)}

pr, pc = [], []
for i, terms in enumerate(val.observed):
    for h in set(terms):
        pr.append(i); pc.append(h2i[h])
P = csr_matrix((np.ones(len(pr), np.float32), (pr, pc)),
               shape=(len(val), len(hpos)), dtype=np.float32)
P = normalize(P, norm="l2", axis=1)

# Pre-index annotation rows by PMID for fast exclusion
refs_by_pmid = {}
for idx, refset in enumerate(ann.ref_set):
    for pmid in refset:
        refs_by_pmid.setdefault(pmid, []).append(idx)

ann_d = ann.database_id.map(d2i).to_numpy()
ann_h = ann.hpo_id.map(h2i).to_numpy()
all_row_indices = np.arange(len(ann))
disease_rows = np.bincount(ann_d, minlength=len(diseases)) > 0

out = []
# Cache profile matrices for cases sharing a publication PMID.
profile_cache = {}

for case_i, row in val.iterrows():
    pmid = row["pmid"]
    if pmid not in profile_cache:
        excluded = np.asarray(refs_by_pmid.get(pmid, []), dtype=int)
        keep = np.ones(len(ann), dtype=bool)
        keep[excluded] = False
        rr = np.flatnonzero(keep)
        prof = csr_matrix(
            (np.ones(len(rr), np.float32), (ann_d[rr], ann_h[rr])),
            shape=(len(diseases), len(hpos)), dtype=np.float32
        )
        prof.data[:] = 1.0
        prof = normalize(prof, norm="l2", axis=1)
        profile_cache[pmid] = prof

    scores = (P.getrow(case_i) @ profile_cache[pmid].T).toarray().ravel()
    scores[~disease_rows] = -1.0

    for target in row["targets"]:
        if target not in d2i or scores[d2i[target]] < 0:
            out.append((row["case_id"], target, False, np.nan, 0, 0, 0, 0.0))
            continue
        ts = scores[d2i[target]]
        rank = 1 + int(np.sum(scores > ts))
        out.append((row["case_id"], target, True, rank,
                    int(rank <= 1), int(rank <= 5), int(rank <= 10), 1.0/rank))

res = pd.DataFrame(out, columns=[
    "case_id", "target", "covered", "rank", "top1", "top5", "top10", "mrr"
])
cov = res[res.covered]

print("\nValidation cases:", val.case_id.nunique())
print("Case-target evaluations:", len(res))
print("Covered targets:", len(cov))
print("Uncovered targets:", int((~res.covered).sum()))
print("Coverage (%):", round(res.covered.mean()*100, 2))

def report(name, frame):
    print("\n" + name)
    if frame.empty:
        print("No cases.")
        return
    for col in ["top1", "top5", "top10", "mrr"]:
        print(f"{col.upper()}: {frame[col].mean():.4f}")
    ranks = frame["rank"].dropna()
    print("Median target rank:", ranks.median() if len(ranks) else "N/A")

report("COVERED TARGETS ONLY", cov)
report("ALL TARGETS (UNCOVERED = MISSES)", res)
print(f"\nElapsed: {time.time()-t0:.1f} seconds")
print("Read-only: no files changed; no model trained.")


=== STAGE 21B: OPTIMIZED PUBLICATION-AWARE BASELINE ===

Validation cases: 906
Case-target evaluations: 906
Covered targets: 906
Uncovered targets: 0
Coverage (%): 100.0

COVERED TARGETS ONLY
TOP1: 0.0375
TOP5: 0.0552
TOP10: 0.0640
MRR: 0.0498
Median target rank: 651.0

ALL TARGETS (UNCOVERED = MISSES)
TOP1: 0.0375
TOP5: 0.0552
TOP10: 0.0640
MRR: 0.0498
Median target rank: 651.0

Elapsed: 9.2 seconds
Read-only: no files changed; no model trained.


In [ ]:

from pathlib import Path
import time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
t0 = time.time()

print("=== STAGE 22: INFORMATION-WEIGHTED PHENOTYPE RANKING ===")

val = pd.read_csv(
    DATA / "patient_level_validation.csv",
    usecols=["case_id", "pmid", "diagnosis_ids", "hpo_ids"]
)
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    usecols=["database_id", "hpo_id", "reference"]
)

def parse_ids(x):
    if pd.isna(x):
        return []
    return [s.strip() for s in str(x).replace(",", ";").split(";") if s.strip()]

val["pmid"] = val["pmid"].astype(str).str.strip()
val["targets"] = val["diagnosis_ids"].apply(parse_ids)
val["observed"] = val["hpo_ids"].apply(parse_ids)

for c in ["database_id", "hpo_id", "reference"]:
    ann[c] = ann[c].fillna("").astype(str).str.strip()

ann = ann[(ann.database_id != "") & (ann.hpo_id != "")]
ann = ann.drop_duplicates(["database_id", "hpo_id", "reference"])
ann["refs"] = ann["reference"].str.findall(r"(?i)PMID:\s*(\d+)")
ann["refset"] = ann["refs"].apply(set)

hpos = sorted(set(h for row in val["observed"] for h in row) |
              set(ann["hpo_id"]))
h2i = {h:i for i,h in enumerate(hpos)}
diseases = sorted(ann["database_id"].unique())
d2i = {d:i for i,d in enumerate(diseases)}

# Patient x HPO sparse matrix
pr, pc = [], []
for i, terms in enumerate(val["observed"]):
    for h in set(terms):
        pr.append(i)
        pc.append(h2i[h])

P = csr_matrix(
    (np.ones(len(pr), np.float32), (pr, pc)),
    shape=(len(val), len(hpos)), dtype=np.float32
)

# IDF weights derived from disease profiles. This weighting is fixed
# across cases and does not use validation outcomes.
disease_hpo = ann[["database_id", "hpo_id"]].drop_duplicates()
hpo_disease_count = disease_hpo.groupby("hpo_id")["database_id"].nunique()
idf = np.ones(len(hpos), dtype=np.float32)
for h, count in hpo_disease_count.items():
    idf[h2i[h]] = np.log1p(len(diseases) / max(int(count), 1))

P = P.multiply(idf).tocsr()
P = normalize(P, norm="l2", axis=1)

ann_d = ann["database_id"].map(d2i).to_numpy()
ann_h = ann["hpo_id"].map(h2i).to_numpy()

# Cache publication-filtered disease profiles by PMID.
ref_rows_by_pmid = {}
for idx, refs in enumerate(ann["refset"]):
    for p in refs:
        ref_rows_by_pmid.setdefault(p, []).append(idx)

profile_cache = {}
disease_has_profile = np.zeros(len(diseases), dtype=bool)
out = []

for i, row in val.iterrows():
    pmid = row["pmid"]

    if pmid not in profile_cache:
        excluded = np.asarray(ref_rows_by_pmid.get(pmid, []), dtype=int)
        keep = np.ones(len(ann), dtype=bool)
        keep[excluded] = False
        rr = np.flatnonzero(keep)

        profile = csr_matrix(
            (idf[ann_h[rr]], (ann_d[rr], ann_h[rr])),
            shape=(len(diseases), len(hpos)), dtype=np.float32
        )
        profile.data[:] = np.repeat(1.0, len(profile.data)) * profile.data
        profile.eliminate_zeros()
        profile = normalize(profile, norm="l2", axis=1)
        profile_cache[pmid] = profile

    scores = (P.getrow(i) @ profile_cache[pmid].T).toarray().ravel()
    scores[profile_cache[pmid].getnnz(axis=1) == 0] = -1.0

    for target in row["targets"]:
        if target not in d2i or scores[d2i[target]] < 0:
            out.append((row["case_id"], target, False, np.nan, 0, 0, 0, 0.0))
            continue

        target_score = scores[d2i[target]]
        rank = 1 + int(np.sum(scores > target_score))
        out.append((
            row["case_id"], target, True, rank,
            int(rank <= 1), int(rank <= 5), int(rank <= 10), 1.0 / rank
        ))

res = pd.DataFrame(out, columns=[
    "case_id", "target", "covered", "rank",
    "top1", "top5", "top10", "mrr"
])
covered = res[res["covered"]]

print("\nEvaluations:", len(res))
print("Covered targets:", len(covered))
print("Coverage:", round(res["covered"].mean() * 100, 2), "%")

for label, frame in [
    ("COVERED TARGETS", covered),
    ("ALL TARGETS (UNCOVERED = MISSES)", res)
]:
    print("\n" + label)
    for metric in ["top1", "top5", "top10", "mrr"]:
        print(f"{metric.upper()}: {frame[metric].mean():.4f}")
    ranks = frame["rank"].dropna()
    print("Median rank:", ranks.median() if len(ranks) else "N/A")

print(f"\nElapsed seconds: {time.time() - t0:.1f}")
print("Read-only: no files changed; no model trained.")


=== STAGE 22: INFORMATION-WEIGHTED PHENOTYPE RANKING ===

Evaluations: 906
Covered targets: 675
Coverage: 74.5 %

COVERED TARGETS
TOP1: 0.0415
TOP5: 0.0696
TOP10: 0.0844
MRR: 0.0570
Median rank: 473.0

ALL TARGETS (UNCOVERED = MISSES)
TOP1: 0.0309
TOP5: 0.0519
TOP10: 0.0629
MRR: 0.0425
Median rank: 473.0

Elapsed seconds: 9.6
Read-only: no files changed; no model trained.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=== STAGE 23: ONTOLOGY AND EXISTING EXPERIMENT INVENTORY ===")

for folder in [DATA, MODEL_DIR]:
    print(f"\n--- {folder.name.upper()} FILES ---")
    if not folder.exists():
        print("Folder missing")
        continue

    files = sorted(
        p for p in folder.rglob("*")
        if p.is_file()
    )

    matches = [
        p for p in files
        if any(term in p.name.lower() for term in [
            "hpo", "ontology", "phenotype", "similarity",
            "recovery", "ablation", "ranker", "disease"
        ])
    ]

    for p in matches[:100]:
        print(f"{p.relative_to(PROJECT)} ({p.stat().st_size:,} bytes)")

    print(f"Matching files: {len(matches)}")

print("\n--- EXISTING RECOVERY AUDIT ---")
audit_path = DATA / "training_only_phenotype_recovery_audit.csv"
if audit_path.exists():
    audit = pd.read_csv(audit_path)
    print("Rows:", len(audit))
    print("Columns:", audit.columns.tolist())
    print(audit.head(5).to_string(index=False))
else:
    print("No existing recovery audit found.")

print("\nInventory complete. No files changed; no model trained.")

=== STAGE 23: ONTOLOGY AND EXISTING EXPERIMENT INVENTORY ===

--- PROCESSED FILES ---
processed/clean_hpo_annotations.csv (35,907,264 bytes)
processed/disease_level_split.csv (162,821 bytes)
processed/disease_ranking_retrieval_metrics.csv (45,159 bytes)
processed/gene_informed_phenotype_recovery_metrics.csv (90,137 bytes)
processed/gene_informed_phenotype_recovery_top10.csv (647,512 bytes)
processed/gnn_phenotype_recovery_metrics.csv (74,139 bytes)
processed/heldout_input_phenotypes.csv (496,634 bytes)
processed/heldout_profile_nearest_training_disease.csv (85,055 bytes)
processed/heldout_target_phenotypes.csv (517,620 bytes)
processed/hpo_symptom_catalogue.csv (1,472,876 bytes)
processed/leave_one_out_similarity_top10.csv (748,237 bytes)
processed/matched_phenotype_gene_comparison.csv (141,741 bytes)
processed/missing_hpo_phenotype_audit.json (592 bytes)
processed/normalized_disease_gene_mapping.csv (230,797 bytes)
processed/ontology_fallback_candidates.csv (1,408 bytes)
processed/ont

In [ ]:

from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL_DIR = PROJECT / "model"

print("=== STAGE 24: EXISTING EXPERIMENT RESULTS ===")

files = [
    DATA / "disease_ranking_retrieval_metrics.csv",
    DATA / "gene_informed_phenotype_recovery_metrics.csv",
    DATA / "gnn_phenotype_recovery_metrics.csv",
    DATA / "phenotype_recovery_baseline_metrics.csv",
    DATA / "phenotype_scoring_method_comparison.csv",
    DATA / "phenotype_disease_ranking_evaluation.csv",
    DATA / "ontology_fallback_coverage_summary.csv",
    DATA / "ontology_fallback_evaluation.csv",
    DATA / "patient_level_ranker_bias_free_experiment_metrics.json",
    DATA / "patient_ranker_bias_softmax_experiment_metrics.json",
    DATA / "patient_ranker_reconstructed_test_metrics.json",
]

for path in files:
    if not path.exists():
        continue
    print(f"\n--- {path.name} ---")
    try:
        if path.suffix.lower() == ".json":
            obj = json.loads(path.read_text())
            print(json.dumps(obj, indent=2)[:5000])
        else:
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            print(df.head(8).to_string(index=False, max_cols=12))
    except Exception as e:
        print("Could not read:", repr(e))

report = DATA / "rare_disease_final_research_report.md"
if report.exists():
    print("\n--- FINAL RESEARCH REPORT: FIRST 180 LINES ---")
    lines = report.read_text(errors="replace").splitlines()
    print("\n".join(lines[:180]))

print("\nRead-only audit complete. No files changed; no models trained.")


=== STAGE 24: EXISTING EXPERIMENT RESULTS ===

--- disease_ranking_retrieval_metrics.csv ---
Shape: (1829, 6)
Columns: ['disease_id', 'rank', 'recall_at_1', 'recall_at_5', 'recall_at_10', 'reciprocal_rank']
 disease_id  rank  recall_at_1  recall_at_5  recall_at_10  reciprocal_rank
OMIM:100100     1            1            1             1              1.0
OMIM:101900     1            1            1             1              1.0
OMIM:102530     1            1            1             1              1.0
OMIM:102900     1            1            1             1              1.0
OMIM:103500     1            1            1             1              1.0
OMIM:103780     1            1            1             1              1.0
OMIM:104290     1            1            1             1              1.0
OMIM:105150     1            1            1             1              1.0

--- gene_informed_phenotype_recovery_metrics.csv ---
Shape: (1832, 7)
Columns: ['disease_id', 'input_phenotype_count'

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 25: METHOD COMPARISON AUDIT ===")

# Compare scoring methods on the same saved case rows.
path = DATA / "phenotype_scoring_method_comparison.csv"
if path.exists():
    df = pd.read_csv(path)
    print("\n--- Phenotype scoring methods ---")
    print("Rows:", len(df), "| Unique cases:", df["case_id"].nunique())
    print(
        df.groupby("method").agg(
            rows=("case_id", "size"),
            unique_cases=("case_id", "nunique"),
            mean_reciprocal_rank=("mrr", "mean"),
            top1=("top1", "mean"),
            top5=("top5", "mean"),
            top10=("top10", "mean"),
            median_target_rank=("target_rank", "median"),
        ).round(4).to_string()
    )

# Compare disease-level recovery metrics, keeping the task distinction clear.
for filename in [
    "phenotype_recovery_baseline_metrics.csv",
    "gene_informed_phenotype_recovery_metrics.csv",
    "gnn_phenotype_recovery_metrics.csv",
]:
    path = DATA / filename
    if not path.exists():
        continue
    df = pd.read_csv(path)
    print(f"\n--- {filename} ---")
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            print(f"{col}: mean={df[col].mean():.4f}, median={df[col].median():.4f}")

# Show the report's matched-profile comparison conclusions.
report = DATA / "rare_disease_final_research_report.md"
if report.exists():
    lines = report.read_text(errors="replace").splitlines()
    print("\n--- Report: matched-profile comparison and conclusion ---")
    print("\n".join(lines[378: min(len(lines), 385)]))
    print("\n".join(lines[-45:]))

print("\nAudit complete. No files changed; no models trained.")


=== STAGE 25: METHOD COMPARISON AUDIT ===

--- Phenotype scoring methods ---
Rows: 1812 | Unique cases: 906
         rows  unique_cases  mean_reciprocal_rank    top1    top5   top10  median_target_rank
method                                                                                       
ic_f1     906           906                 0.336  0.2638  0.4073  0.4890                12.0
jaccard   906           906                 0.295  0.2285  0.3389  0.4437                18.0

--- phenotype_recovery_baseline_metrics.csv ---
Value: mean=610.9106, median=0.5189

--- gene_informed_phenotype_recovery_metrics.csv ---
input_phenotype_count: mean=11.8160, median=9.0000
hidden_target_count: mean=12.3155, median=10.0000
candidate_gene_count: mean=1993.9995, median=2094.0000
candidate_phenotype_count: mean=9970.3619, median=9973.0000
recall_at_10: mean=0.1339, median=0.0909
reciprocal_rank: mean=0.3787, median=0.2000

--- gnn_phenotype_recovery_metrics.csv ---
input_count: mean=11.8456, media

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("=== STAGE 26: VALIDATION RESULT FILE AUDIT ===")

for filename in [
    "patient_phenotype_only_validation.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_ranker_bias_softmax_experiment_validation.csv",
    "phenotype_scoring_method_comparison.csv",
    "patient_phenotype_baseline_metrics_audit.csv",
]:
    path = DATA / filename
    if not path.exists():
        print(f"\nMissing: {filename}")
        continue

    df = pd.read_csv(path)
    print(f"\n--- {filename} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False, max_cols=10))

    for col in ["method", "model", "split", "target_rank", "mrr", "top1", "top5", "top10"]:
        if col in df.columns and df[col].nunique(dropna=True) < 20:
            print(f"{col} values:", df[col].value_counts(dropna=False).to_dict())

print("\nRead-only audit complete. No files changed; no models trained.")


=== STAGE 26: VALIDATION RESULT FILE AUDIT ===

--- patient_phenotype_only_validation.csv ---
Shape: (906, 10)
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'n_recognized_hpo', 'target_in_candidate_set', 'rank', 'top1', 'top5', 'top10', 'mrr']
                                       case_id     pmid diagnosis_ids  n_recognized_hpo  target_in_candidate_set  rank  top1  top5  top10      mrr
                       PMID_28830446_Patient_2 28830446   OMIM:615688                14                     True    19     0     0      0 0.052632
                       PMID_28830446_Patient_1 28830446   OMIM:615688                17                     True    19     0     0      0 0.052632
PMID_32480058_Case_F5P1_original_pedigree_II_2 32480058   OMIM:618052                 5                     True   133     0     0      0 0.007519
top1 values: {0: 667, 1: 239}
top5 values: {0: 537, 1: 369}
top10 values: {0: 463, 1: 443}

--- patient_phenotype_only_baseline.csv ---
Shape: (1959, 8)
Columns: ['case

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ic = pd.read_csv(DATA / "phenotype_scoring_method_comparison.csv")
ic = ic[ic["method"] == "ic_f1"].copy()
neural = pd.read_csv(DATA / "patient_ranker_bias_softmax_experiment_validation.csv")

ic = ic.rename(columns={
    "target_rank": "ic_rank",
    "mrr": "ic_mrr",
    "top1": "ic_top1",
    "top5": "ic_top5",
    "top10": "ic_top10"
})
neural = neural.rename(columns={
    "rank": "neural_rank",
    "mrr": "neural_mrr",
    "top1": "neural_top1",
    "top5": "neural_top5",
    "top10": "neural_top10"
})

print("=== STAGE 27: PAIRED VALIDATION COMPARISON ===")
print("IC-F1 rows / unique cases:", len(ic), ic["case_id"].nunique())
print("Neural rows / unique cases:", len(neural), neural["case_id"].nunique())

paired = ic.merge(neural, on="case_id", how="inner", validate="one_to_one")
print("Matched cases:", len(paired))
print("IC-F1-only cases:", len(set(ic.case_id) - set(neural.case_id)))
print("Neural-only cases:", len(set(neural.case_id) - set(ic.case_id)))

if "diagnosis_id" in paired.columns:
    # IC-F1 comparison file doesn't store target labels, so join the
    # saved baseline validation file to verify case-to-target mapping.
    base = pd.read_csv(DATA / "patient_phenotype_only_validation.csv")
    targets = base[["case_id", "diagnosis_ids"]].drop_duplicates()
    paired = paired.merge(targets, on="case_id", how="left", validate="many_to_one")
    paired["target_match"] = (
        paired["diagnosis_id"].astype(str) == paired["diagnosis_ids"].astype(str)
    )
    print("Target diagnosis agreement:", paired["target_match"].value_counts(dropna=False).to_dict())

for metric in ["mrr", "top1", "top5", "top10"]:
    a, b = f"ic_{metric}", f"neural_{metric}"
    print(f"\n{metric.upper()}:")
    print(f"  IC-F1 mean: {paired[a].mean():.4f}")
    print(f"  Neural mean: {paired[b].mean():.4f}")
    print(f"  Mean paired delta (IC-F1 - neural): {(paired[a] - paired[b]).mean():.4f}")

wins = (paired["ic_mrr"] > paired["neural_mrr"]).sum()
losses = (paired["ic_mrr"] < paired["neural_mrr"]).sum()
ties = (paired["ic_mrr"] == paired["neural_mrr"]).sum()
print("\nPer-case MRR outcomes — IC-F1 vs neural:")
print({"IC-F1 wins": int(wins), "Neural wins": int(losses), "Ties": int(ties)})

print("\nRead-only comparison complete. No files changed; no models trained.")


=== STAGE 27: PAIRED VALIDATION COMPARISON ===
IC-F1 rows / unique cases: 906 906
Neural rows / unique cases: 906 906
Matched cases: 906
IC-F1-only cases: 0
Neural-only cases: 0
Target diagnosis agreement: {True: 906}

MRR:
  IC-F1 mean: 0.3360
  Neural mean: 0.3341
  Mean paired delta (IC-F1 - neural): 0.0019

TOP1:
  IC-F1 mean: 0.2638
  Neural mean: 0.3013
  Mean paired delta (IC-F1 - neural): -0.0375

TOP5:
  IC-F1 mean: 0.4073
  Neural mean: 0.3631
  Mean paired delta (IC-F1 - neural): 0.0442

TOP10:
  IC-F1 mean: 0.4890
  Neural mean: 0.3951
  Mean paired delta (IC-F1 - neural): 0.0938

Per-case MRR outcomes — IC-F1 vs neural:
{'IC-F1 wins': 530, 'Neural wins': 350, 'Ties': 26}

Read-only comparison complete. No files changed; no models trained.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
N_BOOT = 10000
SEED = 42

ic = pd.read_csv(DATA / "phenotype_scoring_method_comparison.csv")
ic = ic[ic["method"] == "ic_f1"].copy()
neural = pd.read_csv(DATA / "patient_ranker_bias_softmax_experiment_validation.csv")

ic = ic.rename(columns={
    "mrr": "ic_mrr", "top1": "ic_top1",
    "top5": "ic_top5", "top10": "ic_top10"
})
neural = neural.rename(columns={
    "mrr": "neural_mrr", "top1": "neural_top1",
    "top5": "neural_top5", "top10": "neural_top10"
})

paired = ic.merge(neural, on="case_id", validate="one_to_one")
rng = np.random.default_rng(SEED)

print("=== STAGE 28: PAIRED BOOTSTRAP UNCERTAINTY ===")
print("Matched cases:", len(paired))
print("Bootstrap replicates:", N_BOOT)

for metric in ["mrr", "top1", "top5", "top10"]:
    delta = (
        paired[f"ic_{metric}"].to_numpy(dtype=float)
        - paired[f"neural_{metric}"].to_numpy(dtype=float)
    )
    indices = rng.integers(0, len(delta), size=(N_BOOT, len(delta)))
    boot = delta[indices].mean(axis=1)
    low, high = np.quantile(boot, [0.025, 0.975])

    print(f"\n{metric.upper()} — IC-F1 minus neural")
    print(f"  Paired difference: {delta.mean():+.4f}")
    print(f"  95% bootstrap CI: [{low:+.4f}, {high:+.4f}]")
    print("  CI excludes zero:", bool(low > 0 or high < 0))

print("\nBootstrap complete. Existing files unchanged; no models trained.")


=== STAGE 28: PAIRED BOOTSTRAP UNCERTAINTY ===
Matched cases: 906
Bootstrap replicates: 10000

MRR — IC-F1 minus neural
  Paired difference: +0.0019
  95% bootstrap CI: [-0.0433, +0.0476]
  CI excludes zero: False

TOP1 — IC-F1 minus neural
  Paired difference: -0.0375
  95% bootstrap CI: [-0.0850, +0.0088]
  CI excludes zero: False

TOP5 — IC-F1 minus neural
  Paired difference: +0.0442
  95% bootstrap CI: [-0.0077, +0.0971]
  CI excludes zero: False

TOP10 — IC-F1 minus neural
  Paired difference: +0.0938
  95% bootstrap CI: [+0.0397, +0.1468]
  CI excludes zero: True

Bootstrap complete. Existing files unchanged; no models trained.


In [ ]:

from pathlib import Path
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

print("=== STAGE 29: FINAL DELIVERABLE AUDIT ===")

required = [
    DATA / "rare_disease_final_research_report.md",
    DATA / "rare_disease_project_summary.csv",
    DATA / "patient_ranker_reconstructed_test_metrics.json",
    DATA / "patient_ranker_reconstructed_test_case_results.csv",
    DATA / "patient_phenotype_baseline_metrics_audit.csv",
    DATA / "phenotype_scoring_method_comparison.csv",
    MODEL / "rare_disease_predictor.py",
    MODEL / "rare_disease_data_loader.py",
    MODEL / "patient_level_ranker_best.pt",
    MODEL / "patient_ranker_bias_softmax_experiment.pt",
    MODEL / "rare_disease_gnn_filtered_best.pt",
]

print("\n--- Required artifact presence ---")
for path in required:
    status = "OK" if path.is_file() and path.stat().st_size > 0 else "MISSING/EMPTY"
    size = f"{path.stat().st_size:,} bytes" if path.is_file() else "-"
    print(f"{status:12} {path.relative_to(PROJECT)} ({size})")

print("\n--- Predictor source review ---")
for filename in ["rare_disease_predictor.py", "rare_disease_data_loader.py"]:
    path = MODEL / filename
    print(f"\n### {filename}")
    if path.exists():
        lines = path.read_text(errors="replace").splitlines()
        print("\n".join(f"{i+1:3}: {line}" for i, line in enumerate(lines[:100])))
    else:
        print("File not found.")

print("\n--- Saved test-metric caveats ---")
metrics_path = DATA / "patient_ranker_reconstructed_test_metrics.json"
if metrics_path.exists():
    obj = json.loads(metrics_path.read_text())
    print("Test cases:", obj.get("test_cases"))
    print("Caveats:")
    for caveat in obj.get("caveats", []):
        print("-", caveat)

print("\nAudit complete. Read-only; no files changed and no models trained.")


=== STAGE 29: FINAL DELIVERABLE AUDIT ===

--- Required artifact presence ---
OK           processed/rare_disease_final_research_report.md (9,093 bytes)
OK           processed/rare_disease_project_summary.csv (536 bytes)
OK           processed/patient_ranker_reconstructed_test_metrics.json (2,163 bytes)
OK           processed/patient_ranker_reconstructed_test_case_results.csv (257,098 bytes)
OK           processed/patient_phenotype_baseline_metrics_audit.csv (260 bytes)
OK           processed/phenotype_scoring_method_comparison.csv (103,269 bytes)
OK           model/rare_disease_predictor.py (2,993 bytes)
OK           model/rare_disease_data_loader.py (1,798 bytes)
OK           model/patient_level_ranker_best.pt (9,387,579 bytes)
OK           model/patient_ranker_bias_softmax_experiment.pt (9,387,364 bytes)
OK           model/rare_disease_gnn_filtered_best.pt (4,859,106 bytes)

--- Predictor source review ---

### rare_disease_predictor.py
  1: """Rare Disease Prediction AI — research 

In [ ]:

from pathlib import Path
import ast
import importlib.util

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL = PROJECT / "model"
source_path = MODEL / "rare_disease_predictor.py"

print("=== STAGE 30: PREDICTOR INITIALIZATION CHECK ===")

source = source_path.read_text(errors="replace")
tree = ast.parse(source)

print("\n--- Module-level assignments and imports ---")
for node in tree.body:
    if isinstance(node, (ast.Import, ast.ImportFrom, ast.Assign, ast.AnnAssign)):
        print(ast.get_source_segment(source, node))

print("\n--- Function definitions ---")
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        print(f"{node.name} (line {node.lineno})")

print("\n--- Initialization names referenced by predictor ---")
for name in ["ic", "phenotype_map", "gene_map", "disease_names"]:
    stores = [
        n.lineno for n in ast.walk(tree)
        if isinstance(n, ast.Name) and n.id == name
        and isinstance(n.ctx, (ast.Store, ast.Param))
    ]
    loads = [
        n.lineno for n in ast.walk(tree)
        if isinstance(n, ast.Name) and n.id == name
        and isinstance(n.ctx, ast.Load)
    ]
    print(f"{name}: assigned/parameter lines={stores}; referenced lines={loads}")

print("\n--- Import smoke test ---")
spec = importlib.util.spec_from_file_location("rare_disease_predictor_audit", source_path)
module = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(module)
    print("Import succeeded.")
    print("Initialization globals present:", {
        name: hasattr(module, name)
        for name in ["ic", "phenotype_map", "gene_map", "disease_names"]
    })
except Exception as exc:
    print(type(exc).__name__ + ":", str(exc))

print("\nRead-only check complete. No files changed; no models trained.")


=== STAGE 30: PREDICTOR INITIALIZATION CHECK ===

--- Module-level assignments and imports ---
import pandas as pd
import numpy as np

--- Function definitions ---
predict_diseases (line 10)
explain_predictions (line 73)

--- Initialization names referenced by predictor ---
ic: assigned/parameter lines=[]; referenced lines=[21, 15]
phenotype_map: assigned/parameter lines=[]; referenced lines=[32, 88]
gene_map: assigned/parameter lines=[]; referenced lines=[43]
disease_names: assigned/parameter lines=[]; referenced lines=[80]

--- Import smoke test ---
Import succeeded.
Initialization globals present: {'ic': False, 'phenotype_map': False, 'gene_map': False, 'disease_names': False}

Read-only check complete. No files changed; no models trained.


In [ ]:

from pathlib import Path
import shutil

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL = PROJECT / "model"

predictor_path = MODEL / "rare_disease_predictor.py"
loader_path = MODEL / "rare_disease_data_loader.py"
backup_path = MODEL / "rare_disease_predictor.py.stage31_backup"

assert predictor_path.exists(), f"Missing: {predictor_path}"
assert loader_path.exists(), f"Missing: {loader_path}"

source = predictor_path.read_text()
if "load_prediction_data" in source:
    print("Initialization import already present; no changes made.")
else:
    assert not backup_path.exists(), (
        f"Backup already exists: {backup_path}. "
        "Review it before making further changes."
    )

    # Keep the original source recoverable.
    shutil.copy2(predictor_path, backup_path)

    initialization = '''
from rare_disease_data_loader import load_prediction_data

_prediction_data = load_prediction_data()
phenotype_map = _prediction_data["phenotype_map"]
gene_map = _prediction_data["gene_map"]
ic = _prediction_data["ic"]
disease_names = _prediction_data["disease_names"]

'''
    # Insert after the module docstring and imports.
    marker = "import numpy as np\n"
    assert marker in source, "Expected import marker not found; source unchanged."
    updated = source.replace(marker, marker + initialization, 1)

    # Validate syntax before saving.
    compile(updated, str(predictor_path), "exec")
    predictor_path.write_text(updated)

    print("Predictor initialization added.")
    print("Backup:", backup_path)
    print("Checkpoint files were not modified.")

print("\nNext: we will import the predictor and test a sample prediction.")


Predictor initialization added.
Backup: /content/drive/MyDrive/Rare Disease ai/model/rare_disease_predictor.py.stage31_backup
Checkpoint files were not modified.

Next: we will import the predictor and test a sample prediction.


In [ ]:

from pathlib import Path
import sys
import traceback

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL = PROJECT / "model"

print("=== STAGE 32: SAMPLE PREDICTION SMOKE TEST ===")
sys.path.insert(0, str(MODEL))

try:
    import rare_disease_predictor as predictor

    print("Predictor imported.")
    print("Disease profiles loaded:", len(predictor.phenotype_map))
    print("HPO information-content terms loaded:", len(predictor.ic))
    print("Disease gene profiles loaded:", len(predictor.gene_map))

    # Select valid terms from the loaded data rather than inventing IDs.
    sample_terms = list(predictor.ic.keys())[:5]
    print("Sample HPO terms:", sample_terms)

    result = predictor.explain_predictions(
        patient_hpo_terms=sample_terms,
        top_k=5
    )

    print("\nTop-5 sample results:")
    print(result.to_string(index=False))

    assert not result.empty, "Prediction returned no results."
    assert "rank" in result.columns, "Missing rank column."
    assert result["rank"].tolist() == list(range(1, len(result) + 1))

    print("\nPASS: prediction returned ranked results.")
    print("Note: this is a software smoke test, not a clinical validation.")

except Exception as exc:
    print("\nFAIL:", type(exc).__name__, str(exc))
    traceback.print_exc()


=== STAGE 32: SAMPLE PREDICTION SMOKE TEST ===
Predictor imported.
Disease profiles loaded: 12879
HPO information-content terms loaded: 9982
Disease gene profiles loaded: 11168
Sample HPO terms: ['HP:0003002', 'HP:0025318', 'HP:0001324', 'HP:0031987', 'HP:0002578']

Top-5 sample results:
 rank  disease_id                                          disease_name  combined_score  matching_phenotypes matching_phenotype_ids matching_genes
    1 OMIM:613399                      Familial breast-ovarian cancer-1        0.625054                    2  HP:0003002;HP:0025318               
    2 OMIM:620442 Breast-ovarian cancer, familial, susceptibility to, 5        0.282488                    1             HP:0003002               
    3 OMIM:114480                                         Breast cancer        0.247526                    1             HP:0003002               
    4 OMIM:612555 Breast-Ovarian cancer, familial, susceptibility to, 2        0.220266                    1             HP

In [ ]:

import numpy as np
import pandas as pd
import rare_disease_predictor as predictor

print("=== STAGE 33: SCORE AND GENE-INPUT CHECK ===")

terms = list(predictor.ic.keys())[:5]
baseline = predictor.predict_diseases(terms, top_k=10)

assert not baseline.empty
assert np.isfinite(baseline["combined_score"]).all()
assert baseline["combined_score"].between(0, 1).all()
assert baseline["rank"].tolist() == list(range(1, len(baseline) + 1))

print("Phenotype-only scores finite and within [0, 1]: PASS")
print("Ranks sequential:", baseline["rank"].tolist())

# Pick a known disease-gene association from the loaded data.
candidate_disease = next(
    (d for d, genes in predictor.gene_map.items() if genes),
    None
)
if candidate_disease is None:
    print("No non-empty disease gene profiles available for gene-input test.")
else:
    test_gene = sorted(predictor.gene_map[candidate_disease])[0]
    print("\nSynthetic gene test disease:", candidate_disease)
    print("Synthetic gene symbol:", test_gene)

    without_gene = predictor.predict_diseases(terms, top_k=10)
    with_gene = predictor.predict_diseases(
        terms, patient_genes=[test_gene], top_k=10
    )

    assert np.isfinite(with_gene["combined_score"]).all()
    assert with_gene["combined_score"].between(0, 1).all()

    disease_row = with_gene[with_gene["disease_id"] == candidate_disease]
    if not disease_row.empty:
        print("Known gene-associated disease appears in Top-10.")
        print(disease_row.to_string(index=False))
    else:
        print("Known gene-associated disease is outside Top-10; this is allowed.")

    print("\nTop-5 without gene:")
    print(without_gene.head(5)[["rank", "disease_id", "combined_score"]].to_string(index=False))
    print("\nTop-5 with synthetic gene:")
    print(with_gene.head(5)[["rank", "disease_id", "combined_score"]].to_string(index=False))

print("\nPASS: scoring checks completed.")
print("Synthetic gene input is a software test, not biological evidence.")


=== STAGE 33: SCORE AND GENE-INPUT CHECK ===
Phenotype-only scores finite and within [0, 1]: PASS
Ranks sequential: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Synthetic gene test disease: OMIM:100100
Synthetic gene symbol: CHRM3
Known gene-associated disease appears in Top-10.
 rank  disease_id  phenotype_score  gene_score  combined_score  matching_phenotypes matching_genes
    3 OMIM:100100              0.0         1.0            0.25                    0          CHRM3

Top-5 without gene:
 rank  disease_id  combined_score
    1 OMIM:613399        0.625054
    2 OMIM:620442        0.282488
    3 OMIM:114480        0.247526
    4 OMIM:612555        0.220266
    5 OMIM:167000        0.198414

Top-5 with synthetic gene:
 rank  disease_id  combined_score
    1 OMIM:613399        0.625054
    2 OMIM:620442        0.282488
    3 OMIM:100100        0.250000
    4  ORPHA:2970        0.250000
    5 OMIM:114480        0.247526

PASS: scoring checks completed.
Synthetic gene input is a software test, not

In [ ]:

from pathlib import Path
from datetime import date

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
out = DATA / "final_project_handoff.md"

print("=== STAGE 34: FINAL PROJECT HANDOFF ===")

if out.exists():
    print("Handoff file already exists; not overwriting it:")
    print(out)
else:
    content = """# Rare Disease Prediction AI — Project Handoff

## Project status
Research prototype for ranking candidate rare diseases from Human Phenotype Ontology (HPO) terms, with optional gene evidence.

## Verified software checks
- Predictor module imports after initialization was connected to the existing data loader.
- Loaded 12,879 disease phenotype profiles, 9,982 HPO information-content terms, and 11,168 disease gene profiles.
- Sample prediction returned a ranked Top-5 list.
- Phenotype-only scores were finite and within [0, 1].
- Sequential ranking and optional gene-input behavior passed smoke tests.
- Predictor source backup: `model/rare_disease_predictor.py.stage31_backup`.

These are software smoke tests, not clinical validation or evidence that the synthetic inputs are biologically meaningful.

## Matched validation comparison
On the same 906 validation cases, with target diagnosis agreement verified:
- IC-F1: MRR 0.3360, Top-1 0.2638, Top-5 0.4073, Top-10 0.4890.
- Neural ranker: MRR 0.3341, Top-1 0.3013, Top-5 0.3631, Top-10 0.3951.

Paired bootstrap (10,000 replicates, seed 42), IC-F1 minus neural:
- MRR difference +0.0019; 95% CI [-0.0433, +0.0476].
- Top-1 difference -0.0375; 95% CI [-0.0850, +0.0088].
- Top-5 difference +0.0442; 95% CI [-0.0077, +0.0971].
- Top-10 difference +0.0938; 95% CI [+0.0397, +0.1468].

The saved results favor IC-F1 for Top-10, while the MRR difference is uncertain. Retain both methods for comparison.

## Held-out test caveats
The saved neural-ranker test evaluation was reconstructed from the checkpoint because the original inference source was not recovered or verified. Of 1,959 test cases, 467 target diagnoses were outside the ranker vocabulary. These metrics must be presented with those caveats.

## Known limitations and next steps
- Publication/reference overlap requires careful leakage review.
- Phenotype-recovery results and disease-ranking results are different tasks and must not be conflated.
- Gene evidence can raise a candidate score even when phenotype overlap is zero; display evidence components separately.
- Verify the final demo with representative valid HPO input and clearly label outputs as research candidates.
- Do not claim clinical validation, diagnostic accuracy, or readiness for patient care.

## Reproducibility
Existing model checkpoints and evaluation artifacts were preserved during the predictor smoke tests. No model was trained in the final validation stages.
"""
    out.write_text(content)
    print("Created:", out)
    print("Size:", f"{out.stat().st_size:,}", "bytes")

print("\nExisting research report and model checkpoints were not overwritten.")


=== STAGE 34: FINAL PROJECT HANDOFF ===
Created: /content/drive/MyDrive/Rare Disease ai/processed/final_project_handoff.md
Size: 2,463 bytes

Existing research report and model checkpoints were not overwritten.
